# AVN YOLO fine-tuning for Colab (accuracy update)

The Android app currently ships a fine-tuned YOLOv8s general detector. Its held-out test mAP50-95
is 0.387; recall is especially weak for bench (0.05), chair (0.18), person (0.52), and
transportation (0.57). The current run used only about 1.5k training images. This notebook adds a
larger, class-stratified COCO training sample while keeping Option E's validation and held-out test
splits frozen. The aim is a meaningful improvement on the same test set, not a cosmetic threshold
change.

**Run order for the accuracy update:** set a Colab GPU runtime; run the install/config cells; run
Option E to reconstruct the current dataset; run Option F to add COCO train-only images; then run
Steps 7A-10A. Do not run the old Steps 2-6 after Option E/F. The new model is exported only if it
passes the promotion gate against the current model. The existing Android asset is preserved until
that gate passes.

The model's eight classes are `person`, `transportation`, `pothole`, `bench`, `chair`, `fire
hydrant`, `stop sign`, and `parking meter`. `transportation` combines bicycle/car/motorcycle/bus/
truck. COCO can improve generic people/vehicles/benches/chairs/signs/hydrants, but it cannot make
metrics Egyptian-scene accuracy. `pothole` still has only a small public source; use real annotated
Egyptian road scenes for the final deployment claim.

Part B is a separate Egyptian-specific detector and remains gated on genuine Egyptian bounding-box
annotations for classes such as `tuk_tuk`, `microbus`, debris and open drains. COCO labels cannot
stand in for those labels. The notebook will not start Part B when its data gate fails.

**Data source note:** Option F uses the official COCO 2017 annotations and downloads only selected
COCO images into Colab. COCO publishes 118k train and 5k validation images; the full train archive
is about 18GB, so this cell samples at most 2,500 train images per COCO source class instead of
pulling the full archive. [COCO download page](https://cocodataset.org/#download).


## Step 0 -- Install dependencies


In [ ]:
!pip -q install "ultralytics>=8.4,<9" onnx onnxruntime roboflow scikit-learn pandas pyyaml


## Step 1 -- Configuration: general classes vs Egyptian target classes

`GENERAL_CLASSES` is Part A's broad, general-purpose taxonomy -- it collapses every vehicle
body type into one `transportation` class and adds `pothole`, on top of the everyday-object
classes (`person`, `bench`, `chair`, `fire hydrant`, `stop sign`, `parking meter`) the shipped
model already targets. It must stay in sync with
`android/app/src/main/resources/config/detector.json`'s `"classes"` **only once** a checkpoint
trained on this exact list has actually been exported -- see Step 10/11.

`EGYPT_TARGET_CLASSES` is the project's actual labeling guideline
(`datasets/labeling-guidelines.md`) -- a separate, optional/future set Part B would be trained
and evaluated on, independent of Part A/`GENERAL_CLASSES` above.
`EGYPT_HAZARD_ONLY_CLASSES` (a subset) have **no COCO equivalent at all**: no amount of COCO data
can ever satisfy their sufficiency floor, so Step 4B's gate on them is the real test of whether
genuine Egyptian-labeled data exists yet.

In [ ]:
GENERAL_CLASSES = [
    "person", "transportation", "pothole",
    "bench", "chair", "fire hydrant", "stop sign", "parking meter",
]  # The broad, general-purpose model (Part A). "transportation" merges every vehicle body type
# (bicycle/car/motorcycle/bus/truck from COCO, 2-/3-/4-wheeler from Option D below) into one
# class -- lane behaviour/hazard profile, not body type, is what this app's hazard logic needs
# (see datasets/data/detection/public_sources/hf_mixed_traffic_visaitech/SOURCE.md). "pothole" has
# no COCO equivalent and comes entirely from Option D's keremberke/pothole-segmentation.
# Keep in sync with android/app/src/main/resources/config/detector.json "classes" --
# but ONLY once a real fine-tuned checkpoint for this exact class list has been trained and
# exported (Step 10); never point detector.json at a class list no on-disk model actually emits.

# The project's own labeling guideline (datasets/labeling-guidelines.md), i.e. the Egyptian
# deliverable's class list -- generic classes shared with COCO plus Egypt-specific hazards.
# This is Part B: a separate, optional/future model gate, independent of Part A/GENERAL_CLASSES
# above. It is NOT required for the general "transportation" model to train or ship, and as of
# this writing it remains blocked on real Egyptian-labeled data regardless of GENERAL_CLASSES --
# see Step 4B. Do not collapse microbus/tuk_tuk here just because Part A uses one broad
# "transportation" class; that is a deliberate, separate design choice for whichever model
# actually ends up with enough real Egyptian annotations to train, not a decision to make now.
EGYPT_TARGET_CLASSES = [
    "person", "car", "microbus", "tuk_tuk", "bus", "truck", "motorcycle",
    "pothole", "debris", "open_drain", "curb_or_step", "low_hanging_obstacle",
]

# Classes with zero COCO equivalent: COCO data can never contribute a single box to these,
# so their counts below are a direct, un-fakeable measure of real Egyptian-labeling progress.
EGYPT_HAZARD_ONLY_CLASSES = [
    c for c in EGYPT_TARGET_CLASSES
    if c not in {"person", "car", "bus", "truck", "motorcycle"}
]

# Pretrained checkpoints to fine-tune and compare -- all are bounding-box detectors with
# published pretrained weights; none is trained from scratch here.
CANDIDATE_WEIGHTS = ["yolov8s.pt"]

EPOCHS = 80
PATIENCE = 20
IMG_SIZE = 640
BATCH_SIZE = 16

# Minimum-sanity floors (NOT an accuracy guarantee) checked before training is allowed to start.
MIN_IMAGES_PER_CLASS = 20
MIN_BOXES_PER_CLASS = 30
MIN_TOTAL_LABELED_IMAGES = 100

# Train/val/test split, performed by *capture session* (see Step 3) to avoid near-duplicate
# video frames leaking across splits.
SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}
SPLIT_SEED = 1337

import re
# Matches a leading session id in filenames like "session042_frame0183.jpg" or "sess-A12_00045.png".
# Any filename that does NOT match this pattern is treated as its own single-image session (the
# correct, leakage-safe default for independent photos such as stock/Roboflow images that were
# never extracted from a shared video).
SESSION_ID_PATTERN = re.compile(r"^(?P<session>[A-Za-z0-9]+(?:[_-][A-Za-z0-9]+)*?)[_-]?(?:frame)?\d{3,}$")

## Step 2 -- Get the data

Every option below ends by setting **`SOURCE_CLASSES`**: the exact ordered class list that the
label `.txt` files under `DATA_ROOT` use (i.e. `SOURCE_CLASSES[i]` is what class id `i` means in
*this* data source). Step 3 uses it to resolve every box to a real class name -- it is never
assumed to match `GENERAL_CLASSES` or `EGYPT_TARGET_CLASSES` unless it genuinely does.

Options A, B and D can all run before Step 3 -- each appends its own images/labels to the same
staging pattern, and Step 3's `discover()` call (run once, after whichever options you used) folds
them all into one `records` list. If you have a source with a different class id scheme per run
(e.g. your own hand-labeled Egyptian sessions), run Step 2 + Step 3 once per source instead, then
concatenate the resulting `records` lists before Step 4: `records = records_general + records_egypt_sessions`.

Use **Option A/B with `datasets/prepared/avn_coco/`** (zipped/Drive-uploaded from your machine)
and **Option D** (no upload needed) together for Part A -- between them they cover every
`GENERAL_CLASSES` entry with real data (see this session's validation report). For Part B you
need a data root containing real, reviewed Egyptian bounding-box labels -- your own
collected+labeled sessions per `datasets/collection-protocol.md` and
`datasets/labeling-guidelines.md` (label directly using `EGYPT_TARGET_CLASSES` order and set
`SOURCE_CLASSES = EGYPT_TARGET_CLASSES`), and/or a Roboflow export whose classes you explicitly
map in Step 2C below. **Never point any option at
`datasets/data/detection/egyptian/images/commons/` alone** -- those are unlabeled Commons
candidates (see Step 1's disclaimer) and Step 3 excludes that exact path regardless.

### Option A -- Upload a ZIP from your machine


In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()  # pick a .zip containing images/ and labels/ (YOLO format)
zip_name = next(iter(uploaded))
DATA_ROOT = pathlib.Path("/content/data")
DATA_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(DATA_ROOT)
print("Extracted to", DATA_ROOT)

# EDIT THIS to the real class list your label .txt files use, in id order. Defaults to the
# common case of uploading datasets/prepared/avn_coco/, whose dataset.yaml id order is
# 0 person, 1 bicycle, 2 car, 3 motorcycle, 4 bus, 5 truck, 6 bench, 7 chair, 8 fire hydrant,
# 9 stop sign, 10 parking meter -- the five vehicle-body-type ids (1-5) are remapped here to
# GENERAL_CLASSES's one broad "transportation" class (see Step 1); change this list if you
# zipped something else (e.g. your own Egyptian-labeled sessions -> SOURCE_CLASSES =
# EGYPT_TARGET_CLASSES, which needs no remapping since its ids already match its names 1:1).
AVN_COCO_SOURCE_CLASSES = [
    "person", "transportation", "transportation", "transportation", "transportation",
    "transportation", "bench", "chair", "fire hydrant", "stop sign", "parking meter",
]
SOURCE_CLASSES = AVN_COCO_SOURCE_CLASSES

### Option B -- Mount Google Drive and point at a folder/ZIP already there


In [ ]:
from google.colab import drive
import pathlib, zipfile

drive.mount("/content/drive")

DRIVE_PATH = "/content/drive/MyDrive/avn_dataset"  # <-- edit to your actual Drive path
source = pathlib.Path(DRIVE_PATH)
if source.is_file() and source.suffix == ".zip":
    DATA_ROOT = pathlib.Path("/content/data")
    DATA_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(source) as zf:
        zf.extractall(DATA_ROOT)
else:
    DATA_ROOT = source
print("Using", DATA_ROOT)

# EDIT THIS to the real class list your label .txt files use, in id order. If this points at
# datasets/prepared/avn_coco/, use the same remap Option A defines (its dataset.yaml id order
# 0 person,1 bicycle,2 car,3 motorcycle,4 bus,5 truck,6 bench,7 chair,8 fire hydrant,9 stop sign,
# 10 parking meter collapses ids 1-5 into GENERAL_CLASSES's one "transportation" class):
# SOURCE_CLASSES = ["person", "transportation", "transportation", "transportation",
#                    "transportation", "transportation", "bench", "chair", "fire hydrant",
#                    "stop sign", "parking meter"]
SOURCE_CLASSES = GENERAL_CLASSES

### Option C -- Download from Roboflow, with an explicit, reviewed class mapping

The API key is read **only** from Colab Secrets (the key icon in the left sidebar) -- it is
never typed into a cell, never printed, and never committed to this repo. Add a secret named
`ROBOFLOW_API_KEY` before running this cell, and grant this notebook access to it when prompted.

**This cell never assumes what a Roboflow project's classes mean.** It reads the exported
`data.yaml`, prints its exact class list, and then requires `ROBOFLOW_CLASS_MAP` (below) to
name every single one of them -- mapped to one of `EGYPT_TARGET_CLASSES`, or explicitly `None`
to discard. Any class missing from the map stops the notebook and asks for human review rather
than guessing a mapping by name similarity. `SOURCE_CLASSES` is then derived automatically from
that reviewed map, so Step 3 resolves every box by the mapping you actually confirmed.

**Known issue, checked live on 2026-11-14:** the project previously noted in
`datasets/README.md` --
`https://universe.roboflow.com/ahmed-saad-fdnmf/egypt_vehicle_detector` -- currently returns
**HTTP 404 "Project Not Found"**. Its class definitions could not be reviewed, so it is **not**
used as the default below. If your team finds a replacement/updated slug, fill in
`ROBOFLOW_WORKSPACE`/`ROBOFLOW_PROJECT` yourself after reviewing its own class list with this
same cell, and update `datasets/README.md`'s stale link.


In [ ]:
from google.colab import userdata
from roboflow import Roboflow
import pathlib, yaml

api_key = userdata.get("ROBOFLOW_API_KEY")  # never hardcode this; never print(api_key)

ROBOFLOW_WORKSPACE = ""  # <-- edit: your workspace slug, after reviewing its class list yourself
ROBOFLOW_PROJECT = ""    # <-- edit: your project slug
ROBOFLOW_VERSION = 1       # <-- edit: dataset version number

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")
DATA_ROOT = pathlib.Path(dataset.location)
print("Downloaded to", DATA_ROOT)
print("RECORD THIS: license/source/attribution for", ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT,
      "version", ROBOFLOW_VERSION, "-- copy it into this repo's datasets/*/SOURCE.md convention,",
      "never the raw images themselves.")

data_yaml_path = DATA_ROOT / "data.yaml"
roboflow_classes = yaml.safe_load(data_yaml_path.read_text())["names"]
print("\nRoboflow project's own class list (data.yaml):", roboflow_classes)

# EDIT THIS: map every single class above to an EGYPT_TARGET_CLASSES name, or to None to discard
# it. Left empty on purpose -- do not fill this in by guessing; have a human confirm each class's
# real meaning (e.g. open the dataset's example images on Roboflow Universe) before mapping it.
ROBOFLOW_CLASS_MAP = {
    # "<roboflow class name>": "<EGYPT_TARGET_CLASSES name or None>",
}

unmapped = [c for c in roboflow_classes if c not in ROBOFLOW_CLASS_MAP]
if unmapped:
    raise SystemExit(
        "STOP -- these Roboflow classes have no reviewed mapping yet: " + str(unmapped) + ".\n"
        "Open the dataset on Roboflow Universe, look at example images for each class, and add "
        "it to ROBOFLOW_CLASS_MAP above (or map it to None to discard it). This notebook refuses "
        "to guess a class's meaning from its name alone."
    )
invalid = {k: v for k, v in ROBOFLOW_CLASS_MAP.items() if v is not None and v not in EGYPT_TARGET_CLASSES}
if invalid:
    raise SystemExit(f"STOP -- these mapped targets are not in EGYPT_TARGET_CLASSES: {invalid}")
print("\nReviewed class mapping in effect:", ROBOFLOW_CLASS_MAP)

# Derived automatically from the human-reviewed map above -- id order matches Roboflow's own
# data.yaml, so Step 3 resolves each box by this, never by re-guessing against a different list.
SOURCE_CLASSES = [ROBOFLOW_CLASS_MAP[c] for c in roboflow_classes]


### Option D -- Two local public datasets already found and reviewed for this project

Unlike Options A-C, this cell needs no upload: it re-fetches two small, licensed,
already-vetted Hugging Face datasets directly (no auth), writes YOLO labels straight into
`GENERAL_CLASSES` id order, and sets `DATA_ROOT`/`SOURCE_CLASSES` accordingly -- a single
Step 3 run below folds both into `records`, feeding Part A (the broad general model), not
Part B. Full provenance for both lives in this repo at
`datasets/data/detection/public_sources/{hf_pothole_keremberke,hf_mixed_traffic_visaitech}/SOURCE.md`
(both already fetched there once by `datasets/tools/import_hf_pothole.py` and
`datasets/tools/import_hf_mixed_traffic.py` -- this cell repeats the fetch so a fresh Colab
runtime does not depend on that local copy being uploaded).

- `keremberke/pothole-segmentation` (CC BY 4.0): single class, maps directly to `pothole`.
  **Caveat (read before trusting any pothole metric)**: these are independent stock photos of
  potholes, not Egyptian road imagery -- 90 images / 120 boxes total. A model trained on this
  learns "what a pothole generically looks like," not Egyptian road conditions specifically.
- `visaitech/vehicle-mixed-traffic-detection` (CC BY 4.0): all 4 of its classes are now kept and
  mapped to the broad taxonomy -- `pedestrian` -> `person`, and `2-wheeler` / `3-wheeler` /
  `4-wheeler` (previously two of these were discarded as unmappable to a *narrow* taxonomy) all
  -> `transportation`, since the broad class makes body-type ambiguity irrelevant: a mixed
  motorcycle/scooter/bicycle box or a mixed car/van/bus/truck box is still unambiguously
  "transportation." Nothing from this source is discarded any more.
  **Caveat**: this is South Asian dashcam footage (6 source videos, 293 frames), not Egyptian,
  and its `pedestrian` contribution is small -- only 39 boxes across the whole set (see
  `SOURCE.md`). Treat any person/transportation metric learned from this source as "generic
  mixed-traffic," not "validated on Egyptian traffic."

Frames from the same source video are named `visaitech_<video_id>_frame<NNNN>` so Step 3's
`SESSION_ID_PATTERN` keeps them one session -- the same video cannot land split across
train/val/test. Pothole images are independent stock photos (no shared video), named with a
content hash plus a trailing literal "img" (not a digit) so that pattern deliberately does *not*
match -- each one becomes its own single-image session, the documented default for independent
photos.

In [ ]:
import hashlib, io, pathlib
from collections import defaultdict
from urllib.request import Request, urlopen

import pandas as pd
from huggingface_hub import snapshot_download

PUBLIC_SOURCES_ROOT = pathlib.Path("/content/data_public_sources")
images_dir = PUBLIC_SOURCES_ROOT / "images"
labels_dir = PUBLIC_SOURCES_ROOT / "labels"
images_dir.mkdir(parents=True, exist_ok=True)
labels_dir.mkdir(parents=True, exist_ok=True)
USER_AGENT = "EgyptianAssistiveVisionDataset/1.0 (research project)"

# -- keremberke/pothole-segmentation: single class, direct match --
POTHOLE_URL = (
    "https://huggingface.co/api/datasets/keremberke/pothole-segmentation/"
    "parquet/full/{split}/0.parquet"
)
pothole_target_id = GENERAL_CLASSES.index("pothole")
pothole_written = 0
for split in ("train", "validation", "test"):
    request = Request(POTHOLE_URL.format(split=split), headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=60) as response:
        df = pd.read_parquet(io.BytesIO(response.read()))
    for _, row in df.iterrows():
        image_bytes = row["image"]["bytes"]
        digest = hashlib.sha256(image_bytes).hexdigest()[:12]
        # Trailing literal "img" (not a digit) guarantees SESSION_ID_PATTERN cannot match --
        # each independent stock photo becomes its own single-image session.
        name = f"pothole_{digest}_img"
        (images_dir / f"{name}.jpg").write_bytes(image_bytes)
        width, height = int(row["width"]), int(row["height"])
        lines = []
        for x, y, box_w, box_h in row["objects"]["bbox"]:
            cx, cy = (x + box_w / 2) / width, (y + box_h / 2) / height
            lines.append(f"{pothole_target_id} {cx:.6f} {cy:.6f} {box_w / width:.6f} {box_h / height:.6f}")
        (labels_dir / f"{name}.txt").write_text("\n".join(lines) + "\n")
        pothole_written += 1
print(f"pothole: {pothole_written} images written (train+validation+test)")

# -- visaitech/vehicle-mixed-traffic-detection: broad-taxonomy mapping, nothing discarded --
# All 4 upstream classes are kept: pedestrian -> person; every vehicle-body-type class
# (2-/3-/4-wheeler) -> the one broad "transportation" class GENERAL_CLASSES uses (see Step 1 --
# this is the opposite of the old narrow mapping, which discarded 2-wheeler/4-wheeler because a
# narrow taxonomy had no single COCO class each box could be unambiguously assigned to).
VISAITECH_CLASS_MAP = {
    "pedestrian": "person",
    "2-wheeler": "transportation",
    "3-wheeler": "transportation",
    "4-wheeler": "transportation",
}
snapshot_dir = pathlib.Path(snapshot_download(
    repo_id="visaitech/vehicle-mixed-traffic-detection",
    repo_type="dataset",
    allow_patterns=["images/**", "labels/**", "classes.txt"],
))
visaitech_classes = (snapshot_dir / "classes.txt").read_text().strip().splitlines()
unmapped = [c for c in visaitech_classes if c not in VISAITECH_CLASS_MAP]
if unmapped:
    raise SystemExit(f"STOP -- unreviewed visaitech classes: {unmapped}")
print("visaitech reviewed class mapping in effect:", VISAITECH_CLASS_MAP)

video_frame_counters = defaultdict(int)
visaitech_written = visaitech_dropped_empty = 0
for split_dir in sorted((snapshot_dir / "images").iterdir()):
    for image_path in sorted(split_dir.glob("*.jpg")):
        label_path = snapshot_dir / "labels" / split_dir.name / f"{image_path.stem}.txt"
        kept_lines = []
        if label_path.is_file():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                target_name = VISAITECH_CLASS_MAP[visaitech_classes[int(parts[0])]]
                if target_name is None:
                    continue
                kept_lines.append(" ".join([str(GENERAL_CLASSES.index(target_name)), *parts[1:]]))
        if not kept_lines:
            visaitech_dropped_empty += 1
            continue  # no boxes in this frame (all 4 classes are mapped now, so this should be rare)
        video_id = image_path.stem.split("__")[0]
        frame_index = video_frame_counters[video_id]
        video_frame_counters[video_id] += 1
        # "<video_id>_frame<NNNN>" makes SESSION_ID_PATTERN group every frame from the same
        # source video into one session, so Step 5 can never split one video across train/val/test.
        name = f"visaitech_{video_id}_frame{frame_index:04d}"
        (images_dir / f"{name}.jpg").write_bytes(image_path.read_bytes())
        (labels_dir / f"{name}.txt").write_text("\n".join(kept_lines) + "\n")
        visaitech_written += 1
print(f"visaitech: {visaitech_written} images written (kept >=1 mapped box), "
      f"{visaitech_dropped_empty} dropped (empty label file), "
      f"{len(video_frame_counters)} distinct source videos")

DATA_ROOT = PUBLIC_SOURCES_ROOT
SOURCE_CLASSES = GENERAL_CLASSES  # labels above were written directly in this id order
print(f"\nPublic-source staging at {DATA_ROOT}; run Step 3 next to fold these into `records`.")

### Option E -- Rebuild the frozen baseline dataset (required before Option F; no upload)

`computer_vision/training/build_general_dataset.py` already ran directly against this repo's
real local files -- it discovered, remapped, session-safe-split, and materialized a real
`GENERAL_CLASSES` dataset, printing honest per-class/per-split box+image counts, a leakage check
(every video session lands wholly in one split), and a test-split class-presence check. Rather
than upload the ~335 MB of resulting image bytes (impractical to automate reliably), this cell
**re-derives the identical dataset from its original public sources** and applies that exact
same split:

- `avn_coco` images (1774 of them) are re-downloaded directly from the official public COCO 2017
  mirror (`images.cocodataset.org`) by filename -- the same real COCO ground truth, no upload.
- `visaitech`/`pothole` are re-fetched the same way Option D already does (HF `snapshot_download`
  / parquet, no upload), but instead of letting Step 5 re-derive a split from filenames, this
  cell looks up the **exact split this session already assigned** (by video_id for visaitech,
  by image content hash for pothole -- both embedded below, computed once from the local files).

The embedded `MANIFEST_JSON` below is small (~640 KB of plain text: filenames, COCO train/val
subfolder, remapped box coordinates, and the split lookups) -- no binary upload, no Drive mount,
no `files.upload()` dialog.

**Do not also run Option A/B/C/D or Steps 3-6 if using this option** -- it sets
`coco_dataset_yaml` and `COCO_GATE_PASSED` itself, which Option F expands before Step 7A. (Part B/Egyptian
is untouched either way.)

In [ ]:
# Embedded manifest from this session's local build_general_dataset.py run -- plain text,
# no binary upload. Contains: avn_coco's 1774 sampled (filename, coco_split, split, boxes)
# entries, visaitech's video_id->split lookup, and pothole's content-hash->split lookup.
MANIFEST_JSON = r'''{"general_classes": ["person", "transportation", "pothole", "bench", "chair", "fire hydrant", "stop sign", "parking meter"], "avn_coco_items": [{"filename": "000000000724.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.499507, 0.29674, 0.35864, 0.30616], [1, 0.358027, 0.58905, 0.057973, 0.06034], [1, 0.3592, 0.54257, 0.034347, 0.0163], [6, 0.565027, 0.54586, 0.050693, 0.05212]]}, {"filename": "000000000831.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.62012, 0.3825, 0.404384, 0.31468], [0, 0.872072, 0.53897, 0.022643, 0.04018], [0, 0.180826, 0.52664, 0.026697, 0.02848], [0, 0.776426, 0.52202, 0.018498, 0.012]]}, {"filename": "000000002139.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.487672, 0.502026, 0.724031, 0.981616]]}, {"filename": "000000002822.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.096523, 0.676396, 0.063078, 0.160417], [1, 0.420727, 0.805271, 0.024391, 0.008083], [1, 0.162305, 0.776458, 0.034672, 0.029]]}, {"filename": "000000003915.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.659039, 0.36251, 0.188859, 0.377729], [4, 0.038859, 0.90026, 0.069937, 0.199479]]}, {"filename": "000000004736.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.86043, 0.301667, 0.104672, 0.065681], [1, 0.586273, 0.300892, 0.108766, 0.056479], [0, 0.636617, 0.603192, 0.218516, 0.321033], [0, 0.661125, 0.411045, 0.113938, 0.20946], [0, 0.078969, 0.452406, 0.157938, 0.227347], [0, 0.53407, 0.469131, 0.051859, 0.091972], [1, 0.110773, 0.295293, 0.072359, 0.033169], [1, 0.690141, 0.304965, 0.074688, 0.052606], [0, 0.965312, 0.333427, 0.016437, 0.0677], [0, 0.974773, 0.450951, 0.030922, 0.077441], [0, 0.744102, 0.355798, 0.017766, 0.035587], [0, 0.459172, 0.291479, 0.017656, 0.054085], [0, 0.789664, 0.582582, 0.217984, 0.398545], [0, 0.776047, 0.354237, 0.020281, 0.03885], [1, 0.330227, 0.300998, 0.063578, 0.048756], [1, 0.219875, 0.300775, 0.086031, 0.046995], [1, 0.465586, 0.294354, 0.113891, 0.053592], [1, 0.792555, 0.302195, 0.046047, 0.046315], [0, 0.727531, 0.395, 0.014094, 0.04784]]}, {"filename": "000000004760.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.475573, 0.624, 0.089979, 0.272906], [3, 0.329052, 0.422375, 0.480354, 0.205031], [3, 0.788625, 0.335344, 0.221458, 0.124781]]}, {"filename": "000000005001.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.062359, 0.244385, 0.091, 0.352812], [0, 0.451688, 0.766292, 0.198875, 0.467417], [0, 0.209219, 0.754563, 0.177031, 0.490125], [0, 0.147805, 0.593469, 0.167234, 0.788271], [0, 0.240406, 0.365677, 0.108219, 0.392021], [0, 0.145547, 0.209448, 0.132312, 0.174937], [0, 0.330281, 0.225625, 0.125594, 0.321833], [0, 0.509719, 0.195302, 0.09125, 0.181729], [0, 0.427008, 0.38524, 0.193234, 0.402104], [0, 0.681742, 0.585396, 0.254484, 0.78875], [0, 0.734656, 0.209938, 0.110719, 0.249333], [0, 0.698594, 0.121344, 0.069094, 0.148312], [1, 0.817445, 0.434854, 0.070953, 0.206792], [0, 0.395023, 0.282604, 0.139672, 0.309792]]}, {"filename": "000000006270.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.541852, 0.447188, 0.803922, 0.831458], [1, 0.129359, 0.598979, 0.032625, 0.068]]}, {"filename": "000000007150.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.578367, 0.5, 0.843266, 1.0]]}, {"filename": "000000008498.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.674344, 0.748937, 0.059312, 0.037625], [1, 0.72725, 0.752352, 0.062792, 0.032328], [1, 0.026906, 0.763859, 0.050312, 0.041719], [1, 0.201552, 0.754266, 0.197437, 0.068562]]}, {"filename": "000000008899.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.250961, 0.827662, 0.062359, 0.177143], [1, 0.075867, 0.738562, 0.077891, 0.192968]]}, {"filename": "000000009483.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.166859, 0.565063, 0.219844, 0.869875]]}, {"filename": "000000009769.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.252187, 0.495052, 0.068406, 0.153937], [5, 0.871102, 0.56075, 0.018109, 0.043125], [0, 0.516766, 0.463927, 0.061844, 0.079938], [0, 0.431867, 0.454094, 0.075328, 0.073396], [1, 0.428422, 0.588104, 0.592625, 0.3935]]}, {"filename": "000000009975.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.50647, 0.49288, 0.3691, 0.79776]]}, {"filename": "000000010363.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.570961, 0.537632, 0.858078, 0.924737], [1, 0.881602, 0.073269, 0.223984, 0.146537]]}, {"filename": "000000011122.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.453391, 0.423969, 0.124313, 0.170729]]}, {"filename": "000000011149.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.62127, 0.484707, 0.44926, 0.483173], [1, 0.13682, 0.55968, 0.27364, 0.752267], [0, 0.07517, 0.166667, 0.15034, 0.313067], [0, 0.91713, 0.423213, 0.16574, 0.838213], [1, 0.28755, 0.385907, 0.22198, 0.457413]]}, {"filename": "000000011511.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.172195, 0.246358, 0.062047, 0.326336], [0, 0.257375, 0.22056, 0.078719, 0.302241], [0, 0.094531, 0.25306, 0.078469, 0.31431], [3, 0.275766, 0.692435, 0.433688, 0.406336], [1, 0.792547, 0.16278, 0.073187, 0.107759], [0, 0.05007, 0.286659, 0.096172, 0.350345]]}, {"filename": "000000011813.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.143453, 0.72328, 0.281141, 0.11564]]}, {"filename": "000000012109.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.575984, 0.535333, 0.664469, 0.646083], [7, 0.154211, 0.546073, 0.234266, 0.656188], [0, 0.248148, 0.293281, 0.016203, 0.048312], [1, 0.047359, 0.406781, 0.027656, 0.100896], [3, 0.331195, 0.299573, 0.056172, 0.037437], [3, 0.262836, 0.299906, 0.025516, 0.034021], [0, 0.04818, 0.298281, 0.007328, 0.035104], [0, 0.883227, 0.263552, 0.011703, 0.044188], [0, 0.949578, 0.215833, 0.022094, 0.049917], [1, 0.631125, 0.312865, 0.125219, 0.106854]]}, {"filename": "000000012147.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.099371, 0.417157, 0.133578, 0.261078], [0, 0.496095, 0.317712, 0.325523, 0.466144]]}, {"filename": "000000012251.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.875586, 0.088027, 0.076047, 0.140572]]}, {"filename": "000000012280.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.350333, 0.477578, 0.104292, 0.200969], [0, 0.722635, 0.439953, 0.029646, 0.097844], [0, 0.651802, 0.452219, 0.022479, 0.027406]]}, {"filename": "000000013546.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.817016, 0.309424, 0.168625, 0.597765], [0, 0.727141, 0.172153, 0.0295, 0.134659], [3, 0.393922, 0.302953, 0.146656, 0.0644], [3, 0.218258, 0.305776, 0.115328, 0.069624], [3, 0.556906, 0.288518, 0.127625, 0.057647], [3, 0.707086, 0.284071, 0.121266, 0.060988]]}, {"filename": "000000013595.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.873745, 0.40332, 0.252259, 0.402453], [0, 0.543086, 0.298875, 0.58069, 0.59325]]}, {"filename": "000000013800.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.493266, 0.445635, 0.069688, 0.198479], [0, 0.178586, 0.460865, 0.008422, 0.033438], [0, 0.272266, 0.467187, 0.0055, 0.02775], [0, 0.21368, 0.466177, 0.007922, 0.025563], [0, 0.336977, 0.47075, 0.010016, 0.019917], [0, 0.871539, 0.468135, 0.014547, 0.030604], [0, 0.884117, 0.472042, 0.013703, 0.02225], [3, 0.707242, 0.393927, 0.024203, 0.028688], [3, 0.734398, 0.385146, 0.023141, 0.022083], [0, 0.734406, 0.384625, 0.010469, 0.028875]]}, {"filename": "000000014180.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.489892, 0.729523, 0.480059, 0.514547], [0, 0.540029, 0.547187, 0.713471, 0.905625]]}, {"filename": "000000014226.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.22332, 0.908302, 0.261234, 0.150562], [4, 0.771914, 0.300438, 0.310109, 0.530333], [0, 0.586148, 0.417792, 0.520266, 0.707208], [4, 0.068258, 0.866292, 0.136516, 0.244958], [4, 0.965617, 0.270052, 0.068766, 0.424021]]}, {"filename": "000000014740.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.37064, 0.56081, 0.087467, 0.21938], [0, 0.187867, 0.57245, 0.188, 0.2073], [0, 0.057933, 0.49056, 0.115867, 0.35412], [0, 0.085413, 0.60816, 0.03104, 0.05468], [1, 0.284893, 0.59571, 0.100613, 0.10002], [5, 0.615907, 0.43289, 0.381147, 0.5617]]}, {"filename": "000000015802.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.317883, 0.36824, 0.263754, 0.2072]]}, {"filename": "000000015859.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.577711, 0.530258, 0.771703, 0.9163], [0, 0.117906, 0.092213, 0.222625, 0.184426], [4, 0.078578, 0.843384, 0.138344, 0.305035]]}, {"filename": "000000016364.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.344562, 0.345646, 0.0315, 0.03275], [5, 0.884289, 0.875, 0.160484, 0.22975]]}, {"filename": "000000017436.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.299813, 0.879109, 0.138378, 0.226875], [3, 0.208212, 0.909906, 0.323659, 0.150531], [3, 0.50762, 0.771562, 0.060146, 0.040375]]}, {"filename": "000000017751.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.77805, 0.418078, 0.04134, 0.045405], [6, 0.6236, 0.202432, 0.06352, 0.119219]]}, {"filename": "000000017905.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.938781, 0.950102, 0.079104, 0.099797], [0, 0.293677, 0.643016, 0.248729, 0.569813]]}, {"filename": "000000017959.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.023812, 0.870199, 0.042312, 0.171967], [0, 0.209898, 0.871569, 0.035547, 0.148712], [0, 0.047203, 0.858454, 0.026344, 0.173162], [0, 0.847602, 0.827155, 0.008953, 0.034637], [0, 0.889102, 0.824005, 0.011109, 0.031382], [0, 0.897969, 0.827283, 0.009719, 0.017611], [0, 0.930281, 0.819953, 0.009656, 0.040234], [0, 0.709789, 0.813829, 0.012797, 0.017213], [0, 0.677937, 0.832436, 0.008406, 0.028197], [0, 0.879492, 0.81397, 0.008359, 0.029766], [0, 0.727711, 0.828548, 0.007109, 0.033724], [0, 0.640836, 0.831639, 0.009859, 0.036253], [0, 0.948469, 0.827096, 0.007594, 0.013958]]}, {"filename": "000000018513.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.836937, 0.786102, 0.326126, 0.159016], [1, 0.167568, 0.793883, 0.177477, 0.096422], [1, 0.026205, 0.774195, 0.05241, 0.047078]]}, {"filename": "000000018794.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.632131, 0.326508, 0.456786, 0.377359]]}, {"filename": "000000019087.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.620727, 0.600316, 0.033797, 0.072389]]}, {"filename": "000000019432.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.594031, 0.234594, 0.112781, 0.226437], [4, 0.335305, 0.076812, 0.074922, 0.087], [4, 0.873875, 0.240885, 0.116281, 0.240437], [4, 0.497461, 0.189198, 0.113172, 0.313063], [4, 0.606656, 0.081854, 0.09825, 0.10325], [4, 0.703547, 0.078823, 0.092906, 0.099104], [4, 0.796375, 0.085542, 0.095625, 0.10675], [4, 0.686516, 0.237583, 0.086344, 0.223458], [4, 0.404531, 0.23451, 0.106063, 0.226938], [4, 0.311781, 0.232115, 0.104062, 0.228771], [0, 0.206547, 0.476865, 0.240406, 0.841979], [4, 0.886516, 0.087646, 0.091, 0.098875], [4, 0.162367, 0.072021, 0.092578, 0.092042]]}, {"filename": "000000019532.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.55593, 0.319008, 0.181203, 0.181203], [1, 0.118344, 0.667758, 0.132719, 0.082016], [1, 0.502094, 0.680852, 0.0605, 0.063172], [1, 0.295945, 0.680547, 0.387141, 0.117781], [1, 0.573859, 0.682352, 0.019875, 0.017641], [1, 0.208219, 0.679969, 0.057156, 0.047406], [1, 0.955555, 0.700844, 0.088891, 0.073813]]}, {"filename": "000000020489.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.837271, 0.347211, 0.057583, 0.056828], [4, 0.961208, 0.351922, 0.069667, 0.063281], [1, 0.164875, 0.575805, 0.32975, 0.690328], [0, 0.95549, 0.294609, 0.088021, 0.115531], [0, 0.702188, 0.288391, 0.049792, 0.142313], [7, 0.531833, 0.539328, 0.338583, 0.894375], [1, 0.97825, 0.699367, 0.037792, 0.222172], [4, 0.791031, 0.335492, 0.049146, 0.053203]]}, {"filename": "000000020640.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.500979, 0.594766, 0.995165, 0.262156], [1, 0.184658, 0.344, 0.087665, 0.04775], [7, 0.455236, 0.51232, 0.253962, 0.336516], [0, 0.053255, 0.310031, 0.04316, 0.0715], [1, 0.54829, 0.364172, 0.663844, 0.17725], [1, 0.849375, 0.324719, 0.30125, 0.065156], [1, 0.02191, 0.324117, 0.043396, 0.044891], [7, 0.202229, 0.294305, 0.036533, 0.017797], [0, 0.566887, 0.320617, 0.074009, 0.039047]]}, {"filename": "000000021343.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.286922, 0.828594, 0.073125, 0.306979], [7, 0.352469, 0.632042, 0.017969, 0.084292], [1, 0.939711, 0.590292, 0.120578, 0.090167], [1, 0.74157, 0.579115, 0.076422, 0.043688], [1, 0.656461, 0.571917, 0.075828, 0.029208], [1, 0.626062, 0.574292, 0.042156, 0.029792], [1, 0.544, 0.56324, 0.024281, 0.023979], [1, 0.830297, 0.577219, 0.102, 0.074229], [1, 0.78668, 0.577896, 0.096078, 0.069333], [7, 0.373945, 0.602031, 0.010672, 0.050021], [7, 0.380898, 0.598427, 0.009828, 0.046729], [1, 0.719328, 0.577635, 0.064625, 0.039104], [1, 0.877984, 0.587719, 0.0995, 0.069937], [1, 0.447352, 0.549771, 0.018328, 0.02925], [7, 0.308656, 0.731844, 0.039156, 0.132771], [7, 0.384172, 0.569562, 0.00625, 0.014208], [1, 0.526367, 0.564146, 0.017109, 0.01475]]}, {"filename": "000000021383.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.32963, 0.729961, 0.13526, 0.193255]]}, {"filename": "000000021469.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.515891, 0.762146, 0.123188, 0.272126], [4, 0.972211, 0.896191, 0.055047, 0.181201], [0, 0.637562, 0.575915, 0.140688, 0.205886], [0, 0.74207, 0.530118, 0.085453, 0.125079], [0, 0.834164, 0.712205, 0.331109, 0.575591], [0, 0.452992, 0.649094, 0.133016, 0.393189], [0, 0.691984, 0.528022, 0.05275, 0.107776], [4, 0.60993, 0.87685, 0.098672, 0.206614], [0, 0.345734, 0.458848, 0.040062, 0.103917], [0, 0.224773, 0.526535, 0.059766, 0.253701], [0, 0.404656, 0.478297, 0.044406, 0.173248], [0, 0.988969, 0.598002, 0.022063, 0.092815], [4, 0.456359, 0.742323, 0.084656, 0.100079], [4, 0.749984, 0.927677, 0.022813, 0.045433], [4, 0.813812, 0.963169, 0.033094, 0.072008]]}, {"filename": "000000022690.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.53932, 0.10136, 0.668516, 0.20272], [0, 0.156484, 0.066213, 0.137688, 0.126946]]}, {"filename": "000000022710.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.932104, 0.469078, 0.091458, 0.066312], [5, 0.982521, 0.624703, 0.034958, 0.069937]]}, {"filename": "000000022802.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.465773, 0.861625, 0.058571, 0.074], [0, 0.503571, 0.858125, 0.038384, 0.067844], [0, 0.43226, 0.859242, 0.043443, 0.072859], [0, 0.387494, 0.857102, 0.033817, 0.070672], [4, 0.733009, 0.899234, 0.125504, 0.047563], [0, 0.194133, 0.851742, 0.034731, 0.060078], [0, 0.125258, 0.851539, 0.033934, 0.050484], [0, 0.596733, 0.863977, 0.045644, 0.062141], [0, 0.15904, 0.858586, 0.048009, 0.080203], [0, 0.365667, 0.853773, 0.023138, 0.068891], [0, 0.02781, 0.854203, 0.02171, 0.018313], [3, 0.735843, 0.899789, 0.111499, 0.040641], [4, 0.04781, 0.874469, 0.092014, 0.029]]}, {"filename": "000000023096.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.768531, 0.330773, 0.446437, 0.217984], [7, 0.413125, 0.256367, 0.265458, 0.275172], [1, 0.201437, 0.187773, 0.045042, 0.013078], [1, 0.148042, 0.260156, 0.293042, 0.141688]]}, {"filename": "000000024610.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.225, 0.795698, 0.240313, 0.408604], [4, 0.014289, 0.842021, 0.017078, 0.071583], [4, 0.003063, 0.844792, 0.004625, 0.081]]}, {"filename": "000000024912.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.298391, 0.863083, 0.187094, 0.251625], [0, 0.548547, 0.801292, 0.249656, 0.397417], [0, 0.86593, 0.768, 0.043453, 0.106875], [0, 0.060953, 0.725771, 0.028437, 0.104208], [0, 0.025555, 0.703344, 0.032297, 0.077021], [1, 0.05575, 0.77101, 0.099969, 0.075187], [1, 0.214531, 0.73874, 0.13175, 0.067563], [7, 0.199969, 0.769531, 0.01475, 0.031063], [4, 0.488211, 0.861708, 0.208266, 0.276583], [0, 0.838336, 0.805177, 0.078547, 0.164354], [1, 0.114062, 0.695167, 0.080063, 0.05], [1, 0.302937, 0.721729, 0.076969, 0.0385], [1, 0.094164, 0.725812, 0.116766, 0.101833], [4, 0.816953, 0.869594, 0.075813, 0.114438], [0, 0.692187, 0.746187, 0.161812, 0.272833], [0, 0.951367, 0.75999, 0.075203, 0.095062], [0, 0.047625, 0.72374, 0.023187, 0.079646]]}, {"filename": "000000025115.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.207813, 0.436344, 0.145656, 0.299479], [5, 0.455219, 0.529656, 0.535313, 0.940687]]}, {"filename": "000000025181.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.162305, 0.55863, 0.019297, 0.048642], [3, 0.064023, 0.589344, 0.048578, 0.031522], [3, 0.152062, 0.576569, 0.024094, 0.018712], [0, 0.092828, 0.559977, 0.022219, 0.083841], [0, 0.166906, 0.551721, 0.007469, 0.021054], [3, 0.289711, 0.554403, 0.006766, 0.018501], [3, 0.308578, 0.553548, 0.006656, 0.017916]]}, {"filename": "000000025244.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.510958, 0.492062, 0.888875, 0.975062], [3, 0.61124, 0.648313, 0.659188, 0.694375], [3, 0.226844, 0.562516, 0.453688, 0.570219]]}, {"filename": "000000026029.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.275586, 0.831247, 0.205578, 0.29359], [4, 0.244156, 0.645559, 0.135562, 0.235734], [4, 0.642266, 0.670758, 0.1335, 0.273916], [4, 0.295906, 0.539883, 0.119844, 0.091748], [0, 0.568391, 0.615956, 0.253219, 0.325758], [0, 0.603187, 0.505967, 0.145563, 0.173333], [0, 0.277734, 0.578671, 0.1785, 0.257063]]}, {"filename": "000000026465.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.85702, 0.25056, 0.27808, 0.213493], [0, 0.13146, 0.395507, 0.26292, 0.786533]]}, {"filename": "000000026879.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.217992, 0.187139, 0.147672, 0.168167], [1, 0.313867, 0.170569, 0.074766, 0.069028], [1, 0.504859, 0.156792, 0.2875, 0.269139], [1, 0.691602, 0.520167, 0.099859, 0.165056], [1, 0.108062, 0.209625, 0.048156, 0.08625], [5, 0.228828, 0.278806, 0.019656, 0.079722]]}, {"filename": "000000026926.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.234137, 0.326617, 0.179716, 0.067266], [1, 0.405106, 0.316406, 0.176028, 0.13325], [5, 0.606135, 0.519977, 0.614019, 0.958578]]}, {"filename": "000000027768.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.968497, 0.677149, 0.062418, 0.132892], [1, 0.468472, 0.629502, 0.936944, 0.711716], [0, 0.322124, 0.655637, 0.066405, 0.080523], [0, 0.111626, 0.644412, 0.04518, 0.074216], [0, 0.709436, 0.622721, 0.076324, 0.133873], [0, 0.273856, 0.639551, 0.054608, 0.082369], [0, 0.061511, 0.642516, 0.052696, 0.07085], [0, 0.317247, 0.636405, 0.087467, 0.036046], [0, 0.026013, 0.643758, 0.035882, 0.068105]]}, {"filename": "000000027917.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.218844, 0.794953, 0.434719, 0.393255], [0, 0.384258, 0.495425, 0.026672, 0.055943], [0, 0.429844, 0.618667, 0.025063, 0.114316], [0, 0.399969, 0.625814, 0.027188, 0.129693], [1, 0.485781, 0.614481, 0.092188, 0.110991], [1, 0.29825, 0.591356, 0.067125, 0.025165], [1, 0.11268, 0.599976, 0.069297, 0.026038], [1, 0.235055, 0.585519, 0.019453, 0.019811], [1, 0.246625, 0.585212, 0.026906, 0.027453], [6, 0.501742, 0.364281, 0.134359, 0.190873]]}, {"filename": "000000028281.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.900865, 0.543234, 0.198271, 0.346938], [0, 0.593094, 0.493242, 0.429437, 0.722984], [4, 0.769208, 0.766469, 0.227875, 0.467063]]}, {"filename": "000000028317.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.14178, 0.196067, 0.21644, 0.302], [1, 0.58174, 0.696627, 0.83652, 0.404507]]}, {"filename": "000000028993.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.604028, 0.673595, 0.263971, 0.626405], [5, 0.326634, 0.589575, 0.155588, 0.394575], [0, 0.247157, 0.421193, 0.016503, 0.046111], [0, 0.110801, 0.42415, 0.017582, 0.045752], [0, 0.156797, 0.427141, 0.012549, 0.035752], [0, 0.168856, 0.425147, 0.014281, 0.038987], [0, 0.19299, 0.424469, 0.011961, 0.040441], [0, 0.228668, 0.422459, 0.018775, 0.04433], [0, 0.059175, 0.421879, 0.022435, 0.060098], [0, 0.095384, 0.428766, 0.013448, 0.030539], [0, 0.210588, 0.424706, 0.011732, 0.039608], [5, 0.464583, 0.632998, 0.151127, 0.489003]]}, {"filename": "000000030213.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.68418, 0.865991, 0.413984, 0.247751]]}, {"filename": "000000031217.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.515609, 0.542559, 0.208531, 0.37493]]}, {"filename": "000000033005.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.470898, 0.567547, 0.089203, 0.427817]]}, {"filename": "000000033104.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.276893, 0.49067, 0.443037, 0.87134], [0, 0.618236, 0.51124, 0.543435, 0.87864], [0, 0.801519, 0.20291, 0.251449, 0.32062], [0, 0.922664, 0.09775, 0.154673, 0.1955], [3, 0.742944, 0.05712, 0.220514, 0.11424], [0, 0.963353, 0.32195, 0.073294, 0.53054], [0, 0.060269, 0.27066, 0.120537, 0.47988]]}, {"filename": "000000033114.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.440727, 0.825469, 0.112922, 0.249438], [7, 0.269633, 0.747604, 0.038641, 0.068125], [7, 0.88975, 0.768823, 0.039125, 0.072063], [7, 0.362195, 0.740635, 0.024016, 0.044937], [7, 0.743219, 0.689698, 0.204344, 0.292813], [7, 0.855187, 0.750385, 0.023625, 0.037896], [7, 0.581305, 0.739229, 0.069359, 0.094]]}, {"filename": "000000033116.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.928906, 0.34399, 0.086313, 0.135646], [0, 0.786828, 0.575906, 0.226125, 0.591104], [0, 0.825594, 0.225146, 0.066156, 0.158417], [0, 0.5875, 0.232802, 0.078625, 0.158229], [0, 0.29907, 0.285927, 0.043734, 0.082729], [0, 0.268789, 0.236406, 0.024203, 0.059479], [0, 0.879148, 0.201781, 0.050828, 0.091729], [0, 0.51768, 0.313906, 0.067453, 0.099854], [0, 0.787367, 0.224417, 0.060922, 0.154583], [0, 0.669023, 0.114094, 0.062922, 0.089063], [0, 0.986109, 0.074083, 0.027781, 0.090708], [0, 0.950508, 0.275375, 0.070484, 0.108], [3, 0.250703, 0.416646, 0.113563, 0.082542], [4, 0.424195, 0.145292, 0.027078, 0.036542], [4, 0.443844, 0.150875, 0.047313, 0.048875], [4, 0.446313, 0.127375, 0.027906, 0.052125], [4, 0.327313, 0.151521, 0.053094, 0.051375], [4, 0.351555, 0.152135, 0.050828, 0.049354], [4, 0.298859, 0.194365, 0.038375, 0.035271], [4, 0.322398, 0.19475, 0.036141, 0.034875], [4, 0.661781, 0.357635, 0.039594, 0.041229], [4, 0.428031, 0.315188, 0.035125, 0.076208], [4, 0.652531, 0.199542, 0.039438, 0.034292], [4, 0.379789, 0.166469, 0.042609, 0.074688], [4, 0.273281, 0.193927, 0.039062, 0.035771], [4, 0.422492, 0.242083, 0.067328, 0.067417], [0, 0.507078, 0.160188, 0.046781, 0.099542]]}, {"filename": "000000033593.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.606133, 0.430843, 0.262234, 0.442998], [0, 0.41925, 0.424555, 0.156281, 0.414403], [4, 0.711906, 0.528735, 0.037625, 0.096019]]}, {"filename": "000000033820.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.599062, 0.737077, 0.429281, 0.391009], [0, 0.209672, 0.832934, 0.042344, 0.161972], [0, 0.264148, 0.834859, 0.033453, 0.159577], [0, 0.936883, 0.848486, 0.036516, 0.166127], [3, 0.313844, 0.88108, 0.08825, 0.071925], [3, 0.162195, 0.876232, 0.073016, 0.072277], [0, 0.841242, 0.842758, 0.040891, 0.169131]]}, {"filename": "000000034071.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.62607, 0.768204, 0.341766, 0.441714], [7, 0.279711, 0.746068, 0.370953, 0.480915]]}, {"filename": "000000034580.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.49025, 0.622073, 0.353937, 0.665729], [1, 0.935297, 0.369083, 0.129406, 0.179667], [1, 0.872383, 0.323594, 0.255234, 0.189562], [1, 0.722047, 0.352885, 0.058781, 0.095604], [1, 0.735281, 0.303573, 0.249312, 0.131146], [1, 0.671086, 0.276594, 0.284078, 0.126521], [1, 0.639117, 0.296802, 0.187141, 0.115521], [1, 0.51243, 0.294615, 0.105328, 0.074438], [1, 0.480875, 0.287594, 0.080969, 0.053688], [1, 0.43825, 0.269094, 0.04775, 0.022771], [1, 0.835313, 0.309906, 0.192, 0.137479], [1, 0.476969, 0.271156, 0.100844, 0.039479], [1, 0.381164, 0.288667, 0.020297, 0.025083]]}, {"filename": "000000036303.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.042977, 0.478646, 0.079203, 0.373042], [0, 0.487016, 0.49351, 0.970125, 0.981812]]}, {"filename": "000000036542.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.17098, 0.123507, 0.12936, 0.146053], [1, 0.33849, 0.11636, 0.1103, 0.086267], [1, 0.28259, 0.096867, 0.04662, 0.03816], [0, 0.55967, 0.542013, 0.34186, 0.89832], [0, 0.33925, 0.191093, 0.0679, 0.294507], [0, 0.24712, 0.190413, 0.10436, 0.297413], [0, 0.17016, 0.125787, 0.0426, 0.140693], [0, 0.1844, 0.19968, 0.06036, 0.275573], [1, 0.10541, 0.1158, 0.06038, 0.149413]]}, {"filename": "000000036748.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.444289, 0.353177, 0.078484, 0.022118], [3, 0.456836, 0.169733, 0.060328, 0.043002], [0, 0.812211, 0.578656, 0.282172, 0.631455], [0, 0.933656, 0.307652, 0.129531, 0.128748], [0, 0.150617, 0.588766, 0.282172, 0.768545], [0, 0.674805, 0.302993, 0.123516, 0.126501], [0, 0.261203, 0.455055, 0.163969, 0.528085], [0, 0.390711, 0.129365, 0.083391, 0.101492], [0, 0.121805, 0.211621, 0.136266, 0.220921], [0, 0.871695, 0.2857, 0.063672, 0.117808], [0, 0.714305, 0.277468, 0.058578, 0.116151], [0, 0.959586, 0.116823, 0.080828, 0.12768], [0, 0.204758, 0.118775, 0.089016, 0.108195], [3, 0.971094, 0.172928, 0.057813, 0.026777], [3, 0.258992, 0.162357, 0.031203, 0.02453], [4, 0.420523, 0.332339, 0.066109, 0.042578], [4, 0.065773, 0.324715, 0.055297, 0.038195], [4, 0.487, 0.333112, 0.067437, 0.035396], [0, 0.519008, 0.514705, 0.380484, 0.484512], [3, 0.660156, 0.171657, 0.076563, 0.036538], [4, 0.761437, 0.340912, 0.039594, 0.056814], [0, 0.458375, 0.179217, 0.903531, 0.358435]]}, {"filename": "000000036861.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.425198, 0.21407, 0.154479, 0.310391], [7, 0.273896, 0.210437, 0.159375, 0.310781]]}, {"filename": "000000037709.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.52925, 0.462271, 0.107375, 0.221375], [1, 0.686758, 0.609344, 0.135172, 0.082062], [7, 0.934477, 0.596542, 0.009672, 0.02775], [7, 0.804141, 0.599469, 0.010094, 0.024729], [7, 0.865531, 0.59425, 0.01, 0.015375], [1, 0.686906, 0.608427, 0.132812, 0.078937]]}, {"filename": "000000037743.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.555492, 0.468536, 0.565234, 0.586534], [3, 0.528742, 0.492834, 0.029891, 0.139344], [3, 0.616266, 0.491101, 0.068312, 0.148665], [3, 0.475641, 0.492927, 0.052531, 0.128571], [3, 0.425047, 0.50808, 0.02775, 0.124122]]}, {"filename": "000000037777.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.162642, 0.967391, 0.174716, 0.063043], [4, 0.401989, 0.879652, 0.142045, 0.11087], [4, 0.760327, 0.88313, 0.145938, 0.197391]]}, {"filename": "000000038048.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.432559, 0.51738, 0.262709, 0.25876], [0, 0.885318, 0.41895, 0.117458, 0.11922], [1, 0.283829, 0.4132, 0.041438, 0.01744]]}, {"filename": "000000038965.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.63651, 0.813907, 0.72698, 0.346907]]}, {"filename": "000000039617.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.495234, 0.621452, 0.075531, 0.425527], [0, 0.799719, 0.490246, 0.087844, 0.44822], [0, 0.176164, 0.625843, 0.088453, 0.420211], [0, 0.424781, 0.660023, 0.108156, 0.512225], [0, 0.667211, 0.409215, 0.074641, 0.136604]]}, {"filename": "000000039785.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.47551, 0.576573, 0.14358, 0.238747], [0, 0.83519, 0.358613, 0.03022, 0.09584], [3, 0.38813, 0.398773, 0.1123, 0.032587], [0, 0.16857, 0.147533, 0.0191, 0.04504]]}, {"filename": "000000039956.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.904828, 0.590527, 0.189312, 0.315152]]}, {"filename": "000000041728.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.37185, 0.60536, 0.02322, 0.052102], [3, 0.2866, 0.612372, 0.0404, 0.038198], [3, 0.36356, 0.627057, 0.04008, 0.058198], [3, 0.31811, 0.618438, 0.0121, 0.045886], [3, 0.34357, 0.618228, 0.02794, 0.046486]]}, {"filename": "000000042528.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.404961, 0.686515, 0.802984, 0.595495], [4, 0.238445, 0.557293, 0.428797, 0.476687]]}, {"filename": "000000042628.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.790539, 0.129894, 0.076734, 0.255282], [0, 0.858836, 0.114519, 0.057484, 0.226784], [0, 0.456289, 0.11216, 0.059328, 0.224319], [0, 0.389008, 0.099554, 0.072266, 0.1923], [0, 0.645844, 0.11811, 0.087688, 0.234953], [5, 0.661695, 0.666209, 0.118422, 0.327066]]}, {"filename": "000000043314.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.5347, 0.403444, 0.2698, 0.284654], [0, 0.53164, 0.407287, 0.27124, 0.278404], [0, 0.12017, 0.441263, 0.02426, 0.057207]]}, {"filename": "000000044580.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.396461, 0.570271, 0.221172, 0.604958], [0, 0.603375, 0.519542, 0.178656, 0.671917]]}, {"filename": "000000044802.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.794187, 0.334145, 0.192906, 0.278501], [0, 0.496102, 0.612061, 0.137359, 0.55452], [0, 0.411633, 0.112728, 0.051641, 0.09993], [0, 0.624922, 0.092307, 0.107281, 0.182365], [0, 0.525117, 0.093923, 0.091391, 0.177354], [0, 0.337344, 0.089883, 0.134937, 0.179766], [0, 0.797133, 0.246218, 0.124891, 0.27829], [0, 0.866313, 0.110375, 0.062063, 0.107447], [0, 0.758641, 0.097752, 0.071969, 0.155035], [0, 0.430703, 0.079157, 0.099531, 0.150023], [0, 0.859844, 0.082775, 0.127437, 0.161803], [4, 0.693164, 0.14103, 0.044141, 0.059672], [0, 0.292242, 0.089578, 0.068172, 0.173302], [0, 0.03268, 0.08089, 0.065297, 0.155176], [3, 0.152156, 0.123782, 0.139531, 0.083489], [0, 0.975867, 0.104778, 0.043797, 0.155878], [4, 0.314047, 0.142213, 0.105719, 0.072482], [4, 0.24018, 0.078009, 0.058766, 0.156019]]}, {"filename": "000000045281.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.427609, 0.437131, 0.206656, 0.510703], [0, 0.865102, 0.167412, 0.101953, 0.321335], [4, 0.275188, 0.26034, 0.052, 0.082459], [4, 0.479844, 0.231827, 0.068031, 0.092272], [0, 0.782461, 0.161358, 0.083078, 0.322717], [0, 0.823891, 0.178782, 0.035781, 0.169321], [0, 0.220758, 0.203595, 0.029578, 0.047143]]}, {"filename": "000000045537.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.941648, 0.663042, 0.116703, 0.349042], [4, 0.962797, 0.489906, 0.060375, 0.067396], [0, 0.951789, 0.917208, 0.096422, 0.165542], [4, 0.654344, 0.717719, 0.2495, 0.336438]]}, {"filename": "000000045792.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.923145, 0.599673, 0.12299, 0.098725], [0, 0.623873, 0.413284, 0.360359, 0.477484], [0, 0.884567, 0.428497, 0.199951, 0.435654], [0, 0.844575, 0.329273, 0.142353, 0.212075], [0, 0.751928, 0.304959, 0.113203, 0.157108], [0, 0.414387, 0.373873, 0.124363, 0.285948], [0, 0.709853, 0.280114, 0.081895, 0.102255], [0, 0.17817, 0.292623, 0.099444, 0.063938], [0, 0.262116, 0.46933, 0.121552, 0.25098], [0, 0.816936, 0.228962, 0.068023, 0.154297], [0, 0.250008, 0.274208, 0.082042, 0.098186], [0, 0.158587, 0.245931, 0.040474, 0.048105], [0, 0.410833, 0.250204, 0.03549, 0.048186], [0, 0.381618, 0.286609, 0.060686, 0.04482], [4, 0.162124, 0.44277, 0.18366, 0.131781], [1, 0.802557, 0.158325, 0.069886, 0.03348]]}, {"filename": "000000046173.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.58623, 0.573018, 0.08958, 0.657327], [0, 0.39585, 0.353604, 0.02882, 0.088889], [3, 0.37054, 0.45003, 0.12252, 0.089429], [0, 0.3498, 0.267793, 0.0088, 0.051201], [0, 0.42176, 0.280871, 0.0274, 0.074054], [0, 0.46084, 0.319054, 0.033, 0.03979], [0, 0.78271, 0.371171, 0.05794, 0.060841], [0, 0.3225, 0.325526, 0.02648, 0.165646], [0, 0.35524, 0.261892, 0.00592, 0.041502], [0, 0.38929, 0.265556, 0.00766, 0.04], [0, 0.50458, 0.322417, 0.02292, 0.051922], [0, 0.03272, 0.607553, 0.06544, 0.363694], [0, 0.04305, 0.381171, 0.08186, 0.186246], [0, 0.72628, 0.393033, 0.05596, 0.121261]]}, {"filename": "000000046743.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.720479, 0.209641, 0.167417, 0.275781], [0, 0.424521, 0.572008, 0.442458, 0.466359], [0, 0.382052, 0.115812, 0.105771, 0.206594], [0, 0.316875, 0.088578, 0.051667, 0.110688], [1, 0.802625, 0.226969, 0.39475, 0.453937], [0, 0.574146, 0.1045, 0.077667, 0.109688], [0, 0.447781, 0.077219, 0.028604, 0.054281], [1, 0.619469, 0.152602, 0.062646, 0.116391], [7, 0.513188, 0.09632, 0.035542, 0.059203], [0, 0.413104, 0.069398, 0.028417, 0.041328], [1, 0.518208, 0.068094, 0.030333, 0.016344], [1, 0.620552, 0.083867, 0.085979, 0.103172], [1, 0.797833, 0.242461, 0.404333, 0.484922]]}, {"filename": "000000046764.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.915523, 0.775235, 0.069078, 0.309061], [4, 0.912008, 0.627758, 0.070234, 0.143216], [4, 0.836641, 0.605775, 0.034938, 0.109859], [4, 0.86093, 0.614296, 0.031516, 0.120047], [0, 0.325336, 0.414613, 0.306641, 0.824718], [0, 0.157414, 0.569812, 0.203891, 0.333333], [0, 0.952039, 0.805141, 0.095922, 0.389718], [0, 0.607289, 0.620786, 0.493609, 0.633685], [0, 0.78607, 0.518357, 0.072391, 0.123568], [0, 0.084516, 0.87865, 0.166031, 0.130352], [4, 0.983367, 0.591585, 0.033266, 0.054437]]}, {"filename": "000000047639.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.337359, 0.564333, 0.243969, 0.584637], [3, 0.256281, 0.378513, 0.084, 0.102131], [0, 0.848289, 0.275714, 0.012516, 0.041452]]}, {"filename": "000000048972.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.897021, 0.80593, 0.11575, 0.155047], [1, 0.63176, 0.7645, 0.122187, 0.141344], [1, 0.789167, 0.79307, 0.175792, 0.207266], [1, 0.155594, 0.794398, 0.104104, 0.152297], [0, 0.30324, 0.758805, 0.112646, 0.249609], [0, 0.051198, 0.724711, 0.035813, 0.062578], [0, 0.127708, 0.72232, 0.026042, 0.063641], [0, 0.15599, 0.697461, 0.017438, 0.021922], [1, 0.347146, 0.791375, 0.230375, 0.124469], [1, 0.695, 0.656813, 0.584625, 0.190156], [0, 0.06825, 0.719234, 0.02125, 0.057813], [0, 0.973375, 0.764406, 0.052375, 0.172531], [1, 0.115229, 0.71868, 0.060625, 0.040109], [1, 0.216698, 0.688641, 0.037896, 0.012719], [0, 0.380042, 0.703422, 0.026208, 0.031187], [1, 0.426646, 0.763133, 0.122042, 0.109953]]}, {"filename": "000000049346.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.47224, 0.816852, 0.0754, 0.280918], [0, 0.74721, 0.74459, 0.1271, 0.51082], [0, 0.90294, 0.690672, 0.14928, 0.618656]]}, {"filename": "000000049985.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.204531, 0.298438, 0.111875, 0.194083], [4, 0.495508, 0.337073, 0.117984, 0.211229], [4, 0.303148, 0.29899, 0.105359, 0.176021], [4, 0.321266, 0.150438, 0.078156, 0.046458], [4, 0.745227, 0.238979, 0.082859, 0.051125], [4, 0.665891, 0.230094, 0.076281, 0.048604], [4, 0.603375, 0.322469, 0.094375, 0.155063], [4, 0.099437, 0.237521, 0.114594, 0.094375], [4, 0.689961, 0.105208, 0.047391, 0.020292], [4, 0.244836, 0.109948, 0.068141, 0.038187], [4, 0.172328, 0.136073, 0.074969, 0.044771], [4, 0.268023, 0.188458, 0.085047, 0.055125], [0, 0.400008, 0.283771, 0.096453, 0.273083], [4, 0.331289, 0.200771, 0.049016, 0.057792]]}, {"filename": "000000050380.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.094906, 0.794918, 0.185312, 0.407119], [0, 0.401969, 0.396393, 0.211875, 0.614848], [0, 0.689672, 0.494192, 0.344844, 0.988384], [0, 0.055453, 0.6013, 0.110906, 0.693513]]}, {"filename": "000000050485.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.833734, 0.561146, 0.0645, 0.083458], [1, 0.44793, 0.560104, 0.138109, 0.102708], [5, 0.544602, 0.596177, 0.104828, 0.207937], [1, 0.407453, 0.557771, 0.125531, 0.072833], [1, 0.495844, 0.513854, 0.148438, 0.076333], [1, 0.4595, 0.613979, 0.093, 0.07125], [1, 0.676383, 0.572823, 0.177891, 0.176854]]}, {"filename": "000000050679.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.811969, 0.458829, 0.137313, 0.129742], [1, 0.129148, 0.483372, 0.156172, 0.079344], [1, 0.623812, 0.487564, 0.053062, 0.052646], [1, 0.572828, 0.474707, 0.063563, 0.08459], [1, 0.679508, 0.488115, 0.052797, 0.051733], [1, 0.964375, 0.497096, 0.030312, 0.032178], [1, 0.724852, 0.485527, 0.043172, 0.056206], [1, 0.930078, 0.499719, 0.025844, 0.028478], [1, 0.044438, 0.484321, 0.088875, 0.067845], [1, 0.902523, 0.503384, 0.022141, 0.020773], [1, 0.657563, 0.469204, 0.047313, 0.029555], [1, 0.886031, 0.49719, 0.010063, 0.034192]]}, {"filename": "000000050943.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.511391, 0.650408, 0.047094, 0.097555]]}, {"filename": "000000051070.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.500375, 0.401125, 0.227708, 0.159562], [1, 0.507365, 0.168055, 0.377479, 0.161047]]}, {"filename": "000000051334.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.21089, 0.740199, 0.12078, 0.256525], [1, 0.27612, 0.255756, 0.05256, 0.062599], [1, 0.28274, 0.164562, 0.05884, 0.053528], [1, 0.31396, 0.139403, 0.02664, 0.024695], [1, 0.27961, 0.213515, 0.04438, 0.025862], [1, 0.39841, 0.172626, 0.0379, 0.049496], [1, 0.95434, 0.692812, 0.07968, 0.21008], [1, 0.87984, 0.714589, 0.13612, 0.167056], [1, 0.75061, 0.775332, 0.13174, 0.054642], [1, 0.37167, 0.138926, 0.0203, 0.040504], [1, 0.57527, 0.796286, 0.06702, 0.022228], [1, 0.42177, 0.245836, 0.06114, 0.074164], [1, 0.41558, 0.194337, 0.05656, 0.052918], [1, 0.68184, 0.786207, 0.04708, 0.041963]]}, {"filename": "000000051938.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.793781, 0.446594, 0.006969, 0.026896], [0, 0.475281, 0.455052, 0.080906, 0.249438], [0, 0.652672, 0.472052, 0.017719, 0.031688], [0, 0.579, 0.504167, 0.027531, 0.074417], [0, 0.642344, 0.491719, 0.016906, 0.056896], [0, 0.832414, 0.435385, 0.012172, 0.034312], [0, 0.780109, 0.44949, 0.008875, 0.024729], [0, 0.428586, 0.523771, 0.011797, 0.045375], [0, 0.628773, 0.439958, 0.010328, 0.042625], [0, 0.537398, 0.492104, 0.009266, 0.027542], [0, 0.804297, 0.437906, 0.010438, 0.022771]]}, {"filename": "000000051961.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.624215, 0.495844, 0.162342, 0.230594]]}, {"filename": "000000052007.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.441948, 0.352805, 0.512354, 0.516859], [0, 0.802583, 0.519914, 0.093875, 0.298734], [0, 0.711312, 0.506547, 0.106708, 0.223281], [0, 0.496958, 0.444734, 0.081083, 0.091], [0, 0.34976, 0.408047, 0.092979, 0.066219], [0, 0.099937, 0.425469, 0.012333, 0.022281], [0, 0.178542, 0.433484, 0.00775, 0.014094]]}, {"filename": "000000052192.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.477217, 0.539914, 0.358868, 0.660734], [0, 0.729811, 0.377836, 0.101226, 0.158516], [5, 0.942854, 0.458375, 0.060236, 0.108094], [0, 0.811851, 0.338063, 0.072712, 0.133781]]}, {"filename": "000000052461.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.896453, 0.297758, 0.18525, 0.591012], [7, 0.669117, 0.228399, 0.187797, 0.456384], [1, 0.498516, 0.495279, 0.997031, 0.988202]]}, {"filename": "000000053977.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.90442, 0.656063, 0.14856, 0.292605], [0, 0.07339, 0.490629, 0.05766, 0.204551], [0, 0.39029, 0.52741, 0.1501, 0.193263], [0, 0.97237, 0.432395, 0.05526, 0.172275], [1, 0.9405, 0.611347, 0.08712, 0.170958], [0, 0.47364, 0.533967, 0.0808, 0.279251]]}, {"filename": "000000053994.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.559137, 0.438141, 0.454491, 0.694375], [1, 0.084657, 0.045633, 0.095509, 0.030609], [1, 0.044262, 0.042648, 0.066861, 0.023297], [1, 0.207225, 0.055914, 0.229168, 0.042328]]}, {"filename": "000000054607.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.154828, 0.246522, 0.27675, 0.47534]]}, {"filename": "000000055184.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.573813, 0.536602, 0.85, 0.926797]]}, {"filename": "000000055425.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.329648, 0.71261, 0.353203, 0.337077], [1, 0.409375, 0.670532, 0.266219, 0.210292], [1, 0.47407, 0.606221, 0.177734, 0.212735], [1, 0.543063, 0.53761, 0.094688, 0.112923], [1, 0.501828, 0.593622, 0.178688, 0.162359], [0, 0.466984, 0.508967, 0.082031, 0.15643], [0, 0.549977, 0.467098, 0.029234, 0.046597], [0, 0.605047, 0.527505, 0.073937, 0.232213], [0, 0.653562, 0.49381, 0.015281, 0.073299], [0, 0.245969, 0.483758, 0.03575, 0.043132], [1, 0.206188, 0.512589, 0.139094, 0.16238], [0, 0.579617, 0.444196, 0.037453, 0.046054], [0, 0.617148, 0.427797, 0.019047, 0.045825], [0, 0.411, 0.445522, 0.011719, 0.017015], [0, 0.466687, 0.448747, 0.013594, 0.026054], [0, 0.663852, 0.466273, 0.012266, 0.048079], [0, 0.441383, 0.464144, 0.010703, 0.028455], [0, 0.269336, 0.508246, 0.021047, 0.091566], [1, 0.546734, 0.459781, 0.03425, 0.021816]]}, {"filename": "000000055629.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.154573, 0.586922, 0.307188, 0.27525], [0, 0.224719, 0.485023, 0.449437, 0.566297]]}, {"filename": "000000056852.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.503365, 0.506742, 0.413479, 0.955047]]}, {"filename": "000000057597.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.164781, 0.218181, 0.104281, 0.037347], [1, 0.593703, 0.279871, 0.148781, 0.071103], [1, 0.534, 0.230904, 0.055719, 0.039601], [0, 0.593484, 0.5502, 0.196, 0.384812], [0, 0.726797, 0.583908, 0.164562, 0.386268], [0, 0.879609, 0.457688, 0.139406, 0.247347], [0, 0.285461, 0.334507, 0.025984, 0.09662], [0, 0.259461, 0.354354, 0.034141, 0.044014], [0, 0.417063, 0.362899, 0.023469, 0.048192], [0, 0.225094, 0.251291, 0.028156, 0.068216], [0, 0.413187, 0.317242, 0.0305, 0.069695], [0, 0.446758, 0.316303, 0.029516, 0.077629], [3, 0.084469, 0.267723, 0.168813, 0.048404], [3, 0.351625, 0.290293, 0.041469, 0.01669], [0, 0.893375, 0.482007, 0.156094, 0.347723], [1, 0.838617, 0.271866, 0.053328, 0.056831], [1, 0.828961, 0.272312, 0.077453, 0.059601], [1, 0.418898, 0.241432, 0.129016, 0.11831], [3, 0.229187, 0.276784, 0.050906, 0.042347], [0, 0.429898, 0.361854, 0.019578, 0.046197], [1, 0.534617, 0.236514, 0.055453, 0.054343], [0, 0.323641, 0.335387, 0.023594, 0.096127], [0, 0.356578, 0.324495, 0.0185, 0.076408], [3, 0.563203, 0.358615, 0.1065, 0.051408], [3, 0.641586, 0.351925, 0.089391, 0.028779], [3, 0.645813, 0.361667, 0.039375, 0.006103]]}, {"filename": "000000057979.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.794657, 0.477844, 0.400618, 0.578469]]}, {"filename": "000000058089.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.18483, 0.733138, 0.36966, 0.516847], [1, 0.13919, 0.358423, 0.15566, 0.150541], [7, 0.58482, 0.524264, 0.59232, 0.925526]]}, {"filename": "000000058735.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.50124, 0.463898, 0.997521, 0.927797], [7, 0.53876, 0.525398, 0.595938, 0.891641]]}, {"filename": "000000059598.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.880562, 0.165635, 0.098875, 0.109229], [0, 0.959062, 0.291135, 0.081438, 0.387354], [4, 0.91082, 0.338833, 0.117672, 0.25275]]}, {"filename": "000000059635.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.796142, 0.86682, 0.150597, 0.204141], [0, 0.668302, 0.382805, 0.208128, 0.217797], [0, 0.405463, 0.197547, 0.05607, 0.118687], [0, 0.795525, 0.119727, 0.054383, 0.106484], [0, 0.183385, 0.712445, 0.063025, 0.078297], [0, 0.391852, 0.654875, 0.048601, 0.052219], [0, 0.794681, 0.611625, 0.051091, 0.106344], [0, 0.23714, 0.63443, 0.042675, 0.097203], [0, 0.956831, 0.601719, 0.048436, 0.128], [0, 0.033313, 0.638453, 0.056667, 0.111969], [0, 0.963241, 0.097758, 0.044383, 0.110109], [0, 0.878076, 0.105508, 0.040391, 0.093953], [1, 0.550802, 0.131148, 0.042263, 0.043734], [1, 0.554187, 0.621352, 0.048498, 0.046516], [1, 0.526728, 0.130172, 0.053868, 0.029156], [1, 0.702109, 0.618922, 0.05072, 0.039875], [1, 0.679722, 0.129102, 0.029938, 0.024078], [0, 0.451502, 0.346758, 0.123128, 0.319484]]}, {"filename": "000000060090.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.871492, 0.462163, 0.018453, 0.097423], [0, 0.841437, 0.478381, 0.013219, 0.095532], [0, 0.850383, 0.472518, 0.009078, 0.103995], [0, 0.971914, 0.49234, 0.014078, 0.112293], [0, 0.871148, 0.549409, 0.019797, 0.127281]]}, {"filename": "000000060347.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.486479, 0.648648, 0.546542, 0.563078], [3, 0.614333, 0.658453, 0.664458, 0.410094]]}, {"filename": "000000060362.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.05848, 0.752669, 0.11696, 0.180562], [1, 0.93845, 0.768764, 0.1231, 0.161573], [7, 0.2584, 0.707865, 0.0176, 0.044944], [7, 0.74713, 0.69309, 0.02282, 0.060337]]}, {"filename": "000000061108.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.586289, 0.636562, 0.804828, 0.722583], [1, 0.744672, 0.031719, 0.040656, 0.052271], [3, 0.107508, 0.471219, 0.194703, 0.316021], [3, 0.223312, 0.257865, 0.274719, 0.285396], [3, 0.345594, 0.30149, 0.170875, 0.285146], [1, 0.777641, 0.025573, 0.027281, 0.029104], [1, 0.846328, 0.028771, 0.035469, 0.026875], [1, 0.894695, 0.032917, 0.042453, 0.02775]]}, {"filename": "000000061181.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.0771, 0.5469, 0.01116, 0.043458], [1, 0.78775, 0.580062, 0.01826, 0.044735], [1, 0.67388, 0.41028, 0.06088, 0.135452], [1, 0.92643, 0.773427, 0.14302, 0.3519], [1, 0.94733, 0.5131, 0.10458, 0.193801], [1, 0.34611, 0.586184, 0.14166, 0.255296], [1, 0.13977, 0.686231, 0.1457, 0.18], [1, 0.32605, 0.771916, 0.16446, 0.222461], [1, 0.27261, 0.397492, 0.07454, 0.11162], [0, 0.87658, 0.499159, 0.02112, 0.100685], [0, 0.7446, 0.460997, 0.01336, 0.028287], [0, 0.75417, 0.462461, 0.00958, 0.01919], [0, 0.0563, 0.49715, 0.02592, 0.049315], [0, 0.78455, 0.461994, 0.01446, 0.022181], [0, 0.09712, 0.518816, 0.0164, 0.100935], [0, 0.10185, 0.493022, 0.01866, 0.093458], [1, 0.88257, 0.90514, 0.15082, 0.163302], [1, 0.82162, 0.953925, 0.189, 0.069657], [1, 0.63148, 0.918162, 0.1506, 0.163676], [1, 0.7431, 0.42933, 0.10688, 0.150374], [1, 0.5642, 0.884704, 0.14048, 0.188224], [1, 0.52907, 0.771667, 0.13306, 0.238474], [1, 0.13751, 0.789361, 0.12838, 0.167134], [1, 0.13594, 0.94419, 0.18004, 0.111558], [1, 0.75031, 0.822648, 0.1439, 0.157632], [1, 0.34666, 0.916838, 0.16768, 0.166324], [1, 0.80728, 0.76486, 0.10548, 0.130841], [1, 0.70297, 0.660919, 0.09078, 0.126573], [1, 0.62626, 0.418972, 0.0388, 0.05595]]}, {"filename": "000000061219.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.633219, 0.710391, 0.126813, 0.123325], [0, 0.478453, 0.669792, 0.033031, 0.141785], [0, 0.263266, 0.661919, 0.035469, 0.132445], [0, 0.370875, 0.66335, 0.031281, 0.130171], [0, 0.757914, 0.664707, 0.025547, 0.140709], [5, 0.916844, 0.693191, 0.054344, 0.164768], [0, 0.848742, 0.676186, 0.016547, 0.027628], [5, 0.029086, 0.711308, 0.029609, 0.081883], [0, 0.855398, 0.664731, 0.005766, 0.015086]]}, {"filename": "000000061526.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.5015, 0.460586, 0.342333, 0.265766]]}, {"filename": "000000062048.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.406906, 0.69482, 0.201187, 0.360359], [0, 0.666667, 0.627406, 0.476417, 0.532594]]}, {"filename": "000000062819.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.64972, 0.636387, 0.1264, 0.204507], [0, 0.33408, 0.304, 0.025, 0.038293], [3, 0.77615, 0.56192, 0.11966, 0.033707], [0, 0.60284, 0.318107, 0.02204, 0.028533], [0, 0.36309, 0.330973, 0.03046, 0.08648]]}, {"filename": "000000064495.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.502242, 0.280521, 0.283141, 0.391]]}, {"filename": "000000065288.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.654273, 0.77096, 0.637047, 0.427963], [4, 0.530008, 0.328056, 0.085672, 0.168806], [1, 0.059617, 0.228899, 0.048641, 0.082529], [0, 0.357289, 0.266089, 0.121766, 0.258735], [0, 0.486992, 0.275855, 0.106047, 0.267354], [0, 0.638633, 0.240012, 0.045734, 0.097166], [0, 0.328695, 0.225902, 0.042047, 0.126276], [0, 0.56107, 0.239813, 0.058797, 0.161733], [0, 0.668734, 0.194895, 0.027844, 0.072412], [0, 0.547773, 0.159778, 0.060047, 0.138899], [4, 0.371867, 0.320562, 0.095609, 0.172272], [0, 0.286398, 0.220843, 0.051547, 0.10356], [1, 0.244664, 0.196651, 0.020734, 0.049742], [1, 0.039602, 0.232365, 0.020734, 0.06623], [1, 0.028344, 0.245433, 0.013125, 0.048852], [0, 0.471039, 0.22671, 0.044953, 0.10377], [0, 0.432227, 0.284403, 0.066891, 0.236815], [0, 0.693937, 0.223396, 0.042594, 0.127681], [0, 0.126594, 0.185632, 0.037719, 0.120398], [4, 0.641781, 0.26219, 0.07425, 0.084286], [4, 0.670531, 0.36418, 0.056813, 0.074309], [0, 0.97132, 0.192213, 0.036391, 0.077916], [4, 0.271984, 0.253689, 0.019406, 0.032061], [4, 0.291133, 0.475328, 0.062391, 0.12466], [4, 0.297258, 0.263361, 0.027328, 0.053326], [4, 0.865625, 0.391546, 0.26875, 0.248197], [4, 0.761047, 0.24911, 0.092406, 0.070351], [4, 0.318656, 0.267342, 0.025844, 0.07459], [4, 0.340906, 0.259368, 0.028406, 0.059766]]}, {"filename": "000000065612.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.590802, 0.497938, 0.187479, 0.164813]]}, {"filename": "000000065984.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.273141, 0.7241, 0.256625, 0.22072], [4, 0.665781, 0.631607, 0.135438, 0.268753], [0, 0.542516, 0.814598, 0.258594, 0.34831]]}, {"filename": "000000066038.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.876086, 0.566437, 0.191266, 0.163417], [4, 0.752594, 0.565552, 0.153969, 0.094313], [4, 0.605609, 0.648406, 0.116719, 0.116104], [4, 0.558219, 0.631167, 0.059531, 0.162125], [0, 0.527555, 0.610437, 0.046141, 0.243542], [0, 0.267852, 0.538135, 0.143703, 0.049021], [0, 0.384352, 0.528271, 0.082609, 0.202917], [0, 0.671398, 0.560312, 0.187734, 0.062917], [0, 0.46757, 0.615844, 0.093797, 0.213687], [0, 0.308328, 0.542708, 0.113094, 0.059292], [4, 0.346211, 0.542646, 0.167516, 0.14075]]}, {"filename": "000000066239.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.604445, 0.685865, 0.334297, 0.473813], [1, 0.721352, 0.970792, 0.064047, 0.035958], [1, 0.659977, 0.977875, 0.082734, 0.043292], [1, 0.789594, 0.985938, 0.074531, 0.02775], [1, 0.883758, 0.985031, 0.022797, 0.029771], [1, 0.516, 0.986167, 0.116531, 0.027667], [0, 0.013469, 0.934969, 0.020969, 0.042604], [1, 0.307977, 0.986031, 0.126703, 0.024313]]}, {"filename": "000000066973.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.470413, 0.43707, 0.44344, 0.74382]]}, {"filename": "000000067084.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.840148, 0.494388, 0.179078, 0.687647], [0, 0.168563, 0.252835, 0.047281, 0.304165], [0, 0.699602, 0.508094, 0.243266, 0.724094]]}, {"filename": "000000067186.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.519613, 0.62218, 0.960773, 0.70136]]}, {"filename": "000000067616.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.725406, 0.817104, 0.227969, 0.217833], [1, 0.978961, 0.781479, 0.042078, 0.079375], [5, 0.362836, 0.886729, 0.045609, 0.137375], [0, 0.041234, 0.85675, 0.082344, 0.278125], [0, 0.207258, 0.809937, 0.058828, 0.107417], [0, 0.273453, 0.791562, 0.040906, 0.075458], [0, 0.230008, 0.790375, 0.025984, 0.056375], [1, 0.505258, 0.85951, 0.041953, 0.123396], [1, 0.927937, 0.796542, 0.139, 0.142333], [7, 0.610453, 0.803573, 0.033937, 0.196687], [4, 0.246367, 0.867625, 0.136203, 0.104917], [4, 0.431828, 0.853375, 0.036156, 0.091375], [4, 0.369586, 0.836844, 0.041391, 0.050187], [4, 0.499539, 0.843573, 0.018578, 0.061021], [4, 0.557617, 0.843917, 0.035953, 0.06775], [4, 0.472219, 0.804865, 0.032313, 0.023896], [4, 0.30207, 0.823125, 0.017641, 0.014333], [0, 0.153039, 0.79126, 0.048141, 0.072229], [1, 0.870555, 0.81776, 0.096891, 0.137063], [4, 0.160008, 0.86349, 0.060547, 0.110354], [4, 0.537555, 0.850625, 0.012359, 0.073625], [4, 0.581203, 0.835417, 0.028469, 0.088667]]}, {"filename": "000000068430.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.546594, 0.173352, 0.314396, 0.189984], [0, 0.216938, 0.21118, 0.282458, 0.422359], [0, 0.082396, 0.264047, 0.134833, 0.528094], [1, 0.475312, 0.18468, 0.086125, 0.081172], [3, 0.732792, 0.034289, 0.254542, 0.068578], [0, 0.504865, 0.140453, 0.176771, 0.280906], [1, 0.961469, 0.043469, 0.077063, 0.086938]]}, {"filename": "000000069213.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.622961, 0.491019, 0.382328, 0.937073], [0, 0.272484, 0.494614, 0.335031, 0.905152], [1, 0.830977, 0.439192, 0.189328, 0.403162], [0, 0.983344, 0.212295, 0.033313, 0.200796], [0, 0.729531, 0.258454, 0.108563, 0.195925], [0, 0.789008, 0.151792, 0.020703, 0.072014], [0, 0.890156, 0.205386, 0.047563, 0.03644], [0, 0.431094, 0.196171, 0.08525, 0.187986]]}, {"filename": "000000069356.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.978305, 0.359168, 0.042328, 0.098218], [0, 0.198586, 0.594178, 0.164172, 0.749426], [0, 0.737742, 0.584317, 0.172766, 0.648238], [0, 0.934469, 0.487931, 0.09575, 0.525248], [0, 0.989461, 0.419535, 0.021078, 0.223941], [0, 0.697609, 0.341901, 0.028437, 0.081069], [0, 0.885836, 0.352238, 0.032078, 0.089941]]}, {"filename": "000000069480.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.138117, 0.420625, 0.218828, 0.515703]]}, {"filename": "000000070229.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.967526, 0.542375, 0.050894, 0.10825], [3, 0.507474, 0.648461, 0.985052, 0.658422], [3, 0.661102, 0.624172, 0.125364, 0.105187]]}, {"filename": "000000070254.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.246906, 0.655964, 0.131531, 0.30849], [0, 0.136859, 0.690312, 0.08225, 0.492135], [0, 0.242109, 0.49276, 0.040188, 0.291094], [0, 0.354375, 0.505182, 0.052375, 0.103333], [0, 0.312484, 0.553268, 0.075031, 0.192995], [0, 0.403008, 0.466966, 0.026266, 0.142943], [3, 0.216945, 0.728594, 0.107859, 0.226875], [0, 0.329281, 0.520898, 0.054719, 0.111536], [3, 0.295281, 0.585951, 0.064719, 0.114609], [3, 0.339984, 0.533099, 0.023563, 0.025]]}, {"filename": "000000070336.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.531891, 0.531066, 0.035844, 0.042646], [1, 0.49693, 0.512576, 0.016297, 0.018923], [1, 0.48093, 0.497963, 0.011797, 0.010726], [1, 0.46575, 0.499801, 0.01025, 0.012623], [5, 0.774984, 0.659251, 0.027594, 0.060047], [1, 0.471148, 0.501464, 0.008547, 0.019227], [1, 0.487047, 0.491546, 0.010125, 0.014098]]}, {"filename": "000000070471.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.4955, 0.383583, 0.401125, 0.638208], [0, 0.5, 0.500563, 1.0, 0.998875]]}, {"filename": "000000071361.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.751059, 0.565438, 0.497883, 0.330156]]}, {"filename": "000000071965.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.900021, 0.628, 0.199958, 0.192406], [7, 0.667416, 0.291539, 0.057762, 0.104953], [7, 0.977646, 0.271031, 0.040188, 0.073281]]}, {"filename": "000000072795.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.525203, 0.615875, 0.079656, 0.180917], [0, 0.720047, 0.597333, 0.068719, 0.209208], [0, 0.659328, 0.634708, 0.0675, 0.122625], [0, 0.102414, 0.455979, 0.038172, 0.115667], [0, 0.396914, 0.926281, 0.217422, 0.119104], [0, 0.597312, 0.88674, 0.145437, 0.226521], [0, 0.904563, 0.75676, 0.190875, 0.466229], [0, 0.219484, 0.946771, 0.149531, 0.106458], [0, 0.708742, 0.842177, 0.086672, 0.136854], [0, 0.55607, 0.878917, 0.070922, 0.201833], [0, 0.488094, 0.93524, 0.089844, 0.129521], [0, 0.622453, 0.414115, 0.022719, 0.030313]]}, {"filename": "000000073109.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.453459, 0.675281, 0.527906, 0.617969], [1, 0.031176, 0.379039, 0.058871, 0.043984], [1, 0.180318, 0.368891, 0.026941, 0.01875]]}, {"filename": "000000074209.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.568297, 0.853052, 0.140594, 0.293896], [4, 0.715695, 0.835906, 0.134547, 0.328187], [4, 0.843539, 0.794385, 0.139891, 0.343813], [4, 0.421875, 0.8685, 0.15625, 0.239583]]}, {"filename": "000000074466.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.055344, 0.566781, 0.110156, 0.069104], [1, 0.418484, 0.550437, 0.0415, 0.025583], [1, 0.448977, 0.547896, 0.025641, 0.029], [0, 0.684633, 0.570833, 0.030297, 0.087167], [0, 0.653953, 0.570375, 0.01825, 0.076167], [1, 0.537305, 0.55124, 0.033078, 0.026396], [1, 0.364844, 0.547958, 0.054344, 0.045417], [1, 0.499023, 0.548469, 0.021422, 0.021896], [5, 0.206078, 0.770115, 0.065438, 0.129479], [1, 0.622641, 0.550188, 0.018438, 0.019458], [1, 0.201445, 0.559729, 0.057859, 0.041375], [1, 0.30457, 0.548115, 0.006016, 0.033021], [1, 0.400164, 0.549021, 0.018641, 0.026792], [1, 0.51618, 0.547552, 0.019078, 0.020187], [1, 0.813312, 0.55325, 0.146094, 0.077917], [1, 0.473859, 0.544958, 0.028844, 0.03175]]}, {"filename": "000000074573.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.337008, 0.433385, 0.031047, 0.077604], [7, 0.304102, 0.435688, 0.030547, 0.079542]]}, {"filename": "000000074733.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.132508, 0.060645, 0.262533, 0.121291], [0, 0.452296, 0.169624, 0.130147, 0.077745], [4, 0.675539, 0.170703, 0.211961, 0.123922], [4, 0.821168, 0.104632, 0.125801, 0.209265]]}, {"filename": "000000074900.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.434367, 0.192653, 0.260984, 0.385212], [0, 0.691594, 0.188066, 0.32975, 0.369387]]}, {"filename": "000000075095.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.820234, 0.388992, 0.084356, 0.120766], [0, 0.898759, 0.383672, 0.105714, 0.133531], [3, 0.36541, 0.613617, 0.649602, 0.503391]]}, {"filename": "000000076468.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.716219, 0.395271, 0.054063, 0.076583], [0, 0.396781, 0.498802, 0.137187, 0.357437], [0, 0.643586, 0.302927, 0.047297, 0.081063], [0, 0.21, 0.370385, 0.044594, 0.088687], [0, 0.511477, 0.498094, 0.102703, 0.355646], [0, 0.809422, 0.392031, 0.041875, 0.058271], [0, 0.601859, 0.260958, 0.051219, 0.084333], [0, 0.082289, 0.37151, 0.040516, 0.078396], [0, 0.735992, 0.140271, 0.044609, 0.075083], [4, 0.766516, 0.936375, 0.153531, 0.12725], [0, 0.598875, 0.06199, 0.031156, 0.063938], [0, 0.674492, 0.19626, 0.037359, 0.065562], [0, 0.257016, 0.37451, 0.046656, 0.092438], [0, 0.716734, 0.179437, 0.046719, 0.048208], [4, 0.924344, 0.935063, 0.15, 0.098708], [4, 0.594336, 0.409719, 0.042891, 0.019812], [4, 0.527102, 0.247406, 0.033828, 0.033521], [4, 0.492414, 0.250792, 0.036609, 0.033708], [4, 0.269219, 0.287719, 0.0395, 0.042021], [4, 0.153586, 0.25524, 0.038453, 0.041313], [4, 0.227758, 0.286479, 0.038891, 0.038458], [4, 0.116672, 0.249438, 0.030812, 0.031583], [4, 0.602305, 0.159448, 0.030797, 0.034687]]}, {"filename": "000000077460.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.650736, 0.910055, 0.027687, 0.058203], [0, 0.434614, 0.888758, 0.006192, 0.010703], [0, 0.560678, 0.888531, 0.021822, 0.0235], [0, 0.251495, 0.910281, 0.041542, 0.082188], [0, 0.280117, 0.877617, 0.015234, 0.014953], [0, 0.296355, 0.875437, 0.014673, 0.018406], [0, 0.348867, 0.877617, 0.012734, 0.021297], [0, 0.374369, 0.879789, 0.01528, 0.028484], [0, 0.407675, 0.882164, 0.013294, 0.021297], [0, 0.456437, 0.884383, 0.013575, 0.032422], [0, 0.527255, 0.890078, 0.021612, 0.021031], [0, 0.18368, 0.899297, 0.05278, 0.080844], [0, 0.739521, 0.895789, 0.012313, 0.027422]]}, {"filename": "000000077517.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.641867, 0.330202, 0.131547, 0.052251], [0, 0.533719, 0.542792, 0.218781, 0.416667], [0, 0.188344, 0.438832, 0.137531, 0.430191], [0, 0.673219, 0.555361, 0.210656, 0.335308], [0, 0.447813, 0.500743, 0.101469, 0.216476], [0, 0.396008, 0.43121, 0.042578, 0.116518], [0, 0.10043, 0.501369, 0.048953, 0.224437], [0, 0.03232, 0.489862, 0.061328, 0.243248], [0, 0.479617, 0.324586, 0.066391, 0.143864], [0, 0.661125, 0.502665, 0.067438, 0.205626], [4, 0.329859, 0.5281, 0.084781, 0.147282], [0, 0.607836, 0.471221, 0.044391, 0.15569], [0, 0.401422, 0.571231, 0.121812, 0.222463], [4, 0.644789, 0.509862, 0.113828, 0.1469], [4, 0.135063, 0.546327, 0.035375, 0.085605], [4, 0.037039, 0.544183, 0.074078, 0.117155]]}, {"filename": "000000077650.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.615273, 0.632177, 0.147922, 0.659181], [0, 0.74832, 0.651778, 0.136766, 0.662349], [0, 0.920172, 0.245636, 0.027313, 0.135582], [0, 0.842078, 0.222608, 0.026969, 0.085474], [0, 0.948305, 0.244515, 0.032578, 0.11528], [0, 0.874328, 0.241692, 0.031719, 0.121056], [0, 0.699586, 0.203879, 0.021609, 0.034914], [0, 0.836633, 0.10945, 0.020797, 0.040323], [5, 0.135305, 0.322155, 0.028453, 0.080948], [4, 0.308672, 0.256724, 0.022344, 0.075086], [4, 0.327937, 0.254795, 0.028969, 0.077177], [4, 0.352625, 0.254903, 0.032094, 0.075711], [4, 0.390578, 0.255744, 0.039469, 0.070668], [4, 0.188664, 0.257478, 0.051516, 0.076379], [4, 0.156305, 0.260937, 0.031609, 0.083168], [4, 0.131602, 0.231239, 0.019234, 0.02222], [4, 0.031242, 0.238319, 0.026859, 0.032931], [4, 0.430328, 0.252877, 0.047875, 0.074246], [4, 0.375703, 0.235183, 0.035188, 0.033642], [4, 0.35482, 0.236509, 0.020641, 0.02681], [0, 0.111664, 0.287306, 0.046672, 0.20931], [0, 0.745875, 0.196369, 0.036875, 0.067047]]}, {"filename": "000000077689.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.73892, 0.590947, 0.06152, 0.069307], [1, 0.15182, 0.300027, 0.0636, 0.116267], [0, 0.20562, 0.283147, 0.12136, 0.233707], [0, 0.39883, 0.42032, 0.14638, 0.419787], [1, 0.83888, 0.606027, 0.04172, 0.05856], [6, 0.82956, 0.200347, 0.03648, 0.02624], [0, 0.84358, 0.573373, 0.02572, 0.038107], [1, 0.20928, 0.36724, 0.08804, 0.046107], [1, 0.51818, 0.432467, 0.37404, 0.464933]]}, {"filename": "000000078032.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.242563, 0.91795, 0.298531, 0.164099], [3, 0.670805, 0.769099, 0.182391, 0.224621]]}, {"filename": "000000078336.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.2454, 0.773105, 0.10104, 0.057242], [1, 0.34501, 0.771126, 0.10338, 0.073368], [1, 0.09402, 0.759726, 0.08108, 0.052295], [6, 0.25409, 0.6956, 0.01938, 0.011663]]}, {"filename": "000000078410.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.64407, 0.745375, 0.559453, 0.40899], [1, 0.228328, 0.73715, 0.166563, 0.093909]]}, {"filename": "000000078716.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.927667, 0.61118, 0.144665, 0.744922], [7, 0.486883, 0.562336, 0.709456, 0.849359]]}, {"filename": "000000079034.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.420969, 0.531333, 0.182063, 0.631667], [1, 0.336242, 0.224021, 0.039828, 0.026542], [1, 0.273625, 0.22101, 0.043719, 0.023521], [1, 0.410945, 0.225917, 0.020734, 0.028708], [1, 0.242773, 0.220417, 0.011484, 0.021167], [1, 0.023125, 0.219479, 0.02825, 0.024167]]}, {"filename": "000000079244.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.445367, 0.657295, 0.111109, 0.348314], [0, 0.464188, 0.584262, 0.044781, 0.188571], [3, 0.456633, 0.717354, 0.179016, 0.074567]]}, {"filename": "000000079268.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.50254, 0.286516, 0.271103, 0.173031], [1, 0.76069, 0.574836, 0.049195, 0.029359], [1, 0.791115, 0.577977, 0.043379, 0.025203], [1, 0.435793, 0.583055, 0.039402, 0.019203], [1, 0.854011, 0.582094, 0.098736, 0.047781], [1, 0.942034, 0.582687, 0.09892, 0.052844], [1, 0.467391, 0.581141, 0.03092, 0.032281], [1, 0.384793, 0.566406, 0.054552, 0.049437], [1, 0.426138, 0.570484, 0.031954, 0.031344], [1, 0.982333, 0.569211, 0.035333, 0.030422], [6, 0.144575, 0.53893, 0.029379, 0.020734]]}, {"filename": "000000080013.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.160703, 0.670802, 0.091844, 0.190613], [0, 0.448789, 0.592748, 0.105516, 0.412429], [0, 0.1905, 0.62717, 0.06525, 0.249387], [0, 0.122992, 0.592983, 0.112672, 0.376769], [0, 0.390664, 0.569068, 0.051609, 0.306722], [4, 0.058609, 0.640271, 0.0455, 0.111486], [0, 0.315172, 0.594646, 0.142906, 0.487642]]}, {"filename": "000000080041.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.517654, 0.598695, 0.527202, 0.771141], [0, 0.305751, 0.532398, 0.59858, 0.911547]]}, {"filename": "000000080340.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.890586, 0.497752, 0.215891, 0.474145], [0, 0.329844, 0.464473, 0.203906, 0.593255], [0, 0.462359, 0.246616, 0.132531, 0.46274], [0, 0.595977, 0.225843, 0.160422, 0.442693], [0, 0.725039, 0.264637, 0.133734, 0.522529], [0, 0.904203, 0.08096, 0.165969, 0.16192], [0, 0.035867, 0.850656, 0.070734, 0.298689], [0, 0.816133, 0.177635, 0.100047, 0.355269]]}, {"filename": "000000080666.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.64382, 0.602688, 0.676391, 0.743812]]}, {"filename": "000000081102.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.469281, 0.260679, 0.212906, 0.31911], [7, 0.4615, 0.619087, 0.124156, 0.618829], [7, 0.731664, 0.596628, 0.230891, 0.662904], [1, 0.054945, 0.329415, 0.084766, 0.103513], [7, 0.345242, 0.432705, 0.057016, 0.174403], [7, 0.302148, 0.425059, 0.039141, 0.143513], [0, 0.127297, 0.433454, 0.040312, 0.155902], [0, 0.930023, 0.552084, 0.139953, 0.7526], [0, 0.249211, 0.475234, 0.083328, 0.335152]]}, {"filename": "000000081594.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.491935, 0.687297, 0.279812, 0.59875]]}, {"filename": "000000082709.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.28861, 0.27111, 0.08638, 0.218174], [0, 0.17747, 0.249087, 0.15554, 0.279972], [0, 0.58453, 0.084157, 0.14502, 0.164663], [0, 0.57652, 0.062935, 0.1834, 0.123062], [0, 0.30484, 0.105169, 0.10748, 0.169438], [0, 0.26165, 0.227893, 0.09318, 0.207135], [0, 0.9136, 0.085393, 0.1248, 0.170787], [0, 0.07461, 0.045197, 0.1143, 0.090393], [0, 0.15847, 0.04257, 0.09674, 0.08514], [0, 0.9824, 0.105618, 0.0352, 0.211236], [0, 0.06745, 0.103272, 0.13422, 0.206545], [0, 0.55355, 0.045534, 0.09346, 0.091067], [0, 0.69187, 0.059775, 0.08026, 0.119551]]}, {"filename": "000000082894.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.700883, 0.137927, 0.383172, 0.214687], [1, 0.966242, 0.14624, 0.067516, 0.118604], [1, 0.94325, 0.375427, 0.1135, 0.202771], [1, 0.627422, 0.305917, 0.073562, 0.1905], [1, 0.044812, 0.261042, 0.083812, 0.183958], [1, 0.70157, 0.139937, 0.392547, 0.214417], [3, 0.331281, 0.312958, 0.580969, 0.189125]]}, {"filename": "000000083382.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.288079, 0.707477, 0.381551, 0.439016], [0, 0.603532, 0.751039, 0.129928, 0.215797]]}, {"filename": "000000083557.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.337922, 0.576406, 0.298313, 0.226979]]}, {"filename": "000000083990.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.942523, 0.515831, 0.029141, 0.026417], [1, 0.759516, 0.518454, 0.020781, 0.022576], [1, 0.887133, 0.515363, 0.022734, 0.02], [1, 0.365445, 0.540539, 0.039953, 0.028337], [1, 0.915797, 0.523021, 0.023625, 0.017705], [1, 0.85125, 0.520351, 0.011719, 0.014145], [1, 0.871977, 0.520585, 0.018141, 0.014473], [1, 0.961922, 0.514836, 0.01525, 0.020632], [1, 0.976086, 0.515773, 0.013266, 0.019087], [1, 0.781258, 0.512506, 0.018953, 0.020937]]}, {"filename": "000000084168.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.396479, 0.186711, 0.129125, 0.148672], [5, 0.600896, 0.597922, 0.082583, 0.175469]]}, {"filename": "000000084170.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.283234, 0.623188, 0.153063, 0.158375], [1, 0.175281, 0.590698, 0.092563, 0.084479], [1, 0.506461, 0.50449, 0.379203, 0.510104], [1, 0.350984, 0.5635, 0.017062, 0.03625], [0, 0.540758, 0.513948, 0.029391, 0.079146], [1, 0.036383, 0.555333, 0.070484, 0.091792], [0, 0.429281, 0.506479, 0.061938, 0.072208]]}, {"filename": "000000084171.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.279133, 0.168815, 0.108797, 0.212887], [4, 0.411734, 0.209765, 0.069906, 0.215869], [4, 0.202875, 0.02966, 0.051438, 0.058474], [1, 0.472672, 0.577523, 0.634219, 0.822465], [0, 0.77332, 0.310117, 0.191453, 0.620235], [0, 0.663008, 0.132805, 0.059672, 0.253451], [0, 0.693023, 0.14757, 0.055297, 0.295141], [0, 0.572891, 0.096631, 0.122656, 0.188756], [1, 0.081977, 0.682512, 0.163953, 0.605915], [0, 0.324266, 0.129695, 0.192781, 0.250892]]}, {"filename": "000000084238.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.080937, 0.504498, 0.161875, 0.191005], [1, 0.198406, 0.48989, 0.035813, 0.112377], [1, 0.721305, 0.465172, 0.098859, 0.103382], [1, 0.359297, 0.413836, 0.030125, 0.056397], [1, 0.192508, 0.472953, 0.048359, 0.106495], [1, 0.538086, 0.394828, 0.273766, 0.242647], [0, 0.226602, 0.697904, 0.100016, 0.480221], [0, 0.45132, 0.565466, 0.037484, 0.19299], [0, 0.588078, 0.493419, 0.045031, 0.170368], [0, 0.873352, 0.49685, 0.062891, 0.125564], [0, 0.330883, 0.662426, 0.134641, 0.552598], [1, 0.899211, 0.801225, 0.201578, 0.397549], [7, 0.633664, 0.640748, 0.046203, 0.152672], [7, 0.52225, 0.602806, 0.046312, 0.139044], [7, 0.436719, 0.580502, 0.028219, 0.118701], [1, 0.299102, 0.308064, 0.029141, 0.03402], [7, 0.133, 0.493946, 0.027906, 0.062157], [7, 0.284813, 0.449583, 0.029219, 0.042206], [0, 0.164742, 0.500882, 0.041703, 0.181127], [0, 0.035148, 0.377586, 0.022234, 0.06973], [0, 0.067477, 0.398824, 0.034422, 0.043775], [0, 0.010297, 0.388554, 0.019688, 0.069657], [1, 0.828828, 0.780086, 0.342344, 0.439828]]}, {"filename": "000000084362.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.494805, 0.228208, 0.251141, 0.057167], [0, 0.663203, 0.422469, 0.15675, 0.251688], [4, 0.5, 0.76999, 1.0, 0.460021]]}, {"filename": "000000084752.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.793117, 0.8313, 0.048484, 0.04644], [1, 0.832734, 0.829848, 0.046844, 0.037541], [1, 0.463477, 0.822155, 0.024047, 0.023794], [1, 0.479594, 0.819192, 0.026969, 0.027424], [1, 0.889164, 0.823056, 0.061141, 0.05548], [1, 0.765055, 0.810585, 0.045453, 0.039157], [1, 0.846109, 0.801089, 0.040094, 0.031639], [1, 0.605172, 0.828829, 0.026844, 0.038829], [1, 0.595234, 0.807119, 0.010656, 0.020468], [1, 0.891656, 0.792213, 0.036719, 0.021616], [1, 0.503156, 0.815667, 0.035313, 0.037986], [1, 0.9405, 0.825785, 0.050969, 0.042482]]}, {"filename": "000000085759.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.447653, 0.162961, 0.855305, 0.325922], [0, 0.49966, 0.458422, 0.553685, 0.723594], [0, 0.911667, 0.036531, 0.057277, 0.069656]]}, {"filename": "000000085817.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.515164, 0.466656, 0.568692, 0.516781]]}, {"filename": "000000086334.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.180508, 0.519007, 0.044547, 0.151425], [3, 0.026188, 0.562979, 0.052375, 0.085397], [0, 0.195039, 0.512617, 0.022984, 0.15215]]}, {"filename": "000000086901.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.911305, 0.807594, 0.174984, 0.187521], [1, 0.753258, 0.802396, 0.164047, 0.122417], [1, 0.519758, 0.750021, 0.047203, 0.057167], [1, 0.347531, 0.726688, 0.045344, 0.041208], [1, 0.408312, 0.713708, 0.026375, 0.025917], [1, 0.390914, 0.711042, 0.019234, 0.02075], [1, 0.563266, 0.75674, 0.05, 0.075146], [1, 0.661484, 0.747938, 0.156281, 0.163583], [1, 0.476562, 0.74001, 0.047219, 0.046896], [1, 0.352734, 0.697917, 0.015375, 0.014875], [1, 0.318484, 0.722125, 0.024875, 0.031], [1, 0.381875, 0.70624, 0.022344, 0.027438], [7, 0.18243, 0.780031, 0.073859, 0.284479], [1, 0.340305, 0.698635, 0.020141, 0.013437], [1, 0.274531, 0.688625, 0.0195, 0.016958]]}, {"filename": "000000087144.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.611742, 0.097844, 0.130391, 0.062146], [0, 0.702805, 0.48876, 0.235953, 0.653937], [0, 0.259547, 0.476031, 0.246062, 0.705604], [3, 0.115711, 0.540542, 0.228047, 0.482], [3, 0.786312, 0.548427, 0.315875, 0.479729], [3, 0.950516, 0.455885, 0.098969, 0.319938], [0, 0.438336, 0.433948, 0.189141, 0.240813], [1, 0.77268, 0.097135, 0.124578, 0.062104], [3, 0.461688, 0.455844, 0.25325, 0.298687]]}, {"filename": "000000087244.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.588933, 0.65247, 0.382667, 0.64574], [0, 0.563707, 0.1093, 0.15248, 0.21464], [3, 0.653893, 0.15043, 0.09328, 0.04714]]}, {"filename": "000000087875.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.346703, 0.67037, 0.240719, 0.651622]]}, {"filename": "000000088582.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.35901, 0.560816, 0.31118, 0.752255], [0, 0.17124, 0.372463, 0.01232, 0.039763], [0, 0.15567, 0.374659, 0.00962, 0.033116], [0, 0.89922, 0.363991, 0.0078, 0.032849], [3, 0.20774, 0.737478, 0.41132, 0.525045], [0, 0.9798, 0.366424, 0.00928, 0.043353]]}, {"filename": "000000088613.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.386521, 0.496781, 0.299625, 0.498875], [7, 0.680146, 0.481336, 0.257667, 0.438203]]}, {"filename": "000000088848.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.293687, 0.221133, 0.177562, 0.342641], [5, 0.708141, 0.337461, 0.134, 0.283016], [0, 0.691281, 0.85432, 0.099, 0.226297], [0, 0.771945, 0.84793, 0.080797, 0.225359], [0, 0.294383, 0.771914, 0.139328, 0.406734], [0, 0.921109, 0.846492, 0.074, 0.224953], [0, 0.867328, 0.847805, 0.115594, 0.226734]]}, {"filename": "000000089045.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.147812, 0.656988, 0.089969, 0.427953], [4, 0.940039, 0.324671, 0.031047, 0.072729]]}, {"filename": "000000089126.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.447984, 0.556174, 0.674594, 0.653944], [3, 0.909313, 0.658732, 0.144562, 0.082864]]}, {"filename": "000000089225.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.349477, 0.695258, 0.106516, 0.159388], [1, 0.613914, 0.673881, 0.234797, 0.101071], [1, 0.199281, 0.676033, 0.231063, 0.121358], [1, 0.484773, 0.715488, 0.079109, 0.187151], [1, 0.131305, 0.816855, 0.236891, 0.307859], [1, 0.50793, 0.631969, 0.064859, 0.037438], [7, 0.582562, 0.691979, 0.031281, 0.075468], [7, 0.632484, 0.682859, 0.031125, 0.080019], [7, 0.291586, 0.672189, 0.023766, 0.062964], [7, 0.233094, 0.674044, 0.01825, 0.059522], [1, 0.879898, 0.673805, 0.216359, 0.124551], [7, 0.056016, 0.647017, 0.021344, 0.028337]]}, {"filename": "000000089556.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.556992, 0.399562, 0.194109, 0.147042], [1, 0.494008, 0.350948, 0.152266, 0.103104], [1, 0.329047, 0.358677, 0.088938, 0.084729], [1, 0.19868, 0.333271, 0.063547, 0.020042], [1, 0.657391, 0.285635, 0.060031, 0.032771], [5, 0.427781, 0.68326, 0.119312, 0.258771], [1, 0.936281, 0.266146, 0.063, 0.020917], [1, 0.585781, 0.29375, 0.037781, 0.017542], [1, 0.95143, 0.343052, 0.067703, 0.090688], [1, 0.232945, 0.575292, 0.462359, 0.458833], [1, 0.416609, 0.346688, 0.048969, 0.095125], [1, 0.741242, 0.475062, 0.478891, 0.41775], [1, 0.062484, 0.347771, 0.120906, 0.115], [1, 0.246297, 0.301031, 0.067187, 0.027063], [1, 0.488141, 0.353573, 0.177031, 0.110354]]}, {"filename": "000000091406.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.243969, 0.46013, 0.07875, 0.150967], [4, 0.313719, 0.403101, 0.089906, 0.110401], [4, 0.183148, 0.408738, 0.057234, 0.066958], [4, 0.361484, 0.388019, 0.028594, 0.019764], [0, 0.465539, 0.411038, 0.164141, 0.594575], [0, 0.584516, 0.467712, 0.182719, 0.574009], [0, 0.703445, 0.424163, 0.150359, 0.707854], [0, 0.825008, 0.541321, 0.179578, 0.793962], [4, 0.30882, 0.447535, 0.033047, 0.101863]]}, {"filename": "000000091495.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.199258, 0.599228, 0.352484, 0.692729], [0, 0.712258, 0.705492, 0.399109, 0.439575], [0, 0.384375, 0.471834, 0.293406, 0.73217]]}, {"filename": "000000091619.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.460527, 0.479031, 0.274356, 0.146906]]}, {"filename": "000000091779.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.106508, 0.102729, 0.148391, 0.202375]]}, {"filename": "000000093382.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.339547, 0.4525, 0.090906, 0.412167], [0, 0.245391, 0.579833, 0.130875, 0.299778], [0, 0.819656, 0.337986, 0.030531, 0.101694], [0, 0.401391, 0.362569, 0.017937, 0.051306], [1, 0.565039, 0.350458, 0.064672, 0.046417], [1, 0.506969, 0.357181, 0.038281, 0.035583], [1, 0.17175, 0.361847, 0.039594, 0.04625], [1, 0.420477, 0.355514, 0.035797, 0.044361], [1, 0.265641, 0.359681, 0.041875, 0.043472], [1, 0.123773, 0.363056, 0.038547, 0.042833], [1, 0.736828, 0.348458, 0.070063, 0.046472], [0, 0.691883, 0.357917, 0.014609, 0.0295], [1, 0.46332, 0.356375, 0.036859, 0.042806], [1, 0.030258, 0.365056, 0.036578, 0.045333], [1, 0.075148, 0.361208, 0.040547, 0.049194], [1, 0.219586, 0.36, 0.033578, 0.041667], [1, 0.377062, 0.362653, 0.015313, 0.034472], [1, 0.299711, 0.359736, 0.027203, 0.045139], [1, 0.846562, 0.346347, 0.043969, 0.036083]]}, {"filename": "000000093451.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.73343, 0.468987, 0.53314, 0.934827]]}, {"filename": "000000093579.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.856312, 0.51478, 0.044344, 0.097023], [4, 0.9455, 0.516981, 0.042719, 0.085996], [4, 0.934781, 0.512065, 0.04075, 0.086688], [4, 0.158984, 0.513407, 0.02275, 0.05805], [0, 0.543633, 0.628375, 0.143797, 0.61109], [0, 0.67225, 0.647044, 0.154719, 0.582138], [3, 0.083437, 0.552579, 0.166875, 0.15174], [0, 0.403695, 0.653574, 0.149016, 0.618679], [0, 0.264953, 0.652243, 0.147, 0.632075], [4, 0.44693, 0.321143, 0.069359, 0.065891]]}, {"filename": "000000094751.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.9174, 0.58408, 0.1652, 0.07216], [0, 0.191427, 0.52187, 0.02776, 0.0333], [0, 0.296027, 0.52192, 0.018773, 0.03076], [0, 0.544267, 0.52104, 0.01744, 0.02728], [0, 0.670133, 0.51189, 0.020213, 0.01686], [0, 0.20888, 0.51954, 0.016587, 0.029], [0, 0.160493, 0.52364, 0.019387, 0.02852], [3, 0.730147, 0.55874, 0.137787, 0.0114], [0, 0.269813, 0.51933, 0.02416, 0.02638], [1, 0.399627, 0.55337, 0.092587, 0.04402], [0, 0.83736, 0.55592, 0.02368, 0.02396], [0, 0.644733, 0.50925, 0.025093, 0.01798], [0, 0.73616, 0.51604, 0.01648, 0.02512], [0, 0.480213, 0.5225, 0.0112, 0.02772]]}, {"filename": "000000094837.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.474551, 0.624559, 0.181487, 0.44866], [0, 0.696732, 0.606127, 0.197778, 0.632582], [0, 0.809755, 0.356601, 0.069118, 0.108758], [0, 0.430172, 0.340891, 0.058905, 0.120899], [0, 0.141111, 0.369044, 0.053922, 0.040931], [0, 0.730931, 0.339673, 0.032451, 0.143333], [0, 0.844395, 0.314412, 0.055033, 0.075131], [3, 0.503382, 0.727475, 0.993235, 0.477467], [0, 0.291642, 0.37018, 0.012271, 0.024641], [0, 0.324028, 0.371487, 0.019526, 0.024118], [0, 0.358554, 0.375016, 0.017696, 0.017712], [0, 0.743603, 0.326193, 0.016454, 0.074444], [0, 0.273742, 0.551658, 0.14915, 0.326944], [0, 0.373399, 0.367908, 0.025327, 0.030523]]}, {"filename": "000000094846.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.791141, 0.227281, 0.417187, 0.422811], [0, 0.32382, 0.527719, 0.589734, 0.921336], [0, 0.082289, 0.32697, 0.164578, 0.155046], [4, 0.373953, 0.868767, 0.624844, 0.257765], [4, 0.084375, 0.851797, 0.166344, 0.296406], [4, 0.422063, 0.076832, 0.307281, 0.151037]]}, {"filename": "000000094926.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.71689, 0.619099, 0.47294, 0.734835]]}, {"filename": "000000095155.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.250555, 0.23137, 0.107922, 0.187143], [0, 0.756703, 0.224286, 0.11925, 0.241686], [0, 0.795719, 0.70048, 0.216625, 0.399742], [0, 0.26182, 0.732143, 0.168547, 0.306066]]}, {"filename": "000000095283.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.556172, 0.439812, 0.329094, 0.463099], [0, 0.825672, 0.434824, 0.068813, 0.258427], [0, 0.856711, 0.416291, 0.042547, 0.269765], [0, 0.971672, 0.540293, 0.046937, 0.484484], [1, 0.231102, 0.435962, 0.381422, 0.364038], [1, 0.29693, 0.63757, 0.593859, 0.724859], [1, 0.834313, 0.400329, 0.275094, 0.218967], [1, 0.142789, 0.245411, 0.220766, 0.054061], [1, 0.921, 0.441643, 0.106406, 0.210376], [0, 0.380445, 0.300493, 0.044266, 0.112254]]}, {"filename": "000000095520.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.26189, 0.690935, 0.51278, 0.595265], [1, 0.29792, 0.528832, 0.34192, 0.213489], [1, 0.13648, 0.399595, 0.03892, 0.039315], [0, 0.23459, 0.487492, 0.0505, 0.086137], [0, 0.78421, 0.413427, 0.03058, 0.051526], [0, 0.26966, 0.477679, 0.03576, 0.057414], [1, 0.19243, 0.369782, 0.03762, 0.021059], [1, 0.24975, 0.692819, 0.4995, 0.614361], [5, 0.99295, 0.470826, 0.0141, 0.04919], [3, 0.94076, 0.398349, 0.02924, 0.036137], [7, 0.62522, 0.65405, 0.27104, 0.689283]]}, {"filename": "000000095843.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.544453, 0.544964, 0.680188, 0.789121]]}, {"filename": "000000095899.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.236273, 0.406833, 0.415359, 0.486542]]}, {"filename": "000000096339.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.702094, 0.499792, 0.035813, 0.11875], [7, 0.515453, 0.490958, 0.031937, 0.108417], [7, 0.348836, 0.48226, 0.034359, 0.109021], [7, 0.21282, 0.474354, 0.038234, 0.102625], [1, 0.129914, 0.341417, 0.037141, 0.033292], [1, 0.025898, 0.349958, 0.048391, 0.088083]]}, {"filename": "000000096466.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.791046, 0.542949, 0.09781, 0.16433]]}, {"filename": "000000096711.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.107375, 0.194215, 0.209031, 0.255363], [1, 0.832875, 0.315738, 0.33425, 0.379297], [1, 0.630047, 0.08623, 0.035406, 0.095082], [0, 0.857273, 0.206077, 0.166797, 0.40541], [0, 0.939312, 0.130902, 0.112438, 0.251686], [0, 0.419055, 0.079778, 0.151422, 0.159555], [0, 0.063062, 0.141194, 0.095938, 0.282389], [3, 0.722664, 0.676405, 0.551734, 0.624707], [3, 0.546039, 0.123911, 0.095797, 0.055831], [0, 0.667, 0.071511, 0.063312, 0.140867], [0, 0.480625, 0.040937, 0.032188, 0.048806], [1, 0.304617, 0.227939, 0.205172, 0.342717], [1, 0.555125, 0.280468, 0.097219, 0.177377], [0, 0.456289, 0.552646, 0.270109, 0.894707], [0, 0.307437, 0.172799, 0.128687, 0.344005]]}, {"filename": "000000098287.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.456434, 0.488414, 0.124482, 0.181609], [0, 0.547446, 0.415305, 0.041831, 0.041703], [0, 0.59647, 0.422391, 0.054482, 0.057906], [0, 0.936096, 0.433406, 0.041976, 0.073563], [0, 0.300458, 0.430836, 0.049108, 0.122078], [0, 0.667795, 0.415813, 0.038723, 0.040437], [0, 0.952145, 0.413727, 0.02053, 0.036828], [0, 0.520084, 0.418539, 0.053277, 0.041016], [0, 0.69094, 0.410969, 0.016434, 0.047812], [0, 0.128687, 0.513859, 0.142627, 0.154375], [0, 0.212253, 0.444836, 0.043157, 0.102453], [0, 0.030241, 0.470789, 0.060482, 0.204172], [0, 0.876096, 0.424766, 0.011807, 0.032156], [3, 0.649675, 0.448578, 0.095108, 0.006594]]}, {"filename": "000000098830.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.073791, 0.754594, 0.086033, 0.166125], [1, 0.041631, 0.751977, 0.083263, 0.157266], [0, 0.612981, 0.726625, 0.155915, 0.102313], [0, 0.854354, 0.703758, 0.057207, 0.096797], [0, 0.772101, 0.694516, 0.106737, 0.127313], [0, 0.673052, 0.65343, 0.035023, 0.038766], [0, 0.7452, 0.616883, 0.043545, 0.039578], [0, 0.70473, 0.636523, 0.044624, 0.031359], [0, 0.570035, 0.690938, 0.076455, 0.088375], [1, 0.476678, 0.768586, 0.074718, 0.101359], [1, 0.396127, 0.643945, 0.068122, 0.033766], [1, 0.033333, 0.655344, 0.047887, 0.061], [0, 0.702383, 0.705352, 0.059507, 0.131141], [0, 0.656467, 0.67743, 0.061526, 0.058234], [1, 0.525047, 0.738031, 0.026197, 0.044094], [1, 0.056737, 0.836125, 0.1077, 0.198688], [1, 0.424613, 0.622789, 0.047911, 0.016359]]}, {"filename": "000000099030.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.691773, 0.862049, 0.165266, 0.272108], [4, 0.839258, 0.85911, 0.160234, 0.28178], [4, 0.126273, 0.739567, 0.136953, 0.255433], [4, 0.647641, 0.655527, 0.083906, 0.05897]]}, {"filename": "000000099180.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.92066, 0.313368, 0.15868, 0.232246], [0, 0.34167, 0.277725, 0.05114, 0.206228], [4, 0.04555, 0.502096, 0.0911, 0.052575], [0, 0.46172, 0.236482, 0.0128, 0.06997], [0, 0.45208, 0.236198, 0.01068, 0.047186], [4, 0.12948, 0.387575, 0.0988, 0.075629], [4, 0.06237, 0.419701, 0.09018, 0.075329], [0, 0.47516, 0.240943, 0.01628, 0.08518]]}, {"filename": "000000100283.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.528031, 0.355604, 0.410313, 0.574], [1, 0.353594, 0.875042, 0.151906, 0.066375], [1, 0.163164, 0.85151, 0.054078, 0.040313], [1, 0.102141, 0.858833, 0.069469, 0.038], [1, 0.194078, 0.863823, 0.065781, 0.040438], [1, 0.552133, 0.866083, 0.189578, 0.052167]]}, {"filename": "000000101860.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.57476, 0.060437, 0.043896, 0.018812], [4, 0.678479, 0.02218, 0.049083, 0.027859], [4, 0.519188, 0.041953, 0.0285, 0.013781], [4, 0.553187, 0.044453, 0.015167, 0.010344], [4, 0.586708, 0.045477, 0.031875, 0.015203], [4, 0.718344, 0.018961, 0.032771, 0.018359], [0, 0.80449, 0.568539, 0.182771, 0.341578], [0, 0.608885, 0.628156, 0.185229, 0.222813], [0, 0.556802, 0.092813, 0.046271, 0.047656], [0, 0.308208, 0.554953, 0.231625, 0.293438], [0, 0.467542, 0.057688, 0.040917, 0.040188], [0, 0.440552, 0.075258, 0.039812, 0.041141], [0, 0.424667, 0.063125, 0.032042, 0.043531], [0, 0.851802, 0.104281, 0.066521, 0.052812], [0, 0.378031, 0.053273, 0.071562, 0.050172], [0, 0.775896, 0.25393, 0.135958, 0.067141], [0, 0.875906, 0.256734, 0.039479, 0.071719], [0, 0.934667, 0.113461, 0.053, 0.038922], [4, 0.747938, 0.019617, 0.025542, 0.013141], [0, 0.839229, 0.071687, 0.0895, 0.059469]]}, {"filename": "000000102411.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.472273, 0.631464, 0.356828, 0.413466], [0, 0.407258, 0.402529, 0.064609, 0.119625], [0, 0.434344, 0.562857, 0.185094, 0.53466], [0, 0.069484, 0.460632, 0.048844, 0.255738], [0, 0.522687, 0.51993, 0.181875, 0.460328], [0, 0.3275, 0.453923, 0.022688, 0.092482], [0, 0.640383, 0.438162, 0.064141, 0.094871], [0, 0.927727, 0.46411, 0.076766, 0.156604], [0, 0.174398, 0.488806, 0.024141, 0.048337], [0, 0.291016, 0.460749, 0.023687, 0.081967], [0, 0.140219, 0.476663, 0.093219, 0.227845], [1, 0.031195, 0.576628, 0.058922, 0.223372], [0, 0.83075, 0.391218, 0.072187, 0.122763]]}, {"filename": "000000102766.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.403633, 0.053979, 0.050922, 0.104792], [0, 0.782625, 0.438573, 0.43475, 0.812729], [0, 0.061656, 0.286031, 0.119938, 0.488729], [0, 0.112031, 0.279635, 0.083906, 0.159521], [0, 0.051156, 0.246365, 0.057094, 0.112271], [1, 0.550758, 0.269, 0.042641, 0.0245], [0, 0.39357, 0.368594, 0.362516, 0.520729]]}, {"filename": "000000103380.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.663594, 0.705383, 0.516116, 0.511828], [0, 0.736362, 0.340719, 0.267098, 0.199625], [0, 0.196663, 0.147914, 0.182031, 0.184984]]}, {"filename": "000000103678.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.390234, 0.398773, 0.009656, 0.010584], [1, 0.385961, 0.449977, 0.012047, 0.020607], [1, 0.388828, 0.412535, 0.011656, 0.014603], [1, 0.392289, 0.4391, 0.012953, 0.018949], [1, 0.421883, 0.395432, 0.006547, 0.014089], [1, 0.396672, 0.424836, 0.013937, 0.010421], [1, 0.265836, 0.415012, 0.013172, 0.021425], [1, 0.355117, 0.439357, 0.017234, 0.013715], [1, 0.381461, 0.405911, 0.007484, 0.014907], [1, 0.409641, 0.396133, 0.010344, 0.012313], [1, 0.39557, 0.406893, 0.007891, 0.013692]]}, {"filename": "000000104592.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.564802, 0.766734, 0.386521, 0.424719], [0, 0.749062, 0.752961, 0.293625, 0.462922], [0, 0.263677, 0.70225, 0.521354, 0.5955], [0, 0.596063, 0.619359, 0.110792, 0.216437], [0, 0.947198, 0.67975, 0.105604, 0.249656], [0, 0.523115, 0.542227, 0.063646, 0.067641], [0, 0.859781, 0.553578, 0.107521, 0.094781], [3, 0.953323, 0.935742, 0.093354, 0.105078], [0, 0.428187, 0.534727, 0.067583, 0.045766], [0, 0.815083, 0.310063, 0.278875, 0.360313], [0, 0.731167, 0.505695, 0.05775, 0.025141], [0, 0.576531, 0.558594, 0.040687, 0.041344], [0, 0.897875, 0.533922, 0.08525, 0.096375], [0, 0.925781, 0.531391, 0.044271, 0.043031]]}, {"filename": "000000104811.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.749156, 0.616854, 0.190437, 0.528083], [1, 0.096516, 0.405594, 0.034156, 0.024396], [1, 0.557312, 0.401125, 0.090719, 0.032917], [1, 0.133641, 0.402187, 0.048656, 0.03075], [1, 0.01425, 0.400865, 0.026, 0.031729], [1, 0.146828, 0.373698, 0.0315, 0.025354], [1, 0.04625, 0.397604, 0.056406, 0.042833]]}, {"filename": "000000105332.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.028787, 0.45849, 0.057573, 0.04446], [1, 0.244333, 0.44893, 0.171173, 0.0473]]}, {"filename": "000000105912.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.11982, 0.39468, 0.08696, 0.069733], [1, 0.52656, 0.379067, 0.32436, 0.16768], [0, 0.69348, 0.45556, 0.03056, 0.110693], [0, 0.49617, 0.438133, 0.0191, 0.07328], [0, 0.48074, 0.43152, 0.02172, 0.08688], [0, 0.44362, 0.420933, 0.02632, 0.076213], [0, 0.79075, 0.426347, 0.00902, 0.036533], [0, 0.07514, 0.38304, 0.02432, 0.069867], [0, 0.12681, 0.386933, 0.03366, 0.080533], [0, 0.02008, 0.37048, 0.01912, 0.095627], [0, 0.73349, 0.45816, 0.02106, 0.08352], [1, 0.03704, 0.372107, 0.07408, 0.084427], [0, 0.76037, 0.460733, 0.02702, 0.084613], [0, 0.87586, 0.419547, 0.06528, 0.278133], [0, 0.62797, 0.425893, 0.07478, 0.212373], [5, 0.31877, 0.558227, 0.1989, 0.59832]]}, {"filename": "000000106216.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.15201, 0.639433, 0.28386, 0.389254], [0, 0.79186, 0.307194, 0.0878, 0.180537], [0, 0.67452, 0.305627, 0.12044, 0.184269], [7, 0.34702, 0.515552, 0.02992, 0.064597], [7, 0.97996, 0.627687, 0.04008, 0.30003], [1, 0.14488, 0.631582, 0.289, 0.402448]]}, {"filename": "000000106235.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.276617, 0.783646, 0.545453, 0.4], [4, 0.918828, 0.475323, 0.162344, 0.363646]]}, {"filename": "000000106964.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.839648, 0.676021, 0.320703, 0.389583]]}, {"filename": "000000108266.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.506203, 0.678896, 0.971781, 0.562083], [1, 0.819102, 0.539323, 0.357297, 0.391021], [0, 0.884125, 0.311656, 0.043688, 0.055896], [0, 0.966727, 0.299521, 0.066547, 0.071542], [0, 0.894039, 0.300365, 0.025703, 0.052396], [0, 0.989133, 0.298094, 0.021734, 0.071562], [0, 0.902234, 0.278521, 0.0125, 0.041458], [0, 0.750961, 0.340417, 0.041828, 0.094667], [0, 0.830273, 0.310958, 0.034547, 0.076333]]}, {"filename": "000000109441.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.725359, 0.945156, 0.018125, 0.037063], [1, 0.211352, 0.889198, 0.031891, 0.020438], [1, 0.258359, 0.88201, 0.015094, 0.020563], [1, 0.178023, 0.880781, 0.023453, 0.018854], [1, 0.14893, 0.882781, 0.028047, 0.014354], [1, 0.104914, 0.882437, 0.021109, 0.015042]]}, {"filename": "000000109992.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.091141, 0.615317, 0.107438, 0.274859]]}, {"filename": "000000111207.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.42749, 0.715625, 0.201255, 0.334625], [3, 0.650042, 0.297547, 0.188368, 0.103906], [3, 0.476266, 0.288813, 0.185711, 0.076063], [0, 0.334142, 0.253406, 0.240753, 0.412031]]}, {"filename": "000000111280.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.90628, 0.775571, 0.13768, 0.386517]]}, {"filename": "000000112378.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.273148, 0.345385, 0.134422, 0.085771], [3, 0.07418, 0.896771, 0.148359, 0.206458]]}, {"filename": "000000114049.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.567647, 0.328086, 0.429271, 0.391797], [0, 0.231176, 0.463914, 0.201506, 0.424234], [0, 0.727353, 0.304383, 0.099082, 0.139047], [4, 0.717071, 0.712398, 0.107271, 0.249953], [4, 0.194506, 0.615258, 0.130471, 0.383516], [0, 0.324365, 0.123898, 0.276259, 0.203297], [0, 0.708118, 0.046133, 0.072565, 0.092266], [0, 0.849141, 0.309633, 0.040118, 0.127484], [4, 0.053588, 0.694547, 0.107176, 0.170281]]}, {"filename": "000000114348.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.05991, 0.7674, 0.11898, 0.458107], [7, 0.67692, 0.342107, 0.281, 0.434347], [7, 0.40028, 0.346827, 0.27472, 0.433707], [0, 0.96827, 0.335467, 0.01786, 0.091947], [1, 0.8868, 0.702253, 0.2264, 0.57304]]}, {"filename": "000000114408.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.232961, 0.749802, 0.215891, 0.193729], [1, 0.070492, 0.766115, 0.138234, 0.251938], [1, 0.565367, 0.687448, 0.097297, 0.093521], [1, 0.629656, 0.674646, 0.078406, 0.073833], [1, 0.661008, 0.672385, 0.038203, 0.069312], [1, 0.972555, 0.66625, 0.025703, 0.043333], [0, 0.757953, 0.66249, 0.017906, 0.081312], [0, 0.287766, 0.652542, 0.032187, 0.052458], [0, 0.440937, 0.661229, 0.028313, 0.118], [0, 0.465484, 0.665135, 0.021125, 0.098062], [0, 0.693867, 0.731125, 0.068641, 0.227333], [0, 0.731328, 0.665052, 0.027, 0.073771], [1, 0.961094, 0.648448, 0.033812, 0.041521], [1, 0.736086, 0.62301, 0.037422, 0.025479], [1, 0.951836, 0.635177, 0.027891, 0.023438], [1, 0.836148, 0.611229, 0.048672, 0.100875], [0, 0.939867, 0.642312, 0.008047, 0.017083], [1, 0.898695, 0.609948, 0.077109, 0.066687]]}, {"filename": "000000114500.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.5, 0.647, 1.0, 0.706], [1, 0.757188, 0.320305, 0.22875, 0.075703], [1, 0.230563, 0.298008, 0.182292, 0.041672], [1, 0.072, 0.314906, 0.140125, 0.049281], [7, 0.49626, 0.638898, 0.591938, 0.681609], [0, 0.865094, 0.290719, 0.035021, 0.085562], [0, 0.632833, 0.28318, 0.031833, 0.039047], [0, 0.594854, 0.282016, 0.018625, 0.035156], [0, 0.368792, 0.279273, 0.025708, 0.027078], [0, 0.354583, 0.286609, 0.028542, 0.020219], [0, 0.024833, 0.289437, 0.021375, 0.029219], [1, 0.95026, 0.333367, 0.099479, 0.073703], [1, 0.474729, 0.278117, 0.230833, 0.063578], [0, 0.035677, 0.287359, 0.011063, 0.018438], [1, 0.470531, 0.282461, 0.239438, 0.055141], [0, 0.089104, 0.283297, 0.023625, 0.016344], [7, 0.647583, 0.294922, 0.020875, 0.032594]]}, {"filename": "000000115070.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.619437, 0.178297, 0.761125, 0.349812]]}, {"filename": "000000116831.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.425773, 0.623958, 0.095891, 0.39555], [0, 0.753047, 0.60459, 0.157563, 0.464918], [0, 0.32357, 0.517084, 0.133172, 0.399227], [0, 0.241328, 0.729824, 0.350562, 0.540351], [4, 0.662969, 0.607283, 0.057656, 0.128712], [4, 0.675852, 0.723384, 0.207953, 0.511686], [4, 0.48632, 0.843009, 0.148078, 0.313981]]}, {"filename": "000000117497.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.187135, 0.860555, 0.140316, 0.050641], [1, 0.097276, 0.851227, 0.08942, 0.051047], [0, 0.826371, 0.836227, 0.019033, 0.027453], [0, 0.767996, 0.837414, 0.017961, 0.014797], [0, 0.634429, 0.82818, 0.009174, 0.020391], [1, 0.022496, 0.840344, 0.044991, 0.03175], [3, 0.354657, 0.897359, 0.045062, 0.040312], [0, 0.64449, 0.827914, 0.015167, 0.023609], [1, 0.056547, 0.844789, 0.025817, 0.023328]]}, {"filename": "000000117563.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.400273, 0.593462, 0.242859, 0.79054], [0, 0.678344, 0.506749, 0.276719, 0.959554], [4, 0.424445, 0.818181, 0.316391, 0.327254]]}, {"filename": "000000117744.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.443677, 0.349102, 0.43766, 0.610359]]}, {"filename": "000000117792.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.929563, 0.85281, 0.113937, 0.271897]]}, {"filename": "000000118019.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.23875, 0.819347, 0.169375, 0.213472], [3, 0.690438, 0.726708, 0.194437, 0.213417], [3, 0.920414, 0.628319, 0.135109, 0.159028], [3, 0.187594, 0.876292, 0.365906, 0.241361]]}, {"filename": "000000118515.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.617398, 0.833278, 0.765203, 0.333445]]}, {"filename": "000000119101.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.39274, 0.521832, 0.17816, 0.367027]]}, {"filename": "000000119808.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.220643, 0.759422, 0.093984, 0.234687], [3, 0.101135, 0.78657, 0.19773, 0.136453]]}, {"filename": "000000120420.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.515937, 0.613083, 0.672813, 0.773833], [0, 0.755125, 0.284656, 0.04625, 0.186021], [0, 0.697992, 0.283417, 0.049891, 0.199542], [0, 0.648977, 0.3105, 0.044516, 0.132417], [3, 0.081445, 0.331073, 0.049734, 0.035812], [0, 0.593961, 0.24326, 0.031672, 0.077937], [0, 0.568234, 0.223229, 0.026875, 0.060042], [0, 0.026539, 0.227667, 0.011578, 0.04275], [0, 0.541547, 0.220365, 0.021531, 0.022813], [0, 0.073125, 0.295698, 0.050313, 0.112646], [0, 0.622477, 0.266115, 0.031672, 0.122979]]}, {"filename": "000000121031.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.275539, 0.493825, 0.148859, 0.478647], [0, 0.640937, 0.450222, 0.060937, 0.18714], [0, 0.394586, 0.487007, 0.128609, 0.315876], [0, 0.430211, 0.401275, 0.054797, 0.124013]]}, {"filename": "000000121417.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.56801, 0.605383, 0.705521, 0.757859], [0, 0.164281, 0.078961, 0.137896, 0.153609], [0, 0.635208, 0.054078, 0.20375, 0.098875], [0, 0.890094, 0.043773, 0.066396, 0.087203], [0, 0.211646, 0.049156, 0.065958, 0.098312], [0, 0.81001, 0.028969, 0.090146, 0.057313], [0, 0.439885, 0.044234, 0.218396, 0.088469], [0, 0.97, 0.040258, 0.059167, 0.080516], [0, 0.467146, 0.026766, 0.082458, 0.052562], [3, 0.845021, 0.714289, 0.30475, 0.140266]]}, {"filename": "000000121506.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.357941, 0.838023, 0.653235, 0.288595], [3, 0.200057, 0.684673, 0.17799, 0.11415], [3, 0.129208, 0.831462, 0.195507, 0.197761], [4, 0.770727, 0.870637, 0.243546, 0.18585], [1, 0.764461, 0.458938, 0.196634, 0.069641], [4, 0.180057, 0.840972, 0.297925, 0.204657]]}, {"filename": "000000121929.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.04742, 0.610136, 0.03108, 0.160331], [1, 0.13428, 0.541114, 0.01416, 0.059819], [0, 0.67442, 0.693946, 0.11016, 0.5], [0, 0.0471, 0.534849, 0.07372, 0.265241], [0, 0.13385, 0.515467, 0.02478, 0.0875], [0, 0.19009, 0.488614, 0.01282, 0.036084], [0, 0.25279, 0.501837, 0.0113, 0.040964], [0, 0.22882, 0.492711, 0.00944, 0.032831], [0, 0.24148, 0.495723, 0.00752, 0.024518], [0, 0.09164, 0.483991, 0.01344, 0.044789], [0, 0.17536, 0.491747, 0.00872, 0.026928], [0, 0.56437, 0.640226, 0.07906, 0.388705], [0, 0.14962, 0.485346, 0.008, 0.026657], [0, 0.21721, 0.485377, 0.00786, 0.021898], [1, 0.19023, 0.501295, 0.00554, 0.020241], [0, 0.10771, 0.484006, 0.0089, 0.025422], [6, 0.86157, 0.379864, 0.02174, 0.111596]]}, {"filename": "000000122046.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.345726, 0.716844, 0.213232, 0.397406], [0, 0.420281, 0.623875, 0.455738, 0.648656]]}, {"filename": "000000123282.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.457273, 0.755979, 0.063047, 0.046736], [1, 0.38425, 0.731305, 0.024594, 0.036292], [1, 0.480758, 0.707206, 0.155328, 0.113786], [0, 0.288828, 0.697846, 0.160938, 0.59141], [3, 0.221219, 0.885013, 0.423625, 0.193264], [1, 0.385727, 0.649478, 0.022766, 0.02705], [1, 0.370063, 0.641697, 0.017687, 0.017807], [1, 0.446742, 0.640953, 0.033391, 0.01846], [1, 0.37075, 0.722219, 0.033625, 0.060627], [1, 0.409539, 0.657493, 0.028578, 0.031227], [1, 0.177484, 0.756775, 0.074594, 0.065927]]}, {"filename": "000000123633.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.1895, 0.884354, 0.118312, 0.205875], [0, 0.359836, 0.765625, 0.338766, 0.467417], [0, 0.718352, 0.654125, 0.307297, 0.45175], [0, 0.675, 0.407687, 0.41375, 0.815375], [0, 0.300672, 0.296167, 0.24325, 0.590083], [0, 0.782969, 0.129667, 0.14175, 0.250542]]}, {"filename": "000000124102.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.7274, 0.830821, 0.26216, 0.155313], [1, 0.94171, 0.818328, 0.08886, 0.146328], [1, 0.32267, 0.639642, 0.08518, 0.051642], [1, 0.97904, 0.873343, 0.04192, 0.253313], [0, 0.43728, 0.631791, 0.01172, 0.04006], [0, 0.89785, 0.743955, 0.01178, 0.044448], [1, 0.24452, 0.645254, 0.04848, 0.048955], [7, 0.16616, 0.636896, 0.08616, 0.175224], [0, 0.63084, 0.742388, 0.01576, 0.024358], [0, 0.76759, 0.745567, 0.0133, 0.042358], [0, 0.58997, 0.747881, 0.0117, 0.055761], [0, 0.86285, 0.752612, 0.0123, 0.037284], [0, 0.93184, 0.728478, 0.02032, 0.052239], [0, 0.7368, 0.747537, 0.01152, 0.015194]]}, {"filename": "000000124466.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.393917, 0.251523, 0.272333, 0.233109], [1, 0.143552, 0.54075, 0.282687, 0.154188], [1, 0.820156, 0.523031, 0.353437, 0.12325], [1, 0.498563, 0.573672, 0.507042, 0.164125], [0, 0.42676, 0.474422, 0.034021, 0.035531], [1, 0.935375, 0.534602, 0.127708, 0.100453], [0, 0.443458, 0.530727, 0.0245, 0.028828]]}, {"filename": "000000124659.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.148328, 0.519219, 0.136094, 0.228938], [4, 0.204867, 0.44974, 0.093547, 0.182521], [4, 0.954414, 0.834021, 0.091172, 0.261]]}, {"filename": "000000124803.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.25296, 0.79194, 0.296213, 0.25088]]}, {"filename": "000000125524.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.401461, 0.637564, 0.361047, 0.724871], [0, 0.042641, 0.339578, 0.076625, 0.36918], [1, 0.477945, 0.369895, 0.695609, 0.739789]]}, {"filename": "000000125643.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.67501, 0.502812, 0.37306, 0.90026]]}, {"filename": "000000125778.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.548891, 0.55574, 0.243219, 0.331188], [1, 0.213789, 0.371969, 0.128297, 0.169354]]}, {"filename": "000000126592.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.513125, 0.526963, 0.214375, 0.312356], [1, 0.645391, 0.689711, 0.300438, 0.133002], [1, 0.238078, 0.624053, 0.127219, 0.090878], [1, 0.044422, 0.657298, 0.088594, 0.121432], [1, 0.178898, 0.644642, 0.078734, 0.080185], [1, 0.360547, 0.618533, 0.089094, 0.064688], [1, 0.092422, 0.640947, 0.114469, 0.101524], [1, 0.973555, 0.675693, 0.052891, 0.15157], [1, 0.310648, 0.62933, 0.049828, 0.055566], [1, 0.686914, 0.624007, 0.172453, 0.098499], [1, 0.410117, 0.610196, 0.048672, 0.053649], [1, 0.713242, 0.602252, 0.073203, 0.036928], [1, 0.718289, 0.603372, 0.059328, 0.029561]]}, {"filename": "000000126608.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.664305, 0.418282, 0.161672, 0.148391], [1, 0.293508, 0.606763, 0.584734, 0.634756], [0, 0.65993, 0.348011, 0.081141, 0.194647], [0, 0.601969, 0.314846, 0.010594, 0.036166], [1, 0.774563, 0.328499, 0.025312, 0.024991], [5, 0.564656, 0.32, 0.007781, 0.021085], [1, 0.794859, 0.319629, 0.00775, 0.012749], [1, 0.787883, 0.321781, 0.010359, 0.012025]]}, {"filename": "000000126994.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.692208, 0.162234, 0.430117, 0.289875]]}, {"filename": "000000127092.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.286016, 0.903469, 0.111844, 0.105271], [1, 0.143961, 0.831812, 0.091578, 0.072625], [0, 0.057016, 0.935083, 0.038094, 0.118208], [5, 0.614289, 0.924125, 0.019203, 0.04825], [6, 0.601906, 0.740573, 0.017719, 0.049479], [1, 0.012219, 0.749333, 0.024438, 0.048708]]}, {"filename": "000000127270.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.895777, 0.0883, 0.208445, 0.17632], [0, 0.247317, 0.07819, 0.22939, 0.15638], [3, 0.512683, 0.07051, 0.198232, 0.06054], [4, 0.207271, 0.11551, 0.077226, 0.12262], [0, 0.10657, 0.09101, 0.209543, 0.1819], [0, 0.69125, 0.08834, 0.272073, 0.17668], [0, 0.496159, 0.54138, 0.753841, 0.88748]]}, {"filename": "000000127394.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.112702, 0.329734, 0.225403, 0.389562], [0, 0.320452, 0.254641, 0.3122, 0.2675], [0, 0.545171, 0.215906, 0.264866, 0.269656], [0, 0.903203, 0.425641, 0.193594, 0.139344], [0, 0.979132, 0.18657, 0.041736, 0.084672], [4, 0.425758, 0.263188, 0.199144, 0.169406], [0, 0.716235, 0.158078, 0.160978, 0.163937]]}, {"filename": "000000127400.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.147281, 0.232762, 0.294563, 0.463347], [1, 0.409609, 0.577709, 0.815844, 0.824331], [0, 0.653523, 0.533944, 0.280078, 0.927803]]}, {"filename": "000000128829.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.692135, 0.290633, 0.611229, 0.406734], [1, 0.369073, 0.151133, 0.280813, 0.082797], [7, 0.706719, 0.380266, 0.411687, 0.687625], [7, 0.250187, 0.429961, 0.4045, 0.617984], [1, 0.055229, 0.244125, 0.110458, 0.198406]]}, {"filename": "000000129062.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.087641, 0.70075, 0.175281, 0.575292], [0, 0.905055, 0.851688, 0.144953, 0.274167], [7, 0.23993, 0.841844, 0.032234, 0.094979]]}, {"filename": "000000129135.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.661516, 0.867042, 0.382594, 0.20675]]}, {"filename": "000000129492.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.494094, 0.577708, 0.498313, 0.835583], [0, 0.648031, 0.530333, 0.267969, 0.462917], [0, 0.559555, 0.19776, 0.306734, 0.386521], [4, 0.536891, 0.208115, 0.412, 0.416229]]}, {"filename": "000000130386.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.756253, 0.95323, 0.06888, 0.0841], [1, 0.53076, 0.90527, 0.145787, 0.0631], [0, 0.086373, 0.89553, 0.02096, 0.05106], [0, 0.101533, 0.89977, 0.0196, 0.04574], [3, 0.164053, 0.93495, 0.045867, 0.03838], [0, 0.1722, 0.96222, 0.096347, 0.07556], [0, 0.2144, 0.92305, 0.025227, 0.07906], [0, 0.589573, 0.86874, 0.023147, 0.03448], [0, 0.514133, 0.86509, 0.01072, 0.01598], [1, 0.73192, 0.95551, 0.0592, 0.07794], [3, 0.363267, 0.97308, 0.121093, 0.05384], [1, 0.944533, 0.96173, 0.076107, 0.07654]]}, {"filename": "000000130733.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.461427, 0.564047, 0.287646, 0.233719], [1, 0.933406, 0.853836, 0.133188, 0.119641]]}, {"filename": "000000131494.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.515758, 0.604496, 0.551734, 0.764028], [1, 0.258727, 0.377717, 0.176203, 0.325738], [0, 0.39282, 0.377529, 0.161922, 0.642693], [0, 0.411461, 0.448501, 0.226266, 0.752693], [0, 0.248281, 0.311276, 0.067031, 0.270468], [0, 0.724883, 0.30055, 0.020984, 0.049719], [1, 0.31007, 0.443396, 0.082578, 0.194754], [7, 0.803992, 0.344368, 0.014359, 0.056464], [1, 0.170305, 0.334368, 0.056109, 0.069485], [1, 0.013922, 0.325375, 0.027531, 0.074965]]}, {"filename": "000000132294.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.500838, 0.360399, 0.998325, 0.71588]]}, {"filename": "000000132375.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.966039, 0.107438, 0.067922, 0.07325], [1, 0.804758, 0.127927, 0.061609, 0.040771], [0, 0.921258, 0.11626, 0.025234, 0.051021], [0, 0.900656, 0.118531, 0.017563, 0.045062], [0, 0.883391, 0.126562, 0.033625, 0.044833]]}, {"filename": "000000132587.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.255219, 0.496129, 0.465594, 0.913671], [0, 0.818187, 0.082988, 0.065781, 0.165976], [0, 0.531328, 0.100047, 0.08675, 0.198212], [0, 0.012852, 0.334329, 0.025703, 0.070729], [0, 0.417516, 0.053941, 0.065031, 0.107882], [3, 0.706117, 0.658224, 0.587766, 0.683553], [0, 0.389086, 0.042341, 0.051328, 0.082518]]}, {"filename": "000000133298.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.173719, 0.976896, 0.0165, 0.031208]]}, {"filename": "000000133418.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.625594, 0.53404, 0.302625, 0.896557]]}, {"filename": "000000133636.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.475656, 0.590329, 0.155562, 0.786526], [7, 0.281211, 0.561796, 0.260266, 0.876408]]}, {"filename": "000000133819.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.556617, 0.525847, 0.656609, 0.651694], [1, 0.875219, 0.456929, 0.249219, 0.312353], [1, 0.983102, 0.504965, 0.033797, 0.240235], [0, 0.17107, 0.5716, 0.032547, 0.159812], [0, 0.063531, 0.537141, 0.044375, 0.079647], [0, 0.122836, 0.577929, 0.036859, 0.180047], [0, 0.107328, 0.536224, 0.023469, 0.0732], [0, 0.085625, 0.532271, 0.028063, 0.084071], [0, 0.01807, 0.537094, 0.036141, 0.081624], [0, 0.154781, 0.530294, 0.013813, 0.066753], [0, 0.780937, 0.520518, 0.029406, 0.052518], [0, 0.750375, 0.519471, 0.034219, 0.064306], [7, 0.935117, 0.685965, 0.068547, 0.171129], [0, 0.79807, 0.508247, 0.026234, 0.063035], [0, 0.699492, 0.520459, 0.037828, 0.074376]]}, {"filename": "000000133827.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.77068, 0.690283, 0.057641, 0.042207], [1, 0.705367, 0.684727, 0.046328, 0.029805], [1, 0.355992, 0.695898, 0.084078, 0.048789], [5, 0.513164, 0.777324, 0.022328, 0.062656], [1, 0.164422, 0.690859, 0.049406, 0.033438], [1, 0.325219, 0.682568, 0.062094, 0.034824], [1, 0.528172, 0.674502, 0.039438, 0.021309], [1, 0.227898, 0.684395, 0.065078, 0.040977], [7, 0.881984, 0.675801, 0.007875, 0.018203], [1, 0.163156, 0.676553, 0.057312, 0.022285], [1, 0.120211, 0.692813, 0.020609, 0.031484], [1, 0.383352, 0.68292, 0.027391, 0.022637], [7, 0.658391, 0.677227, 0.004219, 0.023203]]}, {"filename": "000000133969.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.651289, 0.606815, 0.317297, 0.547541], [0, 0.39532, 0.581124, 0.118641, 0.335878], [0, 0.372219, 0.385141, 0.089344, 0.250937], [0, 0.746828, 0.525902, 0.102187, 0.454941], [0, 0.550656, 0.380316, 0.09575, 0.190375], [0, 0.900805, 0.577084, 0.102672, 0.322037], [0, 0.231539, 0.685609, 0.272797, 0.382927], [0, 0.09025, 0.598208, 0.179938, 0.551874], [0, 0.463289, 0.492131, 0.098953, 0.48089], [0, 0.956297, 0.501288, 0.087406, 0.478923], [0, 0.609617, 0.335258, 0.045891, 0.114169], [0, 0.528805, 0.578841, 0.108359, 0.329204], [0, 0.707961, 0.447963, 0.042797, 0.290913], [3, 0.486711, 0.595059, 0.723828, 0.023232]]}, {"filename": "000000134722.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.034344, 0.641813, 0.068687, 0.073625]]}, {"filename": "000000135348.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.534249, 0.622188, 0.2523, 0.300813], [0, 0.50473, 0.622102, 0.509789, 0.732578], [0, 0.950235, 0.661992, 0.099531, 0.401891], [0, 0.13196, 0.475227, 0.25716, 0.430172], [0, 0.929448, 0.539539, 0.10115, 0.343547], [0, 0.732969, 0.510828, 0.064531, 0.171406], [0, 0.36527, 0.376367, 0.096972, 0.152578], [0, 0.850505, 0.536406, 0.116268, 0.207938], [0, 0.961538, 0.441789, 0.076925, 0.132641], [0, 0.261538, 0.419891, 0.03115, 0.047562]]}, {"filename": "000000135410.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.055979, 0.417602, 0.111958, 0.143484], [1, 0.829375, 0.444234, 0.268333, 0.110969], [1, 0.563958, 0.480641, 0.391542, 0.173094], [7, 0.242125, 0.587828, 0.298333, 0.499312], [0, 0.514844, 0.421422, 0.169438, 0.229438], [1, 0.555677, 0.420117, 0.469396, 0.146578], [1, 0.105854, 0.884617, 0.211708, 0.230766], [1, 0.074917, 0.371445, 0.145583, 0.071797], [1, 0.854385, 0.354078, 0.149354, 0.031094], [1, 0.42124, 0.341227, 0.353813, 0.059891]]}, {"filename": "000000137745.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.810615, 0.691922, 0.130063, 0.297438], [1, 0.076927, 0.665156, 0.093687, 0.054719]]}, {"filename": "000000138550.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.696219, 0.166051, 0.135063, 0.208364], [4, 0.24532, 0.326121, 0.193297, 0.463645], [5, 0.494422, 0.542325, 0.162312, 0.519089]]}, {"filename": "000000140758.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.508406, 0.214359, 0.265188, 0.384531], [7, 0.226219, 0.205813, 0.260688, 0.384656]]}, {"filename": "000000142259.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.376281, 0.815082, 0.139687, 0.293357], [1, 0.917305, 0.745423, 0.163547, 0.278028], [1, 0.027328, 0.556714, 0.054, 0.119061], [0, 0.640188, 0.607805, 0.136094, 0.465047], [0, 0.264313, 0.688134, 0.105719, 0.5973], [0, 0.065602, 0.477512, 0.031891, 0.135962], [1, 0.11618, 0.563181, 0.151203, 0.171667], [0, 0.366016, 0.755927, 0.142625, 0.444859]]}, {"filename": "000000142346.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.743266, 0.653937, 0.175281, 0.471917], [0, 0.439742, 0.46301, 0.210203, 0.553812], [0, 0.22668, 0.378802, 0.089328, 0.222479], [0, 0.071141, 0.397104, 0.042125, 0.111], [0, 0.364094, 0.380302, 0.055531, 0.230437], [0, 0.271133, 0.406292, 0.032016, 0.097917], [0, 0.002086, 0.38075, 0.004172, 0.017833]]}, {"filename": "000000143711.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.642781, 0.56899, 0.005312, 0.040348], [0, 0.772898, 0.647892, 0.008891, 0.062892], [0, 0.651102, 0.649425, 0.012172, 0.063171], [0, 0.914695, 0.690226, 0.022609, 0.094878], [0, 0.558453, 0.553955, 0.005375, 0.030697], [0, 0.079422, 0.710244, 0.026219, 0.097631], [0, 0.59832, 0.667143, 0.013359, 0.054774], [0, 0.630039, 0.567491, 0.005922, 0.025087], [0, 0.795078, 0.575314, 0.00575, 0.037875], [0, 0.665312, 0.544216, 0.005344, 0.025296], [0, 0.760656, 0.636672, 0.007313, 0.046098], [0, 0.474086, 0.550557, 0.006797, 0.0177], [0, 0.576945, 0.554338, 0.003641, 0.025401]]}, {"filename": "000000143781.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.819229, 0.424203, 0.263333, 0.123969], [0, 0.585615, 0.464641, 0.258062, 0.63275]]}, {"filename": "000000144300.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.489875, 0.537155, 0.8385, 0.747752], [0, 0.849852, 0.290691, 0.108578, 0.261007], [0, 0.973484, 0.379543, 0.053031, 0.438946], [1, 0.250383, 0.273033, 0.500766, 0.541569], [1, 0.252047, 0.274075, 0.504094, 0.54815], [1, 0.528813, 0.234567, 0.121906, 0.232084], [1, 0.668016, 0.26733, 0.222938, 0.269789], [1, 0.663734, 0.268419, 0.219031, 0.275808], [1, 0.938016, 0.374918, 0.081844, 0.086511]]}, {"filename": "000000144582.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.692344, 0.483305, 0.348031, 0.269661], [3, 0.515664, 0.569873, 0.704203, 0.52596]]}, {"filename": "000000144694.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.949172, 0.454635, 0.101656, 0.618824], [0, 0.890141, 0.462929, 0.219375, 0.759553], [4, 0.176086, 0.643812, 0.107453, 0.217976], [0, 0.387992, 0.496624, 0.456641, 0.696635]]}, {"filename": "000000144706.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.659109, 0.799802, 0.079031, 0.054104], [1, 0.815219, 0.806427, 0.082844, 0.076604], [1, 0.817313, 0.805802, 0.077031, 0.070271]]}, {"filename": "000000145163.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.78892, 0.723147, 0.24284, 0.43648]]}, {"filename": "000000145718.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.109711, 0.515738, 0.112484, 0.478033], [0, 0.598664, 0.599895, 0.146078, 0.320539], [0, 0.729688, 0.496054, 0.4955, 0.984192], [0, 0.051391, 0.510585, 0.095969, 0.484731], [1, 0.174758, 0.486639, 0.039203, 0.069953], [1, 0.374547, 0.500972, 0.079062, 0.06171], [0, 0.272367, 0.613536, 0.221578, 0.397658], [0, 0.481516, 0.444508, 0.011281, 0.024239], [0, 0.410367, 0.439602, 0.015734, 0.03466], [1, 0.500203, 0.495398, 0.14125, 0.090468], [1, 0.009859, 0.50377, 0.019094, 0.065948], [1, 0.422633, 0.487037, 0.073578, 0.064239], [7, 0.471172, 0.476944, 0.004812, 0.026066], [1, 0.501125, 0.488466, 0.140125, 0.082881], [7, 0.461867, 0.476956, 0.007141, 0.025059]]}, {"filename": "000000146363.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.62985, 0.710413, 0.40918, 0.247493], [4, 0.32692, 0.91496, 0.236, 0.17008]]}, {"filename": "000000146498.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.182383, 0.641896, 0.029766, 0.019333], [1, 0.209328, 0.629271, 0.012719, 0.018083], [1, 0.188188, 0.577365, 0.023656, 0.010687], [1, 0.215664, 0.572396, 0.021391, 0.010417], [1, 0.184602, 0.588969, 0.027953, 0.017188], [1, 0.163781, 0.575521, 0.014781, 0.006]]}, {"filename": "000000149414.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.717289, 0.552307, 0.082891, 0.038384], [1, 0.248328, 0.511335, 0.021031, 0.021218], [1, 0.259063, 0.507002, 0.018344, 0.023981], [1, 0.280766, 0.504262, 0.024969, 0.020796], [1, 0.495133, 0.561932, 0.380172, 0.427939], [0, 0.136281, 0.539707, 0.014219, 0.045082], [0, 0.101281, 0.542681, 0.0155, 0.067752], [0, 0.179164, 0.552014, 0.028109, 0.084543], [0, 0.175609, 0.510199, 0.009469, 0.029391], [5, 0.224648, 0.663536, 0.023641, 0.109836], [1, 0.227539, 0.501347, 0.010578, 0.016698], [3, 0.125805, 0.541874, 0.015297, 0.028946]]}, {"filename": "000000150224.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.769141, 0.862916, 0.197906, 0.24719], [0, 0.387563, 0.813478, 0.133438, 0.269672], [0, 0.737656, 0.755047, 0.194906, 0.256183], [0, 0.601672, 0.718396, 0.122, 0.435667], [0, 0.67225, 0.69459, 0.071, 0.257002], [0, 0.667906, 0.819778, 0.153188, 0.316932], [0, 0.507625, 0.818197, 0.037312, 0.059953]]}, {"filename": "000000150773.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.640594, 0.284578, 0.424031, 0.464844]]}, {"filename": "000000151231.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.737422, 0.511232, 0.314125, 0.469648], [1, 0.361617, 0.857923, 0.051859, 0.040446], [1, 0.304414, 0.874319, 0.035609, 0.032535]]}, {"filename": "000000151371.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.33718, 0.824731, 0.668609, 0.350539], [1, 0.435023, 0.461148, 0.855016, 0.922295], [1, 0.046039, 0.565105, 0.085141, 0.122389]]}, {"filename": "000000151820.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.3235, 0.86356, 0.236813, 0.272881], [4, 0.673055, 0.834965, 0.256484, 0.304028], [4, 0.886984, 0.815925, 0.218469, 0.361171], [4, 0.964531, 0.568583, 0.062438, 0.148407], [4, 0.567187, 0.460152, 0.029844, 0.051452], [0, 0.457281, 0.438197, 0.134937, 0.224731], [0, 0.293586, 0.462237, 0.159828, 0.307845], [0, 0.683312, 0.68534, 0.295063, 0.627307], [0, 0.624906, 0.435164, 0.142844, 0.212342], [0, 0.214398, 0.619098, 0.230891, 0.559555], [4, 0.400953, 0.473372, 0.023937, 0.063794], [4, 0.085453, 0.760667, 0.062969, 0.456183], [0, 0.893492, 0.476124, 0.089828, 0.246862], [0, 0.863094, 0.62137, 0.151844, 0.420539]]}, {"filename": "000000152056.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.935484, 0.448609, 0.129031, 0.072887], [7, 0.500273, 0.539331, 0.067797, 0.188769]]}, {"filename": "000000152764.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.491367, 0.602242, 0.704516, 0.710117], [1, 0.254359, 0.441784, 0.491375, 0.502488], [1, 0.183984, 0.304495, 0.341031, 0.285376], [1, 0.142461, 0.243322, 0.254578, 0.201479], [0, 0.278273, 0.116796, 0.020797, 0.069366], [0, 0.452203, 0.135082, 0.016906, 0.113357], [0, 0.610711, 0.179953, 0.079609, 0.285305], [0, 0.708062, 0.23696, 0.08325, 0.431995], [0, 0.378437, 0.119096, 0.065813, 0.188756], [0, 0.900555, 0.156467, 0.021922, 0.134108], [0, 0.563977, 0.162347, 0.039391, 0.215962], [0, 0.650961, 0.167347, 0.036953, 0.214836], [0, 0.793313, 0.132958, 0.019188, 0.09216], [0, 0.872477, 0.158627, 0.032547, 0.113451], [0, 0.781766, 0.140528, 0.021094, 0.092887], [0, 0.80775, 0.136338, 0.024563, 0.101033], [0, 0.501273, 0.148322, 0.016391, 0.104437], [6, 0.097305, 0.125563, 0.012672, 0.020751]]}, {"filename": "000000152771.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.612969, 0.679081, 0.177375, 0.177495], [1, 0.911914, 0.601921, 0.048078, 0.059958], [5, 0.391391, 0.65024, 0.028187, 0.068372], [1, 0.052797, 0.57548, 0.100781, 0.066618], [1, 0.972508, 0.604259, 0.047172, 0.05833], [5, 0.966289, 0.640866, 0.013922, 0.046618]]}, {"filename": "000000153217.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.553933, 0.42097, 0.892133, 0.8337]]}, {"filename": "000000153568.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.49978, 0.467425, 0.99956, 0.92135]]}, {"filename": "000000154425.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.431719, 0.496047, 0.425719, 0.986447]]}, {"filename": "000000155341.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.425797, 0.729521, 0.376813, 0.272458], [1, 0.249164, 0.543844, 0.340609, 0.196854], [0, 0.044555, 0.590042, 0.026578, 0.099833], [0, 0.291383, 0.583333, 0.025328, 0.056292], [0, 0.377625, 0.667573, 0.046344, 0.058604], [0, 0.36907, 0.57299, 0.027578, 0.047271], [0, 0.320422, 0.589063, 0.033531, 0.019042], [0, 0.445609, 0.586552, 0.024344, 0.040271], [1, 0.675922, 0.574073, 0.529625, 0.229646]]}, {"filename": "000000155364.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.917406, 0.61279, 0.165, 0.220662], [0, 0.177594, 0.513783, 0.020781, 0.032057], [0, 0.404102, 0.511537, 0.017891, 0.020662], [0, 0.174867, 0.494764, 0.008891, 0.00896], [0, 0.167859, 0.516395, 0.010969, 0.023853], [0, 0.253953, 0.515485, 0.02025, 0.02539], [0, 0.370297, 0.508818, 0.018938, 0.029787], [0, 0.651609, 0.499681, 0.019562, 0.0326], [0, 0.607586, 0.493995, 0.028234, 0.046288], [6, 0.516109, 0.175863, 0.041594, 0.286619]]}, {"filename": "000000155376.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.547135, 0.486617, 0.337271, 0.403359]]}, {"filename": "000000155739.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.59085, 0.51653, 0.75482, 0.897117], [1, 0.89466, 0.067402, 0.21068, 0.125836], [1, 0.69966, 0.042135, 0.17932, 0.083132], [1, 0.50381, 0.04911, 0.2519, 0.097367], [1, 0.25322, 0.022616, 0.06364, 0.04516], [1, 0.01896, 0.044395, 0.03736, 0.070996], [7, 0.11813, 0.337189, 0.20962, 0.589181], [1, 0.13197, 0.101886, 0.19322, 0.2], [1, 0.514, 0.043256, 0.24248, 0.08089], [1, 0.20775, 0.021797, 0.02094, 0.03363], [1, 0.18655, 0.018043, 0.04086, 0.035018]]}, {"filename": "000000155997.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.385437, 0.779646, 0.648813, 0.419708], [1, 0.553648, 0.539917, 0.260734, 0.175958], [0, 0.51007, 0.44876, 0.107391, 0.351229], [0, 0.179969, 0.549823, 0.144937, 0.687646], [0, 0.592906, 0.30575, 0.054156, 0.083875], [0, 0.312914, 0.576281, 0.095641, 0.494396], [0, 0.704852, 0.389396, 0.055172, 0.142625], [0, 0.805781, 0.396438, 0.055437, 0.15575], [0, 0.249094, 0.391146, 0.053531, 0.192583], [0, 0.327516, 0.35426, 0.062875, 0.114313], [7, 0.610297, 0.370312, 0.029594, 0.054083], [7, 0.592141, 0.402073, 0.059719, 0.118938], [0, 0.76193, 0.3895, 0.037297, 0.1295], [0, 0.627906, 0.338188, 0.030687, 0.055708], [0, 0.050648, 0.481948, 0.101297, 0.520938], [0, 0.6535, 0.347104, 0.038281, 0.067792], [0, 0.557695, 0.334219, 0.041172, 0.085021]]}, {"filename": "000000156643.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.531391, 0.515917, 0.443937, 0.468625], [0, 0.848836, 0.473948, 0.108203, 0.203187], [0, 0.937359, 0.48301, 0.125281, 0.297479], [0, 0.806828, 0.364177, 0.021781, 0.053604], [0, 0.03632, 0.480094, 0.072641, 0.435396], [0, 0.458859, 0.348552, 0.02225, 0.089854], [0, 0.29157, 0.342698, 0.168547, 0.680896], [4, 0.955523, 0.722198, 0.088953, 0.222188], [4, 0.872711, 0.603167, 0.098141, 0.076208], [0, 0.423375, 0.370292, 0.032719, 0.0375], [0, 0.819789, 0.409135, 0.055016, 0.075313], [4, 0.911391, 0.431687, 0.047438, 0.060083], [0, 0.769641, 0.33375, 0.052031, 0.13875]]}, {"filename": "000000156747.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.728914, 0.306913, 0.150391, 0.331103], [1, 0.520859, 0.737887, 0.955156, 0.5], [0, 0.500336, 0.630329, 0.414328, 0.716854]]}, {"filename": "000000157109.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.622031, 0.950781, 0.069366, 0.052625]]}, {"filename": "000000157610.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.418469, 0.669719, 0.266656, 0.171944]]}, {"filename": "000000158818.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.594473, 0.565914, 0.710679, 0.768547], [3, 0.117354, 0.338023, 0.234707, 0.384797], [3, 0.871417, 0.500938, 0.257166, 0.169813]]}, {"filename": "000000158922.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.222891, 0.114115, 0.039469, 0.128396], [0, 0.255305, 0.09126, 0.012984, 0.044354], [0, 0.628555, 0.1135, 0.008734, 0.029125], [0, 0.062906, 0.111885, 0.037344, 0.032521], [0, 0.413227, 0.553042, 0.209609, 0.602708], [1, 0.130484, 0.072906, 0.091, 0.058104], [1, 0.983844, 0.128312, 0.032312, 0.062292], [0, 0.024359, 0.102823, 0.027938, 0.031563], [0, 0.028266, 0.097656, 0.0165, 0.019062], [0, 0.618766, 0.112031, 0.0075, 0.031271], [0, 0.828836, 0.128615, 0.012453, 0.029813], [0, 0.438344, 0.119885, 0.235313, 0.239771], [0, 0.135828, 0.080417, 0.016219, 0.046333], [0, 0.095477, 0.118031, 0.035047, 0.039854], [0, 0.753773, 0.123292, 0.005297, 0.024625]]}, {"filename": "000000158956.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.51707, 0.5382, 0.505891, 0.892118], [4, 0.974937, 0.869341, 0.049312, 0.261318]]}, {"filename": "000000160812.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.520553, 0.702469, 0.193341, 0.2]]}, {"filename": "000000160970.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.516305, 0.447906, 0.213734, 0.306312], [0, 0.473594, 0.343448, 0.124719, 0.247188], [0, 0.315289, 0.711156, 0.032172, 0.115437]]}, {"filename": "000000161128.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.78084, 0.80611, 0.125413, 0.05302], [0, 0.271173, 0.80374, 0.022827, 0.05956], [1, 0.855387, 0.77965, 0.078347, 0.0273], [1, 0.580827, 0.7948, 0.099893, 0.16644], [5, 0.96792, 0.8075, 0.012427, 0.02132], [0, 0.376707, 0.7968, 0.02312, 0.04852], [0, 0.295427, 0.80646, 0.021253, 0.05424], [1, 0.633693, 0.79181, 0.02696, 0.03542], [1, 0.15852, 0.79791, 0.100293, 0.0741], [1, 0.3988, 0.7733, 0.02128, 0.0118], [1, 0.584347, 0.79803, 0.101227, 0.16486], [1, 0.066387, 0.82952, 0.132773, 0.09464], [1, 0.783333, 0.77265, 0.047627, 0.01962], [1, 0.014947, 0.85685, 0.026747, 0.11486]]}, {"filename": "000000161218.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.078792, 0.7039, 0.157583, 0.56625]]}, {"filename": "000000161840.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.885461, 0.726937, 0.060266, 0.148096]]}, {"filename": "000000162130.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.353859, 0.776719, 0.333281, 0.257062]]}, {"filename": "000000162573.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.338086, 0.584262, 0.430297, 0.804496]]}, {"filename": "000000163118.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.5005, 0.583357, 0.173562, 0.833286], [0, 0.349203, 0.529413, 0.227469, 0.704319]]}, {"filename": "000000163314.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.22082, 0.798082, 0.04588, 0.15574], [0, 0.65518, 0.702341, 0.01888, 0.079245], [0, 0.29308, 0.776511, 0.03144, 0.160937], [0, 0.84351, 0.800045, 0.04906, 0.174592], [0, 0.14654, 0.80139, 0.04948, 0.191541], [0, 0.62444, 0.793142, 0.05628, 0.18435], [0, 0.10972, 0.788701, 0.04204, 0.161813], [0, 0.37604, 0.767719, 0.036, 0.147341], [0, 0.74303, 0.784366, 0.03258, 0.167402], [0, 0.59846, 0.725498, 0.03176, 0.089184], [0, 0.6055, 0.781208, 0.0202, 0.166284], [0, 0.87961, 0.797432, 0.05282, 0.160423], [0, 0.71725, 0.784184, 0.03742, 0.18148]]}, {"filename": "000000163640.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.307398, 0.65, 0.294578, 0.699125], [0, 0.826688, 0.644869, 0.256313, 0.710263], [0, 0.523664, 0.515295, 0.141016, 0.323939]]}, {"filename": "000000165518.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.567908, 0.631626, 0.281699, 0.404755], [0, 0.548725, 0.518489, 0.285915, 0.468611]]}, {"filename": "000000165874.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.714563, 0.758031, 0.027219, 0.120563]]}, {"filename": "000000166391.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.251141, 0.850644, 0.04175, 0.040539], [1, 0.33757, 0.848876, 0.016234, 0.017658], [1, 0.598023, 0.846241, 0.022203, 0.028735], [1, 0.184891, 0.848946, 0.054406, 0.049555], [1, 0.951109, 0.840644, 0.097312, 0.105831], [1, 0.356875, 0.846698, 0.014969, 0.017658], [1, 0.227273, 0.853091, 0.018547, 0.033255], [1, 0.752133, 0.841452, 0.081453, 0.081874], [1, 0.548602, 0.839227, 0.045641, 0.061311], [1, 0.093648, 0.838911, 0.099078, 0.089719], [1, 0.283336, 0.844859, 0.025703, 0.035667], [1, 0.499859, 0.842307, 0.024437, 0.029813], [1, 0.949805, 0.841569, 0.097453, 0.105621], [1, 0.184336, 0.847506, 0.053922, 0.052904], [1, 0.387313, 0.849297, 0.018719, 0.018548], [1, 0.367516, 0.845714, 0.010344, 0.014473]]}, {"filename": "000000167027.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.31407, 0.689637, 0.025891, 0.061101], [0, 0.237211, 0.706124, 0.031203, 0.090843], [0, 0.045672, 0.719415, 0.041719, 0.192787], [0, 0.01675, 0.718326, 0.0335, 0.202084], [0, 0.296437, 0.698056, 0.020281, 0.041124], [0, 0.439008, 0.679473, 0.024922, 0.053372], [4, 0.235695, 0.733677, 0.040516, 0.029789], [0, 0.980711, 0.710492, 0.033484, 0.083185], [4, 0.343156, 0.804813, 0.038375, 0.113326], [4, 0.316242, 0.789356, 0.025484, 0.096557]]}, {"filename": "000000167126.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.282117, 0.23303, 0.050391, 0.031465], [1, 0.429133, 0.345328, 0.422234, 0.265758], [1, 0.720758, 0.326717, 0.416141, 0.294192], [1, 0.881594, 0.243927, 0.180219, 0.145631], [1, 0.080648, 0.298876, 0.144609, 0.188763], [1, 0.206641, 0.320139, 0.13575, 0.17553], [4, 0.064828, 0.367045, 0.048094, 0.101364], [4, 0.092672, 0.435821, 0.071063, 0.153157], [4, 0.086547, 0.404179, 0.084187, 0.182348], [0, 0.716219, 0.241048, 0.065594, 0.067449], [1, 0.791039, 0.200909, 0.116953, 0.059394], [1, 0.576625, 0.214369, 0.153844, 0.094141], [1, 0.010383, 0.31048, 0.020766, 0.15404], [5, 0.733492, 0.730644, 0.098359, 0.372753], [7, 0.020648, 0.281793, 0.021672, 0.063384], [1, 0.208555, 0.221869, 0.094328, 0.040909], [1, 0.395133, 0.208283, 0.066797, 0.026919]]}, {"filename": "000000167486.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.47543, 0.712698, 0.208297, 0.459813], [0, 0.418969, 0.465615, 0.054062, 0.148979], [0, 0.801867, 0.476469, 0.063453, 0.309438], [0, 0.321914, 0.528094, 0.185391, 0.719104], [0, 0.769266, 0.470313, 0.038719, 0.133583], [0, 0.599875, 0.491792, 0.047875, 0.255292], [1, 0.512477, 0.671219, 0.064797, 0.082812], [0, 0.341102, 0.318635, 0.040984, 0.077604], [0, 0.364969, 0.449677, 0.052781, 0.076271]]}, {"filename": "000000168330.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.225109, 0.521385, 0.038813, 0.047487], [5, 0.300055, 0.575821, 0.023016, 0.084923], [1, 0.156328, 0.533103, 0.063312, 0.049385], [1, 0.057062, 0.530167, 0.076281, 0.105103], [1, 0.020789, 0.537244, 0.041578, 0.102128]]}, {"filename": "000000169097.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.496602, 0.176843, 0.071797, 0.2427], [7, 0.669609, 0.330258, 0.032406, 0.119953], [7, 0.763641, 0.416667, 0.020469, 0.06493], [7, 0.731547, 0.395258, 0.027, 0.081127]]}, {"filename": "000000169243.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.292625, 0.407109, 0.137167, 0.124469]]}, {"filename": "000000169322.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.927039, 0.403354, 0.145141, 0.257333], [4, 0.637195, 0.308937, 0.183797, 0.085]]}, {"filename": "000000169331.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.603484, 0.294073, 0.135719, 0.149646], [1, 0.078766, 0.228302, 0.157531, 0.205479], [1, 0.657664, 0.280844, 0.062516, 0.114771], [1, 0.69143, 0.272375, 0.053609, 0.098458], [1, 0.214047, 0.433708, 0.421344, 0.467417], [0, 0.874719, 0.389896, 0.185406, 0.730333], [0, 0.973977, 0.275865, 0.052047, 0.286646], [0, 0.181211, 0.198125, 0.033078, 0.046625], [7, 0.461961, 0.363885, 0.253828, 0.615521]]}, {"filename": "000000169802.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.302109, 0.5, 0.598219, 1.0], [0, 0.173164, 0.415726, 0.244391, 0.525831], [0, 0.601336, 0.589614, 0.256484, 0.815574]]}, {"filename": "000000169996.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.321773, 0.816125, 0.169359, 0.2], [1, 0.688477, 0.788219, 0.127297, 0.204979], [1, 0.485156, 0.761875, 0.306156, 0.259167], [1, 0.927703, 0.660646, 0.078438, 0.095542], [0, 0.325703, 0.73174, 0.116125, 0.267563], [1, 0.208609, 0.718469, 0.410469, 0.335604], [0, 0.253742, 0.606833, 0.021359, 0.039667], [0, 0.541734, 0.616448, 0.021844, 0.039313], [0, 0.690813, 0.702875, 0.117094, 0.224667], [0, 0.490508, 0.692135, 0.041953, 0.051854], [1, 0.519156, 0.629875, 0.274688, 0.168833]]}, {"filename": "000000170393.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.477352, 0.518115, 0.904359, 0.440729], [7, 0.286555, 0.36774, 0.055297, 0.167521], [0, 0.919148, 0.2595, 0.047047, 0.127], [0, 0.553055, 0.274417, 0.033672, 0.123917], [0, 0.379188, 0.261156, 0.029406, 0.090938]]}, {"filename": "000000170474.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.476859, 0.528382, 0.475063, 0.943235], [4, 0.850414, 0.922731, 0.171047, 0.129202]]}, {"filename": "000000170545.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.460289, 0.623156, 0.034578, 0.030438], [1, 0.210727, 0.599281, 0.040047, 0.028229]]}, {"filename": "000000170670.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.905414, 0.188439, 0.189172, 0.19223], [4, 0.211797, 0.142864, 0.261063, 0.280516]]}, {"filename": "000000170835.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.417984, 0.505791, 0.378344, 0.169004]]}, {"filename": "000000171050.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.671156, 0.690641, 0.617229, 0.5955]]}, {"filename": "000000172856.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.320055, 0.615115, 0.228484, 0.416271]]}, {"filename": "000000174011.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.358242, 0.494378, 0.716484, 0.988756]]}, {"filename": "000000174231.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.200933, 0.7032, 0.401867, 0.5936]]}, {"filename": "000000174313.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.510958, 0.649281, 0.973542, 0.603188], [0, 0.577885, 0.493797, 0.517687, 0.636563]]}, {"filename": "000000175013.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.505336, 0.641576, 0.693047, 0.671907], [1, 0.835813, 0.613473, 0.327937, 0.406712], [0, 0.344062, 0.502451, 0.185062, 0.932879], [0, 0.842383, 0.316946, 0.025641, 0.04358], [0, 0.949672, 0.326138, 0.053125, 0.138191]]}, {"filename": "000000176249.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.06067, 0.746813, 0.12134, 0.235973]]}, {"filename": "000000176318.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.931344, 0.755617, 0.137313, 0.462908], [0, 0.434703, 0.506736, 0.751906, 0.955063]]}, {"filename": "000000176721.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.534439, 0.46955, 0.36018, 0.16586]]}, {"filename": "000000177213.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.633586, 0.195669, 0.732828, 0.391337]]}, {"filename": "000000177489.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.775664, 0.771088, 0.428609, 0.43682], [4, 0.249766, 0.782793, 0.35175, 0.408598], [0, 0.744359, 0.507301, 0.424625, 0.671925], [0, 0.319734, 0.618128, 0.454844, 0.741569], [4, 0.538492, 0.462165, 0.069172, 0.073159]]}, {"filename": "000000177934.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.903078, 0.390385, 0.072906, 0.082062], [1, 0.525438, 0.417073, 0.117344, 0.125396], [1, 0.6955, 0.382531, 0.062531, 0.083979], [1, 0.791422, 0.40525, 0.152688, 0.154125], [1, 0.513945, 0.378042, 0.076703, 0.062875], [1, 0.310953, 0.58651, 0.402813, 0.355063], [1, 0.326063, 0.480125, 0.081813, 0.187583], [0, 0.592094, 0.513521, 0.081813, 0.383375], [0, 0.389953, 0.460667, 0.068594, 0.2355], [0, 0.663836, 0.454656, 0.065859, 0.292104], [0, 0.776969, 0.447187, 0.1045, 0.265167], [0, 0.331094, 0.380146, 0.070031, 0.157833], [1, 0.20068, 0.346938, 0.401359, 0.267583], [1, 0.457547, 0.359354, 0.113063, 0.1005], [0, 0.627258, 0.390937, 0.029516, 0.153583], [1, 0.883305, 0.370719, 0.065609, 0.071396], [1, 0.67368, 0.332333, 0.027984, 0.026]]}, {"filename": "000000179174.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.863248, 0.227008, 0.042196, 0.097266], [0, 0.752325, 0.100883, 0.061238, 0.102922], [0, 0.801203, 0.221422, 0.03993, 0.079125], [0, 0.818633, 0.203062, 0.028201, 0.045875], [0, 0.983621, 0.223422, 0.032757, 0.087], [0, 0.942407, 0.229617, 0.059206, 0.110203], [0, 0.331098, 0.198922, 0.036682, 0.042062], [0, 0.758259, 0.19518, 0.063575, 0.089516]]}, {"filename": "000000179265.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.566148, 0.482031, 0.237828, 0.258427], [1, 0.397945, 0.414894, 0.097297, 0.05885], [1, 0.766641, 0.411021, 0.089344, 0.081948], [0, 0.614945, 0.258779, 0.009641, 0.033615], [7, 0.448992, 0.398908, 0.010328, 0.038803], [3, 0.730781, 0.528333, 0.050188, 0.045164], [3, 0.691703, 0.519319, 0.014656, 0.06277], [0, 0.918469, 0.385399, 0.008125, 0.025493]]}, {"filename": "000000179765.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.545742, 0.411115, 0.797141, 0.794771]]}, {"filename": "000000179898.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.740773, 0.458823, 0.132547, 0.308729], [0, 0.633336, 0.357062, 0.140672, 0.226375], [0, 0.869375, 0.411323, 0.094562, 0.262646], [0, 0.223977, 0.481885, 0.116391, 0.152771], [0, 0.099437, 0.685396, 0.198875, 0.530333], [0, 0.869547, 0.591208, 0.260906, 0.423458], [0, 0.887055, 0.80976, 0.216109, 0.380479], [0, 0.04793, 0.838823, 0.095859, 0.300438], [4, 0.254336, 0.388813, 0.134047, 0.133708], [4, 0.108672, 0.273115, 0.099125, 0.052979], [4, 0.066547, 0.491802, 0.130438, 0.213896], [0, 0.450047, 0.521406, 0.608625, 0.957187], [4, 0.044781, 0.252125, 0.082, 0.131458], [4, 0.233766, 0.254542, 0.070125, 0.083125]]}, {"filename": "000000180765.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.872578, 0.596771, 0.254844, 0.806458], [0, 0.886672, 0.150958, 0.106188, 0.209625], [0, 0.482609, 0.138615, 0.053937, 0.115646], [1, 0.873273, 0.594073, 0.253453, 0.800521], [0, 0.951219, 0.110396, 0.097563, 0.220792], [0, 0.812883, 0.419854, 0.245922, 0.170167]]}, {"filename": "000000181753.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.273281, 0.783853, 0.404344, 0.428416], [4, 0.902531, 0.821749, 0.193781, 0.356501]]}, {"filename": "000000181816.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.713922, 0.250771, 0.565438, 0.237125], [3, 0.520555, 0.649323, 0.676672, 0.415896], [1, 0.449711, 0.144, 0.083453, 0.052583], [0, 0.027492, 0.118531, 0.030234, 0.122479], [0, 0.383469, 0.564969, 0.199156, 0.810729], [0, 0.191516, 0.099927, 0.021281, 0.083937]]}, {"filename": "000000182551.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.250763, 0.321891, 0.036362, 0.072625], [1, 0.405106, 0.323344, 0.112371, 0.056688], [1, 0.135469, 0.32832, 0.208404, 0.063266], [1, 0.829167, 0.319422, 0.070822, 0.042281], [1, 0.868204, 0.315469, 0.118099, 0.043281], [1, 0.653545, 0.315398, 0.136948, 0.060609], [1, 0.271702, 0.318492, 0.18439, 0.072734], [1, 0.751115, 0.314984, 0.103122, 0.059063], [1, 0.019049, 0.3305, 0.038099, 0.068625], [1, 0.914918, 0.313203, 0.089366, 0.047375], [1, 0.508427, 0.315266, 0.186244, 0.067844], [5, 0.666995, 0.691891, 0.149671, 0.307844]]}, {"filename": "000000183391.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.139509, 0.208836, 0.05481, 0.113453], [0, 0.150063, 0.227422, 0.040823, 0.078625], [0, 0.091147, 0.337797, 0.079066, 0.105187], [0, 0.213647, 0.340398, 0.076978, 0.081953], [0, 0.509747, 0.435305, 0.076266, 0.123234], [0, 0.416013, 0.382, 0.068734, 0.091], [0, 0.398877, 0.351352, 0.063956, 0.099641], [0, 0.376891, 0.306945, 0.071946, 0.107172], [0, 0.366179, 0.263523, 0.064984, 0.087984], [0, 0.430206, 0.541289, 0.082405, 0.093141], [0, 0.447255, 0.447187, 0.066503, 0.132906], [4, 0.523267, 0.509297, 0.069351, 0.048906], [4, 0.243489, 0.503367, 0.059161, 0.040453], [4, 0.176788, 0.502875, 0.073228, 0.039562], [4, 0.256353, 0.555375, 0.067991, 0.047469], [4, 0.197532, 0.554008, 0.044177, 0.038203], [4, 0.148608, 0.369742, 0.043449, 0.044578], [4, 0.137136, 0.323297, 0.042057, 0.033344], [4, 0.536646, 0.557031, 0.048354, 0.04775], [4, 0.096574, 0.518117, 0.020839, 0.010672], [4, 0.024351, 0.406305, 0.048703, 0.035766], [4, 0.141828, 0.407594, 0.045997, 0.032438], [4, 0.167081, 0.554203, 0.022547, 0.039844], [0, 0.167611, 0.429992, 0.080253, 0.077891], [4, 0.229604, 0.473555, 0.064652, 0.075172], [0, 0.55288, 0.551719, 0.439525, 0.752375]]}, {"filename": "000000184313.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.035758, 0.659927, 0.071516, 0.143104], [0, 0.541977, 0.534427, 0.057766, 0.101771], [0, 0.27368, 0.563156, 0.014453, 0.052438], [1, 0.035305, 0.65351, 0.070609, 0.147687], [5, 0.117687, 0.620146, 0.019562, 0.050917], [0, 0.456297, 0.550115, 0.034531, 0.046604], [0, 0.443047, 0.540979, 0.008719, 0.016792]]}, {"filename": "000000185373.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.397539, 0.244167, 0.048422, 0.037042], [1, 0.506383, 0.252885, 0.120734, 0.105062], [1, 0.819102, 0.475281, 0.357297, 0.384271], [1, 0.523844, 0.402448, 0.303312, 0.244896], [1, 0.406719, 0.370958, 0.213844, 0.198125], [1, 0.289609, 0.366125, 0.063969, 0.122542], [1, 0.20425, 0.323427, 0.128312, 0.163771], [1, 0.185313, 0.334594, 0.123969, 0.110854], [1, 0.075789, 0.255135, 0.049172, 0.045354], [1, 0.196891, 0.264281, 0.082375, 0.059146], [1, 0.338258, 0.238937, 0.069266, 0.049833], [1, 0.151719, 0.242625, 0.029469, 0.030292], [1, 0.77893, 0.273031, 0.442141, 0.253937], [7, 0.435539, 0.266906, 0.027422, 0.075312], [1, 0.968391, 0.256031, 0.063219, 0.060313], [1, 0.049516, 0.25, 0.018031, 0.022417], [1, 0.201422, 0.235687, 0.053719, 0.031292], [1, 0.372539, 0.290635, 0.083266, 0.093562]]}, {"filename": "000000185472.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.714703, 0.634823, 0.098125, 0.265229], [0, 0.780336, 0.537073, 0.165172, 0.417979], [0, 0.656484, 0.339552, 0.031813, 0.025021], [0, 0.959453, 0.464594, 0.081094, 0.349896], [1, 0.983914, 0.684135, 0.032172, 0.061021], [0, 0.964813, 0.854313, 0.070375, 0.256792]]}, {"filename": "000000185807.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.391852, 0.556187, 0.517422, 0.865167], [0, 0.482336, 0.427615, 0.327109, 0.553479], [0, 0.816555, 0.648052, 0.366891, 0.672729]]}, {"filename": "000000186026.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.606977, 0.604727, 0.236484, 0.522516]]}, {"filename": "000000186109.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.64843, 0.626972, 0.360484, 0.746056], [0, 0.413164, 0.439754, 0.104766, 0.384484], [0, 0.365055, 0.529977, 0.113172, 0.516479], [0, 0.495727, 0.455387, 0.117297, 0.420023], [0, 0.660344, 0.342946, 0.072125, 0.156221], [0, 0.098102, 0.478685, 0.153953, 0.603192], [0, 0.187391, 0.359237, 0.065969, 0.231995], [0, 0.219883, 0.658862, 0.320109, 0.65392], [0, 0.358242, 0.679777, 0.315609, 0.613498], [0, 0.848781, 0.416444, 0.061219, 0.274671], [1, 0.253422, 0.301362, 0.039969, 0.04784], [0, 0.024156, 0.524319, 0.047688, 0.472535], [0, 0.768938, 0.504941, 0.150781, 0.537019], [1, 0.873406, 0.322394, 0.165375, 0.103099], [0, 0.912977, 0.424178, 0.065078, 0.271221]]}, {"filename": "000000186599.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.870539, 0.057565, 0.045141, 0.064871], [1, 0.493945, 0.367412, 0.892391, 0.546071], [1, 0.682336, 0.165847, 0.035453, 0.034565], [1, 0.273281, 0.182612, 0.036906, 0.063435], [1, 0.846156, 0.164059, 0.0365, 0.029341], [1, 0.715633, 0.164976, 0.038016, 0.041435]]}, {"filename": "000000186906.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.359508, 0.619344, 0.373109, 0.350771], [1, 0.564195, 0.468375, 0.090641, 0.135333], [1, 0.113227, 0.611562, 0.225797, 0.297542], [1, 0.518563, 0.417625, 0.05275, 0.070333], [1, 0.125992, 0.656396, 0.177984, 0.289542], [1, 0.341289, 0.415729, 0.291578, 0.292125], [0, 0.761289, 0.57676, 0.477422, 0.819354], [0, 0.599562, 0.434104, 0.020219, 0.035958], [0, 0.297375, 0.458333, 0.069937, 0.0975], [1, 0.497164, 0.431677, 0.031109, 0.069354], [1, 0.591672, 0.39325, 0.031062, 0.011583], [1, 0.552859, 0.388677, 0.051406, 0.034062], [1, 0.144125, 0.409927, 0.157406, 0.201896], [0, 0.051844, 0.453146, 0.024063, 0.032667], [0, 0.111734, 0.565427, 0.133187, 0.277771], [1, 0.580344, 0.369719, 0.015844, 0.015187]]}, {"filename": "000000187144.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.323594, 0.524031, 0.640437, 0.546063], [1, 0.927, 0.491583, 0.146, 0.071625], [0, 0.641805, 0.597552, 0.041047, 0.160479], [0, 0.666195, 0.58676, 0.039203, 0.154229], [0, 0.599008, 0.604448, 0.050016, 0.167063], [1, 0.728078, 0.49325, 0.175313, 0.065208], [0, 0.72175, 0.57574, 0.039594, 0.180937], [0, 0.699789, 0.564531, 0.030953, 0.111271], [0, 0.763687, 0.563625, 0.026687, 0.10525], [0, 0.676391, 0.581073, 0.025969, 0.127813], [0, 0.850633, 0.531312, 0.016453, 0.04775], [0, 0.646844, 0.527062, 0.012188, 0.039125]]}, {"filename": "000000187236.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.067031, 0.345729, 0.118531, 0.266833]]}, {"filename": "000000187249.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.399703, 0.460667, 0.37225, 0.395167], [0, 0.563766, 0.465917, 0.291563, 0.40225]]}, {"filename": "000000187427.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.118708, 0.490602, 0.237083, 0.221141], [0, 0.557042, 0.552, 0.545194, 0.869344], [4, 0.218958, 0.551023, 0.080361, 0.131016], [4, 0.138333, 0.777648, 0.269333, 0.390047]]}, {"filename": "000000188009.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.504875, 0.603396, 0.136125, 0.484309], [0, 0.782891, 0.447073, 0.258469, 0.873864], [0, 0.70518, 0.562049, 0.104141, 0.193185], [0, 0.897445, 0.637307, 0.075609, 0.499906], [0, 0.043477, 0.669672, 0.086953, 0.507869], [1, 0.290523, 0.765445, 0.168422, 0.124988], [0, 0.631523, 0.715082, 0.110891, 0.243372], [0, 0.141461, 0.817834, 0.079078, 0.185129], [0, 0.95057, 0.636745, 0.095141, 0.476721], [1, 0.186305, 0.77493, 0.091578, 0.102927]]}, {"filename": "000000188296.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.094086, 0.555805, 0.183891, 0.164056], [1, 0.601367, 0.45156, 0.050641, 0.050246], [0, 0.25207, 0.580411, 0.248828, 0.49642], [0, 0.431242, 0.434154, 0.271578, 0.681117], [0, 0.496828, 0.379294, 0.185625, 0.649097], [0, 0.81257, 0.603366, 0.265141, 0.748309], [0, 0.81132, 0.315599, 0.042453, 0.039278], [0, 0.943297, 0.524869, 0.113406, 0.477061], [1, 0.605438, 0.420057, 0.08, 0.041396], [1, 0.327711, 0.532323, 0.086109, 0.077947], [1, 0.624734, 0.510131, 0.173594, 0.24775], [0, 0.690453, 0.583473, 0.11975, 0.356962]]}, {"filename": "000000188439.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.37618, 0.722966, 0.514016, 0.421176], [0, 0.642539, 0.664779, 0.018703, 0.077255]]}, {"filename": "000000188992.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.050977, 0.39818, 0.083496, 0.152453], [0, 0.12581, 0.414227, 0.086889, 0.157484], [0, 0.012326, 0.596383, 0.024653, 0.118703]]}, {"filename": "000000189509.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.510865, 0.244945, 0.422479, 0.314609], [0, 0.446937, 0.720859, 0.391625, 0.526906]]}, {"filename": "000000189752.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.21368, 0.112615, 0.116547, 0.220729], [0, 0.075969, 0.11899, 0.151188, 0.237604], [0, 0.610641, 0.251125, 0.775344, 0.5], [4, 0.166273, 0.486125, 0.332547, 0.081458]]}, {"filename": "000000189820.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.770492, 0.640867, 0.393141, 0.649461]]}, {"filename": "000000189903.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.295135, 0.140453, 0.590271, 0.276406]]}, {"filename": "000000190007.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.443375, 0.530459, 0.117906, 0.249265]]}, {"filename": "000000190187.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.253323, 0.546133, 0.150854, 0.135859], [4, 0.416344, 0.51107, 0.078438, 0.097547], [4, 0.277844, 0.485898, 0.067896, 0.045797]]}, {"filename": "000000190753.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.257117, 0.674623, 0.025141, 0.084142], [4, 0.350328, 0.632929, 0.021281, 0.048243], [0, 0.302648, 0.6875, 0.083359, 0.347427], [0, 0.385188, 0.592887, 0.068813, 0.116862], [0, 0.450719, 0.657751, 0.141656, 0.253452], [0, 0.568969, 0.587646, 0.057062, 0.128096], [0, 0.580383, 0.518504, 0.053078, 0.089142], [0, 0.812398, 0.488033, 0.030172, 0.057071], [0, 0.935383, 0.475837, 0.025609, 0.040209], [3, 0.154805, 0.596946, 0.123203, 0.096527], [1, 0.712516, 0.583494, 0.053344, 0.091632], [3, 0.526188, 0.746433, 0.323094, 0.325084], [3, 0.750969, 0.627155, 0.144813, 0.18431], [3, 0.522523, 0.541726, 0.063141, 0.057427], [3, 0.811844, 0.500251, 0.04375, 0.034226], [3, 0.92507, 0.484498, 0.024641, 0.025272], [3, 0.988328, 0.473891, 0.016156, 0.017155], [3, 0.248227, 0.591998, 0.102609, 0.093912]]}, {"filename": "000000190923.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.937347, 0.79688, 0.125307, 0.40624], [0, 0.044947, 0.61348, 0.089893, 0.3236], [0, 0.8616, 0.62081, 0.027253, 0.06458], [0, 0.820773, 0.61353, 0.023253, 0.0671], [0, 0.80372, 0.6144, 0.020987, 0.07068], [0, 0.78976, 0.60928, 0.018133, 0.07052], [0, 0.901667, 0.62451, 0.036347, 0.06386], [0, 0.58124, 0.57327, 0.01912, 0.05534], [0, 0.560373, 0.56975, 0.014773, 0.0511], [0, 0.735693, 0.59571, 0.017467, 0.06486], [0, 0.54972, 0.56107, 0.018107, 0.04498], [1, 0.083653, 0.45141, 0.111627, 0.11782], [0, 0.482053, 0.54513, 0.014293, 0.03602], [0, 0.44644, 0.53918, 0.016453, 0.04444]]}, {"filename": "000000190981.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.102055, 0.966396, 0.018109, 0.026125]]}, {"filename": "000000190998.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.543492, 0.504485, 0.181422, 0.35281], [0, 0.51743, 0.40918, 0.110422, 0.381546], [0, 0.397336, 0.881803, 0.175328, 0.236393]]}, {"filename": "000000191456.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.31857, 0.353612, 0.260234, 0.587835], [6, 0.024969, 0.236847, 0.049938, 0.113506]]}, {"filename": "000000191471.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.493242, 0.462833, 0.611484, 0.828833]]}, {"filename": "000000191913.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.657301, 0.5695, 0.074042, 0.1145], [0, 0.67097, 0.491266, 0.110864, 0.19025], [7, 0.292617, 0.525273, 0.044533, 0.064641], [7, 0.53028, 0.283477, 0.036916, 0.050234], [7, 0.673458, 0.1375, 0.029439, 0.043094], [7, 0.784556, 0.030766, 0.022243, 0.025844]]}, {"filename": "000000192607.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.368594, 0.732212, 0.731219, 0.476424], [0, 0.761859, 0.713894, 0.096969, 0.374141]]}, {"filename": "000000192670.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.128437, 0.364157, 0.254594, 0.17963], [1, 0.483859, 0.314954, 0.279906, 0.190185], [0, 0.687891, 0.559099, 0.227594, 0.55963], [0, 0.893, 0.357887, 0.065375, 0.253695], [0, 0.864633, 0.298314, 0.031109, 0.129007], [0, 0.941891, 0.352621, 0.057531, 0.263626], [0, 0.054508, 0.3847, 0.076516, 0.197945], [0, 0.969758, 0.340866, 0.058859, 0.284088], [0, 0.819703, 0.369781, 0.052656, 0.213972], [4, 0.162609, 0.424353, 0.076875, 0.134942], [4, 0.080594, 0.436005, 0.02625, 0.106005], [0, 0.28468, 0.720023, 0.230859, 0.368868], [0, 0.026906, 0.42455, 0.053812, 0.13358], [1, 0.684492, 0.291212, 0.244547, 0.153972], [1, 0.307523, 0.304792, 0.095703, 0.141455], [1, 0.169125, 0.300266, 0.164938, 0.130139], [1, 0.883578, 0.206755, 0.066938, 0.041178]]}, {"filename": "000000192699.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.825742, 0.811236, 0.348516, 0.350569], [0, 0.246453, 0.71291, 0.492906, 0.547484], [1, 0.31243, 0.914398, 0.326859, 0.171204], [0, 0.611789, 0.545481, 0.416547, 0.909037]]}, {"filename": "000000192964.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.893236, 0.829844, 0.213528, 0.088562], [0, 0.442325, 0.322914, 0.453341, 0.456641]]}, {"filename": "000000193166.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.679211, 0.525844, 0.455047, 0.485396], [7, 0.305055, 0.498875, 0.461797, 0.99325], [1, 0.92768, 0.244281, 0.029703, 0.022146]]}, {"filename": "000000193213.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.25434, 0.12892, 0.21028, 0.256827], [0, 0.06054, 0.370973, 0.11772, 0.302693], [0, 0.77612, 0.108987, 0.09944, 0.217973], [0, 0.29773, 0.467533, 0.20314, 0.427547], [0, 0.93567, 0.153933, 0.12866, 0.298853], [0, 0.13745, 0.354293, 0.12994, 0.447467], [0, 0.77872, 0.277027, 0.10812, 0.286053], [0, 0.95136, 0.606307, 0.09728, 0.604933], [3, 0.50056, 0.394373, 0.99776, 0.45168], [4, 0.29753, 0.513747, 0.21938, 0.316347], [0, 0.68889, 0.538267, 0.3963, 0.849387]]}, {"filename": "000000193579.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.30093, 0.83726, 0.076984, 0.268271], [0, 0.170633, 0.787062, 0.050922, 0.148125], [5, 0.27525, 0.954885, 0.061375, 0.090229], [5, 0.193281, 0.917604, 0.06375, 0.150208], [0, 0.338023, 0.873719, 0.071172, 0.246104], [0, 0.079953, 0.773562, 0.062781, 0.162917], [0, 0.063734, 0.721479, 0.032062, 0.050667]]}, {"filename": "000000193717.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.619376, 0.334578, 0.121286, 0.037219], [1, 0.726219, 0.335125, 0.093436, 0.045312], [5, 0.422351, 0.501875, 0.695643, 0.905625], [0, 0.865998, 0.331031, 0.195835, 0.29125], [1, 0.039299, 0.309148, 0.078599, 0.075359], [1, 0.957322, 0.352203, 0.085355, 0.029594], [1, 0.200643, 0.322602, 0.084012, 0.028234]]}, {"filename": "000000194346.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.486742, 0.6535, 0.198078, 0.361167], [0, 0.816578, 0.127135, 0.045844, 0.162688], [0, 0.323664, 0.394333, 0.064734, 0.123042], [0, 0.243367, 0.423312, 0.097297, 0.136167], [0, 0.67332, 0.072667, 0.038766, 0.069667], [0, 0.036492, 0.125104, 0.036453, 0.054], [0, 0.411234, 0.132052, 0.033188, 0.103271], [0, 0.877703, 0.060083, 0.030312, 0.095208], [4, 0.182711, 0.125271, 0.022891, 0.02625], [0, 0.031086, 0.931417, 0.060984, 0.134875], [3, 0.042563, 0.450917, 0.083906, 0.124208], [3, 0.303102, 0.341385, 0.033453, 0.068687], [0, 0.378852, 0.131021, 0.047078, 0.125542], [0, 0.238102, 0.150896, 0.096797, 0.200042]]}, {"filename": "000000194461.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.545938, 0.655594, 0.083563, 0.051531], [0, 0.108109, 0.68468, 0.040531, 0.105859], [1, 0.736086, 0.695078, 0.025297, 0.052531], [0, 0.266, 0.653383, 0.016219, 0.052016], [1, 0.425813, 0.673492, 0.059188, 0.075328], [0, 0.699211, 0.657344, 0.012391, 0.060125], [0, 0.688805, 0.652586, 0.016703, 0.047641]]}, {"filename": "000000194518.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.346945, 0.655199, 0.080859, 0.080796], [0, 0.422453, 0.717026, 0.062969, 0.200562], [3, 0.411641, 0.757248, 0.228812, 0.150562], [0, 0.485797, 0.719555, 0.081, 0.211475]]}, {"filename": "000000194724.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.898312, 0.192135, 0.198875, 0.208979], [4, 0.027875, 0.318698, 0.05575, 0.234229]]}, {"filename": "000000194907.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.087477, 0.466708, 0.110016, 0.431708], [0, 0.172906, 0.391771, 0.230531, 0.41125], [0, 0.320805, 0.512073, 0.273766, 0.675646], [0, 0.528031, 0.660312, 0.3565, 0.656958], [0, 0.776578, 0.585219, 0.446844, 0.822229], [0, 0.708508, 0.452813, 0.134141, 0.382458], [1, 0.65918, 0.383302, 0.092109, 0.079312]]}, {"filename": "000000195842.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.765422, 0.542854, 0.235438, 0.887333], [4, 0.955656, 0.762927, 0.085688, 0.219687]]}, {"filename": "000000197300.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.529211, 0.466292, 0.532578, 0.5865]]}, {"filename": "000000197870.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.846328, 0.383565, 0.114313, 0.435506], [3, 0.396344, 0.568694, 0.789687, 0.522518]]}, {"filename": "000000198426.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.529008, 0.516337, 0.076922, 0.266825], [4, 0.62243, 0.59305, 0.145578, 0.223705], [0, 0.788031, 0.497758, 0.423937, 0.982033], [0, 0.303922, 0.207019, 0.012344, 0.041448], [4, 0.05682, 0.7039, 0.099078, 0.109081], [4, 0.259539, 0.283593, 0.039516, 0.11805]]}, {"filename": "000000198441.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.082305, 0.880468, 0.116078, 0.145527], [0, 0.539281, 0.779403, 0.540031, 0.42377]]}, {"filename": "000000198489.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.72236, 0.147633, 0.55528, 0.172422], [0, 0.445386, 0.40968, 0.401986, 0.531172]]}, {"filename": "000000198928.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.187698, 0.599516, 0.029354, 0.005187], [0, 0.577177, 0.764195, 0.128271, 0.284109], [1, 0.241208, 0.596484, 0.051083, 0.010625], [1, 0.345479, 0.589094, 0.167667, 0.027813]]}, {"filename": "000000198947.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.841219, 0.33118, 0.317563, 0.240859], [1, 0.680146, 0.132203, 0.467417, 0.175281], [5, 0.205719, 0.423289, 0.109687, 0.162047], [3, 0.564271, 0.843313, 0.871458, 0.295969], [1, 0.23301, 0.144508, 0.465104, 0.206016]]}, {"filename": "000000198972.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.666578, 0.544942, 0.359313, 0.255291], [0, 0.695164, 0.505618, 0.191297, 0.332587]]}, {"filename": "000000200611.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.761391, 0.555189, 0.121156, 0.44967], [0, 0.586742, 0.578479, 0.169328, 0.417052], [0, 0.488312, 0.541191, 0.339437, 0.521344], [0, 0.309117, 0.501344, 0.107922, 0.371745], [0, 0.723391, 0.339292, 0.015906, 0.081981], [0, 0.650594, 0.422476, 0.041688, 0.188774], [0, 0.194422, 0.454988, 0.055531, 0.110495], [4, 0.212477, 0.475259, 0.029953, 0.068726], [4, 0.179523, 0.444917, 0.016109, 0.032665], [4, 0.144391, 0.474458, 0.041312, 0.068774], [4, 0.043352, 0.476144, 0.047391, 0.06592], [4, 0.407164, 0.463974, 0.045141, 0.08941], [0, 0.083273, 0.455142, 0.042672, 0.10967], [0, 0.158344, 0.446156, 0.040188, 0.075943], [0, 0.127328, 0.44138, 0.050687, 0.089033], [4, 0.092117, 0.48237, 0.021891, 0.055684], [4, 0.038641, 0.445413, 0.021063, 0.026344], [0, 0.704242, 0.472972, 0.025203, 0.088019], [4, 0.081437, 0.49421, 0.012062, 0.028844], [0, 0.015016, 0.432807, 0.0295, 0.087972], [0, 0.774664, 0.386262, 0.026859, 0.052524]]}, {"filename": "000000200749.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.234227, 0.552667, 0.111859, 0.116125], [1, 0.499398, 0.543563, 0.131859, 0.172583], [1, 0.772789, 0.542687, 0.060484, 0.084667], [1, 0.903609, 0.544865, 0.192781, 0.232604], [1, 0.651555, 0.535458, 0.040672, 0.056042], [1, 0.616211, 0.534563, 0.034109, 0.050667], [1, 0.730172, 0.527781, 0.038063, 0.048771], [1, 0.699031, 0.528646, 0.032969, 0.043042], [1, 0.676898, 0.529604, 0.020859, 0.034833], [1, 0.571773, 0.534062, 0.017891, 0.03875], [1, 0.356234, 0.541969, 0.058438, 0.059188], [1, 0.093195, 0.552323, 0.186391, 0.162354], [1, 0.58993, 0.534, 0.021734, 0.045125]]}, {"filename": "000000200839.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.433148, 0.50674, 0.589891, 0.312354], [0, 0.026242, 0.527531, 0.045328, 0.120854]]}, {"filename": "000000201236.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.34061, 0.533836, 0.116338, 0.086484], [0, 0.452394, 0.617297, 0.263333, 0.548312], [0, 0.665951, 0.545656, 0.061385, 0.096219], [0, 0.634894, 0.537289, 0.186643, 0.114297], [0, 0.112488, 0.543023, 0.132817, 0.104547], [4, 0.716749, 0.687289, 0.151526, 0.125266]]}, {"filename": "000000202070.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.806461, 0.486958, 0.112922, 0.274167], [0, 0.598531, 0.403896, 0.039562, 0.077667], [0, 0.647016, 0.3355, 0.035406, 0.104125], [0, 0.588664, 0.3385, 0.029047, 0.088542], [0, 0.549594, 0.319927, 0.028531, 0.098104], [1, 0.470961, 0.299135, 0.046984, 0.073188], [0, 0.566414, 0.405948, 0.028234, 0.054271], [0, 0.537594, 0.397302, 0.039313, 0.078604]]}, {"filename": "000000202582.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.942445, 0.460094, 0.042234, 0.069396], [0, 0.829883, 0.445229, 0.013766, 0.0175], [0, 0.813492, 0.442479, 0.016234, 0.050375], [0, 0.869469, 0.457719, 0.026031, 0.040187], [0, 0.854945, 0.454583, 0.005016, 0.021458], [0, 0.786031, 0.444927, 0.007844, 0.016646], [4, 0.874812, 0.468156, 0.014281, 0.019687], [4, 0.955656, 0.465135, 0.027906, 0.049313], [0, 0.998703, 0.440385, 0.002594, 0.015312], [0, 0.932672, 0.427021, 0.010187, 0.0175], [0, 0.901328, 0.423792, 0.006687, 0.008917], [0, 0.927812, 0.434927, 0.005125, 0.004021]]}, {"filename": "000000203580.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.69175, 0.638279, 0.340406, 0.249016]]}, {"filename": "000000203986.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.740833, 0.563695, 0.064542, 0.146484], [0, 0.701323, 0.49632, 0.053854, 0.082109], [0, 0.752687, 0.482547, 0.03975, 0.056625], [0, 0.836552, 0.49107, 0.045271, 0.072891], [0, 0.464948, 0.471281, 0.101729, 0.075312], [0, 0.598458, 0.464445, 0.071583, 0.077609], [0, 0.32074, 0.395352, 0.045479, 0.052266], [0, 0.615958, 0.389672, 0.045042, 0.059688], [0, 0.59275, 0.41925, 0.057792, 0.055375], [0, 0.576896, 0.389836, 0.031333, 0.029984], [3, 0.671583, 0.452641, 0.103625, 0.037469], [0, 0.972604, 0.504703, 0.054792, 0.071938], [0, 0.188833, 0.684086, 0.377667, 0.557828]]}, {"filename": "000000204423.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.593133, 0.689013, 0.014953, 0.031373], [0, 0.065406, 0.6447, 0.009344, 0.017468], [0, 0.518461, 0.683069, 0.013516, 0.025279], [0, 0.553, 0.68382, 0.016188, 0.025579], [0, 0.680461, 0.681384, 0.013516, 0.019335], [0, 0.756188, 0.684292, 0.013438, 0.02073], [0, 0.786188, 0.595483, 0.01075, 0.021738], [0, 0.157891, 0.629861, 0.006844, 0.013498], [0, 0.57793, 0.684903, 0.012797, 0.022425], [0, 0.464695, 0.684635, 0.010422, 0.025365], [0, 0.503719, 0.682393, 0.007719, 0.023069], [0, 0.538453, 0.683004, 0.013844, 0.026738], [0, 0.481359, 0.682811, 0.014031, 0.020558]]}, {"filename": "000000204700.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.29082, 0.521708, 0.010672, 0.03175], [0, 0.303125, 0.51976, 0.008781, 0.027104], [3, 0.97957, 0.538667, 0.040859, 0.035333], [0, 0.565367, 0.57099, 0.046797, 0.069771]]}, {"filename": "000000204757.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.538286, 0.587271, 0.923429, 0.825457]]}, {"filename": "000000204871.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.67607, 0.248873, 0.137696, 0.13317], [1, 0.951413, 0.247214, 0.097173, 0.109788], [1, 0.773513, 0.25549, 0.118007, 0.090392], [0, 0.328056, 0.240915, 0.040098, 0.140359], [1, 0.822492, 0.219984, 0.169199, 0.123235], [0, 0.295482, 0.260008, 0.017042, 0.040637], [0, 0.787859, 0.226846, 0.017255, 0.021405], [1, 0.953194, 0.185114, 0.09067, 0.065098], [5, 0.536364, 0.530425, 0.672729, 0.907974]]}, {"filename": "000000205915.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.764406, 0.551167, 0.012594, 0.015118], [1, 0.780117, 0.501863, 0.010359, 0.016226]]}, {"filename": "000000206391.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.28061, 0.304111, 0.23182, 0.332128], [1, 0.76646, 0.651939, 0.15904, 0.067784]]}, {"filename": "000000207941.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.85643, 0.336561, 0.287141, 0.673122]]}, {"filename": "000000208824.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.5, 0.380627, 1.0, 0.761253], [7, 0.54404, 0.550467, 0.39412, 0.899067]]}, {"filename": "000000209290.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.97424, 0.567786, 0.05152, 0.204853], [0, 0.586569, 0.398162, 0.509673, 0.579069], [4, 0.610212, 0.467917, 0.602614, 0.384886]]}, {"filename": "000000210099.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.507992, 0.500344, 0.903891, 0.758457]]}, {"filename": "000000210782.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.054773, 0.480896, 0.106172, 0.125833], [0, 0.581734, 0.569656, 0.128, 0.327562], [0, 0.210672, 0.48501, 0.067406, 0.116854], [0, 0.140516, 0.497177, 0.084656, 0.160271], [3, 0.857023, 0.705625, 0.284828, 0.368542], [0, 0.623984, 0.563708, 0.116312, 0.288167], [0, 0.98068, 0.577052, 0.038641, 0.459979], [0, 0.935172, 0.346667, 0.129656, 0.15375], [0, 0.440477, 0.522302, 0.021703, 0.111937], [0, 0.489148, 0.455542, 0.104297, 0.32675]]}, {"filename": "000000211653.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.78989, 0.523227, 0.42022, 0.921333], [7, 0.37837, 0.657453, 0.31854, 0.658427], [7, 0.10758, 0.69, 0.21372, 0.602133], [7, 0.16601, 0.397747, 0.26122, 0.593253], [7, 0.04298, 0.267413, 0.08596, 0.467413]]}, {"filename": "000000212226.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.62935, 0.529209, 0.31618, 0.555075], [1, 0.4729, 0.495836, 0.72696, 0.986776]]}, {"filename": "000000212941.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.193617, 0.724028, 0.387234, 0.42728], [1, 0.653719, 0.595505, 0.541063, 0.375829], [1, 0.884148, 0.451036, 0.231703, 0.249637], [7, 0.585508, 0.371218, 0.029828, 0.132591], [1, 0.063859, 0.849417, 0.127719, 0.301166]]}, {"filename": "000000213593.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.303781, 0.358157, 0.196562, 0.304249], [1, 0.66893, 0.967077, 0.110828, 0.065845], [1, 0.370797, 0.94304, 0.116219, 0.11392], [1, 0.713172, 0.956432, 0.081062, 0.083944], [1, 0.452039, 0.947946, 0.092078, 0.098052], [0, 0.835828, 0.946103, 0.047375, 0.107793], [0, 0.869172, 0.943838, 0.025375, 0.112324], [1, 0.522117, 0.956819, 0.061141, 0.075657], [1, 0.218141, 0.966854, 0.146625, 0.066291]]}, {"filename": "000000213623.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.380563, 0.54966, 0.035594, 0.096792]]}, {"filename": "000000214192.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.641625, 0.769897, 0.182875, 0.246078], [0, 0.802195, 0.275287, 0.195953, 0.469656], [0, 0.537602, 0.597064, 0.342578, 0.499679], [1, 0.88225, 0.387385, 0.202531, 0.231972]]}, {"filename": "000000214295.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.298397, 0.578086, 0.309282, 0.101328], [1, 0.720825, 0.582719, 0.220311, 0.083406], [7, 0.529868, 0.64382, 0.137632, 0.361797]]}, {"filename": "000000215072.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.146609, 0.73867, 0.148187, 0.416098], [4, 0.526133, 0.760133, 0.249172, 0.459468], [0, 0.480109, 0.601353, 0.265062, 0.385144], [0, 0.281242, 0.587705, 0.202234, 0.515676]]}, {"filename": "000000215374.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.794871, 0.808969, 0.071054, 0.117938], [4, 0.400468, 0.817164, 0.19363, 0.145672], [4, 0.537424, 0.760656, 0.076862, 0.071125], [0, 0.285515, 0.688258, 0.022787, 0.046453], [0, 0.619133, 0.695516, 0.028806, 0.067469], [0, 0.312014, 0.686047, 0.017424, 0.031625], [4, 0.604227, 0.731117, 0.058056, 0.028172], [4, 0.482799, 0.739609, 0.043021, 0.037688], [0, 0.464098, 0.677055, 0.016628, 0.036047], [0, 0.572857, 0.674984, 0.012646, 0.037781], [0, 0.64733, 0.696305, 0.025574, 0.081172], [0, 0.479824, 0.675555, 0.017775, 0.040422], [0, 0.259918, 0.707602, 0.045316, 0.034359], [0, 0.77685, 0.680844, 0.03089, 0.039031]]}, {"filename": "000000215600.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.1441, 0.131467, 0.17696, 0.24944], [7, 0.50253, 0.49664, 0.99494, 0.984267], [1, 0.9489, 0.013907, 0.06208, 0.02728]]}, {"filename": "000000215616.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.950617, 0.715779, 0.073422, 0.126017], [0, 0.437445, 0.548485, 0.069766, 0.232944], [0, 0.458367, 0.668139, 0.124984, 0.396147], [0, 0.149383, 0.619903, 0.098516, 0.420584], [0, 0.777727, 0.648745, 0.073672, 0.384329], [0, 0.703914, 0.726201, 0.156828, 0.547597], [0, 0.329641, 0.43092, 0.104313, 0.372835], [0, 0.823813, 0.606126, 0.05225, 0.273766], [0, 0.851273, 0.601548, 0.039141, 0.17145], [0, 0.849094, 0.520097, 0.016437, 0.04119], [4, 0.895844, 0.70947, 0.024188, 0.156429]]}, {"filename": "000000216794.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.890766, 0.617344, 0.218281, 0.187563]]}, {"filename": "000000218508.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.4206, 0.577823, 0.12344, 0.396847]]}, {"filename": "000000218751.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.824054, 0.48073, 0.107027, 0.06202], [1, 0.915405, 0.47397, 0.047628, 0.01806], [5, 0.445075, 0.66576, 0.332553, 0.63312]]}, {"filename": "000000219674.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.415059, 0.5, 0.830117, 1.0], [3, 0.5, 0.791008, 1.0, 0.143828], [4, 0.636455, 0.584781, 0.247934, 0.074313], [4, 0.713709, 0.610625, 0.2277, 0.224469], [4, 0.619507, 0.467422, 0.246432, 0.049437], [4, 0.795059, 0.36068, 0.232934, 0.038203], [4, 0.90304, 0.340539, 0.192653, 0.059172], [4, 0.71412, 0.425039, 0.234624, 0.057422], [4, 0.699096, 0.334547, 0.212324, 0.046469], [4, 0.59, 0.374688, 0.163756, 0.039156], [4, 0.865951, 0.528086, 0.246455, 0.157297], [4, 0.102887, 0.366633, 0.153615, 0.043828], [4, 0.91081, 0.43593, 0.163263, 0.073516], [4, 0.483873, 0.684414, 0.124883, 0.070141], [0, 0.063099, 0.679227, 0.119718, 0.089859], [4, 0.520775, 0.547227, 0.205822, 0.102203]]}, {"filename": "000000219756.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.530906, 0.226056, 0.214938, 0.244883], [1, 0.361234, 0.183592, 0.136125, 0.166291], [1, 0.207164, 0.215176, 0.103203, 0.226972], [1, 0.930445, 0.390669, 0.138047, 0.325376], [1, 0.101445, 0.175939, 0.052047, 0.168263], [7, 0.150406, 0.176538, 0.061625, 0.219507], [7, 0.734602, 0.5298, 0.317297, 0.909319], [1, 0.273367, 0.153533, 0.074484, 0.085047], [7, 0.082867, 0.142793, 0.022141, 0.115305]]}, {"filename": "000000221565.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.376406, 0.567412, 0.603938, 0.568525], [0, 0.502, 0.497138, 0.062156, 0.098725], [0, 0.845844, 0.722212, 0.063781, 0.389075], [0, 0.320734, 0.484738, 0.029969, 0.057325], [0, 0.882641, 0.730687, 0.069937, 0.381975], [0, 0.373758, 0.496512, 0.021047, 0.060825], [0, 0.606445, 0.829325, 0.029984, 0.0455], [0, 0.951844, 0.708762, 0.070656, 0.415425]]}, {"filename": "000000222094.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.26545, 0.317973, 0.09942, 0.2], [1, 0.13356, 0.610293, 0.18024, 0.072907], [1, 0.55512, 0.605347, 0.066, 0.076507], [1, 0.23299, 0.60992, 0.03646, 0.043253], [1, 0.29219, 0.591533, 0.07374, 0.066213], [1, 0.64261, 0.576813, 0.03514, 0.04216], [1, 0.36235, 0.582027, 0.04382, 0.05392], [1, 0.64472, 0.579013, 0.02568, 0.041813], [1, 0.55383, 0.604853, 0.06646, 0.077013], [1, 0.60669, 0.59444, 0.0549, 0.047387], [1, 0.49289, 0.624587, 0.11486, 0.110613]]}, {"filename": "000000222458.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.808617, 0.518597, 0.020547, 0.064882], [3, 0.812906, 0.637099, 0.103687, 0.094434], [3, 0.960164, 0.598998, 0.067922, 0.06658], [0, 0.840656, 0.521191, 0.011844, 0.042571], [3, 0.896359, 0.529422, 0.026, 0.020495], [0, 0.730141, 0.565531, 0.051844, 0.205967], [0, 0.949797, 0.521851, 0.023156, 0.077759], [0, 0.928031, 0.523726, 0.021562, 0.077877], [3, 0.476359, 0.713349, 0.222906, 0.208302]]}, {"filename": "000000223182.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.190789, 0.193253, 0.245578, 0.269665], [0, 0.566102, 0.508559, 0.427891, 0.932435], [0, 0.906891, 0.488736, 0.18175, 0.968476], [0, 0.031078, 0.269554, 0.062156, 0.103643], [3, 0.319945, 0.780381, 0.639891, 0.35645], [4, 0.385445, 0.422268, 0.695484, 0.401338], [4, 0.795312, 0.049693, 0.0925, 0.099387], [0, 0.584438, 0.122333, 0.195844, 0.244665]]}, {"filename": "000000223203.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.414447, 0.101617, 0.426079, 0.098922], [5, 0.797355, 0.238203, 0.116023, 0.215719]]}, {"filename": "000000223747.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.68679, 0.530338, 0.37582, 0.525865]]}, {"filename": "000000225946.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.129891, 0.500216, 0.049875, 0.117495]]}, {"filename": "000000226417.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.94897, 0.653814, 0.09994, 0.155976], [1, 0.35355, 0.602072, 0.08834, 0.116276], [1, 0.6418, 0.689835, 0.11784, 0.268679], [1, 0.42603, 0.691111, 0.07326, 0.195556], [1, 0.24888, 0.685526, 0.08972, 0.242462], [0, 0.05089, 0.686141, 0.10178, 0.357327], [0, 0.41681, 0.623589, 0.10026, 0.294384], [0, 0.24098, 0.633529, 0.10268, 0.301532], [0, 0.59191, 0.621231, 0.0631, 0.257598], [0, 0.9762, 0.609009, 0.04696, 0.237357], [0, 0.51453, 0.546246, 0.02002, 0.04018], [0, 0.65568, 0.617342, 0.0954, 0.262432], [1, 0.74799, 0.680616, 0.04854, 0.148258], [5, 0.87615, 0.668033, 0.03146, 0.074024], [0, 0.74339, 0.597357, 0.0775, 0.203784], [0, 0.5685, 0.579009, 0.0234, 0.107087], [0, 0.3671, 0.534279, 0.01084, 0.024414], [1, 0.57522, 0.594745, 0.02552, 0.103844], [1, 0.51355, 0.584174, 0.01958, 0.052312]]}, {"filename": "000000226536.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.783417, 0.555222, 0.2262, 0.498489]]}, {"filename": "000000226705.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.148805, 0.748474, 0.294672, 0.503052], [1, 0.505156, 0.512289, 0.387031, 0.62608], [0, 0.504516, 0.374331, 0.194906, 0.524343], [1, 0.189133, 0.348052, 0.126109, 0.456291], [1, 0.260234, 0.175012, 0.061031, 0.052887], [3, 0.845117, 0.211984, 0.061172, 0.02284], [3, 0.28468, 0.16608, 0.048953, 0.031315], [3, 0.833984, 0.184073, 0.125531, 0.008615]]}, {"filename": "000000227307.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.761773, 0.615584, 0.011516, 0.017804], [1, 0.77557, 0.613049, 0.016609, 0.010491], [1, 0.803781, 0.6175, 0.011781, 0.017477], [1, 0.78957, 0.616005, 0.012797, 0.019065], [1, 0.743797, 0.612453, 0.013594, 0.011963], [1, 0.731383, 0.616507, 0.009953, 0.016238], [1, 0.661695, 0.617325, 0.009266, 0.011846], [1, 0.716812, 0.618633, 0.012312, 0.01493], [1, 0.253773, 0.614077, 0.019797, 0.01479], [1, 0.704266, 0.619836, 0.0115, 0.008598], [1, 0.492508, 0.613096, 0.015172, 0.01278], [1, 0.687, 0.618353, 0.011406, 0.016706], [1, 0.592109, 0.614498, 0.020344, 0.012921]]}, {"filename": "000000227478.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.270141, 0.613187, 0.483219, 0.617458], [0, 0.21657, 0.551687, 0.345516, 0.564042], [3, 0.390414, 0.234771, 0.401016, 0.147792], [3, 0.335484, 0.652469, 0.428812, 0.668146], [3, 0.21657, 0.107865, 0.244391, 0.130354]]}, {"filename": "000000227491.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.364891, 0.660823, 0.281469, 0.656188], [3, 0.476484, 0.855406, 0.825906, 0.289187]]}, {"filename": "000000227656.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.399667, 0.392062, 0.579542, 0.549656], [4, 0.693229, 0.225094, 0.109625, 0.139125]]}, {"filename": "000000227898.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.95093, 0.580649, 0.098141, 0.275259], [0, 0.565125, 0.451356, 0.114781, 0.248561], [0, 0.769133, 0.455932, 0.072391, 0.089599], [0, 0.728656, 0.557335, 0.084406, 0.175283], [0, 0.821797, 0.537075, 0.077406, 0.197736], [0, 0.012164, 0.596804, 0.024328, 0.048844], [3, 0.130555, 0.625153, 0.097797, 0.105307]]}, {"filename": "000000228214.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.809149, 0.871641, 0.276387, 0.256719], [0, 0.962413, 0.659867, 0.075175, 0.127922], [0, 0.90014, 0.663477, 0.060886, 0.113922], [0, 0.452587, 0.501273, 0.898462, 0.943828]]}, {"filename": "000000228794.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.322633, 0.300677, 0.179047, 0.184688], [3, 0.537141, 0.328844, 0.159969, 0.228729]]}, {"filename": "000000228915.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.667109, 0.502459, 0.665781, 0.995082]]}, {"filename": "000000228942.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.921148, 0.563354, 0.157703, 0.346082], [1, 0.298289, 0.676411, 0.593641, 0.518589], [0, 0.861617, 0.436646, 0.026422, 0.114232], [0, 0.905438, 0.432429, 0.025875, 0.076583], [0, 0.833562, 0.477884, 0.030125, 0.148182], [0, 0.147203, 0.389498, 0.039875, 0.093166], [0, 0.139125, 0.396458, 0.033187, 0.075987], [0, 0.925367, 0.433103, 0.015203, 0.069342], [0, 0.94843, 0.429498, 0.016391, 0.046395], [0, 0.294039, 0.535502, 0.072453, 0.117712], [0, 0.964031, 0.412429, 0.015906, 0.041097], [0, 0.896594, 0.430564, 0.021406, 0.088213], [0, 0.824727, 0.471395, 0.018797, 0.152759]]}, {"filename": "000000229058.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.670023, 0.821521, 0.071703, 0.120292], [4, 0.54318, 0.564073, 0.038922, 0.064521], [4, 0.573687, 0.756177, 0.069094, 0.105271], [4, 0.667383, 0.738917, 0.068234, 0.0355], [4, 0.666141, 0.764833, 0.065219, 0.02875], [4, 0.741172, 0.833729, 0.091312, 0.102708], [4, 0.442453, 0.559135, 0.045344, 0.076188], [4, 0.282406, 0.648323, 0.083719, 0.067104], [4, 0.166938, 0.643677, 0.075219, 0.067771], [4, 0.092594, 0.638167, 0.091312, 0.065917], [0, 0.339711, 0.241552, 0.006484, 0.014271], [0, 0.379023, 0.241354, 0.007141, 0.009417], [0, 0.392422, 0.25175, 0.006813, 0.01725], [0, 0.444156, 0.248927, 0.009187, 0.011604], [4, 0.664281, 0.751948, 0.074031, 0.038979], [4, 0.495273, 0.553135, 0.052047, 0.066437], [0, 0.384281, 0.231208, 0.011437, 0.010625], [0, 0.310273, 0.240687, 0.005172, 0.009833], [0, 0.412992, 0.25225, 0.007172, 0.009167], [0, 0.607102, 0.401438, 0.021453, 0.028583], [3, 0.281383, 0.651854, 0.084422, 0.065208], [3, 0.494859, 0.548958, 0.044375, 0.053875], [3, 0.093734, 0.640354, 0.092906, 0.068917], [3, 0.169531, 0.643375, 0.075063, 0.065667], [3, 0.035344, 0.636323, 0.070688, 0.061563], [4, 0.228719, 0.648146, 0.086969, 0.065625], [3, 0.441094, 0.554823, 0.039687, 0.071979], [3, 0.541539, 0.554208, 0.039609, 0.046417], [3, 0.586836, 0.556562, 0.052016, 0.054625]]}, {"filename": "000000229318.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.431854, 0.597539, 0.522167, 0.697047], [0, 0.647385, 0.034992, 0.036438, 0.055797], [0, 0.294365, 0.031766, 0.054271, 0.061937], [0, 0.505906, 0.530328, 0.643104, 0.803875], [0, 0.499104, 0.528578, 0.6545, 0.799937], [0, 0.50026, 0.463562, 0.191979, 0.366281]]}, {"filename": "000000229507.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.35935, 0.23812, 0.11314, 0.189467], [5, 0.88125, 0.36168, 0.05786, 0.143787], [1, 0.67561, 0.24832, 0.05266, 0.138613]]}, {"filename": "000000229659.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.33807, 0.669813, 0.309859, 0.448], [4, 0.503969, 0.433594, 0.057969, 0.049937], [4, 0.081898, 0.738542, 0.115047, 0.24925], [4, 0.386219, 0.442177, 0.107687, 0.025229], [0, 0.398406, 0.378927, 0.079375, 0.102646], [0, 0.812523, 0.182719, 0.128547, 0.250312], [4, 0.553469, 0.490927, 0.048156, 0.147813], [4, 0.450141, 0.429615, 0.048125, 0.044604], [4, 0.176102, 0.666937, 0.169453, 0.462625], [0, 0.266672, 0.356052, 0.129625, 0.175313]]}, {"filename": "000000230450.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.527914, 0.532062, 0.114891, 0.032125], [7, 0.345758, 0.658458, 0.122516, 0.366167], [7, 0.248609, 0.596031, 0.078844, 0.199854], [7, 0.826687, 0.702396, 0.3455, 0.559542], [7, 0.173437, 0.544562, 0.0235, 0.093958], [7, 0.192836, 0.550969, 0.037234, 0.108979], [1, 0.596656, 0.536083, 0.063563, 0.031375]]}, {"filename": "000000230819.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.378719, 0.495336, 0.126, 0.280579]]}, {"filename": "000000230996.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.388156, 0.506737, 0.216875, 0.869671], [0, 0.62168, 0.308169, 0.048953, 0.196667], [0, 0.567672, 0.308697, 0.0275, 0.160681], [0, 0.154906, 0.329131, 0.035156, 0.134977], [1, 0.135852, 0.325505, 0.040609, 0.106596], [0, 0.668945, 0.269085, 0.025203, 0.126573], [0, 0.640234, 0.326796, 0.021469, 0.109178], [0, 0.062844, 0.337793, 0.059906, 0.171174], [0, 0.187672, 0.346467, 0.073281, 0.16561], [1, 0.2645, 0.286455, 0.044844, 0.062817], [1, 0.815773, 0.260235, 0.368453, 0.520469], [1, 0.021, 0.303415, 0.042, 0.155469], [0, 0.474859, 0.32027, 0.034187, 0.159319], [0, 0.225008, 0.344577, 0.069016, 0.177512], [0, 0.113883, 0.345129, 0.056453, 0.178286], [0, 0.491117, 0.289014, 0.023422, 0.095117]]}, {"filename": "000000231125.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.42824, 0.45353, 0.429813, 0.39766], [0, 0.186453, 0.36324, 0.184427, 0.2232]]}, {"filename": "000000231206.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.575398, 0.576276, 0.544797, 0.807588], [1, 0.200609, 0.495492, 0.401219, 0.984239]]}, {"filename": "000000231534.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.543422, 0.511721, 0.178375, 0.170375], [0, 0.601359, 0.674496, 0.040281, 0.080749], [0, 0.582008, 0.476651, 0.035641, 0.034192], [1, 0.135047, 0.249192, 0.116813, 0.096183], [1, 0.190758, 0.486066, 0.133578, 0.154707]]}, {"filename": "000000231831.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.182256, 0.128578, 0.278815, 0.257156]]}, {"filename": "000000231984.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.416361, 0.799023, 0.248119, 0.259609]]}, {"filename": "000000232538.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.026094, 0.612854, 0.052187, 0.049125], [1, 0.066133, 0.615031, 0.061016, 0.044146], [3, 0.893883, 0.676813, 0.124578, 0.074042]]}, {"filename": "000000232646.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.853992, 0.281582, 0.152578, 0.193672]]}, {"filename": "000000232886.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.711242, 0.643504, 0.368109, 0.35875], [4, 0.702539, 0.634867, 0.379547, 0.326856], [0, 0.574008, 0.568892, 0.508641, 0.822822]]}, {"filename": "000000234022.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.106367, 0.443128, 0.147078, 0.429198]]}, {"filename": "000000234757.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.615617, 0.500896, 0.102953, 0.126458], [7, 0.501125, 0.454292, 0.114344, 0.594167], [3, 0.271836, 0.068917, 0.066016, 0.051833]]}, {"filename": "000000235034.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.035967, 0.141336, 0.023978, 0.040266], [7, 0.175669, 0.252891, 0.265019, 0.266844]]}, {"filename": "000000235870.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.477797, 0.508048, 0.771875, 0.921692]]}, {"filename": "000000236148.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.163094, 0.367656, 0.326187, 0.209354], [1, 0.5, 0.719604, 1.0, 0.540542], [1, 0.499719, 0.256188, 0.992688, 0.498875], [0, 0.288016, 0.151177, 0.200375, 0.246479]]}, {"filename": "000000237984.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.278125, 0.379087, 0.205406, 0.314614], [3, 0.521008, 0.64719, 0.952047, 0.660656]]}, {"filename": "000000238345.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.869664, 0.848313, 0.070797, 0.074167], [1, 0.401609, 0.80901, 0.136531, 0.166062], [1, 0.132477, 0.787656, 0.087703, 0.090354], [1, 0.497508, 0.805052, 0.044141, 0.047729], [1, 0.649336, 0.841375, 0.241609, 0.290833], [1, 0.315047, 0.776365, 0.017094, 0.019021], [1, 0.342234, 0.774323, 0.014063, 0.019937], [1, 0.229219, 0.789021, 0.123719, 0.128958], [1, 0.770984, 0.812021, 0.036281, 0.082792]]}, {"filename": "000000238887.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.824805, 0.272305, 0.050922, 0.216761]]}, {"filename": "000000239048.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.26028, 0.74774, 0.24372, 0.34009], [0, 0.26035, 0.497889, 0.15962, 0.186856]]}, {"filename": "000000239347.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.575172, 0.468469, 0.505062, 0.281521], [0, 0.566477, 0.404719, 0.179328, 0.242271]]}, {"filename": "000000239837.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.407961, 0.771083, 0.023828, 0.095292], [0, 0.4145, 0.697708, 0.013563, 0.063417], [0, 0.578164, 0.754104, 0.020516, 0.093], [0, 0.672398, 0.785927, 0.037266, 0.119896], [0, 0.388406, 0.754271, 0.025531, 0.065333], [0, 0.549906, 0.766437, 0.031156, 0.097917], [0, 0.7955, 0.76526, 0.028813, 0.074479], [0, 0.574297, 0.672417, 0.013781, 0.059125], [0, 0.721055, 0.720427, 0.020828, 0.078604], [1, 0.576766, 0.620687, 0.027031, 0.021542], [1, 0.606758, 0.618865, 0.021016, 0.010521], [1, 0.522492, 0.608823, 0.017672, 0.017354], [3, 0.628664, 0.649208, 0.056641, 0.021292], [3, 0.644055, 0.68351, 0.019141, 0.017563], [0, 0.558922, 0.976885, 0.058156, 0.046229], [0, 0.501484, 0.777177, 0.034844, 0.084354], [0, 0.899227, 0.751385, 0.025609, 0.082354], [3, 0.739031, 0.656521, 0.043063, 0.029333], [3, 0.707648, 0.647375, 0.027641, 0.021], [0, 0.823469, 0.758323, 0.026875, 0.081812], [3, 0.475086, 0.795125, 0.194703, 0.036667]]}, {"filename": "000000239843.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.48118, 0.701133, 0.16012, 0.57528]]}, {"filename": "000000240038.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.331633, 0.421021, 0.048141, 0.095939]]}, {"filename": "000000240250.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.580915, 0.324224, 0.508301, 0.620866], [0, 0.298464, 0.117516, 0.056732, 0.069444], [0, 0.127222, 0.174984, 0.129477, 0.303856], [0, 0.245735, 0.101912, 0.049608, 0.055359], [4, 0.074158, 0.415727, 0.148317, 0.337075], [4, 0.315082, 0.262369, 0.115425, 0.188007], [0, 0.239224, 0.193742, 0.079722, 0.179641], [0, 0.189077, 0.160874, 0.082565, 0.095866], [0, 0.605645, 0.265449, 0.362892, 0.196618], [0, 0.305278, 0.189395, 0.100556, 0.121863], [4, 0.197173, 0.3354, 0.136895, 0.275278], [0, 0.051119, 0.264142, 0.102239, 0.528284], [4, 0.184853, 0.194959, 0.027941, 0.035016], [4, 0.632018, 0.503676, 0.445507, 0.196993]]}, {"filename": "000000240648.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.934801, 0.116578, 0.130397, 0.123031], [1, 0.752103, 0.299648, 0.468975, 0.142516], [1, 0.348473, 0.283742, 0.344435, 0.112484], [0, 0.439278, 0.359547, 0.024079, 0.098875], [0, 0.329456, 0.355453, 0.044895, 0.118781], [1, 0.120105, 0.323906, 0.127824, 0.070719], [1, 0.066977, 0.279836, 0.027469, 0.018234], [1, 0.224163, 0.324281, 0.171381, 0.071219], [0, 0.359163, 0.357805, 0.044351, 0.108672]]}, {"filename": "000000241677.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.78493, 0.590208, 0.025922, 0.0575], [0, 0.733805, 0.621448, 0.044297, 0.055563], [0, 0.583484, 0.638875, 0.037781, 0.056458]]}, {"filename": "000000241860.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.677352, 0.596145, 0.101953, 0.152243], [4, 0.49343, 0.589311, 0.083234, 0.128435], [4, 0.867555, 0.534357, 0.041359, 0.204463], [4, 0.935805, 0.573376, 0.128391, 0.19021]]}, {"filename": "000000241963.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.773867, 0.081734, 0.136266, 0.060375], [7, 0.48575, 0.506781, 0.427469, 0.888594], [1, 0.152352, 0.099391, 0.200609, 0.09375], [1, 0.225406, 0.085242, 0.100656, 0.054766], [1, 0.374484, 0.07182, 0.133156, 0.030891]]}, {"filename": "000000242721.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.798958, 0.212461, 0.283792, 0.129328], [7, 0.367042, 0.494383, 0.728083, 0.979766], [1, 0.062917, 0.691008, 0.125833, 0.433703]]}, {"filename": "000000242869.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.36594, 0.515734, 0.28956, 0.40226]]}, {"filename": "000000243034.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.304052, 0.87893, 0.047307, 0.046703], [0, 0.25671, 0.85457, 0.01281, 0.033578], [0, 0.308466, 0.849813, 0.022553, 0.019313], [0, 0.217904, 0.906859, 0.068923, 0.043844], [0, 0.705703, 0.88093, 0.043396, 0.050766], [0, 0.676124, 0.851, 0.014543, 0.038531], [0, 0.945995, 0.843391, 0.016393, 0.02875], [0, 0.089239, 0.861437, 0.017963, 0.028406], [0, 0.058888, 0.856133, 0.023817, 0.036766], [0, 0.244684, 0.853836, 0.013489, 0.030516]]}, {"filename": "000000243810.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.470733, 0.113156, 0.229591, 0.08325], [1, 0.334111, 0.105539, 0.141827, 0.062172], [6, 0.339267, 0.024641, 0.04488, 0.030625]]}, {"filename": "000000244428.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.533361, 0.165539, 0.533372, 0.238734]]}, {"filename": "000000244442.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.660562, 0.656625, 0.06275, 0.092133], [1, 0.23518, 0.872692, 0.157453, 0.084224], [1, 0.551039, 0.862464, 0.215547, 0.088033]]}, {"filename": "000000244496.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.496649, 0.55056, 0.920036, 0.89888]]}, {"filename": "000000244597.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.304055, 0.83774, 0.406609, 0.324521], [0, 0.3, 0.596635, 0.391, 0.784271]]}, {"filename": "000000245513.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.25897, 0.565892, 0.01538, 0.019711]]}, {"filename": "000000246308.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.142906, 0.940023, 0.168938, 0.119953]]}, {"filename": "000000246436.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.419094, 0.497977, 0.838187, 0.982109]]}, {"filename": "000000246963.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.776984, 0.646167, 0.114062, 0.218952], [1, 0.644719, 0.528452, 0.057844, 0.136143], [1, 0.313406, 0.468833, 0.103, 0.217476], [1, 0.380813, 0.424202, 0.095656, 0.197595], [1, 0.187836, 0.418393, 0.093078, 0.186833], [0, 0.173195, 0.288952, 0.084891, 0.111905], [0, 0.28707, 0.398583, 0.103734, 0.21631], [0, 0.389219, 0.390738, 0.08725, 0.260905], [0, 0.755797, 0.530345, 0.140094, 0.346071], [0, 0.624414, 0.449738, 0.121797, 0.281905], [6, 0.717313, 0.246857, 0.066219, 0.105381]]}, {"filename": "000000247351.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.050805, 0.107167, 0.101609, 0.21075]]}, {"filename": "000000247849.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.74243, 0.503373, 0.51514, 0.972987], [0, 0.56798, 0.570107, 0.48876, 0.831467], [0, 0.29663, 0.66152, 0.5311, 0.632107]]}, {"filename": "000000248591.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.878115, 0.607703, 0.243771, 0.442094], [4, 0.423979, 0.415039, 0.668167, 0.512359], [0, 0.451896, 0.382766, 0.56375, 0.760625]]}, {"filename": "000000248834.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.044309, 0.739852, 0.088571, 0.042828]]}, {"filename": "000000249219.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.477656, 0.668549, 0.666031, 0.568539]]}, {"filename": "000000249273.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.110716, 0.265305, 0.214624, 0.204078], [5, 0.503052, 0.58632, 0.251408, 0.444484], [1, 0.627723, 0.256547, 0.12831, 0.0295]]}, {"filename": "000000249277.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.927198, 0.554609, 0.145604, 0.082594], [3, 0.802146, 0.534523, 0.11625, 0.061484], [0, 0.250979, 0.483289, 0.029708, 0.083328], [0, 0.828458, 0.520219, 0.128833, 0.2], [0, 0.358385, 0.492289, 0.050229, 0.028078], [0, 0.386073, 0.4785, 0.026854, 0.040906], [0, 0.456073, 0.483703, 0.047313, 0.114219], [0, 0.666219, 0.484469, 0.031146, 0.084875], [0, 0.873813, 0.48875, 0.025333, 0.081937], [0, 0.746969, 0.521758, 0.088562, 0.091547], [0, 0.692365, 0.524781, 0.084438, 0.080906], [0, 0.351813, 0.464539, 0.016458, 0.040453], [0, 0.56025, 0.49493, 0.0475, 0.106141], [0, 0.288917, 0.495562, 0.022042, 0.043406], [1, 0.571396, 0.505797, 0.107583, 0.045906], [0, 0.75651, 0.472875, 0.020896, 0.032719]]}, {"filename": "000000249537.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.346641, 0.371432, 0.278344, 0.337653]]}, {"filename": "000000249786.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.50343, 0.350713, 0.072953, 0.134696], [0, 0.829367, 0.459182, 0.011609, 0.020093], [0, 0.548109, 0.546636, 0.005094, 0.010654], [0, 0.54282, 0.557558, 0.009734, 0.032126], [0, 0.498344, 0.561484, 0.006562, 0.016005], [0, 0.152398, 0.732161, 0.008797, 0.028715], [0, 0.388641, 0.600023, 0.008562, 0.022757], [0, 0.404937, 0.628026, 0.005719, 0.020444], [0, 0.160367, 0.796577, 0.012922, 0.025724], [0, 0.147875, 0.789918, 0.014375, 0.030864], [0, 0.531789, 0.562021, 0.012516, 0.030304]]}, {"filename": "000000250192.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.186445, 0.514611, 0.097328, 0.310111], [0, 0.337375, 0.503625, 0.069375, 0.28475], [0, 0.381992, 0.441375, 0.026797, 0.219806], [0, 0.511664, 0.167431, 0.133578, 0.331528], [0, 0.851305, 0.459431, 0.036078, 0.212139], [0, 0.937953, 0.497, 0.037094, 0.193444], [0, 0.980055, 0.432486, 0.035359, 0.103583], [0, 0.921078, 0.427083, 0.019875, 0.092833], [0, 0.776055, 0.511389, 0.042484, 0.234944], [0, 0.890898, 0.476125, 0.040984, 0.17825], [3, 0.859227, 0.587167, 0.211797, 0.164222]]}, {"filename": "000000250268.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.855148, 0.672477, 0.289703, 0.59284], [7, 0.286438, 0.549871, 0.243812, 0.710117], [1, 0.79775, 0.423146, 0.163313, 0.218216]]}, {"filename": "000000251448.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.914609, 0.658427, 0.170781, 0.660687], [0, 0.048359, 0.742958, 0.096094, 0.513542], [0, 0.032781, 0.506677, 0.065562, 0.219521], [0, 0.70025, 0.594396, 0.069125, 0.23025], [3, 0.736492, 0.810417, 0.051672, 0.148833], [0, 0.6335, 0.709042, 0.179219, 0.444167], [0, 0.793828, 0.724677, 0.126625, 0.301312]]}, {"filename": "000000252468.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.405062, 0.730431, 0.810125, 0.539138], [3, 0.659508, 0.250513, 0.415547, 0.501027], [0, 0.521641, 0.5, 0.956719, 1.0]]}, {"filename": "000000252639.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.197797, 0.524333, 0.056781, 0.068103], [1, 0.123039, 0.506335, 0.042547, 0.045738], [1, 0.100703, 0.496909, 0.033094, 0.045293], [1, 0.406477, 0.489578, 0.364891, 0.315457], [1, 0.181727, 0.470948, 0.091172, 0.052998], [1, 0.168555, 0.501194, 0.034734, 0.039251], [1, 0.769992, 0.507084, 0.091078, 0.042131], [1, 0.88782, 0.491932, 0.146203, 0.074871], [1, 0.711836, 0.499204, 0.090516, 0.054707], [0, 0.537352, 0.437787, 0.038578, 0.081475], [1, 0.614961, 0.506417, 0.056359, 0.04459], [1, 0.043141, 0.494824, 0.028844, 0.032646], [1, 0.663055, 0.50534, 0.031703, 0.040515], [1, 0.055953, 0.480258, 0.0285, 0.039157], [1, 0.042727, 0.467787, 0.019234, 0.019227]]}, {"filename": "000000253958.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.350625, 0.788068, 0.096156, 0.074543], [4, 0.426531, 0.929052, 0.039062, 0.141897], [4, 0.343039, 0.933946, 0.119359, 0.132108]]}, {"filename": "000000254714.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.427195, 0.724668, 0.118891, 0.105273], [1, 0.381133, 0.701797, 0.068359, 0.124062], [5, 0.242453, 0.434375, 0.144656, 0.753906], [0, 0.388453, 0.722988, 0.020094, 0.122539], [0, 0.361008, 0.727051, 0.019109, 0.098555]]}, {"filename": "000000255536.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.510837, 0.685225, 0.187275, 0.23835], [4, 0.13115, 0.656787, 0.10855, 0.215325], [4, 0.344712, 0.687287, 0.139375, 0.212625], [4, 0.239462, 0.675575, 0.122275, 0.1723], [1, 0.370638, 0.352463, 0.145275, 0.053975], [0, 0.372388, 0.337375, 0.073725, 0.07405], [0, 0.342462, 0.339838, 0.059975, 0.084175]]}, {"filename": "000000255718.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.510854, 0.519102, 0.59625, 0.862922]]}, {"filename": "000000256236.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.483109, 0.665542, 0.246625, 0.587833], [0, 0.971336, 0.576448, 0.055141, 0.206688], [3, 0.301695, 0.603031, 0.109328, 0.065438], [3, 0.333953, 0.588708, 0.078656, 0.061042]]}, {"filename": "000000256743.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.540945, 0.659553, 0.299953, 0.680894], [0, 0.976219, 0.457494, 0.047563, 0.223647], [0, 0.055023, 0.775212, 0.030891, 0.058], [6, 0.146758, 0.748, 0.030297, 0.047718], [0, 0.101, 0.788282, 0.030063, 0.061365], [0, 0.170219, 0.822576, 0.033844, 0.057435], [0, 0.22068, 0.833118, 0.028516, 0.061953], [0, 0.254883, 0.848941, 0.032047, 0.042165], [0, 0.283422, 0.848541, 0.023, 0.050729], [0, 0.843773, 0.886871, 0.087609, 0.221835], [0, 0.311781, 0.855329, 0.017594, 0.042941], [0, 0.961016, 0.838694, 0.018625, 0.043365], [0, 0.390477, 0.874282, 0.011422, 0.032518], [0, 0.33182, 0.855588, 0.020578, 0.043318], [1, 0.363633, 0.821682, 0.056891, 0.045059], [1, 0.070633, 0.775859, 0.139297, 0.127906]]}, {"filename": "000000256868.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.340375, 0.654965, 0.248906, 0.470864], [0, 0.708281, 0.558061, 0.139031, 0.152477], [0, 0.567383, 0.363528, 0.159453, 0.447991], [3, 0.84682, 0.603166, 0.076578, 0.144743], [3, 0.753313, 0.602103, 0.135937, 0.14486], [0, 0.030703, 0.477278, 0.028156, 0.064836], [0, 0.928961, 0.618271, 0.019172, 0.071682], [3, 0.169289, 0.560783, 0.075828, 0.065631], [3, 0.637234, 0.569474, 0.110125, 0.085117], [3, 0.08793, 0.556203, 0.035984, 0.064089], [3, 0.225258, 0.567932, 0.069703, 0.079416]]}, {"filename": "000000257231.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.041055, 0.465531, 0.081172, 0.030479], [1, 0.698063, 0.451656, 0.04025, 0.046229], [1, 0.96743, 0.461719, 0.065141, 0.050188], [0, 0.399438, 0.414604, 0.101125, 0.375292], [0, 0.2195, 0.553354, 0.120781, 0.334542], [0, 0.655188, 0.402375, 0.104969, 0.413083], [0, 0.664922, 0.516115, 0.235406, 0.247438], [0, 0.736516, 0.55326, 0.283156, 0.258437], [0, 0.486133, 0.585844, 0.216672, 0.420229], [3, 0.920664, 0.473292, 0.150828, 0.053], [1, 0.139016, 0.469844, 0.123438, 0.034438], [1, 0.585562, 0.478167, 0.03225, 0.046083]]}, {"filename": "000000257478.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.68559, 0.496872, 0.35614, 0.881432]]}, {"filename": "000000257566.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.437867, 0.247438, 0.019859, 0.036542], [0, 0.403961, 0.278677, 0.021047, 0.065771], [0, 0.478164, 0.244865, 0.045984, 0.077062], [0, 0.522711, 0.245677, 0.052359, 0.053896], [0, 0.453656, 0.25649, 0.016594, 0.018813], [0, 0.346625, 0.298156, 0.040375, 0.143271], [0, 0.362086, 0.260188, 0.015641, 0.06475], [0, 0.050516, 0.743521, 0.101031, 0.512958], [0, 0.36375, 0.29751, 0.01275, 0.128104]]}, {"filename": "000000258554.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.429008, 0.449881, 0.446484, 0.65843]]}, {"filename": "000000258559.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.631797, 0.763015, 0.165969, 0.327724], [1, 0.700758, 0.660206, 0.040672, 0.052663], [1, 0.497008, 0.835424, 0.400797, 0.300436], [0, 0.493773, 0.638184, 0.199266, 0.255157], [0, 0.927766, 0.659322, 0.025437, 0.111138], [5, 0.892406, 0.697676, 0.016062, 0.066295], [1, 0.714758, 0.650993, 0.028672, 0.047361], [1, 0.466898, 0.834116, 0.344672, 0.308232]]}, {"filename": "000000258793.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.943883, 0.504244, 0.037547, 0.028697], [7, 0.759594, 0.890399, 0.145094, 0.201807], [0, 0.293477, 0.525441, 0.024672, 0.076639], [1, 0.916625, 0.500924, 0.03325, 0.023824]]}, {"filename": "000000258878.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.05643, 0.643009, 0.067078, 0.087283], [1, 0.042406, 0.61541, 0.055313, 0.04637], [1, 0.021516, 0.662927, 0.042, 0.116253], [1, 0.01032, 0.60329, 0.020641, 0.026581]]}, {"filename": "000000259690.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.798555, 0.274083, 0.202766, 0.078875], [1, 0.984469, 0.246615, 0.030844, 0.010979], [1, 0.947734, 0.283542, 0.101812, 0.069292], [1, 0.911992, 0.230958, 0.117922, 0.043333], [0, 0.397055, 0.505531, 0.330922, 0.442979], [0, 0.565102, 0.491802, 0.214672, 0.627688], [0, 0.277203, 0.260448, 0.031688, 0.085979], [0, 0.183875, 0.339521, 0.056187, 0.157375], [0, 0.077672, 0.27325, 0.072687, 0.1085], [0, 0.305063, 0.330667, 0.062125, 0.157375], [0, 0.411852, 0.31174, 0.035609, 0.091854], [3, 0.157266, 0.374833, 0.203938, 0.070417], [3, 0.326242, 0.365042, 0.104078, 0.056333], [0, 0.256406, 0.338396, 0.050094, 0.162583], [0, 0.137937, 0.290458, 0.056, 0.136125], [0, 0.127766, 0.347427, 0.04525, 0.156188], [0, 0.395359, 0.318125, 0.026969, 0.102417], [1, 0.679461, 0.300365, 0.077422, 0.059104], [1, 0.652625, 0.279719, 0.042656, 0.020021], [4, 0.384242, 0.640542, 0.279109, 0.435208], [0, 0.334406, 0.244344, 0.021687, 0.064354], [3, 0.221117, 0.314031, 0.027641, 0.007146], [0, 0.319961, 0.259094, 0.034047, 0.084812], [1, 0.924594, 0.239302, 0.150812, 0.152854]]}, {"filename": "000000259884.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.626125, 0.457302, 0.739875, 0.591021], [1, 0.119063, 0.472896, 0.090875, 0.145583], [0, 0.40725, 0.649323, 0.088375, 0.351562], [0, 0.550445, 0.66875, 0.093141, 0.394917], [0, 0.465398, 0.605125, 0.071172, 0.314708], [0, 0.357305, 0.596865, 0.044078, 0.260396], [0, 0.378305, 0.501437, 0.031297, 0.068542], [0, 0.313633, 0.627792, 0.096641, 0.393792], [0, 0.208172, 0.644083, 0.09, 0.372833], [0, 0.109555, 0.515687, 0.029516, 0.104375], [0, 0.092156, 0.491792, 0.043313, 0.126792], [0, 0.094797, 0.651229, 0.135906, 0.349], [0, 0.422742, 0.621729, 0.031609, 0.250833], [0, 0.445906, 0.498479, 0.018094, 0.031]]}, {"filename": "000000260261.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.419566, 0.498867, 0.82561, 0.454953], [0, 0.607359, 0.609227, 0.395892, 0.734234], [0, 0.936326, 0.354383, 0.012606, 0.013547]]}, {"filename": "000000260470.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.148734, 0.30993, 0.259969, 0.351897], [0, 0.482734, 0.219953, 0.095688, 0.189836], [0, 0.97432, 0.220023, 0.051359, 0.439672], [0, 0.41043, 0.179473, 0.023047, 0.106534], [0, 0.425609, 0.177377, 0.016563, 0.097143], [1, 0.591266, 0.216733, 0.163781, 0.157354], [0, 0.294312, 0.20219, 0.036437, 0.14007]]}, {"filename": "000000260888.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.800211, 0.508937, 0.234484, 0.153762], [3, 0.682281, 0.551308, 0.183344, 0.105607], [0, 0.511883, 0.617897, 0.509641, 0.219112], [0, 0.570445, 0.711822, 0.079203, 0.347103], [3, 0.278273, 0.654416, 0.556547, 0.311495]]}, {"filename": "000000260953.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.637719, 0.278609, 0.293229, 0.310719], [0, 0.379281, 0.400305, 0.758563, 0.800609], [4, 0.458552, 0.203422, 0.242188, 0.339312], [4, 0.842854, 0.562344, 0.314292, 0.490906], [1, 0.756375, 0.062008, 0.153625, 0.124016], [1, 0.814656, 0.228414, 0.231646, 0.125891]]}, {"filename": "000000261097.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.48985, 0.61089, 0.205465, 0.46978], [0, 0.711411, 0.22362, 0.058859, 0.06056], [0, 0.105105, 0.22682, 0.058318, 0.03384], [0, 0.04015, 0.29694, 0.077177, 0.07132], [0, 0.173724, 0.41093, 0.08012, 0.04362], [0, 0.384655, 0.34388, 0.03964, 0.0328], [0, 0.778574, 0.39998, 0.078889, 0.06236], [0, 0.857793, 0.39616, 0.067297, 0.06012], [0, 0.917387, 0.94832, 0.165225, 0.10336], [0, 0.723168, 0.35237, 0.075586, 0.04582], [4, 0.316502, 0.53695, 0.114444, 0.09678], [0, 0.395435, 0.11337, 0.048529, 0.0547], [3, 0.735375, 0.40336, 0.084505, 0.05628], [3, 0.862628, 0.36666, 0.227778, 0.0154], [3, 0.341246, 0.35974, 0.097508, 0.0182], [3, 0.312673, 0.40666, 0.043123, 0.02512], [3, 0.335886, 0.38339, 0.083664, 0.01562], [3, 0.84467, 0.41702, 0.310661, 0.12112], [3, 0.35467, 0.18019, 0.195345, 0.01642], [3, 0.107492, 0.37483, 0.191441, 0.0263], [3, 0.786727, 0.45531, 0.197057, 0.0467], [3, 0.601336, 0.43555, 0.117207, 0.01186], [0, 0.605045, 0.39856, 0.078378, 0.06936], [0, 0.25452, 0.49848, 0.107538, 0.10816]]}, {"filename": "000000261161.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.19775, 0.598545, 0.3925, 0.659202]]}, {"filename": "000000261360.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.280195, 0.334656, 0.560391, 0.661396], [7, 0.319445, 0.516052, 0.490734, 0.938271], [7, 0.736898, 0.623656, 0.526203, 0.725812]]}, {"filename": "000000261888.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.677766, 0.763134, 0.295687, 0.341197], [0, 0.640891, 0.630411, 0.151844, 0.58716], [6, 0.376156, 0.530892, 0.014312, 0.022582]]}, {"filename": "000000262048.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.658031, 0.751005, 0.403807, 0.33152], [0, 0.275147, 0.743979, 0.473627, 0.232598], [0, 0.92777, 0.550907, 0.063448, 0.108611], [0, 0.941479, 0.442786, 0.023219, 0.019788], [0, 0.223873, 0.568284, 0.051634, 0.109706], [0, 0.904828, 0.423374, 0.008644, 0.01835], [0, 0.887304, 0.430065, 0.019575, 0.016242], [0, 0.893268, 0.461634, 0.021667, 0.063333], [0, 0.122337, 0.47683, 0.123072, 0.353562], [0, 0.853897, 0.438799, 0.013317, 0.021422]]}, {"filename": "000000262347.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.692453, 0.529356, 0.317, 0.33815]]}, {"filename": "000000263098.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.251609, 0.522523, 0.241719, 0.934676], [0, 0.091766, 0.494387, 0.1805, 0.988773], [1, 0.429914, 0.228565, 0.173109, 0.177917], [0, 0.834211, 0.15522, 0.093016, 0.11206], [5, 0.667031, 0.173299, 0.066656, 0.142894]]}, {"filename": "000000263418.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.089967, 0.74294, 0.179935, 0.3132], [0, 0.391891, 0.60541, 0.626217, 0.60502]]}, {"filename": "000000263440.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.481005, 0.483453, 0.638318, 0.668125]]}, {"filename": "000000264535.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.787949, 0.529935, 0.355016, 0.538758], [0, 0.954428, 0.435131, 0.031503, 0.089837], [0, 0.835065, 0.435204, 0.023562, 0.090016], [3, 0.396626, 0.676552, 0.631454, 0.534837]]}, {"filename": "000000265173.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.917398, 0.835703, 0.165203, 0.328595], [7, 0.68068, 0.618735, 0.209891, 0.501124], [0, 0.453539, 0.733021, 0.256391, 0.505621], [0, 0.812195, 0.78603, 0.091672, 0.407658], [0, 0.559844, 0.701358, 0.038438, 0.257564], [0, 0.523531, 0.681534, 0.077031, 0.250047], [1, 0.87282, 0.682225, 0.102453, 0.132178]]}, {"filename": "000000265244.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.923758, 0.52751, 0.057859, 0.147187]]}, {"filename": "000000265454.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.530109, 0.469323, 0.794312, 0.647729], [1, 0.958539, 0.619927, 0.082922, 0.170062], [1, 0.958063, 0.61449, 0.083875, 0.162646]]}, {"filename": "000000265799.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.781875, 0.336089, 0.43625, 0.307869], [1, 0.524008, 0.450714, 0.919078, 0.851686], [0, 0.626305, 0.227237, 0.025109, 0.050679], [0, 0.495266, 0.261663, 0.02025, 0.13733], [0, 0.517883, 0.262951, 0.030484, 0.143513], [0, 0.578742, 0.261604, 0.028047, 0.127752], [0, 0.427898, 0.256323, 0.016391, 0.07363], [1, 0.411914, 0.250258, 0.087859, 0.07541], [1, 0.5405, 0.265773, 0.026875, 0.02719], [1, 0.485344, 0.264848, 0.039375, 0.032787], [1, 0.385141, 0.264754, 0.056187, 0.053677], [0, 0.763281, 0.223958, 0.035937, 0.065574], [0, 0.462789, 0.256827, 0.011078, 0.053232], [1, 0.340477, 0.252225, 0.089234, 0.072272], [1, 0.694555, 0.252026, 0.237984, 0.144426], [1, 0.050469, 0.24993, 0.100937, 0.126042]]}, {"filename": "000000266082.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.51384, 0.76831, 0.43248, 0.46338]]}, {"filename": "000000267199.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.505471, 0.435953, 0.655921, 0.516844], [4, 0.478619, 0.713039, 0.267782, 0.351703]]}, {"filename": "000000267324.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.504469, 0.576167, 0.183375, 0.704375], [0, 0.977195, 0.475229, 0.045609, 0.335583], [0, 0.303086, 0.316823, 0.085109, 0.225938], [0, 0.252812, 0.384271, 0.148312, 0.426958], [0, 0.619969, 0.459115, 0.165813, 0.556312], [0, 0.792039, 0.362115, 0.087453, 0.230938], [0, 0.875844, 0.522521, 0.18075, 0.578833], [0, 0.161875, 0.557312, 0.32375, 0.885375], [0, 0.391266, 0.467344, 0.111813, 0.443646], [0, 0.705102, 0.429604, 0.104547, 0.447208]]}, {"filename": "000000268577.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.771648, 0.616802, 0.120391, 0.117358], [7, 0.516344, 0.45821, 0.035656, 0.194346], [7, 0.326703, 0.406272, 0.048969, 0.237975], [7, 0.201219, 0.404494, 0.061156, 0.233728], [7, 0.604703, 0.473556, 0.041437, 0.200889], [7, 0.842133, 0.499988, 0.028922, 0.132864], [7, 0.900336, 0.500395, 0.028172, 0.131012], [7, 0.987453, 0.593679, 0.025094, 0.090568], [7, 0.702937, 0.486877, 0.028688, 0.157506], [7, 0.778711, 0.490321, 0.030453, 0.160691]]}, {"filename": "000000268729.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.029977, 0.487271, 0.020547, 0.024417], [0, 0.560703, 0.42824, 0.010875, 0.025896], [0, 0.008266, 0.486562, 0.013437, 0.034917], [0, 0.591945, 0.43524, 0.007641, 0.013437], [0, 0.602367, 0.435823, 0.010953, 0.013729]]}, {"filename": "000000269863.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.485359, 0.623875, 0.916656, 0.722969], [4, 0.886516, 0.835953, 0.119094, 0.067406], [4, 0.716836, 0.836883, 0.165078, 0.109609], [4, 0.742734, 0.785438, 0.136562, 0.106813], [4, 0.744664, 0.678242, 0.106797, 0.048297], [4, 0.465867, 0.632594, 0.068891, 0.049875], [4, 0.105148, 0.75525, 0.085016, 0.078313], [4, 0.520867, 0.647391, 0.070203, 0.045625], [4, 0.200023, 0.685602, 0.107109, 0.075953], [0, 0.497164, 0.37425, 0.017828, 0.046437], [0, 0.465992, 0.383242, 0.017891, 0.048109], [0, 0.874078, 0.638508, 0.24075, 0.135422], [0, 0.092062, 0.448813, 0.050187, 0.046], [0, 0.249055, 0.337039, 0.014422, 0.036172], [0, 0.050203, 0.514938, 0.087063, 0.070844], [0, 0.552531, 0.538008, 0.048281, 0.074984], [0, 0.594141, 0.524711, 0.048688, 0.070828], [0, 0.048859, 0.41318, 0.041281, 0.023766], [0, 0.046477, 0.391117, 0.011891, 0.013078], [4, 0.742852, 0.713977, 0.124453, 0.069797], [4, 0.592891, 0.817102, 0.091, 0.072359], [4, 0.734781, 0.646797, 0.065562, 0.022625], [4, 0.139531, 0.655422, 0.1585, 0.1155], [0, 0.677711, 0.40868, 0.029859, 0.059953], [4, 0.866625, 0.750578, 0.1365, 0.074594], [0, 0.480898, 0.518961, 0.043203, 0.066234]]}, {"filename": "000000270440.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.344797, 0.448432, 0.264437, 0.556054], [0, 0.063836, 0.250297, 0.120859, 0.349892], [0, 0.158289, 0.211905, 0.053359, 0.234243], [0, 0.196609, 0.18773, 0.07125, 0.203459], [0, 0.607461, 0.099784, 0.033703, 0.060595], [0, 0.495125, 0.120784, 0.041563, 0.199892], [0, 0.562125, 0.122054, 0.047938, 0.241297], [0, 0.680531, 0.121081, 0.046656, 0.237676], [0, 0.744109, 0.107851, 0.0465, 0.21127], [0, 0.321469, 0.129297, 0.057031, 0.255622], [0, 0.784359, 0.13927, 0.052594, 0.200541], [3, 0.917945, 0.167892, 0.061109, 0.056432]]}, {"filename": "000000270612.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.277876, 0.689312, 0.128333, 0.049375], [1, 0.290704, 0.677758, 0.149484, 0.044953], [1, 0.319143, 0.665328, 0.060305, 0.0325], [6, 0.907312, 0.541797, 0.024296, 0.08125], [1, 0.069941, 0.661766, 0.046268, 0.018594], [1, 0.146408, 0.648797, 0.076291, 0.019219], [1, 0.018322, 0.660523, 0.036643, 0.037359], [1, 0.166467, 0.71275, 0.331197, 0.084844], [1, 0.172523, 0.689, 0.331948, 0.064469]]}, {"filename": "000000271116.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.087094, 0.717288, 0.174187, 0.514623], [4, 0.974273, 0.750778, 0.051453, 0.391509], [0, 0.761867, 0.550672, 0.476266, 0.889646], [0, 0.264555, 0.5, 0.529109, 1.0], [0, 0.485945, 0.441191, 0.119859, 0.15842], [0, 0.653758, 0.445236, 0.116328, 0.16566], [0, 0.615633, 0.448255, 0.055141, 0.106415], [0, 0.411523, 0.437559, 0.030516, 0.054929], [0, 0.459063, 0.421167, 0.028719, 0.046392], [0, 0.389641, 0.427252, 0.040281, 0.078137], [0, 0.553586, 0.456427, 0.022266, 0.035024], [0, 0.103055, 0.212535, 0.101547, 0.047571], [0, 0.581648, 0.457901, 0.025922, 0.03434], [0, 0.715109, 0.478396, 0.039625, 0.117406], [0, 0.563867, 0.452158, 0.035297, 0.06908]]}, {"filename": "000000271457.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.663, 0.806306, 0.672, 0.36036]]}, {"filename": "000000272148.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.321609, 0.68082, 0.020219, 0.036614], [0, 0.44918, 0.717037, 0.027516, 0.046085], [0, 0.242312, 0.612103, 0.030125, 0.034206], [0, 0.193023, 0.472989, 0.009891, 0.011005], [0, 0.18157, 0.557725, 0.013703, 0.013122], [0, 0.866047, 0.432447, 0.001969, 0.020026], [0, 0.844234, 0.419577, 0.008, 0.022751], [0, 0.23357, 0.580013, 0.009516, 0.027381], [0, 0.442023, 0.880066, 0.076141, 0.042989], [0, 0.391789, 0.451495, 0.008828, 0.005582], [0, 0.44843, 0.491521, 0.004453, 0.013042], [0, 0.208328, 0.49578, 0.025719, 0.016429]]}, {"filename": "000000272728.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.085967, 0.597398, 0.171935, 0.174797]]}, {"filename": "000000272743.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.652914, 0.764198, 0.034578, 0.056771], [7, 0.615352, 0.76374, 0.035828, 0.057813], [0, 0.34507, 0.855719, 0.068547, 0.272937], [6, 0.454273, 0.405104, 0.050891, 0.119375]]}, {"filename": "000000273132.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.490062, 0.641651, 0.596125, 0.688483], [3, 0.478508, 0.707291, 0.918172, 0.547472]]}, {"filename": "000000273198.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.33571, 0.050773, 0.22074, 0.099147], [1, 0.20649, 0.025387, 0.05554, 0.05072], [0, 0.10272, 0.27452, 0.20544, 0.54904], [1, 0.64367, 0.04436, 0.25798, 0.08872], [5, 0.19951, 0.38452, 0.10346, 0.3596]]}, {"filename": "000000273617.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.229859, 0.432307, 0.05525, 0.090281]]}, {"filename": "000000274537.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.771352, 0.114615, 0.457297, 0.152812]]}, {"filename": "000000275037.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.78476, 0.356067, 0.12016, 0.165627], [6, 0.08953, 0.408773, 0.08446, 0.11712], [1, 0.11054, 0.589, 0.08444, 0.0804], [1, 0.62941, 0.606533, 0.0893, 0.07008], [1, 0.72231, 0.59708, 0.04098, 0.04856], [0, 0.85846, 0.622573, 0.03196, 0.134533], [0, 0.75391, 0.607467, 0.03714, 0.144], [0, 0.82953, 0.613573, 0.0309, 0.1328], [1, 0.67213, 0.59864, 0.06234, 0.075413], [0, 0.79038, 0.607013, 0.02208, 0.13888]]}, {"filename": "000000275190.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.926242, 0.530026, 0.147516, 0.827282], [0, 0.696219, 0.614868, 0.221406, 0.659921], [0, 0.298258, 0.547929, 0.399141, 0.82066], [0, 0.416375, 0.562427, 0.083031, 0.251821], [0, 0.513187, 0.584208, 0.120031, 0.278443], [0, 0.22357, 0.076412, 0.090484, 0.103377], [0, 0.289438, 0.05843, 0.067875, 0.11686], [0, 0.13918, 0.063918, 0.090203, 0.127836], [0, 0.325367, 0.066293, 0.113109, 0.132586], [0, 0.905492, 0.065726, 0.107391, 0.131451], [0, 0.834875, 0.067098, 0.096719, 0.131082], [3, 0.589391, 0.615422, 0.101938, 0.213219], [0, 0.459305, 0.064301, 0.107609, 0.128602], [0, 0.109805, 0.065475, 0.034391, 0.057071], [3, 0.075164, 0.655765, 0.150328, 0.029842]]}, {"filename": "000000275913.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.435198, 0.688898, 0.389437, 0.402047]]}, {"filename": "000000276506.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.39607, 0.84551, 0.411234, 0.287646], [5, 0.438625, 0.932333, 0.065906, 0.134042]]}, {"filename": "000000276545.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.77135, 0.430147, 0.12526, 0.179013], [1, 0.48202, 0.886667, 0.96068, 0.2]]}, {"filename": "000000276720.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.459422, 0.624125, 0.053344, 0.074792], [1, 0.400477, 0.604958, 0.040422, 0.043292], [1, 0.333703, 0.596979, 0.021031, 0.025083], [1, 0.459109, 0.622552, 0.054937, 0.071729], [1, 0.401008, 0.603677, 0.040297, 0.046396]]}, {"filename": "000000276735.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.859141, 0.601285, 0.138, 0.760967], [0, 0.778883, 0.550672, 0.149203, 0.779269], [0, 0.65357, 0.520224, 0.133984, 0.635967], [0, 0.230016, 0.599021, 0.269469, 0.779788], [4, 0.981523, 0.446439, 0.036953, 0.013679], [4, 0.940672, 0.446285, 0.028031, 0.016627], [4, 0.540453, 0.533703, 0.091219, 0.2025], [4, 0.943711, 0.640389, 0.112578, 0.244505]]}, {"filename": "000000277020.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.415266, 0.60425, 0.245594, 0.766222], [0, 0.194031, 0.512361, 0.388063, 0.975278], [0, 0.634977, 0.561694, 0.185953, 0.406056], [4, 0.306531, 0.951681, 0.165594, 0.096639], [0, 0.848375, 0.498389, 0.30325, 0.986667], [4, 0.290227, 0.883653, 0.026734, 0.115583]]}, {"filename": "000000277094.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.950516, 0.627698, 0.060375, 0.093354], [1, 0.686766, 0.256927, 0.20725, 0.144021], [0, 0.544773, 0.453354, 0.046734, 0.150833], [6, 0.443516, 0.481812, 0.03475, 0.065958], [0, 0.881555, 0.554531, 0.062922, 0.165187], [0, 0.954281, 0.584281, 0.059625, 0.159854], [0, 0.045336, 0.263042, 0.023297, 0.059458], [0, 0.203867, 0.554281, 0.020297, 0.047937], [1, 0.415547, 0.766896, 0.375, 0.301792], [1, 0.204773, 0.589885, 0.358984, 0.285396], [1, 0.152523, 0.403375, 0.261234, 0.141583], [1, 0.530242, 0.269812, 0.117516, 0.070375], [1, 0.806516, 0.261448, 0.036438, 0.039563], [1, 0.84275, 0.255802, 0.044344, 0.039104], [1, 0.972328, 0.262167, 0.053406, 0.046708], [7, 0.635641, 0.673187, 0.023719, 0.061958], [7, 0.082742, 0.982552, 0.033359, 0.034229], [1, 0.025312, 0.558854, 0.050625, 0.129292], [1, 0.034555, 0.335448, 0.069109, 0.138021], [1, 0.389328, 0.211719, 0.027875, 0.038771], [1, 0.235664, 0.17675, 0.024547, 0.037458], [1, 0.897555, 0.247458, 0.054047, 0.035583], [0, 0.078609, 0.278052, 0.025938, 0.089188], [1, 0.489266, 0.249521, 0.068594, 0.041708], [0, 0.418797, 0.44375, 0.061188, 0.160292]]}, {"filename": "000000277267.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.83572, 0.624428, 0.19984, 0.419578], [4, 0.42787, 0.836235, 0.16334, 0.200542], [0, 0.41314, 0.617937, 0.21588, 0.712982], [0, 0.84611, 0.593117, 0.16038, 0.699789], [0, 0.42128, 0.242861, 0.17956, 0.277771]]}, {"filename": "000000277461.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.287852, 0.600428, 0.013422, 0.041065], [0, 0.62832, 0.635449, 0.015359, 0.054572], [0, 0.176938, 0.609196, 0.009156, 0.02119], [1, 0.094422, 0.612641, 0.057, 0.059102], [1, 0.053547, 0.662954, 0.063156, 0.058643], [1, 0.166805, 0.611712, 0.023172, 0.024134], [0, 0.522703, 0.686775, 0.028219, 0.091044]]}, {"filename": "000000277623.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.4211, 0.105375, 0.4002, 0.201802]]}, {"filename": "000000278101.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.259492, 0.593976, 0.112234, 0.078541], [1, 0.432609, 0.600765, 0.078156, 0.075318], [1, 0.355641, 0.595118, 0.099406, 0.087365], [5, 0.53275, 0.621812, 0.012531, 0.053976], [0, 0.609609, 0.6798, 0.102937, 0.362565], [0, 0.165195, 0.633094, 0.103203, 0.358235], [0, 0.07757, 0.621682, 0.078359, 0.388118], [3, 0.955961, 0.672541, 0.085266, 0.083341], [0, 0.560391, 0.658447, 0.086844, 0.250824], [0, 0.680781, 0.696835, 0.093531, 0.304918], [1, 0.53225, 0.612376, 0.09075, 0.074541], [0, 0.004883, 0.584506, 0.009516, 0.108353]]}, {"filename": "000000278344.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.847594, 0.658133, 0.303854, 0.260578], [1, 0.408656, 0.48325, 0.103479, 0.030375]]}, {"filename": "000000278375.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.46172, 0.484217, 0.11844, 0.49]]}, {"filename": "000000278525.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.190086, 0.657732, 0.100672, 0.263475], [3, 0.80018, 0.668342, 0.175328, 0.257533]]}, {"filename": "000000278638.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.87257, 0.259129, 0.125672, 0.113976], [4, 0.735773, 0.2712, 0.106922, 0.140424], [0, 0.750633, 0.211788, 0.109578, 0.259059], [0, 0.80482, 0.181247, 0.036984, 0.0592], [0, 0.923625, 0.202824, 0.10175, 0.216847], [0, 0.98282, 0.214212, 0.034078, 0.164706], [0, 0.981383, 0.203953, 0.019859, 0.057082]]}, {"filename": "000000279145.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.516891, 0.551802, 0.429063, 0.490979]]}, {"filename": "000000279714.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.761082, 0.812297, 0.477835, 0.268469], [1, 0.047941, 0.813672, 0.095882, 0.290094], [7, 0.345259, 0.822969, 0.542282, 0.316063]]}, {"filename": "000000279775.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.84175, 0.560729, 0.273219, 0.22875]]}, {"filename": "000000280325.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.82992, 0.688037, 0.30648, 0.524562]]}, {"filename": "000000281179.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.385797, 0.285094, 0.018562, 0.044479], [3, 0.98457, 0.322552, 0.030859, 0.077438]]}, {"filename": "000000281227.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.157938, 0.77699, 0.312625, 0.401729], [1, 0.873586, 0.558875, 0.198047, 0.240167], [1, 0.595969, 0.791396, 0.775813, 0.309667], [0, 0.554367, 0.596187, 0.384859, 0.8055], [0, 0.782164, 0.422979, 0.032609, 0.055625], [1, 0.789961, 0.481333, 0.062141, 0.06125], [1, 0.756102, 0.457146, 0.015359, 0.01125], [1, 0.164578, 0.206563, 0.062438, 0.078917], [1, 0.863555, 0.495542, 0.110922, 0.136167], [1, 0.852641, 0.447979, 0.056625, 0.031], [1, 0.169516, 0.128177, 0.065687, 0.082938], [1, 0.739805, 0.46251, 0.072203, 0.037437], [6, 0.839813, 0.137042, 0.101844, 0.14075]]}, {"filename": "000000281382.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.783555, 0.546073, 0.127516, 0.241604], [4, 0.14832, 0.903375, 0.286516, 0.170792]]}, {"filename": "000000281502.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.372484, 0.869344, 0.3255, 0.239354]]}, {"filename": "000000282293.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.250359, 0.878096, 0.166594, 0.221145], [1, 0.470969, 0.568248, 0.041781, 0.04271], [0, 0.23218, 0.725841, 0.139766, 0.453925], [0, 0.735703, 0.72271, 0.113063, 0.50028], [0, 0.831055, 0.690631, 0.087172, 0.397757], [0, 0.631117, 0.629217, 0.081766, 0.254322], [0, 0.044789, 0.573867, 0.012109, 0.057874], [0, 0.053453, 0.573224, 0.008594, 0.055421], [0, 0.061711, 0.570012, 0.015766, 0.066238], [0, 0.590156, 0.564159, 0.00975, 0.056402], [7, 0.076484, 0.87528, 0.130875, 0.249439], [0, 0.577977, 0.561729, 0.009859, 0.045888], [0, 0.690727, 0.574287, 0.038297, 0.161051], [1, 0.309828, 0.843119, 0.083625, 0.299369], [0, 0.568734, 0.547383, 0.008031, 0.028598], [0, 0.782, 0.612558, 0.029375, 0.15021], [1, 0.434539, 0.556682, 0.024609, 0.026402], [1, 0.386367, 0.567301, 0.039297, 0.032921]]}, {"filename": "000000282329.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.530492, 0.515039, 0.555738, 0.892141], [3, 0.5, 0.680898, 1.0, 0.391016], [1, 0.237131, 0.206859, 0.474262, 0.218844]]}, {"filename": "000000282354.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.203953, 0.738719, 0.394306, 0.501125], [0, 0.833129, 0.479125, 0.333741, 0.711062]]}, {"filename": "000000282479.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.42133, 0.473378, 0.04598, 0.097327]]}, {"filename": "000000283254.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.083274, 0.271922, 0.166548, 0.543844], [0, 0.288574, 0.181164, 0.214672, 0.362328], [0, 0.752608, 0.10207, 0.12334, 0.204141], [0, 0.812336, 0.088172, 0.065385, 0.105], [0, 0.500816, 0.507992, 0.998368, 0.984016], [0, 0.655901, 0.0825, 0.203171, 0.165], [0, 0.135863, 0.188687, 0.108011, 0.346469]]}, {"filename": "000000283318.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.828234, 0.73534, 0.157125, 0.176393], [1, 0.769234, 0.707916, 0.078031, 0.133817], [1, 0.729484, 0.655492, 0.036188, 0.047799], [1, 0.668758, 0.644426, 0.017172, 0.033115], [5, 0.814859, 0.861171, 0.04125, 0.150539], [0, 0.945344, 0.643267, 0.010688, 0.045222], [0, 0.933156, 0.646909, 0.008313, 0.04534], [1, 0.745562, 0.637482, 0.021625, 0.01192], [0, 0.583961, 0.644953, 0.016141, 0.05534]]}, {"filename": "000000283835.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.443445, 0.482416, 0.584953, 0.855318]]}, {"filename": "000000284297.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.90175, 0.716188, 0.070781, 0.108612], [0, 0.431039, 0.799212, 0.090828, 0.369953], [0, 0.563031, 0.738894, 0.035562, 0.160047], [0, 0.947227, 0.739612, 0.034328, 0.105529], [3, 0.147539, 0.903365, 0.266203, 0.1768], [1, 0.823516, 0.716753, 0.224125, 0.101976], [1, 0.974312, 0.725529, 0.042219, 0.075482], [3, 0.635703, 0.736329, 0.041188, 0.029035]]}, {"filename": "000000285579.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.660508, 0.532095, 0.137828, 0.567338], [3, 0.043961, 0.40103, 0.058703, 0.058819], [3, 0.17543, 0.398113, 0.063047, 0.025208], [3, 0.218336, 0.437951, 0.027266, 0.045718], [7, 0.48857, 0.367569, 0.013078, 0.055926], [3, 0.219906, 0.404826, 0.020938, 0.023634]]}, {"filename": "000000286033.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.4323, 0.470787, 0.22416, 0.73032], [0, 0.13302, 0.55696, 0.23712, 0.8592]]}, {"filename": "000000286553.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.72057, 0.911323, 0.099203, 0.150229], [4, 0.055617, 0.679146, 0.062422, 0.234667], [0, 0.34382, 0.393406, 0.522484, 0.737063], [0, 0.716547, 0.561271, 0.566906, 0.877417]]}, {"filename": "000000286576.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.57848, 0.69118, 0.553067, 0.44172], [4, 0.455573, 0.70084, 0.297013, 0.44356], [4, 0.894013, 0.61854, 0.211973, 0.21572], [4, 0.68576, 0.67709, 0.357653, 0.41406]]}, {"filename": "000000288378.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.134984, 0.796303, 0.184844, 0.169977], [3, 0.189719, 0.824965, 0.182344, 0.176831], [3, 0.070836, 0.781467, 0.138859, 0.187864], [3, 0.795242, 0.87554, 0.110766, 0.236948], [3, 0.573352, 0.815411, 0.091547, 0.165986]]}, {"filename": "000000288450.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.15132, 0.16642, 0.20052, 0.3306], [0, 0.72338, 0.54109, 0.12216, 0.05918], [0, 0.74976, 0.62753, 0.1814, 0.07318], [0, 0.9564, 0.47018, 0.08712, 0.08712], [0, 0.92806, 0.41484, 0.12512, 0.15932], [5, 0.91722, 0.16912, 0.12976, 0.30648], [0, 0.83827, 0.41594, 0.10886, 0.15996], [0, 0.52248, 0.81127, 0.29476, 0.2519], [0, 0.41696, 0.44579, 0.17268, 0.15526], [0, 0.73209, 0.43388, 0.13278, 0.14992]]}, {"filename": "000000288472.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.403367, 0.7, 0.406734, 0.424719], [1, 0.708992, 0.728352, 0.159547, 0.125828], [1, 0.805258, 0.680609, 0.068547, 0.05025], [1, 0.883953, 0.679805, 0.069375, 0.049266], [1, 0.980547, 0.731375, 0.038906, 0.092281], [1, 0.666086, 0.67068, 0.071641, 0.042516], [1, 0.836094, 0.672, 0.05325, 0.046], [1, 0.934766, 0.681531, 0.023719, 0.018719], [6, 0.944578, 0.642609, 0.014063, 0.014187]]}, {"filename": "000000288584.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.328992, 0.496768, 0.096578, 0.212272], [0, 0.241672, 0.488548, 0.111719, 0.584169], [0, 0.265016, 0.801042, 0.130844, 0.394028], [0, 0.694172, 0.635948, 0.140937, 0.597752], [0, 0.777859, 0.403361, 0.063656, 0.38541], [0, 0.887984, 0.306171, 0.139031, 0.248454], [0, 0.901547, 0.17685, 0.097906, 0.172951], [0, 0.097281, 0.844309, 0.134875, 0.308431], [0, 0.907703, 0.424696, 0.115312, 0.301241], [0, 0.983555, 0.395094, 0.032891, 0.243443], [4, 0.051359, 0.062799, 0.082906, 0.125597]]}, {"filename": "000000288828.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.293906, 0.312389, 0.279375, 0.254988], [7, 0.632602, 0.315691, 0.062797, 0.226464], [0, 0.719898, 0.470094, 0.165672, 0.670445], [0, 0.365898, 0.524778, 0.217891, 0.921171], [0, 0.505602, 0.517447, 0.207016, 0.843747], [0, 0.238461, 0.433665, 0.098297, 0.650984], [1, 0.4805, 0.194496, 0.149844, 0.069696], [1, 0.123523, 0.199848, 0.047422, 0.037728], [1, 0.066875, 0.200808, 0.042344, 0.028407], [1, 0.006859, 0.199321, 0.013719, 0.037705], [1, 0.865812, 0.428829, 0.268375, 0.826651], [7, 0.577867, 0.287471, 0.052797, 0.166183]]}, {"filename": "000000289417.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.65426, 0.32868, 0.2222, 0.51848], [0, 0.88927, 0.199493, 0.06142, 0.151733], [0, 0.54404, 0.192813, 0.07612, 0.127867], [0, 0.45208, 0.14472, 0.0998, 0.251627], [0, 0.93028, 0.15544, 0.0532, 0.18352], [0, 0.55645, 0.0928, 0.07862, 0.147947], [3, 0.3289, 0.208173, 0.08728, 0.030587], [3, 0.22188, 0.204973, 0.08248, 0.027013], [3, 0.13393, 0.20664, 0.04422, 0.02096]]}, {"filename": "000000289532.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.353477, 0.395453, 0.086484, 0.180594], [3, 0.813484, 0.715734, 0.348313, 0.460656], [3, 0.597453, 0.653594, 0.26125, 0.34325], [3, 0.462961, 0.588406, 0.179109, 0.280625], [3, 0.391, 0.505578, 0.116656, 0.181969], [3, 0.335688, 0.308969, 0.027344, 0.040625], [3, 0.369719, 0.44825, 0.074, 0.1275], [3, 0.454, 0.367688, 0.017781, 0.092063], [3, 0.368719, 0.307016, 0.036563, 0.046469], [3, 0.365914, 0.411109, 0.053484, 0.067469], [3, 0.416859, 0.390969, 0.036469, 0.084562], [3, 0.447383, 0.309062, 0.020484, 0.025125], [3, 0.460211, 0.3515, 0.022078, 0.07075], [3, 0.415945, 0.320531, 0.031172, 0.0605]]}, {"filename": "000000289594.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.084807, 0.448859, 0.081214, 0.057031], [0, 0.711811, 0.541555, 0.128439, 0.360422], [1, 0.064403, 0.432125, 0.053353, 0.04075]]}, {"filename": "000000290201.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.238865, 0.576406, 0.157812, 0.154375], [0, 0.608375, 0.446414, 0.582958, 0.531391], [0, 0.268417, 0.444836, 0.208667, 0.352141], [3, 0.223708, 0.576352, 0.12, 0.156078]]}, {"filename": "000000290761.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.588008, 0.354833, 0.064484, 0.137583]]}, {"filename": "000000291098.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.271844, 0.142225, 0.102346, 0.28445], [0, 0.126508, 0.080117, 0.090931, 0.158689], [4, 0.337058, 0.294262, 0.115903, 0.177049], [0, 0.370074, 0.094016, 0.076946, 0.188033], [0, 0.085866, 0.098384, 0.032328, 0.111241], [0, 0.020093, 0.103255, 0.038212, 0.173162]]}, {"filename": "000000291503.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.141281, 0.781885, 0.134625, 0.386771], [0, 0.10143, 0.796615, 0.159047, 0.356604], [1, 0.366414, 0.634833, 0.112703, 0.110875], [1, 0.431953, 0.638531, 0.04525, 0.089896], [1, 0.444398, 0.631573, 0.069734, 0.089646], [1, 0.487109, 0.631208, 0.042781, 0.082625], [1, 0.539391, 0.62526, 0.04525, 0.066437], [1, 0.511523, 0.628281, 0.041484, 0.068771], [1, 0.612852, 0.623781, 0.021047, 0.044479], [1, 0.669828, 0.635208, 0.09775, 0.083083], [1, 0.563984, 0.623375, 0.026312, 0.055208], [1, 0.587234, 0.619906, 0.03975, 0.056354], [1, 0.749133, 0.615833, 0.015328, 0.024333], [1, 0.679992, 0.607792, 0.071203, 0.040583], [5, 0.788547, 0.882719, 0.073375, 0.160479], [1, 0.737281, 0.628594, 0.022, 0.053604], [6, 0.718398, 0.396167, 0.112141, 0.148], [6, 0.145234, 0.487969, 0.040156, 0.055313]]}, {"filename": "000000291658.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.174625, 0.46068, 0.34925, 0.293234], [7, 0.536323, 0.308016, 0.179771, 0.310125], [0, 0.46675, 0.493156, 0.032833, 0.053937], [0, 0.505198, 0.491117, 0.036521, 0.072391], [1, 0.104042, 0.694187, 0.2065, 0.520969]]}, {"filename": "000000291664.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.379983, 0.411383, 0.313095, 0.619328]]}, {"filename": "000000292053.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.25388, 0.679627, 0.44768, 0.614358], [1, 0.04027, 0.663985, 0.0745, 0.47591], [1, 0.83891, 0.47594, 0.22174, 0.371821], [1, 0.61949, 0.284, 0.06158, 0.167522], [1, 0.24993, 0.380896, 0.16562, 0.213493], [1, 0.11581, 0.405537, 0.1691, 0.346537], [1, 0.72117, 0.300746, 0.09602, 0.152955], [1, 0.88355, 0.303582, 0.09706, 0.124836], [0, 0.41234, 0.491866, 0.2418, 0.562299], [0, 0.96061, 0.43997, 0.07878, 0.603522], [0, 0.16683, 0.17403, 0.10442, 0.246866], [0, 0.91122, 0.153239, 0.03276, 0.082179], [0, 0.71599, 0.243045, 0.06634, 0.263701], [0, 0.65244, 0.23206, 0.06548, 0.284239], [0, 0.49913, 0.390597, 0.19186, 0.619701], [0, 0.82579, 0.182955, 0.06878, 0.12394], [0, 0.78684, 0.14397, 0.04164, 0.084537], [0, 0.62143, 0.175478, 0.03902, 0.126179], [0, 0.31247, 0.138791, 0.02422, 0.064627], [0, 0.27912, 0.154463, 0.03252, 0.089821], [0, 0.73784, 0.136731, 0.01768, 0.045821], [1, 0.79641, 0.278567, 0.09446, 0.166328]]}, {"filename": "000000292058.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.760859, 0.405264, 0.126469, 0.192854], [6, 0.990586, 0.365032, 0.018828, 0.162156], [0, 0.314945, 0.528647, 0.058734, 0.284271], [0, 0.114523, 0.582378, 0.079578, 0.378245], [0, 0.610273, 0.545137, 0.077422, 0.368499], [0, 0.151375, 0.546533, 0.081687, 0.321395], [0, 0.557203, 0.509429, 0.068094, 0.292135], [1, 0.939859, 0.423679, 0.032594, 0.03315], [1, 0.325695, 0.480359, 0.131797, 0.123467], [0, 0.423172, 0.522537, 0.075906, 0.27167], [6, 0.734812, 0.229366, 0.141469, 0.19556], [6, 0.835273, 0.272304, 0.091266, 0.229514], [6, 0.89125, 0.196543, 0.065437, 0.196892], [6, 0.941977, 0.284841, 0.116047, 0.191924], [6, 0.857773, 0.395412, 0.041797, 0.112347], [6, 0.92543, 0.163277, 0.030922, 0.07814], [1, 0.897859, 0.42297, 0.024, 0.029535], [0, 0.204531, 0.531702, 0.06175, 0.311776], [6, 0.985156, 0.639006, 0.029687, 0.230994]]}, {"filename": "000000292146.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.25927, 0.304618, 0.03938, 0.051804], [1, 0.40266, 0.299771, 0.06584, 0.069694], [1, 0.33329, 0.291239, 0.03154, 0.046514], [6, 0.65997, 0.23604, 0.02418, 0.036116], [0, 0.74717, 0.479511, 0.13114, 0.419572], [0, 0.91948, 0.412584, 0.09724, 0.295321], [0, 0.53025, 0.361177, 0.05246, 0.192691], [0, 0.8001, 0.320382, 0.029, 0.058746], [0, 0.45837, 0.334786, 0.03146, 0.131407], [0, 0.10063, 0.340168, 0.04414, 0.067492], [0, 0.97637, 0.346651, 0.03394, 0.133425], [1, 0.45633, 0.781713, 0.43306, 0.402263], [1, 0.74295, 0.60156, 0.11786, 0.282997], [1, 0.91812, 0.540963, 0.081, 0.227064], [1, 0.65015, 0.376804, 0.04498, 0.07104], [1, 0.06393, 0.56292, 0.12786, 0.31685], [1, 0.52935, 0.409725, 0.0329, 0.118899], [1, 0.11647, 0.537737, 0.13138, 0.216636], [1, 0.45805, 0.37055, 0.0241, 0.082813], [1, 0.55261, 0.372462, 0.02166, 0.087798], [1, 0.25482, 0.755535, 0.19704, 0.334557], [1, 0.25072, 0.379113, 0.0352, 0.066667], [0, 0.55314, 0.299924, 0.02312, 0.058746], [0, 0.61869, 0.319755, 0.0153, 0.070275], [0, 0.64752, 0.310336, 0.02028, 0.063792], [0, 0.67298, 0.309434, 0.0262, 0.049388], [0, 0.33867, 0.302813, 0.03258, 0.077492], [1, 0.96923, 0.476972, 0.06154, 0.077187], [1, 0.26566, 0.294587, 0.0334, 0.04844], [0, 0.57647, 0.34737, 0.02266, 0.132661], [1, 0.62887, 0.73292, 0.13586, 0.215076]]}, {"filename": "000000292209.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.551906, 0.497975, 0.896187, 0.99595]]}, {"filename": "000000292358.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.444102, 0.759552, 0.264609, 0.453937], [1, 0.827664, 0.535125, 0.126734, 0.1015], [1, 0.526617, 0.53351, 0.484453, 0.300521], [0, 0.848547, 0.567562, 0.0265, 0.064167], [1, 0.528031, 0.533594, 0.483094, 0.302187], [0, 0.586125, 0.466646, 0.020437, 0.022417], [0, 0.009125, 0.602344, 0.01825, 0.152437], [0, 0.503539, 0.452677, 0.046516, 0.058438], [0, 0.517422, 0.438146, 0.014, 0.019083], [0, 0.164648, 0.70375, 0.314578, 0.58975], [0, 0.259438, 0.565844, 0.027281, 0.087854]]}, {"filename": "000000293245.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.589047, 0.61349, 0.648875, 0.651687]]}, {"filename": "000000293468.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.197313, 0.592086, 0.049125, 0.078625]]}, {"filename": "000000293474.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.50136, 0.637793, 0.15644, 0.375826]]}, {"filename": "000000293906.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.518547, 0.524375, 0.337094, 0.455917], [1, 0.091953, 0.320885, 0.120344, 0.051062], [1, 0.14032, 0.301948, 0.067766, 0.049687], [1, 0.657195, 0.303833, 0.070172, 0.0355], [1, 0.589203, 0.301021, 0.0725, 0.048958], [1, 0.715516, 0.300073, 0.082875, 0.034104], [1, 0.823828, 0.302052, 0.052812, 0.027021], [1, 0.224219, 0.306, 0.08675, 0.038542], [1, 0.559102, 0.305469, 0.030422, 0.036354], [1, 0.783687, 0.29949, 0.047656, 0.029187], [1, 0.752469, 0.301073, 0.054281, 0.033854]]}, {"filename": "000000294349.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.682117, 0.491265, 0.519422, 0.977611]]}, {"filename": "000000295138.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.428531, 0.408428, 0.350812, 0.815437], [3, 0.363586, 0.676253, 0.143953, 0.170709]]}, {"filename": "000000295809.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.916531, 0.242598, 0.166312, 0.061523], [1, 0.479937, 0.251855, 0.034594, 0.035117], [1, 0.512297, 0.247969, 0.018094, 0.026641], [1, 0.561008, 0.251982, 0.012203, 0.014043], [0, 0.393977, 0.245283, 0.011203, 0.052754], [1, 0.461164, 0.766514, 0.529328, 0.395059], [1, 0.157516, 0.238447, 0.157844, 0.073418], [1, 0.49118, 0.2475, 0.028172, 0.037344], [0, 0.460992, 0.244951, 0.008578, 0.032715], [5, 0.679891, 0.261924, 0.005312, 0.013926]]}, {"filename": "000000296002.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.15068, 0.138827, 0.30136, 0.27088], [4, 0.50097, 0.041667, 0.36914, 0.075547], [4, 0.05347, 0.09792, 0.10646, 0.1952], [4, 0.958, 0.095333, 0.084, 0.190667]]}, {"filename": "000000296128.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.293859, 0.520972, 0.251875, 0.723583], [1, 0.485227, 0.262272, 0.247141, 0.212319], [3, 0.655227, 0.420667, 0.511609, 0.271311], [1, 0.681719, 0.299192, 0.151594, 0.122272]]}, {"filename": "000000296489.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.488, 0.644218, 0.666667, 0.516047], [1, 0.748644, 0.888599, 0.502711, 0.222802], [1, 0.167944, 0.475944, 0.279444, 0.258083], [1, 0.079078, 0.407124, 0.148156, 0.171829], [1, 0.043356, 0.340516, 0.086711, 0.115959], [5, 0.827067, 0.661932, 0.1412, 0.239027], [0, 0.947611, 0.403584, 0.104778, 0.403569], [0, 0.711478, 0.350546, 0.043844, 0.203687], [0, 0.839022, 0.356445, 0.064622, 0.257375], [0, 0.565022, 0.320546, 0.067556, 0.213776], [0, 0.907067, 0.365324, 0.0528, 0.258319]]}, {"filename": "000000296649.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.834305, 0.880293, 0.331391, 0.218431], [1, 0.384422, 0.699415, 0.054344, 0.075831], [1, 0.560258, 0.703618, 0.031203, 0.058759], [1, 0.540492, 0.884508, 0.270766, 0.230984], [1, 0.136891, 0.880222, 0.272469, 0.239555], [0, 0.554867, 0.830492, 0.101703, 0.298876], [0, 0.467367, 0.835913, 0.079609, 0.30363], [0, 0.092414, 0.807436, 0.178828, 0.364848], [0, 0.746648, 0.785386, 0.167922, 0.303372], [0, 0.425437, 0.69918, 0.040656, 0.082201], [0, 0.451273, 0.695855, 0.024234, 0.096768], [1, 0.421617, 0.809953, 0.040703, 0.159204], [1, 0.383406, 0.689719, 0.052281, 0.104215], [0, 0.178336, 0.660609, 0.029391, 0.068056], [0, 0.201563, 0.664684, 0.025437, 0.059485], [0, 0.411781, 0.708033, 0.02, 0.098407], [0, 0.427258, 0.66356, 0.013891, 0.041639], [1, 0.776563, 0.897459, 0.239125, 0.106628], [0, 0.894258, 0.778501, 0.050141, 0.108009], [0, 0.845266, 0.795258, 0.143875, 0.295246], [1, 0.386297, 0.654754, 0.050938, 0.038501]]}, {"filename": "000000296994.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.606615, 0.440312, 0.354354, 0.540531], [0, 0.195823, 0.190352, 0.087979, 0.182234], [1, 0.498042, 0.217398, 0.049167, 0.076859], [1, 0.778052, 0.165258, 0.121521, 0.046266], [3, 0.026875, 0.190734, 0.048042, 0.026562], [3, 0.056323, 0.183758, 0.038271, 0.026641], [1, 0.294177, 0.166633, 0.035146, 0.027016], [1, 0.808323, 0.181906, 0.024729, 0.011]]}, {"filename": "000000297022.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.518531, 0.471885, 0.782188, 0.4475], [0, 0.450063, 0.471875, 0.0885, 0.386448], [0, 0.245563, 0.533879, 0.101594, 0.242837], [0, 0.341602, 0.539077, 0.083734, 0.198194]]}, {"filename": "000000297323.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.8102, 0.795908, 0.22056, 0.170483], [0, 0.9536, 0.802069, 0.03592, 0.062069], [6, 0.2612, 0.697034, 0.04824, 0.062391], [1, 0.50636, 0.822471, 0.34408, 0.224713], [1, 0.50884, 0.82692, 0.34812, 0.213747], [0, 0.35354, 0.812322, 0.02652, 0.042989]]}, {"filename": "000000297387.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.599695, 0.593062, 0.048391, 0.022833], [1, 0.680016, 0.593365, 0.047625, 0.027813], [1, 0.823828, 0.595177, 0.038438, 0.017729], [1, 0.532898, 0.595052, 0.054609, 0.023771], [1, 0.92818, 0.596021, 0.033016, 0.01775], [1, 0.989742, 0.594323, 0.020516, 0.015104], [1, 0.772359, 0.59576, 0.036688, 0.017604], [1, 0.345836, 0.587125, 0.073391, 0.038], [0, 0.316305, 0.699458, 0.042109, 0.096792], [0, 0.105469, 0.583854, 0.013031, 0.030542], [0, 0.632867, 0.614417, 0.014766, 0.024708], [1, 0.637789, 0.595021, 0.035266, 0.019833], [1, 0.7255, 0.596625, 0.04025, 0.021292], [1, 0.881242, 0.594354, 0.030297, 0.021542], [0, 0.317984, 0.619656, 0.011844, 0.024229], [0, 0.624422, 0.620115, 0.008344, 0.018729], [1, 0.958086, 0.596187, 0.023953, 0.016958]]}, {"filename": "000000297562.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.721352, 0.81124, 0.208984, 0.346063], [0, 0.818789, 0.330354, 0.117453, 0.501125], [0, 0.673531, 0.319594, 0.105156, 0.455354], [0, 0.698367, 0.246573, 0.050109, 0.265146], [0, 0.95425, 0.162094, 0.0915, 0.320729], [0, 0.905836, 0.510073, 0.187797, 0.710688], [0, 0.510133, 0.708896, 0.324328, 0.582208], [0, 0.679945, 0.14501, 0.040828, 0.099146], [0, 0.384945, 0.305167, 0.054672, 0.201625], [1, 0.44832, 0.231167, 0.045484, 0.052625], [0, 0.336234, 0.715729, 0.618531, 0.568542], [0, 0.723594, 0.3935, 0.056375, 0.181208], [4, 0.625328, 0.423375, 0.120781, 0.077917]]}, {"filename": "000000297944.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.704336, 0.644677, 0.050672, 0.022354], [1, 0.665875, 0.642083, 0.035813, 0.023625], [5, 0.875875, 0.742427, 0.025156, 0.093104], [1, 0.866742, 0.65475, 0.082578, 0.034958], [1, 0.774094, 0.661427, 0.062875, 0.047021], [1, 0.95675, 0.644833, 0.084094, 0.058833], [6, 0.81407, 0.522156, 0.079422, 0.104729], [1, 0.99818, 0.688531, 0.003641, 0.043979], [1, 0.66368, 0.634896, 0.037016, 0.013292], [1, 0.790023, 0.637927, 0.057547, 0.023396]]}, {"filename": "000000299970.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.35049, 0.617664, 0.533937, 0.344516], [1, 0.560708, 0.474258, 0.09325, 0.060109], [1, 0.556271, 0.502734, 0.0515, 0.059844], [0, 0.144198, 0.460203, 0.023854, 0.032344], [0, 0.129115, 0.471516, 0.023354, 0.036781], [0, 0.614573, 0.489195, 0.035521, 0.089328], [0, 0.097896, 0.458289, 0.020917, 0.025422], [1, 0.14975, 0.484039, 0.059042, 0.050734], [1, 0.556042, 0.442672, 0.029042, 0.00725], [7, 0.726854, 0.516617, 0.052083, 0.211703], [1, 0.06774, 0.544664, 0.134229, 0.179734], [5, 0.65351, 0.523398, 0.030729, 0.065266]]}, {"filename": "000000300811.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.170859, 0.677411, 0.124937, 0.150536], [4, 0.411031, 0.537701, 0.06975, 0.045536], [4, 0.579688, 0.794096, 0.16875, 0.165647], [0, 0.823016, 0.714286, 0.350781, 0.489777], [0, 0.364945, 0.657299, 0.283141, 0.325848], [0, 0.096797, 0.555413, 0.193594, 0.310112], [4, 0.952141, 0.933906, 0.095719, 0.100134], [4, 0.067625, 0.688404, 0.057937, 0.040871], [0, 0.813875, 0.312388, 0.069594, 0.084643], [4, 0.647461, 0.635134, 0.123359, 0.084643], [4, 0.370797, 0.772868, 0.213438, 0.118862], [4, 0.807766, 0.658092, 0.122313, 0.091272], [4, 0.234422, 0.562344, 0.072906, 0.0625], [4, 0.397039, 0.590915, 0.038391, 0.064732], [4, 0.765609, 0.546853, 0.084313, 0.121116]]}, {"filename": "000000300848.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.514648, 0.544026, 0.208078, 0.533545], [0, 0.633828, 0.536009, 0.153281, 0.535352], [0, 0.712516, 0.505939, 0.134563, 0.593709], [0, 0.29093, 0.489519, 0.345516, 0.777535], [0, 0.217641, 0.479777, 0.08825, 0.321338], [0, 0.265289, 0.519214, 0.073547, 0.257254], [0, 0.082797, 0.499742, 0.115781, 0.602394], [0, 0.149352, 0.512019, 0.041422, 0.240282], [3, 0.864016, 0.510411, 0.271969, 0.320305]]}, {"filename": "000000301262.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.504875, 0.229219, 0.500375, 0.373031], [5, 0.794896, 0.931875, 0.087875, 0.125]]}, {"filename": "000000301376.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.823709, 0.79045, 0.167177, 0.09014], [1, 0.14952, 0.87036, 0.29009, 0.17156], [0, 0.933198, 0.81006, 0.123514, 0.15264], [1, 0.772703, 0.7778, 0.136396, 0.093], [1, 0.920571, 0.74199, 0.090991, 0.03262], [0, 0.580195, 0.76699, 0.066456, 0.1041], [0, 0.637718, 0.77078, 0.026967, 0.0764], [0, 0.607823, 0.76467, 0.033003, 0.09206], [7, 0.662147, 0.74882, 0.016847, 0.02304], [1, 0.955465, 0.84895, 0.068709, 0.10534], [1, 0.857402, 0.73379, 0.037027, 0.02082], [1, 0.98509, 0.771, 0.02982, 0.06164], [1, 0.68967, 0.76181, 0.087628, 0.06954]]}, {"filename": "000000301421.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.14707, 0.829028, 0.276922, 0.311827], [1, 0.525633, 0.294941, 0.126328, 0.135644], [4, 0.938266, 0.728677, 0.123469, 0.542646]]}, {"filename": "000000301817.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.987594, 0.88701, 0.024812, 0.042521], [1, 0.008023, 0.84701, 0.016047, 0.047312], [0, 0.967648, 0.871448, 0.019578, 0.037229], [0, 0.862461, 0.876969, 0.012766, 0.061063], [0, 0.039477, 0.84351, 0.009703, 0.051437], [0, 0.964867, 0.917219, 0.010547, 0.054479], [0, 0.050703, 0.837542, 0.011156, 0.043292], [0, 0.02425, 0.843042, 0.010625, 0.047667], [0, 0.066539, 0.842729, 0.013578, 0.064458], [0, 0.075836, 0.842052, 0.012609, 0.053562]]}, {"filename": "000000301851.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.662101, 0.849211, 0.289038, 0.176734]]}, {"filename": "000000301998.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.30975, 0.507898, 0.4065, 0.932432], [0, 0.71442, 0.568844, 0.29596, 0.656847], [3, 0.87056, 0.728393, 0.25888, 0.505075], [3, 0.681, 0.470736, 0.288, 0.281532], [5, 0.62714, 0.026426, 0.0082, 0.023904], [4, 0.87328, 0.73033, 0.25292, 0.512372], [4, 0.68372, 0.477117, 0.28436, 0.283183], [1, 0.45104, 0.020135, 0.04544, 0.04027], [1, 0.37521, 0.019489, 0.03066, 0.031712], [1, 0.34593, 0.026036, 0.0333, 0.025586], [1, 0.31729, 0.022207, 0.04114, 0.036186], [1, 0.0638, 0.049189, 0.03196, 0.030751], [1, 0.09309, 0.048318, 0.01978, 0.019399], [3, 0.84098, 0.567222, 0.28776, 0.398529], [1, 0.28127, 0.023348, 0.03958, 0.029099]]}, {"filename": "000000302318.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.95738, 0.619573, 0.08524, 0.125867], [1, 0.87733, 0.586707, 0.10526, 0.098853], [1, 0.81045, 0.567427, 0.0591, 0.06904], [1, 0.7458, 0.5454, 0.0468, 0.051653], [1, 0.70846, 0.54372, 0.03572, 0.040293], [0, 0.89095, 0.617093, 0.04146, 0.07696], [0, 0.77957, 0.663733, 0.06534, 0.274987], [0, 0.90235, 0.63752, 0.07162, 0.110827], [3, 0.91842, 0.71036, 0.08516, 0.121947], [0, 0.88549, 0.701773, 0.09218, 0.098587], [0, 0.68564, 0.553147, 0.01144, 0.029173], [0, 0.6464, 0.552027, 0.01176, 0.037013], [1, 0.84584, 0.538067, 0.03412, 0.015867], [3, 0.68428, 0.597893, 0.02084, 0.050667], [1, 0.9289, 0.54232, 0.04712, 0.03216], [1, 0.98469, 0.54836, 0.03062, 0.018747]]}, {"filename": "000000302319.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.491511, 0.568445, 0.405738, 0.680984]]}, {"filename": "000000302882.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.38837, 0.569462, 0.77674, 0.861075]]}, {"filename": "000000303713.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.407553, 0.749437, 0.397447, 0.478656], [0, 0.090105, 0.685867, 0.066112, 0.048609], [0, 0.730656, 0.737688, 0.176674, 0.160219], [0, 0.855492, 0.743133, 0.116089, 0.158766], [0, 0.691358, 0.591609, 0.054192, 0.049281], [0, 0.202881, 0.757508, 0.157658, 0.176172], [0, 0.175199, 0.53975, 0.106276, 0.077969], [0, 0.633372, 0.46532, 0.05541, 0.048078], [0, 0.784731, 0.331602, 0.075597, 0.047047], [0, 0.023255, 0.684281, 0.046511, 0.056125], [4, 0.136159, 0.754406, 0.049602, 0.060156], [4, 0.799977, 0.741594, 0.037424, 0.051844], [4, 0.815433, 0.708117, 0.021218, 0.013984], [4, 0.58959, 0.780656, 0.085644, 0.114906], [4, 0.388173, 0.775844, 0.047494, 0.052812], [4, 0.821663, 0.668164, 0.060047, 0.014141], [0, 0.734543, 0.490203, 0.103607, 0.062656], [0, 0.575457, 0.621859, 0.081499, 0.069781], [0, 0.333115, 0.541703, 0.064543, 0.051406]]}, {"filename": "000000303863.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.973984, 0.596369, 0.031938, 0.186798], [0, 0.909227, 0.582436, 0.049078, 0.202923], [0, 0.212633, 0.486427, 0.017234, 0.044176], [0, 0.186234, 0.482262, 0.012875, 0.033341], [0, 0.03393, 0.494095, 0.011234, 0.016311], [0, 0.792375, 0.454698, 0.026187, 0.055522], [0, 0.969047, 0.578643, 0.05525, 0.224524]]}, {"filename": "000000303897.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.3566, 0.524727, 0.41292, 0.698879], [0, 0.5267, 0.484052, 0.08668, 0.151437], [0, 0.64098, 0.598463, 0.16496, 0.320661], [0, 0.79799, 0.56681, 0.07494, 0.33023], [0, 0.86883, 0.560388, 0.14454, 0.361178], [0, 0.10245, 0.580905, 0.14858, 0.433707], [0, 0.45348, 0.657428, 0.18436, 0.209109], [3, 0.81537, 0.649468, 0.36926, 0.10181], [3, 0.20889, 0.674813, 0.1433, 0.106408], [3, 0.37358, 0.708017, 0.13644, 0.023678]]}, {"filename": "000000303946.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.621375, 0.730323, 0.336, 0.355313], [1, 0.16507, 0.853833, 0.330141, 0.268625], [0, 0.484008, 0.730177, 0.187297, 0.430188], [0, 0.149164, 0.741573, 0.237641, 0.440437]]}, {"filename": "000000305460.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.721852, 0.641909, 0.125766, 0.114403], [4, 0.658945, 0.924719, 0.370328, 0.150562], [4, 0.921977, 0.910386, 0.156047, 0.153232], [4, 0.691844, 0.821838, 0.211969, 0.084801], [4, 0.598, 0.722752, 0.123031, 0.25623]]}, {"filename": "000000305788.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.454104, 0.585313, 0.904, 0.827313]]}, {"filename": "000000305959.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.058023, 0.786148, 0.116047, 0.157307], [1, 0.942641, 0.718434, 0.113687, 0.058455], [1, 0.920781, 0.693215, 0.091406, 0.040042], [1, 0.856312, 0.695219, 0.077969, 0.030438], [0, 0.228297, 0.769666, 0.034125, 0.117119], [3, 0.811805, 0.756649, 0.067234, 0.038225], [0, 0.243266, 0.766441, 0.020031, 0.109332], [3, 0.475508, 0.778111, 0.074547, 0.042944], [4, 0.808125, 0.793987, 0.053406, 0.064426], [0, 0.312492, 0.824614, 0.071547, 0.251232], [6, 0.904211, 0.651127, 0.018891, 0.033194]]}, {"filename": "000000306136.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.142103, 0.786859, 0.116262, 0.187906], [1, 0.616612, 0.732023, 0.762804, 0.451672], [0, 0.256098, 0.700203, 0.063692, 0.058719], [0, 0.645432, 0.662859, 0.112126, 0.082875], [0, 0.928049, 0.669875, 0.050397, 0.070781], [0, 0.228178, 0.769313, 0.080234, 0.153687], [0, 0.711238, 0.647078, 0.045, 0.106813]]}, {"filename": "000000306207.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.26175, 0.593867, 0.289125, 0.112922], [1, 0.051896, 0.609445, 0.103625, 0.065703], [1, 0.626375, 0.566594, 0.074417, 0.027813], [1, 0.414062, 0.581078, 0.13625, 0.049188], [1, 0.973229, 0.603992, 0.053542, 0.091641], [1, 0.054406, 0.583125, 0.107813, 0.038812], [0, 0.575385, 0.55225, 0.009604, 0.012594], [1, 0.25851, 0.5935, 0.297687, 0.116156], [1, 0.71325, 0.566523, 0.078125, 0.057297], [7, 0.674958, 0.584031, 0.018292, 0.023969], [0, 0.131865, 0.663984, 0.059229, 0.172594], [1, 0.739365, 0.530875, 0.107896, 0.020125], [1, 0.498396, 0.569523, 0.0845, 0.038609], [1, 0.564375, 0.566398, 0.064167, 0.034047], [1, 0.118906, 0.53718, 0.129437, 0.053578]]}, {"filename": "000000306437.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.497207, 0.579492, 0.181502, 0.201359], [1, 0.288721, 0.618406, 0.161385, 0.063219], [1, 0.381784, 0.617812, 0.060188, 0.057844], [1, 0.98007, 0.629047, 0.039859, 0.040344], [1, 0.943615, 0.619844, 0.08662, 0.066656], [3, 0.688392, 0.823813, 0.623216, 0.276187]]}, {"filename": "000000306893.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.05932, 0.758478, 0.11472, 0.22288], [1, 0.95115, 0.765611, 0.0977, 0.216984], [5, 0.1831, 0.692677, 0.0406, 0.191277], [5, 0.82649, 0.707092, 0.0339, 0.178043], [0, 0.23006, 0.524918, 0.04084, 0.065435], [0, 0.15451, 0.533832, 0.02066, 0.03875], [0, 0.12603, 0.531739, 0.02518, 0.044239], [0, 0.34031, 0.541291, 0.01662, 0.024266], [0, 0.84954, 0.551508, 0.02708, 0.039049], [0, 0.2019, 0.54356, 0.0098, 0.021087], [0, 0.70527, 0.549253, 0.00906, 0.030027], [0, 0.80084, 0.556848, 0.01116, 0.021522], [0, 0.8805, 0.556576, 0.01316, 0.023207]]}, {"filename": "000000307074.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.84717, 0.856982, 0.01842, 0.02964], [1, 0.31143, 0.853679, 0.03622, 0.036787], [1, 0.45172, 0.834144, 0.03548, 0.030571], [1, 0.40391, 0.841426, 0.03742, 0.028799], [1, 0.34911, 0.852342, 0.0467, 0.033694], [1, 0.37095, 0.837327, 0.02694, 0.023243], [1, 0.26793, 0.867943, 0.05062, 0.043333], [1, 0.13157, 0.835946, 0.0403, 0.029129], [1, 0.49158, 0.808844, 0.01228, 0.014144], [1, 0.44852, 0.8203, 0.02036, 0.014775], [1, 0.2716, 0.822057, 0.01356, 0.015706], [1, 0.25467, 0.823213, 0.01514, 0.020841], [1, 0.60914, 0.846456, 0.06324, 0.034955]]}, {"filename": "000000307194.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.937458, 0.714227, 0.125083, 0.386516], [1, 0.14374, 0.849859, 0.287479, 0.300281], [1, 0.062333, 0.423883, 0.11775, 0.470141], [7, 0.502167, 0.531172, 0.768833, 0.9065]]}, {"filename": "000000307359.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.590727, 0.508009, 0.224891, 0.903372]]}, {"filename": "000000307652.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.37932, 0.674599, 0.214828, 0.628113], [0, 0.50618, 0.499634, 0.053609, 0.191014], [3, 0.802727, 0.666167, 0.194797, 0.166203], [4, 0.248594, 0.389552, 0.07, 0.107311]]}, {"filename": "000000307949.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.683195, 0.579407, 0.624328, 0.434672], [1, 0.162156, 0.469356, 0.324313, 0.28846], [5, 0.315523, 0.711629, 0.035984, 0.108308]]}, {"filename": "000000307993.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.556242, 0.494254, 0.887516, 0.983893], [0, 0.520102, 0.470571, 0.327609, 0.680163]]}, {"filename": "000000308575.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.108242, 0.200958, 0.216484, 0.331708]]}, {"filename": "000000308631.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.46832, 0.512523, 0.865172, 0.791948], [0, 0.843062, 0.351068, 0.171594, 0.447911], [0, 0.717977, 0.17865, 0.107703, 0.352793], [0, 0.454602, 0.121338, 0.140141, 0.242676], [0, 0.317492, 0.133979, 0.124234, 0.267958], [0, 0.948281, 0.358275, 0.103438, 0.707488], [0, 0.10682, 0.088862, 0.057891, 0.177723], [0, 0.176117, 0.082207, 0.104422, 0.164178], [0, 0.913547, 0.077371, 0.085094, 0.15108], [0, 0.5925, 0.103897, 0.173687, 0.207793], [0, 0.031664, 0.310235, 0.063078, 0.617465]]}, {"filename": "000000308641.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.393467, 0.371531, 0.291698, 0.40225], [7, 0.708927, 0.406742, 0.393467, 0.494391]]}, {"filename": "000000308687.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.357328, 0.473844, 0.229656, 0.261229], [4, 0.905516, 0.429635, 0.188969, 0.259104], [0, 0.245703, 0.617875, 0.373781, 0.761625], [0, 0.845289, 0.441312, 0.230859, 0.290583], [4, 0.074695, 0.136146, 0.149391, 0.272292]]}, {"filename": "000000308847.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.471289, 0.472969, 0.516891, 0.281521], [7, 0.107937, 0.461531, 0.09925, 0.186896], [0, 0.763969, 0.558396, 0.124281, 0.418042], [0, 0.864164, 0.387583, 0.019797, 0.040583], [0, 0.976359, 0.392573, 0.022719, 0.069979], [0, 0.396305, 0.635708, 0.113922, 0.349], [0, 0.954602, 0.390958, 0.018516, 0.064292], [0, 0.88482, 0.396917, 0.026297, 0.075958]]}, {"filename": "000000309173.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.568104, 0.38513, 0.766418, 0.6419], [0, 0.283149, 0.78874, 0.082179, 0.09056]]}, {"filename": "000000309964.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.923555, 0.380527, 0.152891, 0.321335], [0, 0.26582, 0.597635, 0.248516, 0.753958], [0, 0.676156, 0.605749, 0.287219, 0.764567]]}, {"filename": "000000310524.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.313906, 0.598865, 0.042219, 0.073354], [0, 0.350102, 0.579083, 0.036703, 0.073417], [0, 0.294055, 0.412042, 0.015453, 0.0425], [0, 0.325234, 0.396667, 0.014438, 0.024542], [0, 0.314156, 0.412781, 0.023844, 0.042146], [3, 0.214047, 0.600156, 0.417969, 0.215729], [3, 0.11807, 0.770031, 0.163047, 0.126229], [3, 0.187648, 0.554438, 0.090984, 0.057375], [3, 0.180531, 0.558063, 0.360312, 0.160708], [3, 0.239508, 0.485917, 0.066703, 0.04075], [3, 0.175781, 0.507729, 0.077844, 0.0505], [3, 0.073078, 0.538146, 0.140031, 0.054875], [3, 0.243734, 0.682615, 0.127656, 0.091937], [3, 0.434984, 0.565823, 0.145687, 0.077438], [0, 0.867203, 0.406458, 0.020562, 0.073375], [0, 0.039961, 0.540177, 0.053734, 0.096604], [0, 0.107836, 0.539687, 0.042047, 0.069667], [0, 0.879359, 0.412469, 0.024594, 0.078688], [0, 0.431109, 0.328396, 0.010062, 0.020375], [0, 0.980953, 0.4055, 0.024469, 0.062042], [0, 0.808367, 0.401854, 0.025547, 0.038333]]}, {"filename": "000000310862.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.476405, 0.495507, 0.934837, 0.977516]]}, {"filename": "000000314174.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.164117, 0.495508, 0.317859, 0.97752], [0, 0.744914, 0.404644, 0.500578, 0.808801], [0, 0.518563, 0.144299, 0.356813, 0.288598]]}, {"filename": "000000316441.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.18327, 0.510289, 0.36654, 0.917234], [7, 0.46382, 0.228131, 0.03848, 0.125441], [7, 0.4321, 0.218116, 0.02732, 0.113556], [0, 0.47297, 0.462675, 0.16954, 0.145532], [1, 0.82223, 0.494498, 0.35322, 0.988997]]}, {"filename": "000000316667.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.43161, 0.445413, 0.21362, 0.805382], [0, 0.83685, 0.322508, 0.06706, 0.225566], [0, 0.90873, 0.317064, 0.08978, 0.184709], [0, 0.79646, 0.360596, 0.09004, 0.207676], [3, 0.51995, 0.780459, 0.35594, 0.226422], [3, 0.16093, 0.588777, 0.3101, 0.287645], [3, 0.11223, 0.496131, 0.22446, 0.179969]]}, {"filename": "000000317622.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.33632, 0.658302, 0.326234, 0.399104], [1, 0.637391, 0.626646, 0.310406, 0.231625], [1, 0.871078, 0.613229, 0.154719, 0.132292], [0, 0.899508, 0.66276, 0.036797, 0.117271], [0, 0.953063, 0.661615, 0.029031, 0.119396], [0, 0.842453, 0.655219, 0.026156, 0.093646], [0, 0.083469, 0.748563, 0.064031, 0.217042], [0, 0.019422, 0.699146, 0.036094, 0.164708], [1, 0.095711, 0.661156, 0.036484, 0.063646], [1, 0.116141, 0.65824, 0.032438, 0.056479], [0, 0.058805, 0.701604, 0.036297, 0.167792], [1, 0.044812, 0.66951, 0.035781, 0.083813], [1, 0.152117, 0.660177, 0.058047, 0.065812], [1, 0.159266, 0.633, 0.052125, 0.031583], [1, 0.154984, 0.651125, 0.054719, 0.041583], [0, 0.105742, 0.625562, 0.008453, 0.011], [0, 0.102211, 0.625229, 0.005172, 0.0125], [0, 0.910117, 0.67951, 0.010922, 0.074771]]}, {"filename": "000000318464.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.479133, 0.741358, 0.119609, 0.209508], [1, 0.696719, 0.770855, 0.09375, 0.2263], [3, 0.409313, 0.701593, 0.091469, 0.04637], [3, 0.681203, 0.618162, 0.043594, 0.037916]]}, {"filename": "000000319369.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.966445, 0.478924, 0.067109, 0.062014], [1, 0.930891, 0.471933, 0.056344, 0.044236], [1, 0.392867, 0.413484, 0.100109, 0.04044], [0, 0.799789, 0.616134, 0.103359, 0.281019], [0, 0.480656, 0.669236, 0.109781, 0.375833], [0, 0.564852, 0.681285, 0.067641, 0.373264], [0, 0.42882, 0.719826, 0.060109, 0.211042], [4, 0.335469, 0.743762, 0.094656, 0.191319], [4, 0.41343, 0.708356, 0.051359, 0.10662], [0, 0.356469, 0.736725, 0.083781, 0.183032], [1, 0.889219, 0.478194, 0.019094, 0.013426], [1, 0.543438, 0.463299, 0.067781, 0.051181]]}, {"filename": "000000319785.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.381737, 0.39215, 0.092275, 0.14086], [7, 0.234102, 0.4505, 0.03012, 0.05204], [7, 0.273877, 0.43676, 0.047814, 0.07756], [7, 0.215749, 0.46075, 0.022036, 0.03774]]}, {"filename": "000000319935.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.699984, 0.580528, 0.070656, 0.138342], [4, 0.601406, 0.562889, 0.053781, 0.124975], [4, 0.283672, 0.760578, 0.239469, 0.367186]]}, {"filename": "000000320171.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.420414, 0.901565, 0.068078, 0.123588], [1, 0.246547, 0.787634, 0.331187, 0.146069], [1, 0.8285, 0.739666, 0.069906, 0.059103], [1, 0.966109, 0.765239, 0.038469, 0.048989], [1, 0.782641, 0.738254, 0.05925, 0.04792], [1, 0.729719, 0.734599, 0.050781, 0.038664], [1, 0.574797, 0.722643, 0.01275, 0.023149], [1, 0.551187, 0.721603, 0.018625, 0.014924], [1, 0.912141, 0.751508, 0.081031, 0.068664], [1, 0.669117, 0.730115, 0.053453, 0.037366], [1, 0.635828, 0.726908, 0.034687, 0.034466], [1, 0.147422, 0.719332, 0.018031, 0.015344], [1, 0.031469, 0.720506, 0.031219, 0.020553], [1, 0.083547, 0.718769, 0.0315, 0.023531]]}, {"filename": "000000320290.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.523281, 0.825843, 0.056813, 0.141571], [0, 0.840016, 0.686456, 0.029, 0.071111], [1, 0.546867, 0.720316, 0.234359, 0.171475], [1, 0.653867, 0.700354, 0.090016, 0.123467], [1, 0.698063, 0.683994, 0.053469, 0.087644], [1, 0.227, 0.68932, 0.125562, 0.108985], [1, 0.440305, 0.672232, 0.083047, 0.064157], [7, 0.627172, 0.676082, 0.015906, 0.043582], [7, 0.769969, 0.650287, 0.007875, 0.013027], [1, 0.349008, 0.680479, 0.126984, 0.084444], [1, 0.045602, 0.704895, 0.063891, 0.124962], [1, 0.731023, 0.670517, 0.056828, 0.068084], [1, 0.777906, 0.666446, 0.041875, 0.029904], [1, 0.493852, 0.661331, 0.034109, 0.047414], [1, 0.39157, 0.649483, 0.031922, 0.024368]]}, {"filename": "000000320554.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.52382, 0.371667, 0.8862, 0.73992]]}, {"filename": "000000321195.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.480336, 0.666729, 0.940453, 0.521333], [1, 0.25618, 0.328531, 0.508984, 0.182021], [1, 0.631727, 0.316958, 0.094078, 0.1095], [1, 0.937781, 0.314271, 0.124437, 0.090458], [1, 0.575859, 0.315865, 0.125188, 0.093271], [1, 0.493773, 0.294187, 0.105359, 0.088417], [1, 0.191711, 0.501052, 0.383422, 0.355604], [7, 0.795867, 0.339865, 0.031734, 0.100688], [1, 0.290234, 0.273781, 0.037688, 0.021688], [7, 0.771773, 0.338969, 0.028297, 0.093521], [7, 0.268336, 0.269417, 0.008297, 0.006125], [1, 0.407023, 0.308667, 0.022016, 0.059625], [1, 0.894203, 0.281042, 0.041188, 0.028042], [1, 0.896742, 0.256177, 0.062141, 0.014146], [1, 0.261109, 0.329625, 0.522219, 0.2], [6, 0.117742, 0.235479, 0.010172, 0.0165], [7, 0.427875, 0.329292, 0.025781, 0.072833], [1, 0.318305, 0.279896, 0.046266, 0.028583], [1, 0.510461, 0.320979, 0.042391, 0.065583]]}, {"filename": "000000321307.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.514898, 0.524717, 0.932734, 0.865175], [1, 0.817805, 0.199825, 0.325203, 0.199784], [1, 0.247977, 0.289434, 0.494172, 0.578869]]}, {"filename": "000000321557.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.485101, 0.50843, 0.938647, 0.943828]]}, {"filename": "000000322023.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.28343, 0.265606, 0.12666, 0.119091], [5, 0.46483, 0.728766, 0.12886, 0.245411], [5, 0.95465, 0.457457, 0.03806, 0.098766]]}, {"filename": "000000322610.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.269664, 0.60224, 0.205609, 0.750562], [0, 0.094344, 0.385396, 0.05675, 0.227667], [0, 0.16732, 0.416156, 0.057016, 0.304271], [0, 0.195062, 0.366167, 0.047031, 0.176458], [0, 0.368938, 0.41126, 0.032281, 0.345771], [0, 0.672469, 0.696635, 0.222469, 0.584271], [0, 0.138156, 0.377104, 0.034781, 0.1615], [0, 0.008617, 0.31849, 0.017078, 0.039271], [3, 0.827531, 0.882771, 0.343812, 0.21125], [0, 0.927258, 0.584698, 0.145484, 0.583063], [0, 0.011234, 0.56426, 0.022469, 0.255729], [0, 0.043453, 0.421792, 0.069156, 0.325125], [0, 0.352805, 0.439438, 0.039016, 0.182917], [0, 0.017453, 0.377823, 0.034906, 0.199438]]}, {"filename": "000000322725.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.217776, 0.110797, 0.246518, 0.118687], [0, 0.210282, 0.626125, 0.407012, 0.722969], [0, 0.950729, 0.110484, 0.098541, 0.217344], [1, 0.361082, 0.145203, 0.1736, 0.106469], [1, 0.160282, 0.213727, 0.245459, 0.131953], [0, 0.650788, 0.493844, 0.698424, 0.974656], [1, 0.360518, 0.156297, 0.186165, 0.084594], [0, 0.408294, 0.286469, 0.211129, 0.140187]]}, {"filename": "000000322844.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.5015, 0.794383, 0.997, 0.411234]]}, {"filename": "000000322895.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.256242, 0.453052, 0.036859, 0.110979], [4, 0.401187, 0.543885, 0.147625, 0.152937], [4, 0.540727, 0.528573, 0.130016, 0.136563]]}, {"filename": "000000323020.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.242859, 0.243062, 0.085531, 0.0775], [1, 0.4595, 0.229635, 0.054219, 0.085188], [5, 0.500281, 0.535552, 0.997188, 0.900687]]}, {"filename": "000000323151.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.668208, 0.03175, 0.033167, 0.0635]]}, {"filename": "000000323496.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.49863, 0.133744, 0.07726, 0.259598], [0, 0.41967, 0.14152, 0.06954, 0.27706], [0, 0.20885, 0.140528, 0.07422, 0.28005], [0, 0.32091, 0.099095, 0.06094, 0.186935], [0, 0.7212, 0.074171, 0.06464, 0.148342], [3, 0.88544, 0.186508, 0.22896, 0.07191], [0, 0.7123, 0.490151, 0.39888, 0.69799], [3, 0.9116, 0.14853, 0.1768, 0.041834], [3, 0.94206, 0.118291, 0.11584, 0.028291]]}, {"filename": "000000324116.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.683399, 0.30562, 0.489848, 0.48988]]}, {"filename": "000000324682.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.31183, 0.684632, 0.087288, 0.048546], [4, 0.363448, 0.660286, 0.11232, 0.087761], [4, 0.544101, 0.641781, 0.059248, 0.034575], [0, 0.632606, 0.588342, 0.107631, 0.12165], [0, 0.698472, 0.583783, 0.042402, 0.077827], [0, 0.053758, 0.607141, 0.047484, 0.149641], [4, 0.132369, 0.688725, 0.138562, 0.103105], [0, 0.567516, 0.55433, 0.066765, 0.168235], [0, 0.905899, 0.632549, 0.127549, 0.217059], [4, 0.905866, 0.763105, 0.128922, 0.058562], [4, 0.459641, 0.661626, 0.074477, 0.037239]]}, {"filename": "000000324882.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.634203, 0.562927, 0.230906, 0.496628], [3, 0.773906, 0.652412, 0.03325, 0.019485]]}, {"filename": "000000325031.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.901523, 0.281003, 0.032828, 0.084276], [0, 0.939688, 0.331234, 0.035281, 0.081316], [0, 0.926391, 0.30111, 0.018094, 0.030411], [0, 0.954508, 0.294465, 0.038016, 0.03278], [0, 0.063492, 0.31338, 0.018234, 0.035641], [0, 0.85257, 0.31139, 0.014109, 0.040345], [0, 0.86843, 0.284128, 0.013953, 0.01625], [0, 0.119148, 0.305502, 0.013172, 0.021234], [0, 0.030047, 0.29537, 0.012594, 0.020937], [0, 0.086477, 0.281061, 0.007672, 0.01653], [0, 0.07382, 0.304688, 0.015328, 0.016612], [0, 0.941523, 0.284104, 0.008203, 0.025313], [3, 0.897047, 0.339975, 0.082281, 0.042089], [0, 0.124484, 0.277541, 0.004781, 0.016661]]}, {"filename": "000000325623.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.760031, 0.781175, 0.089563, 0.18195], [4, 0.864094, 0.787688, 0.103469, 0.174475], [4, 0.564906, 0.773012, 0.086469, 0.184325], [4, 0.815875, 0.657037, 0.071687, 0.018325], [0, 0.42043, 0.516363, 0.064891, 0.252825], [0, 0.25818, 0.6947, 0.093109, 0.4876], [0, 0.122453, 0.675037, 0.141031, 0.513125], [0, 0.581719, 0.583362, 0.070406, 0.139975], [4, 0.953773, 0.777337, 0.041734, 0.224275], [4, 0.645133, 0.7769, 0.097203, 0.1879], [4, 0.707844, 0.655438, 0.07975, 0.038875]]}, {"filename": "000000325665.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.282354, 0.6646, 0.264338, 0.11372], [3, 0.327877, 0.83049, 0.3216, 0.18978], [4, 0.6216, 0.6923, 0.102338, 0.07484]]}, {"filename": "000000325991.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.48821, 0.564746, 0.39266, 0.870509]]}, {"filename": "000000326627.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.49668, 0.869438, 0.084672, 0.082875], [1, 0.566242, 0.864146, 0.023859, 0.046458], [1, 0.610414, 0.848698, 0.040266, 0.034521], [1, 0.631125, 0.844125, 0.016812, 0.028], [1, 0.786242, 0.833552, 0.014297, 0.016479], [1, 0.548531, 0.866302, 0.032156, 0.053604], [1, 0.661016, 0.840813, 0.029563, 0.044083], [1, 0.634328, 0.838042, 0.023406, 0.031375], [1, 0.400617, 0.858375, 0.022234, 0.023708], [1, 0.735734, 0.82426, 0.009906, 0.009063]]}, {"filename": "000000326689.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.501437, 0.188173, 0.997125, 0.376347], [1, 0.500445, 0.526628, 0.998672, 0.924262], [0, 0.442289, 0.449438, 0.512766, 0.898876]]}, {"filename": "000000326702.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.352016, 0.732615, 0.125844, 0.534771], [0, 0.969414, 0.72851, 0.061172, 0.330146], [1, 0.903844, 0.613646, 0.032844, 0.031542], [0, 0.033023, 0.645302, 0.066047, 0.249354], [0, 0.112336, 0.620552, 0.025516, 0.109479], [0, 0.905328, 0.597156, 0.01725, 0.043729], [0, 0.896594, 0.588604, 0.008219, 0.037625], [0, 0.885812, 0.593125, 0.012469, 0.046458], [0, 0.854375, 0.597865, 0.013562, 0.042813], [0, 0.835258, 0.593594, 0.010547, 0.043813], [0, 0.82475, 0.595083, 0.012031, 0.044], [0, 0.812688, 0.605448, 0.021969, 0.059104], [1, 0.537641, 0.628833, 0.667406, 0.660667], [0, 0.171187, 0.576979, 0.018813, 0.021125], [1, 0.85582, 0.616187, 0.034766, 0.031583], [1, 0.048578, 0.701177, 0.097156, 0.123979], [0, 0.933492, 0.658677, 0.041672, 0.200979]]}, {"filename": "000000326911.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.259477, 0.358601, 0.510297, 0.707506]]}, {"filename": "000000327263.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.5255, 0.532201, 0.730156, 0.602248]]}, {"filename": "000000327368.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.385437, 0.748241, 0.074781, 0.066667]]}, {"filename": "000000327788.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.017164, 0.830329, 0.028359, 0.316847], [0, 0.51707, 0.618129, 0.207422, 0.741576], [0, 0.865914, 0.551835, 0.268172, 0.896329], [0, 0.40875, 0.547047, 0.163375, 0.903318], [0, 0.276695, 0.618235, 0.179484, 0.743247], [0, 0.678922, 0.565624, 0.199063, 0.868753], [0, 0.113961, 0.575306, 0.211516, 0.834565], [4, 0.011047, 0.723094, 0.022094, 0.152212]]}, {"filename": "000000327890.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.957391, 0.434083, 0.026187, 0.025625], [1, 0.983969, 0.434396, 0.031969, 0.022417], [1, 0.846977, 0.436927, 0.025422, 0.021062], [1, 0.896797, 0.433802, 0.019031, 0.022354], [1, 0.817812, 0.435021, 0.026187, 0.024708], [1, 0.783594, 0.437344, 0.023688, 0.019563], [1, 0.880102, 0.429417, 0.019141, 0.029708], [3, 0.408227, 0.721969, 0.490016, 0.289521], [3, 0.514883, 0.57375, 0.281141, 0.13175], [3, 0.105945, 0.609865, 0.164797, 0.183854], [1, 0.912914, 0.433125, 0.022016, 0.024083], [3, 0.756109, 0.500719, 0.084219, 0.051396], [1, 0.937039, 0.435604, 0.018422, 0.021875]]}, {"filename": "000000328111.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.207727, 0.790083, 0.017203, 0.040208], [0, 0.315469, 0.690021, 0.007063, 0.019792], [1, 0.518469, 0.749646, 0.042156, 0.046833], [1, 0.37332, 0.755854, 0.044516, 0.039167], [1, 0.584063, 0.696302, 0.023625, 0.022562], [1, 0.781281, 0.634229, 0.009594, 0.009208], [1, 0.787211, 0.636667, 0.009922, 0.009833], [1, 0.246125, 0.72049, 0.03225, 0.034063], [1, 0.633727, 0.668083, 0.017766, 0.016417], [1, 0.37982, 0.694646, 0.027703, 0.025583], [1, 0.554219, 0.671146, 0.02175, 0.014875], [1, 0.206594, 0.804, 0.022625, 0.029667], [1, 0.187648, 0.745104, 0.034484, 0.0315], [1, 0.519633, 0.74825, 0.041953, 0.051083], [1, 0.364414, 0.90326, 0.105109, 0.118313], [0, 0.294391, 0.666281, 0.011313, 0.026021], [0, 0.311867, 0.664896, 0.004766, 0.017667], [1, 0.101734, 0.765208, 0.023625, 0.04675]]}, {"filename": "000000328683.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.481323, 0.215508, 0.281021, 0.318391]]}, {"filename": "000000328751.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.818555, 0.440906, 0.026984, 0.112918], [0, 0.845789, 0.390565, 0.010391, 0.032847], [3, 0.96807, 0.705365, 0.063641, 0.166918], [3, 0.940945, 0.556812, 0.059078, 0.103412], [0, 0.448977, 0.375847, 0.056172, 0.073576], [0, 0.860336, 0.464271, 0.047078, 0.197106], [0, 0.83275, 0.403576, 0.021125, 0.062212]]}, {"filename": "000000329125.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.452818, 0.603922, 0.497921, 0.792156], [1, 0.383406, 0.681664, 0.766813, 0.240266]]}, {"filename": "000000330177.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.535203, 0.758156, 0.113063, 0.261729], [0, 0.524328, 0.246302, 0.380875, 0.460854], [0, 0.100945, 0.700365, 0.031953, 0.070354], [3, 0.038766, 0.762927, 0.077531, 0.042688], [3, 0.19207, 0.787177, 0.254609, 0.037604]]}, {"filename": "000000330439.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.355414, 0.747213, 0.289016, 0.44829]]}, {"filename": "000000330726.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.388742, 0.645, 0.106391, 0.29375], [0, 0.31707, 0.669052, 0.071672, 0.413396], [0, 0.959461, 0.63401, 0.043922, 0.211729], [0, 0.567156, 0.678396, 0.108156, 0.39075], [0, 0.507023, 0.574365, 0.057141, 0.143604], [0, 0.834898, 0.601448, 0.028359, 0.110812], [0, 0.821445, 0.644333, 0.129078, 0.154], [0, 0.745633, 0.638198, 0.055297, 0.205688], [0, 0.701594, 0.584406, 0.032281, 0.099938], [1, 0.404891, 0.790146, 0.156281, 0.264375], [1, 0.070844, 0.784583, 0.1195, 0.162833], [1, 0.278828, 0.799031, 0.085656, 0.179229], [1, 0.786133, 0.679177, 0.056828, 0.119062], [1, 0.666477, 0.688115, 0.099453, 0.127271], [1, 0.178805, 0.667615, 0.030703, 0.077813], [1, 0.143477, 0.671292, 0.052922, 0.072083], [1, 0.462789, 0.671854, 0.053547, 0.115375], [1, 0.935625, 0.680104, 0.033656, 0.112917], [1, 0.982969, 0.651979, 0.034062, 0.103333], [0, 0.079281, 0.648469, 0.106625, 0.408688], [0, 0.674516, 0.63824, 0.050281, 0.194437], [0, 0.862133, 0.605198, 0.033484, 0.090354], [1, 0.567328, 0.743365, 0.249469, 0.214688], [0, 0.150789, 0.632292, 0.036078, 0.156333], [1, 0.355852, 0.543958, 0.021172, 0.039167], [1, 0.813547, 0.676781, 0.037281, 0.099896]]}, {"filename": "000000330993.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.872633, 0.495269, 0.254734, 0.983575], [7, 0.332031, 0.479299, 0.144562, 0.480234], [7, 0.763281, 0.236016, 0.079594, 0.272313], [1, 0.407336, 0.354579, 0.090078, 0.176636]]}, {"filename": "000000331031.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.869328, 0.692133, 0.261344, 0.593266], [0, 0.612096, 0.666289, 0.708736, 0.631453], [0, 0.089672, 0.546727, 0.179344, 0.442516], [0, 0.437216, 0.566289, 0.46944, 0.665172], [3, 0.072616, 0.926219, 0.143536, 0.143406], [0, 0.636096, 0.426578, 0.130912, 0.113406]]}, {"filename": "000000331298.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.487898, 0.683875, 0.327422, 0.606458], [0, 0.423992, 0.494365, 0.425672, 0.927938]]}, {"filename": "000000331518.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.551477, 0.614859, 0.465828, 0.747752], [0, 0.224648, 0.542787, 0.223891, 0.894145]]}, {"filename": "000000331799.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.926039, 0.377522, 0.147922, 0.516853], [0, 0.518555, 0.424881, 0.445391, 0.836315], [0, 0.185352, 0.667425, 0.360859, 0.475108], [0, 0.039523, 0.407565, 0.078891, 0.21181]]}, {"filename": "000000332543.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.51875, 0.403613, 0.2625, 0.334825], [1, 0.83863, 0.748437, 0.03982, 0.047775], [1, 0.70043, 0.735387, 0.03454, 0.026875], [1, 0.7575, 0.7316, 0.01144, 0.01875], [1, 0.73875, 0.730275, 0.02762, 0.023], [5, 0.84165, 0.82915, 0.01438, 0.0443], [1, 0.72274, 0.737125, 0.02104, 0.025], [1, 0.81484, 0.732688, 0.02588, 0.024225], [6, 0.33501, 0.660438, 0.02518, 0.038125]]}, {"filename": "000000332824.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.365906, 0.722736, 0.119531, 0.148919], [1, 0.234492, 0.681774, 0.037984, 0.072466], [1, 0.612766, 0.700389, 0.117563, 0.105912], [0, 0.960516, 0.694527, 0.019875, 0.161959], [5, 0.578547, 0.883851, 0.054031, 0.165203], [1, 0.453375, 0.623209, 0.247469, 0.253716], [1, 0.276453, 0.679679, 0.054188, 0.128412], [1, 0.200758, 0.666605, 0.032484, 0.039899], [1, 0.582398, 0.61799, 0.036234, 0.059561], [1, 0.151703, 0.637449, 0.022, 0.037331], [1, 0.179977, 0.650372, 0.031484, 0.044865]]}, {"filename": "000000333090.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.451754, 0.415203, 0.510927, 0.767344]]}, {"filename": "000000333436.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.609546, 0.564047, 0.473193, 0.849437], [0, 0.255624, 0.540773, 0.343274, 0.917484], [1, 0.107447, 0.282164, 0.209222, 0.171266], [1, 0.867877, 0.203484, 0.124635, 0.068781], [1, 0.065413, 0.542797, 0.130827, 0.376125], [1, 0.401621, 0.469461, 0.07812, 0.132047], [1, 0.39406, 0.343625, 0.096224, 0.12175], [6, 0.54654, 0.245172, 0.298525, 0.286406]]}, {"filename": "000000333697.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.414296, 0.236117, 0.110094, 0.158641]]}, {"filename": "000000333710.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.5, 0.492277, 1.0, 0.984554]]}, {"filename": "000000334006.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.140922, 0.634002, 0.278844, 0.283779], [1, 0.57793, 0.543768, 0.466234, 0.515751], [0, 0.872734, 0.569894, 0.035469, 0.110634], [1, 0.860547, 0.518674, 0.12375, 0.183263], [6, 0.982523, 0.496045, 0.019078, 0.025798]]}, {"filename": "000000334039.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.989445, 0.58228, 0.020859, 0.08546], [3, 0.959797, 0.687655, 0.080375, 0.154146], [0, 0.739555, 0.555985, 0.329641, 0.864578]]}, {"filename": "000000334187.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.121914, 0.198698, 0.237109, 0.338562], [0, 0.806539, 0.225, 0.386922, 0.447292]]}, {"filename": "000000334328.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.503211, 0.517729, 0.024516, 0.123668]]}, {"filename": "000000335177.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.910844, 0.639885, 0.094531, 0.045198], [1, 0.386734, 0.673038, 0.191906, 0.082359], [6, 0.555539, 0.536952, 0.052359, 0.072359]]}, {"filename": "000000335434.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.535109, 0.451688, 0.092687, 0.323583], [0, 0.400008, 0.293021, 0.033453, 0.041667], [0, 0.445836, 0.291198, 0.041234, 0.042396], [0, 0.466945, 0.462396, 0.092797, 0.29525], [0, 0.392773, 0.560552, 0.070766, 0.131854], [0, 0.388172, 0.390615, 0.092125, 0.192688], [0, 0.491555, 0.294208, 0.039297, 0.040792], [0, 0.313289, 0.438375, 0.071734, 0.278042]]}, {"filename": "000000335529.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.364891, 0.783146, 0.632031, 0.379792]]}, {"filename": "000000335986.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.666438, 0.14452, 0.297031, 0.284122], [1, 0.266125, 0.148314, 0.52025, 0.274145], [7, 0.407789, 0.123466, 0.097266, 0.219251], [7, 0.858414, 0.175574, 0.034453, 0.116159], [7, 0.977125, 0.190515, 0.024875, 0.067916], [0, 0.831609, 0.191885, 0.034938, 0.097073], [1, 0.944937, 0.181112, 0.017625, 0.034403], [1, 0.900391, 0.19034, 0.023438, 0.034356]]}, {"filename": "000000336232.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.196805, 0.624075, 0.354734, 0.285293], [1, 0.531414, 0.265445, 0.165547, 0.259415], [0, 0.678859, 0.199567, 0.010656, 0.05192], [0, 0.098352, 0.306007, 0.023953, 0.058618], [1, 0.736391, 0.444988, 0.207719, 0.234754], [1, 0.790016, 0.753372, 0.419969, 0.470726], [1, 0.039344, 0.468981, 0.078687, 0.242365], [1, 0.071805, 0.45123, 0.120609, 0.22096], [1, 0.111227, 0.413724, 0.127609, 0.172131], [1, 0.177898, 0.386042, 0.087953, 0.130258], [1, 0.713602, 0.168396, 0.066859, 0.131686], [1, 0.716781, 0.162166, 0.069125, 0.119368], [1, 0.821477, 0.266089, 0.115297, 0.113396], [1, 0.769805, 0.217424, 0.100047, 0.075644], [0, 0.974773, 0.416323, 0.050453, 0.17185], [1, 0.289375, 0.33096, 0.089719, 0.117564], [1, 0.690789, 0.308946, 0.092641, 0.114895], [1, 0.414609, 0.317834, 0.080438, 0.101616], [1, 0.271133, 0.356522, 0.076422, 0.135527], [1, 0.773258, 0.288384, 0.086422, 0.094567], [1, 0.659336, 0.226206, 0.051391, 0.062155], [1, 0.180648, 0.421522, 0.053641, 0.121171], [1, 0.898844, 0.399215, 0.130312, 0.164052], [1, 0.706453, 0.059684, 0.006219, 0.009016], [0, 0.205391, 0.515679, 0.141063, 0.407002]]}, {"filename": "000000336980.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.233203, 0.547687, 0.070344, 0.109958], [0, 0.579242, 0.564906, 0.046297, 0.087271], [0, 0.812781, 0.571156, 0.063531, 0.080479]]}, {"filename": "000000338104.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.628398, 0.545708, 0.163141, 0.182083]]}, {"filename": "000000338219.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.43125, 0.558428, 0.790969, 0.802261], [1, 0.769844, 0.425671, 0.432219, 0.317562], [1, 0.928031, 0.385292, 0.143937, 0.264576], [0, 0.331531, 0.157535, 0.150656, 0.284399], [0, 0.256117, 0.274214, 0.031016, 0.058463], [0, 0.059336, 0.290945, 0.019266, 0.071007], [0, 0.144094, 0.278083, 0.03075, 0.044435], [0, 0.173328, 0.271599, 0.023906, 0.050159], [4, 0.092867, 0.318127, 0.047141, 0.054806], [0, 0.958742, 0.298322, 0.022828, 0.021661], [0, 0.194008, 0.281899, 0.021078, 0.03023], [0, 0.110039, 0.290168, 0.012109, 0.052279], [0, 0.23732, 0.285018, 0.013328, 0.032191], [0, 0.119133, 0.297235, 0.009578, 0.064258], [0, 0.270891, 0.285362, 0.016844, 0.043375], [1, 0.029641, 0.315345, 0.05825, 0.068004], [0, 0.102695, 0.285981, 0.010391, 0.030336], [0, 0.214867, 0.272323, 0.043953, 0.056095]]}, {"filename": "000000338560.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.20176, 0.285914, 0.169437, 0.055234], [1, 0.291781, 0.279195, 0.157396, 0.058203], [1, 0.06151, 0.291195, 0.121521, 0.060766], [1, 0.907156, 0.261664, 0.128354, 0.047859], [1, 0.124625, 0.272562, 0.100375, 0.030344], [5, 0.469969, 0.608109, 0.387396, 0.698188]]}, {"filename": "000000338705.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.745812, 0.745383, 0.109833, 0.254609], [1, 0.268438, 0.531516, 0.160458, 0.12325], [0, 0.760625, 0.619906, 0.208792, 0.411625], [0, 0.252823, 0.490984, 0.085646, 0.124969], [0, 0.381031, 0.464359, 0.036604, 0.057125], [1, 0.946135, 0.479781, 0.093104, 0.053656], [3, 0.330146, 0.486477, 0.099333, 0.032453], [3, 0.606021, 0.466617, 0.072042, 0.034047]]}, {"filename": "000000338978.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.29924, 0.494492, 0.522187, 0.292234], [1, 0.544229, 0.375516, 0.1005, 0.046375], [1, 0.831708, 0.345961, 0.118042, 0.038297], [1, 0.472521, 0.356125, 0.123792, 0.038437], [1, 0.817979, 0.438203, 0.311625, 0.148313], [5, 0.81901, 0.547937, 0.082896, 0.094937], [6, 0.486063, 0.326078, 0.062625, 0.055563], [1, 0.963865, 0.523164, 0.072271, 0.092359], [1, 0.931375, 0.32918, 0.061333, 0.026953], [1, 0.701021, 0.355359, 0.139417, 0.0505], [1, 0.980802, 0.424117, 0.038396, 0.103391], [1, 0.313729, 0.345891, 0.093417, 0.019719], [1, 0.544615, 0.431125, 0.161729, 0.117406], [1, 0.299427, 0.494172, 0.514479, 0.290313], [1, 0.052021, 0.392172, 0.038375, 0.055562], [1, 0.657896, 0.42732, 0.132542, 0.094984]]}, {"filename": "000000339177.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.592867, 0.692424, 0.164828, 0.324848], [4, 0.771211, 0.753115, 0.196578, 0.423653], [4, 0.430648, 0.6487, 0.139484, 0.265082], [4, 0.287203, 0.5826, 0.083031, 0.182998], [4, 0.962727, 0.792119, 0.074547, 0.39356], [4, 0.104836, 0.513595, 0.027547, 0.0863], [0, 0.487, 0.544614, 0.113063, 0.37733], [0, 0.303531, 0.508337, 0.060125, 0.257611], [0, 0.156234, 0.440948, 0.050969, 0.075621], [0, 0.246539, 0.481511, 0.047266, 0.119555], [0, 0.118891, 0.486628, 0.06, 0.122342], [0, 0.068562, 0.476593, 0.047656, 0.07281], [4, 0.173469, 0.540984, 0.028344, 0.127494], [0, 0.088961, 0.718958, 0.177922, 0.259321], [4, 0.124547, 0.521159, 0.02025, 0.102319], [4, 0.215109, 0.564262, 0.062594, 0.162248], [4, 0.144258, 0.526651, 0.024984, 0.105152], [4, 0.04332, 0.823372, 0.086641, 0.32726]]}, {"filename": "000000339186.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.318203, 0.56292, 0.262469, 0.217988]]}, {"filename": "000000339589.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.799922, 0.764977, 0.066063, 0.247049], [1, 0.461641, 0.779016, 0.376875, 0.441967], [1, 0.827039, 0.727869, 0.111766, 0.235738], [1, 0.207656, 0.734262, 0.412312, 0.514614], [1, 0.59532, 0.728946, 0.235391, 0.422951], [1, 0.925328, 0.719415, 0.073156, 0.14178], [7, 0.744547, 0.590012, 0.075344, 0.342272], [0, 0.982961, 0.705843, 0.034078, 0.218712], [1, 0.886602, 0.739297, 0.041203, 0.163372], [1, 0.955242, 0.68877, 0.046609, 0.074496], [1, 0.968727, 0.635937, 0.024641, 0.036932]]}, {"filename": "000000340175.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.179875, 0.805673, 0.17, 0.388655], [4, 0.935391, 0.715787, 0.129219, 0.223959], [3, 0.362695, 0.696827, 0.177578, 0.31802], [4, 0.167883, 0.539404, 0.036703, 0.050482], [4, 0.096266, 0.706713, 0.057344, 0.306269], [4, 0.368664, 0.524619, 0.081359, 0.164924], [4, 0.221344, 0.501561, 0.026938, 0.033731]]}, {"filename": "000000340451.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.478805, 0.632488, 0.823609, 0.478467], [0, 0.49457, 0.50421, 0.245109, 0.792901]]}, {"filename": "000000341039.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.855557, 0.274773, 0.227951, 0.254516], [6, 0.261058, 0.208992, 0.349154, 0.314609], [0, 0.370724, 0.73793, 0.185367, 0.457391], [1, 0.716136, 0.714406, 0.567728, 0.218406], [6, 0.146826, 0.361195, 0.162249, 0.176984], [4, 0.360579, 0.816078, 0.17216, 0.303156]]}, {"filename": "000000341137.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.610802, 0.50101, 0.484375, 0.93946], [1, 0.908967, 0.17743, 0.182065, 0.34906], [1, 0.913818, 0.17966, 0.16981, 0.35624], [1, 0.092337, 0.03017, 0.182717, 0.0575], [1, 0.518777, 0.0252, 0.282337, 0.04676]]}, {"filename": "000000341247.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.533781, 0.802312, 0.053781, 0.293098], [0, 0.40868, 0.75885, 0.069953, 0.283257], [0, 0.351, 0.68459, 0.09375, 0.239658], [0, 0.477258, 0.180103, 0.041609, 0.053554], [0, 0.07943, 0.591036, 0.073484, 0.221982], [0, 0.489836, 0.449465, 0.063734, 0.224989], [0, 0.547367, 0.537711, 0.078016, 0.22959], [0, 0.570508, 0.657062, 0.079672, 0.251708], [0, 0.93757, 0.709021, 0.065766, 0.269522], [0, 0.239687, 0.597027, 0.073094, 0.206856], [0, 0.721094, 0.368269, 0.047656, 0.137768], [4, 0.157961, 0.211219, 0.022672, 0.024624], [4, 0.375414, 0.200399, 0.028828, 0.068269], [4, 0.184758, 0.20992, 0.024203, 0.021162], [4, 0.262617, 0.217084, 0.027297, 0.026834], [4, 0.445766, 0.043144, 0.0375, 0.047198], [4, 0.41582, 0.233667, 0.028828, 0.02615], [4, 0.476219, 0.245478, 0.026219, 0.034692], [4, 0.969187, 0.282232, 0.03275, 0.03549], [4, 0.780172, 0.299305, 0.030875, 0.035285], [4, 0.291094, 0.218622, 0.02575, 0.022528], [4, 0.130914, 0.212711, 0.028797, 0.029066], [0, 0.384141, 0.391253, 0.070875, 0.195854], [0, 0.587484, 0.552084, 0.081156, 0.268633], [4, 0.485781, 0.068155, 0.020875, 0.012073], [4, 0.237016, 0.215695, 0.022875, 0.023781]]}, {"filename": "000000341639.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.324937, 0.435792, 0.229063, 0.219417], [1, 0.687773, 0.840052, 0.146734, 0.071396], [1, 0.309852, 0.709344, 0.020516, 0.032813], [0, 0.666391, 0.855083, 0.028812, 0.072167], [1, 0.09082, 0.63375, 0.031453, 0.029833]]}, {"filename": "000000341921.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.569953, 0.373431, 0.023031, 0.02113], [1, 0.361641, 0.361726, 0.009031, 0.011736], [1, 0.781086, 0.38954, 0.028828, 0.014268], [0, 0.541883, 0.660931, 0.183328, 0.297343], [3, 0.59232, 0.79568, 0.449453, 0.384791], [1, 0.693078, 0.382364, 0.026344, 0.017155], [1, 0.856625, 0.386914, 0.025812, 0.013703]]}, {"filename": "000000341973.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.72954, 0.55868, 0.54092, 0.858747], [3, 0.05983, 0.953933, 0.11966, 0.06968], [0, 0.35552, 0.657147, 0.49676, 0.649333]]}, {"filename": "000000342004.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.490195, 0.587667, 0.761766, 0.612125]]}, {"filename": "000000343315.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.860883, 0.285711, 0.096422, 0.309079]]}, {"filename": "000000343496.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.287883, 0.256896, 0.062453, 0.106209], [1, 0.440656, 0.539059, 0.082437, 0.071908], [5, 0.591273, 0.635585, 0.028547, 0.102875], [0, 0.130406, 0.671539, 0.086938, 0.305623], [1, 0.975914, 0.54757, 0.016484, 0.012137]]}, {"filename": "000000343704.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.514227, 0.50773, 0.567569, 0.8261], [3, 0.439793, 0.7613, 0.254834, 0.0612]]}, {"filename": "000000344422.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.266516, 0.297422, 0.2005, 0.211719], [1, 0.150891, 0.76343, 0.262813, 0.256516], [1, 0.020648, 0.755164, 0.037266, 0.130859]]}, {"filename": "000000344805.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.575344, 0.584719, 0.290281, 0.279229], [0, 0.678523, 0.657927, 0.305516, 0.361604], [4, 0.300977, 0.616885, 0.243516, 0.340271]]}, {"filename": "000000344909.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.603469, 0.494922, 0.749438, 0.981938], [1, 0.944156, 0.338664, 0.111688, 0.067953], [1, 0.853104, 0.337547, 0.120583, 0.06375], [1, 0.78524, 0.335867, 0.059021, 0.062172]]}, {"filename": "000000345047.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.120961, 0.244085, 0.239047, 0.488169], [3, 0.856656, 0.403592, 0.208938, 0.30493], [4, 0.622805, 0.49581, 0.196391, 0.293545]]}, {"filename": "000000346351.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.674984, 0.945446, 0.341375, 0.109061], [3, 0.250656, 0.908028, 0.25875, 0.047746], [0, 0.620211, 0.826843, 0.099766, 0.32439], [0, 0.795758, 0.832582, 0.095734, 0.30338], [0, 0.409977, 0.843451, 0.092391, 0.313099], [0, 0.309289, 0.82946, 0.093922, 0.34108], [0, 0.211039, 0.836326, 0.108016, 0.327347], [0, 0.712469, 0.834566, 0.093687, 0.330869], [0, 0.270305, 0.717101, 0.040891, 0.070023], [0, 0.101078, 0.828392, 0.116688, 0.343216]]}, {"filename": "000000346472.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.06332, 0.756427, 0.015891, 0.052979], [1, 0.304781, 0.623406, 0.025125, 0.029187], [1, 0.215352, 0.700302, 0.045672, 0.054938], [1, 0.311109, 0.701083, 0.019281, 0.035167], [1, 0.571078, 0.504375, 0.034063, 0.042417], [1, 0.610094, 0.500177, 0.005, 0.012312]]}, {"filename": "000000346581.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.263258, 0.268545, 0.224359, 0.366291], [1, 0.330789, 0.887641, 0.151203, 0.090352], [1, 0.882062, 0.896185, 0.062625, 0.043263], [5, 0.127336, 0.874472, 0.012203, 0.04669]]}, {"filename": "000000347104.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.165008, 0.211288, 0.143484, 0.421499]]}, {"filename": "000000347163.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.354492, 0.467775, 0.500016, 0.747213]]}, {"filename": "000000347254.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.569493, 0.42056, 0.19056, 0.12932], [4, 0.845893, 0.77718, 0.135093, 0.1594], [0, 0.426427, 0.52027, 0.852853, 0.9347], [0, 0.75864, 0.39393, 0.168427, 0.17246], [0, 0.900587, 0.54772, 0.196107, 0.55504], [0, 0.669187, 0.3641, 0.0604, 0.21912], [0, 0.70568, 0.32858, 0.089173, 0.14884]]}, {"filename": "000000348019.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.524644, 0.395828, 0.434812, 0.379312]]}, {"filename": "000000348379.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.552828, 0.544731, 0.342406, 0.869977], [0, 0.302789, 0.489859, 0.277984, 0.979719], [0, 0.796516, 0.52377, 0.213781, 0.435972], [0, 0.936703, 0.235984, 0.073188, 0.471967], [0, 0.127617, 0.602002, 0.255234, 0.787471], [0, 0.653727, 0.185351, 0.167547, 0.363208], [6, 0.586609, 0.576628, 0.358719, 0.4], [6, 0.324555, 0.233009, 0.290859, 0.334965], [6, 0.805227, 0.160375, 0.249797, 0.285714]]}, {"filename": "000000348400.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.551356, 0.675196, 0.880948, 0.333431], [1, 0.246364, 0.532541, 0.094036, 0.051781], [1, 0.287753, 0.521675, 0.033676, 0.035572], [0, 0.385065, 0.664216, 0.132582, 0.445163], [0, 0.965621, 0.55344, 0.051176, 0.123873], [0, 0.939926, 0.549199, 0.038709, 0.08598], [0, 0.898832, 0.527157, 0.026748, 0.055458], [0, 0.82652, 0.520425, 0.018301, 0.030654], [0, 0.854379, 0.531993, 0.02098, 0.037157], [0, 0.911487, 0.53085, 0.023954, 0.037255], [0, 0.375025, 0.561095, 0.062663, 0.067124], [5, 0.133783, 0.549542, 0.018807, 0.04098]]}, {"filename": "000000349184.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.249217, 0.542797, 0.410864, 0.691437], [0, 0.420257, 0.332273, 0.053879, 0.056797], [0, 0.946133, 0.342172, 0.026192, 0.051219], [3, 0.366273, 0.652805, 0.719089, 0.667422], [0, 0.886086, 0.341469, 0.043294, 0.055781], [0, 0.841799, 0.340609, 0.046121, 0.048438], [0, 0.506881, 0.33425, 0.059136, 0.056562], [0, 0.914007, 0.349906, 0.029229, 0.039938]]}, {"filename": "000000349393.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.206508, 0.622259, 0.054891, 0.225584], [0, 0.153211, 0.535279, 0.038047, 0.166294], [0, 0.111625, 0.483756, 0.0415, 0.084264], [0, 0.275461, 0.314772, 0.052453, 0.205635], [0, 0.660906, 0.36783, 0.039719, 0.194239], [0, 0.785117, 0.714734, 0.084359, 0.16033], [0, 0.924055, 0.508959, 0.040953, 0.212893], [0, 0.763859, 0.129023, 0.037844, 0.196168], [0, 0.465391, 0.132157, 0.042156, 0.188782], [0, 0.351977, 0.31736, 0.054578, 0.188477], [0, 0.123984, 0.601536, 0.032031, 0.16967], [0, 0.21068, 0.320228, 0.036734, 0.169239], [0, 0.590961, 0.372272, 0.031516, 0.192563]]}, {"filename": "000000349575.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.64151, 0.746242, 0.210396, 0.068359], [1, 0.054354, 0.777766, 0.108708, 0.096125], [1, 0.372854, 0.707313, 0.044875, 0.024031], [1, 0.34324, 0.701281, 0.042938, 0.02425], [1, 0.321302, 0.699164, 0.023771, 0.018828], [5, 0.239594, 0.891414, 0.108063, 0.217172], [1, 0.395552, 0.705812, 0.008896, 0.018406], [1, 0.547885, 0.713258, 0.064854, 0.024641], [1, 0.310823, 0.697758, 0.018396, 0.018141], [1, 0.29825, 0.698094, 0.014833, 0.016344], [1, 0.244219, 0.701539, 0.037563, 0.020047], [1, 0.419115, 0.71468, 0.042271, 0.027422]]}, {"filename": "000000349822.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.4228, 0.530766, 0.663, 0.91012], [0, 0.27493, 0.637312, 0.06774, 0.091802], [3, 0.80712, 0.834685, 0.15244, 0.132132]]}, {"filename": "000000349860.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.575367, 0.854038, 0.056609, 0.068545], [0, 0.582484, 0.209977, 0.290187, 0.357324], [3, 0.545141, 0.887617, 0.037594, 0.041526], [0, 0.086758, 0.844941, 0.026922, 0.085376], [0, 0.348008, 0.790927, 0.014984, 0.07392], [1, 0.448789, 0.854448, 0.017984, 0.066643], [3, 0.587094, 0.906268, 0.024031, 0.038873]]}, {"filename": "000000349905.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.628961, 0.479778, 0.166422, 0.289906], [0, 0.230664, 0.648653, 0.461328, 0.702693], [0, 0.469727, 0.573548, 0.260297, 0.852904], [0, 0.765531, 0.522553, 0.468938, 0.954895]]}, {"filename": "000000350148.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.813771, 0.245898, 0.215875, 0.081453], [4, 0.214719, 0.404281, 0.069062, 0.225219], [0, 0.555812, 0.295125, 0.59625, 0.573031], [0, 0.287646, 0.360672, 0.19775, 0.357313], [0, 0.951292, 0.474687, 0.097417, 0.254625], [0, 0.906979, 0.277398, 0.186042, 0.195047], [1, 0.086781, 0.263641, 0.173563, 0.174344], [4, 0.882156, 0.465516, 0.145354, 0.16225], [4, 0.877375, 0.359023, 0.047375, 0.029078]]}, {"filename": "000000351268.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.132731, 0.400556, 0.090525, 0.034213], [5, 0.500108, 0.660359, 0.440154, 0.669421], [1, 0.196019, 0.402546, 0.053395, 0.025648], [1, 0.355062, 0.403657, 0.034259, 0.020231], [1, 0.03466, 0.402928, 0.053704, 0.023866], [1, 0.327207, 0.405463, 0.020278, 0.010926], [1, 0.319475, 0.40309, 0.077654, 0.021505]]}, {"filename": "000000351351.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.103656, 0.493021, 0.171437, 0.362042], [0, 0.578937, 0.334458, 0.197187, 0.665167], [0, 0.283148, 0.352813, 0.060672, 0.175292], [0, 0.407805, 0.358573, 0.096297, 0.341146], [0, 0.592336, 0.477156, 0.229766, 0.352021], [7, 0.07882, 0.242188, 0.057859, 0.162708], [0, 0.325156, 0.334906, 0.043312, 0.148146]]}, {"filename": "000000351386.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.471736, 0.305766, 0.103428, 0.151125], [7, 0.320546, 0.080539, 0.099083, 0.126141], [0, 0.611419, 0.447414, 0.307642, 0.558266]]}, {"filename": "000000351530.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.339187, 0.515886, 0.254625, 0.440661], [0, 0.300023, 0.42873, 0.180516, 0.243122], [1, 0.103508, 0.523122, 0.206484, 0.269101], [0, 0.271281, 0.534352, 0.075219, 0.370556], [4, 0.47125, 0.560225, 0.170594, 0.40172], [4, 0.551742, 0.57832, 0.101016, 0.437434], [4, 0.594758, 0.585728, 0.112672, 0.467328], [4, 0.847078, 0.64832, 0.305844, 0.57754], [4, 0.699453, 0.55955, 0.185813, 0.458413], [4, 0.766508, 0.599854, 0.227578, 0.528862], [4, 0.87932, 0.442712, 0.090047, 0.145741], [4, 0.938375, 0.785582, 0.12325, 0.424286], [4, 0.345477, 0.618228, 0.051359, 0.256772], [4, 0.911391, 0.736164, 0.177219, 0.419577], [3, 0.618859, 0.564458, 0.229031, 0.45082], [3, 0.848492, 0.641574, 0.296172, 0.558228], [0, 0.090977, 0.369418, 0.172891, 0.331429], [3, 0.937984, 0.780516, 0.124031, 0.412989], [3, 0.912781, 0.738968, 0.171813, 0.423386], [4, 0.491344, 0.62914, 0.061469, 0.299603], [4, 0.378773, 0.616759, 0.049547, 0.260714], [4, 0.665234, 0.426111, 0.157719, 0.185714]]}, {"filename": "000000351810.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.13087, 0.520547, 0.17114, 0.4452], [0, 0.06503, 0.38548, 0.0917, 0.119333], [0, 0.04469, 0.29784, 0.0887, 0.33552], [0, 0.21356, 0.36816, 0.04464, 0.093547], [4, 0.17337, 0.772907, 0.08438, 0.082187], [0, 0.3732, 0.366533, 0.04252, 0.155893]]}, {"filename": "000000352145.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.349203, 0.80825, 0.073656, 0.157531], [0, 0.958273, 0.78493, 0.080234, 0.101672], [0, 0.927227, 0.688508, 0.023484, 0.055297], [0, 0.795359, 0.709875, 0.022719, 0.056031], [0, 0.741453, 0.687742, 0.014531, 0.029453], [0, 0.722602, 0.685516, 0.016172, 0.030531], [3, 0.462305, 0.827211, 0.153016, 0.047922], [3, 0.950789, 0.845047, 0.098422, 0.062937], [0, 0.953281, 0.694367, 0.018781, 0.063516], [0, 0.862937, 0.717742, 0.016781, 0.061516], [0, 0.885828, 0.713469, 0.028344, 0.07375], [0, 0.831578, 0.712086, 0.019937, 0.051516], [0, 0.75875, 0.714, 0.027625, 0.048406]]}, {"filename": "000000352148.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.412484, 0.580712, 0.536781, 0.838577]]}, {"filename": "000000352761.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.38354, 0.520225, 0.50288, 0.806737], [3, 0.23568, 0.674162, 0.36628, 0.507844], [1, 0.69259, 0.516587, 0.01586, 0.023772], [1, 0.97014, 0.537201, 0.00668, 0.019491]]}, {"filename": "000000352828.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.102094, 0.222981, 0.125219, 0.414783], [1, 0.05007, 0.652081, 0.091391, 0.089752], [1, 0.130617, 0.656211, 0.032578, 0.052112]]}, {"filename": "000000354268.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.416125, 0.639485, 0.120562, 0.400749], [6, 0.014586, 0.447026, 0.020797, 0.057752], [5, 0.552398, 0.629578, 0.020891, 0.051663]]}, {"filename": "000000354547.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.162904, 0.168063, 0.038829, 0.031656], [0, 0.43041, 0.541664, 0.415199, 0.891891]]}, {"filename": "000000354819.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.165547, 0.25225, 0.158781, 0.2635], [1, 0.234586, 0.064573, 0.287547, 0.129146], [1, 0.733938, 0.032396, 0.13025, 0.064792]]}, {"filename": "000000354829.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.45337, 0.430333, 0.57978, 0.411227], [5, 0.66858, 0.622453, 0.06016, 0.142827]]}, {"filename": "000000354921.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.128406, 0.428141, 0.170094, 0.843046], [1, 0.911164, 0.151061, 0.069359, 0.046618], [1, 0.271242, 0.180872, 0.152703, 0.110567], [3, 0.545859, 0.222658, 0.134438, 0.071113], [0, 0.418219, 0.178803, 0.0285, 0.10105], [3, 0.936836, 0.212899, 0.124828, 0.086765]]}, {"filename": "000000354938.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.84398, 0.588469, 0.192388, 0.464031], [3, 0.143097, 0.633711, 0.171716, 0.368547]]}, {"filename": "000000356060.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.261172, 0.514333, 0.206094, 0.423917], [0, 0.530109, 0.573146, 0.166781, 0.450708], [1, 0.336836, 0.648125, 0.213328, 0.126917], [1, 0.838531, 0.414885, 0.033, 0.086604], [1, 0.063859, 0.458917, 0.062031, 0.130458], [1, 0.814156, 0.419177, 0.025719, 0.076604], [1, 0.71775, 0.425958, 0.024281, 0.063083], [0, 0.023641, 0.453719, 0.047281, 0.327646], [1, 0.361719, 0.303594, 0.159344, 0.122479], [0, 0.457328, 0.555635, 0.083406, 0.263896], [0, 0.977945, 0.400125, 0.029266, 0.116208], [0, 0.767016, 0.378562, 0.016812, 0.085208], [0, 0.735961, 0.36451, 0.021391, 0.048104], [0, 0.709586, 0.388687, 0.022984, 0.080917], [0, 0.659555, 0.40875, 0.029516, 0.1465], [1, 0.605273, 0.391396, 0.098734, 0.089708], [1, 0.749305, 0.419927, 0.036359, 0.075688], [1, 0.728055, 0.417135, 0.012078, 0.034021], [1, 0.081203, 0.47275, 0.017719, 0.099417], [1, 0.127664, 0.388677, 0.023141, 0.043187], [1, 0.240148, 0.324135, 0.136703, 0.184729], [0, 0.509266, 0.377354, 0.032031, 0.092417], [1, 0.769, 0.395688, 0.036281, 0.051292], [1, 0.605445, 0.394208, 0.099828, 0.08075], [1, 0.115828, 0.387656, 0.014625, 0.041229]]}, {"filename": "000000356169.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.533664, 0.688104, 0.028016, 0.029583], [1, 0.583492, 0.685937, 0.028609, 0.029917], [1, 0.062844, 0.69974, 0.055781, 0.082938], [1, 0.557352, 0.681719, 0.016422, 0.019229], [5, 0.084281, 0.73626, 0.027062, 0.078562], [1, 0.542563, 0.683021, 0.019813, 0.026208], [1, 0.454484, 0.682885, 0.039219, 0.019812], [1, 0.021391, 0.685083, 0.042781, 0.137667]]}, {"filename": "000000356181.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.42356, 0.68143, 0.53096, 0.48546]]}, {"filename": "000000356531.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.321914, 0.512365, 0.640453, 0.952813]]}, {"filename": "000000357290.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.72074, 0.613258, 0.056313, 0.109109], [1, 0.468271, 0.554359, 0.033458, 0.023781], [0, 0.395583, 0.550164, 0.012167, 0.026609], [0, 0.267135, 0.560156, 0.022104, 0.04775], [0, 0.432198, 0.550086, 0.012979, 0.022672], [0, 0.574771, 0.557773, 0.0135, 0.027672], [1, 0.513073, 0.550641, 0.069104, 0.062406], [0, 0.473281, 0.539258, 0.005979, 0.007016], [0, 0.302781, 0.559539, 0.022604, 0.038578], [0, 0.404406, 0.549656, 0.008854, 0.026562], [0, 0.415844, 0.550812, 0.009604, 0.026], [0, 0.454292, 0.545539, 0.007333, 0.010234], [6, 0.694573, 0.230086, 0.263187, 0.192891]]}, {"filename": "000000357734.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.66302, 0.537851, 0.35568, 0.858684]]}, {"filename": "000000357737.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.354031, 0.626885, 0.566125, 0.686021], [1, 0.361031, 0.385698, 0.52475, 0.301687], [1, 0.30982, 0.235187, 0.197953, 0.060083], [0, 0.775344, 0.496625, 0.283781, 0.905417], [1, 0.523719, 0.59599, 0.334531, 0.737521], [0, 0.207055, 0.234781, 0.028422, 0.063021], [0, 0.256117, 0.207052, 0.032297, 0.040938], [1, 0.070469, 0.329073, 0.080531, 0.067104], [1, 0.906086, 0.186906, 0.011047, 0.006979], [1, 0.172523, 0.264667, 0.258203, 0.160417], [0, 0.266359, 0.206625, 0.008531, 0.016917]]}, {"filename": "000000357841.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.661047, 0.600574, 0.248813, 0.516511]]}, {"filename": "000000357994.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.627781, 0.228051, 0.049594, 0.047239], [1, 0.68343, 0.258306, 0.047016, 0.065151], [5, 0.463656, 0.421566, 0.304031, 0.704292]]}, {"filename": "000000358345.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.39851, 0.499625, 0.485396, 0.784281], [1, 0.174948, 0.224375, 0.137396, 0.062344]]}, {"filename": "000000358756.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.1055, 0.807768, 0.038937, 0.137797], [0, 0.627445, 0.79727, 0.016359, 0.057759], [0, 0.668742, 0.788563, 0.011703, 0.037625], [0, 0.695094, 0.796264, 0.013906, 0.055402], [0, 0.736258, 0.797854, 0.019797, 0.057471], [0, 0.788672, 0.793458, 0.025156, 0.064119], [0, 0.359555, 0.804828, 0.019172, 0.026667], [0, 0.562367, 0.800498, 0.017266, 0.051609], [0, 0.604992, 0.796638, 0.019766, 0.058142], [0, 0.710742, 0.794588, 0.017422, 0.063199], [0, 0.225305, 0.807672, 0.013297, 0.031207], [1, 0.187313, 0.801466, 0.047688, 0.042318], [0, 0.663187, 0.797605, 0.014969, 0.059349]]}, {"filename": "000000358817.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.156383, 0.746219, 0.021453, 0.025854], [1, 0.040172, 0.817813, 0.078938, 0.096375], [5, 0.847117, 0.857104, 0.032859, 0.066583], [0, 0.331812, 0.789563, 0.020125, 0.086792], [0, 0.310508, 0.796687, 0.022984, 0.0775]]}, {"filename": "000000358939.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.821527, 0.357914, 0.356946, 0.187297], [4, 0.168295, 0.353758, 0.219895, 0.177297]]}, {"filename": "000000359098.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.431461, 0.562927, 0.862922, 0.847187], [0, 0.861828, 0.515917, 0.267375, 0.3565], [0, 0.792047, 0.342354, 0.104063, 0.157917]]}, {"filename": "000000359106.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.53928, 0.55625, 0.58144, 0.516774]]}, {"filename": "000000359695.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.130187, 0.346521, 0.116406, 0.170083], [0, 0.303383, 0.352365, 0.061109, 0.290646], [0, 0.207313, 0.375615, 0.070531, 0.269896], [0, 0.051164, 0.34525, 0.091047, 0.4515], [1, 0.764852, 0.373573, 0.470297, 0.742146], [0, 0.250078, 0.345073, 0.048687, 0.191854], [1, 0.488859, 0.32176, 0.094656, 0.145313], [6, 0.122938, 0.058583, 0.055188, 0.1155], [0, 0.658852, 0.575156, 0.063453, 0.157562], [0, 0.327539, 0.244438, 0.016078, 0.064292], [0, 0.608984, 0.548531, 0.0425, 0.123188], [5, 0.269445, 0.581479, 0.198141, 0.303708]]}, {"filename": "000000361399.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.694187, 0.220572, 0.131656, 0.441145], [0, 0.370195, 0.347418, 0.317453, 0.646051]]}, {"filename": "000000361648.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.329305, 0.828427, 0.145984, 0.133021]]}, {"filename": "000000361730.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.253289, 0.831687, 0.061203, 0.153667], [0, 0.906531, 0.825479, 0.05525, 0.227667], [0, 0.87243, 0.797615, 0.017016, 0.069937], [0, 0.600023, 0.855104, 0.029141, 0.066625], [0, 0.689766, 0.803438, 0.045437, 0.141375], [0, 0.416766, 0.809927, 0.008312, 0.037896], [0, 0.539727, 0.79026, 0.010109, 0.038771], [0, 0.372039, 0.843021, 0.053641, 0.093833], [3, 0.298375, 0.870937, 0.106969, 0.062583], [0, 0.179227, 0.874719, 0.030359, 0.072312], [3, 0.115141, 0.902406, 0.133125, 0.078937], [0, 0.750312, 0.894198, 0.052187, 0.163854], [0, 0.649461, 0.845698, 0.042234, 0.093604], [0, 0.957102, 0.84275, 0.044328, 0.19775], [0, 0.197281, 0.824698, 0.0115, 0.040063]]}, {"filename": "000000361942.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.52374, 0.565617, 0.8744, 0.737078]]}, {"filename": "000000362591.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.159797, 0.650117, 0.100437, 0.267354], [0, 0.508672, 0.820433, 0.092531, 0.14377], [0, 0.9315, 0.622037, 0.137, 0.339672], [0, 0.331094, 0.642354, 0.019906, 0.09267], [0, 0.022406, 0.629883, 0.023781, 0.055972], [0, 0.428219, 0.550012, 0.015937, 0.054215]]}, {"filename": "000000362682.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.491625, 0.439602, 0.724281, 0.612834], [0, 0.679422, 0.383759, 0.04475, 0.115855], [1, 0.065937, 0.406358, 0.128594, 0.238712], [1, 0.921852, 0.507494, 0.156297, 0.39082], [0, 0.08182, 0.369368, 0.032047, 0.066745]]}, {"filename": "000000363171.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.230352, 0.498181, 0.299172, 0.359554], [1, 0.823977, 0.770681, 0.153016, 0.231033]]}, {"filename": "000000363188.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.960961, 0.593247, 0.078078, 0.2504], [0, 0.416344, 0.659553, 0.235781, 0.680894], [0, 0.693156, 0.6146, 0.422312, 0.748306], [0, 0.566703, 0.516459, 0.1155, 0.2932], [0, 0.162656, 0.711235, 0.325313, 0.546047], [0, 0.092523, 0.656929, 0.185047, 0.662941], [0, 0.313594, 0.467071, 0.083781, 0.198565], [0, 0.204062, 0.410082, 0.047906, 0.091271], [0, 0.174469, 0.445976, 0.044469, 0.108424], [0, 0.268445, 0.410482, 0.060234, 0.087553], [0, 0.026875, 0.503976, 0.05375, 0.182447], [0, 0.257586, 0.367976, 0.013141, 0.023859], [0, 0.232336, 0.402071, 0.032391, 0.073176], [0, 0.144281, 0.426188, 0.036156, 0.106682]]}, {"filename": "000000363207.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.484711, 0.495312, 0.131203, 0.428458], [0, 0.895227, 0.840448, 0.209547, 0.319104], [0, 0.856461, 0.579208, 0.287078, 0.496625], [0, 0.78257, 0.625021, 0.059203, 0.112625], [0, 0.573703, 0.554542, 0.157781, 0.392208], [0, 0.699039, 0.60426, 0.097391, 0.160896]]}, {"filename": "000000365207.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.219789, 0.615306, 0.221766, 0.34483], [1, 0.79075, 0.592134, 0.276656, 0.582024], [3, 0.631133, 0.620876, 0.087297, 0.049643], [1, 0.690133, 0.677534, 0.133234, 0.314184]]}, {"filename": "000000365511.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.879211, 0.837833, 0.241578, 0.305625]]}, {"filename": "000000366141.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.270078, 0.755177, 0.369156, 0.318646]]}, {"filename": "000000366178.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.536213, 0.35392, 0.040693, 0.02492], [1, 0.025467, 0.39297, 0.04992, 0.04026], [5, 0.57128, 0.7918, 0.130293, 0.14976]]}, {"filename": "000000366789.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.824609, 0.582026, 0.197906, 0.521335], [0, 0.486805, 0.510597, 0.070234, 0.045316], [0, 0.445406, 0.520433, 0.086906, 0.049251], [0, 0.242781, 0.458337, 0.047656, 0.044871], [1, 0.246758, 0.698993, 0.340484, 0.293115], [0, 0.195031, 0.462213, 0.027125, 0.025785], [0, 0.344719, 0.427459, 0.023656, 0.036651], [0, 0.58043, 0.446838, 0.076359, 0.033583], [0, 0.634875, 0.443981, 0.0345, 0.047635], [0, 0.806328, 0.4174, 0.030906, 0.036487], [0, 0.266109, 0.454473, 0.013937, 0.036487], [1, 0.818414, 0.750129, 0.329359, 0.375667], [1, 0.069234, 0.519801, 0.071375, 0.086393], [1, 0.067062, 0.684274, 0.133438, 0.298665], [1, 0.607961, 0.631944, 0.305203, 0.346136], [1, 0.398062, 0.551557, 0.136469, 0.159087], [1, 0.103875, 0.587916, 0.135063, 0.077564], [1, 0.683437, 0.564578, 0.12075, 0.195246], [1, 0.736523, 0.525363, 0.092609, 0.089696], [0, 0.036914, 0.45685, 0.073828, 0.147939], [0, 0.630023, 0.435504, 0.015141, 0.047119], [1, 0.387672, 0.655785, 0.144, 0.22993], [1, 0.108836, 0.528033, 0.049484, 0.085012], [1, 0.279297, 0.583126, 0.124875, 0.181897]]}, {"filename": "000000367082.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.482992, 0.53871, 0.895492, 0.90106]]}, {"filename": "000000367095.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.211633, 0.826885, 0.124828, 0.34623], [4, 0.173023, 0.519333, 0.128078, 0.225597], [4, 0.854922, 0.391183, 0.090813, 0.157635], [0, 0.264234, 0.295211, 0.070219, 0.130749], [0, 0.405562, 0.638197, 0.403313, 0.723607], [0, 0.894094, 0.34644, 0.096844, 0.247588], [4, 0.857359, 0.583279, 0.089906, 0.072131], [0, 0.518055, 0.286885, 0.063359, 0.129977], [0, 0.032836, 0.319742, 0.042672, 0.129321], [4, 0.500789, 0.296475, 0.004391, 0.007939]]}, {"filename": "000000367107.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.194463, 0.318609, 0.241402, 0.419281]]}, {"filename": "000000367891.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.717492, 0.718611, 0.412565, 0.419281], [0, 0.712361, 0.267565, 0.38652, 0.471895], [0, 0.25, 0.525907, 0.486471, 0.900899], [4, 0.856168, 0.173415, 0.257042, 0.299608]]}, {"filename": "000000368249.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.3191, 0.61686, 0.28096, 0.743816], [0, 0.66706, 0.625833, 0.17304, 0.523599], [0, 0.93527, 0.375857, 0.07954, 0.283696], [0, 0.81441, 0.438986, 0.0907, 0.330773], [0, 0.49491, 0.379529, 0.07046, 0.277802], [0, 0.86893, 0.376413, 0.06326, 0.276401], [0, 0.65784, 0.267367, 0.05648, 0.077778], [1, 0.43801, 0.347802, 0.06026, 0.105121], [0, 0.73252, 0.348792, 0.05144, 0.19971], [0, 0.98335, 0.385242, 0.0333, 0.196908]]}, {"filename": "000000368335.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.822178, 0.522633, 0.355645, 0.411641], [1, 0.293431, 0.418211, 0.219367, 0.182922], [1, 0.152555, 0.478609, 0.298102, 0.657656], [1, 0.150474, 0.484273, 0.300949, 0.667422], [1, 0.669428, 0.423773, 0.114672, 0.073297]]}, {"filename": "000000369144.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.582583, 0.631758, 0.5045, 0.445953], [5, 0.332469, 0.910391, 0.025229, 0.058906]]}, {"filename": "000000369442.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.706469, 0.476698, 0.046938, 0.134104], [0, 0.934437, 0.565281, 0.0775, 0.279146], [0, 0.891422, 0.542792, 0.066656, 0.283042], [0, 0.635391, 0.478656, 0.047187, 0.121354], [0, 0.606906, 0.487958, 0.033906, 0.136417], [0, 0.78068, 0.521865, 0.031891, 0.089271], [0, 0.509125, 0.486323, 0.045937, 0.119979], [0, 0.399008, 0.532771, 0.044016, 0.2015], [0, 0.129781, 0.494385, 0.252812, 0.988771], [0, 0.348234, 0.543031, 0.04275, 0.190021], [0, 0.434438, 0.542823, 0.037469, 0.193479], [0, 0.322508, 0.543479, 0.047828, 0.180542], [1, 0.863633, 0.542479, 0.026297, 0.105208], [6, 0.600227, 0.314354, 0.025984, 0.058]]}, {"filename": "000000369751.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.674695, 0.155086, 0.14723, 0.100734], [1, 0.434108, 0.382133, 0.324789, 0.076141], [1, 0.776549, 0.347547, 0.446901, 0.120219], [7, 0.66581, 0.302898, 0.044437, 0.039484], [7, 0.656749, 0.352867, 0.054812, 0.076516], [7, 0.602488, 0.383953, 0.105164, 0.080688], [7, 0.599847, 0.468992, 0.093075, 0.124703], [7, 0.362923, 0.810109, 0.246455, 0.330344], [7, 0.418627, 0.578883, 0.129695, 0.173578], [7, 0.74743, 0.307227, 0.018991, 0.025516]]}, {"filename": "000000370208.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.50645, 0.59032, 0.7613, 0.608587], [0, 0.04812, 0.608253, 0.09624, 0.17736], [7, 0.4109, 0.163653, 0.0966, 0.231147], [7, 0.48936, 0.16372, 0.11028, 0.22984]]}, {"filename": "000000371552.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.504383, 0.348407, 0.056016, 0.153068], [0, 0.139242, 0.508501, 0.187109, 0.082763], [0, 0.243148, 0.440351, 0.065734, 0.179016], [0, 0.913094, 0.472248, 0.0745, 0.087775], [0, 0.369656, 0.505141, 0.165563, 0.744473], [0, 0.570172, 0.421066, 0.051562, 0.190843], [3, 0.494352, 0.484707, 0.121984, 0.060328], [3, 0.491094, 0.438841, 0.119438, 0.036557]]}, {"filename": "000000372203.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.538891, 0.851306, 0.037906, 0.135167], [3, 0.338945, 0.738694, 0.232766, 0.267556]]}, {"filename": "000000372317.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.483148, 0.403376, 0.858109, 0.433715], [7, 0.032258, 0.434533, 0.017734, 0.053037]]}, {"filename": "000000372819.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.755961, 0.23115, 0.129766, 0.304695], [0, 0.901648, 0.229519, 0.105734, 0.355704], [3, 0.880203, 0.2825, 0.23925, 0.199225]]}, {"filename": "000000372980.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.564602, 0.557312, 0.212359, 0.692125], [0, 0.098875, 0.513484, 0.1955, 0.712344], [0, 0.082586, 0.396625, 0.165172, 0.78875], [0, 0.870805, 0.499891, 0.217859, 0.735031], [0, 0.379211, 0.421344, 0.095516, 0.81575], [0, 0.711758, 0.561234, 0.025422, 0.137344], [0, 0.749828, 0.581, 0.026719, 0.087563], [0, 0.774781, 0.584437, 0.03825, 0.083625], [0, 0.437992, 0.6445, 0.025078, 0.068687], [0, 0.397305, 0.612813, 0.021016, 0.107875], [0, 0.952883, 0.588766, 0.017109, 0.099781], [0, 0.935055, 0.573734, 0.018953, 0.105594], [0, 0.912289, 0.552359, 0.018141, 0.050469]]}, {"filename": "000000373492.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.465164, 0.632583, 0.461797, 0.712375]]}, {"filename": "000000373945.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.450961, 0.349562, 0.240766, 0.406625], [0, 0.701344, 0.591938, 0.379188, 0.794167], [4, 0.920945, 0.526854, 0.080203, 0.168542], [4, 0.902914, 0.770813, 0.040828, 0.113375], [4, 0.751039, 0.959479, 0.061891, 0.081042]]}, {"filename": "000000374672.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.711227, 0.62529, 0.069391, 0.289585], [4, 0.649898, 0.630768, 0.060922, 0.305934], [4, 0.673523, 0.497199, 0.035203, 0.078797]]}, {"filename": "000000374858.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.506141, 0.738305, 0.97875, 0.508578], [0, 0.183563, 0.373875, 0.349094, 0.50675]]}, {"filename": "000000375053.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.352136, 0.366242, 0.199351, 0.197672], [1, 0.091286, 0.411844, 0.169776, 0.47875], [0, 0.932517, 0.377953, 0.087763, 0.140688], [0, 0.831119, 0.355289, 0.049888, 0.083766], [7, 0.527785, 0.296547, 0.086846, 0.123906], [7, 0.621275, 0.327055, 0.054855, 0.075359], [7, 0.755794, 0.344125, 0.031678, 0.049781], [0, 0.981902, 0.363758, 0.034452, 0.096703], [7, 0.202685, 0.207797, 0.197092, 0.270125], [7, 0.651555, 0.332484, 0.027136, 0.058094], [7, 0.716734, 0.356898, 0.019687, 0.039141]]}, {"filename": "000000375430.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.86882, 0.38685, 0.262359, 0.637916], [4, 0.545227, 0.344087, 0.195141, 0.361288]]}, {"filename": "000000375493.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.494969, 0.519229, 0.201344, 0.935125], [5, 0.599164, 0.770563, 0.131391, 0.329333], [6, 0.654828, 0.214958, 0.010344, 0.014625]]}, {"filename": "000000375590.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.105617, 0.542695, 0.148328, 0.132578]]}, {"filename": "000000376365.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.74397, 0.399987, 0.10374, 0.183867], [4, 0.96378, 0.521693, 0.03728, 0.102693], [4, 0.62528, 0.475347, 0.06492, 0.032453]]}, {"filename": "000000377319.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.287251, 0.061172, 0.190278, 0.10075], [0, 0.72297, 0.043867, 0.203527, 0.073141], [0, 0.845209, 0.032273, 0.120998, 0.042703], [0, 0.635893, 0.744305, 0.511369, 0.417141], [0, 0.128654, 0.069672, 0.228538, 0.118813], [0, 0.454176, 0.052156, 0.180046, 0.084375], [0, 0.595476, 0.045234, 0.118097, 0.0715]]}, {"filename": "000000377368.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.826883, 0.097973, 0.340234, 0.195946], [3, 0.546531, 0.573524, 0.906938, 0.852952]]}, {"filename": "000000377575.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.660023, 0.528094, 0.219437, 0.449437]]}, {"filename": "000000377592.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.520523, 0.483864, 0.141891, 0.270679], [1, 0.622805, 0.480703, 0.024547, 0.047869], [0, 0.320937, 0.497635, 0.020688, 0.082389], [0, 0.168352, 0.458724, 0.017828, 0.041569], [0, 0.61025, 0.510632, 0.0155, 0.048806], [0, 0.622617, 0.513934, 0.024516, 0.043044], [0, 0.236648, 0.484859, 0.019766, 0.061991], [0, 0.104742, 0.486792, 0.023016, 0.082155], [0, 0.840523, 0.219438, 0.010172, 0.030211], [1, 0.278148, 0.252623, 0.152016, 0.11733], [0, 0.054117, 0.478642, 0.021047, 0.062061], [1, 0.602312, 0.474965, 0.022312, 0.056159]]}, {"filename": "000000378448.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.581625, 0.259589, 0.017875, 0.026812], [0, 0.401883, 0.188659, 0.013422, 0.061618], [1, 0.587469, 0.338128, 0.197875, 0.145435], [1, 0.255648, 0.355483, 0.112766, 0.13715]]}, {"filename": "000000379441.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.215766, 0.857833, 0.353594, 0.277]]}, {"filename": "000000379468.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.592437, 0.602034, 0.074437, 0.102941], [0, 0.326336, 0.395245, 0.017734, 0.035392], [0, 0.495602, 0.360797, 0.016484, 0.05027], [1, 0.025234, 0.37973, 0.047625, 0.050392], [0, 0.674305, 0.436262, 0.005109, 0.009828], [1, 0.734297, 0.142586, 0.026406, 0.008848]]}, {"filename": "000000379754.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.56208, 0.320227, 0.29156, 0.62696]]}, {"filename": "000000379800.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.239211, 0.651227, 0.039734, 0.038114], [1, 0.130234, 0.626268, 0.076562, 0.033955], [1, 0.939742, 0.705619, 0.120516, 0.117728], [1, 0.866656, 0.712769, 0.036063, 0.044523], [6, 0.917477, 0.514016, 0.055391, 0.076024], [6, 0.162289, 0.600771, 0.011516, 0.017809]]}, {"filename": "000000380588.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.275493, 0.856117, 0.162913, 0.109328], [1, 0.474438, 0.861977, 0.112638, 0.096516], [1, 0.879209, 0.761594, 0.240528, 0.059312], [7, 0.034174, 0.795352, 0.017798, 0.018047], [7, 0.014736, 0.795633, 0.020023, 0.016172]]}, {"filename": "000000381457.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.358945, 0.366812, 0.254859, 0.242625]]}, {"filename": "000000381463.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.744227, 0.766719, 0.119891, 0.135687], [1, 0.887922, 0.760552, 0.104094, 0.131729], [1, 0.592125, 0.766406, 0.044375, 0.048938], [1, 0.827859, 0.762115, 0.042906, 0.082271], [1, 0.679531, 0.775021, 0.012094, 0.059375], [1, 0.672422, 0.761625, 0.038125, 0.069208], [1, 0.810117, 0.751656, 0.025391, 0.058771], [1, 0.622234, 0.771083, 0.02575, 0.050125], [1, 0.299719, 0.703208, 0.449531, 0.219125], [6, 0.416312, 0.657542, 0.036781, 0.054208], [1, 0.945672, 0.785292, 0.108656, 0.125667]]}, {"filename": "000000381971.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.777781, 0.661206, 0.090906, 0.041475], [0, 0.302313, 0.661745, 0.083969, 0.213232], [0, 0.236594, 0.683033, 0.037625, 0.077775], [0, 0.219953, 0.684309, 0.034219, 0.069368], [0, 0.193328, 0.716522, 0.036062, 0.1537], [0, 0.506, 0.657002, 0.040344, 0.056628], [5, 0.894273, 0.833302, 0.058047, 0.166698], [1, 0.681844, 0.662518, 0.075094, 0.038806], [1, 0.983742, 0.653126, 0.026078, 0.016183], [6, 0.68025, 0.598759, 0.013844, 0.021124], [0, 0.611758, 0.803806, 0.052422, 0.182365], [0, 0.514461, 0.845667, 0.030422, 0.085667]]}, {"filename": "000000382339.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.26566, 0.495511, 0.09728, 0.528078]]}, {"filename": "000000383339.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.407672, 0.714206, 0.032875, 0.06271]]}, {"filename": "000000383386.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.072719, 0.317292, 0.145437, 0.634583]]}, {"filename": "000000383639.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.673586, 0.821083, 0.080703, 0.135958], [0, 0.119617, 0.818844, 0.024297, 0.041646], [0, 0.914445, 0.910135, 0.065672, 0.179729], [0, 0.136641, 0.816562, 0.026062, 0.05], [0, 0.086594, 0.433469, 0.031969, 0.046687], [1, 0.822141, 0.883646, 0.302219, 0.21725], [6, 0.513516, 0.312917, 0.480312, 0.613958]]}, {"filename": "000000384616.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.644055, 0.684979, 0.127797, 0.57175]]}, {"filename": "000000384935.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.322578, 0.494625, 0.645156, 0.98925]]}, {"filename": "000000385139.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.198656, 0.624719, 0.397313, 0.710094], [3, 0.471102, 0.603478, 0.936266, 0.793044]]}, {"filename": "000000385196.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.508266, 0.514602, 0.151313, 0.646019], [0, 0.829109, 0.524719, 0.140938, 0.604473], [0, 0.229859, 0.523712, 0.092531, 0.402693], [0, 0.354586, 0.524719, 0.115453, 0.402225], [0, 0.891828, 0.572553, 0.051781, 0.123138], [0, 0.666922, 0.52733, 0.057469, 0.221967], [0, 0.278531, 0.516733, 0.071438, 0.328173], [0, 0.083211, 0.497377, 0.163422, 0.986511], [0, 0.573906, 0.543712, 0.060062, 0.369719], [0, 0.945719, 0.596581, 0.092844, 0.248478], [0, 0.90157, 0.489496, 0.027297, 0.057073], [0, 0.393227, 0.535574, 0.055547, 0.232319], [3, 0.67925, 0.615375, 0.205594, 0.06911], [0, 0.293234, 0.407342, 0.027875, 0.062646]]}, {"filename": "000000385701.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.401422, 0.676896, 0.1835, 0.531333], [0, 0.66925, 0.518906, 0.064844, 0.137563], [0, 0.903703, 0.561365, 0.064687, 0.084854], [0, 0.733211, 0.559698, 0.051797, 0.083896], [0, 0.865828, 0.535656, 0.035875, 0.089021], [0, 0.853289, 0.44376, 0.052234, 0.080937], [0, 0.816508, 0.375219, 0.044703, 0.081854], [0, 0.735023, 0.50724, 0.070422, 0.111521], [0, 0.794516, 0.545667, 0.041094, 0.109583], [3, 0.524258, 0.584104, 0.127328, 0.060833], [4, 0.694703, 0.338417, 0.036375, 0.054292], [4, 0.780016, 0.59199, 0.056781, 0.020896], [4, 0.696633, 0.492365, 0.036578, 0.036313], [4, 0.635188, 0.495677, 0.0325, 0.037396], [4, 0.753133, 0.48325, 0.024828, 0.019083], [4, 0.741359, 0.345938, 0.019219, 0.0205], [4, 0.574672, 0.218979, 0.018719, 0.023792], [4, 0.570133, 0.239667, 0.018734, 0.033583], [4, 0.549789, 0.204948, 0.030328, 0.019521], [4, 0.514422, 0.205146, 0.031219, 0.015708], [0, 0.276141, 0.543, 0.056688, 0.111375], [0, 0.612547, 0.542302, 0.064375, 0.110354], [4, 0.663352, 0.59824, 0.043922, 0.014271], [4, 0.605578, 0.600094, 0.050938, 0.014771], [0, 0.656813, 0.397115, 0.050812, 0.072271], [3, 0.972109, 0.473646, 0.0555, 0.021125], [3, 0.975336, 0.424333, 0.049328, 0.013792], [3, 0.534664, 0.541833, 0.116453, 0.029], [3, 0.969727, 0.614594, 0.060547, 0.028646], [4, 0.975313, 0.253813, 0.044375, 0.070458], [0, 0.839641, 0.552594, 0.06525, 0.101438], [3, 0.56975, 0.468458, 0.113344, 0.02825], [3, 0.689883, 0.351344, 0.035609, 0.028396]]}, {"filename": "000000386876.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.291938, 0.535312, 0.332813, 0.929375], [0, 0.708711, 0.496635, 0.466859, 0.979771], [3, 0.890727, 0.337083, 0.214047, 0.19325], [3, 0.5, 0.756552, 1.0, 0.360854], [3, 0.274938, 0.401865, 0.542094, 0.259062], [3, 0.751297, 0.050646, 0.497406, 0.101292]]}, {"filename": "000000386912.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.500281, 0.586177, 0.343469, 0.575313], [4, 0.637016, 0.695063, 0.074125, 0.358917]]}, {"filename": "000000386984.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.17132, 0.627646, 0.038641, 0.046833], [1, 0.188906, 0.61376, 0.017562, 0.020438], [1, 0.592766, 0.622427, 0.775219, 0.520188], [0, 0.149078, 0.642365, 0.012719, 0.071438], [0, 0.114164, 0.635292, 0.016484, 0.047333], [0, 0.070961, 0.657458, 0.025297, 0.075083], [0, 0.045367, 0.653656, 0.022734, 0.094229], [0, 0.02282, 0.648781, 0.018016, 0.089438], [0, 0.123773, 0.627521, 0.008328, 0.051833], [0, 0.099914, 0.630656, 0.016297, 0.057896], [0, 0.087242, 0.642167, 0.016891, 0.070708], [0, 0.063219, 0.635687, 0.019, 0.053125], [0, 0.034164, 0.643229, 0.013172, 0.071833], [1, 0.198734, 0.610312, 0.013625, 0.016625], [1, 0.117617, 0.600427, 0.019172, 0.007854], [0, 0.061375, 0.591719, 0.014344, 0.024729], [0, 0.957672, 0.62649, 0.02075, 0.081271], [5, 0.09768, 0.685906, 0.024922, 0.054271], [0, 0.990141, 0.643094, 0.019719, 0.106479]]}, {"filename": "000000387244.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.7482, 0.177973, 0.17584, 0.351733]]}, {"filename": "000000387387.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.510039, 0.615177, 0.890578, 0.348021]]}, {"filename": "000000387601.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.391859, 0.566292, 0.146625, 0.310125], [1, 0.164063, 0.5755, 0.318094, 0.401792], [0, 0.709187, 0.669312, 0.351969, 0.661375], [0, 0.509414, 0.719198, 0.091703, 0.451896], [7, 0.323336, 0.749875, 0.040172, 0.091625], [0, 0.092867, 0.799188, 0.097234, 0.180042], [0, 0.487594, 0.544531, 0.026406, 0.206396], [0, 0.53982, 0.740469, 0.152641, 0.485396], [1, 0.162961, 0.60124, 0.325922, 0.461729], [0, 0.283297, 0.790198, 0.097344, 0.192188]]}, {"filename": "000000387731.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.02493, 0.480146, 0.011484, 0.018375], [0, 0.266609, 0.663219, 0.023125, 0.040604], [0, 0.387094, 0.680427, 0.028906, 0.022979], [0, 0.433141, 0.678698, 0.011937, 0.031812], [1, 0.452617, 0.500917, 0.065516, 0.020917], [1, 0.952859, 0.47575, 0.065531, 0.068208]]}, {"filename": "000000387928.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.263113, 0.289273, 0.165764, 0.199203], [0, 0.190243, 0.172211, 0.121736, 0.129859], [0, 0.648356, 0.127844, 0.086991, 0.138344], [0, 0.421319, 0.690492, 0.388009, 0.343922], [0, 0.420671, 0.369734, 0.145787, 0.159], [0, 0.155903, 0.386797, 0.16375, 0.158469], [0, 0.089907, 0.279766, 0.121296, 0.150531], [0, 0.160984, 0.248164, 0.092894, 0.096828], [0, 0.615891, 0.402414, 0.102616, 0.086922], [0, 0.072674, 0.12443, 0.087708, 0.130453], [0, 0.18691, 0.101453, 0.076597, 0.082906], [0, 0.357454, 0.405852, 0.128843, 0.099391], [3, 0.80037, 0.524562, 0.301157, 0.144562], [3, 0.410764, 0.1195, 0.236157, 0.006781], [3, 0.352697, 0.069352, 0.427245, 0.049703], [3, 0.83515, 0.338508, 0.30169, 0.020141], [3, 0.663056, 0.171859, 0.638704, 0.023875], [4, 0.278449, 0.438258, 0.303565, 0.387328], [3, 0.18206, 0.362063, 0.330185, 0.017875], [0, 0.901516, 0.391352, 0.083958, 0.070828], [3, 0.448438, 0.155992, 0.215486, 0.007453], [3, 0.701528, 0.19757, 0.567037, 0.012141], [3, 0.877141, 0.413375, 0.214375, 0.034313], [3, 0.885174, 0.392664, 0.195116, 0.024297], [3, 0.548345, 0.247234, 0.095069, 0.013344], [3, 0.247488, 0.159547, 0.024745, 0.010281], [3, 0.035035, 0.086547, 0.035949, 0.025219]]}, {"filename": "000000388258.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.35818, 0.613985, 0.623203, 0.673117], [3, 0.234664, 0.65091, 0.459234, 0.520272]]}, {"filename": "000000388487.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.05068, 0.125, 0.10136, 0.24776]]}, {"filename": "000000389174.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.274047, 0.856265, 0.050906, 0.135948], [3, 0.307844, 0.888876, 0.164437, 0.080515], [3, 0.178758, 0.899461, 0.077234, 0.0674], [0, 0.321297, 0.852002, 0.052469, 0.135386]]}, {"filename": "000000390301.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.42885, 0.512926, 0.15978, 0.571841]]}, {"filename": "000000390555.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.299617, 0.606011, 0.103797, 0.352851], [0, 0.340242, 0.399425, 0.103641, 0.243724], [0, 0.535414, 0.406885, 0.101766, 0.255011], [0, 0.897242, 0.457172, 0.122828, 0.37269], [0, 0.86293, 0.818299, 0.176391, 0.344506], [0, 0.636992, 0.811506, 0.153266, 0.368621], [0, 0.468539, 0.614276, 0.113984, 0.394115], [0, 0.619352, 0.543828, 0.102328, 0.283149], [0, 0.122477, 0.770034, 0.149016, 0.416115], [0, 0.248969, 0.832575, 0.183281, 0.307862], [0, 0.39143, 0.828943, 0.155047, 0.331264], [0, 0.541461, 0.814598, 0.151203, 0.325839], [0, 0.705578, 0.585586, 0.085031, 0.323862]]}, {"filename": "000000390685.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.612944, 0.338969, 0.235748, 0.653156], [6, 0.321121, 0.869617, 0.097617, 0.144953]]}, {"filename": "000000391375.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.50525, 0.648312, 0.249812, 0.523595], [3, 0.509961, 0.811394, 0.579578, 0.187364]]}, {"filename": "000000391397.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.95956, 0.982973, 0.0494, 0.026587], [1, 0.83603, 0.980907, 0.0297, 0.036107], [0, 0.36211, 0.938787, 0.1065, 0.122427], [0, 0.61637, 0.975693, 0.02146, 0.047173], [0, 0.41295, 0.960333, 0.04654, 0.079333], [0, 0.80388, 0.97908, 0.0214, 0.0404], [0, 0.87142, 0.98876, 0.01096, 0.02248], [1, 0.80423, 0.993013, 0.02602, 0.013653], [0, 0.50617, 0.973093, 0.01682, 0.031413], [0, 0.85856, 0.98564, 0.01008, 0.020933], [1, 0.61327, 0.99164, 0.01914, 0.01672]]}, {"filename": "000000391895.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.649055, 0.702653, 0.175703, 0.59325], [0, 0.651281, 0.479236, 0.240437, 0.835361], [0, 0.765, 0.546861, 0.056125, 0.133611], [1, 0.78332, 0.557778, 0.047859, 0.097167]]}, {"filename": "000000392776.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.388969, 0.489801, 0.326031, 0.979603], [0, 0.255484, 0.449229, 0.130469, 0.08285], [3, 0.489289, 0.548843, 0.074484, 0.106238]]}, {"filename": "000000392905.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.965141, 0.542441, 0.069625, 0.122254], [3, 0.887031, 0.645845, 0.140719, 0.089296]]}, {"filename": "000000393093.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.500687, 0.892111, 0.118375, 0.109552], [1, 0.616891, 0.799021, 0.085656, 0.080354], [1, 0.710164, 0.715413, 0.048578, 0.059693], [0, 0.924117, 0.926486, 0.086203, 0.147028], [1, 0.749172, 0.751828, 0.048125, 0.06375], [1, 0.74468, 0.692193, 0.035797, 0.042123], [1, 0.656617, 0.749682, 0.059047, 0.064033], [1, 0.760086, 0.676675, 0.037078, 0.032264]]}, {"filename": "000000394483.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.70157, 0.617234, 0.178453, 0.700227], [3, 0.334555, 0.411735, 0.049547, 0.079887], [3, 0.356398, 0.411735, 0.029703, 0.026689]]}, {"filename": "000000394510.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.283258, 0.206573, 0.566484, 0.413146], [0, 0.572148, 0.471562, 0.184609, 0.659167], [1, 0.782844, 0.226042, 0.296312, 0.393917], [1, 0.929578, 0.173354, 0.140844, 0.278667], [1, 0.818156, 0.125792, 0.363687, 0.251583]]}, {"filename": "000000394580.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.668203, 0.454281, 0.10727, 0.037125], [0, 0.462033, 0.497664, 0.567855, 0.547203], [0, 0.882006, 0.391195, 0.047855, 0.030766], [0, 0.952549, 0.396422, 0.040808, 0.024313], [4, 0.765543, 0.438, 0.089916, 0.050438]]}, {"filename": "000000395007.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.399235, 0.837742, 0.149671, 0.080016], [1, 0.341918, 0.827508, 0.102659, 0.066484], [1, 0.300047, 0.823406, 0.034635, 0.053219], [1, 0.255565, 0.820484, 0.087176, 0.048531], [1, 0.151282, 0.811617, 0.076682, 0.037828], [6, 0.158247, 0.743758, 0.033765, 0.023734]]}, {"filename": "000000395180.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.125547, 0.7371, 0.070844, 0.381185], [0, 0.156055, 0.580094, 0.210859, 0.652703], [0, 0.707367, 0.407193, 0.018703, 0.032723], [0, 0.784164, 0.408898, 0.021641, 0.033098], [0, 0.805813, 0.413659, 0.015875, 0.025946], [0, 0.758547, 0.409012, 0.024969, 0.028337], [0, 0.736195, 0.404532, 0.020453, 0.027983]]}, {"filename": "000000395204.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.776577, 0.2185, 0.133634, 0.089], [1, 0.753078, 0.3148, 0.057508, 0.01992], [5, 0.977237, 0.55432, 0.045526, 0.1452], [1, 0.853228, 0.30646, 0.016727, 0.00944], [6, 0.321306, 0.23013, 0.035826, 0.02726]]}, {"filename": "000000395491.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.80843, 0.574156, 0.383141, 0.829229], [0, 0.209148, 0.502865, 0.418297, 0.963146], [3, 0.392656, 0.757958, 0.058438, 0.073458], [3, 0.467805, 0.759031, 0.021297, 0.023729]]}, {"filename": "000000395801.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.687312, 0.650333, 0.029469, 0.052167], [1, 0.728531, 0.603135, 0.017469, 0.016146], [0, 0.710391, 0.870792, 0.173594, 0.231458], [0, 0.437117, 0.820844, 0.080828, 0.308063], [0, 0.711461, 0.688135, 0.027359, 0.123438], [0, 0.751148, 0.683323, 0.036828, 0.131854], [0, 0.361289, 0.658979, 0.026453, 0.086333], [0, 0.772937, 0.665552, 0.024531, 0.087812], [1, 0.768766, 0.593125, 0.014187, 0.014667], [3, 0.761969, 0.959604, 0.040656, 0.072917], [0, 0.734906, 0.656073, 0.021531, 0.100979], [0, 0.796336, 0.643062, 0.017234, 0.057042], [0, 0.786375, 0.629771, 0.008938, 0.029375], [0, 0.705609, 0.645292, 0.014438, 0.050333], [0, 0.771977, 0.619031, 0.010391, 0.023396], [0, 0.714555, 0.624573, 0.008547, 0.015271], [1, 0.781156, 0.590312, 0.012781, 0.014625], [1, 0.794125, 0.574771, 0.009938, 0.012833], [1, 0.792805, 0.601115, 0.015141, 0.015521], [0, 0.642234, 0.604042, 0.006813, 0.0265], [1, 0.773477, 0.610656, 0.014234, 0.017438]]}, {"filename": "000000395903.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.818992, 0.540069, 0.165078, 0.444694]]}, {"filename": "000000396518.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.76348, 0.50112, 0.28988, 0.534827]]}, {"filename": "000000396580.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.90375, 0.391779, 0.077187, 0.039093]]}, {"filename": "000000398097.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.10759, 0.328258, 0.21234, 0.159399], [1, 0.372, 0.381562, 0.07728, 0.074535], [1, 0.42679, 0.398333, 0.04154, 0.045796], [1, 0.45497, 0.405541, 0.02842, 0.033784], [1, 0.56084, 0.43455, 0.0388, 0.044474], [5, 0.41239, 0.733288, 0.32346, 0.531081], [1, 0.4721, 0.407718, 0.01988, 0.024024]]}, {"filename": "000000398438.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.713, 0.336712, 0.574, 0.668919]]}, {"filename": "000000399235.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.084508, 0.338337, 0.169016, 0.198126], [1, 0.694156, 0.416557, 0.611688, 0.71911]]}, {"filename": "000000401061.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.258227, 0.497582, 0.462369, 0.43], [1, 0.480253, 0.596291, 0.535801, 0.6]]}, {"filename": "000000401534.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.49282, 0.737082, 0.60696, 0.503366], [0, 0.52159, 0.548765, 0.65338, 0.901114], [0, 0.34432, 0.456925, 0.30628, 0.510121], [0, 0.88619, 0.490678, 0.04638, 0.161985], [0, 0.92032, 0.473765, 0.03832, 0.166465], [1, 0.12692, 0.445775, 0.02732, 0.036489], [1, 0.16526, 0.444588, 0.03532, 0.03247], [1, 0.91877, 0.414782, 0.1269, 0.060412], [1, 0.2686, 0.445266, 0.04184, 0.043608], [1, 0.2044, 0.443777, 0.04096, 0.04678], [0, 0.08539, 0.45109, 0.02094, 0.048571], [0, 0.30097, 0.456017, 0.02398, 0.051308]]}, {"filename": "000000402615.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.843062, 0.556639, 0.088844, 0.266347], [0, 0.858617, 0.405023, 0.041578, 0.088267], [0, 0.722492, 0.389766, 0.017703, 0.075644], [0, 0.750359, 0.364368, 0.025156, 0.034075], [0, 0.634031, 0.326956, 0.014625, 0.024684], [0, 0.244984, 0.347963, 0.008625, 0.019906], [0, 0.356922, 0.37829, 0.013875, 0.056159], [3, 0.91532, 0.311897, 0.036766, 0.009415], [3, 0.926656, 0.290644, 0.032062, 0.0174], [0, 0.961898, 0.31904, 0.022391, 0.057939], [0, 0.741719, 0.255562, 0.0175, 0.029766], [0, 0.764398, 0.239719, 0.018297, 0.036534], [0, 0.463539, 0.162763, 0.008453, 0.025012], [0, 0.443109, 0.165105, 0.00975, 0.029836], [0, 0.265594, 0.163244, 0.008281, 0.02719]]}, {"filename": "000000402683.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.760203, 0.468021, 0.178344, 0.139333], [5, 0.250203, 0.644198, 0.154188, 0.120771]]}, {"filename": "000000402742.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.566745, 0.510659, 0.866509, 0.880818]]}, {"filename": "000000402818.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.823992, 0.892042, 0.066078, 0.153458], [1, 0.521633, 0.725615, 0.058484, 0.060729], [1, 0.278297, 0.748365, 0.102031, 0.071729], [1, 0.382578, 0.724031, 0.050562, 0.051688], [1, 0.550188, 0.714073, 0.019875, 0.029104], [1, 0.4415, 0.718031, 0.030469, 0.027563], [1, 0.461031, 0.711594, 0.023062, 0.023187], [1, 0.516125, 0.685906, 0.015844, 0.015354], [1, 0.482219, 0.703281, 0.011531, 0.021396], [1, 0.559047, 0.687927, 0.013031, 0.012479], [1, 0.569391, 0.706635, 0.018844, 0.015646], [1, 0.469422, 0.705583, 0.021812, 0.029083], [1, 0.503453, 0.694875, 0.012375, 0.015625]]}, {"filename": "000000403982.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.665531, 0.397879, 0.326688, 0.230977], [0, 0.505516, 0.576351, 0.334281, 0.557443], [0, 0.261586, 0.608721, 0.125266, 0.228087], [0, 0.754227, 0.684906, 0.273453, 0.286445]]}, {"filename": "000000404678.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.718797, 0.768129, 0.243656, 0.463741], [4, 0.759016, 0.473694, 0.048875, 0.115012], [4, 0.973031, 0.520224, 0.053937, 0.264494], [0, 0.34857, 0.481365, 0.243703, 0.522306], [0, 0.257453, 0.672718, 0.355656, 0.645059], [0, 0.225133, 0.606941, 0.201141, 0.528471], [0, 0.454102, 0.297941, 0.091297, 0.1548], [0, 0.657398, 0.438353, 0.173609, 0.399388], [0, 0.671609, 0.292965, 0.092938, 0.167294], [0, 0.352766, 0.352847, 0.043781, 0.171624], [0, 0.591617, 0.302318, 0.054641, 0.179318], [0, 0.984164, 0.359871, 0.031672, 0.1124], [3, 0.138859, 0.647588, 0.275969, 0.698706]]}, {"filename": "000000405800.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.507602, 0.546167, 0.366547, 0.157667], [0, 0.854477, 0.5795, 0.016016, 0.0685], [0, 0.183391, 0.612865, 0.031219, 0.091479], [0, 0.824961, 0.560302, 0.022672, 0.051938], [1, 0.270422, 0.616687, 0.09525, 0.122958], [0, 0.863094, 0.546229, 0.012469, 0.058333], [6, 0.805438, 0.461094, 0.011656, 0.037271], [0, 0.164445, 0.595781, 0.012672, 0.045187], [1, 0.237047, 0.561365, 0.048031, 0.085229]]}, {"filename": "000000405999.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.68048, 0.571414, 0.619133, 0.857172]]}, {"filename": "000000406948.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.082541, 0.854648, 0.118197, 0.035797], [3, 0.183443, 0.850273, 0.086932, 0.027172], [1, 0.060094, 0.840805, 0.120187, 0.042453], [1, 0.119239, 0.826836, 0.094871, 0.024516], [1, 0.927623, 0.801391, 0.014356, 0.011719], [1, 0.877518, 0.804094, 0.018126, 0.010344]]}, {"filename": "000000407378.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.45443, 0.516973, 0.0763, 0.068667], [1, 0.75368, 0.486907, 0.11392, 0.116373], [1, 0.54873, 0.500133, 0.05422, 0.05808], [1, 0.59992, 0.50172, 0.075, 0.069947], [1, 0.39205, 0.505787, 0.02534, 0.049227], [1, 0.66398, 0.4988, 0.0674, 0.067573], [1, 0.77004, 0.486787, 0.14572, 0.122693], [1, 0.54616, 0.5016, 0.0508, 0.058027], [1, 0.93661, 0.490667, 0.12658, 0.130453], [5, 0.87654, 0.636987, 0.07944, 0.45072], [1, 0.50996, 0.507773, 0.03168, 0.041413]]}, {"filename": "000000407590.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.663789, 0.137447, 0.081922, 0.107681], [0, 0.557383, 0.625, 0.304828, 0.75], [0, 0.134789, 0.528782, 0.221391, 0.548525], [0, 0.724766, 0.392787, 0.078469, 0.407635], [0, 0.638758, 0.385656, 0.086766, 0.331827], [0, 0.533, 0.343443, 0.0315, 0.089883], [0, 0.53568, 0.276838, 0.047047, 0.068571]]}, {"filename": "000000408120.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.472914, 0.20726, 0.107484, 0.067026], [1, 0.574594, 0.206499, 0.071, 0.059508], [1, 0.39018, 0.188548, 0.021891, 0.054895], [0, 0.471547, 0.36582, 0.051063, 0.241007]]}, {"filename": "000000408405.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.526883, 0.30751, 0.142641, 0.146063], [0, 0.109555, 0.724719, 0.208984, 0.523604], [0, 0.846352, 0.729208, 0.307297, 0.541583], [0, 0.097969, 0.529281, 0.192562, 0.398646], [0, 0.287633, 0.436271, 0.146703, 0.191208], [0, 0.613484, 0.51249, 0.16675, 0.259396], [0, 0.746836, 0.619542, 0.233922, 0.346542], [4, 0.811211, 0.837438, 0.042922, 0.062125], [4, 0.040805, 0.492094, 0.081609, 0.188812], [4, 0.157906, 0.857906, 0.043469, 0.070729]]}, {"filename": "000000408818.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.937602, 0.413322, 0.057266, 0.023638], [0, 0.646344, 0.430376, 0.075375, 0.104319], [0, 0.775563, 0.444566, 0.100219, 0.45392], [0, 0.503258, 0.447113, 0.087109, 0.112582], [1, 0.886336, 0.415329, 0.042484, 0.026009], [1, 0.983227, 0.412453, 0.033547, 0.025986], [1, 0.82593, 0.415775, 0.057172, 0.025446], [0, 0.524133, 0.432805, 0.050891, 0.081479], [0, 0.367258, 0.423533, 0.056109, 0.088991], [1, 0.977289, 0.404178, 0.032453, 0.020469], [0, 0.013852, 0.45534, 0.027703, 0.113779]]}, {"filename": "000000409023.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.265328, 0.775863, 0.02625, 0.068349], [3, 0.269672, 0.778537, 0.122406, 0.056023], [0, 0.288516, 0.774418, 0.028375, 0.068124]]}, {"filename": "000000409198.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.515016, 0.494532, 0.397313, 0.80897], [3, 0.926797, 0.162553, 0.047125, 0.029977], [3, 0.985539, 0.162237, 0.028547, 0.029672], [3, 0.61575, 0.159075, 0.041406, 0.028548], [3, 0.331813, 0.13185, 0.044719, 0.043841]]}, {"filename": "000000409374.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.901531, 0.412448, 0.1115, 0.060646], [5, 0.417109, 0.760698, 0.086281, 0.352104], [0, 0.642125, 0.460135, 0.055531, 0.176396], [0, 0.529289, 0.435646, 0.012516, 0.050792], [0, 0.233227, 0.430521, 0.012484, 0.040875], [0, 0.800188, 0.447792, 0.038594, 0.133417], [0, 0.362789, 0.433604, 0.006672, 0.035042], [0, 0.375992, 0.429698, 0.011109, 0.045354], [0, 0.344133, 0.430875, 0.011891, 0.046542], [0, 0.174016, 0.431594, 0.009281, 0.034563], [0, 0.183633, 0.429927, 0.007859, 0.036229], [0, 0.220969, 0.433115, 0.0075, 0.032271], [1, 0.852828, 0.405281, 0.10625, 0.062104], [5, 0.098055, 0.451052, 0.009703, 0.028896], [1, 0.139812, 0.42874, 0.032094, 0.018729], [0, 0.30868, 0.432927, 0.013203, 0.043979], [0, 0.760648, 0.453281, 0.029641, 0.121479], [0, 0.571508, 0.439323, 0.017766, 0.079437]]}, {"filename": "000000409475.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.334219, 0.641469, 0.270563, 0.692312], [0, 0.725875, 0.662094, 0.350969, 0.675812]]}, {"filename": "000000409903.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.29894, 0.426966, 0.08044, 0.332564]]}, {"filename": "000000410680.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.662672, 0.761289, 0.32325, 0.240873], [0, 0.647687, 0.638202, 0.187469, 0.390998]]}, {"filename": "000000410880.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.07819, 0.763453, 0.15302, 0.44168], [0, 0.60814, 0.211347, 0.0506, 0.258213], [1, 0.5646, 0.19868, 0.06292, 0.061947], [1, 0.76482, 0.19584, 0.02812, 0.037013], [1, 0.24185, 0.221427, 0.07426, 0.09288], [4, 0.6172, 0.52948, 0.0222, 0.07544], [4, 0.46575, 0.760493, 0.30926, 0.34568]]}, {"filename": "000000411155.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.773945, 0.774326, 0.387578, 0.357683], [3, 0.019703, 0.774019, 0.038906, 0.261939], [1, 0.803977, 0.67383, 0.194484, 0.153853], [1, 0.628203, 0.742967, 0.118656, 0.301064], [1, 0.908141, 0.789208, 0.183719, 0.283191]]}, {"filename": "000000411303.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.400758, 0.831927, 0.135828, 0.336146], [4, 0.536438, 0.842635, 0.052, 0.063354], [0, 0.437359, 0.714604, 0.207313, 0.471917], [0, 0.532305, 0.609719, 0.099266, 0.215562], [0, 0.493391, 0.549, 0.063125, 0.152875], [0, 0.161242, 0.559, 0.139766, 0.188], [0, 0.045422, 0.628021, 0.090844, 0.272125], [0, 0.717422, 0.582031, 0.118031, 0.166437], [0, 0.935992, 0.578385, 0.128016, 0.164021], [0, 0.838461, 0.586562, 0.146734, 0.156042], [4, 0.880914, 0.956542, 0.089922, 0.086917], [4, 0.078164, 0.606177, 0.048516, 0.028646], [0, 0.282695, 0.674083, 0.157734, 0.394208], [4, 0.691406, 0.945583, 0.01725, 0.018792]]}, {"filename": "000000411754.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.507609, 0.528167, 0.795594, 0.923417], [0, 0.372469, 0.428792, 0.136812, 0.27575], [0, 0.284984, 0.362, 0.041719, 0.172667], [0, 0.420078, 0.319125, 0.043094, 0.120667], [0, 0.235687, 0.342125, 0.024125, 0.0905], [4, 0.938094, 0.878583, 0.123813, 0.242833]]}, {"filename": "000000412531.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.852219, 0.637219, 0.284094, 0.208396], [1, 0.700969, 0.634437, 0.132906, 0.15225], [7, 0.154211, 0.514198, 0.100078, 0.321729], [1, 0.56482, 0.632917, 0.086234, 0.096833]]}, {"filename": "000000413277.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.800842, 0.330163, 0.112206, 0.072941], [1, 0.427925, 0.545049, 0.85585, 0.826569], [0, 0.643015, 0.68018, 0.322075, 0.531536], [5, 0.737884, 0.802157, 0.148415, 0.289902]]}, {"filename": "000000413404.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.966898, 0.770831, 0.063891, 0.439508], [0, 0.930242, 0.770831, 0.060234, 0.371874], [3, 0.39957, 0.747143, 0.194547, 0.103138], [3, 0.812805, 0.740527, 0.186172, 0.100726], [3, 0.654844, 0.724122, 0.116562, 0.07822], [3, 0.159398, 0.747108, 0.036078, 0.087564], [3, 0.869938, 0.687998, 0.066531, 0.012061], [3, 0.681797, 0.680785, 0.080063, 0.050515]]}, {"filename": "000000413689.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.212602, 0.389038, 0.028609, 0.045258], [1, 0.043562, 0.36662, 0.082531, 0.055305], [1, 0.501141, 0.390411, 0.033969, 0.039601], [0, 0.61725, 0.638662, 0.097687, 0.401784], [0, 0.731742, 0.59669, 0.078734, 0.426338], [0, 0.437961, 0.399261, 0.020859, 0.073451], [0, 0.212625, 0.373885, 0.018562, 0.068333], [0, 0.962102, 0.387559, 0.010609, 0.060516], [0, 0.519805, 0.396408, 0.014828, 0.058685], [0, 0.615352, 0.370728, 0.008797, 0.012958], [0, 0.464414, 0.402289, 0.019953, 0.072559], [6, 0.385641, 0.327124, 0.008562, 0.014061]]}, {"filename": "000000413749.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.372687, 0.605887, 0.169187, 0.368611], [0, 0.537344, 0.634466, 0.272781, 0.361795]]}, {"filename": "000000414673.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.35392, 0.414232, 0.0958, 0.261355], [0, 0.96243, 0.893253, 0.07162, 0.191024], [0, 0.63238, 0.373268, 0.10516, 0.36991], [0, 0.41108, 0.57753, 0.17756, 0.37753], [0, 0.29496, 0.478614, 0.0686, 0.11994], [0, 0.86359, 0.386717, 0.06754, 0.102108], [0, 0.95179, 0.548223, 0.09474, 0.376747], [0, 0.40518, 0.37134, 0.05824, 0.177319], [0, 0.25956, 0.350151, 0.07964, 0.152349], [0, 0.19771, 0.425196, 0.07726, 0.286054], [4, 0.02974, 0.353705, 0.05224, 0.118012], [0, 0.1105, 0.342907, 0.06584, 0.164849], [0, 0.02099, 0.334036, 0.04198, 0.150663], [0, 0.52486, 0.397967, 0.06792, 0.148946], [4, 0.2547, 0.37125, 0.06408, 0.13509], [4, 0.41388, 0.379413, 0.0614, 0.13491], [4, 0.67358, 0.395873, 0.02624, 0.098795], [4, 0.96061, 0.354157, 0.04666, 0.05012], [4, 0.11513, 0.364157, 0.08314, 0.118976]]}, {"filename": "000000415190.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.232945, 0.711422, 0.188203, 0.049562], [1, 0.635633, 0.735141, 0.233891, 0.048969], [1, 0.35418, 0.704898, 0.112453, 0.043453], [1, 0.900055, 0.620781, 0.035766, 0.016594], [1, 0.863836, 0.62107, 0.020547, 0.010641]]}, {"filename": "000000415238.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.758898, 0.660084, 0.116016, 0.133038], [4, 0.489227, 0.881466, 0.276141, 0.213354]]}, {"filename": "000000416095.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.833781, 0.801328, 0.106937, 0.199531], [0, 0.322312, 0.782367, 0.046667, 0.124078], [0, 0.666094, 0.765914, 0.061104, 0.124578], [0, 0.397531, 0.80732, 0.109146, 0.212828], [0, 0.587656, 0.761562, 0.052688, 0.116812], [0, 0.442875, 0.754945, 0.059167, 0.098016], [0, 0.49175, 0.767141, 0.041167, 0.116031], [0, 0.203708, 0.712695, 0.024417, 0.029047], [0, 0.541104, 0.755984, 0.036417, 0.098906], [0, 0.629625, 0.790781, 0.0435, 0.049094], [0, 0.710417, 0.737281, 0.026833, 0.05325]]}, {"filename": "000000416125.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.360154, 0.305648, 0.141965, 0.112172], [5, 0.82579, 0.782453, 0.172081, 0.351281]]}, {"filename": "000000416991.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.878766, 0.526033, 0.060437, 0.088541], [4, 0.737937, 0.459119, 0.029031, 0.034468], [4, 0.664617, 0.483602, 0.017859, 0.055532], [4, 0.938156, 0.507128, 0.021687, 0.07304], [4, 0.563367, 0.485334, 0.025391, 0.069757], [4, 0.685469, 0.484802, 0.025, 0.110699], [4, 0.971383, 0.514574, 0.021047, 0.128784], [4, 0.796523, 0.500957, 0.027484, 0.108116], [4, 0.62343, 0.482568, 0.028047, 0.101064], [4, 0.597445, 0.487158, 0.029391, 0.077903], [4, 0.53607, 0.466216, 0.027141, 0.099179], [4, 0.85893, 0.470198, 0.021578, 0.044286], [3, 0.519078, 0.454271, 0.045531, 0.01231], [3, 0.800539, 0.513617, 0.036297, 0.027112], [3, 0.817016, 0.474529, 0.082656, 0.019453], [4, 0.698836, 0.471125, 0.014859, 0.079574]]}, {"filename": "000000417779.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.357414, 0.255125, 0.090016, 0.091], [1, 0.016297, 0.274615, 0.032594, 0.098771], [5, 0.522469, 0.7, 0.219094, 0.573042], [1, 0.310531, 0.243948, 0.018781, 0.058313], [0, 0.750977, 0.222594, 0.029359, 0.089021], [6, 0.257875, 0.114094, 0.026187, 0.033521]]}, {"filename": "000000419019.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.656229, 0.87382, 0.686042, 0.225953], [4, 0.106125, 0.721523, 0.21225, 0.551578], [0, 0.461031, 0.562602, 0.571229, 0.799141], [0, 0.052844, 0.648023, 0.100396, 0.488516], [1, 0.810479, 0.145508, 0.368542, 0.280891]]}, {"filename": "000000419064.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.544684, 0.502141, 0.675172, 0.817562]]}, {"filename": "000000419232.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.354781, 0.443549, 0.126906, 0.300334], [0, 0.305414, 0.21738, 0.198016, 0.430501], [0, 0.454484, 0.402203, 0.104281, 0.274927], [0, 0.810703, 0.668424, 0.165594, 0.475595], [0, 0.160977, 0.570491, 0.245641, 0.373257], [0, 0.764328, 0.185658, 0.091563, 0.188392], [0, 0.844648, 0.128069, 0.125672, 0.167912], [0, 0.933219, 0.441441, 0.133563, 0.349102], [0, 0.976586, 0.275574, 0.046828, 0.268351], [4, 0.252117, 0.501889, 0.116828, 0.17], [4, 0.200984, 0.278653, 0.085781, 0.076388], [4, 0.590203, 0.318967, 0.065156, 0.239645], [4, 0.652688, 0.336367, 0.055813, 0.118246], [4, 0.708695, 0.258664, 0.020234, 0.078664], [4, 0.730852, 0.271211, 0.042797, 0.047557], [4, 0.605281, 0.376054, 0.072, 0.116326], [4, 0.901078, 0.70786, 0.194, 0.446117], [4, 0.433883, 0.205908, 0.053609, 0.052902], [4, 0.759555, 0.894384, 0.131766, 0.211232], [4, 0.123695, 0.433372, 0.145484, 0.148873], [0, 0.6265, 0.784269, 0.3145, 0.408998], [0, 0.188547, 0.12737, 0.06325, 0.136367], [4, 0.210078, 0.228935, 0.050781, 0.056534], [0, 0.917336, 0.222401, 0.041859, 0.09382], [4, 0.532484, 0.340094, 0.073937, 0.241148]]}, {"filename": "000000419408.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.263555, 0.483706, 0.085922, 0.064382], [0, 0.45943, 0.45324, 0.210891, 0.386527], [0, 0.874805, 0.555163, 0.225672, 0.335594], [3, 0.793078, 0.651678, 0.227469, 0.256177], [3, 0.887156, 0.498998, 0.092719, 0.048392], [1, 0.800859, 0.47465, 0.100438, 0.059837], [3, 0.08482, 0.518601, 0.055797, 0.032308], [3, 0.988586, 0.56338, 0.022828, 0.155478], [1, 0.088297, 0.468951, 0.118938, 0.061772], [1, 0.565758, 0.465117, 0.033172, 0.031352]]}, {"filename": "000000419974.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.43338, 0.523375, 0.037559, 0.086969], [0, 0.310223, 0.387234, 0.109178, 0.114437], [3, 0.21223, 0.443578, 0.211127, 0.121312], [0, 0.643838, 0.427211, 0.438005, 0.538328]]}, {"filename": "000000420066.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.483551, 0.533766, 0.493271, 0.787]]}, {"filename": "000000420069.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.173305, 0.661794, 0.154047, 0.676411], [0, 0.366508, 0.440842, 0.019391, 0.063042], [0, 0.647805, 0.443578, 0.018016, 0.058053], [0, 0.819039, 0.469617, 0.031609, 0.121729], [0, 0.896859, 0.459442, 0.010969, 0.032188], [0, 0.918734, 0.459048, 0.023156, 0.058578], [0, 0.076883, 0.447582, 0.043234, 0.062779], [0, 0.438539, 0.42733, 0.012203, 0.030284], [0, 0.011516, 0.460919, 0.014063, 0.051641], [0, 0.507531, 0.534759, 0.073656, 0.29372], [4, 0.065391, 0.445777, 0.021813, 0.034792]]}, {"filename": "000000420231.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.13397, 0.718236, 0.1841, 0.360345], [1, 0.79759, 0.418833, 0.18082, 0.070663], [0, 0.55191, 0.591989, 0.89618, 0.792308], [1, 0.21942, 0.37191, 0.28636, 0.195491], [4, 0.89435, 0.564019, 0.06142, 0.111114]]}, {"filename": "000000420236.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.047828, 0.713178, 0.048844, 0.194252], [0, 0.843344, 0.706379, 0.053531, 0.213505], [0, 0.904742, 0.729836, 0.038734, 0.097804], [0, 0.982938, 0.740689, 0.029625, 0.067687], [0, 0.951094, 0.630035, 0.017875, 0.025444], [0, 0.879281, 0.720794, 0.034938, 0.18271], [0, 0.939453, 0.724393, 0.036656, 0.102196], [0, 0.988219, 0.690736, 0.010344, 0.036893], [0, 0.749156, 0.635783, 0.005687, 0.016846], [1, 0.514156, 0.677921, 0.687844, 0.52986]]}, {"filename": "000000420629.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.081674, 0.498633, 0.162881, 0.185266], [1, 0.425258, 0.51807, 0.11281, 0.249234], [1, 0.164614, 0.449164, 0.087213, 0.111703], [1, 0.782295, 0.504711, 0.43541, 0.468609], [1, 0.300644, 0.525773, 0.201429, 0.204141], [1, 0.186019, 0.513664, 0.156159, 0.190828], [1, 0.505738, 0.571641, 0.12548, 0.186313], [1, 0.919461, 0.574359, 0.161077, 0.360906], [1, 0.867927, 0.160555, 0.264145, 0.108547]]}, {"filename": "000000420916.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.891648, 0.611625, 0.132609, 0.35215], [0, 0.768734, 0.654012, 0.146344, 0.366925], [3, 0.703648, 0.80075, 0.584266, 0.3753], [0, 0.582859, 0.687638, 0.275281, 0.408975]]}, {"filename": "000000421804.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.424305, 0.167406, 0.122078, 0.312358], [1, 0.826453, 0.529858, 0.13725, 0.100566], [1, 0.931086, 0.520566, 0.090234, 0.084811], [7, 0.7345, 0.490354, 0.009781, 0.031415], [7, 0.986586, 0.486592, 0.005922, 0.019599], [6, 0.632563, 0.398573, 0.01775, 0.042712]]}, {"filename": "000000422517.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.28324, 0.470856, 0.35448, 0.63], [3, 0.49651, 0.558889, 0.78098, 0.5]]}, {"filename": "000000422836.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.355547, 0.722417, 0.017719, 0.073167], [4, 0.455031, 0.722708, 0.040125, 0.083417], [4, 0.314156, 0.723146, 0.020313, 0.083208], [4, 0.277844, 0.727792, 0.057875, 0.10575], [0, 0.505609, 0.488771, 0.414594, 0.973042], [0, 0.921344, 0.49775, 0.157312, 0.982], [0, 0.091844, 0.7185, 0.042375, 0.145583], [0, 0.398109, 0.69275, 0.049406, 0.07125], [4, 0.496156, 0.725812, 0.025625, 0.089125], [4, 0.149094, 0.730958, 0.043688, 0.089583], [4, 0.401344, 0.72075, 0.040875, 0.081333], [4, 0.506047, 0.715562, 0.024719, 0.070125], [4, 0.057516, 0.725208, 0.048656, 0.09175], [4, 0.333281, 0.724062, 0.017875, 0.083708], [4, 0.346344, 0.723688, 0.010188, 0.064292]]}, {"filename": "000000424307.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.872365, 0.704977, 0.116979, 0.343328], [4, 0.945937, 0.749758, 0.106708, 0.225234], [4, 0.98074, 0.486, 0.038521, 0.059781]]}, {"filename": "000000424408.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.28358, 0.589438, 0.465047, 0.510063], [0, 0.614812, 0.458359, 0.4023, 0.373844], [0, 0.667782, 0.363477, 0.346502, 0.234922]]}, {"filename": "000000424788.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.56082, 0.762927, 0.549891, 0.280271]]}, {"filename": "000000425158.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.352156, 0.817898, 0.147396, 0.130453], [1, 0.5, 0.859555, 1.0, 0.258422]]}, {"filename": "000000425187.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.500563, 0.663677, 0.330344, 0.550562]]}, {"filename": "000000425361.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.140828, 0.630631, 0.205625, 0.704969], [0, 0.368078, 0.480896, 0.408594, 0.876415]]}, {"filename": "000000426049.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.957586, 0.738208, 0.084828, 0.501125], [7, 0.813055, 0.72074, 0.156391, 0.468604], [7, 0.651625, 0.678135, 0.126125, 0.410313], [7, 0.419664, 0.604115, 0.117984, 0.391021], [7, 0.048305, 0.531333, 0.069484, 0.217375], [7, 0.532383, 0.628844, 0.120516, 0.417479], [7, 0.108469, 0.535729, 0.067687, 0.252542], [7, 0.270414, 0.487396, 0.070766, 0.158083], [7, 0.209414, 0.585719, 0.099359, 0.251938], [7, 0.145508, 0.567635, 0.058734, 0.229896], [7, 0.339953, 0.440396, 0.068187, 0.058292], [7, 0.026398, 0.485083, 0.051609, 0.131958], [7, 0.304398, 0.602833, 0.131359, 0.288458]]}, {"filename": "000000426300.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.74285, 0.7742, 0.33226, 0.4516], [0, 0.88351, 0.718293, 0.13206, 0.356053], [0, 0.17949, 0.749067, 0.15674, 0.267413], [0, 0.12909, 0.677467, 0.08962, 0.251733], [0, 0.32526, 0.814147, 0.17384, 0.371707], [0, 0.62363, 0.516053, 0.04646, 0.108427], [0, 0.52954, 0.600013, 0.08428, 0.196827], [0, 0.6, 0.632107, 0.15252, 0.172213], [0, 0.47933, 0.792333, 0.1185, 0.413787], [0, 0.27038, 0.65436, 0.05804, 0.307547], [0, 0.47915, 0.560227, 0.0479, 0.144453], [0, 0.67781, 0.517147, 0.0401, 0.090453], [0, 0.36068, 0.642387, 0.07292, 0.258907]]}, {"filename": "000000427055.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.535281, 0.67676, 0.072438, 0.065604], [1, 0.870414, 0.63101, 0.022016, 0.024146], [1, 0.829313, 0.633281, 0.020219, 0.021479], [1, 0.929367, 0.627052, 0.027391, 0.019937], [6, 0.797531, 0.599198, 0.017656, 0.023021], [1, 0.952445, 0.632688, 0.029391, 0.027958], [1, 0.471914, 0.646302, 0.047547, 0.041771], [1, 0.992586, 0.641552, 0.014828, 0.022396]]}, {"filename": "000000427500.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.60535, 0.591344, 0.399486, 0.256812], [1, 0.050654, 0.111258, 0.101308, 0.204734]]}, {"filename": "000000428605.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.460362, 0.591766, 0.577967, 0.752812], [0, 0.79965, 0.348953, 0.035935, 0.061]]}, {"filename": "000000428769.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.31444, 0.5, 0.62888, 1.0], [7, 0.75656, 0.491003, 0.48336, 0.977515]]}, {"filename": "000000428867.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.97794, 0.554985, 0.03568, 0.205225], [4, 0.86025, 0.424625, 0.0279, 0.050751]]}, {"filename": "000000429690.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.867555, 0.472518, 0.216141, 0.108407], [0, 0.516508, 0.551686, 0.223391, 0.613489], [0, 0.177664, 0.5, 0.349328, 1.0], [0, 0.57207, 0.419333, 0.076203, 0.266066], [0, 0.658648, 0.443712, 0.103078, 0.307658], [0, 0.737281, 0.424918, 0.102125, 0.274848], [0, 0.900688, 0.426241, 0.083844, 0.22719], [0, 0.946875, 0.346253, 0.05625, 0.133489], [0, 0.747305, 0.356288, 0.042766, 0.166112], [0, 0.843922, 0.428665, 0.133813, 0.247822], [0, 0.443469, 0.322752, 0.045125, 0.154122], [3, 0.763414, 0.565269, 0.418734, 0.033115]]}, {"filename": "000000429980.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.107508, 0.672873, 0.040734, 0.209693], [0, 0.073914, 0.672456, 0.035547, 0.254298], [3, 0.474219, 0.76057, 0.03175, 0.095965], [3, 0.094461, 0.735154, 0.016172, 0.053465], [0, 0.496617, 0.717346, 0.027422, 0.127939], [0, 0.521953, 0.672697, 0.022094, 0.140746], [0, 0.550305, 0.729627, 0.044109, 0.152149]]}, {"filename": "000000430073.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.591391, 0.494417, 0.055844, 0.107917], [4, 0.804875, 0.50099, 0.042594, 0.132313], [4, 0.893805, 0.484583, 0.057984, 0.10275], [4, 0.946164, 0.583917, 0.106734, 0.137792], [4, 0.722078, 0.539365, 0.070188, 0.123271], [4, 0.745008, 0.414948, 0.022828, 0.031521], [4, 0.846555, 0.598229, 0.100828, 0.142625], [0, 0.048031, 0.66325, 0.096062, 0.526417], [0, 0.372617, 0.272021, 0.024234, 0.070792], [0, 0.823516, 0.53725, 0.094125, 0.212667], [0, 0.154211, 0.589896, 0.133141, 0.496625], [0, 0.297258, 0.327385, 0.034359, 0.118104], [0, 0.200109, 0.31449, 0.036031, 0.116854], [0, 0.743625, 0.510333, 0.080563, 0.182625], [0, 0.924172, 0.523062, 0.138375, 0.190375], [0, 0.315969, 0.300823, 0.025219, 0.111563], [0, 0.658539, 0.433448, 0.069984, 0.215604], [0, 0.419664, 0.282042, 0.034641, 0.051208], [1, 0.01907, 0.352667, 0.038141, 0.06775], [0, 0.284133, 0.414146, 0.061109, 0.082958]]}, {"filename": "000000430086.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.091503, 0.62258, 0.055612, 0.04092], [1, 0.17262, 0.62838, 0.06242, 0.03424], [0, 0.644029, 0.80193, 0.533165, 0.39614], [0, 0.278032, 0.66662, 0.085106, 0.18556], [5, 0.469096, 0.6822, 0.011809, 0.01428]]}, {"filename": "000000430155.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.351336, 0.477149, 0.051453, 0.049719], [1, 0.364836, 0.443494, 0.036109, 0.036627], [1, 0.598953, 0.447871, 0.0435, 0.056667], [1, 0.566453, 0.446737, 0.026594, 0.047329], [1, 0.482219, 0.40993, 0.029, 0.03002], [1, 0.465453, 0.394869, 0.0195, 0.02251], [1, 0.424281, 0.397902, 0.0185, 0.021707], [1, 0.384484, 0.417661, 0.027062, 0.024558], [0, 0.640422, 0.450221, 0.016531, 0.056064], [0, 0.334852, 0.425371, 0.010922, 0.029618], [1, 0.080672, 0.490582, 0.05675, 0.05751], [6, 0.618578, 0.384116, 0.013187, 0.015823], [1, 0.515023, 0.402279, 0.021609, 0.02247], [1, 0.350602, 0.449538, 0.047859, 0.027992], [1, 0.02957, 0.493082, 0.056547, 0.097169], [1, 0.498414, 0.395542, 0.010109, 0.014659]]}, {"filename": "000000430871.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.175078, 0.545374, 0.226938, 0.876402]]}, {"filename": "000000431795.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.78118, 0.494385, 0.436516, 0.984271], [0, 0.482258, 0.438469, 0.039859, 0.125438]]}, {"filename": "000000433204.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.435664, 0.596667, 0.104266, 0.13775], [0, 0.451187, 0.581823, 0.038938, 0.144479], [0, 0.679109, 0.554271, 0.033437, 0.133167], [0, 0.704039, 0.556188, 0.024891, 0.113125], [0, 0.73543, 0.547625, 0.023703, 0.108375], [1, 0.639273, 0.530448, 0.053484, 0.027104], [1, 0.753914, 0.519719, 0.032297, 0.047229], [0, 0.457398, 0.543948, 0.011484, 0.023687]]}, {"filename": "000000433515.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.668344, 0.66751, 0.011313, 0.041771], [0, 0.717484, 0.654385, 0.014813, 0.056437], [0, 0.454266, 0.667198, 0.023719, 0.063271], [0, 0.570367, 0.657729, 0.013766, 0.051958], [0, 0.545391, 0.681271, 0.012031, 0.024042], [1, 0.226773, 0.669906, 0.051266, 0.037021], [0, 0.795438, 0.643865, 0.008219, 0.035479], [0, 0.486844, 0.665792, 0.014969, 0.059875], [0, 0.632031, 0.666, 0.016344, 0.048292], [0, 0.867078, 0.616833, 0.011219, 0.027125], [0, 0.648164, 0.668688, 0.014891, 0.042333], [0, 0.501461, 0.666031, 0.017297, 0.054146], [1, 0.686242, 0.620031, 0.015203, 0.016187], [1, 0.831516, 0.599958, 0.024844, 0.018167], [1, 0.963602, 0.570958, 0.036109, 0.012542], [1, 0.929086, 0.578458, 0.028078, 0.014875], [1, 0.616211, 0.626219, 0.024359, 0.018396], [1, 0.74293, 0.61374, 0.029859, 0.019187], [1, 0.378047, 0.645521, 0.018531, 0.015042], [1, 0.777594, 0.607354, 0.024062, 0.019417], [0, 0.138063, 0.673625, 0.015875, 0.0305], [1, 0.426469, 0.64699, 0.051594, 0.022646], [1, 0.847156, 0.597854, 0.017281, 0.013625], [1, 0.85693, 0.586219, 0.012734, 0.017688], [0, 0.685727, 0.661594, 0.017109, 0.059104]]}, {"filename": "000000433774.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.780383, 0.543138, 0.178422, 0.570773], [7, 0.686766, 0.508466, 0.069281, 0.498852], [1, 0.207523, 0.311628, 0.014328, 0.017728], [1, 0.235297, 0.315035, 0.021, 0.027354]]}, {"filename": "000000433892.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.168711, 0.689979, 0.034328, 0.07925], [1, 0.565578, 0.593698, 0.144156, 0.060104], [6, 0.205656, 0.494021, 0.066094, 0.088125]]}, {"filename": "000000434204.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.3539, 0.342928, 0.23892, 0.672643], [0, 0.16719, 0.068468, 0.06626, 0.12997], [3, 0.70706, 0.332958, 0.1538, 0.217508], [3, 0.90484, 0.862538, 0.19032, 0.251682], [3, 0.64552, 0.147357, 0.03484, 0.076156], [0, 0.39674, 0.075541, 0.04756, 0.148679], [0, 0.5674, 0.229865, 0.15568, 0.45973]]}, {"filename": "000000434992.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.28314, 0.506747, 0.49888, 0.964053], [0, 0.74645, 0.41424, 0.2057, 0.572], [0, 0.65747, 0.304733, 0.0853, 0.23976], [3, 0.33884, 0.697987, 0.45908, 0.59416], [0, 0.65258, 0.179347, 0.05356, 0.077627], [0, 0.60324, 0.20524, 0.05992, 0.04616], [0, 0.48435, 0.211813, 0.04662, 0.057067], [0, 0.51008, 0.198067, 0.01596, 0.056133], [0, 0.08163, 0.287587, 0.16326, 0.264133], [3, 0.09546, 0.562333, 0.18312, 0.26232], [4, 0.65511, 0.36616, 0.08346, 0.23424], [3, 0.19064, 0.54416, 0.05412, 0.201867]]}, {"filename": "000000435081.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.2923, 0.57995, 0.0774, 0.15482]]}, {"filename": "000000436617.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.749867, 0.607859, 0.267109, 0.573035], [0, 0.094008, 0.477165, 0.160797, 0.342141], [0, 0.673773, 0.447494, 0.053078, 0.224824], [0, 0.569305, 0.411882, 0.184016, 0.266447], [4, 0.04982, 0.5976, 0.093141, 0.267906], [4, 0.130734, 0.547624, 0.105344, 0.240753], [0, 0.451, 0.416812, 0.114656, 0.221435], [0, 0.621938, 0.383082, 0.053094, 0.162024], [0, 0.222453, 0.347706, 0.041188, 0.058612], [0, 0.185758, 0.378659, 0.049922, 0.109365], [0, 0.681078, 0.505694, 0.051844, 0.131671], [0, 0.411648, 0.502565, 0.239141, 0.406494], [0, 0.245484, 0.706376, 0.308906, 0.564047], [3, 0.72768, 0.671, 0.544641, 0.636353], [4, 0.050258, 0.892859, 0.097078, 0.202894], [0, 0.250844, 0.397365, 0.097906, 0.110588], [0, 0.030109, 0.392247, 0.057563, 0.107271]]}, {"filename": "000000436883.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.947281, 0.614228, 0.105438, 0.493539], [1, 0.515906, 0.497755, 0.721375, 0.815748], [0, 0.910016, 0.46196, 0.0305, 0.128005], [0, 0.883445, 0.463064, 0.023047, 0.131829], [0, 0.886164, 0.394133, 0.015922, 0.03715], [1, 0.020695, 0.413492, 0.026609, 0.125843], [1, 0.092844, 0.432257, 0.141281, 0.366556], [0, 0.301844, 0.36791, 0.02325, 0.091544], [1, 0.945039, 0.61728, 0.109922, 0.518504], [0, 0.904094, 0.386366, 0.026656, 0.100475], [0, 0.922242, 0.382126, 0.014984, 0.082257]]}, {"filename": "000000437465.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.608219, 0.424156, 0.384219, 0.610729], [0, 0.846352, 0.565167, 0.307297, 0.474167], [0, 0.698977, 0.29175, 0.072359, 0.070792], [0, 0.772484, 0.196135, 0.016719, 0.050521], [0, 0.93125, 0.231385, 0.060812, 0.177937], [0, 0.89357, 0.286448, 0.130578, 0.138896], [0, 0.310508, 0.48176, 0.232953, 0.435104], [0, 0.30193, 0.208594, 0.093609, 0.131896], [4, 0.066977, 0.444594, 0.130672, 0.211563]]}, {"filename": "000000437514.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.357266, 0.632581, 0.640344, 0.555054], [0, 0.59875, 0.350531, 0.096969, 0.083975]]}, {"filename": "000000437540.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.726352, 0.460583, 0.283797, 0.657667]]}, {"filename": "000000437859.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.940461, 0.880844, 0.119078, 0.139104], [1, 0.057172, 0.858969, 0.114344, 0.204021], [0, 0.705352, 0.814729, 0.058984, 0.19775], [0, 0.657703, 0.789479, 0.037437, 0.118625], [1, 0.428398, 0.862073, 0.511297, 0.190937], [7, 0.700523, 0.836229, 0.036672, 0.147708], [1, 0.032828, 0.74099, 0.045031, 0.044854], [1, 0.107656, 0.780094, 0.017813, 0.019229], [1, 0.185812, 0.814406, 0.033656, 0.021271], [1, 0.13168, 0.81299, 0.033484, 0.016729], [1, 0.154508, 0.801875, 0.024953, 0.023917], [1, 0.104484, 0.767406, 0.0205, 0.017396]]}, {"filename": "000000438028.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.722542, 0.63582, 0.200833, 0.413297]]}, {"filename": "000000438095.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.3455, 0.421854, 0.057219, 0.132042], [0, 0.234148, 0.520781, 0.044109, 0.159854], [0, 0.176273, 0.425646, 0.057234, 0.150625], [0, 0.878594, 0.765594, 0.154656, 0.223562], [0, 0.859727, 0.747281, 0.055734, 0.163562], [0, 0.980414, 0.447427, 0.039172, 0.077479], [4, 0.907305, 0.733885, 0.048672, 0.058938], [4, 0.989398, 0.722469, 0.021203, 0.035521], [0, 0.970453, 0.731677, 0.059094, 0.088312], [4, 0.899695, 0.777177, 0.195391, 0.132938], [4, 0.737664, 0.841844, 0.327953, 0.143396]]}, {"filename": "000000438862.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.212148, 0.480902, 0.286359, 0.741569], [0, 0.239672, 0.220726, 0.085656, 0.283794], [3, 0.950344, 0.231452, 0.087562, 0.047775], [3, 0.418141, 0.241288, 0.046187, 0.043981], [3, 0.496438, 0.233302, 0.052125, 0.029508], [0, 0.538953, 0.58363, 0.298063, 0.789602], [0, 0.714586, 0.351909, 0.152609, 0.647799], [0, 0.783172, 0.211792, 0.058156, 0.236838], [0, 0.771227, 0.251557, 0.031734, 0.081101]]}, {"filename": "000000439325.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.381477, 0.607, 0.390359, 0.786], [3, 0.290539, 0.800677, 0.581078, 0.398646], [3, 0.976828, 0.419844, 0.033531, 0.028771], [0, 0.987258, 0.395406, 0.025484, 0.074979]]}, {"filename": "000000439623.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.496438, 0.369062, 0.685875, 0.273656], [1, 0.817, 0.319914, 0.146875, 0.082828], [1, 0.093708, 0.322008, 0.08025, 0.037734], [1, 0.153969, 0.327141, 0.052479, 0.031469], [5, 0.417604, 0.530414, 0.471292, 0.697984], [1, 0.06625, 0.32407, 0.058458, 0.031922], [1, 0.020417, 0.308414, 0.040833, 0.083453]]}, {"filename": "000000439854.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.51579, 0.853769, 0.10818, 0.227417], [0, 0.3112, 0.578949, 0.17308, 0.225225], [0, 0.86671, 0.744174, 0.05806, 0.139459], [0, 0.61555, 0.765646, 0.09698, 0.351051], [0, 0.52591, 0.786246, 0.10974, 0.342282], [0, 0.94422, 0.58009, 0.02128, 0.050751], [0, 0.66839, 0.672508, 0.02382, 0.074084], [0, 0.30687, 0.773333, 0.08854, 0.158739], [0, 0.67501, 0.600901, 0.0159, 0.068529], [0, 0.97931, 0.617868, 0.0125, 0.04036], [0, 0.988, 0.643844, 0.01812, 0.026727], [0, 0.63081, 0.640586, 0.02478, 0.094144]]}, {"filename": "000000440298.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.533555, 0.499094, 0.677859, 0.903813], [1, 0.923688, 0.674333, 0.152625, 0.233292], [0, 0.178789, 0.613479, 0.040547, 0.193208], [0, 0.036289, 0.516865, 0.025047, 0.051479], [5, 0.292781, 0.828937, 0.043125, 0.077417], [1, 0.167969, 0.523031, 0.020031, 0.021479]]}, {"filename": "000000440390.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.032289, 0.646729, 0.063172, 0.0905], [1, 0.498602, 0.494396, 0.997203, 0.988792], [0, 0.480836, 0.574531, 0.228828, 0.169396], [0, 0.781352, 0.589583, 0.114484, 0.154583], [0, 0.257602, 0.603604, 0.103047, 0.105875], [0, 0.065227, 0.612417, 0.023953, 0.068125], [0, 0.982109, 0.590781, 0.035406, 0.159021], [0, 0.366555, 0.588948, 0.055234, 0.106187], [1, 0.028859, 0.690417, 0.056281, 0.122042], [0, 0.837641, 0.587646, 0.070781, 0.137083], [0, 0.449875, 0.577927, 0.063406, 0.093563], [0, 0.585641, 0.57426, 0.04975, 0.085437], [0, 0.658563, 0.563042, 0.095406, 0.116333], [0, 0.795945, 0.555906, 0.040109, 0.076479]]}, {"filename": "000000441182.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.905977, 0.208802, 0.188047, 0.153188]]}, {"filename": "000000441211.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.509291, 0.348, 0.079109, 0.042563], [1, 0.482034, 0.343734, 0.089332, 0.041531], [1, 0.562935, 0.352, 0.054049, 0.028406]]}, {"filename": "000000441468.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.300962, 0.250461, 0.13892, 0.071797], [1, 0.543568, 0.228023, 0.099859, 0.067766], [1, 0.605692, 0.216977, 0.053592, 0.035047], [5, 0.702383, 0.747773, 0.251385, 0.345141], [0, 0.727324, 0.215633, 0.012535, 0.037172], [0, 0.676678, 0.213961, 0.013967, 0.023641], [0, 0.455387, 0.219156, 0.008286, 0.017656], [0, 0.384061, 0.22093, 0.008638, 0.017578], [0, 0.467958, 0.218461, 0.009061, 0.017359], [0, 0.714988, 0.216047, 0.009413, 0.031344], [0, 0.887899, 0.239367, 0.045657, 0.092266]]}, {"filename": "000000441586.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.205641, 0.516129, 0.317031, 0.318282], [1, 0.642641, 0.435906, 0.048844, 0.184659], [1, 0.891992, 0.425024, 0.107797, 0.104259], [0, 0.557875, 0.341224, 0.158531, 0.599106], [0, 0.405687, 0.3326, 0.030844, 0.120682], [1, 0.597656, 0.634835, 0.332781, 0.631459], [0, 0.168688, 0.298694, 0.025562, 0.046094], [0, 0.191953, 0.296718, 0.025406, 0.050141], [6, 0.839383, 0.201071, 0.037578, 0.053012], [0, 0.646273, 0.376471, 0.054953, 0.109412]]}, {"filename": "000000442480.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.148938, 0.581518, 0.1445, 0.114753], [0, 0.411789, 0.597776, 0.022203, 0.067788], [0, 0.489961, 0.543576, 0.027266, 0.085882]]}, {"filename": "000000442746.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.67625, 0.375573, 0.1049, 0.052747], [0, 0.41629, 0.445387, 0.1045, 0.330347]]}, {"filename": "000000442840.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.88382, 0.628091, 0.232359, 0.721335], [3, 0.127438, 0.631077, 0.254875, 0.705621], [0, 0.129109, 0.372588, 0.258219, 0.745176]]}, {"filename": "000000443592.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.845344, 0.598162, 0.048594, 0.033888], [0, 0.711641, 0.656768, 0.222594, 0.411569], [0, 0.346281, 0.617869, 0.271656, 0.756159], [1, 0.78993, 0.664403, 0.043953, 0.093021], [1, 0.801266, 0.587752, 0.033531, 0.053396], [1, 0.949219, 0.632272, 0.059969, 0.035129], [0, 0.898797, 0.61363, 0.007469, 0.022389]]}, {"filename": "000000443602.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.813203, 0.59775, 0.372469, 0.777542], [0, 0.166563, 0.401344, 0.068562, 0.088229], [0, 0.076711, 0.401562, 0.120797, 0.098958], [0, 0.783039, 0.28576, 0.034578, 0.099521], [0, 0.635477, 0.563042, 0.365047, 0.873917], [6, 0.072, 0.308656, 0.020594, 0.044313], [0, 0.314539, 0.371531, 0.091234, 0.117688]]}, {"filename": "000000443844.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.81525, 0.392984, 0.158687, 0.566154], [3, 0.670312, 0.721352, 0.256062, 0.287646]]}, {"filename": "000000443969.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.501634, 0.426822, 0.211765, 0.507042], [0, 0.10817, 0.56317, 0.21634, 0.768137], [0, 0.184175, 0.261234, 0.145899, 0.368938], [0, 0.5454, 0.227304, 0.500441, 0.426928], [0, 0.209739, 0.180294, 0.029412, 0.080882], [0, 0.308031, 0.167802, 0.022369, 0.065637]]}, {"filename": "000000444340.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.217234, 0.643446, 0.179281, 0.546065]]}, {"filename": "000000445371.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.116297, 0.407865, 0.229219, 0.186521], [1, 0.958516, 0.438229, 0.080719, 0.302708], [4, 0.924367, 0.386125, 0.041953, 0.070958], [4, 0.656813, 0.34674, 0.010937, 0.018604], [4, 0.734273, 0.371229, 0.032516, 0.063833], [4, 0.621602, 0.361823, 0.022641, 0.030188], [4, 0.639711, 0.36101, 0.020516, 0.031271], [3, 0.522469, 0.522469, 0.471906, 0.343813], [4, 0.718938, 0.365875, 0.011781, 0.040667], [4, 0.726672, 0.368771, 0.020406, 0.007417]]}, {"filename": "000000446206.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.156578, 0.837104, 0.27125, 0.253325], [0, 0.214219, 0.860325, 0.013, 0.030675], [0, 0.242539, 0.851545, 0.014172, 0.034935], [0, 0.229875, 0.852818, 0.011531, 0.038312], [0, 0.563711, 0.969766, 0.020609, 0.060468]]}, {"filename": "000000447297.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.547289, 0.220599, 0.117641, 0.119916], [1, 0.356391, 0.207577, 0.179563, 0.13415], [1, 0.204375, 0.214025, 0.122, 0.114345], [1, 0.095383, 0.270279, 0.180672, 0.488747], [5, 0.661781, 0.495947, 0.224375, 0.710111], [0, 0.950742, 0.241574, 0.034547, 0.065487], [0, 0.928227, 0.197855, 0.011641, 0.060056], [0, 0.867234, 0.19851, 0.031156, 0.064596], [0, 0.832469, 0.198538, 0.039094, 0.081922], [0, 0.8715, 0.123273, 0.035313, 0.061588], [0, 0.902398, 0.123092, 0.030391, 0.064178], [0, 0.806477, 0.130543, 0.023547, 0.070947], [1, 0.046141, 0.264652, 0.069563, 0.106295], [0, 0.576867, 0.18376, 0.019891, 0.030084], [0, 0.929969, 0.127507, 0.011281, 0.058635], [0, 0.896125, 0.205669, 0.027187, 0.080084], [0, 0.963125, 0.199819, 0.038687, 0.146212], [0, 0.957148, 0.124554, 0.016391, 0.058412]]}, {"filename": "000000447448.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.462167, 0.81125, 0.271167, 0.323583], [0, 0.613056, 0.779312, 0.103611, 0.139625], [4, 0.084333, 0.662458, 0.168667, 0.309333], [4, 0.090042, 0.897396, 0.176639, 0.174042]]}, {"filename": "000000447611.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.82574, 0.324865, 0.34852, 0.64494], [0, 0.15912, 0.186512, 0.31524, 0.368533]]}, {"filename": "000000448317.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.417273, 0.547971, 0.527703, 0.904059], [0, 0.705195, 0.821663, 0.161797, 0.35228], [0, 0.786969, 0.769435, 0.134594, 0.443598], [0, 0.93275, 0.566444, 0.129531, 0.189205], [4, 0.878187, 0.925837, 0.148188, 0.148326], [4, 0.732539, 0.884906, 0.146797, 0.182992], [0, 0.613438, 0.685837, 0.103437, 0.187866], [0, 0.549672, 0.811234, 0.172875, 0.337071], [0, 0.931172, 0.685711, 0.131906, 0.270126], [0, 0.959945, 0.914153, 0.080109, 0.151695]]}, {"filename": "000000448410.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.066742, 0.786739, 0.081328, 0.319108], [0, 0.105273, 0.609577, 0.088828, 0.182906], [0, 0.117742, 0.577517, 0.072516, 0.172975], [0, 0.190586, 0.552952, 0.052984, 0.248741], [0, 0.274367, 0.392746, 0.033141, 0.158261], [0, 0.342016, 0.392597, 0.04525, 0.188627], [0, 0.179703, 0.544668, 0.038688, 0.109153], [0, 0.08543, 0.633455, 0.104797, 0.180366], [0, 0.312453, 0.401373, 0.03325, 0.12833], [0, 0.572234, 0.453341, 0.049875, 0.073638], [3, 0.012922, 0.724645, 0.025844, 0.038215], [0, 0.180383, 0.66095, 0.057359, 0.266522]]}, {"filename": "000000449198.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.57672, 0.493559, 0.01712, 0.031261]]}, {"filename": "000000449312.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.336141, 0.355649, 0.502188, 0.70628], [1, 0.627508, 0.176799, 0.303078, 0.171176]]}, {"filename": "000000449384.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.409117, 0.350469, 0.079516, 0.034688], [1, 0.132125, 0.341313, 0.125125, 0.054417], [1, 0.630648, 0.371469, 0.026359, 0.019812], [0, 0.274945, 0.567521, 0.247203, 0.632292], [0, 0.584828, 0.641573, 0.094375, 0.294396], [1, 0.350609, 0.355, 0.038781, 0.023625], [1, 0.210961, 0.350073, 0.033859, 0.025813], [7, 0.078719, 0.306771, 0.018312, 0.040667], [1, 0.523148, 0.357427, 0.050516, 0.026146]]}, {"filename": "000000449579.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.294508, 0.43637, 0.209266, 0.628618]]}, {"filename": "000000449850.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.104297, 0.445609, 0.065719, 0.094215], [4, 0.517086, 0.376581, 0.034922, 0.059063], [0, 0.356086, 0.691007, 0.295359, 0.595504], [0, 0.04118, 0.3474, 0.081297, 0.263372], [0, 0.310664, 0.407775, 0.119734, 0.115785]]}, {"filename": "000000450359.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.213016, 0.743521, 0.416312, 0.490292], [4, 0.036289, 0.782802, 0.072578, 0.408604], [0, 0.874445, 0.64401, 0.246297, 0.441479], [0, 0.578086, 0.237073, 0.134828, 0.253938], [0, 0.39182, 0.247125, 0.071328, 0.193125], [0, 0.105016, 0.237156, 0.210031, 0.299188]]}, {"filename": "000000451359.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.055844, 0.441854, 0.111094, 0.21493]]}, {"filename": "000000451435.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.094391, 0.287195, 0.058656, 0.231377]]}, {"filename": "000000451859.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.858359, 0.833781, 0.031125, 0.023979], [0, 0.132406, 0.952052, 0.041188, 0.092813], [0, 0.86725, 0.855198, 0.020469, 0.056854], [1, 0.809812, 0.840885, 0.029719, 0.016521], [1, 0.909617, 0.831021, 0.016484, 0.022375]]}, {"filename": "000000452891.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.500785, 0.494805, 0.987678, 0.984422]]}, {"filename": "000000453302.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.93386, 0.58732, 0.09488, 0.065893]]}, {"filename": "000000453313.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.311284, 0.316398, 0.028747, 0.017609], [1, 0.432035, 0.33218, 0.02929, 0.024703], [1, 0.262255, 0.319617, 0.027557, 0.017609], [1, 0.198069, 0.320469, 0.038601, 0.021344], [7, 0.309969, 0.503195, 0.10261, 0.069703], [7, 0.315219, 0.573477, 0.104342, 0.067109], [7, 0.456775, 0.304063, 0.015678, 0.050875], [1, 0.225752, 0.318773, 0.020021, 0.013922]]}, {"filename": "000000453334.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.753648, 0.546073, 0.492703, 0.885396], [0, 0.184148, 0.633302, 0.020922, 0.082479], [1, 0.52382, 0.488604, 0.012516, 0.007708], [1, 0.546438, 0.489573, 0.016156, 0.009479]]}, {"filename": "000000453584.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.601859, 0.532167, 0.414781, 0.37925], [4, 0.416055, 0.431521, 0.086703, 0.236458], [4, 0.540828, 0.436479, 0.07625, 0.231292], [4, 0.649, 0.357823, 0.064656, 0.049562], [4, 0.794148, 0.393333, 0.046234, 0.144083], [4, 0.707938, 0.359073, 0.058469, 0.034521], [4, 0.504531, 0.371281, 0.020188, 0.065729]]}, {"filename": "000000453621.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.528088, 0.502688, 0.485392, 0.957304]]}, {"filename": "000000453722.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.165055, 0.799473, 0.326641, 0.36719]]}, {"filename": "000000453841.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.60301, 0.721888, 0.40334, 0.488437], [1, 0.9586, 0.544499, 0.07832, 0.084336], [1, 0.15234, 0.604233, 0.29612, 0.351003], [0, 0.54479, 0.564735, 0.03882, 0.041268], [5, 0.95206, 0.707271, 0.04004, 0.112891], [0, 0.78661, 0.595147, 0.0281, 0.11649], [1, 0.03111, 0.75059, 0.06222, 0.26354], [1, 0.38437, 0.597684, 0.0183, 0.03354], [1, 0.39809, 0.594336, 0.0161, 0.030265], [1, 0.76558, 0.572065, 0.02052, 0.025074]]}, {"filename": "000000454053.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.30842, 0.938467, 0.16668, 0.105627], [1, 0.47828, 0.948653, 0.17776, 0.079013], [1, 0.75671, 0.974693, 0.21478, 0.046987], [6, 0.52162, 0.58808, 0.11292, 0.229227]]}, {"filename": "000000454404.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.535867, 0.461115, 0.302766, 0.824726], [3, 0.253547, 0.672665, 0.406781, 0.631456], [4, 0.267469, 0.657297, 0.445781, 0.6]]}, {"filename": "000000455157.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.384375, 0.517406, 0.271469, 0.540625], [3, 0.523547, 0.726492, 0.390281, 0.310078], [3, 0.771914, 0.644945, 0.438203, 0.260672], [3, 0.927609, 0.54175, 0.144781, 0.075063]]}, {"filename": "000000455758.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.629636, 0.486328, 0.52331, 0.943812], [0, 0.906397, 0.115187, 0.187207, 0.230375], [1, 0.045575, 0.210547, 0.08439, 0.319094], [0, 0.405904, 0.147289, 0.171808, 0.292266], [0, 0.267606, 0.077687, 0.146291, 0.144687], [0, 0.764883, 0.066672, 0.139765, 0.133344], [0, 0.550763, 0.065398, 0.166033, 0.129234]]}, {"filename": "000000455952.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.253531, 0.514219, 0.048875, 0.069479]]}, {"filename": "000000456239.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.274508, 0.537078, 0.474918, 0.692125]]}, {"filename": "000000456545.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.616102, 0.696916, 0.159891, 0.317997], [1, 0.12007, 0.462218, 0.102328, 0.13938], [1, 0.058133, 0.443871, 0.058453, 0.09725], [6, 0.683922, 0.158728, 0.171312, 0.183021], [0, 0.398648, 0.483148, 0.099391, 0.220223], [0, 0.635875, 0.573068, 0.198813, 0.417456], [0, 0.772359, 0.495795, 0.154969, 0.337727], [0, 0.191898, 0.590246, 0.159859, 0.445866], [0, 0.295836, 0.46093, 0.135734, 0.260715], [0, 0.527234, 0.482377, 0.069281, 0.218617], [0, 0.924852, 0.515739, 0.094641, 0.362766], [0, 0.548883, 0.6612, 0.256578, 0.315151], [0, 0.032023, 0.455064, 0.059641, 0.258426], [0, 0.861023, 0.455159, 0.082203, 0.279952], [1, 0.717031, 0.449022, 0.046312, 0.090986], [0, 0.46057, 0.470342, 0.030547, 0.072385], [0, 0.959602, 0.56372, 0.078734, 0.429952]]}, {"filename": "000000456638.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.431461, 0.505375, 0.377953, 0.989251]]}, {"filename": "000000457726.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.447805, 0.873431, 0.228422, 0.229063]]}, {"filename": "000000458223.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.31632, 0.661893, 0.042578, 0.184911], [0, 0.288219, 0.525059, 0.1155, 0.423669]]}, {"filename": "000000458702.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.309871, 0.349875, 0.107799, 0.220219], [1, 0.183396, 0.739086, 0.349883, 0.079766], [7, 0.480656, 0.347164, 0.138689, 0.215922], [1, 0.136077, 0.739391, 0.272155, 0.084813]]}, {"filename": "000000458857.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.258086, 0.65053, 0.050703, 0.107576], [0, 0.51632, 0.574167, 0.062109, 0.632879], [0, 0.447344, 0.663295, 0.082688, 0.648636], [0, 0.391672, 0.604792, 0.075687, 0.573371], [0, 0.446953, 0.339886, 0.040375, 0.254924], [0, 0.183914, 0.617917, 0.061609, 0.441818], [0, 0.268008, 0.56464, 0.052578, 0.179432], [0, 0.300891, 0.53303, 0.05625, 0.08447], [6, 0.245344, 0.162367, 0.06225, 0.164886], [1, 0.946922, 0.475701, 0.082687, 0.097008]]}, {"filename": "000000459634.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.746188, 0.804803, 0.507625, 0.382552], [1, 0.158406, 0.584447, 0.022375, 0.042251], [0, 0.157477, 0.576904, 0.023172, 0.042589]]}, {"filename": "000000459662.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.759117, 0.568604, 0.481766, 0.862458], [0, 0.288656, 0.55249, 0.573937, 0.853271], [4, 0.304688, 0.354177, 0.605469, 0.695312]]}, {"filename": "000000460147.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.416469, 0.766592, 0.094844, 0.161061], [1, 0.465227, 0.485743, 0.044016, 0.060259], [1, 0.790781, 0.50059, 0.073719, 0.072217], [1, 0.918805, 0.763078, 0.162391, 0.25408], [1, 0.523133, 0.37441, 0.024672, 0.027783], [1, 0.89432, 0.579068, 0.116766, 0.123373], [1, 0.561258, 0.34592, 0.010109, 0.017406], [1, 0.743852, 0.458785, 0.048734, 0.064929], [1, 0.938336, 0.435908, 0.044109, 0.074552], [1, 0.666117, 0.431108, 0.037203, 0.043868], [1, 0.485734, 0.425967, 0.026687, 0.038302], [1, 0.650227, 0.408479, 0.010641, 0.023278], [1, 0.594695, 0.322465, 0.028828, 0.039505], [1, 0.566469, 0.314186, 0.033406, 0.033844], [1, 0.695352, 0.38033, 0.037359, 0.067783], [0, 0.84743, 0.422146, 0.006703, 0.044717], [0, 0.902062, 0.563715, 0.02125, 0.025212], [1, 0.503531, 0.394363, 0.02225, 0.033962], [1, 0.649758, 0.365106, 0.009359, 0.022429], [1, 0.637836, 0.380802, 0.011203, 0.020519], [1, 0.597328, 0.596262, 0.116969, 0.174741], [1, 0.798031, 0.606368, 0.120531, 0.133302], [1, 0.071055, 0.62145, 0.141297, 0.12625], [0, 0.649586, 0.391639, 0.012734, 0.017712], [1, 0.540734, 0.36237, 0.019281, 0.02125]]}, {"filename": "000000460646.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.928305, 0.503937, 0.115641, 0.080958], [1, 0.448312, 0.484271, 0.256187, 0.276417]]}, {"filename": "000000460652.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.38518, 0.414187, 0.125078, 0.1725], [1, 0.051898, 0.753229, 0.103797, 0.12175], [1, 0.12018, 0.731844, 0.088172, 0.080313], [1, 0.550289, 0.716052, 0.099766, 0.051854], [1, 0.666797, 0.707833, 0.134031, 0.059375], [1, 0.462859, 0.716958, 0.076625, 0.047458], [0, 0.883898, 0.673677, 0.042141, 0.100562], [0, 0.837523, 0.692, 0.024984, 0.070125], [1, 0.328047, 0.724656, 0.135844, 0.055396], [0, 0.694187, 0.669813, 0.015625, 0.019875], [1, 0.959031, 0.681198, 0.01325, 0.074604], [1, 0.953352, 0.695219, 0.041828, 0.046479], [1, 0.14968, 0.705115, 0.160703, 0.105688], [1, 0.110984, 0.757802, 0.039094, 0.075812], [1, 0.39193, 0.720115, 0.026547, 0.040812], [1, 0.048094, 0.90624, 0.096188, 0.187521], [1, 0.010789, 0.691031, 0.021578, 0.017604], [0, 0.579875, 0.679302, 0.017438, 0.020688], [0, 0.584195, 0.67925, 0.010672, 0.019708]]}, {"filename": "000000461015.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.771542, 0.642695, 0.20675, 0.656172], [0, 0.920927, 0.725063, 0.026104, 0.063469], [0, 0.865375, 0.717195, 0.015667, 0.015234], [1, 0.66151, 0.732828, 0.045187, 0.035281], [1, 0.915583, 0.731336, 0.098792, 0.039172], [1, 0.97176, 0.730258, 0.056479, 0.059484], [1, 0.34226, 0.737148, 0.176437, 0.053203], [1, 0.679156, 0.727398, 0.047771, 0.033516], [1, 0.103375, 0.70843, 0.20075, 0.161797], [1, 0.585083, 0.730375, 0.127167, 0.084156], [1, 0.442688, 0.730492, 0.078333, 0.038328], [1, 0.20199, 0.705031, 0.119188, 0.133688], [1, 0.101677, 0.706438, 0.193771, 0.154], [1, 0.58375, 0.730219, 0.122042, 0.082188]]}, {"filename": "000000461573.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.69071, 0.520053, 0.55894, 0.727979], [3, 0.46797, 0.377952, 0.37774, 0.490426]]}, {"filename": "000000461620.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.415813, 0.715958, 0.2405, 0.568083], [0, 0.115711, 0.795052, 0.231422, 0.389646], [0, 0.19582, 0.367521, 0.125859, 0.216917], [0, 0.440359, 0.37826, 0.169844, 0.412021], [0, 0.329781, 0.445271, 0.11675, 0.228333], [0, 0.506859, 0.291792, 0.054781, 0.181792], [0, 0.579922, 0.230563, 0.107844, 0.289542], [0, 0.702594, 0.191854, 0.126094, 0.323917], [0, 0.332836, 0.310021, 0.104141, 0.221542], [0, 0.241102, 0.3995, 0.111766, 0.215792], [0, 0.739375, 0.099979, 0.028281, 0.051792], [0, 0.149469, 0.748708, 0.293438, 0.490042], [0, 0.769414, 0.142625, 0.070703, 0.178042], [4, 0.612656, 0.252583, 0.039469, 0.081292]]}, {"filename": "000000462238.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.314141, 0.75636, 0.628281, 0.465326], [0, 0.525484, 0.803163, 0.135688, 0.10214], [0, 0.643023, 0.959477, 0.016922, 0.066721], [1, 0.896359, 0.943523, 0.199187, 0.094116], [0, 0.325773, 0.842802, 0.051922, 0.102302]]}, {"filename": "000000462677.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.525031, 0.308016, 0.321946, 0.494375]]}, {"filename": "000000462728.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.167086, 0.362658, 0.043453, 0.083583]]}, {"filename": "000000463325.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.405937, 0.334085, 0.134312, 0.210751], [0, 0.514437, 0.633709, 0.200156, 0.531221], [6, 0.188055, 0.463674, 0.042953, 0.057441]]}, {"filename": "000000464824.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.876635, 0.058375, 0.246729, 0.11675], [5, 0.560937, 0.479609, 0.163375, 0.138563], [0, 0.636792, 0.555844, 0.726417, 0.862344]]}, {"filename": "000000465179.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.499452, 0.59085, 0.998904, 0.79458]]}, {"filename": "000000465675.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.666805, 0.542854, 0.266953, 0.266849]]}, {"filename": "000000465941.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.31542, 0.543, 0.06016, 0.090907], [0, 0.54354, 0.568547, 0.30168, 0.840453], [3, 0.32224, 0.53972, 0.11372, 0.087387], [3, 0.16776, 0.54096, 0.09072, 0.088693], [3, 0.42659, 0.53972, 0.05662, 0.073573], [4, 0.34311, 0.52416, 0.05314, 0.05328], [4, 0.42223, 0.541867, 0.0437, 0.01808]]}, {"filename": "000000466156.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.5, 0.489787, 1.0, 0.879573], [1, 0.50255, 0.10356, 0.04778, 0.037147], [1, 0.56549, 0.0968, 0.03386, 0.022773], [1, 0.0113, 0.05128, 0.0226, 0.02848]]}, {"filename": "000000466263.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.533609, 0.61826, 0.349875, 0.762396], [1, 0.05282, 0.469167, 0.089797, 0.0305]]}, {"filename": "000000467315.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.677055, 0.832512, 0.114266, 0.240329], [4, 0.081586, 0.836561, 0.114516, 0.232277], [0, 0.35943, 0.338005, 0.073109, 0.200469], [4, 0.054461, 0.889613, 0.105453, 0.194812], [4, 0.727039, 0.893392, 0.131266, 0.213216], [4, 0.356203, 0.719401, 0.245313, 0.099977]]}, {"filename": "000000467572.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.248243, 0.356193, 0.449657, 0.145196]]}, {"filename": "000000467848.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.637992, 0.66975, 0.616016, 0.435417], [1, 0.215727, 0.69625, 0.357297, 0.20225], [0, 0.946063, 0.744104, 0.079969, 0.244542]]}, {"filename": "000000468284.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.759421, 0.493789, 0.370728, 0.871328]]}, {"filename": "000000469721.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.340844, 0.439188, 0.261271, 0.551813]]}, {"filename": "000000470028.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.29557, 0.323548, 0.071828, 0.255995], [0, 0.594469, 0.518817, 0.195906, 0.779274], [1, 0.499641, 0.496628, 0.999281, 0.979719], [0, 0.613523, 0.140773, 0.048078, 0.145293]]}, {"filename": "000000471015.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.612898, 0.848726, 0.101859, 0.161734], [4, 0.639687, 0.708276, 0.051, 0.063619], [0, 0.605945, 0.68545, 0.040828, 0.11364], [0, 0.532641, 0.693608, 0.049844, 0.137152], [3, 0.232461, 0.923062, 0.088641, 0.117516], [0, 0.657937, 0.731081, 0.037125, 0.105632], [0, 0.777063, 0.641842, 0.035031, 0.054004], [4, 0.553648, 0.846424, 0.046141, 0.072077], [4, 0.536094, 0.787966, 0.036156, 0.105867], [4, 0.488719, 0.818865, 0.073438, 0.127923]]}, {"filename": "000000471023.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.261094, 0.573286, 0.046156, 0.090939], [0, 0.35575, 0.572664, 0.035531, 0.071761], [0, 0.471852, 0.561725, 0.018484, 0.110587], [0, 0.504563, 0.540469, 0.03125, 0.093897], [0, 0.536164, 0.521502, 0.017484, 0.070188], [0, 0.586508, 0.51216, 0.010484, 0.072817], [0, 0.609641, 0.509225, 0.009719, 0.053897], [0, 0.436734, 0.561561, 0.02175, 0.102136], [4, 0.971555, 0.706127, 0.056891, 0.33493], [0, 0.341437, 0.560106, 0.017531, 0.078333], [0, 0.405977, 0.598239, 0.024641, 0.096244], [0, 0.317812, 0.551491, 0.022406, 0.060587], [4, 0.968023, 0.509988, 0.063953, 0.070869], [4, 0.969227, 0.560246, 0.061547, 0.147441], [4, 0.946977, 0.551608, 0.051391, 0.156878], [0, 0.960867, 0.420575, 0.078266, 0.141244], [0, 0.416594, 0.542958, 0.025812, 0.060423]]}, {"filename": "000000471439.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.162133, 0.580474, 0.036797, 0.077456]]}, {"filename": "000000471991.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.386414, 0.590743, 0.029516, 0.045967]]}, {"filename": "000000472030.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.888484, 0.878656, 0.223031, 0.220229]]}, {"filename": "000000472654.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.458781, 0.616391, 0.093938, 0.220375], [4, 0.231289, 0.594359, 0.282141, 0.291594], [0, 0.270961, 0.497852, 0.184984, 0.677047], [0, 0.737078, 0.4985, 0.170781, 0.683156], [0, 0.872734, 0.338281, 0.085688, 0.168625], [0, 0.123594, 0.348312, 0.103375, 0.20675], [0, 0.434023, 0.529664, 0.139203, 0.388859], [0, 0.166563, 0.336961, 0.062625, 0.165328], [0, 0.486602, 0.295281, 0.036391, 0.084906], [0, 0.624719, 0.421352, 0.080906, 0.415734], [0, 0.926922, 0.449164, 0.144562, 0.233859], [0, 0.357484, 0.52632, 0.079094, 0.355266], [0, 0.446609, 0.387578, 0.096656, 0.292469], [4, 0.901766, 0.591836, 0.196469, 0.289047], [0, 0.831234, 0.348586, 0.061125, 0.190641], [0, 0.032828, 0.453891, 0.065656, 0.369094]]}, {"filename": "000000473261.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.519016, 0.322354, 0.092594, 0.186417], [4, 0.918891, 0.272677, 0.071406, 0.162646], [4, 0.815586, 0.248573, 0.060547, 0.098646], [0, 0.252805, 0.515354, 0.461797, 0.941583], [0, 0.620219, 0.604646, 0.39775, 0.755042], [0, 0.505758, 0.254604, 0.101141, 0.258167], [0, 0.866453, 0.143562, 0.057875, 0.097042], [0, 0.898187, 0.225802, 0.096719, 0.258271], [4, 0.833867, 0.382573, 0.248422, 0.290771], [4, 0.69743, 0.213062, 0.019891, 0.051625], [4, 0.972406, 0.338958, 0.055188, 0.205208], [0, 0.966234, 0.131177, 0.058437, 0.111687], [0, 0.783188, 0.155698, 0.069063, 0.095021], [4, 0.401852, 0.834156, 0.162953, 0.301229]]}, {"filename": "000000474378.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.59775, 0.713484, 0.164792, 0.083156], [7, 0.626292, 0.704719, 0.038917, 0.062375], [1, 0.711208, 0.693859, 0.030958, 0.04975], [1, 0.121802, 0.653937, 0.105271, 0.027156], [1, 0.658375, 0.680906, 0.085792, 0.03875], [1, 0.046812, 0.693664, 0.093625, 0.097297], [1, 0.281635, 0.661766, 0.046187, 0.05775], [1, 0.034375, 0.775039, 0.06875, 0.121078], [1, 0.539875, 0.661937, 0.0215, 0.016125], [1, 0.716698, 0.665023, 0.028729, 0.015047], [1, 0.061062, 0.650242, 0.08725, 0.033641], [1, 0.434563, 0.662727, 0.042625, 0.025141], [1, 0.502687, 0.654883, 0.050542, 0.017766], [7, 0.692292, 0.692164, 0.018833, 0.025297], [7, 0.72774, 0.679867, 0.014229, 0.026203], [1, 0.693979, 0.704922, 0.042167, 0.053531]]}, {"filename": "000000474399.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.981229, 0.427984, 0.037542, 0.089], [1, 0.934854, 0.386469, 0.033333, 0.022187], [7, 0.904938, 0.40818, 0.02925, 0.050547]]}, {"filename": "000000475484.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.921937, 0.580828, 0.146784, 0.076719], [7, 0.791232, 0.228938, 0.287207, 0.385281], [0, 0.025047, 0.546977, 0.048826, 0.085266], [0, 0.200211, 0.541578, 0.019812, 0.051625], [0, 0.275646, 0.538563, 0.024343, 0.03825], [0, 0.050728, 0.534289, 0.020986, 0.045672], [0, 0.289906, 0.534555, 0.015023, 0.021891], [0, 0.150927, 0.54657, 0.039225, 0.063953], [1, 0.484155, 0.567445, 0.307371, 0.080359]]}, {"filename": "000000475882.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.668094, 0.904156, 0.094229, 0.064281], [7, 0.65525, 0.929852, 0.028792, 0.033547], [7, 0.604646, 0.902656, 0.016833, 0.026125], [1, 0.802865, 0.88868, 0.049563, 0.024203], [1, 0.757521, 0.881969, 0.038875, 0.023531], [1, 0.717521, 0.884695, 0.032375, 0.017234], [1, 0.599531, 0.883164, 0.016187, 0.018016], [6, 0.745208, 0.392617, 0.490875, 0.358391], [1, 0.781573, 0.887031, 0.026688, 0.021219], [1, 0.73475, 0.880906, 0.028375, 0.021063], [0, 0.548615, 0.934062, 0.075354, 0.129]]}, {"filename": "000000476119.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.535086, 0.321253, 0.317922, 0.279649], [0, 0.331352, 0.481194, 0.376141, 0.885902], [1, 0.05243, 0.257529, 0.077203, 0.086956], [1, 0.534898, 0.177588, 0.231703, 0.093583]]}, {"filename": "000000476738.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.71368, 0.764219, 0.139297, 0.471562], [4, 0.352125, 0.562214, 0.093281, 0.148718], [0, 0.454555, 0.57303, 0.614016, 0.853939], [4, 0.071, 0.739557, 0.142, 0.520886]]}, {"filename": "000000477622.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.636234, 0.585958, 0.119094, 0.069542], [0, 0.181781, 0.612438, 0.050812, 0.230583], [0, 0.730453, 0.57451, 0.018156, 0.058146], [0, 0.742742, 0.563406, 0.012141, 0.051521], [1, 0.906266, 0.605823, 0.017406, 0.027771], [6, 0.542773, 0.316177, 0.026859, 0.140813], [1, 0.976008, 0.596, 0.017891, 0.041958]]}, {"filename": "000000477831.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.311632, 0.458781, 0.04201, 0.03725], [5, 0.509285, 0.58057, 0.203682, 0.335266], [1, 0.35709, 0.45932, 0.042797, 0.032203], [1, 0.398207, 0.463187, 0.033521, 0.021625], [1, 0.088481, 0.467617, 0.119598, 0.073234], [1, 0.2259, 0.46393, 0.041994, 0.049453], [1, 0.266672, 0.459047, 0.071543, 0.049031], [1, 0.141929, 0.45275, 0.018682, 0.043031], [1, 0.119928, 0.445219, 0.033232, 0.016812]]}, {"filename": "000000477974.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.70716, 0.558438, 0.38764, 0.680901], [0, 0.46166, 0.762883, 0.05048, 0.254715], [0, 0.42529, 0.761321, 0.05218, 0.273814]]}, {"filename": "000000478721.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.626359, 0.544719, 0.100906, 0.096062], [1, 0.088172, 0.516167, 0.07875, 0.083833], [0, 0.508828, 0.542625, 0.022094, 0.079917], [0, 0.492875, 0.543208, 0.017063, 0.072208], [0, 0.917898, 0.579917, 0.035391, 0.049167], [1, 0.721688, 0.542646, 0.109781, 0.106708], [1, 0.823656, 0.56124, 0.092813, 0.084604], [1, 0.012719, 0.508479, 0.025438, 0.096208], [1, 0.626289, 0.54651, 0.101516, 0.101688], [1, 0.954969, 0.565302, 0.090062, 0.090563], [0, 0.559094, 0.620781, 0.105187, 0.290896], [0, 0.429508, 0.576885, 0.104297, 0.232896]]}, {"filename": "000000479099.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.648344, 0.386729, 0.098156, 0.073667]]}, {"filename": "000000480587.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.618242, 0.722916, 0.203328, 0.494356], [1, 0.516508, 0.686511, 0.955047, 0.591007], [1, 0.313398, 0.606733, 0.076297, 0.282506], [0, 0.231359, 0.556616, 0.11925, 0.374918], [0, 0.403883, 0.614672, 0.215422, 0.580867], [0, 0.51632, 0.392424, 0.090266, 0.173513], [0, 0.69418, 0.442002, 0.110953, 0.355059], [1, 0.248094, 0.595574, 0.087031, 0.244075], [1, 0.789492, 0.558735, 0.084109, 0.304684], [0, 0.634953, 0.580902, 0.205406, 0.644941], [0, 0.480133, 0.375948, 0.037391, 0.082295], [0, 0.896109, 0.5263, 0.207781, 0.578407], [0, 0.465531, 0.551429, 0.082469, 0.411241], [0, 0.179227, 0.6713, 0.254203, 0.570796], [0, 0.963477, 0.294707, 0.073047, 0.17274], [0, 0.325836, 0.403923, 0.056266, 0.144005], [1, 0.518008, 0.558419, 0.121453, 0.343817], [1, 0.033867, 0.783044, 0.064797, 0.254005], [1, 0.109336, 0.495012, 0.046328, 0.069461], [1, 0.776961, 0.538454, 0.032672, 0.195925], [1, 0.954828, 0.183489, 0.083531, 0.100749], [0, 0.790898, 0.333735, 0.095734, 0.202646], [0, 0.345953, 0.437201, 0.092969, 0.181194], [1, 0.721711, 0.460468, 0.033328, 0.108197], [1, 0.961836, 0.451499, 0.076328, 0.267166], [1, 0.270297, 0.469871, 0.045063, 0.095246]]}, {"filename": "000000480591.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.716297, 0.738354, 0.246062, 0.487625], [0, 0.507938, 0.649552, 0.139156, 0.238521], [0, 0.386766, 0.805198, 0.258969, 0.372188], [0, 0.093367, 0.766969, 0.182016, 0.458437], [0, 0.928187, 0.745667, 0.143625, 0.508667], [4, 0.501992, 0.700969, 0.052516, 0.074812]]}, {"filename": "000000480944.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.593635, 0.565188, 0.123354, 0.048688], [1, 0.372833, 0.584828, 0.193083, 0.055625], [1, 0.370417, 0.563367, 0.193292, 0.050109], [1, 0.150615, 0.588141, 0.213438, 0.068125], [1, 0.796854, 0.566086, 0.040875, 0.034641], [1, 0.822229, 0.56682, 0.036542, 0.032047], [1, 0.982812, 0.559445, 0.032042, 0.029734], [1, 0.541562, 0.556023, 0.164333, 0.079234], [1, 0.68826, 0.535945, 0.102521, 0.032828], [1, 0.89249, 0.557156, 0.088896, 0.047844], [6, 0.195771, 0.393227, 0.092208, 0.110266]]}, {"filename": "000000481390.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.401016, 0.207792, 0.273188, 0.091417], [0, 0.607734, 0.56626, 0.149656, 0.291479], [0, 0.112414, 0.808115, 0.121047, 0.353937], [0, 0.4805, 0.719198, 0.192219, 0.292479], [0, 0.540414, 0.747781, 0.200953, 0.376563], [0, 0.489867, 0.103604, 0.067578, 0.119375], [0, 0.900336, 0.59399, 0.114797, 0.352854], [0, 0.755852, 0.39299, 0.136453, 0.299312], [0, 0.423258, 0.457271, 0.200922, 0.295917], [0, 0.094828, 0.24125, 0.104438, 0.336583], [0, 0.150695, 0.146604, 0.099391, 0.276625], [0, 0.708711, 0.746073, 0.193828, 0.287646], [0, 0.352359, 0.06974, 0.071188, 0.101104], [4, 0.307969, 0.177125, 0.073219, 0.123083], [4, 0.433109, 0.189313, 0.069313, 0.099333], [4, 0.388875, 0.194115, 0.071438, 0.124896], [4, 0.479047, 0.191875, 0.066219, 0.078042], [4, 0.521383, 0.206969, 0.066016, 0.093896], [4, 0.520117, 0.074271, 0.026922, 0.032167], [4, 0.565953, 0.083948, 0.023531, 0.032187], [4, 0.610812, 0.09599, 0.033531, 0.035354], [4, 0.259234, 0.12574, 0.02225, 0.098563], [4, 0.351906, 0.188833, 0.066156, 0.133], [4, 0.846672, 0.057385, 0.051219, 0.065021], [0, 0.767703, 0.250479, 0.057062, 0.126792], [4, 0.813508, 0.050375, 0.027984, 0.066708], [4, 0.764602, 0.167854, 0.034734, 0.047708]]}, {"filename": "000000481582.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.206727, 0.642377, 0.153734, 0.702646], [0, 0.690484, 0.618396, 0.18925, 0.763208], [0, 0.59882, 0.563443, 0.238859, 0.427166]]}, {"filename": "000000482330.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.635414, 0.534162, 0.159797, 0.457106], [0, 0.420938, 0.528493, 0.075594, 0.112675], [0, 0.236727, 0.746467, 0.190953, 0.507066], [0, 0.882547, 0.698603, 0.232438, 0.388782], [0, 0.789055, 0.669082, 0.242266, 0.543872], [0, 0.778719, 0.596737, 0.098187, 0.29004], [0, 0.166437, 0.449341, 0.188844, 0.344431], [4, 0.119875, 0.601277, 0.067531, 0.141397]]}, {"filename": "000000482436.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.538281, 0.490561, 0.138219, 0.252145], [0, 0.273766, 0.626135, 0.285062, 0.727481]]}, {"filename": "000000482472.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.342672, 0.294177, 0.678875, 0.454729], [1, 0.801094, 0.194344, 0.348375, 0.253562], [1, 0.962484, 0.148021, 0.075031, 0.134333], [0, 0.031656, 0.027719, 0.029844, 0.047104], [0, 0.154797, 0.040417, 0.01875, 0.041292], [1, 0.667227, 0.07026, 0.126328, 0.081646], [1, 0.385094, 0.06749, 0.144625, 0.065021], [1, 0.531719, 0.075667, 0.154063, 0.084125], [7, 0.913023, 0.21175, 0.070172, 0.276375], [3, 0.506461, 0.485396, 0.369109, 0.570792], [1, 0.161867, 0.094625, 0.323734, 0.171167], [1, 0.883781, 0.072323, 0.087906, 0.051354], [0, 0.763437, 0.046604, 0.016656, 0.023417], [1, 0.962711, 0.075188, 0.074578, 0.045833], [1, 0.72225, 0.090583, 0.166062, 0.078458], [1, 0.796125, 0.058948, 0.076156, 0.027229], [7, 0.427453, 0.04099, 0.008531, 0.018563]]}, {"filename": "000000482719.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.584055, 0.144042, 0.392016, 0.288083], [4, 0.290148, 0.057073, 0.211047, 0.114146]]}, {"filename": "000000483297.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.493431, 0.490328, 0.986862, 0.977531]]}, {"filename": "000000483352.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.501867, 0.45917, 0.512373, 0.77078], [7, 0.805733, 0.35275, 0.021227, 0.03042], [7, 0.104707, 0.35979, 0.01064, 0.0235], [7, 0.73924, 0.35386, 0.019707, 0.03036], [7, 0.19316, 0.35872, 0.015333, 0.02492]]}, {"filename": "000000484029.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.162562, 0.387234, 0.325125, 0.277406], [1, 0.580406, 0.823961, 0.110146, 0.047109], [1, 0.641271, 0.817906, 0.071042, 0.047406], [1, 0.508, 0.803258, 0.078667, 0.030484], [1, 0.79099, 0.8135, 0.039729, 0.029594], [1, 0.746135, 0.816398, 0.074979, 0.031734], [1, 0.840323, 0.82857, 0.023021, 0.051109], [6, 0.268948, 0.71432, 0.046979, 0.035234]]}, {"filename": "000000484351.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.352953, 0.504871, 0.096312, 0.131009], [4, 0.901758, 0.526033, 0.112547, 0.272347], [0, 0.509172, 0.448169, 0.152562, 0.256197], [0, 0.749391, 0.628087, 0.284187, 0.721338], [0, 0.234836, 0.249437, 0.086766, 0.364038], [0, 0.070461, 0.779272, 0.140922, 0.416667], [0, 0.652914, 0.361796, 0.070297, 0.179789], [0, 0.448453, 0.175622, 0.051312, 0.12885], [0, 0.558047, 0.300528, 0.085656, 0.427441], [0, 0.320094, 0.247183, 0.065813, 0.265164], [0, 0.388062, 0.241866, 0.072594, 0.248803], [0, 0.704078, 0.271268, 0.032969, 0.058685], [0, 0.554273, 0.753732, 0.351828, 0.492535], [4, 0.600648, 0.368345, 0.030266, 0.13392], [4, 0.983875, 0.563392, 0.03225, 0.18885], [0, 0.83632, 0.381303, 0.123016, 0.244812], [0, 0.063695, 0.308228, 0.124672, 0.337394]]}, {"filename": "000000484404.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.163594, 0.515875, 0.327188, 0.212542], [0, 0.485711, 0.612542, 0.043859, 0.052125], [1, 0.7995, 0.476708, 0.104625, 0.075833], [0, 0.504789, 0.638521, 0.071641, 0.0555], [4, 0.514891, 0.825844, 0.726406, 0.348312], [4, 0.908586, 0.789792, 0.141141, 0.286292], [4, 0.608602, 0.638281, 0.066953, 0.051437], [4, 0.565859, 0.61375, 0.053156, 0.085375], [4, 0.748375, 0.573479, 0.02225, 0.061958], [0, 0.614406, 0.612396, 0.057344, 0.061583], [1, 0.553203, 0.491396, 0.124844, 0.135792], [4, 0.819758, 0.729979, 0.105734, 0.22975], [4, 0.570984, 0.625396, 0.03825, 0.07], [4, 0.947148, 0.644656, 0.054828, 0.057437], [4, 0.658484, 0.602729, 0.037312, 0.131167], [4, 0.502539, 0.583396, 0.037266, 0.02175]]}, {"filename": "000000484498.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.46984, 0.61891, 0.073227, 0.10346], [0, 0.593987, 0.61439, 0.05784, 0.08366], [0, 0.652947, 0.59557, 0.04136, 0.0717], [0, 0.70652, 0.58323, 0.04504, 0.07278], [0, 0.157213, 0.68835, 0.230213, 0.27398]]}, {"filename": "000000485019.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.666922, 0.51539, 0.334565, 0.25418], [3, 0.564144, 0.64753, 0.733333, 0.20874]]}, {"filename": "000000486040.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.526281, 0.45768, 0.203854, 0.131141]]}, {"filename": "000000487245.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.661312, 0.717344, 0.126687, 0.518021]]}, {"filename": "000000488075.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.792789, 0.730646, 0.290266, 0.413875], [4, 0.244109, 0.912323, 0.170812, 0.175354], [4, 0.951195, 0.801698, 0.097609, 0.248438]]}, {"filename": "000000488151.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.837488, 0.289734, 0.269089, 0.137469], [0, 0.466636, 0.586727, 0.26743, 0.399578]]}, {"filename": "000000488327.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.116129, 0.523297, 0.124467, 0.065531], [1, 0.440893, 0.499219, 0.023573, 0.009375], [1, 0.192022, 0.521797, 0.076402, 0.049969], [1, 0.4084, 0.499711, 0.036104, 0.014484], [1, 0.336811, 0.507328, 0.02464, 0.011844], [1, 0.246055, 0.509273, 0.075434, 0.044328], [1, 0.298139, 0.509227, 0.039702, 0.023891], [1, 0.312667, 0.507695, 0.029404, 0.025172], [6, 0.731675, 0.156688, 0.223648, 0.139937]]}, {"filename": "000000488592.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.621729, 0.813844, 0.102875, 0.135], [1, 0.113021, 0.806008, 0.214583, 0.128766], [1, 0.394646, 0.734648, 0.085958, 0.040484], [0, 0.60051, 0.760852, 0.093438, 0.172797], [0, 0.543427, 0.70368, 0.026187, 0.060422], [0, 0.511354, 0.702906, 0.0215, 0.060313], [0, 0.495677, 0.695594, 0.013354, 0.037062], [0, 0.461198, 0.702352, 0.026854, 0.054828], [0, 0.529146, 0.692953, 0.021333, 0.041844], [0, 0.475719, 0.696648, 0.018271, 0.041609], [1, 0.396646, 0.74043, 0.047458, 0.035891], [1, 0.362719, 0.744055, 0.020104, 0.040734]]}, {"filename": "000000488977.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.667414, 0.40299, 0.556172, 0.613479], [4, 0.45193, 0.231115, 0.150578, 0.125312]]}, {"filename": "000000489070.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.496922, 0.398573, 0.079625, 0.112979]]}, {"filename": "000000489764.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.582406, 0.516413, 0.835187, 0.967175], [4, 0.06332, 0.608262, 0.126641, 0.267677]]}, {"filename": "000000490055.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.202377, 0.462555, 0.238087, 0.117641], [1, 0.856548, 0.477992, 0.107887, 0.135453], [7, 0.629499, 0.3835, 0.275792, 0.530813], [1, 0.331002, 0.447086, 0.090565, 0.082109]]}, {"filename": "000000490936.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.757635, 0.195563, 0.046438, 0.057344], [0, 0.052417, 0.276562, 0.080417, 0.150094], [0, 0.57426, 0.164953, 0.015063, 0.043469], [0, 0.3685, 0.188578, 0.04425, 0.078438], [0, 0.485594, 0.193586, 0.035021, 0.076922], [0, 0.462667, 0.198148, 0.024917, 0.066547], [0, 0.417854, 0.198953, 0.031458, 0.066687], [0, 0.428646, 0.178789, 0.0155, 0.041891], [0, 0.599906, 0.171875, 0.014438, 0.031187], [0, 0.397208, 0.192563, 0.023167, 0.066594], [1, 0.180021, 0.31768, 0.076583, 0.103891], [1, 0.709531, 0.260742, 0.212979, 0.068297], [1, 0.961396, 0.131516, 0.077208, 0.11475], [5, 0.422615, 0.249047, 0.027354, 0.051688], [0, 0.72775, 0.147461, 0.01325, 0.027734], [1, 0.747333, 0.388984, 0.077833, 0.061438], [1, 0.863115, 0.19482, 0.156479, 0.135609], [1, 0.619844, 0.384367, 0.444271, 0.168734], [1, 0.806438, 0.378563, 0.149583, 0.109563], [1, 0.772042, 0.317672, 0.154625, 0.078906], [1, 0.841656, 0.323328, 0.043188, 0.060312], [1, 0.722167, 0.243672, 0.136, 0.019719], [1, 0.763406, 0.240664, 0.193646, 0.082859], [0, 0.099979, 0.175977, 0.043458, 0.031109], [1, 0.547312, 0.466406, 0.640792, 0.29725], [0, 0.55074, 0.174281, 0.019937, 0.059813], [0, 0.632698, 0.143789, 0.012771, 0.014547], [1, 0.479656, 0.751156, 0.959313, 0.45975], [1, 0.729948, 0.281984, 0.219146, 0.050687], [1, 0.544313, 0.353375, 0.135167, 0.040844], [1, 0.661719, 0.242688, 0.080313, 0.039844]]}, {"filename": "000000491213.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.284195, 0.338781, 0.380047, 0.338562], [1, 0.385016, 0.242938, 0.044531, 0.021708], [0, 0.080641, 0.273281, 0.014938, 0.049812], [1, 0.652781, 0.496292, 0.694438, 0.982708], [1, 0.152914, 0.240083, 0.153172, 0.140917]]}, {"filename": "000000491464.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.472633, 0.638201, 0.262984, 0.701121], [1, 0.286094, 0.513703, 0.079844, 0.030724], [1, 0.363781, 0.517044, 0.076281, 0.021752], [1, 0.820562, 0.532255, 0.030813, 0.034042], [1, 0.885484, 0.533283, 0.051844, 0.033154], [1, 0.556937, 0.511285, 0.067125, 0.011495], [1, 0.925937, 0.534977, 0.031781, 0.035794], [1, 0.843625, 0.529825, 0.020187, 0.029136], [1, 0.790594, 0.531647, 0.025, 0.029276]]}, {"filename": "000000492077.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.567477, 0.384521, 0.017703, 0.01225], [1, 0.566016, 0.378687, 0.017438, 0.011125], [1, 0.519742, 0.391104, 0.029797, 0.04425], [1, 0.143594, 0.260562, 0.287188, 0.521125], [1, 0.35593, 0.781135, 0.129141, 0.345646], [1, 0.443453, 0.597063, 0.053375, 0.197167], [0, 0.78125, 0.408792, 0.045594, 0.121625], [0, 0.351508, 0.627271, 0.159453, 0.599875], [0, 0.437414, 0.494635, 0.100016, 0.328813], [0, 0.511984, 0.431104, 0.028156, 0.118125], [0, 0.488031, 0.484677, 0.054281, 0.263104], [1, 0.500242, 0.533917, 0.010172, 0.049], [1, 0.549367, 0.385937, 0.014516, 0.011583], [0, 0.160953, 0.615729, 0.318531, 0.764042], [0, 0.796359, 0.377281, 0.014844, 0.043312], [0, 0.807828, 0.377885, 0.015344, 0.054271], [0, 0.733609, 0.374458, 0.01, 0.034875], [0, 0.688016, 0.391094, 0.008469, 0.033229], [0, 0.80393, 0.372427, 0.006078, 0.026646], [1, 0.110227, 0.92849, 0.077141, 0.143021]]}, {"filename": "000000492107.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.762922, 0.490323, 0.474156, 0.977521], [0, 0.182023, 0.532583, 0.364047, 0.934833], [0, 0.687078, 0.32125, 0.266781, 0.615208], [4, 0.61175, 0.509417, 0.057188, 0.249875], [4, 0.977094, 0.941479, 0.044688, 0.102417], [4, 0.248742, 0.531354, 0.127078, 0.202]]}, {"filename": "000000492110.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.085898, 0.476815, 0.074109, 0.123044], [0, 0.624266, 0.500574, 0.462375, 0.981944], [0, 0.16268, 0.428091, 0.163422, 0.3663], [0, 0.089703, 0.3237, 0.047188, 0.110773], [4, 0.416055, 0.548314, 0.100453, 0.211241], [4, 0.262195, 0.528419, 0.159734, 0.189906], [4, 0.039961, 0.610679, 0.079922, 0.265621], [4, 0.857313, 0.800386, 0.127906, 0.362693]]}, {"filename": "000000493760.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.55882, 0.624715, 0.309766, 0.210138], [1, 0.06318, 0.597421, 0.122797, 0.163661], [0, 0.283391, 0.523858, 0.132687, 0.312126], [0, 0.343406, 0.454272, 0.056969, 0.104803], [0, 0.922242, 0.328711, 0.053797, 0.125177], [0, 0.010773, 0.596496, 0.021547, 0.202165], [1, 0.900438, 0.483415, 0.199125, 0.573996], [7, 0.74243, 0.564646, 0.016766, 0.043425], [7, 0.724758, 0.565305, 0.015453, 0.044193], [0, 0.975992, 0.9181, 0.048016, 0.163799]]}, {"filename": "000000493905.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.54803, 0.472945, 0.262154, 0.557766], [0, 0.162811, 0.181688, 0.131821, 0.15075], [0, 0.478888, 0.117383, 0.086462, 0.145172], [0, 0.73669, 0.086203, 0.087145, 0.172406], [0, 0.8612, 0.242922, 0.08944, 0.076812], [0, 0.759475, 0.229102, 0.106095, 0.097234], [0, 0.649956, 0.218844, 0.104151, 0.107594], [0, 0.552583, 0.191758, 0.100193, 0.128703], [0, 0.460972, 0.181852, 0.113222, 0.137703], [0, 0.351016, 0.186898, 0.109282, 0.161359], [0, 0.045412, 0.175719, 0.090823, 0.125406], [4, 0.953056, 0.2645, 0.093888, 0.036187], [4, 0.957198, 0.22718, 0.085429, 0.070891], [4, 0.947119, 0.147328, 0.093187, 0.04625], [4, 0.748581, 0.155117, 0.093065, 0.064109], [4, 0.82676, 0.146227, 0.05352, 0.041578], [4, 0.648555, 0.146141, 0.091856, 0.054844], [4, 0.548485, 0.258367, 0.092102, 0.025547], [4, 0.448511, 0.258203, 0.095762, 0.028125], [4, 0.347618, 0.255164, 0.096392, 0.026828], [4, 0.652977, 0.216016, 0.092785, 0.052219], [4, 0.045814, 0.24743, 0.090963, 0.031891], [4, 0.145849, 0.249234, 0.095306, 0.034313], [0, 0.559886, 0.124844, 0.094623, 0.162469], [4, 0.246349, 0.254445, 0.096165, 0.026078], [0, 0.24845, 0.189961, 0.103065, 0.154547]]}, {"filename": "000000495919.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.493273, 0.473782, 0.512766, 0.768548], [1, 0.165266, 0.198115, 0.141781, 0.064614], [0, 0.787141, 0.384438, 0.033656, 0.097166], [0, 0.868367, 0.316511, 0.011891, 0.054707], [0, 0.942641, 0.340433, 0.018031, 0.066815], [0, 0.849734, 0.337061, 0.020063, 0.070328], [1, 0.871914, 0.564192, 0.256172, 0.315316], [1, 0.491609, 0.489848, 0.514687, 0.815948], [1, 0.11743, 0.495398, 0.231859, 0.663677], [0, 0.930547, 0.282178, 0.009437, 0.034052], [0, 0.958125, 0.324988, 0.020438, 0.052272], [0, 0.805859, 0.300398, 0.012031, 0.022998], [1, 0.900742, 0.249883, 0.039109, 0.031288], [1, 0.941172, 0.253185, 0.044031, 0.033536], [1, 0.869148, 0.566534, 0.261703, 0.323934], [0, 0.926789, 0.280972, 0.006016, 0.029719]]}, {"filename": "000000496401.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.803773, 0.32453, 0.125413, 0.07378], [1, 0.906973, 0.27307, 0.186053, 0.09946], [0, 0.632133, 0.35248, 0.069813, 0.16292], [0, 0.58376, 0.32849, 0.04816, 0.11038], [1, 0.682947, 0.2992, 0.088027, 0.07308], [1, 0.686027, 0.30483, 0.077013, 0.07918], [1, 0.594387, 0.27738, 0.030853, 0.01512], [0, 0.562587, 0.3005, 0.021333, 0.06396], [1, 0.149627, 0.365, 0.1464, 0.08412], [5, 0.471253, 0.42529, 0.225867, 0.7133]]}, {"filename": "000000496722.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.457383, 0.435458, 0.023672, 0.03175], [1, 0.478305, 0.430931, 0.025234, 0.017972], [1, 0.541195, 0.423306, 0.014266, 0.029167], [1, 0.55393, 0.426111, 0.018922, 0.033056], [1, 0.438008, 0.433111, 0.018484, 0.021444], [1, 0.63418, 0.422875, 0.034797, 0.02875], [1, 0.514672, 0.420125, 0.009344, 0.017583], [1, 0.525977, 0.417597, 0.013328, 0.015083], [7, 0.924977, 0.426014, 0.109797, 0.281361], [1, 0.499414, 0.432306, 0.020828, 0.032444]]}, {"filename": "000000497438.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.171195, 0.425043, 0.108391, 0.299017], [4, 0.533391, 0.406836, 0.176781, 0.284741], [0, 0.434789, 0.70225, 0.344172, 0.5955], [0, 0.263133, 0.429629, 0.209297, 0.343052], [4, 0.377078, 0.360552, 0.128406, 0.18531], [4, 0.035586, 0.51875, 0.071172, 0.173293], [0, 0.699109, 0.766233, 0.244813, 0.441569]]}, {"filename": "000000498709.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.50075, 0.542694, 0.9985, 0.905624], [1, 0.022883, 0.496494, 0.039859, 0.165082], [1, 0.109758, 0.513671, 0.124109, 0.133694], [1, 0.290508, 0.460847, 0.068516, 0.106682]]}, {"filename": "000000499181.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.908477, 0.442865, 0.144359, 0.081896], [0, 0.184055, 0.87424, 0.024203, 0.036146], [1, 0.167859, 0.90576, 0.273656, 0.185812], [1, 0.056211, 0.614125, 0.112422, 0.08375], [1, 0.841234, 0.536438, 0.145438, 0.090167], [1, 0.947758, 0.664813, 0.104484, 0.172375], [1, 0.562781, 0.539396, 0.097469, 0.025167], [6, 0.405883, 0.707802, 0.034859, 0.080938]]}, {"filename": "000000499768.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.19725, 0.179217, 0.261125, 0.17222], [5, 0.457805, 0.308937, 0.158578, 0.514556], [1, 0.036047, 0.197921, 0.072094, 0.095888]]}, {"filename": "000000500478.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.11959, 0.296336, 0.115957, 0.029453], [4, 0.130441, 0.212078, 0.118997, 0.027313], [0, 0.569559, 0.465102, 0.519726, 0.677609], [0, 0.756185, 0.358906, 0.155532, 0.110813], [0, 0.18965, 0.330031, 0.178815, 0.095875], [0, 0.050365, 0.330406, 0.100365, 0.100063], [0, 0.719574, 0.074078, 0.199818, 0.1165], [0, 0.113511, 0.15382, 0.227021, 0.125703], [0, 0.344909, 0.078469, 0.154073, 0.155719], [0, 0.468328, 0.051578, 0.125532, 0.091], [4, 0.142538, 0.242406, 0.140091, 0.030594], [4, 0.259362, 0.214336, 0.127295, 0.032297], [4, 0.146337, 0.272125, 0.131824, 0.029687], [4, 0.044696, 0.269055, 0.089392, 0.033672], [4, 0.246945, 0.184078, 0.136322, 0.029687], [4, 0.368663, 0.185484, 0.108754, 0.033062], [0, 0.695456, 0.237852, 0.128055, 0.099672], [0, 0.571733, 0.045609, 0.151611, 0.089594], [4, 0.03614, 0.238805, 0.07228, 0.027141], [4, 0.033997, 0.212922, 0.060942, 0.028344], [4, 0.26041, 0.241547, 0.125502, 0.031969], [0, 0.94304, 0.357172, 0.113921, 0.111938], [0, 0.286657, 0.32607, 0.097933, 0.111328], [0, 0.856018, 0.320242, 0.087112, 0.076047]]}, {"filename": "000000500650.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.355148, 0.556279, 0.014734, 0.018239], [7, 0.529562, 0.700282, 0.038719, 0.119155], [1, 0.393203, 0.554472, 0.017687, 0.020446], [7, 0.388234, 0.642324, 0.023312, 0.06784], [1, 0.37707, 0.556608, 0.016766, 0.017535], [1, 0.035969, 0.570376, 0.036781, 0.075681]]}, {"filename": "000000501023.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.17023, 0.085133, 0.07102, 0.096773], [6, 0.56395, 0.32712, 0.10466, 0.12144]]}, {"filename": "000000501176.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.30516, 0.60021, 0.29852, 0.342943], [0, 0.30607, 0.455, 0.19038, 0.558829], [0, 0.71587, 0.23509, 0.07134, 0.276847], [0, 0.14071, 0.27518, 0.08514, 0.239069], [0, 0.69681, 0.523649, 0.1393, 0.472282]]}, {"filename": "000000501549.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.868547, 0.709812, 0.162906, 0.438708], [4, 0.602414, 0.639792, 0.243547, 0.393542], [4, 0.831453, 0.71076, 0.246781, 0.436563], [1, 0.410555, 0.536646, 0.126078, 0.273708], [1, 0.040094, 0.818771, 0.071813, 0.332208], [0, 0.289734, 0.485635, 0.177906, 0.667938], [0, 0.549, 0.648906, 0.191344, 0.463396], [0, 0.762906, 0.710958, 0.244969, 0.574792], [1, 0.157992, 0.477552, 0.177672, 0.225396], [1, 0.038656, 0.453854, 0.076969, 0.167792], [1, 0.119102, 0.483844, 0.108172, 0.175021]]}, {"filename": "000000502012.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.057086, 0.318282, 0.114172, 0.058588], [1, 0.492453, 0.566282, 0.949094, 0.844941], [1, 0.065656, 0.279776, 0.125344, 0.065153], [1, 0.18282, 0.408647, 0.359078, 0.422235]]}, {"filename": "000000502738.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.067145, 0.538961, 0.13429, 0.142859], [5, 0.227326, 0.790766, 0.282897, 0.26525]]}, {"filename": "000000503232.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.047646, 0.504172, 0.095292, 0.13825], [7, 0.504542, 0.643281, 0.19475, 0.498875]]}, {"filename": "000000503252.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.492167, 0.548312, 0.584625, 0.841156]]}, {"filename": "000000503557.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.87682, 0.801625, 0.246359, 0.391333], [0, 0.422859, 0.582115, 0.638219, 0.821271], [4, 0.977141, 0.582969, 0.045719, 0.060229]]}, {"filename": "000000504074.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.823141, 0.463536, 0.133375, 0.191991], [4, 0.14618, 0.459555, 0.166422, 0.276393], [4, 0.568562, 0.540597, 0.134781, 0.343583], [0, 0.237422, 0.510141, 0.162281, 0.664403], [4, 0.908563, 0.726967, 0.182875, 0.523583], [4, 0.919906, 0.709473, 0.134031, 0.484754], [0, 0.636063, 0.602307, 0.458719, 0.795386]]}, {"filename": "000000504425.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.820547, 0.09876, 0.358907, 0.10836], [1, 0.243573, 0.0914, 0.487147, 0.152], [1, 0.244773, 0.08803, 0.489547, 0.15802], [5, 0.52568, 0.55602, 0.280907, 0.34896]]}, {"filename": "000000506178.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.91312, 0.31245, 0.17376, 0.22818], [0, 0.494773, 0.58547, 0.612853, 0.80718]]}, {"filename": "000000506379.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.72125, 0.664476, 0.342531, 0.620218]]}, {"filename": "000000506454.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.516047, 0.458333, 0.383438, 0.268], [0, 0.871469, 0.167115, 0.009375, 0.032646]]}, {"filename": "000000506707.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.947461, 0.532342, 0.105078, 0.515972], [0, 0.832109, 0.557307, 0.194906, 0.597752], [0, 0.332094, 0.589883, 0.118438, 0.370773], [0, 0.501211, 0.533677, 0.125453, 0.67452], [0, 0.659625, 0.504204, 0.12925, 0.424286], [0, 0.754492, 0.501429, 0.077859, 0.265246], [1, 0.351773, 0.497131, 0.154953, 0.276183], [1, 0.59025, 0.527553, 0.03675, 0.272295], [3, 0.751195, 0.634215, 0.066672, 0.025527], [0, 0.129227, 0.570948, 0.171297, 0.626112], [0, 0.557336, 0.524075, 0.074734, 0.444918], [4, 0.024805, 0.622225, 0.049609, 0.254426], [1, 0.031055, 0.407787, 0.062109, 0.073326], [4, 0.226766, 0.595023, 0.023438, 0.035386]]}, {"filename": "000000507091.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.095414, 0.365622, 0.190828, 0.124155], [1, 0.346227, 0.371174, 0.213328, 0.103099], [1, 0.564555, 0.364319, 0.046984, 0.06615], [1, 0.826758, 0.355305, 0.157141, 0.088592], [1, 0.95368, 0.474695, 0.092641, 0.222958], [7, 0.637594, 0.307453, 0.112875, 0.399085], [7, 0.195047, 0.332042, 0.007406, 0.032582]]}, {"filename": "000000507826.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.466468, 0.514031, 0.809356, 0.949469], [3, 0.961408, 0.694078, 0.076086, 0.179906]]}, {"filename": "000000508370.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.411522, 0.669992, 0.823044, 0.427016], [0, 0.810035, 0.396625, 0.360398, 0.626969], [0, 0.59685, 0.368227, 0.234731, 0.480984]]}, {"filename": "000000508917.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.064383, 0.599122, 0.053828, 0.247432], [0, 0.093008, 0.569014, 0.023266, 0.131108], [0, 0.130773, 0.601608, 0.048297, 0.182459], [0, 0.227039, 0.598811, 0.039953, 0.096595], [0, 0.246, 0.596595, 0.045094, 0.102162], [0, 0.255656, 0.589946, 0.032375, 0.102703], [0, 0.274031, 0.587405, 0.037406, 0.107135], [0, 0.841125, 0.621986, 0.050594, 0.237919], [0, 0.443828, 0.495365, 0.0165, 0.022838], [0, 0.205859, 0.600095, 0.037719, 0.112297], [3, 0.186711, 0.612986, 0.047422, 0.082027], [0, 0.744055, 0.536203, 0.007172, 0.038514], [0, 0.921641, 0.605081, 0.036781, 0.121297]]}, {"filename": "000000509258.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.199281, 0.816682, 0.050719, 0.06215]]}, {"filename": "000000510035.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.534477, 0.596557, 0.102266, 0.032974], [1, 0.584906, 0.648677, 0.058031, 0.030984], [1, 0.5705, 0.633103, 0.033406, 0.027658], [0, 0.863008, 0.626206, 0.021516, 0.086323], [6, 0.152305, 0.454707, 0.039734, 0.091803]]}, {"filename": "000000510095.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.752539, 0.578446, 0.179641, 0.534977], [5, 0.703117, 0.092792, 0.012672, 0.031986]]}, {"filename": "000000511301.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.755383, 0.759021, 0.488516, 0.4745], [7, 0.649156, 0.523521, 0.01875, 0.031], [0, 0.667367, 0.518927, 0.041766, 0.081271], [0, 0.62018, 0.550802, 0.025922, 0.110104], [0, 0.925289, 0.510042, 0.019766, 0.043917], [1, 0.536266, 0.615156, 0.039156, 0.094812], [7, 0.276805, 0.493792, 0.550234, 0.984208], [0, 0.903789, 0.509729, 0.014141, 0.039667], [1, 0.779633, 0.521896, 0.100266, 0.02175]]}, {"filename": "000000513181.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.782484, 0.676147, 0.028313, 0.025343], [1, 0.498992, 0.669303, 0.039141, 0.02253], [1, 0.594906, 0.672163, 0.022281, 0.013073], [1, 0.797906, 0.674598, 0.009594, 0.018558], [1, 0.547313, 0.671229, 0.019313, 0.017967], [1, 0.824938, 0.671489, 0.019031, 0.020757], [1, 0.880617, 0.678333, 0.020453, 0.012411], [1, 0.903359, 0.673428, 0.01275, 0.014043], [1, 0.949062, 0.67818, 0.029594, 0.023121], [1, 0.741789, 0.676371, 0.020391, 0.010378], [1, 0.734297, 0.670355, 0.027687, 0.015461], [1, 0.616422, 0.672813, 0.033437, 0.025296]]}, {"filename": "000000513708.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.32157, 0.517531, 0.087859, 0.091854]]}, {"filename": "000000514376.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.924875, 0.580792, 0.052594, 0.04825], [1, 0.947672, 0.545698, 0.022344, 0.021604], [1, 0.618836, 0.543719, 0.336328, 0.325104], [1, 0.911859, 0.55876, 0.030687, 0.030146], [3, 0.081398, 0.615865, 0.094797, 0.029187]]}, {"filename": "000000515077.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.910672, 0.835261, 0.178656, 0.295449], [4, 0.807172, 0.673967, 0.081937, 0.20618], [4, 0.213516, 0.735887, 0.114719, 0.230438], [4, 0.34482, 0.633873, 0.076359, 0.144154], [0, 0.089578, 0.623466, 0.179156, 0.744969], [4, 0.949148, 0.922443, 0.101703, 0.155115], [4, 0.899797, 0.774729, 0.063813, 0.054593], [4, 0.985875, 0.61762, 0.02825, 0.137119]]}, {"filename": "000000515107.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.223703, 0.60785, 0.080375, 0.105047], [1, 0.700492, 0.61222, 0.045422, 0.170607], [1, 0.512719, 0.512079, 0.036781, 0.043925], [1, 0.56557, 0.510549, 0.044953, 0.050164], [1, 0.596313, 0.497804, 0.03075, 0.04271], [1, 0.019094, 0.668061, 0.034875, 0.149813], [0, 0.830133, 0.716893, 0.120516, 0.429626], [0, 0.481375, 0.492208, 0.017344, 0.033061], [1, 0.54793, 0.496542, 0.041953, 0.051402], [1, 0.710508, 0.539614, 0.054984, 0.086799], [3, 0.155297, 0.565794, 0.021781, 0.027757], [1, 0.480633, 0.519661, 0.034734, 0.028902], [1, 0.583625, 0.532114, 0.044469, 0.020397], [1, 0.510344, 0.678096, 0.20775, 0.291706]]}, {"filename": "000000515266.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.964047, 0.275726, 0.056969, 0.058431], [1, 0.703352, 0.424578, 0.560578, 0.384614], [3, 0.569562, 0.69507, 0.860875, 0.547096], [1, 0.280055, 0.337037, 0.490922, 0.377775]]}, {"filename": "000000515577.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.45198, 0.463413, 0.0838, 0.225808], [0, 0.54237, 0.375045, 0.05538, 0.204341]]}, {"filename": "000000515579.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.24618, 0.530617, 0.4532, 0.916114], [0, 0.62121, 0.221476, 0.07122, 0.278313], [3, 0.60782, 0.227274, 0.15612, 0.14509], [0, 0.70623, 0.159849, 0.06654, 0.196988]]}, {"filename": "000000516143.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.479031, 0.595292, 0.760062, 0.425042], [0, 0.87268, 0.637219, 0.027766, 0.152104]]}, {"filename": "000000516812.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.784869, 0.380605, 0.108595, 0.127712], [5, 0.409208, 0.621601, 0.208938, 0.290196]]}, {"filename": "000000517069.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.254414, 0.541073, 0.069734, 0.027521], [1, 0.231203, 0.533687, 0.063062, 0.026458], [1, 0.748523, 0.680938, 0.193859, 0.115792], [1, 0.385109, 0.583146, 0.055625, 0.047208], [0, 0.893062, 0.730813, 0.164219, 0.510167], [3, 0.737828, 0.842573, 0.524344, 0.314854], [0, 0.491797, 0.743115, 0.332531, 0.513771], [1, 0.267461, 0.552813, 0.052859, 0.032792], [1, 0.359352, 0.55124, 0.044422, 0.036563], [1, 0.234258, 0.516833, 0.089672, 0.028542]]}, {"filename": "000000517403.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.733992, 0.38651, 0.311797, 0.755062], [0, 0.457586, 0.451688, 0.207297, 0.507875], [0, 0.284836, 0.434448, 0.559547, 0.851688]]}, {"filename": "000000518649.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.821313, 0.633113, 0.034, 0.036698], [1, 0.772102, 0.629222, 0.031516, 0.04184], [1, 0.721406, 0.622925, 0.02075, 0.021557], [1, 0.686719, 0.622229, 0.029812, 0.022901], [0, 0.910195, 0.7973, 0.128328, 0.385118], [0, 0.051953, 0.772653, 0.09775, 0.42158], [0, 0.689117, 0.666403, 0.031609, 0.090542], [0, 0.971953, 0.761509, 0.056094, 0.32283], [5, 0.267469, 0.871144, 0.070187, 0.158514], [0, 0.939891, 0.631969, 0.011063, 0.056533], [0, 0.673867, 0.637665, 0.011703, 0.027406], [0, 0.959258, 0.631167, 0.013859, 0.047524], [0, 0.517672, 0.60421, 0.018375, 0.077241], [1, 0.792539, 0.630153, 0.018953, 0.033703], [0, 0.712117, 0.812807, 0.133578, 0.374387], [0, 0.836953, 0.749104, 0.056438, 0.240708], [0, 0.70743, 0.665896, 0.016641, 0.086132], [0, 0.759523, 0.668679, 0.026734, 0.123019]]}, {"filename": "000000519039.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.392914, 0.534548, 0.250922, 0.335719]]}, {"filename": "000000519555.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.37393, 0.430177, 0.177922, 0.295038]]}, {"filename": "000000520272.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.069742, 0.652742, 0.040516, 0.079032], [3, 0.515062, 0.48712, 0.122094, 0.116175], [3, 0.87243, 0.504309, 0.110766, 0.095438], [0, 0.509602, 0.430253, 0.065891, 0.074055], [0, 0.287109, 0.418329, 0.035438, 0.070668], [3, 0.296305, 0.520818, 0.153297, 0.110576], [0, 0.29518, 0.453882, 0.124609, 0.166382], [0, 0.869867, 0.45682, 0.093203, 0.086359], [0, 0.736992, 0.483157, 0.014234, 0.044839]]}, {"filename": "000000520531.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.10918, 0.434792, 0.215984, 0.491833]]}, {"filename": "000000520910.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.470937, 0.495258, 0.191569, 0.483266]]}, {"filename": "000000521543.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.619289, 0.741073, 0.047453, 0.168561]]}, {"filename": "000000521689.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.185333, 0.337891, 0.220542, 0.135875], [1, 0.784448, 0.295375, 0.431104, 0.136813], [0, 0.444229, 0.446883, 0.34675, 0.638234], [0, 0.201219, 0.274313, 0.138521, 0.188438], [0, 0.550802, 0.242813, 0.080896, 0.092406], [0, 0.31225, 0.23793, 0.062458, 0.053859], [0, 0.977563, 0.250945, 0.034125, 0.042891], [0, 0.826146, 0.21818, 0.021625, 0.019828], [1, 0.250177, 0.250023, 0.055104, 0.026266], [1, 0.053656, 0.277727, 0.018854, 0.023141], [6, 0.250719, 0.079133, 0.055979, 0.055453]]}, {"filename": "000000522406.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.401198, 0.210508, 0.611479, 0.420578], [1, 0.572479, 0.821172, 0.106042, 0.057594], [1, 0.321271, 0.802406, 0.053625, 0.028344], [1, 0.176542, 0.809781, 0.094833, 0.048844], [1, 0.608177, 0.804594, 0.065854, 0.048531], [1, 0.066708, 0.817406, 0.133417, 0.055625], [1, 0.349542, 0.801078, 0.027583, 0.028313], [1, 0.456979, 0.792672, 0.031833, 0.022344], [5, 0.636823, 0.853867, 0.038979, 0.040203], [1, 0.265583, 0.805633, 0.083208, 0.034766], [1, 0.63851, 0.796086, 0.035437, 0.030234], [1, 0.63349, 0.783234, 0.045396, 0.017312], [7, 0.657938, 0.811602, 0.013125, 0.026766]]}, {"filename": "000000522579.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.635945, 0.6, 0.666859, 0.503361], [1, 0.353492, 0.187637, 0.706984, 0.370777], [0, 0.115328, 0.326964, 0.096937, 0.267416], [0, 0.644273, 0.467384, 0.213922, 0.609349], [0, 0.429625, 0.311355, 0.101469, 0.236029], [0, 0.726195, 0.318214, 0.094828, 0.320252], [0, 0.534156, 0.234748, 0.054063, 0.101092], [0, 0.519922, 0.33541, 0.081375, 0.201282], [0, 0.914883, 0.247321, 0.051016, 0.179853], [0, 0.588859, 0.323918, 0.058938, 0.184853], [0, 0.933531, 0.251145, 0.039937, 0.174643], [0, 0.968477, 0.272647, 0.063047, 0.301975]]}, {"filename": "000000522940.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.368444, 0.313523, 0.703268, 0.572703]]}, {"filename": "000000523807.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.47866, 0.764053, 0.4382, 0.44944], [0, 0.91016, 0.595053, 0.17968, 0.78648]]}, {"filename": "000000524061.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.457102, 0.301835, 0.300766, 0.562776], [3, 0.188492, 0.476071, 0.115766, 0.086118]]}, {"filename": "000000524108.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.414328, 0.6281, 0.823031, 0.72135], [1, 0.561094, 0.568538, 0.877812, 0.844925], [1, 0.465203, 0.206138, 0.444656, 0.188325], [1, 0.323563, 0.052587, 0.240531, 0.081725], [1, 0.36175, 0.161887, 0.441, 0.284725], [1, 0.961437, 0.399312, 0.077125, 0.422425], [1, 0.92907, 0.22865, 0.130609, 0.2899], [1, 0.096914, 0.186512, 0.185391, 0.350575], [1, 0.52868, 0.067088, 0.236078, 0.134175], [1, 0.879047, 0.496262, 0.211969, 0.314225], [1, 0.740914, 0.122838, 0.157672, 0.152825], [1, 0.342672, 0.090213, 0.305125, 0.076275]]}, {"filename": "000000525083.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.406602, 0.623364, 0.111578, 0.186402], [4, 0.502734, 0.567792, 0.051969, 0.072079]]}, {"filename": "000000526166.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.557054, 0.525992, 0.688709, 0.925828], [3, 0.380469, 0.293266, 0.506291, 0.40925], [0, 0.783967, 0.10857, 0.156479, 0.217141], [0, 0.964765, 0.112688, 0.070469, 0.225375]]}, {"filename": "000000526204.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.119365, 0.508758, 0.238729, 0.457922], [0, 0.7985, 0.549438, 0.326583, 0.375281], [0, 0.705271, 0.268234, 0.194167, 0.436875], [0, 0.532333, 0.241711, 0.15875, 0.334422], [0, 0.397292, 0.247578, 0.135625, 0.312031], [0, 0.2675, 0.331391, 0.227625, 0.220937], [0, 0.881646, 0.208312, 0.103458, 0.199375], [0, 0.820646, 0.153094, 0.091292, 0.114437], [0, 0.870958, 0.126656, 0.058458, 0.095375], [0, 0.470417, 0.113484, 0.059917, 0.051687], [0, 0.93101, 0.274383, 0.137979, 0.332891], [0, 0.985875, 0.11032, 0.02825, 0.041516]]}, {"filename": "000000526728.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.84133, 0.493093, 0.1261, 0.073333], [1, 0.36599, 0.480653, 0.07934, 0.05256], [0, 0.56699, 0.571627, 0.14546, 0.3424], [0, 0.45422, 0.54944, 0.1264, 0.34832], [1, 0.96499, 0.49736, 0.07002, 0.079147], [1, 0.31884, 0.464267, 0.02304, 0.02816]]}, {"filename": "000000526874.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.429625, 0.762033, 0.19225, 0.190875], [1, 0.059273, 0.747754, 0.118547, 0.208511], [1, 0.240617, 0.770095, 0.184172, 0.179764], [1, 0.940758, 0.845875, 0.118484, 0.202813], [7, 0.646016, 0.761903, 0.029312, 0.172884]]}, {"filename": "000000527059.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.820773, 0.820958, 0.219234, 0.220042], [7, 0.820469, 0.332427, 0.194625, 0.664438], [7, 0.973195, 0.262698, 0.053609, 0.525396], [5, 0.285547, 0.891875, 0.053906, 0.105208]]}, {"filename": "000000527830.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.214852, 0.682436, 0.423609, 0.61485], [1, 0.389328, 0.35545, 0.063187, 0.079908], [7, 0.582672, 0.410785, 0.043656, 0.224988], [1, 0.121758, 0.34418, 0.105609, 0.066882]]}, {"filename": "000000527960.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.593703, 0.727927, 0.519344, 0.442131]]}, {"filename": "000000528076.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.44717, 0.552703, 0.32678, 0.893732]]}, {"filename": "000000528731.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.679539, 0.621021, 0.193484, 0.57325]]}, {"filename": "000000528980.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.451221, 0.854734, 0.040141, 0.067094], [1, 0.310857, 0.973578, 0.237535, 0.052844], [1, 0.129836, 0.983023, 0.176995, 0.033953], [1, 0.194695, 0.950109, 0.115775, 0.028094], [1, 0.273322, 0.957711, 0.13392, 0.047078], [1, 0.363779, 0.914203, 0.132207, 0.031875], [1, 0.198204, 0.926117, 0.115469, 0.040078], [1, 0.055563, 0.940484, 0.061784, 0.018219]]}, {"filename": "000000530289.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.398789, 0.39482, 0.772282, 0.72808], [0, 0.148254, 0.2575, 0.195606, 0.1138], [0, 0.298944, 0.2609, 0.104535, 0.11076], [0, 0.142366, 0.19644, 0.28107, 0.22948], [0, 0.305239, 0.05621, 0.112958, 0.1093], [0, 0.094944, 0.04831, 0.183577, 0.09662], [0, 0.06, 0.53722, 0.12, 0.60988], [0, 0.676775, 0.09777, 0.309944, 0.19554], [0, 0.942366, 0.18613, 0.115155, 0.1495], [0, 0.765521, 0.22601, 0.25662, 0.0841]]}, {"filename": "000000530975.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.60318, 0.5085, 0.310547, 0.1595], [0, 0.054953, 0.389958, 0.109906, 0.18775], [0, 0.675687, 0.372396, 0.134187, 0.184708], [0, 0.068117, 0.432281, 0.089953, 0.255854], [4, 0.722734, 0.45674, 0.050219, 0.076521], [0, 0.560242, 0.374375, 0.121766, 0.151833], [4, 0.794273, 0.479563, 0.032891, 0.08125], [4, 0.817703, 0.472302, 0.036469, 0.133396], [4, 0.024141, 0.503958, 0.045781, 0.1515], [4, 0.243969, 0.419708, 0.025687, 0.040083], [4, 0.277758, 0.418031, 0.005828, 0.039938], [0, 0.361789, 0.366802, 0.013703, 0.029229], [0, 0.207625, 0.385854, 0.075469, 0.275875]]}, {"filename": "000000531423.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.339719, 0.56166, 0.162469, 0.464118], [6, 0.496547, 0.175247, 0.154125, 0.213909]]}, {"filename": "000000532085.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.18237, 0.608133, 0.1965, 0.129707], [7, 0.22366, 0.53464, 0.08744, 0.239893], [7, 0.01575, 0.555053, 0.01678, 0.046373], [7, 0.0325, 0.549787, 0.01556, 0.041493], [7, 0.29927, 0.536307, 0.07258, 0.22184], [1, 0.76492, 0.57568, 0.25608, 0.166187]]}, {"filename": "000000532426.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.84592, 0.38039, 0.30816, 0.411652], [1, 0.27843, 0.501171, 0.31602, 0.668288], [7, 0.05518, 0.11467, 0.03604, 0.067057], [7, 0.09384, 0.118273, 0.03348, 0.060991], [7, 0.75958, 0.208889, 0.03848, 0.100541]]}, {"filename": "000000532481.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.44668, 0.471127, 0.109547, 0.1523], [1, 0.69075, 0.699847, 0.021031, 0.018333], [1, 0.708625, 0.698908, 0.019, 0.017958], [1, 0.729633, 0.694683, 0.019922, 0.019836], [1, 0.64593, 0.685869, 0.019766, 0.023146], [1, 0.974781, 0.688462, 0.018125, 0.018662], [1, 0.817016, 0.696843, 0.033094, 0.013075], [0, 0.455602, 0.892406, 0.018859, 0.045094]]}, {"filename": "000000533827.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.084336, 0.518002, 0.167953, 0.566853], [0, 0.898148, 0.336025, 0.068922, 0.373665], [0, 0.967547, 0.362019, 0.064906, 0.306729], [0, 0.874516, 0.338965, 0.0425, 0.324306], [0, 0.79393, 0.340021, 0.081047, 0.359627], [0, 0.165195, 0.255549, 0.026484, 0.068406], [0, 0.257703, 0.373437, 0.098344, 0.360248], [0, 0.098891, 0.181491, 0.066781, 0.105093], [0, 0.834672, 0.353002, 0.081438, 0.38766], [0, 0.200555, 0.354534, 0.057516, 0.196936], [0, 0.109742, 0.37382, 0.084422, 0.27294], [0, 0.590508, 0.408592, 0.090172, 0.203975], [0, 0.162414, 0.388054, 0.057328, 0.240994], [1, 0.552008, 0.544555, 0.607109, 0.784638], [0, 0.322023, 0.216429, 0.041078, 0.05058]]}, {"filename": "000000533889.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.281586, 0.444677, 0.414484, 0.351229], [0, 0.86168, 0.460115, 0.021703, 0.028354], [1, 0.931938, 0.45475, 0.025281, 0.015458], [3, 0.905961, 0.504292, 0.187984, 0.066083], [1, 0.877328, 0.513427, 0.04, 0.038812], [1, 0.588758, 0.506583, 0.032109, 0.051083], [1, 0.858352, 0.513667, 0.022016, 0.038833]]}, {"filename": "000000534041.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.59157, 0.543133, 0.4045, 0.88088], [0, 0.72331, 0.496627, 0.55338, 0.97976], [0, 0.85112, 0.203373, 0.18876, 0.406747], [0, 0.72434, 0.110347, 0.0652, 0.216853], [3, 0.86527, 0.384267, 0.26946, 0.258293], [4, 0.14941, 0.079427, 0.0879, 0.151067], [4, 0.04973, 0.075587, 0.09798, 0.1492], [3, 0.15137, 0.10436, 0.10966, 0.110587]]}, {"filename": "000000534673.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.496914, 0.503583, 0.859297, 0.677125], [0, 0.63132, 0.443552, 0.038859, 0.086646], [0, 0.032992, 0.56901, 0.064609, 0.208729]]}, {"filename": "000000534702.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.462859, 0.272296, 0.415219, 0.494378], [5, 0.771813, 0.476685, 0.022094, 0.071352]]}, {"filename": "000000535981.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.071505, 0.083422, 0.143011, 0.112531], [1, 0.164437, 0.083891, 0.095528, 0.082156], [1, 0.282967, 0.081078, 0.124349, 0.076594], [1, 0.485599, 0.068234, 0.130775, 0.095656], [1, 0.654481, 0.055563, 0.101074, 0.081], [1, 0.556972, 0.061453, 0.077289, 0.054094], [1, 0.811725, 0.046047, 0.06243, 0.074781], [5, 0.483847, 0.919297, 0.179384, 0.161406], [1, 0.601637, 0.068344, 0.025739, 0.047188], [1, 0.750986, 0.049688, 0.066303, 0.053938], [1, 0.88765, 0.040812, 0.060933, 0.054375]]}, {"filename": "000000537153.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.702906, 0.92524, 0.020625, 0.063409], [0, 0.677961, 0.93779, 0.021016, 0.081692], [3, 0.675758, 0.952803, 0.105672, 0.071919]]}, {"filename": "000000537355.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.428187, 0.711811, 0.035813, 0.126752]]}, {"filename": "000000537812.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.561328, 0.718417, 0.200844, 0.36225]]}, {"filename": "000000537964.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.207211, 0.485, 0.262328, 0.892558]]}, {"filename": "000000538463.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.74118, 0.515094, 0.051141, 0.207892], [0, 0.839664, 0.510386, 0.032703, 0.183255], [0, 0.810406, 0.502998, 0.032375, 0.181546], [0, 0.784148, 0.509403, 0.040859, 0.182881], [0, 0.673461, 0.498864, 0.047547, 0.191218], [0, 0.694391, 0.424356, 0.0255, 0.072412], [0, 0.931922, 0.537494, 0.051094, 0.127354], [0, 0.923758, 0.469098, 0.036047, 0.181288], [3, 0.941398, 0.613208, 0.073516, 0.15733], [0, 0.821398, 0.488653, 0.022516, 0.197681], [0, 0.868586, 0.443759, 0.025672, 0.070515], [0, 0.705883, 0.477881, 0.030797, 0.155527], [0, 0.891852, 0.529356, 0.063234, 0.281475], [0, 0.700508, 0.525105, 0.023609, 0.138314]]}, {"filename": "000000538872.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.246516, 0.570602, 0.150063, 0.694373], [0, 0.599344, 0.283145, 0.124625, 0.341572], [0, 0.755539, 0.443698, 0.210547, 0.819828], [0, 0.474961, 0.444472, 0.126359, 0.679656], [0, 0.873273, 0.338047, 0.095891, 0.491818]]}, {"filename": "000000539883.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.654219, 0.792037, 0.674375, 0.415925], [0, 0.492227, 0.45459, 0.035891, 0.226838]]}, {"filename": "000000539904.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.4862, 0.55964, 0.574853, 0.73588], [0, 0.8788, 0.45102, 0.110133, 0.21148], [0, 0.971773, 0.47384, 0.018853, 0.04816]]}, {"filename": "000000539962.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.061781, 0.60949, 0.0405, 0.052604], [0, 0.093484, 0.552146, 0.020781, 0.055375], [0, 0.536852, 0.655948, 0.018047, 0.024979], [0, 0.541633, 0.659167, 0.041828, 0.032375], [0, 0.127672, 0.572313, 0.018906, 0.076625], [4, 0.022492, 0.619552, 0.024859, 0.058146], [4, 0.089578, 0.606375, 0.020875, 0.0525], [4, 0.102445, 0.607469, 0.018859, 0.050271], [4, 0.149414, 0.598969, 0.022891, 0.036938], [4, 0.169688, 0.597969, 0.0205, 0.042354], [4, 0.184648, 0.594073, 0.017766, 0.037396], [4, 0.037148, 0.618594, 0.016203, 0.056771]]}, {"filename": "000000540466.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.831563, 0.64856, 0.300688, 0.437447]]}, {"filename": "000000540502.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.023297, 0.581089, 0.037562, 0.200445], [4, 0.059602, 0.644742, 0.043641, 0.193607]]}, {"filename": "000000540547.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.162938, 0.55588, 0.182125, 0.231296], [1, 0.585508, 0.59625, 0.825172, 0.6], [7, 0.155664, 0.525775, 0.028891, 0.089699]]}, {"filename": "000000541223.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.09799, 0.915734, 0.0497, 0.146617]]}, {"filename": "000000542423.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.042508, 0.519588, 0.032516, 0.019976], [0, 0.590875, 0.642118, 0.089531, 0.329176], [3, 0.674305, 0.7102, 0.328297, 0.199035], [0, 0.502336, 0.533106, 0.010203, 0.0288], [0, 0.280602, 0.529141, 0.010547, 0.029012], [1, 0.188094, 0.520859, 0.031906, 0.021529], [1, 0.101562, 0.520447, 0.031156, 0.016282], [1, 0.138039, 0.518812, 0.023922, 0.009906], [1, 0.95675, 0.541341, 0.035219, 0.018682], [1, 0.803227, 0.543435, 0.014484, 0.013365], [1, 0.776789, 0.544541, 0.006328, 0.011529], [0, 0.249414, 0.518718, 0.004703, 0.016682]]}, {"filename": "000000542856.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.467781, 0.644941, 0.923563, 0.494379], [0, 0.331008, 0.680585, 0.078984, 0.120703], [0, 0.412258, 0.679824, 0.080703, 0.117869], [0, 0.250437, 0.681101, 0.078313, 0.121311], [0, 0.766273, 0.605258, 0.017328, 0.098712]]}, {"filename": "000000543515.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.119359, 0.200344, 0.103281, 0.388271], [0, 0.330672, 0.142875, 0.121469, 0.28575], [0, 0.569047, 0.105354, 0.246688, 0.200208]]}, {"filename": "000000544094.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.593203, 0.267062, 0.039656, 0.132625], [3, 0.32618, 0.794531, 0.523672, 0.366854]]}, {"filename": "000000544121.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.833891, 0.855927, 0.332219, 0.257271]]}, {"filename": "000000545500.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.33725, 0.5844, 0.09874, 0.225867], [0, 0.59767, 0.53556, 0.11374, 0.28952], [0, 0.32417, 0.349613, 0.07458, 0.155013], [0, 0.39997, 0.342733, 0.08386, 0.146587], [0, 0.30424, 0.128747, 0.05, 0.085227], [0, 0.77915, 0.29788, 0.08382, 0.120187], [0, 0.87337, 0.27372, 0.05658, 0.137893], [0, 0.12956, 0.365747, 0.06892, 0.15976], [0, 0.24481, 0.36452, 0.06818, 0.165627], [0, 0.20145, 0.362493, 0.04858, 0.1764], [0, 0.34282, 0.316747, 0.04416, 0.113653], [3, 0.65605, 0.343493, 0.07498, 0.01744], [3, 0.67127, 0.344467, 0.32678, 0.073627], [0, 0.40971, 0.5962, 0.0859, 0.208773], [0, 0.70817, 0.298747, 0.0651, 0.140533]]}, {"filename": "000000546219.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.706164, 0.929215, 0.161922, 0.11459], [4, 0.123219, 0.921171, 0.14125, 0.137377], [0, 0.073359, 0.790632, 0.1465, 0.418735], [0, 0.167273, 0.655984, 0.107609, 0.283302], [0, 0.222648, 0.784262, 0.187422, 0.404496], [4, 0.309297, 0.70452, 0.030125, 0.105527], [0, 0.601969, 0.76966, 0.286375, 0.4337], [0, 0.809617, 0.76966, 0.164922, 0.460679], [0, 0.020945, 0.597564, 0.041891, 0.192319], [0, 0.60843, 0.630796, 0.062109, 0.269461], [0, 0.824609, 0.667307, 0.239438, 0.406979], [0, 0.468531, 0.631464, 0.154437, 0.220211], [4, 0.902805, 0.844426, 0.067703, 0.207026], [4, 0.463586, 0.646745, 0.117641, 0.014895]]}, {"filename": "000000546229.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.08993, 0.551531, 0.130891, 0.178271], [1, 0.09118, 0.696833, 0.182359, 0.247583], [0, 0.610109, 0.58876, 0.151687, 0.642687], [0, 0.474219, 0.549552, 0.232062, 0.793729], [0, 0.234273, 0.5955, 0.235953, 0.737083], [0, 0.913133, 0.584792, 0.173734, 0.608208], [1, 0.587219, 0.561365, 0.536719, 0.367312], [1, 0.320031, 0.484208, 0.057719, 0.169583], [1, 0.723313, 0.652698, 0.225531, 0.228062], [1, 0.515, 0.671542, 0.112156, 0.208292], [1, 0.955367, 0.701927, 0.067109, 0.078771]]}, {"filename": "000000547130.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.554898, 0.5, 0.699766, 1.0]]}, {"filename": "000000548339.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.561984, 0.460043, 0.541906, 0.812483], [0, 0.687641, 0.17079, 0.238656, 0.332587], [0, 0.926742, 0.216484, 0.146516, 0.429219], [0, 0.264523, 0.160304, 0.184578, 0.319948], [3, 0.287609, 0.42717, 0.438969, 0.396667], [0, 0.421289, 0.114974, 0.176328, 0.229948], [3, 0.809828, 0.461745, 0.380344, 0.283108]]}, {"filename": "000000548528.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.454696, 0.471906, 0.687096, 0.930344]]}, {"filename": "000000548734.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.202289, 0.55506, 0.286891, 0.862912], [3, 0.43182, 0.281134, 0.044141, 0.066277]]}, {"filename": "000000548780.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.669438, 0.514578, 0.321625, 0.728689], [0, 0.497766, 0.517986, 0.257875, 0.698876], [0, 0.506758, 0.168162, 0.245891, 0.332576], [0, 0.341844, 0.246066, 0.134937, 0.492131], [0, 0.224648, 0.247752, 0.124734, 0.493255], [0, 0.832281, 0.17219, 0.072344, 0.26363], [3, 0.566805, 0.555023, 0.616859, 0.607799]]}, {"filename": "000000549230.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.961727, 0.621375, 0.076547, 0.098667], [0, 0.180727, 0.532312, 0.080828, 0.358792], [0, 0.388258, 0.515771, 0.097859, 0.11275], [0, 0.777828, 0.545885, 0.131969, 0.142604], [4, 0.815211, 0.595281, 0.196297, 0.240187], [0, 0.613922, 0.530573, 0.139656, 0.089854]]}, {"filename": "000000550019.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.499262, 0.539266, 0.998525, 0.473156], [7, 0.596499, 0.399328, 0.460398, 0.634531]]}, {"filename": "000000550147.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.749195, 0.530104, 0.124203, 0.298917], [1, 0.82032, 0.386656, 0.085016, 0.124437], [1, 0.457711, 0.312021, 0.023016, 0.042208], [1, 0.53068, 0.313927, 0.101297, 0.066854], [1, 0.140492, 0.278521, 0.049203, 0.048542], [1, 0.527656, 0.305688, 0.113031, 0.057125], [0, 0.667945, 0.42576, 0.235859, 0.793937], [0, 0.458617, 0.725219, 0.201016, 0.358104], [0, 0.155625, 0.30099, 0.15625, 0.351146], [0, 0.949219, 0.690396, 0.101562, 0.609375], [0, 0.818125, 0.34024, 0.071563, 0.208396], [0, 0.929984, 0.285938, 0.08675, 0.125708], [6, 0.929875, 0.211625, 0.027906, 0.047833], [0, 0.028508, 0.253979, 0.020297, 0.05875], [0, 0.415492, 0.26201, 0.016984, 0.027062], [1, 0.148609, 0.405271, 0.233687, 0.241708], [1, 0.908109, 0.408083, 0.046625, 0.150667], [1, 0.027328, 0.279583, 0.006125, 0.033167], [1, 0.430406, 0.303354, 0.020875, 0.029417], [1, 0.058195, 0.273896, 0.032047, 0.033708], [1, 0.900297, 0.323156, 0.031687, 0.033229], [0, 0.255992, 0.279781, 0.015078, 0.060146], [0, 0.269719, 0.2795, 0.011437, 0.057083]]}, {"filename": "000000550273.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.429427, 0.492117, 0.858854, 0.711703], [0, 0.124344, 0.298875, 0.248688, 0.170781], [0, 0.916479, 0.529211, 0.167042, 0.793266], [4, 0.933531, 0.511242, 0.126646, 0.170516]]}, {"filename": "000000550543.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.09967, 0.233107, 0.0473, 0.09912], [6, 0.41337, 0.129067, 0.06278, 0.16512], [1, 0.96507, 0.36328, 0.0387, 0.05328], [1, 0.84403, 0.366533, 0.08178, 0.067947], [1, 0.90003, 0.377893, 0.14226, 0.081707], [1, 0.98982, 0.3608, 0.02036, 0.048693]]}, {"filename": "000000550666.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.907281, 0.75385, 0.185437, 0.124507], [1, 0.724648, 0.690587, 0.062641, 0.06784], [1, 0.603438, 0.68939, 0.119813, 0.100563], [1, 0.669656, 0.766045, 0.11075, 0.115], [1, 0.773875, 0.760822, 0.110469, 0.117606], [1, 0.470242, 0.725622, 0.024578, 0.056549], [1, 0.496398, 0.713908, 0.030891, 0.064718], [1, 0.544289, 0.752124, 0.043922, 0.084296], [0, 0.297828, 0.716432, 0.021, 0.133944], [0, 0.365617, 0.724437, 0.023297, 0.09216], [0, 0.449188, 0.705998, 0.03025, 0.109601], [0, 0.338539, 0.693908, 0.020922, 0.036268], [0, 0.731719, 0.701608, 0.021125, 0.050915], [0, 0.263, 0.658721, 0.013, 0.036315], [0, 0.76968, 0.697124, 0.031797, 0.080962], [0, 0.78182, 0.691667, 0.017922, 0.066338], [1, 0.605773, 0.688732, 0.123141, 0.10385], [0, 0.278906, 0.667735, 0.023281, 0.055892], [0, 0.812477, 0.689624, 0.019422, 0.065869], [0, 0.392055, 0.695446, 0.023484, 0.097277], [0, 0.922211, 0.676796, 0.017953, 0.033545], [0, 0.427227, 0.705739, 0.022891, 0.115939], [1, 0.582547, 0.746185, 0.041281, 0.093592], [1, 0.665555, 0.725681, 0.031984, 0.037089], [1, 0.782492, 0.714941, 0.029703, 0.044624], [1, 0.40807, 0.717899, 0.026859, 0.054624]]}, {"filename": "000000550691.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.248039, 0.66951, 0.132859, 0.147979], [1, 0.710891, 0.612771, 0.036656, 0.034833], [1, 0.710523, 0.591229, 0.022266, 0.007417], [1, 0.495508, 0.489896, 0.391016, 0.732583], [1, 0.134828, 0.565167, 0.252812, 0.226958], [0, 0.774664, 0.602813, 0.014984, 0.049375], [0, 0.42007, 0.55876, 0.064641, 0.071687]]}, {"filename": "000000551350.jpg", "coco_split": "val2017", "split": "train", "boxes": [[3, 0.751688, 0.570948, 0.152031, 0.184688], [3, 0.388961, 0.580146, 0.216359, 0.121958], [3, 0.796789, 0.172063, 0.088641, 0.007458], [3, 0.933953, 0.190896, 0.037312, 0.010167]]}, {"filename": "000000551823.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.433052, 0.737375, 0.214771, 0.351937], [0, 0.746104, 0.343273, 0.102, 0.199828], [1, 0.50274, 0.291727, 0.037354, 0.070203], [1, 0.500188, 0.283047, 0.029167, 0.04975]]}, {"filename": "000000552089.jpg", "coco_split": "train2017", "split": "train", "boxes": [[5, 0.499711, 0.903302, 0.180828, 0.193397]]}, {"filename": "000000553373.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.193648, 0.213604, 0.234672, 0.147583], [0, 0.361516, 0.55999, 0.281469, 0.847187], [0, 0.607586, 0.547187, 0.197203, 0.79775], [0, 0.902211, 0.537031, 0.195578, 0.900229], [0, 0.770922, 0.172875, 0.017656, 0.082667], [0, 0.746, 0.176958, 0.041844, 0.083875], [0, 0.793461, 0.178073, 0.024609, 0.078896]]}, {"filename": "000000553511.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.23946, 0.870255, 0.28412, 0.177748], [1, 0.41515, 0.866486, 0.09494, 0.113333], [1, 0.70952, 0.844535, 0.12024, 0.092132], [5, 0.81388, 0.944685, 0.03792, 0.110631], [0, 0.89962, 0.832688, 0.0404, 0.108438], [0, 0.21096, 0.831336, 0.03348, 0.036126], [1, 0.6534, 0.7953, 0.01888, 0.025315], [1, 0.95404, 0.782988, 0.09192, 0.033423]]}, {"filename": "000000553776.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.58893, 0.697761, 0.594078, 0.577526], [0, 0.502227, 0.597556, 0.319453, 0.637526]]}, {"filename": "000000554006.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.890083, 0.325844, 0.219835, 0.116844], [0, 0.543491, 0.454484, 0.385991, 0.752031], [0, 0.103302, 0.470367, 0.143585, 0.214359], [1, 0.8475, 0.296492, 0.302972, 0.103922], [1, 0.230142, 0.364344, 0.460283, 0.28475], [1, 0.241568, 0.198063, 0.460684, 0.114156], [1, 0.975767, 0.343586, 0.048467, 0.070016], [1, 0.200755, 0.253937, 0.386651, 0.177344], [1, 0.726403, 0.273062, 0.054976, 0.028063]]}, {"filename": "000000554106.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.32796, 0.796633, 0.529561, 0.384266], [4, 0.071098, 0.710391, 0.142197, 0.397], [4, 0.158609, 0.538141, 0.23818, 0.172125], [4, 0.261412, 0.446969, 0.127301, 0.061437], [4, 0.527594, 0.376852, 0.122385, 0.061578], [4, 0.621245, 0.3705, 0.10136, 0.071094], [4, 0.818556, 0.837359, 0.362887, 0.325281], [4, 0.454571, 0.404375, 0.139603, 0.034375]]}, {"filename": "000000554636.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.896626, 0.473333, 0.206748, 0.693791], [0, 0.058701, 0.636544, 0.116356, 0.226127], [0, 0.211193, 0.616536, 0.038007, 0.128562], [0, 0.270221, 0.621961, 0.03701, 0.116601], [0, 0.13576, 0.574853, 0.031127, 0.040294], [1, 0.563693, 0.593824, 0.493693, 0.237222], [1, 0.095261, 0.605065, 0.190523, 0.245098], [0, 0.120613, 0.555384, 0.024297, 0.057598], [0, 0.149028, 0.554248, 0.027206, 0.086176], [0, 0.315229, 0.536078, 0.029935, 0.034314], [0, 0.786985, 0.587198, 0.028317, 0.044265], [1, 0.095204, 0.598088, 0.190408, 0.237059]]}, {"filename": "000000555050.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.69804, 0.672857, 0.03752, 0.096584], [4, 0.73552, 0.671289, 0.03804, 0.094814], [4, 0.65924, 0.670978, 0.04708, 0.094627], [4, 0.9709, 0.682919, 0.0582, 0.105093], [4, 0.707, 0.582422, 0.03448, 0.098882], [1, 0.23589, 0.693261, 0.4689, 0.285404], [1, 0.10499, 0.49264, 0.04726, 0.072112], [0, 0.22286, 0.619845, 0.05732, 0.086087], [4, 0.63456, 0.661941, 0.0162, 0.09854]]}, {"filename": "000000555534.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.524539, 0.543052, 0.221141, 0.496229], [0, 0.718234, 0.419875, 0.039156, 0.153917], [0, 0.738883, 0.396927, 0.034734, 0.145187], [0, 0.175164, 0.439052, 0.036328, 0.092896], [0, 0.165172, 0.44251, 0.046063, 0.062104], [0, 0.583023, 0.385823, 0.024016, 0.038188], [3, 0.781977, 0.413469, 0.067328, 0.044479], [4, 0.833547, 0.416698, 0.029156, 0.045854]]}, {"filename": "000000555597.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.361539, 0.773346, 0.243922, 0.154778], [0, 0.353336, 0.76293, 0.019297, 0.051509], [1, 0.832984, 0.882263, 0.307281, 0.185803], [1, 0.548789, 0.785948, 0.141266, 0.074874], [1, 0.489047, 0.903723, 0.401031, 0.168182], [1, 0.170867, 0.796132, 0.159547, 0.078375], [1, 0.882109, 0.765658, 0.167719, 0.056925], [7, 0.692508, 0.879323, 0.014234, 0.045648], [1, 0.681203, 0.779023, 0.148219, 0.070116], [1, 0.564039, 0.739671, 0.078328, 0.025106], [1, 0.038812, 0.806489, 0.056656, 0.064932], [1, 0.935852, 0.727369, 0.069672, 0.029478], [1, 0.151375, 0.73589, 0.042938, 0.020174], [1, 0.280328, 0.735116, 0.050062, 0.022418], [1, 0.673273, 0.730145, 0.033203, 0.014178], [7, 0.271031, 0.926373, 0.084313, 0.119594]]}, {"filename": "000000556064.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.871969, 0.437547, 0.249188, 0.252156], [4, 0.187615, 0.497438, 0.130063, 0.160156], [0, 0.521021, 0.469594, 0.801792, 0.76125], [0, 0.101875, 0.55393, 0.20375, 0.393266], [0, 0.063167, 0.252672, 0.126333, 0.172906], [4, 0.748167, 0.370148, 0.129917, 0.072391], [4, 0.973833, 0.321695, 0.052333, 0.040203], [0, 0.267781, 0.282055, 0.166521, 0.219109]]}, {"filename": "000000556246.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.408258, 0.450237, 0.115203, 0.294911], [0, 0.603062, 0.538299, 0.121312, 0.376124], [0, 0.692867, 0.597973, 0.162953, 0.481982], [0, 0.268852, 0.352618, 0.097016, 0.215414], [0, 0.516547, 0.755873, 0.174906, 0.488254], [1, 0.200727, 0.185843, 0.034641, 0.027071], [1, 0.161258, 0.188018, 0.041953, 0.033728], [1, 0.026234, 0.203462, 0.0385, 0.036864], [1, 0.549195, 0.156109, 0.058797, 0.051331], [1, 0.134484, 0.185414, 0.016375, 0.037988], [1, 0.081602, 0.193284, 0.016766, 0.027811], [1, 0.413953, 0.19003, 0.024625, 0.025858], [5, 0.581586, 0.192707, 0.012234, 0.033462], [1, 0.676406, 0.133669, 0.021094, 0.017396], [1, 0.317563, 0.173831, 0.038656, 0.026183], [1, 0.516945, 0.162145, 0.036766, 0.047308], [1, 0.238617, 0.176109, 0.012891, 0.029083], [1, 0.103805, 0.197426, 0.046703, 0.031538]]}, {"filename": "000000557451.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.684273, 0.49775, 0.631453, 0.9955], [1, 0.576164, 0.5, 0.847672, 1.0]]}, {"filename": "000000559160.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.247634, 0.440227, 0.080769, 0.080047], [0, 0.115455, 0.404336, 0.107972, 0.089016], [0, 0.300536, 0.41182, 0.113986, 0.162766], [3, 0.122483, 0.43068, 0.192517, 0.053828], [0, 0.519604, 0.474945, 0.072168, 0.126016], [0, 0.636305, 0.599234, 0.41, 0.63875], [0, 0.574662, 0.362867, 0.172401, 0.129797]]}, {"filename": "000000559482.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.766364, 0.391087, 0.43567, 0.778775], [0, 0.147647, 0.365989, 0.255033, 0.257271]]}, {"filename": "000000559884.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.491367, 0.27865, 0.297672, 0.462934], [0, 0.674617, 0.611491, 0.365797, 0.772512], [1, 0.307742, 0.506491, 0.546328, 0.955845]]}, {"filename": "000000559995.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.308285, 0.650891, 0.058273, 0.054125], [4, 0.391472, 0.653516, 0.047859, 0.058125], [4, 0.430097, 0.646477, 0.03528, 0.060422], [4, 0.462165, 0.627344, 0.045499, 0.048688], [4, 0.328114, 0.642656, 0.04528, 0.047969], [4, 0.347981, 0.629031, 0.02292, 0.026844], [1, 0.905681, 0.621313, 0.176715, 0.088031], [0, 0.506423, 0.671586, 0.067591, 0.112422], [0, 0.517871, 0.601656, 0.069124, 0.096312], [0, 0.577555, 0.628586, 0.056375, 0.103578], [1, 0.136788, 0.698188, 0.263504, 0.155531], [1, 0.910876, 0.567562, 0.03382, 0.013563], [1, 0.859672, 0.59882, 0.094574, 0.046797], [5, 0.878066, 0.955133, 0.169027, 0.089734], [1, 0.970961, 0.56868, 0.033017, 0.025078], [4, 0.620304, 0.640977, 0.038321, 0.051672], [4, 0.420693, 0.625922, 0.034039, 0.017344], [4, 0.438333, 0.616836, 0.03253, 0.007703], [1, 0.83719, 0.567797, 0.052092, 0.023625], [4, 0.38427, 0.623383, 0.031314, 0.005641], [4, 0.305122, 0.648203, 0.036667, 0.05775]]}, {"filename": "000000560513.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.950313, 0.824055, 0.097833, 0.248453], [5, 0.541198, 0.767, 0.113313, 0.211219], [0, 0.687406, 0.467703, 0.258313, 0.395625], [0, 0.494385, 0.548312, 0.185771, 0.431469], [0, 0.350937, 0.565438, 0.183833, 0.427344], [0, 0.134479, 0.29732, 0.2055, 0.270203], [0, 0.337948, 0.250578, 0.231687, 0.356687], [0, 0.477927, 0.262602, 0.260771, 0.372109], [0, 0.630531, 0.16618, 0.148521, 0.246766], [0, 0.771646, 0.162797, 0.044625, 0.091719]]}, {"filename": "000000560880.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.398031, 0.652913, 0.041781, 0.181063]]}, {"filename": "000000562558.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.45102, 0.304186, 0.90204, 0.604942]]}, {"filename": "000000563070.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.462531, 0.718724, 0.583219, 0.382037], [0, 0.286492, 0.821792, 0.047016, 0.228735], [0, 0.565445, 0.789988, 0.040453, 0.217166], [3, 0.943016, 0.753829, 0.089531, 0.066628], [0, 0.453266, 0.66007, 0.02625, 0.039344], [0, 0.355602, 0.656593, 0.017047, 0.02719], [0, 0.893688, 0.715867, 0.024156, 0.127845], [0, 0.274695, 0.64055, 0.020641, 0.050843], [0, 0.247992, 0.638899, 0.013016, 0.045667], [0, 0.671852, 0.655632, 0.046984, 0.071405], [0, 0.387445, 0.656569, 0.025328, 0.036885]]}, {"filename": "000000563281.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.375072, 0.49605, 0.288507, 0.98418]]}, {"filename": "000000563364.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.504227, 0.414611, 0.123109, 0.550563], [0, 0.176383, 0.399357, 0.218141, 0.584718], [0, 0.275719, 0.39063, 0.105344, 0.608981], [0, 0.363602, 0.394142, 0.126016, 0.592359], [0, 0.538812, 0.158123, 0.076375, 0.209866], [0, 0.592008, 0.373928, 0.089703, 0.612735], [0, 0.6405, 0.371756, 0.096437, 0.625121], [0, 0.700031, 0.373043, 0.127031, 0.615737], [0, 0.850039, 0.362105, 0.137203, 0.616595], [0, 0.784305, 0.341635, 0.075766, 0.659357], [0, 0.096273, 0.429772, 0.145328, 0.531609], [0, 0.432719, 0.388512, 0.081688, 0.604209], [4, 0.972602, 0.401354, 0.054797, 0.153378]]}, {"filename": "000000563470.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.236469, 0.48904, 0.202094, 0.535785], [4, 0.034125, 0.038864, 0.062781, 0.038946], [4, 0.053469, 0.131206, 0.074156, 0.210562]]}, {"filename": "000000564336.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.250844, 0.753375, 0.293906, 0.49325], [4, 0.447305, 0.572986, 0.142297, 0.137472], [4, 0.339766, 0.589694, 0.0755, 0.158833], [4, 0.629016, 0.563069, 0.079812, 0.101361], [0, 0.232586, 0.404653, 0.133984, 0.251694], [1, 0.539594, 0.520125, 0.219906, 0.173917], [1, 0.704047, 0.509153, 0.120969, 0.071472], [1, 0.945945, 0.439181, 0.108109, 0.168917], [4, 0.048914, 0.589694, 0.097453, 0.277611], [1, 0.95443, 0.492278, 0.025547, 0.019278], [7, 0.682594, 0.478569, 0.032375, 0.120583]]}, {"filename": "000000564589.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.590766, 0.466698, 0.080219, 0.041521], [1, 0.955688, 0.42774, 0.036719, 0.036521], [0, 0.728094, 0.578365, 0.053125, 0.217396], [0, 0.566086, 0.58599, 0.067203, 0.136979], [0, 0.390164, 0.557302, 0.062359, 0.161812], [0, 0.11075, 0.575729, 0.052656, 0.195375], [0, 0.021273, 0.58675, 0.042297, 0.194333], [0, 0.523969, 0.582073, 0.056594, 0.141771], [0, 0.664469, 0.560365, 0.057125, 0.212354], [0, 0.761797, 0.567979, 0.057313, 0.1955], [0, 0.470656, 0.596052, 0.044375, 0.114104], [1, 0.836695, 0.423844, 0.032922, 0.032521], [3, 0.564141, 0.547812, 0.195188, 0.113333], [1, 0.892039, 0.428198, 0.061484, 0.040521], [1, 0.794383, 0.422865, 0.020953, 0.019229], [0, 0.621883, 0.585562, 0.050203, 0.136042]]}, {"filename": "000000565012.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.254117, 0.475776, 0.030234, 0.031082], [1, 0.875578, 0.505247, 0.071094, 0.084], [1, 0.253211, 0.476506, 0.031016, 0.032024], [1, 0.623445, 0.4194, 0.010172, 0.013059], [1, 0.552344, 0.429118, 0.022906, 0.019788], [7, 0.06057, 0.523659, 0.006391, 0.019647]]}, {"filename": "000000565153.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.365594, 0.38458, 0.150976, 0.11248], [7, 0.480185, 0.67563, 0.097625, 0.13302]]}, {"filename": "000000565424.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.416359, 0.033291, 0.052, 0.064883], [4, 0.806219, 0.715732, 0.127344, 0.294395], [4, 0.247844, 0.522261, 0.186625, 0.39586], [4, 0.168711, 0.317633, 0.100797, 0.197558], [0, 0.592062, 0.510106, 0.377062, 0.629214], [0, 0.19807, 0.327707, 0.270172, 0.569809], [0, 0.036297, 0.309416, 0.066, 0.269066], [0, 0.095961, 0.519947, 0.191922, 0.445372], [4, 0.069281, 0.490064, 0.071688, 0.084501], [0, 0.071188, 0.207346, 0.142375, 0.40327]]}, {"filename": "000000565607.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.830156, 0.457476, 0.266031, 0.914952]]}, {"filename": "000000566524.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.770148, 0.476615, 0.007359, 0.028979], [0, 0.800297, 0.518021, 0.02125, 0.11075], [0, 0.777102, 0.475708, 0.007328, 0.021417], [0, 0.784602, 0.47626, 0.005547, 0.020146], [0, 0.569664, 0.398438, 0.020859, 0.030125], [0, 0.869953, 0.50826, 0.040656, 0.126771]]}, {"filename": "000000566536.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.592177, 0.493828, 0.815646, 0.987656], [4, 0.5, 0.559367, 1.0, 0.880922]]}, {"filename": "000000566923.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.285695, 0.758052, 0.332078, 0.460657], [0, 0.551195, 0.512359, 0.680578, 0.948333], [0, 0.279719, 0.353932, 0.233344, 0.69439], [0, 0.433781, 0.300563, 0.1765, 0.537089], [0, 0.848766, 0.332594, 0.128063, 0.386925], [0, 0.937516, 0.582031, 0.124969, 0.808991], [0, 0.64543, 0.519108, 0.339547, 0.934836], [0, 0.174617, 0.234683, 0.125359, 0.457394], [0, 0.141523, 0.156174, 0.102578, 0.308216], [0, 0.025805, 0.103439, 0.050141, 0.205423], [0, 0.063578, 0.138556, 0.064875, 0.245986]]}, {"filename": "000000567037.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.610906, 0.552476, 0.429219, 0.440189], [1, 0.337047, 0.415861, 0.133406, 0.147335], [1, 0.667039, 0.325295, 0.095766, 0.067099], [1, 0.533891, 0.323561, 0.050875, 0.023962], [1, 0.478898, 0.32625, 0.046484, 0.038255], [0, 0.848039, 0.317913, 0.020984, 0.082854], [0, 0.876531, 0.319351, 0.022219, 0.081958], [0, 0.893711, 0.322005, 0.016016, 0.078538], [1, 0.449766, 0.325059, 0.033469, 0.031722], [1, 0.386531, 0.32605, 0.023438, 0.022429], [1, 0.37018, 0.324517, 0.022141, 0.021627], [1, 0.301906, 0.325047, 0.021125, 0.023396], [1, 0.425227, 0.321309, 0.045516, 0.041675], [7, 0.422063, 0.432925, 0.047313, 0.162925], [1, 0.223211, 0.328278, 0.018297, 0.019245], [1, 0.589367, 0.316887, 0.067109, 0.041557], [0, 0.824961, 0.320248, 0.016766, 0.072854]]}, {"filename": "000000567205.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.289891, 0.500563, 0.576406, 0.667417], [1, 0.681742, 0.260677, 0.443266, 0.211229], [7, 0.505617, 0.494542, 0.084266, 0.4], [0, 0.634555, 0.524719, 0.436516, 0.928104]]}, {"filename": "000000567837.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.501125, 0.496854, 0.524656, 0.618833]]}, {"filename": "000000568115.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.315648, 0.687948, 0.045672, 0.039854], [1, 0.437516, 0.698583, 0.034594, 0.034167], [1, 0.513633, 0.704573, 0.038516, 0.045354], [1, 0.4965, 0.676927, 0.013187, 0.012562], [1, 0.524313, 0.683781, 0.024469, 0.021937], [1, 0.944305, 0.670937, 0.111391, 0.084833]]}, {"filename": "000000568147.jpg", "coco_split": "val2017", "split": "train", "boxes": [[7, 0.52526, 0.363508, 0.434896, 0.638516], [0, 0.038042, 0.077852, 0.076083, 0.056609], [0, 0.831719, 0.027398, 0.064896, 0.054078], [1, 0.889542, 0.0195, 0.220917, 0.039], [1, 0.691729, 0.022172, 0.064708, 0.044281], [0, 0.511948, 0.022219, 0.032146, 0.043094], [0, 0.483583, 0.02275, 0.027667, 0.0455]]}, {"filename": "000000568647.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.870352, 0.152521, 0.138641, 0.185833], [1, 0.454219, 0.684271, 0.672469, 0.460667], [5, 0.811781, 0.648031, 0.0815, 0.193979], [1, 0.468586, 0.40899, 0.107609, 0.055562], [1, 0.680758, 0.409917, 0.078359, 0.070125], [6, 0.384234, 0.331073, 0.017656, 0.029062], [1, 0.557008, 0.398677, 0.081172, 0.049396], [1, 0.176719, 0.429073, 0.112844, 0.054104], [1, 0.968086, 0.441969, 0.063828, 0.080604], [1, 0.132703, 0.415354, 0.040969, 0.044167], [1, 0.086437, 0.425208, 0.059469, 0.068042]]}, {"filename": "000000568710.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.891867, 0.836475, 0.216266, 0.327049], [4, 0.726164, 0.426803, 0.043328, 0.053981]]}, {"filename": "000000569030.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.90421, 0.363373, 0.10338, 0.099547], [5, 0.63652, 0.6666, 0.13792, 0.379493], [0, 0.14462, 0.259987, 0.16816, 0.201787], [0, 0.67703, 0.321813, 0.03042, 0.108427], [0, 0.53139, 0.322253, 0.0249, 0.12856], [0, 0.77171, 0.323747, 0.03518, 0.13416], [3, 0.17275, 0.651693, 0.3455, 0.42248], [3, 0.16509, 0.398813, 0.28178, 0.086693], [0, 0.51048, 0.315547, 0.03352, 0.126453], [0, 0.60246, 0.334027, 0.01728, 0.109387], [1, 0.11751, 0.27748, 0.23254, 0.179973], [1, 0.95855, 0.46452, 0.0829, 0.31896], [0, 0.67227, 0.265453, 0.00954, 0.032293], [0, 0.72226, 0.311787, 0.02044, 0.093493], [0, 0.95855, 0.277827, 0.02966, 0.059173], [0, 0.44732, 0.318893, 0.03044, 0.096773], [0, 0.49042, 0.293227, 0.01928, 0.053547], [1, 0.2691, 0.31592, 0.07352, 0.04704], [0, 0.57783, 0.33196, 0.0271, 0.1092]]}, {"filename": "000000569059.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.802578, 0.911917, 0.394844, 0.150208], [0, 0.313477, 0.3955, 0.020859, 0.046333], [0, 0.325688, 0.4145, 0.02325, 0.059708], [0, 0.340242, 0.413344, 0.023297, 0.059146], [0, 0.352297, 0.402573, 0.017406, 0.071312], [0, 0.361695, 0.383917, 0.013141, 0.022958], [0, 0.296977, 0.407104, 0.028484, 0.070833], [0, 0.636773, 0.344812, 0.033641, 0.056333], [0, 0.3365, 0.387104, 0.0125, 0.033542], [0, 0.365477, 0.398552, 0.022766, 0.079938]]}, {"filename": "000000569272.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.06425, 0.653766, 0.125281, 0.335481], [0, 0.170422, 0.217197, 0.269781, 0.434393]]}, {"filename": "000000569273.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.764289, 0.610125, 0.264016, 0.757833], [0, 0.373703, 0.258573, 0.084125, 0.157937], [0, 0.279172, 0.221542, 0.077031, 0.113417], [0, 0.105508, 0.282469, 0.044484, 0.053937], [0, 0.038359, 0.265104, 0.035344, 0.042917], [0, 0.076805, 0.257427, 0.046672, 0.096313]]}, {"filename": "000000570471.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.173227, 0.27527, 0.174773, 0.14574], [0, 0.564413, 0.49438, 0.871173, 0.98876], [0, 0.765173, 0.34157, 0.469653, 0.33258]]}, {"filename": "000000570539.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.584462, 0.4427, 0.419839, 0.36404], [1, 0.496142, 0.50133, 0.992285, 0.99734], [0, 0.098038, 0.3676, 0.19371, 0.32352], [0, 0.19203, 0.39415, 0.180349, 0.18802], [0, 0.303925, 0.28707, 0.031398, 0.03766], [0, 0.333683, 0.29249, 0.028011, 0.02842], [0, 0.375995, 0.28752, 0.031183, 0.03664]]}, {"filename": "000000570588.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.382781, 0.514274, 0.023, 0.032857], [0, 0.404258, 0.509532, 0.013922, 0.043466], [3, 0.942664, 0.734426, 0.072922, 0.08281], [0, 0.600461, 0.510129, 0.010078, 0.042037], [0, 0.576086, 0.516991, 0.017516, 0.027799], [0, 0.570242, 0.509145, 0.008734, 0.041944], [0, 0.925125, 0.507389, 0.01275, 0.046253], [0, 0.127727, 0.773642, 0.156266, 0.432436]]}, {"filename": "000000571008.jpg", "coco_split": "val2017", "split": "train", "boxes": [[6, 0.499914, 0.494388, 0.540203, 0.777529]]}, {"filename": "000000571182.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.58957, 0.488289, 0.680266, 0.199547], [0, 0.874672, 0.585102, 0.051719, 0.142766], [0, 0.451977, 0.709047, 0.106703, 0.227219], [0, 0.13268, 0.602859, 0.056547, 0.109438], [1, 0.947133, 0.513352, 0.074516, 0.043016], [1, 0.97732, 0.512687, 0.045359, 0.042406], [0, 0.012375, 0.480211, 0.014656, 0.037859], [0, 0.200742, 0.484945, 0.012609, 0.044453], [5, 0.940219, 0.619531, 0.031469, 0.051844], [0, 0.103766, 0.468172, 0.009313, 0.023281]]}, {"filename": "000000572147.jpg", "coco_split": "train2017", "split": "train", "boxes": [[4, 0.354273, 0.093797, 0.176766, 0.186085], [4, 0.530133, 0.02921, 0.132859, 0.05842], [4, 0.769812, 0.049104, 0.198125, 0.096792], [4, 0.061508, 0.151226, 0.123016, 0.302028], [0, 0.211406, 0.498868, 0.404938, 0.975283], [0, 0.759203, 0.544835, 0.481594, 0.874434]]}, {"filename": "000000572252.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.744898, 0.547195, 0.460703, 0.874155], [0, 0.232594, 0.602254, 0.465188, 0.768545]]}, {"filename": "000000572283.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.530062, 0.809113, 0.113687, 0.230057], [7, 0.421531, 0.820782, 0.1055, 0.239084]]}, {"filename": "000000572303.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.015211, 0.640698, 0.030422, 0.116771]]}, {"filename": "000000572462.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.43049, 0.796969, 0.143497, 0.365474], [0, 0.632508, 0.849322, 0.154069, 0.267892], [0, 0.73357, 0.794542, 0.085016, 0.344216], [3, 0.910678, 0.879191, 0.134951, 0.203252], [0, 0.575515, 0.711038, 0.060114, 0.09835], [0, 0.601201, 0.690817, 0.019134, 0.040915], [0, 0.112745, 0.765466, 0.031503, 0.036618], [0, 0.628464, 0.696487, 0.043431, 0.068595], [0, 0.261789, 0.211446, 0.087435, 0.069951], [0, 0.906781, 0.77482, 0.067451, 0.27281]]}, {"filename": "000000572678.jpg", "coco_split": "val2017", "split": "train", "boxes": [[4, 0.884703, 0.813976, 0.135656, 0.337647], [4, 0.463414, 0.623659, 0.095672, 0.202141], [4, 0.595984, 0.518424, 0.128344, 0.113882], [4, 0.398438, 0.9, 0.226813, 0.177506], [4, 0.756398, 0.568294, 0.088672, 0.112353], [4, 0.348883, 0.655847, 0.191328, 0.231318]]}, {"filename": "000000573062.jpg", "coco_split": "train2017", "split": "train", "boxes": [[3, 0.544945, 0.739014, 0.239359, 0.139061], [3, 0.285813, 0.724401, 0.252781, 0.123873]]}, {"filename": "000000573485.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.897102, 0.552937, 0.014516, 0.017292], [1, 0.914266, 0.551656, 0.022813, 0.030271], [1, 0.557023, 0.413479, 0.537641, 0.768542], [0, 0.969172, 0.649479, 0.060594, 0.313583], [1, 0.931406, 0.557177, 0.004344, 0.010687], [1, 0.875281, 0.550875, 0.027531, 0.031583], [1, 0.94032, 0.555042, 0.014984, 0.013542], [1, 0.914172, 0.551979, 0.023344, 0.030125], [1, 0.835375, 0.556677, 0.029687, 0.034604], [1, 0.158336, 0.6, 0.316672, 0.345667]]}, {"filename": "000000574425.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.613477, 0.481055, 0.713484, 0.854668], [1, 0.186703, 0.457275, 0.155812, 0.231754]]}, {"filename": "000000575260.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.310953, 0.416854, 0.618531, 0.761792], [1, 0.972086, 0.888469, 0.030953, 0.040646], [1, 0.921203, 0.868104, 0.045344, 0.058167], [0, 0.266906, 0.412271, 0.055813, 0.093542], [0, 0.132719, 0.326792, 0.044063, 0.077917], [0, 0.469742, 0.522427, 0.069453, 0.085437]]}, {"filename": "000000577035.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.153961, 0.479927, 0.204766, 0.113146], [0, 0.789586, 0.528813, 0.137922, 0.297083], [0, 0.049609, 0.525781, 0.095844, 0.329604], [1, 0.262078, 0.415354, 0.466844, 0.161792]]}, {"filename": "000000577096.jpg", "coco_split": "train2017", "split": "train", "boxes": [[7, 0.528844, 0.49482, 0.476396, 0.860672], [1, 0.223729, 0.147523, 0.297292, 0.072078], [1, 0.633021, 0.388453, 0.733958, 0.762906]]}, {"filename": "000000577213.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.78782, 0.485105, 0.111359, 0.143138], [1, 0.948477, 0.500176, 0.041422, 0.043068], [1, 0.913352, 0.494157, 0.036484, 0.04555], [5, 0.278867, 0.640445, 0.188922, 0.525855], [1, 0.895375, 0.489192, 0.017594, 0.040726], [1, 0.686813, 0.500738, 0.052375, 0.038103], [1, 0.980234, 0.49466, 0.039531, 0.0563], [5, 0.716172, 0.506686, 0.009875, 0.033653]]}, {"filename": "000000578713.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.190445, 0.612792, 0.067422, 0.094375], [1, 0.155055, 0.470542, 0.060984, 0.029833], [1, 0.145219, 0.481823, 0.077937, 0.026604], [1, 0.249742, 0.4755, 0.039578, 0.023125], [1, 0.037937, 0.483458, 0.075406, 0.030792], [1, 0.434516, 0.466385, 0.027813, 0.025396], [1, 0.406156, 0.459229, 0.017156, 0.015958], [1, 0.447586, 0.462865, 0.009484, 0.018687]]}, {"filename": "000000578715.jpg", "coco_split": "train2017", "split": "train", "boxes": [[6, 0.527531, 0.535948, 0.370781, 0.487646]]}, {"filename": "000000579070.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.586172, 0.338888, 0.302125, 0.560211], [0, 0.063516, 0.416347, 0.127031, 0.411241], [0, 0.307641, 0.305691, 0.298406, 0.472974], [4, 0.759172, 0.477014, 0.0465, 0.189766], [0, 0.885797, 0.524977, 0.228406, 0.510703], [4, 0.188109, 0.180867, 0.139563, 0.3563], [4, 0.81093, 0.393326, 0.115484, 0.378735], [0, 0.083484, 0.10493, 0.163688, 0.209859], [0, 0.835328, 0.800785, 0.329344, 0.148454]]}, {"filename": "000000579145.jpg", "coco_split": "train2017", "split": "train", "boxes": [[0, 0.642778, 0.7, 0.691682, 0.56852], [0, 0.227748, 0.63446, 0.428529, 0.70336]]}, {"filename": "000000579232.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.246578, 0.817375, 0.069125, 0.147583], [1, 0.32057, 0.767319, 0.073797, 0.12225], [1, 0.65868, 0.600042, 0.169609, 0.202639], [1, 0.676633, 0.477458, 0.089547, 0.116583], [1, 0.621359, 0.284083, 0.031594, 0.039333], [1, 0.612766, 0.264208, 0.017687, 0.026972], [1, 0.608258, 0.257514, 0.021422, 0.033861], [1, 0.599703, 0.245639, 0.013344, 0.024556], [0, 0.55282, 0.516625, 0.021828, 0.106139], [0, 0.52743, 0.373208, 0.012734, 0.062361], [1, 0.521234, 0.439028, 0.04825, 0.059222], [1, 0.343391, 0.744611, 0.064469, 0.125889], [0, 0.860148, 0.321333, 0.012797, 0.059556], [1, 0.531297, 0.416708, 0.035562, 0.044139], [1, 0.593688, 0.230806, 0.015531, 0.020556], [1, 0.661477, 0.234236, 0.057547, 0.103806], [0, 0.547453, 0.379042, 0.009438, 0.041306], [0, 0.635109, 0.239778, 0.011188, 0.030944], [6, 0.833336, 0.435097, 0.014984, 0.048861]]}, {"filename": "000000580294.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.821203, 0.497901, 0.357594, 0.982032]]}, {"filename": "000000580418.jpg", "coco_split": "val2017", "split": "train", "boxes": [[1, 0.966984, 0.71736, 0.063469, 0.256262], [1, 0.930086, 0.692477, 0.097047, 0.222944], [1, 0.314977, 0.547442, 0.149328, 0.127126], [6, 0.95368, 0.518972, 0.022547, 0.043271], [0, 0.755711, 0.54535, 0.021766, 0.032757], [6, 0.182938, 0.350654, 0.026656, 0.089579]]}, {"filename": "000000580757.jpg", "coco_split": "val2017", "split": "train", "boxes": [[5, 0.706594, 0.572341, 0.362625, 0.813482]]}, {"filename": "000000581081.jpg", "coco_split": "train2017", "split": "train", "boxes": [[1, 0.23643, 0.481005, 0.046141, 0.040024], [1, 0.197, 0.482683, 0.047969, 0.036288], [0, 0.499008, 0.486726, 0.008922, 0.050757], [0, 0.471023, 0.486099, 0.033203, 0.061797], [1, 0.083484, 0.465296, 0.166969, 0.200709], [1, 0.365313, 0.450591, 0.210594, 0.129125], [0, 0.392188, 0.485603, 0.013687, 0.050024], [0, 0.6125, 0.478073, 0.010969, 0.055816], [0, 0.632555, 0.463286, 0.012234, 0.028038], [0, 0.988742, 0.448239, 0.010078, 0.041537], [0, 0.447203, 0.494054, 0.016156, 0.033452], [1, 0.852219, 0.456087, 0.084844, 0.045035]]}, {"filename": "000000581357.jpg", "coco_split": "val2017", "split": "train", "boxes": [[0, 0.608987, 0.354681, 0.359542, 0.404493], [0, 0.719387, 0.691062, 0.037075, 0.07415], [0, 0.813105, 0.692525, 0.032876, 0.038088], [0, 0.865956, 0.690507, 0.020801, 0.060458], [0, 0.922998, 0.677377, 0.035114, 0.085539], [0, 0.95616, 0.656642, 0.021013, 0.041487], [3, 0.338415, 0.741422, 0.046275, 0.030261], [0, 0.31281, 0.751127, 0.034477, 0.06268], [0, 0.418007, 0.705743, 0.021569, 0.040082], [0, 0.364167, 0.719404, 0.038203, 0.065408], [0, 0.391814, 0.718194, 0.031078, 0.061029], [3, 0.48732, 0.734216, 0.172288, 0.029216], [0, 0.055858, 0.489902, 0.047892, 0.125196]]}, {"filename": "000000004988.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.890867, 0.523146, 0.022609, 0.0145], [1, 0.380898, 0.550562, 0.519109, 0.498875], [0, 0.96532, 0.526729, 0.008641, 0.034542], [0, 0.941875, 0.523031, 0.005562, 0.019937], [1, 0.861102, 0.530396, 0.023391, 0.027542], [3, 0.802555, 0.688042, 0.202422, 0.168208], [0, 0.255102, 0.518719, 0.048922, 0.076813], [0, 0.392555, 0.611687, 0.032078, 0.276333], [0, 0.574055, 0.525823, 0.015266, 0.056604], [0, 0.600391, 0.514417, 0.017094, 0.029333], [0, 0.97557, 0.5235, 0.012078, 0.041958], [0, 0.990047, 0.519542, 0.009719, 0.041667], [1, 0.063703, 0.600708, 0.127406, 0.222333]]}, {"filename": "000000007278.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.391727, 0.221691, 0.190391, 0.196867]]}, {"filename": "000000008211.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.307297, 0.514575, 0.173063, 0.325664], [4, 0.54693, 0.648878, 0.086109, 0.204641], [4, 0.672656, 0.678791, 0.057531, 0.126645], [0, 0.695969, 0.573878, 0.104781, 0.356993], [0, 0.575359, 0.606743, 0.132156, 0.328083], [1, 0.51775, 0.487669, 0.072063, 0.14732]]}, {"filename": "000000015687.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.71329, 0.497514, 0.34266, 0.947486], [0, 0.40064, 0.386508, 0.3572, 0.728101], [4, 0.48002, 0.391466, 0.17808, 0.197849]]}, {"filename": "000000019109.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.087359, 0.761262, 0.1235, 0.326589], [1, 0.177148, 0.746308, 0.132078, 0.308318], [1, 0.261258, 0.739907, 0.076172, 0.277196], [1, 0.395852, 0.744614, 0.209922, 0.221986], [1, 0.535336, 0.707921, 0.093547, 0.240421], [1, 0.598461, 0.70722, 0.067578, 0.214533], [1, 0.72, 0.690853, 0.093344, 0.199696], [1, 0.656344, 0.693411, 0.129594, 0.222804], [0, 0.025258, 0.56222, 0.037516, 0.098084], [0, 0.095031, 0.592442, 0.051031, 0.137921], [0, 0.560914, 0.58215, 0.036547, 0.092336], [0, 0.461414, 0.563551, 0.028859, 0.133318], [0, 0.704484, 0.593423, 0.016344, 0.056472], [0, 0.886062, 0.573411, 0.045906, 0.082477], [0, 0.85575, 0.561495, 0.021656, 0.03285], [0, 0.94368, 0.578902, 0.023297, 0.121636], [0, 0.057172, 0.570315, 0.034969, 0.096145], [0, 0.591172, 0.594498, 0.036844, 0.126799], [0, 0.014063, 0.710315, 0.028125, 0.321332], [0, 0.748172, 0.581612, 0.011219, 0.009065], [1, 0.915328, 0.671986, 0.082938, 0.175561], [1, 0.947102, 0.681706, 0.044078, 0.106355], [1, 0.968633, 0.651916, 0.061016, 0.084813], [1, 0.487742, 0.701495, 0.088641, 0.231168], [0, 0.107305, 0.475935, 0.041078, 0.13486], [1, 0.769141, 0.687593, 0.109469, 0.199112]]}, {"filename": "000000030828.jpg", "coco_split": "val2017", "split": "val", "boxes": [[7, 0.406305, 0.275714, 0.107953, 0.197752], [0, 0.592055, 0.498876, 0.616109, 0.24548], [3, 0.513914, 0.579953, 0.847516, 0.425667], [1, 0.01625, 0.473021, 0.031438, 0.127119]]}, {"filename": "000000035628.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.618945, 0.259543, 0.300904, 0.339329], [1, 0.518096, 0.87878, 0.123753, 0.099207]]}, {"filename": "000000036844.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.277422, 0.674188, 0.183875, 0.333333], [4, 0.102766, 0.686167, 0.190594, 0.335583]]}, {"filename": "000000046816.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.80321, 0.6515, 0.30622, 0.61885], [3, 0.45804, 0.700913, 0.73036, 0.433025]]}, {"filename": "000000047451.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.778516, 0.84065, 0.442968, 0.3187], [1, 0.185909, 0.84709, 0.371818, 0.30062], [7, 0.657888, 0.79967, 0.18107, 0.37246], [7, 0.40381, 0.80311, 0.190508, 0.38746]]}, {"filename": "000000048396.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.518688, 0.739789, 0.064562, 0.468806], [0, 0.37707, 0.494379, 0.364328, 0.984262], [0, 0.786992, 0.50596, 0.306672, 0.772201]]}, {"filename": "000000048747.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.653523, 0.237354, 0.343953, 0.281875], [0, 0.391555, 0.441844, 0.330047, 0.188146], [3, 0.336375, 0.545323, 0.516531, 0.415354]]}, {"filename": "000000049810.jpg", "coco_split": "val2017", "split": "val", "boxes": [[3, 0.64357, 0.572724, 0.712391, 0.823373]]}, {"filename": "000000054593.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.866219, 0.478056, 0.101281, 0.144075], [4, 0.724031, 0.471768, 0.114094, 0.147611], [4, 0.540438, 0.433852, 0.038063, 0.119274], [0, 0.628953, 0.521347, 0.178406, 0.548314], [0, 0.230367, 0.591464, 0.198516, 0.387892], [0, 0.864398, 0.414379, 0.101672, 0.303185], [0, 0.090086, 0.550492, 0.170922, 0.443091], [0, 0.721281, 0.402026, 0.111188, 0.286019], [1, 0.328898, 0.302178, 0.311359, 0.252927], [1, 0.803195, 0.278068, 0.305828, 0.206019], [0, 0.576922, 0.4337, 0.072656, 0.254098], [1, 0.050234, 0.272623, 0.100469, 0.274192], [1, 0.055945, 0.37685, 0.111891, 0.19192]]}, {"filename": "000000055618.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.725555, 0.339885, 0.329484, 0.236063], [1, 0.384227, 0.253271, 0.206234, 0.122583], [1, 0.060883, 0.242802, 0.052766, 0.061937], [1, 0.6045, 0.238667, 0.105063, 0.107292], [1, 0.309523, 0.198313, 0.061047, 0.033792], [1, 0.522617, 0.177406, 0.134672, 0.082854], [7, 0.377445, 0.579094, 0.116672, 0.342896], [0, 0.530898, 0.59401, 0.215734, 0.788771], [0, 0.752789, 0.557385, 0.318016, 0.869396], [0, 0.247758, 0.570104, 0.259547, 0.831458], [7, 0.103547, 0.403146, 0.03975, 0.264208], [7, 0.050773, 0.275771, 0.017891, 0.038708], [7, 0.029375, 0.259792, 0.0115, 0.030667], [7, 0.018164, 0.250781, 0.009766, 0.023937], [1, 0.314602, 0.220937, 0.078453, 0.050458]]}, {"filename": "000000056127.jpg", "coco_split": "val2017", "split": "val", "boxes": [[3, 0.785922, 0.774734, 0.428155, 0.265625], [3, 0.918457, 0.944266, 0.163086, 0.111469]]}, {"filename": "000000062276.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.700375, 0.394955, 0.062188, 0.04435], [1, 0.484945, 0.374339, 0.170953, 0.095942], [1, 0.133258, 0.391872, 0.151109, 0.071682], [0, 0.527883, 0.381043, 0.033578, 0.091682], [0, 0.397016, 0.380605, 0.030062, 0.086278], [0, 0.356563, 0.37741, 0.036625, 0.091816], [0, 0.391883, 0.343195, 0.015516, 0.03213], [0, 0.702094, 0.375336, 0.020344, 0.086233], [1, 0.483805, 0.37176, 0.171422, 0.096883], [3, 0.649898, 0.491592, 0.088953, 0.056816], [0, 0.079992, 0.380045, 0.020047, 0.052332]]}, {"filename": "000000064086.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.488172, 0.491344, 0.621625, 0.813771], [1, 0.090719, 0.34776, 0.12, 0.054146]]}, {"filename": "000000064902.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.883734, 0.540609, 0.040906, 0.131007], [0, 0.807984, 0.456393, 0.128844, 0.506838], [0, 0.94457, 0.494052, 0.110859, 0.4674]]}, {"filename": "000000067593.jpg", "coco_split": "train2017", "split": "val", "boxes": [[7, 0.082163, 0.876523, 0.029109, 0.043297], [7, 0.109543, 0.871008, 0.024826, 0.030766]]}, {"filename": "000000068198.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.655281, 0.529134, 0.114219, 0.2282], [1, 0.864188, 0.680991, 0.246906, 0.479112]]}, {"filename": "000000073790.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.54775, 0.452813, 0.9, 0.905625]]}, {"filename": "000000074088.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.578797, 0.836297, 0.309844, 0.327407], [0, 0.520703, 0.315724, 0.105219, 0.19229]]}, {"filename": "000000076547.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.153703, 0.743125, 0.307406, 0.508583], [0, 0.782992, 0.6955, 0.268766, 0.538625], [3, 0.644656, 0.799625, 0.574719, 0.364042], [3, 0.796953, 0.92925, 0.188031, 0.1415], [4, 0.842773, 0.802094, 0.134766, 0.075521], [4, 0.579922, 0.780323, 0.132094, 0.044729], [4, 0.869938, 0.806073, 0.205344, 0.172354]]}, {"filename": "000000076962.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.606747, 0.53302, 0.716107, 0.90112]]}, {"filename": "000000077396.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.288258, 0.596833, 0.175234, 0.272125]]}, {"filename": "000000079408.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.354633, 0.484227, 0.586047, 0.934684]]}, {"filename": "000000082846.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.50407, 0.676795, 0.746109, 0.599097], [6, 0.104195, 0.4075, 0.021328, 0.036057], [6, 0.088453, 0.431344, 0.029281, 0.048326]]}, {"filename": "000000085157.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.086516, 0.892896, 0.173031, 0.214208], [4, 0.854086, 0.823604, 0.165172, 0.352792], [4, 0.963977, 0.559552, 0.057453, 0.188771], [0, 0.448148, 0.390396, 0.176578, 0.309417], [0, 0.821094, 0.549479, 0.355469, 0.731375], [0, 0.201859, 0.602479, 0.403719, 0.795042], [0, 0.278937, 0.425833, 0.123031, 0.334833], [0, 0.663938, 0.611469, 0.375875, 0.744729], [0, 0.858297, 0.358969, 0.198, 0.329562], [0, 0.039344, 0.307542, 0.075781, 0.168125], [4, 0.343969, 0.49651, 0.036844, 0.066437]]}, {"filename": "000000085376.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.450656, 0.560547, 0.208937, 0.084469], [1, 0.241021, 0.544164, 0.323042, 0.085359], [1, 0.097385, 0.650188, 0.182771, 0.141562], [1, 0.35076, 0.517313, 0.142354, 0.037375], [1, 0.505812, 0.51482, 0.152292, 0.017422], [1, 0.088406, 0.495711, 0.119396, 0.031422], [1, 0.37524, 0.811438, 0.690563, 0.331688], [0, 0.01925, 0.525625, 0.0385, 0.090719], [1, 0.545813, 0.530258, 0.074042, 0.022922], [1, 0.940917, 0.574656, 0.118167, 0.149937], [1, 0.711687, 0.584414, 0.419042, 0.181828]]}, {"filename": "000000086107.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.815961, 0.430187, 0.230422, 0.10271], [1, 0.98457, 0.431379, 0.030859, 0.06472], [1, 0.586461, 0.494626, 0.496703, 0.299766], [1, 0.819203, 0.436075, 0.235625, 0.10743], [0, 0.844625, 0.495514, 0.071906, 0.236963], [0, 0.963266, 0.439988, 0.032094, 0.115678], [0, 0.919727, 0.416414, 0.021922, 0.045584], [0, 0.945398, 0.435082, 0.025797, 0.113621], [0, 0.664445, 0.404136, 0.020234, 0.035421], [0, 0.207898, 0.487956, 0.098078, 0.293341]]}, {"filename": "000000094944.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.64475, 0.736546, 0.170531, 0.42377], [0, 0.295859, 0.533068, 0.041469, 0.179859], [0, 0.933969, 0.615023, 0.087969, 0.150141]]}, {"filename": "000000098716.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.683125, 0.430585, 0.219062, 0.248301], [0, 0.109195, 0.48266, 0.032672, 0.093064], [3, 0.526195, 0.69032, 0.496516, 0.396407], [3, 0.957375, 0.587646, 0.079406, 0.357298], [0, 0.040539, 0.479011, 0.031203, 0.103036]]}, {"filename": "000000100510.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.371192, 0.211738, 0.14155, 0.300075], [0, 0.910933, 0.4669, 0.072333, 0.22885], [0, 0.014958, 0.078587, 0.029917, 0.123875], [4, 0.972775, 0.516088, 0.025383, 0.109225], [4, 0.5628, 0.527262, 0.041867, 0.112375]]}, {"filename": "000000104198.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.78455, 0.577533, 0.41966, 0.52584]]}, {"filename": "000000104782.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.5705, 0.586507, 0.13988, 0.8]]}, {"filename": "000000105923.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.113906, 0.482692, 0.130031, 0.108675], [0, 0.326727, 0.420694, 0.005828, 0.021432], [0, 0.295836, 0.419893, 0.006641, 0.019487], [0, 0.205578, 0.415812, 0.006156, 0.020043], [0, 0.275312, 0.420598, 0.006, 0.022393], [0, 0.160984, 0.418643, 0.005875, 0.01985], [0, 0.220789, 0.420726, 0.004234, 0.021154], [0, 0.355281, 0.431827, 0.007969, 0.060107]]}, {"filename": "000000110626.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.42925, 0.264242, 0.256333, 0.188641]]}, {"filename": "000000111574.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.918258, 0.335857, 0.087391, 0.051103], [1, 0.975687, 0.336315, 0.046969, 0.053662], [1, 0.59768, 0.36081, 0.132016, 0.223122], [1, 0.869719, 0.317887, 0.084563, 0.072347], [0, 0.316, 0.485798, 0.369844, 0.683662], [3, 0.14168, 0.415023, 0.039516, 0.252066]]}, {"filename": "000000114313.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.860086, 0.282635, 0.279828, 0.192436], [1, 0.558805, 0.280843, 0.386453, 0.527447], [1, 0.228266, 0.057705, 0.083562, 0.099391], [1, 0.720938, 0.188009, 0.11125, 0.111148], [1, 0.122734, 0.048747, 0.045813, 0.074403], [1, 0.274383, 0.044614, 0.035297, 0.040515], [5, 0.398063, 0.526593, 0.118438, 0.339321], [0, 0.338852, 0.363747, 0.208859, 0.630632], [0, 0.670289, 0.121663, 0.024484, 0.050773], [1, 0.270625, 0.108618, 0.145438, 0.11911], [0, 0.544813, 0.082869, 0.0135, 0.043817], [0, 0.068828, 0.02623, 0.013781, 0.049696], [1, 0.143266, 0.064063, 0.038813, 0.105925], [1, 0.342758, 0.074227, 0.054797, 0.034684]]}, {"filename": "000000117317.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.45717, 0.486512, 0.30058, 0.883132], [3, 0.06378, 0.776032, 0.12504, 0.101103]]}, {"filename": "000000122745.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.597135, 0.283445, 0.293271, 0.222234]]}, {"filename": "000000122973.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.972797, 0.617201, 0.048781, 0.339789], [1, 0.515352, 0.515703, 0.922984, 0.92377], [1, 0.973062, 0.616089, 0.053875, 0.336628], [0, 0.066859, 0.533454, 0.012844, 0.043536], [0, 0.017922, 0.51637, 0.008781, 0.038548], [0, 0.743437, 0.376628, 0.110906, 0.181827], [0, 0.02725, 0.546311, 0.008719, 0.048407], [3, 0.453797, 0.591335, 0.129156, 0.205667], [3, 0.353492, 0.605035, 0.100016, 0.2237]]}, {"filename": "000000128148.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.049828, 0.616086, 0.099656, 0.239743], [4, 0.12325, 0.503621, 0.034219, 0.070047], [4, 0.130141, 0.487079, 0.044406, 0.074019]]}, {"filename": "000000131126.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.53478, 0.829573, 0.2766, 0.340853]]}, {"filename": "000000132796.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.155609, 0.672477, 0.162312, 0.257254], [0, 0.542266, 0.549331, 0.175719, 0.778662], [0, 0.741906, 0.360822, 0.230344, 0.433709], [0, 0.115031, 0.558627, 0.230063, 0.433451], [3, 0.77275, 0.485469, 0.26925, 0.314601], [3, 0.154062, 0.668533, 0.191469, 0.258427]]}, {"filename": "000000133403.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.535953, 0.294531, 0.209, 0.355063], [1, 0.492375, 0.69826, 0.458781, 0.525937]]}, {"filename": "000000135303.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.361729, 0.625203, 0.241167, 0.364719]]}, {"filename": "000000137246.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.462, 0.604745, 0.381, 0.77027], [3, 0.22899, 0.667417, 0.1347, 0.215736], [3, 0.0706, 0.669249, 0.1412, 0.13003], [3, 0.10225, 0.488033, 0.2045, 0.054084], [3, 0.26559, 0.487312, 0.02638, 0.035405], [3, 0.29041, 0.531276, 0.08518, 0.057207], [0, 0.79377, 0.506291, 0.11146, 0.448438]]}, {"filename": "000000137479.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.23832, 0.270784, 0.147611, 0.235947], [0, 0.237075, 0.634763, 0.098198, 0.493432]]}, {"filename": "000000137810.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.535981, 0.521375, 0.023294, 0.044562], [0, 0.529155, 0.191352, 0.017014, 0.035328], [0, 0.766032, 0.04357, 0.01872, 0.039328], [0, 0.309113, 0.639375, 0.037201, 0.047531], [0, 0.833191, 0.602484, 0.032014, 0.034187], [1, 0.90791, 0.051992, 0.09384, 0.083266], [0, 0.723259, 0.085063, 0.017543, 0.037469], [0, 0.031817, 0.544344, 0.013942, 0.029094], [0, 0.022645, 0.549641, 0.017235, 0.037469], [0, 0.883729, 0.612992, 0.024761, 0.051734]]}, {"filename": "000000140556.jpg", "coco_split": "val2017", "split": "val", "boxes": [[5, 0.682914, 0.763895, 0.173703, 0.47221], [0, 0.370734, 0.499431, 0.319156, 0.97291], [0, 0.84775, 0.511466, 0.238563, 0.950678], [0, 0.178914, 0.569661, 0.181328, 0.829212], [1, 0.040891, 0.583414, 0.06575, 0.032516], [0, 0.511008, 0.423643, 0.133391, 0.52186]]}, {"filename": "000000142263.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.114969, 0.602083, 0.107875, 0.074958], [1, 0.556414, 0.585542, 0.084359, 0.045375], [1, 0.416461, 0.58651, 0.077922, 0.044937], [1, 0.282094, 0.605927, 0.058094, 0.066604], [1, 0.866047, 0.591344, 0.015969, 0.029646], [1, 0.906836, 0.592, 0.031297, 0.029792], [1, 0.950516, 0.593312, 0.033531, 0.030042], [0, 0.320367, 0.579698, 0.009078, 0.033896], [0, 0.614203, 0.581646, 0.007156, 0.032167], [0, 0.309281, 0.578094, 0.010812, 0.035854], [1, 0.512477, 0.565458, 0.112109, 0.06375], [6, 0.246156, 0.467406, 0.052281, 0.077437], [1, 0.664008, 0.58425, 0.035797, 0.027292], [1, 0.833883, 0.591052, 0.015859, 0.025271], [1, 0.636141, 0.585042, 0.025469, 0.0215], [0, 0.324109, 0.561823, 0.004625, 0.011688]]}, {"filename": "000000143277.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.475164, 0.622459, 0.118984, 0.311522], [0, 0.597055, 0.738724, 0.041359, 0.104941], [0, 0.24607, 0.465714, 0.011109, 0.054145], [0, 0.255812, 0.468115, 0.010125, 0.046347], [0, 0.28218, 0.455972, 0.018172, 0.05185], [0, 0.759398, 0.48877, 0.019484, 0.051686], [0, 0.646898, 0.406534, 0.007516, 0.045059], [0, 0.272539, 0.393267, 0.013234, 0.043255], [1, 0.355266, 0.547646, 0.063656, 0.108923], [1, 0.888367, 0.880023, 0.180297, 0.192365], [1, 0.314102, 0.457307, 0.055484, 0.078642], [1, 0.66482, 0.505867, 0.064141, 0.07459], [1, 0.712914, 0.574157, 0.088453, 0.105597], [1, 0.558242, 0.367728, 0.071297, 0.041311], [1, 0.372438, 0.351721, 0.025219, 0.03267], [1, 0.379141, 0.339824, 0.023031, 0.034169], [1, 0.618883, 0.441218, 0.040234, 0.04356], [5, 0.3375, 0.387061, 0.007969, 0.018806], [4, 0.887266, 0.753326, 0.030219, 0.056487], [0, 0.669805, 0.406475, 0.009297, 0.039461], [0, 0.357445, 0.312529, 0.004203, 0.021311], [0, 0.699727, 0.413618, 0.013672, 0.046628], [0, 0.661562, 0.404965, 0.011062, 0.046089], [0, 0.700539, 0.3813, 0.012172, 0.030843], [1, 0.632984, 0.467658, 0.051844, 0.060141], [1, 0.553453, 0.336347, 0.025063, 0.021265], [1, 0.174625, 0.709871, 0.144969, 0.249391], [1, 0.793031, 0.683302, 0.1355, 0.174239], [1, 0.694805, 0.368794, 0.038484, 0.066487], [0, 0.283461, 0.38808, 0.007297, 0.035644], [1, 0.541305, 0.330574, 0.018516, 0.024333]]}, {"filename": "000000143961.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.281875, 0.716854, 0.15375, 0.54382], [0, 0.576898, 0.695365, 0.186234, 0.342303], [0, 0.847445, 0.737233, 0.186953, 0.470534], [0, 0.955898, 0.671166, 0.085203, 0.63514], [0, 0.149203, 0.614185, 0.119781, 0.368652], [0, 0.788789, 0.875983, 0.179547, 0.248034], [0, 0.183891, 0.195871, 0.108625, 0.30118], [0, 0.421602, 0.167725, 0.118516, 0.203315], [0, 0.363828, 0.318469, 0.227562, 0.226433], [0, 0.2175, 0.287317, 0.070031, 0.221152], [0, 0.472469, 0.829621, 0.159313, 0.316039], [0, 0.065016, 0.655843, 0.127156, 0.522079], [0, 0.093992, 0.483118, 0.117359, 0.269719]]}, {"filename": "000000145620.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.786984, 0.218406, 0.026063, 0.12117], [0, 0.824602, 0.219839, 0.020547, 0.127202], [0, 0.56643, 0.401124, 0.183703, 0.375275], [0, 0.810828, 0.215436, 0.015875, 0.12078], [1, 0.572445, 0.550229, 0.204953, 0.38578]]}, {"filename": "000000147338.jpg", "coco_split": "val2017", "split": "val", "boxes": [[5, 0.515354, 0.511234, 0.527333, 0.730344], [1, 0.19626, 0.225844, 0.386521, 0.209], [1, 0.69476, 0.204492, 0.610479, 0.283141], [1, 0.926, 0.07782, 0.148, 0.072672], [1, 0.052177, 0.171719, 0.104354, 0.029031], [1, 0.192792, 0.224039, 0.385583, 0.210391]]}, {"filename": "000000147740.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.142133, 0.790354, 0.284266, 0.394], [0, 0.079078, 0.516625, 0.144562, 0.428583], [0, 0.152414, 0.543635, 0.095953, 0.388937], [0, 0.195852, 0.597917, 0.059141, 0.275083], [0, 0.630805, 0.388187, 0.065797, 0.239792], [0, 0.506461, 0.79775, 0.409547, 0.373042], [0, 0.602289, 0.335594, 0.029797, 0.070396], [1, 0.069992, 0.617198, 0.139984, 0.388938], [1, 0.63157, 0.592083, 0.085641, 0.323083], [0, 0.935547, 0.576833, 0.128906, 0.807292]]}, {"filename": "000000148761.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.491812, 0.622914, 0.185281, 0.555058], [3, 0.319312, 0.519534, 0.287844, 0.275851]]}, {"filename": "000000150265.jpg", "coco_split": "val2017", "split": "val", "boxes": [[5, 0.677021, 0.336328, 0.08075, 0.161375], [0, 0.462927, 0.785391, 0.416479, 0.39775], [0, 0.195115, 0.126711, 0.108771, 0.161391], [0, 0.102062, 0.081367, 0.127333, 0.162734], [0, 0.366031, 0.061156, 0.058688, 0.121531], [0, 0.18251, 0.375281, 0.314729, 0.505812], [0, 0.231063, 0.053914, 0.082458, 0.107828]]}, {"filename": "000000151962.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.418594, 0.59428, 0.632906, 0.780752], [0, 0.113523, 0.885543, 0.058859, 0.188747]]}, {"filename": "000000158227.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.48917, 0.506204, 0.53026, 0.959356]]}, {"filename": "000000161242.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.617906, 0.63668, 0.764188, 0.700234], [0, 0.944812, 0.968867, 0.110376, 0.062266], [0, 0.067094, 0.792016, 0.134188, 0.366187], [0, 0.637988, 0.448641, 0.719882, 0.799406]]}, {"filename": "000000165086.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.08193, 0.62046, 0.156891, 0.100493], [1, 0.717133, 0.580642, 0.122391, 0.107537], [1, 0.635703, 0.595642, 0.2005, 0.096724], [1, 0.284656, 0.587291, 0.10175, 0.095653], [1, 0.97143, 0.564122, 0.057141, 0.125418], [1, 0.488906, 0.577034, 0.118844, 0.074497], [1, 0.343945, 0.640964, 0.074172, 0.099358], [1, 0.784828, 0.679101, 0.043281, 0.097859], [1, 0.66743, 0.633704, 0.136859, 0.126895], [0, 0.555742, 0.560931, 0.015547, 0.031028], [0, 0.86475, 0.630364, 0.079063, 0.191863], [0, 0.825031, 0.584486, 0.029375, 0.092527], [0, 0.477531, 0.585257, 0.024469, 0.07621], [0, 0.52575, 0.606124, 0.047063, 0.118287], [0, 0.428664, 0.618051, 0.049828, 0.144154], [0, 0.34525, 0.609283, 0.05925, 0.119336], [0, 0.660242, 0.608394, 0.054234, 0.132548], [1, 0.435484, 0.655996, 0.112469, 0.101756], [1, 0.884273, 0.665503, 0.187516, 0.172163], [1, 0.236867, 0.866124, 0.253422, 0.241071], [1, 0.535375, 0.645685, 0.106594, 0.083148], [0, 0.201523, 0.783919, 0.153984, 0.40334], [1, 0.369188, 0.57166, 0.045063, 0.040278], [1, 0.482625, 0.614165, 0.051625, 0.066488], [0, 0.433883, 0.569925, 0.017453, 0.035482], [0, 0.935023, 0.527655, 0.005141, 0.00636], [0, 0.83443, 0.575803, 0.022141, 0.054004], [1, 0.407594, 0.550846, 0.030063, 0.045289], [1, 0.934281, 0.53364, 0.019187, 0.010064]]}, {"filename": "000000166747.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.256173, 0.62472, 0.140827, 0.31012], [0, 0.462387, 0.25462, 0.081627, 0.15976], [0, 0.3266, 0.56786, 0.07192, 0.18876], [0, 0.383107, 0.55236, 0.040293, 0.086], [0, 0.917933, 0.60081, 0.02888, 0.04838], [0, 0.438067, 0.21673, 0.04328, 0.06666], [0, 0.690413, 0.28942, 0.154373, 0.13344], [0, 0.708667, 0.25801, 0.038667, 0.05994], [0, 0.91548, 0.59178, 0.024773, 0.06408], [0, 0.595533, 0.24556, 0.046, 0.05936], [0, 0.54128, 0.27826, 0.08672, 0.12264], [0, 0.420067, 0.53469, 0.043067, 0.05674], [3, 0.611253, 0.29845, 0.129973, 0.07002], [0, 0.38388, 0.49735, 0.033733, 0.02878], [1, 0.600533, 0.42985, 0.364267, 0.38858]]}, {"filename": "000000167122.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.52557, 0.724802, 0.948859, 0.550396], [0, 0.625977, 0.435917, 0.016453, 0.046458], [1, 0.423602, 0.438063, 0.133391, 0.036042], [1, 0.305961, 0.441896, 0.079078, 0.025833], [1, 0.883266, 0.4565, 0.043344, 0.023708]]}, {"filename": "000000168337.jpg", "coco_split": "val2017", "split": "val", "boxes": [[5, 0.511484, 0.730281, 0.078719, 0.179646], [0, 0.876938, 0.499333, 0.070406, 0.279792], [0, 0.112516, 0.63951, 0.0875, 0.329188]]}, {"filename": "000000180095.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.605741, 0.35618, 0.412847, 0.361797]]}, {"filename": "000000183648.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.804305, 0.709463, 0.183766, 0.581075], [0, 0.439711, 0.730853, 0.272734, 0.538294], [0, 0.290797, 0.737079, 0.214906, 0.494393]]}, {"filename": "000000185313.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.464273, 0.511393, 0.236859, 0.576045]]}, {"filename": "000000186233.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.65631, 0.777813, 0.06154, 0.173653]]}, {"filename": "000000190515.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.174227, 0.585055, 0.055141, 0.075881], [1, 0.032219, 0.701388, 0.064438, 0.039251], [5, 0.23632, 0.812004, 0.019922, 0.103789]]}, {"filename": "000000190648.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.571656, 0.537073, 0.167094, 0.130854]]}, {"filename": "000000191614.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.595866, 0.528407, 0.055229, 0.109265]]}, {"filename": "000000193094.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.304531, 0.804458, 0.092281, 0.369125]]}, {"filename": "000000194184.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.866373, 0.52092, 0.267253, 0.10252], [0, 0.08568, 0.63485, 0.155147, 0.19514], [1, 0.875187, 0.63669, 0.242053, 0.23614], [1, 0.399107, 0.54735, 0.188347, 0.10314], [1, 0.633467, 0.55115, 0.332107, 0.20162], [1, 0.404347, 0.47795, 0.209333, 0.09374], [0, 0.348533, 0.51794, 0.01408, 0.0126], [4, 0.064413, 0.84606, 0.128827, 0.23596], [1, 0.131173, 0.5298, 0.2608, 0.17412], [1, 0.285347, 0.50178, 0.110427, 0.07644], [1, 0.13196, 0.50837, 0.236507, 0.1311]]}, {"filename": "000000196754.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.424453, 0.67121, 0.216427, 0.1689], [4, 0.78764, 0.80192, 0.42472, 0.39556]]}, {"filename": "000000202800.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.092961, 0.376405, 0.167922, 0.739321], [0, 0.280125, 0.365703, 0.404188, 0.729859]]}, {"filename": "000000203669.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.70757, 0.575375, 0.442797, 0.822881], [7, 0.067797, 0.429813, 0.041344, 0.083372]]}, {"filename": "000000203931.jpg", "coco_split": "val2017", "split": "val", "boxes": [[3, 0.188178, 0.658422, 0.369626, 0.071906], [0, 0.490607, 0.607867, 0.443551, 0.644953], [0, 0.589825, 0.181555, 0.069416, 0.051078], [0, 0.439276, 0.185336, 0.060888, 0.042141], [0, 0.346671, 0.202141, 0.066379, 0.050406], [0, 0.248458, 0.208437, 0.071822, 0.081625], [0, 0.204182, 0.221227, 0.07229, 0.067297], [0, 0.176811, 0.128914, 0.048528, 0.055422], [0, 0.089568, 0.072766, 0.097921, 0.07725], [0, 0.380619, 0.064867, 0.050631, 0.036453], [0, 0.657932, 0.627602, 0.133107, 0.108578], [0, 0.692395, 0.231063, 0.046799, 0.043562], [0, 0.746706, 0.361586, 0.060187, 0.063734], [0, 0.783458, 0.406133, 0.078458, 0.073547]]}, {"filename": "000000205324.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.19393, 0.344494, 0.379891, 0.592329], [0, 0.187703, 0.327424, 0.213875, 0.653718]]}, {"filename": "000000205656.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.20168, 0.524708, 0.034703, 0.033875], [1, 0.139461, 0.506885, 0.020859, 0.014896], [1, 0.56882, 0.478646, 0.665734, 0.59325], [0, 0.579203, 0.421656, 0.03875, 0.105687], [1, 0.861602, 0.607115, 0.017547, 0.049229], [0, 0.407141, 0.366969, 0.038344, 0.091563], [1, 0.137398, 0.495312, 0.028328, 0.034625]]}, {"filename": "000000205887.jpg", "coco_split": "train2017", "split": "val", "boxes": [[7, 0.482802, 0.554078, 0.600896, 0.8565]]}, {"filename": "000000209200.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.886852, 0.537865, 0.226297, 0.293104], [1, 0.062742, 0.499271, 0.125266, 0.200792], [1, 0.171172, 0.472802, 0.122937, 0.087771], [1, 0.414891, 0.497729, 0.567531, 0.370083], [1, 0.796617, 0.477875, 0.178516, 0.214833], [6, 0.129195, 0.365229, 0.047922, 0.081958], [0, 0.423891, 0.427198, 0.020437, 0.046063], [0, 0.650203, 0.554833, 0.08325, 0.312958], [0, 0.833945, 0.568188, 0.055609, 0.307375], [1, 0.226953, 0.446604, 0.0425, 0.020708], [5, 0.090078, 0.545583, 0.045031, 0.182625]]}, {"filename": "000000209405.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.394219, 0.493483, 0.235781, 0.137124], [1, 0.709477, 0.726101, 0.581047, 0.547798], [7, 0.245078, 0.614371, 0.075062, 0.431798], [0, 0.917203, 0.600191, 0.1025, 0.151573], [1, 0.970539, 0.415843, 0.058297, 0.092225]]}, {"filename": "000000212072.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.77097, 0.276493, 0.30034, 0.395973]]}, {"filename": "000000213780.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.164109, 0.640022, 0.302594, 0.32512], [5, 0.890594, 0.601783, 0.013719, 0.028425], [0, 0.113797, 0.543698, 0.034, 0.034354], [1, 0.383461, 0.611586, 0.061547, 0.037965], [1, 0.705688, 0.596499, 0.167094, 0.14]]}, {"filename": "000000214200.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.486698, 0.480336, 0.515316, 0.397766], [1, 0.533677, 0.870414, 0.932646, 0.217984], [1, 0.852529, 0.798844, 0.284403, 0.074344], [1, 0.883126, 0.73718, 0.232763, 0.054391]]}, {"filename": "000000215776.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.505617, 0.552115, 0.269672, 0.620229], [0, 0.415742, 0.417281, 0.453703, 0.745688]]}, {"filename": "000000221820.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.670922, 0.59082, 0.359719, 0.532459]]}, {"filename": "000000223754.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.712453, 0.688688, 0.066719, 0.364708], [0, 0.647227, 0.733656, 0.107891, 0.385479], [0, 0.482023, 0.761427, 0.117984, 0.391021], [0, 0.939453, 0.540813, 0.052969, 0.223458], [1, 0.060633, 0.459479, 0.121266, 0.135667], [3, 0.261445, 0.79624, 0.148766, 0.113062], [0, 0.417133, 0.779781, 0.085953, 0.350563], [0, 0.310414, 0.573448, 0.051422, 0.165563], [0, 0.586375, 0.738792, 0.075375, 0.381208], [0, 0.219453, 0.583313, 0.030125, 0.151417], [0, 0.574875, 0.480469, 0.018656, 0.029187], [0, 0.897906, 0.536104, 0.043938, 0.201792], [0, 0.7815, 0.683927, 0.083813, 0.351104], [0, 0.980984, 0.575625, 0.038031, 0.244], [1, 0.048969, 0.660469, 0.095344, 0.259771], [0, 0.118156, 0.652771, 0.090312, 0.211333]]}, {"filename": "000000223955.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.53595, 0.595646, 0.37134, 0.787808]]}, {"filename": "000000224200.jpg", "coco_split": "val2017", "split": "val", "boxes": [[5, 0.821352, 0.384273, 0.357297, 0.764047]]}, {"filename": "000000229866.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.480082, 0.594227, 0.379321, 0.483203]]}, {"filename": "000000235836.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.69822, 0.695141, 0.365059, 0.508625], [4, 0.336698, 0.486187, 0.354988, 0.266313], [4, 0.83089, 0.501211, 0.337002, 0.282578], [4, 0.080925, 0.510563, 0.16185, 0.169187]]}, {"filename": "000000235876.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.305922, 0.786265, 0.441812, 0.425363], [0, 0.3965, 0.629122, 0.088094, 0.231827], [0, 0.273492, 0.731979, 0.183328, 0.443677], [0, 0.481023, 0.594543, 0.074047, 0.19466], [0, 0.799805, 0.429801, 0.104391, 0.275105], [0, 0.964797, 0.664789, 0.052, 0.307658], [0, 0.516094, 0.504742, 0.035469, 0.056838], [0, 0.098703, 0.521698, 0.037844, 0.047705], [0, 0.932562, 0.593255, 0.041969, 0.233724], [0, 0.068789, 0.591967, 0.048078, 0.247728], [0, 0.1695, 0.606593, 0.051156, 0.266112], [1, 0.686156, 0.531194, 0.139844, 0.057986], [3, 0.086961, 0.642693, 0.173922, 0.202248], [0, 0.991875, 0.648103, 0.016188, 0.255878], [1, 0.453711, 0.699356, 0.203766, 0.182787], [1, 0.432609, 0.515035, 0.041219, 0.021218], [1, 0.501094, 0.511499, 0.179344, 0.07452], [1, 0.060437, 0.520012, 0.033, 0.04555], [1, 0.069805, 0.492681, 0.020766, 0.013934]]}, {"filename": "000000238065.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.93361, 0.801727, 0.128435, 0.083922], [0, 0.544229, 0.445578, 0.613131, 0.575531]]}, {"filename": "000000243328.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.539789, 0.984118, 0.354797, 0.031765]]}, {"filename": "000000245638.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.944328, 0.372141, 0.111344, 0.076155], [1, 0.497062, 0.501125, 0.793187, 0.970777], [1, 0.908047, 0.36761, 0.084406, 0.068486], [1, 0.833883, 0.32003, 0.067422, 0.04747], [6, 0.84618, 0.448078, 0.075797, 0.123048], [1, 0.88257, 0.298635, 0.050891, 0.021614], [1, 0.98168, 0.314552, 0.036641, 0.038147]]}, {"filename": "000000252092.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.411562, 0.62204, 0.306219, 0.5969], [0, 0.701055, 0.552786, 0.328422, 0.774056], [0, 0.462539, 0.253298, 0.052141, 0.089441], [0, 0.424383, 0.160734, 0.041859, 0.107529], [0, 0.689937, 0.062296, 0.042188, 0.078671], [0, 0.839539, 0.126235, 0.034797, 0.158625], [0, 0.881523, 0.117343, 0.044734, 0.177902], [0, 0.941437, 0.101655, 0.033594, 0.1538], [0, 0.442367, 0.356667, 0.043859, 0.151235], [0, 0.176875, 0.30859, 0.051313, 0.074942], [0, 0.763289, 0.164242, 0.041984, 0.081119], [0, 0.828023, 0.28789, 0.041672, 0.082494], [0, 0.239133, 0.319172, 0.034234, 0.095641]]}, {"filename": "000000252332.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.485398, 0.474906, 0.711234, 0.946063]]}, {"filename": "000000253221.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.755852, 0.906156, 0.180146, 0.067688], [0, 0.184002, 0.909164, 0.095696, 0.111484], [0, 0.837755, 0.890008, 0.017089, 0.010766], [1, 0.976653, 0.922172, 0.046694, 0.0375], [1, 0.875322, 0.906992, 0.159792, 0.060453]]}, {"filename": "000000255500.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.202453, 0.817302, 0.152875, 0.330979], [4, 0.3105, 0.677812, 0.335188, 0.305042], [4, 0.785828, 0.70649, 0.305719, 0.392812], [4, 0.402289, 0.867198, 0.205609, 0.265604]]}, {"filename": "000000258078.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.402664, 0.595719, 0.392512, 0.74775], [7, 0.513216, 0.217836, 0.244648, 0.255641], [1, 0.507441, 0.579211, 0.985117, 0.480203]]}, {"filename": "000000264718.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.968333, 0.627086, 0.063333, 0.063078], [1, 0.898594, 0.59093, 0.155438, 0.056734], [1, 0.961531, 0.563617, 0.071396, 0.011328], [0, 0.414604, 0.574117, 0.010292, 0.022266], [0, 0.332469, 0.577867, 0.008979, 0.024672], [0, 0.76749, 0.594, 0.012063, 0.047562], [1, 0.562854, 0.601203, 0.089125, 0.052375], [7, 0.680885, 0.648094, 0.040646, 0.068594], [7, 0.649771, 0.612164, 0.021917, 0.037453], [7, 0.640187, 0.588719, 0.014917, 0.020563], [1, 0.912948, 0.596398, 0.144146, 0.054734], [1, 0.925979, 0.60757, 0.148042, 0.061734], [1, 0.91924, 0.635133, 0.042396, 0.040703], [1, 0.839323, 0.573711, 0.038021, 0.019797], [1, 0.727958, 0.581039, 0.049375, 0.025578], [7, 0.062229, 0.582906, 0.009208, 0.018125], [7, 0.158, 0.578734, 0.0095, 0.018594], [7, 0.246667, 0.57643, 0.006333, 0.011359], [7, 0.293583, 0.575891, 0.006208, 0.011562], [7, 0.338917, 0.575047, 0.007458, 0.011062], [7, 0.367521, 0.573695, 0.004042, 0.010172]]}, {"filename": "000000268681.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.820117, 0.690301, 0.046297, 0.043408], [1, 0.793648, 0.690167, 0.017609, 0.022027], [3, 0.902633, 0.866915, 0.103172, 0.05686], [3, 0.062602, 0.878575, 0.100141, 0.063786], [1, 0.916195, 0.690757, 0.020516, 0.01902], [1, 0.984891, 0.677884, 0.030219, 0.016214], [3, 0.345414, 0.873062, 0.094297, 0.058463], [3, 0.17518, 0.879666, 0.046453, 0.057194], [3, 0.240297, 0.875824, 0.032125, 0.061047], [1, 0.843297, 0.68422, 0.015938, 0.025412], [1, 0.889828, 0.693586, 0.015344, 0.016437], [0, 0.20857, 0.843463, 0.022141, 0.041715]]}, {"filename": "000000276285.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.550246, 0.111484, 0.729157, 0.216219], [4, 0.106101, 0.062914, 0.205457, 0.112359], [4, 0.915375, 0.103523, 0.169251, 0.151828]]}, {"filename": "000000279278.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.349492, 0.340093, 0.206828, 0.673427], [0, 0.644961, 0.374627, 0.212766, 0.749254], [0, 0.42982, 0.288322, 0.122922, 0.569883], [0, 0.25418, 0.2519, 0.070484, 0.445198], [0, 0.941078, 0.512984, 0.117844, 0.945828], [1, 0.772203, 0.244464, 0.095469, 0.216434], [1, 0.201094, 0.221911, 0.06175, 0.27641], [1, 0.038039, 0.281608, 0.070516, 0.185221], [1, 0.013141, 0.303473, 0.026281, 0.189138], [1, 0.083914, 0.129744, 0.167828, 0.209044], [1, 0.849219, 0.197075, 0.188719, 0.2838], [0, 0.498398, 0.213683, 0.096484, 0.425268], [1, 0.546844, 0.240874, 0.047562, 0.253497], [1, 0.892305, 0.203578, 0.214578, 0.290606], [1, 0.219234, 0.794114, 0.435313, 0.38352], [1, 0.207, 0.157214, 0.049906, 0.125991], [0, 0.176656, 0.093823, 0.072, 0.187646]]}, {"filename": "000000282037.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.967086, 0.213619, 0.007516, 0.02069], [0, 0.831773, 0.716704, 0.336453, 0.566592], [0, 0.666391, 0.341125, 0.038875, 0.098196], [0, 0.738586, 0.343864, 0.027297, 0.097261], [0, 0.951187, 0.342238, 0.025969, 0.121225], [0, 0.251797, 0.335768, 0.107688, 0.279911], [4, 0.874664, 0.137394, 0.015672, 0.011359], [4, 0.915359, 0.115746, 0.009844, 0.007261], [4, 0.919516, 0.074488, 0.005406, 0.005813], [4, 0.66582, 0.179399, 0.014172, 0.00833], [4, 0.866086, 0.088363, 0.008484, 0.006303], [4, 0.593836, 0.178396, 0.013734, 0.0098], [4, 0.805312, 0.13098, 0.010469, 0.007082], [4, 0.737813, 0.205746, 0.013781, 0.022673], [0, 0.825875, 0.282416, 0.016656, 0.026971], [0, 0.768328, 0.286526, 0.015688, 0.01951], [0, 0.788469, 0.2851, 0.010687, 0.023474], [0, 0.817812, 0.203675, 0.011219, 0.041381], [0, 0.940656, 0.127472, 0.008125, 0.021314], [0, 0.147859, 0.328575, 0.01675, 0.056793], [0, 0.124367, 0.324666, 0.016453, 0.065189], [4, 0.866672, 0.123185, 0.011125, 0.010067], [4, 0.891453, 0.101971, 0.007625, 0.005367], [4, 0.900242, 0.080857, 0.006766, 0.006704], [0, 0.135664, 0.327572, 0.014859, 0.058575]]}, {"filename": "000000284445.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.184281, 0.57117, 0.059719, 0.020791], [0, 0.365141, 0.758359, 0.210188, 0.483283], [0, 0.77207, 0.703872, 0.106922, 0.340236], [0, 0.840617, 0.647593, 0.068797, 0.233131], [0, 0.501445, 0.720976, 0.107766, 0.339865], [0, 0.600969, 0.71021, 0.10325, 0.359882], [0, 0.636672, 0.608695, 0.061375, 0.154024], [0, 0.971602, 0.749646, 0.056516, 0.421549], [0, 0.740555, 0.577466, 0.030953, 0.047828], [0, 0.891477, 0.709226, 0.085547, 0.307744], [0, 0.437375, 0.60697, 0.026375, 0.064141], [0, 0.708812, 0.629032, 0.048031, 0.146077], [1, 0.029437, 0.565034, 0.046156, 0.049731], [0, 0.462688, 0.612551, 0.037875, 0.092946], [1, 0.917266, 0.581684, 0.046594, 0.070875], [0, 0.940977, 0.648215, 0.060703, 0.214916]]}, {"filename": "000000284651.jpg", "coco_split": "train2017", "split": "val", "boxes": [[7, 0.669629, 0.615133, 0.075622, 0.106328], [0, 0.577739, 0.656, 0.224569, 0.362656]]}, {"filename": "000000290002.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.557367, 0.513118, 0.255172, 0.685388]]}, {"filename": "000000291634.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.517644, 0.689648, 0.850531, 0.407328], [1, 0.541905, 0.553414, 0.816605, 0.345547], [0, 0.848383, 0.369422, 0.082402, 0.146719], [0, 0.680797, 0.376344, 0.044827, 0.151656], [0, 0.524192, 0.351086, 0.042933, 0.086453], [0, 0.049954, 0.367648, 0.057229, 0.131641], [0, 0.756836, 0.376586, 0.058984, 0.146047], [0, 0.397344, 0.344273, 0.043603, 0.055891], [0, 0.318279, 0.335313, 0.016467, 0.044031], [1, 0.214099, 0.475359, 0.349677, 0.2505], [0, 0.491478, 0.347281, 0.035889, 0.081875], [0, 0.017229, 0.374547, 0.034457, 0.140281], [1, 0.22821, 0.44432, 0.385889, 0.207391], [1, 0.070554, 0.449086, 0.05358, 0.069078], [0, 0.331212, 0.336406, 0.020069, 0.048156], [1, 0.089018, 0.367133, 0.048661, 0.069578], [1, 0.100069, 0.393602, 0.048499, 0.039547]]}, {"filename": "000000292024.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.627541, 0.371895, 0.744918, 0.735882]]}, {"filename": "000000293071.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.135672, 0.37268, 0.270149, 0.13204], [1, 0.530149, 0.37999, 0.621791, 0.16438], [5, 0.371194, 0.72222, 0.251881, 0.459], [1, 0.905761, 0.42083, 0.178388, 0.11642]]}, {"filename": "000000293804.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.75941, 0.772733, 0.4809, 0.428589]]}, {"filename": "000000296657.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.195883, 0.46637, 0.218828, 0.299813], [0, 0.310773, 0.470632, 0.148984, 0.296862], [0, 0.806625, 0.5, 0.254156, 0.534379], [0, 0.60332, 0.531522, 0.130734, 0.472974], [5, 0.311484, 0.65993, 0.008625, 0.041358], [0, 0.486328, 0.55692, 0.082281, 0.381007], [0, 0.389867, 0.495843, 0.297172, 0.593279]]}, {"filename": "000000299115.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.49879, 0.84112, 0.21138, 0.30932], [5, 0.81634, 0.50786, 0.15772, 0.3094], [5, 0.48315, 0.16292, 0.18426, 0.30788], [5, 0.16629, 0.5, 0.2337, 0.29888], [5, 0.1663, 0.84045, 0.22472, 0.29214], [5, 0.84045, 0.8382, 0.21574, 0.26516], [5, 0.15618, 0.17415, 0.15504, 0.2809], [5, 0.8231, 0.17762, 0.2134, 0.28996]]}, {"filename": "000000305309.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.703422, 0.524951, 0.222094, 0.793596], [0, 0.146883, 0.62569, 0.293766, 0.740493], [0, 0.200516, 0.592894, 0.153844, 0.521847]]}, {"filename": "000000310072.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.551977, 0.090183, 0.404422, 0.180366], [1, 0.106242, 0.09158, 0.212484, 0.173029], [1, 0.322531, 0.048251, 0.111031, 0.096501], [3, 0.506523, 0.521292, 0.637359, 0.585196], [1, 0.993742, 0.039373, 0.012516, 0.040366], [1, 0.103023, 0.09329, 0.206047, 0.18658], [1, 0.554141, 0.090052, 0.393531, 0.179112]]}, {"filename": "000000315209.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.974656, 0.720603, 0.034938, 0.028607], [1, 0.822492, 0.710447, 0.027859, 0.015198], [1, 0.91875, 0.70868, 0.018437, 0.02131]]}, {"filename": "000000319534.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.526687, 0.579396, 0.291563, 0.8045], [0, 0.232133, 0.390135, 0.061484, 0.382562], [0, 0.882133, 0.462427, 0.232641, 0.407354], [0, 0.369023, 0.462823, 0.133141, 0.284896], [0, 0.141836, 0.425219, 0.053547, 0.288563], [0, 0.024656, 0.368562, 0.038937, 0.137458], [0, 0.058367, 0.484344, 0.111516, 0.156479], [1, 0.50093, 0.497531, 0.998141, 0.980229]]}, {"filename": "000000322654.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.507766, 0.741367, 0.046406, 0.099463], [0, 0.413852, 0.75868, 0.067016, 0.128715], [0, 0.458398, 0.765304, 0.042297, 0.116075], [0, 0.834711, 0.664299, 0.023328, 0.097944], [0, 0.537477, 0.720105, 0.039859, 0.087266], [5, 0.775195, 0.717453, 0.015703, 0.047336], [1, 0.167187, 0.631577, 0.105969, 0.087266], [1, 0.344234, 0.636449, 0.076094, 0.056589], [1, 0.400273, 0.630607, 0.064172, 0.063178], [1, 0.472422, 0.631098, 0.065062, 0.060421], [1, 0.04825, 0.637967, 0.0965, 0.081916], [1, 0.57418, 0.635935, 0.055672, 0.043364], [1, 0.731328, 0.636729, 0.271906, 0.040981], [1, 0.524156, 0.630339, 0.050063, 0.058808], [1, 0.804828, 0.631379, 0.032719, 0.031916], [1, 0.874211, 0.636624, 0.020641, 0.01993], [1, 0.905234, 0.632009, 0.041875, 0.034673]]}, {"filename": "000000328238.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.410445, 0.641514, 0.276078, 0.716972], [0, 0.80407, 0.362195, 0.075141, 0.146127], [0, 0.859492, 0.341256, 0.065391, 0.13561], [0, 0.713547, 0.318157, 0.027125, 0.064624], [0, 0.739016, 0.323545, 0.029125, 0.089108], [0, 0.758172, 0.332535, 0.035469, 0.084977], [3, 0.639883, 0.385822, 0.068203, 0.03338], [0, 0.890867, 0.386761, 0.081422, 0.090845], [4, 0.852031, 0.390047, 0.052344, 0.094836], [4, 0.784812, 0.386995, 0.051094, 0.089577], [4, 0.872672, 0.379085, 0.033563, 0.071268]]}, {"filename": "000000328808.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.495595, 0.484266, 0.688, 0.851688]]}, {"filename": "000000332845.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.680648, 0.604302, 0.138703, 0.116146], [0, 0.600547, 0.463604, 0.224094, 0.381625]]}, {"filename": "000000334614.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.77743, 0.646565, 0.159453, 0.429741]]}, {"filename": "000000336053.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.746359, 0.106781, 0.076156, 0.175771], [4, 0.668078, 0.108104, 0.065875, 0.173417], [4, 0.24125, 0.263062, 0.338125, 0.265208], [4, 0.961938, 0.086708, 0.076125, 0.059042], [4, 0.79618, 0.050833, 0.021672, 0.044333], [0, 0.204156, 0.248823, 0.367406, 0.415729], [0, 0.84375, 0.492115, 0.3125, 0.763521], [0, 0.062828, 0.132104, 0.125656, 0.264208], [0, 0.856164, 0.061448, 0.114891, 0.122896], [4, 0.971594, 0.568406, 0.056813, 0.221438]]}, {"filename": "000000336445.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.854021, 0.328156, 0.2915, 0.360687], [1, 0.722802, 0.18025, 0.029229, 0.016687], [1, 0.69049, 0.180313, 0.023563, 0.013375], [1, 0.659781, 0.180555, 0.011729, 0.006516], [1, 0.714437, 0.178031, 0.028833, 0.0205], [1, 0.774333, 0.165078, 0.044917, 0.022125], [1, 0.600021, 0.19018, 0.021875, 0.021891], [6, 0.627969, 0.121516, 0.058688, 0.042437], [1, 0.854979, 0.331172, 0.290042, 0.355844]]}, {"filename": "000000336587.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.461125, 0.101104, 0.148312, 0.2], [1, 0.973234, 0.65701, 0.052844, 0.121438]]}, {"filename": "000000338625.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.106453, 0.534823, 0.180656, 0.086021], [1, 0.866039, 0.466115, 0.073234, 0.080354], [1, 0.486492, 0.416667, 0.547297, 0.49775], [0, 0.959125, 0.467177, 0.044781, 0.136021], [0, 0.973687, 0.47076, 0.024312, 0.109396], [0, 0.188414, 0.506302, 0.023547, 0.067313], [0, 0.217852, 0.489719, 0.019859, 0.038729], [0, 0.070195, 0.504292, 0.016672, 0.033083], [0, 0.019328, 0.527542, 0.025938, 0.079667], [0, 0.644344, 0.442896, 0.029094, 0.047], [0, 0.994516, 0.442771, 0.010969, 0.083458], [1, 0.82582, 0.444885, 0.012109, 0.016146], [1, 0.18918, 0.536865, 0.017234, 0.029521]]}, {"filename": "000000340128.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.419083, 0.305586, 0.832208, 0.440703], [1, 0.947937, 0.514609, 0.104125, 0.332594], [1, 0.728229, 0.099102, 0.531542, 0.195953], [7, 0.398562, 0.168391, 0.191375, 0.336781]]}, {"filename": "000000345617.jpg", "coco_split": "train2017", "split": "val", "boxes": [[3, 0.094453, 0.308219, 0.167844, 0.106729], [4, 0.630937, 0.220781, 0.049875, 0.076604], [4, 0.59943, 0.2125, 0.039203, 0.065708], [0, 0.032758, 0.463438, 0.061859, 0.068708], [4, 0.936813, 0.132302, 0.043313, 0.070604]]}, {"filename": "000000346785.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.169617, 0.6262, 0.027453, 0.060259]]}, {"filename": "000000351428.jpg", "coco_split": "train2017", "split": "val", "boxes": [[3, 0.480148, 0.696636, 0.960297, 0.579766]]}, {"filename": "000000355240.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.081687, 0.255986, 0.039, 0.049361]]}, {"filename": "000000357312.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.279346, 0.632688, 0.24506, 0.373563]]}, {"filename": "000000357748.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.541477, 0.444135, 0.917047, 0.880354], [0, 0.070461, 0.437865, 0.020109, 0.084729], [0, 0.050211, 0.442031, 0.024328, 0.070187], [0, 0.023156, 0.436552, 0.022594, 0.078354], [0, 0.078234, 0.439125, 0.006531, 0.071417]]}, {"filename": "000000361506.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.820898, 0.568802, 0.139734, 0.058479], [0, 0.378367, 0.150552, 0.284828, 0.283146], [6, 0.550344, 0.511698, 0.022625, 0.027562]]}, {"filename": "000000361603.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.751672, 0.524469, 0.293563, 0.391396], [0, 0.224937, 0.334083, 0.018219, 0.055167], [0, 0.156914, 0.372594, 0.031078, 0.050813]]}, {"filename": "000000367063.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.045172, 0.614647, 0.088344, 0.262141]]}, {"filename": "000000376284.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.13439, 0.250225, 0.11946, 0.21973], [1, 0.32548, 0.255646, 0.36532, 0.34991], [1, 0.84771, 0.110736, 0.30258, 0.216907], [1, 0.54893, 0.179189, 0.11114, 0.133874], [1, 0.4817, 0.132192, 0.12388, 0.095676], [1, 0.08408, 0.241622, 0.09332, 0.177417], [5, 0.68321, 0.502252, 0.2679, 0.973033], [1, 0.01973, 0.2553, 0.03878, 0.102613], [1, 0.01933, 0.209264, 0.03866, 0.032462]]}, {"filename": "000000378028.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.847922, 0.645319, 0.093719, 0.211678], [4, 0.952523, 0.665934, 0.094047, 0.226572], [4, 0.543656, 0.754267, 0.07075, 0.303617], [0, 0.403242, 0.610106, 0.105453, 0.550567]]}, {"filename": "000000379024.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.550281, 0.492135, 0.294937, 0.476396]]}, {"filename": "000000379332.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.25257, 0.696961, 0.14326, 0.541766], [0, 0.93061, 0.681841, 0.13878, 0.634521], [0, 0.54941, 0.553937, 0.81362, 0.86518], [4, 0.15687, 0.868982, 0.10658, 0.22024]]}, {"filename": "000000380337.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.51459, 0.523813, 0.97082, 0.753656], [0, 0.574282, 0.747203, 0.203645, 0.371438]]}, {"filename": "000000385425.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.564169, 0.142016, 0.239157, 0.2]]}, {"filename": "000000390298.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.58882, 0.494614, 0.605359, 0.702576], [0, 0.511266, 0.414602, 0.326844, 0.721335]]}, {"filename": "000000394275.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.549203, 0.579784, 0.814156, 0.737074], [0, 0.671461, 0.483219, 0.036734, 0.130712], [0, 0.618398, 0.508511, 0.020703, 0.077176], [0, 0.590133, 0.511094, 0.019641, 0.066972], [0, 0.356617, 0.476883, 0.048234, 0.10883], [0, 0.971406, 0.644338, 0.00775, 0.028677], [0, 0.862289, 0.512099, 0.020422, 0.081654], [0, 0.431836, 0.466094, 0.058016, 0.12374], [0, 0.278844, 0.433562, 0.093688, 0.146107], [0, 0.764719, 0.501921, 0.012125, 0.094478]]}, {"filename": "000000396200.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.626609, 0.473806, 0.051344, 0.046581], [1, 0.985594, 0.507213, 0.028469, 0.182904], [1, 0.801352, 0.479133, 0.029703, 0.06], [0, 0.854891, 0.558981, 0.232469, 0.399227], [1, 0.710641, 0.453162, 0.125188, 0.096674], [0, 0.546469, 0.451768, 0.12775, 0.425644]]}, {"filename": "000000398419.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.383641, 0.727156, 0.098656, 0.265104], [4, 0.773391, 0.851615, 0.153219, 0.270979], [1, 0.62532, 0.736365, 0.303891, 0.319479]]}, {"filename": "000000407365.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.754613, 0.481195, 0.490773, 0.953016], [1, 0.273299, 0.69707, 0.458196, 0.173734]]}, {"filename": "000000413955.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.320641, 0.725854, 0.286813, 0.313125], [0, 0.350211, 0.476094, 0.096172, 0.203313], [0, 0.22557, 0.43849, 0.067578, 0.165813], [0, 0.507305, 0.506625, 0.081422, 0.128375], [0, 0.449891, 0.460396, 0.050594, 0.222292], [0, 0.675844, 0.678083, 0.296625, 0.429208], [0, 0.706195, 0.442125, 0.078234, 0.173208], [0, 0.739523, 0.443073, 0.059391, 0.173854], [0, 0.209172, 0.15624, 0.098781, 0.112771], [0, 0.781578, 0.442219, 0.079969, 0.172146], [0, 0.808734, 0.449052, 0.048969, 0.190063], [3, 0.445898, 0.536573, 0.290266, 0.060354], [4, 0.305844, 0.306531, 0.036281, 0.031229], [4, 0.264484, 0.305135, 0.032437, 0.032271], [4, 0.124797, 0.282052, 0.043562, 0.032271], [4, 0.165852, 0.278667, 0.041484, 0.029667], [4, 0.205648, 0.278958, 0.038641, 0.033875], [4, 0.083664, 0.28225, 0.043859, 0.033875], [4, 0.283617, 0.281771, 0.040266, 0.033958], [4, 0.318727, 0.280104, 0.039609, 0.031292], [4, 0.397172, 0.281562, 0.037344, 0.033667], [4, 0.359563, 0.282677, 0.042469, 0.036188], [4, 0.579641, 0.178812, 0.033156, 0.028208], [0, 0.140727, 0.685385, 0.166859, 0.417979], [4, 0.430844, 0.278208, 0.036844, 0.026083], [0, 0.565297, 0.502656, 0.062719, 0.128854], [3, 0.090289, 0.559073, 0.090109, 0.052187], [4, 0.457734, 0.305448, 0.036156, 0.029562]]}, {"filename": "000000415716.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.712906, 0.794302, 0.291219, 0.408854], [4, 0.073313, 0.770792, 0.146625, 0.161792]]}, {"filename": "000000417248.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.351698, 0.395437, 0.215646, 0.099688], [1, 0.211896, 0.36432, 0.122, 0.056578], [1, 0.857188, 0.3715, 0.274583, 0.075812], [1, 0.094458, 0.352242, 0.062417, 0.032422], [1, 0.056677, 0.347945, 0.029521, 0.025109], [1, 0.522344, 0.484531, 0.181646, 0.163719], [0, 0.515406, 0.432805, 0.181521, 0.222016], [1, 0.13874, 0.346305, 0.046479, 0.025203], [1, 0.753219, 0.365523, 0.169854, 0.055484], [1, 0.501865, 0.774156, 0.985771, 0.424719], [1, 0.673031, 0.362656, 0.183688, 0.055219], [1, 0.577677, 0.331625, 0.208229, 0.081312], [1, 0.024646, 0.362445, 0.049292, 0.093891]]}, {"filename": "000000417608.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.970422, 0.217981, 0.059156, 0.305634], [4, 0.678242, 0.081033, 0.121641, 0.162066], [4, 0.788375, 0.098709, 0.089906, 0.17662], [4, 0.853625, 0.077641, 0.048156, 0.149178], [4, 0.519117, 0.114671, 0.173328, 0.229343]]}, {"filename": "000000420658.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.784625, 0.409906, 0.163844, 0.322063], [1, 0.572688, 0.948396, 0.169969, 0.087167], [1, 0.387516, 0.886198, 0.048469, 0.041562], [1, 0.414523, 0.888198, 0.022297, 0.038312], [1, 0.724414, 0.924656, 0.118359, 0.097604], [1, 0.8995, 0.918062, 0.106188, 0.07775], [1, 0.970898, 0.917031, 0.032359, 0.057021], [1, 0.444117, 0.892865, 0.031016, 0.036979], [1, 0.327773, 0.938271, 0.266672, 0.093833]]}, {"filename": "000000420840.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.499477, 0.50112, 0.445141, 0.966297], [0, 0.896766, 0.295047, 0.205906, 0.583349], [0, 0.497992, 0.064045, 0.343891, 0.12809], [4, 0.235344, 0.923502, 0.111656, 0.152807]]}, {"filename": "000000421254.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.471607, 0.26145, 0.084054, 0.04026], [1, 0.563363, 0.24643, 0.128589, 0.06794], [1, 0.811847, 0.27853, 0.177327, 0.06194], [5, 0.571787, 0.68188, 0.403213, 0.5088], [1, 0.446126, 0.23629, 0.061682, 0.02914], [7, 0.117462, 0.25286, 0.08976, 0.08988], [1, 0.395165, 0.23483, 0.041381, 0.0231], [1, 0.812913, 0.27705, 0.185706, 0.06258]]}, {"filename": "000000424998.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.407578, 0.561673, 0.075594, 0.074943], [1, 0.35018, 0.543527, 0.045422, 0.047015], [1, 0.747828, 0.65692, 0.364969, 0.252319], [1, 0.946063, 0.559819, 0.087844, 0.075418], [1, 0.740609, 0.52924, 0.051531, 0.014563], [1, 0.648828, 0.533983, 0.090937, 0.069601], [7, 0.691836, 0.592766, 0.029953, 0.073327], [0, 0.272445, 0.576549, 0.030922, 0.119373], [0, 0.302766, 0.577063, 0.037937, 0.122414], [0, 0.200563, 0.618992, 0.0625, 0.20943], [1, 0.561141, 0.540941, 0.051187, 0.035418], [1, 0.872187, 0.555551, 0.12625, 0.071255], [1, 0.593664, 0.543108, 0.027266, 0.036825], [1, 0.531961, 0.540941, 0.021328, 0.025342], [1, 0.329164, 0.535884, 0.012297, 0.017243], [1, 0.229445, 0.553165, 0.034484, 0.085875]]}, {"filename": "000000427823.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.334219, 0.581896, 0.144969, 0.197375], [4, 0.235125, 0.596448, 0.056719, 0.291438], [4, 0.035398, 0.565167, 0.060672, 0.1865], [4, 0.479914, 0.443771, 0.044828, 0.041125], [4, 0.089945, 0.561823, 0.057172, 0.269063], [0, 0.540453, 0.507865, 0.919094, 0.961813], [0, 0.815734, 0.335948, 0.050563, 0.078646], [0, 0.420305, 0.352479, 0.052266, 0.215583], [0, 0.519461, 0.338396, 0.059109, 0.143583], [0, 0.465445, 0.376458, 0.029141, 0.050792], [0, 0.490219, 0.375323, 0.023062, 0.039812], [0, 0.57968, 0.328781, 0.089016, 0.209187], [0, 0.051492, 0.385229, 0.100203, 0.191208], [0, 0.386781, 0.37301, 0.039062, 0.168646], [0, 0.915453, 0.383146, 0.169094, 0.465167], [0, 0.33257, 0.451052, 0.179203, 0.327521]]}, {"filename": "000000429623.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.613187, 0.16075, 0.379969, 0.318458]]}, {"filename": "000000429761.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.387727, 0.544953, 0.189359, 0.725841], [0, 0.552281, 0.486519, 0.166812, 0.694393]]}, {"filename": "000000432444.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.513172, 0.25309, 0.196219, 0.386745], [0, 0.249648, 0.505932, 0.425891, 0.589693], [0, 0.240289, 0.496757, 0.336391, 0.75875], [0, 0.104445, 0.305177, 0.146234, 0.610354], [0, 0.211195, 0.165767, 0.143766, 0.322146], [0, 0.678133, 0.217972, 0.168234, 0.435943], [0, 0.906602, 0.253373, 0.186797, 0.5], [0, 0.73193, 0.576769, 0.388953, 0.603396], [0, 0.573164, 0.15441, 0.132109, 0.308821], [0, 0.418344, 0.119104, 0.175688, 0.229198], [0, 0.849477, 0.130177, 0.171422, 0.260354]]}, {"filename": "000000437224.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.264031, 0.111206, 0.139125, 0.219742], [0, 0.171672, 0.121347, 0.073469, 0.130328], [0, 0.38168, 0.141827, 0.086172, 0.086557], [0, 0.342102, 0.120878, 0.054016, 0.150281], [0, 0.267273, 0.214297, 0.091422, 0.064473], [4, 0.910953, 0.74308, 0.178094, 0.513841], [0, 0.494109, 0.191323, 0.076063, 0.0863], [0, 0.405109, 0.112494, 0.031781, 0.05445], [4, 0.752695, 0.4976, 0.249141, 0.520632], [4, 0.656797, 0.286792, 0.135344, 0.131101], [4, 0.485227, 0.288314, 0.259953, 0.119485], [4, 0.358133, 0.27363, 0.108484, 0.150211], [4, 0.65282, 0.24171, 0.142328, 0.055972], [4, 0.441852, 0.663642, 0.723078, 0.672717], [4, 0.590602, 0.214883, 0.076797, 0.056792], [0, 0.384977, 0.107857, 0.024047, 0.079883], [0, 0.45968, 0.13329, 0.039203, 0.078337], [0, 0.15025, 0.333782, 0.165781, 0.234778], [4, 0.906078, 0.672436, 0.187844, 0.634567]]}, {"filename": "000000441608.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.624906, 0.442853, 0.291531, 0.624709], [3, 0.638336, 0.568255, 0.473578, 0.543712]]}, {"filename": "000000455624.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.539922, 0.620199, 0.519594, 0.586909], [0, 0.03775, 0.480984, 0.0755, 0.393349], [0, 0.25025, 0.424169, 0.052844, 0.080468], [0, 0.218562, 0.43589, 0.035375, 0.052342], [0, 0.752055, 0.388794, 0.055109, 0.128407], [0, 0.724672, 0.368607, 0.079906, 0.163864], [0, 0.679383, 0.370504, 0.051391, 0.113466], [0, 0.628906, 0.329543, 0.034625, 0.080867], [0, 0.848516, 0.387763, 0.065031, 0.153607], [0, 0.944602, 0.410234, 0.074641, 0.143232], [0, 0.579383, 0.453981, 0.297047, 0.63363], [0, 0.816039, 0.383232, 0.084297, 0.15541], [0, 0.144703, 0.412834, 0.102469, 0.255269], [0, 0.980109, 0.409356, 0.039781, 0.147845]]}, {"filename": "000000458768.jpg", "coco_split": "val2017", "split": "val", "boxes": [[3, 0.491344, 0.565972, 0.101125, 0.055785]]}, {"filename": "000000459912.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.219766, 0.321522, 0.021125, 0.058829], [0, 0.073633, 0.350222, 0.096172, 0.29281], [0, 0.642805, 0.325422, 0.048891, 0.15459], [0, 0.94818, 0.34733, 0.052078, 0.245433], [1, 0.440047, 0.732576, 0.880094, 0.512365], [0, 0.704695, 0.346112, 0.032859, 0.104684], [0, 0.130719, 0.353724, 0.033594, 0.141967], [0, 0.544461, 0.33507, 0.016453, 0.058571], [0, 0.596672, 0.333981, 0.015875, 0.052506], [0, 0.157781, 0.343525, 0.06725, 0.186815], [0, 0.330578, 0.334707, 0.025094, 0.058782], [0, 0.580586, 0.340574, 0.021297, 0.071218], [1, 0.452547, 0.605187, 0.329844, 0.201616], [1, 0.702766, 0.454169, 0.144187, 0.273208], [1, 0.910961, 0.591183, 0.178078, 0.547424], [1, 0.360031, 0.470913, 0.2035, 0.09007], [1, 0.356211, 0.464836, 0.106453, 0.070187], [1, 0.4175, 0.419192, 0.085219, 0.071499], [1, 0.802117, 0.506487, 0.192891, 0.315972], [1, 0.513977, 0.674075, 0.431609, 0.227166], [1, 0.406117, 0.525574, 0.352984, 0.13274], [1, 0.387266, 0.5, 0.249375, 0.111382], [1, 0.498586, 0.386956, 0.114422, 0.139578], [0, 0.897906, 0.343384, 0.105375, 0.256557], [0, 0.197172, 0.340012, 0.016281, 0.050141], [1, 0.615742, 0.44671, 0.101516, 0.223489]]}, {"filename": "000000460229.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.44111, 0.301109, 0.613081, 0.351219], [1, 0.79453, 0.593695, 0.401436, 0.249797]]}, {"filename": "000000460813.jpg", "coco_split": "train2017", "split": "val", "boxes": [[7, 0.745728, 0.145633, 0.259882, 0.210766]]}, {"filename": "000000465537.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.492281, 0.301719, 0.984562, 0.515219], [7, 0.741719, 0.257758, 0.241104, 0.403391], [1, 0.697698, 0.048195, 0.067438, 0.027703], [1, 0.075302, 0.040391, 0.150604, 0.0615], [1, 0.503375, 0.062922, 0.341583, 0.125844], [1, 0.731198, 0.050102, 0.060062, 0.034891], [1, 0.890729, 0.085453, 0.218542, 0.170844], [1, 0.178344, 0.030969, 0.109646, 0.033719]]}, {"filename": "000000468505.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.437914, 0.315854, 0.496641, 0.407167], [1, 0.634492, 0.267969, 0.234828, 0.159021]]}, {"filename": "000000469192.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.636234, 0.661427, 0.719656, 0.384271], [0, 0.156438, 0.608781, 0.009719, 0.035021], [0, 0.115078, 0.612792, 0.009969, 0.020667]]}, {"filename": "000000470762.jpg", "coco_split": "train2017", "split": "val", "boxes": [[5, 0.470417, 0.506367, 0.629208, 0.815734]]}, {"filename": "000000472678.jpg", "coco_split": "val2017", "split": "val", "boxes": [[4, 0.963711, 0.875271, 0.069359, 0.223667], [0, 0.358055, 0.521687, 0.017234, 0.022], [0, 0.38475, 0.522094, 0.017219, 0.024229], [0, 0.38975, 0.445604, 0.016812, 0.023542], [0, 0.390266, 0.408583, 0.018375, 0.022833]]}, {"filename": "000000483999.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.100687, 0.720654, 0.201375, 0.525841], [0, 0.266594, 0.668925, 0.189781, 0.558551], [0, 0.379453, 0.366297, 0.13375, 0.238201], [0, 0.711, 0.548201, 0.172281, 0.445888], [0, 0.86443, 0.520175, 0.217422, 0.215257], [0, 0.73068, 0.779778, 0.538641, 0.413481], [0, 0.628289, 0.411811, 0.117391, 0.272453], [0, 0.250219, 0.381659, 0.145781, 0.22472], [0, 0.024281, 0.395794, 0.048125, 0.187757], [0, 0.737023, 0.35875, 0.138672, 0.176005], [0, 0.596875, 0.331086, 0.056, 0.100818], [4, 0.47818, 0.344089, 0.037328, 0.104299], [4, 0.895789, 0.386612, 0.048766, 0.085561], [4, 0.78843, 0.375736, 0.035297, 0.109416], [4, 0.54775, 0.340164, 0.029094, 0.063037], [4, 0.853445, 0.392477, 0.041078, 0.084299], [4, 0.043883, 0.409778, 0.026922, 0.13764], [4, 0.433859, 0.377967, 0.0485, 0.157009]]}, {"filename": "000000485680.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.546977, 0.263186, 0.172255, 0.176732], [1, 0.060727, 0.562786, 0.040278, 0.030637], [1, 0.132949, 0.533399, 0.065049, 0.021699], [1, 0.123227, 0.511503, 0.021062, 0.019542], [1, 0.145212, 0.461332, 0.015588, 0.012141], [0, 0.011936, 0.609828, 0.020212, 0.077239], [0, 0.348399, 0.574444, 0.02415, 0.065621], [1, 0.151912, 0.547753, 0.033105, 0.024101], [1, 0.194322, 0.56009, 0.043252, 0.018284], [1, 0.198039, 0.587557, 0.062941, 0.058284], [1, 0.14143, 0.47, 0.016683, 0.013693]]}, {"filename": "000000486536.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.533797, 0.495604, 0.932406, 0.987125]]}, {"filename": "000000496959.jpg", "coco_split": "train2017", "split": "val", "boxes": [[3, 0.443641, 0.585025, 0.752063, 0.451679]]}, {"filename": "000000497807.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.795148, 0.700047, 0.012922, 0.039767], [0, 0.690664, 0.689161, 0.009641, 0.037529], [0, 0.670789, 0.692226, 0.010547, 0.035268], [0, 0.653625, 0.692681, 0.007969, 0.034126], [0, 0.640109, 0.690478, 0.007469, 0.031538], [0, 0.584195, 0.691655, 0.013328, 0.046387], [0, 0.593484, 0.69366, 0.007969, 0.036643], [0, 0.415672, 0.686573, 0.005438, 0.02345], [0, 0.366016, 0.692646, 0.005563, 0.028695], [0, 0.460922, 0.690501, 0.005406, 0.025058], [0, 0.491656, 0.685396, 0.008063, 0.027483], [0, 0.405023, 0.687436, 0.004953, 0.019767], [0, 0.4765, 0.674697, 0.003406, 0.016527]]}, {"filename": "000000497819.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.459711, 0.387829, 0.020984, 0.032976], [1, 0.919305, 0.511549, 0.141984, 0.129488], [1, 0.166086, 0.418244, 0.037641, 0.043854], [1, 0.291609, 0.419244, 0.059531, 0.037415], [1, 0.340023, 0.430378, 0.030484, 0.042073], [0, 0.483719, 0.693427, 0.20375, 0.479976], [0, 0.105523, 0.414354, 0.011266, 0.039146], [0, 0.469602, 0.452963, 0.011391, 0.060902], [0, 0.14468, 0.426573, 0.009828, 0.051488], [0, 0.051758, 0.421488, 0.019328, 0.076], [0, 0.187594, 0.421537, 0.010563, 0.047024], [1, 0.241687, 0.408073, 0.012688, 0.011122], [7, 0.790289, 0.517549, 0.014234, 0.049976], [7, 0.696992, 0.491646, 0.011953, 0.043341], [1, 0.979906, 0.549963, 0.040188, 0.084317], [1, 0.946156, 0.518354, 0.107594, 0.145439], [0, 0.493633, 0.438683, 0.016703, 0.040634], [1, 0.324234, 0.421976, 0.01025, 0.02678]]}, {"filename": "000000497847.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.505055, 0.477422, 0.266453, 0.490335], [0, 0.060406, 0.434944, 0.109125, 0.200737], [0, 0.464305, 0.610357, 0.260141, 0.759018], [0, 0.742555, 0.557165, 0.193141, 0.519464], [0, 0.902961, 0.560871, 0.158172, 0.504643], [0, 0.980641, 0.428549, 0.0375, 0.13875], [3, 0.176563, 0.676016, 0.139687, 0.136763], [0, 0.356063, 0.417154, 0.07725, 0.178772], [3, 0.429437, 0.808996, 0.440437, 0.359554], [3, 0.968664, 0.609475, 0.062672, 0.049799], [0, 0.270914, 0.485614, 0.115172, 0.299174], [0, 0.654875, 0.446105, 0.036062, 0.128504], [3, 0.127273, 0.757143, 0.189078, 0.075313], [3, 0.819094, 0.779219, 0.361812, 0.327277], [3, 0.58082, 0.724319, 0.310297, 0.066763], [3, 0.645914, 0.604721, 0.027016, 0.014978], [3, 0.056211, 0.644877, 0.112422, 0.040067], [0, 0.144547, 0.470134, 0.18, 0.311696]]}, {"filename": "000000502358.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.865648, 0.242159, 0.040047, 0.052203], [0, 0.790063, 0.252247, 0.075094, 0.087841], [0, 0.814266, 0.627059, 0.084875, 0.158524], [0, 0.825195, 0.744945, 0.349609, 0.487643], [0, 0.721336, 0.646068, 0.062172, 0.186498], [0, 0.623516, 0.645022, 0.083312, 0.129471], [0, 0.389734, 0.719846, 0.0905, 0.396476], [0, 0.32293, 0.646597, 0.062953, 0.131167], [0, 0.262906, 0.715804, 0.077719, 0.269009], [0, 0.676695, 0.680011, 0.130172, 0.206894], [0, 0.229484, 0.654967, 0.047812, 0.122093], [0, 0.149453, 0.743106, 0.111156, 0.325022], [3, 0.755398, 0.891718, 0.039078, 0.074317], [4, 0.916914, 0.380363, 0.044328, 0.026586], [4, 0.866305, 0.415947, 0.045203, 0.039207], [0, 0.476297, 0.571432, 0.392469, 0.84674], [4, 0.983266, 0.440275, 0.033469, 0.037203], [4, 0.808125, 0.433172, 0.041625, 0.061938], [4, 0.914055, 0.421575, 0.045641, 0.029978]]}, {"filename": "000000504850.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.431117, 0.373943, 0.053734, 0.086134], [0, 0.473805, 0.55518, 0.087391, 0.283763]]}, {"filename": "000000505862.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.48843, 0.498415, 0.139141, 0.153592], [4, 0.333531, 0.563181, 0.241531, 0.215986], [0, 0.597703, 0.495082, 0.796938, 0.975047], [0, 0.371008, 0.313592, 0.200547, 0.283568], [0, 0.47325, 0.312617, 0.078094, 0.218286], [0, 0.213148, 0.299448, 0.157047, 0.175282], [0, 0.229852, 0.197477, 0.060297, 0.14007], [0, 0.149289, 0.206444, 0.106953, 0.321385], [0, 0.608031, 0.311009, 0.145094, 0.278638], [1, 0.492477, 0.219589, 0.199391, 0.189178], [0, 0.548398, 0.240164, 0.029828, 0.080329], [0, 0.519539, 0.261667, 0.040641, 0.121033], [0, 0.572516, 0.247958, 0.038344, 0.091455]]}, {"filename": "000000505942.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.15798, 0.482067, 0.14484, 0.4284], [0, 0.07966, 0.47916, 0.09088, 0.394373], [1, 0.95946, 0.4916, 0.081, 0.1136]]}, {"filename": "000000506933.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.5, 0.495135, 1.0, 0.977521], [4, 0.920586, 0.496896, 0.158828, 0.322083]]}, {"filename": "000000507830.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.732437, 0.584354, 0.341781, 0.318125], [7, 0.74725, 0.437917, 0.045281, 0.125333], [7, 0.683531, 0.433948, 0.044219, 0.118938]]}, {"filename": "000000508730.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.730133, 0.614351, 0.334922, 0.739916], [4, 0.704914, 0.888766, 0.409516, 0.222469], [0, 0.259547, 0.612071, 0.421406, 0.775858]]}, {"filename": "000000508815.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.298746, 0.37303, 0.537358, 0.7281]]}, {"filename": "000000519055.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.553055, 0.135873, 0.700891, 0.264568], [0, 0.502375, 0.559683, 0.832406, 0.8581], [0, 0.88332, 0.489194, 0.232859, 0.386065], [0, 0.061281, 0.46404, 0.122562, 0.352802], [0, 0.398656, 0.486334, 0.125812, 0.405873], [0, 0.184812, 0.463896, 0.213031, 0.328791], [0, 0.700414, 0.489837, 0.050078, 0.184549], [1, 0.955023, 0.205269, 0.089953, 0.097985], [4, 0.708695, 0.541046, 0.069953, 0.279251], [4, 0.198836, 0.566689, 0.077922, 0.120403]]}, {"filename": "000000520532.jpg", "coco_split": "train2017", "split": "val", "boxes": [[3, 0.477969, 0.750565, 0.76475, 0.39551]]}, {"filename": "000000521849.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.862459, 0.458156, 0.274707, 0.131812], [5, 0.434227, 0.623258, 0.486206, 0.642047], [0, 0.253689, 0.466109, 0.055902, 0.081281], [0, 0.730527, 0.409953, 0.042787, 0.041875], [0, 0.2076, 0.468523, 0.03993, 0.079922], [0, 0.238138, 0.444242, 0.024707, 0.034797], [0, 0.274485, 0.431969, 0.019274, 0.018375], [0, 0.720937, 0.457258, 0.034988, 0.084766], [0, 0.23171, 0.478797, 0.026417, 0.038844]]}, {"filename": "000000523638.jpg", "coco_split": "train2017", "split": "val", "boxes": [[6, 0.546547, 0.470748, 0.439344, 0.730935]]}, {"filename": "000000525438.jpg", "coco_split": "train2017", "split": "val", "boxes": [[7, 0.11132, 0.65039, 0.03856, 0.137417], [0, 0.4282, 0.647598, 0.05076, 0.192072], [7, 0.07243, 0.654925, 0.03602, 0.142943], [0, 0.24245, 0.728093, 0.17362, 0.503393], [0, 0.72009, 0.593288, 0.0367, 0.118108], [1, 0.62871, 0.672658, 0.30946, 0.148198]]}, {"filename": "000000526070.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.371106, 0.486531, 0.504894, 0.879375], [0, 0.781965, 0.327664, 0.1844, 0.442172], [0, 0.614894, 0.337891, 0.209553, 0.446125], [5, 0.716518, 0.713711, 0.375482, 0.457172]]}, {"filename": "000000528962.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.81321, 0.676827, 0.10062, 0.34992], [1, 0.96207, 0.627093, 0.07586, 0.226133], [0, 0.80983, 0.540453, 0.1399, 0.55504], [1, 0.27074, 0.446187, 0.54148, 0.721973], [0, 0.95706, 0.572133, 0.05604, 0.32624], [0, 0.12338, 0.26616, 0.1136, 0.120533], [3, 0.79214, 0.447747, 0.37416, 0.47192], [1, 0.62524, 0.496493, 0.04952, 0.126427], [3, 0.78409, 0.167533, 0.36818, 0.04936]]}, {"filename": "000000531707.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.332727, 0.63651, 0.135578, 0.382896], [0, 0.455328, 0.631271, 0.125, 0.378375], [0, 0.672242, 0.646292, 0.143797, 0.358], [3, 0.515727, 0.708979, 0.461797, 0.267417], [0, 0.565875, 0.62275, 0.094594, 0.324333]]}, {"filename": "000000535424.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.484656, 0.516495, 0.086937, 0.188879]]}, {"filename": "000000536806.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.563109, 0.435218, 0.031844, 0.091476], [0, 0.445953, 0.407734, 0.014063, 0.048524], [0, 0.48182, 0.410166, 0.014734, 0.048274], [0, 0.614102, 0.421403, 0.006391, 0.010416], [3, 0.26175, 0.604366, 0.084219, 0.04815], [0, 0.606492, 0.419792, 0.005203, 0.009023], [4, 0.075109, 0.512765, 0.101969, 0.079127], [4, 0.353977, 0.550811, 0.113078, 0.092848], [4, 0.655641, 0.649044, 0.162844, 0.142786], [4, 0.545, 0.530624, 0.121875, 0.103867], [4, 0.162023, 0.635936, 0.312328, 0.264927], [4, 0.287977, 0.50262, 0.164984, 0.049023], [4, 0.082375, 0.58474, 0.16475, 0.045946], [4, 0.013852, 0.517921, 0.025734, 0.095509], [0, 0.301945, 0.399709, 0.007359, 0.007443], [0, 0.211406, 0.388659, 0.004219, 0.005634], [0, 0.388703, 0.393358, 0.002969, 0.0058]]}, {"filename": "000000538364.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.864516, 0.395698, 0.106438, 0.184937], [1, 0.517094, 0.42399, 0.216938, 0.271312], [1, 0.687641, 0.587635, 0.434844, 0.451688], [0, 0.101164, 0.406583, 0.060703, 0.303708], [0, 0.963648, 0.479031, 0.072703, 0.454437], [3, 0.202094, 0.382625, 0.083437, 0.067167], [0, 0.439945, 0.320146, 0.034703, 0.109417], [0, 0.564242, 0.272896, 0.013672, 0.029542], [0, 0.584359, 0.276969, 0.009187, 0.022313], [0, 0.608625, 0.282302, 0.01375, 0.019979], [0, 0.632906, 0.276573, 0.014438, 0.018021], [0, 0.548781, 0.276146, 0.010844, 0.019542], [0, 0.691188, 0.544646, 0.20975, 0.502792], [0, 0.033719, 0.400427, 0.067437, 0.334813], [1, 0.781023, 0.312063, 0.051734, 0.025583], [1, 0.809734, 0.41274, 0.118906, 0.177229]]}, {"filename": "000000542651.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.785289, 0.602393, 0.194828, 0.714603], [0, 0.930141, 0.639328, 0.139719, 0.622485], [0, 0.212914, 0.534837, 0.298266, 0.817984], [3, 0.52582, 0.603371, 0.372391, 0.55057]]}, {"filename": "000000544410.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.809961, 0.649227, 0.337641, 0.300578], [1, 0.573359, 0.495391, 0.017531, 0.011031], [7, 0.689023, 0.594227, 0.034172, 0.094953], [7, 0.668086, 0.593727, 0.025391, 0.091172], [1, 0.605078, 0.512656, 0.015906, 0.014313], [1, 0.587844, 0.485023, 0.015562, 0.008266]]}, {"filename": "000000546717.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.430541, 0.551312, 0.757261, 0.869656]]}, {"filename": "000000547336.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.446047, 0.482854, 0.580125, 0.861083], [5, 0.863352, 0.546135, 0.273297, 0.421063]]}, {"filename": "000000549220.jpg", "coco_split": "val2017", "split": "val", "boxes": [[0, 0.662385, 0.181148, 0.291229, 0.353328], [0, 0.873031, 0.210117, 0.253937, 0.316859], [0, 0.907927, 0.176789, 0.184146, 0.326516], [0, 0.960115, 0.114727, 0.079771, 0.118578], [3, 0.811063, 0.291125, 0.119083, 0.125688]]}, {"filename": "000000549539.jpg", "coco_split": "train2017", "split": "val", "boxes": [[0, 0.548594, 0.350187, 0.591563, 0.687625], [0, 0.202641, 0.290615, 0.401906, 0.576229], [4, 0.679078, 0.360854, 0.616344, 0.70425], [4, 0.267883, 0.415771, 0.240391, 0.411542]]}, {"filename": "000000552504.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.230359, 0.199652, 0.258625, 0.207161], [0, 0.373523, 0.609804, 0.156266, 0.602321], [7, 0.654703, 0.14858, 0.0305, 0.097446], [7, 0.032711, 0.154938, 0.020203, 0.090125]]}, {"filename": "000000558213.jpg", "coco_split": "val2017", "split": "val", "boxes": [[1, 0.232664, 0.701412, 0.071328, 0.064954], [0, 0.444602, 0.658935, 0.091578, 0.317824], [0, 0.228562, 0.66478, 0.041375, 0.172292]]}, {"filename": "000000564352.jpg", "coco_split": "train2017", "split": "val", "boxes": [[3, 0.975977, 0.549146, 0.048047, 0.036625], [0, 0.176969, 0.670792, 0.350562, 0.631458], [0, 0.640125, 0.522406, 0.03875, 0.081938], [0, 0.602781, 0.532573, 0.020812, 0.034854], [0, 0.488438, 0.545792, 0.019937, 0.046917], [0, 0.452148, 0.535281, 0.035766, 0.063021], [0, 0.666328, 0.519813, 0.018531, 0.041708], [0, 0.567992, 0.536167, 0.021703, 0.035333], [0, 0.617508, 0.52525, 0.015078, 0.01825], [0, 0.85132, 0.532958, 0.028547, 0.079667], [0, 0.884203, 0.531885, 0.037625, 0.087813], [4, 0.541242, 0.795854, 0.062891, 0.170833], [4, 0.825211, 0.551406, 0.023922, 0.050854], [4, 0.930148, 0.539188, 0.024734, 0.055958], [4, 0.62282, 0.542792, 0.016922, 0.016167], [4, 0.98675, 0.536229, 0.02525, 0.056042], [4, 0.757297, 0.549875, 0.021125, 0.046], [4, 0.825695, 0.519865, 0.019391, 0.010187], [4, 0.895227, 0.547229, 0.047141, 0.058833], [4, 0.957609, 0.552021, 0.028531, 0.035042], [4, 0.943422, 0.546969, 0.019375, 0.033187], [4, 0.592203, 0.548073, 0.013594, 0.025562], [0, 0.458578, 0.556562, 0.024906, 0.025875], [0, 0.909945, 0.503979, 0.017578, 0.030125], [3, 0.847773, 0.346875, 0.030172, 0.029708], [0, 0.456891, 0.571375, 0.063875, 0.078833], [4, 0.315695, 0.644719, 0.057672, 0.023729], [4, 0.33568, 0.722115, 0.078078, 0.156146]]}, {"filename": "000000568668.jpg", "coco_split": "train2017", "split": "val", "boxes": [[1, 0.157969, 0.642506, 0.071906, 0.050116], [1, 0.283969, 0.631323, 0.01975, 0.020046], [1, 0.086266, 0.638956, 0.06325, 0.047657], [1, 0.244992, 0.631206, 0.033391, 0.029234], [1, 0.200586, 0.636705, 0.027078, 0.029977], [0, 0.376094, 0.728387, 0.158531, 0.500812], [1, 0.300227, 0.63116, 0.019016, 0.032947], [5, 0.304414, 0.714292, 0.011734, 0.060557]]}, {"filename": "000000579893.jpg", "coco_split": "val2017", "split": "val", "boxes": [[6, 0.84671, 0.684789, 0.27926, 0.608494]]}, {"filename": "000000580120.jpg", "coco_split": "train2017", "split": "val", "boxes": [[4, 0.311133, 0.914167, 0.158828, 0.171667], [0, 0.301258, 0.66674, 0.150109, 0.378062], [0, 0.447867, 0.65575, 0.054203, 0.111], [0, 0.463305, 0.679344, 0.122453, 0.140021], [0, 0.621172, 0.68001, 0.032469, 0.056187], [0, 0.525836, 0.641573, 0.037078, 0.069646], [0, 0.803797, 0.75749, 0.08925, 0.158271], [0, 0.163477, 0.796635, 0.239328, 0.379771], [0, 0.655125, 0.646198, 0.048, 0.058688], [0, 0.540555, 0.716917, 0.129141, 0.152458], [0, 0.590367, 0.760302, 0.208016, 0.179396], [0, 0.142883, 0.612, 0.140828, 0.188625], [0, 0.211375, 0.734469, 0.094656, 0.177271], [4, 0.406836, 0.88975, 0.051672, 0.141042], [4, 0.26475, 0.868521, 0.050563, 0.131917], [0, 0.732133, 0.729229, 0.082828, 0.143542]]}, {"filename": "000000002347.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.187016, 0.629875, 0.374031, 0.454542]]}, {"filename": "000000007228.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.6835, 0.496507, 0.04548, 0.05808], [1, 0.27312, 0.753133, 0.08712, 0.246533], [1, 0.17785, 0.492053, 0.19258, 0.201813], [1, 0.37082, 0.486653, 0.09628, 0.132507], [1, 0.51994, 0.45168, 0.08596, 0.05392], [1, 0.42037, 0.477333, 0.06826, 0.105493], [1, 0.72903, 0.41512, 0.04526, 0.04224], [1, 0.49169, 0.425133, 0.0489, 0.041947], [1, 0.02572, 0.46128, 0.05144, 0.096], [0, 0.36738, 0.758973, 0.1438, 0.482053], [0, 0.29039, 0.58612, 0.09738, 0.350853], [0, 0.78097, 0.465987, 0.05198, 0.137467], [0, 0.47623, 0.54868, 0.07702, 0.214213], [0, 0.74412, 0.7074, 0.2102, 0.57624], [0, 0.58989, 0.782027, 0.1517, 0.413493], [0, 0.06068, 0.60636, 0.09776, 0.408987], [0, 0.98039, 0.491373, 0.03922, 0.0796], [0, 0.68374, 0.45032, 0.04576, 0.135093], [1, 0.47461, 0.62384, 0.07518, 0.156533], [7, 0.4274, 0.41012, 0.02404, 0.0852], [0, 0.20992, 0.445387, 0.02404, 0.038773], [0, 0.35326, 0.458, 0.0184, 0.024533], [1, 0.05775, 0.763907, 0.10462, 0.245627], [1, 0.98141, 0.483973, 0.03718, 0.0992], [1, 0.84189, 0.41724, 0.0151, 0.017573], [1, 0.56926, 0.429707, 0.05904, 0.031467], [1, 0.80654, 0.427907, 0.01792, 0.02696], [1, 0.05781, 0.4108, 0.11562, 0.0512]]}, {"filename": "000000009556.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.460656, 0.529282, 0.095719, 0.101341], [0, 0.808812, 0.493259, 0.382031, 0.982], [0, 0.683422, 0.326494, 0.196531, 0.526918], [0, 0.184375, 0.563765, 0.36875, 0.848753]]}, {"filename": "000000015440.jpg", "coco_split": "val2017", "split": "test", "boxes": [[6, 0.42755, 0.264937, 0.280149, 0.168094], [1, 0.562611, 0.346484, 0.173045, 0.121906], [1, 0.319703, 0.347969, 0.077079, 0.096375]]}, {"filename": "000000015746.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.493653, 0.603961, 0.652412, 0.590109]]}, {"filename": "000000020013.jpg", "coco_split": "train2017", "split": "test", "boxes": [[3, 0.594301, 0.455211, 0.554113, 0.117828]]}, {"filename": "000000020107.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.681802, 0.58109, 0.636396, 0.8139]]}, {"filename": "000000021711.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.863117, 0.74695, 0.186984, 0.255905], [1, 0.919984, 0.500265, 0.160031, 0.269777], [1, 0.175469, 0.633969, 0.345313, 0.414568], [1, 0.573078, 0.63429, 0.305938, 0.369972], [0, 0.14507, 0.550655, 0.130578, 0.612061], [0, 0.588609, 0.346574, 0.059187, 0.104011], [0, 0.679586, 0.407618, 0.036922, 0.16649], [0, 0.598055, 0.451922, 0.173703, 0.190139], [0, 0.81832, 0.51337, 0.062266, 0.296212], [4, 0.805586, 0.560724, 0.095672, 0.178719], [4, 0.772648, 0.550669, 0.015016, 0.049694]]}, {"filename": "000000025102.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.676838, 0.892386, 0.252565, 0.176601], [1, 0.503366, 0.453137, 0.97085, 0.594183], [0, 0.927239, 0.842606, 0.10719, 0.278186], [0, 0.186985, 0.896454, 0.262467, 0.166863], [0, 0.547557, 0.913717, 0.101324, 0.128546]]}, {"filename": "000000029984.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.517258, 0.741148, 0.234797, 0.299654], [0, 0.23975, 0.474573, 0.085219, 0.071911], [0, 0.377898, 0.469522, 0.071703, 0.091809], [0, 0.456273, 0.379197, 0.029828, 0.046402], [4, 0.700688, 0.739583, 0.152156, 0.294289], [0, 0.539687, 0.359309, 0.042, 0.082683]]}, {"filename": "000000034847.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.466445, 0.374937, 0.154359, 0.588375]]}, {"filename": "000000035825.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.148133, 0.665927, 0.296266, 0.435937], [0, 0.348773, 0.577552, 0.364016, 0.841604]]}, {"filename": "000000036012.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.611578, 0.503569, 0.380031, 0.684583], [1, 0.123875, 0.535958, 0.242688, 0.573028], [1, 0.323602, 0.530333, 0.237641, 0.588778], [1, 0.86018, 0.623222, 0.274297, 0.523611], [1, 0.862594, 0.615847, 0.274813, 0.510417]]}, {"filename": "000000041305.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.543539, 0.412365, 0.315172, 0.379771], [0, 0.885977, 0.463969, 0.146953, 0.518021], [0, 0.552766, 0.322948, 0.138469, 0.449396], [0, 0.729359, 0.454927, 0.129219, 0.461771], [0, 0.969883, 0.436365, 0.059672, 0.156646], [0, 0.33143, 0.295438, 0.025359, 0.082667], [0, 0.200258, 0.33124, 0.020266, 0.043354], [0, 0.171742, 0.361896, 0.025547, 0.058333], [0, 0.138867, 0.399552, 0.057766, 0.138313], [0, 0.830805, 0.323573, 0.057141, 0.185521], [3, 0.117141, 0.440958, 0.222094, 0.062], [3, 0.36532, 0.399354, 0.110828, 0.043875], [3, 0.292086, 0.400271, 0.074078, 0.03025], [0, 0.301086, 0.343052, 0.039391, 0.065813], [0, 0.27125, 0.350552, 0.0405, 0.078937], [3, 0.028844, 0.404833, 0.051469, 0.0115], [0, 0.980508, 0.343042, 0.038984, 0.090917], [3, 0.228633, 0.395177, 0.021766, 0.033479], [3, 0.372359, 0.371521, 0.165375, 0.030792]]}, {"filename": "000000042070.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.534828, 0.464043, 0.731687, 0.856172], [1, 0.882742, 0.611045, 0.041422, 0.011934], [0, 0.023719, 0.655088, 0.047438, 0.111113]]}, {"filename": "000000043353.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.525842, 0.27768, 0.458415, 0.47415], [5, 0.384788, 0.930866, 0.107386, 0.089477], [1, 0.153538, 0.75509, 0.252696, 0.158513]]}, {"filename": "000000045494.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.514039, 0.51895, 0.828828, 0.513285], [0, 0.06207, 0.504823, 0.035266, 0.116466], [0, 0.028813, 0.498971, 0.024375, 0.083264], [0, 0.972047, 0.562734, 0.055906, 0.246507]]}, {"filename": "000000046230.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.484617, 0.688056, 0.957203, 0.603607]]}, {"filename": "000000048504.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.870766, 0.787541, 0.089563, 0.320141], [3, 0.119383, 0.483279, 0.238766, 0.023466], [3, 0.165781, 0.429508, 0.331562, 0.022576], [0, 0.050484, 0.6476, 0.052469, 0.121897]]}, {"filename": "000000051610.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.731367, 0.585222, 0.382109, 0.70918]]}, {"filename": "000000064621.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.72107, 0.363292, 0.015484, 0.030333], [0, 0.502148, 0.371833, 0.029359, 0.091542], [0, 0.593617, 0.367958, 0.021047, 0.053042], [0, 0.69357, 0.353927, 0.016641, 0.037354], [0, 0.04557, 0.370865, 0.034172, 0.132437], [0, 0.078961, 0.390354, 0.027328, 0.142458], [0, 0.32875, 0.353594, 0.030938, 0.111354], [6, 0.150094, 0.170854, 0.046406, 0.029833], [0, 0.414695, 0.390323, 0.084953, 0.217188], [0, 0.288422, 0.357729, 0.038406, 0.131208], [0, 0.948422, 0.392583, 0.015938, 0.060708]]}, {"filename": "000000067534.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.487549, 0.5, 0.956275, 1.0], [4, 0.939886, 0.670656, 0.120229, 0.223625]]}, {"filename": "000000072852.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.802465, 0.540828, 0.072335, 0.120313], [0, 0.730248, 0.54143, 0.084458, 0.117609], [1, 0.499469, 0.814477, 0.368278, 0.241172]]}, {"filename": "000000073731.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.1225, 0.880062, 0.243875, 0.238656], [1, 0.106594, 0.594406, 0.211854, 0.149844], [1, 0.954208, 0.549414, 0.0755, 0.027453], [1, 0.889385, 0.541344, 0.048979, 0.013469], [7, 0.789708, 0.741094, 0.201958, 0.358562], [1, 0.938927, 0.535586, 0.049729, 0.016016], [1, 0.851948, 0.535398, 0.037354, 0.014859], [5, 0.983937, 0.609883, 0.032125, 0.061734], [1, 0.988031, 0.556055, 0.023938, 0.024203]]}, {"filename": "000000074309.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.224341, 0.336906, 0.10002, 0.123031], [0, 0.059521, 0.330461, 0.028184, 0.058516], [0, 0.511307, 0.320273, 0.212156, 0.155016], [0, 0.033812, 0.33168, 0.031737, 0.055672], [0, 0.022735, 0.235578, 0.011337, 0.009781], [0, 0.078942, 0.243898, 0.017525, 0.018203]]}, {"filename": "000000074316.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.050828, 0.552787, 0.052969, 0.188852], [4, 0.122, 0.515773, 0.057969, 0.111593]]}, {"filename": "000000075173.jpg", "coco_split": "train2017", "split": "test", "boxes": [[7, 0.213953, 0.467664, 0.050062, 0.148427], [7, 0.94807, 0.43885, 0.029297, 0.163333], [3, 0.522781, 0.492136, 0.587844, 0.4]]}, {"filename": "000000080445.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.503352, 0.520352, 0.290828, 0.293078]]}, {"filename": "000000082662.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.55554, 0.474709, 0.47908, 0.647196], [0, 0.56488, 0.68119, 0.06944, 0.221852]]}, {"filename": "000000082715.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.372984, 0.381475, 0.213, 0.205761]]}, {"filename": "000000083178.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.307359, 0.346066, 0.605719, 0.534848], [0, 0.290859, 0.758431, 0.131937, 0.483138], [0, 0.251695, 0.507881, 0.040578, 0.162178], [0, 0.522086, 0.821745, 0.176516, 0.334075], [0, 0.57007, 0.690117, 0.064141, 0.212319], [0, 0.811547, 0.546183, 0.070312, 0.33185], [0, 0.499516, 0.536792, 0.102781, 0.2526], [0, 0.670891, 0.671745, 0.163125, 0.241288], [0, 0.661641, 0.803817, 0.144781, 0.391756], [0, 0.325109, 0.506499, 0.091312, 0.180539], [0, 0.873484, 0.537412, 0.059094, 0.320867], [0, 0.038055, 0.538091, 0.076109, 0.163396], [0, 0.177234, 0.751686, 0.155281, 0.474145], [1, 0.62425, 0.315035, 0.036375, 0.058876], [0, 0.16425, 0.550749, 0.147469, 0.210304]]}, {"filename": "000000083953.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.5, 0.238948, 1.0, 0.460687], [5, 0.787078, 0.69775, 0.144937, 0.294375]]}, {"filename": "000000088432.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.367115, 0.820156, 0.190854, 0.244531], [1, 0.322771, 0.521, 0.616958, 0.209531]]}, {"filename": "000000088462.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.732648, 0.420222, 0.534703, 0.579765], [1, 0.083617, 0.545535, 0.089797, 0.110601], [1, 0.317, 0.522337, 0.092687, 0.102689], [1, 0.256992, 0.515836, 0.062328, 0.077415], [1, 0.165938, 0.503995, 0.039781, 0.055509], [1, 0.141969, 0.494295, 0.029812, 0.046397], [1, 0.135234, 0.484021, 0.025781, 0.042245], [6, 0.536258, 0.460065, 0.023922, 0.065927], [1, 0.225164, 0.512846, 0.041484, 0.060313], [1, 0.19457, 0.509465, 0.038422, 0.053394], [1, 0.122766, 0.477781, 0.023156, 0.034099], [1, 0.083773, 0.476762, 0.027078, 0.03295], [1, 0.103055, 0.472337, 0.016453, 0.023943], [1, 0.090266, 0.428211, 0.0115, 0.020339], [1, 0.078727, 0.413198, 0.008328, 0.013916], [1, 0.074789, 0.429478, 0.016109, 0.026684]]}, {"filename": "000000094326.jpg", "coco_split": "val2017", "split": "test", "boxes": [[3, 0.398507, 0.60562, 0.71312, 0.32136], [0, 0.328853, 0.56539, 0.4544, 0.60314]]}, {"filename": "000000100723.jpg", "coco_split": "val2017", "split": "test", "boxes": [[6, 0.53505, 0.307853, 0.24694, 0.35955], [0, 0.5985, 0.547297, 0.171, 0.425646], [0, 0.53355, 0.766291, 0.2649, 0.444955], [0, 0.70055, 0.377928, 0.05734, 0.317838], [0, 0.15737, 0.788859, 0.30054, 0.422282], [0, 0.84915, 0.781261, 0.1851, 0.431952], [0, 0.45933, 0.632492, 0.11174, 0.407928], [0, 0.95877, 0.080826, 0.05586, 0.104535], [0, 0.25669, 0.275375, 0.05606, 0.25982], [0, 0.30865, 0.746471, 0.1775, 0.485856], [0, 0.14956, 0.482447, 0.07572, 0.116126], [0, 0.05163, 0.585661, 0.07342, 0.532643], [0, 0.11843, 0.216066, 0.05394, 0.070691], [0, 0.17232, 0.395721, 0.11348, 0.289099]]}, {"filename": "000000102906.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.169078, 0.782021, 0.020094, 0.064369], [5, 0.316945, 0.765315, 0.009422, 0.025911], [1, 0.359195, 0.720678, 0.028297, 0.028972]]}, {"filename": "000000109118.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.755977, 0.504492, 0.488045, 0.878641], [0, 0.479877, 0.606102, 0.487695, 0.770797], [0, 0.197387, 0.452117, 0.341728, 0.881422], [4, 0.462521, 0.802234, 0.223889, 0.243844]]}, {"filename": "000000109417.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.539656, 0.581823, 0.217719, 0.185188], [4, 0.338672, 0.832385, 0.159, 0.256104], [4, 0.259609, 0.592438, 0.16675, 0.216042]]}, {"filename": "000000115245.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.845408, 0.236683, 0.166144, 0.146732], [4, 0.72674, 0.173538, 0.069624, 0.11152], [4, 0.535188, 0.121936, 0.039395, 0.058905], [4, 0.479804, 0.094975, 0.02634, 0.040114], [4, 0.960842, 0.293971, 0.078317, 0.09585], [0, 0.021814, 0.039657, 0.043497, 0.078791], [0, 0.199845, 0.036413, 0.095082, 0.07201], [4, 0.25665, 0.033023, 0.015588, 0.020131], [0, 0.40991, 0.058039, 0.132925, 0.115]]}, {"filename": "000000115946.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.464635, 0.801688, 0.021729, 0.048875], [1, 0.304062, 0.818109, 0.1115, 0.065844], [1, 0.430917, 0.806281, 0.087, 0.042188], [1, 0.519802, 0.798578, 0.038812, 0.025063], [1, 0.698354, 0.799133, 0.065292, 0.037609], [1, 0.657885, 0.793266, 0.031937, 0.022125], [1, 0.355823, 0.809336, 0.068062, 0.055359], [1, 0.481604, 0.799625, 0.020125, 0.022125], [1, 0.541302, 0.797164, 0.019479, 0.021703], [1, 0.559969, 0.795609, 0.028021, 0.016969]]}, {"filename": "000000125352.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.459883, 0.551798, 0.912953, 0.891901], [0, 0.169047, 0.491901, 0.178438, 0.253926], [0, 0.59118, 0.523574, 0.147328, 0.487562], [1, 0.936539, 0.094298, 0.126922, 0.188595], [0, 0.629984, 0.471281, 0.256688, 0.528884]]}, {"filename": "000000126305.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.307359, 0.601557, 0.212906, 0.319087], [1, 0.931758, 0.435703, 0.136484, 0.265831], [1, 0.542289, 0.589895, 0.071516, 0.091639], [1, 0.086195, 0.868021, 0.073422, 0.053981], [1, 0.664188, 0.517412, 0.046219, 0.084707], [5, 0.95332, 0.58411, 0.093359, 0.239461]]}, {"filename": "000000126590.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.772854, 0.155586, 0.082917, 0.045141], [1, 0.71949, 0.146867, 0.041312, 0.032422], [1, 0.139177, 0.130125, 0.186437, 0.048937], [7, 0.529146, 0.259156, 0.143917, 0.228187], [1, 0.533052, 0.130883, 0.019271, 0.016859], [1, 0.879135, 0.157234, 0.142313, 0.068719], [1, 0.991365, 0.14593, 0.017271, 0.046922]]}, {"filename": "000000138639.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.700422, 0.706802, 0.056813, 0.044771], [1, 0.630281, 0.704292, 0.044187, 0.043458], [0, 0.661594, 0.820135, 0.072594, 0.286479], [0, 0.943164, 0.841073, 0.108422, 0.317688], [0, 0.175328, 0.68774, 0.026188, 0.053646], [0, 0.252875, 0.751115, 0.084594, 0.171479], [0, 0.266539, 0.675115, 0.015672, 0.027729], [0, 0.299953, 0.672885, 0.013813, 0.038604], [0, 0.460797, 0.683052, 0.012437, 0.050271], [0, 0.33625, 0.674573, 0.013062, 0.040771], [0, 0.317031, 0.677615, 0.021719, 0.050271], [0, 0.112969, 0.651448, 0.008469, 0.034812], [0, 0.098727, 0.653438, 0.010266, 0.041958], [1, 0.253219, 0.811958, 0.118531, 0.120875], [0, 0.068203, 0.674375, 0.032469, 0.064958], [0, 0.131937, 0.652583, 0.0115, 0.04175]]}, {"filename": "000000139426.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.614555, 0.598336, 0.506487, 0.600578]]}, {"filename": "000000140420.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.664641, 0.633037, 0.071219, 0.176822]]}, {"filename": "000000142620.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.393531, 0.438063, 0.292344, 0.387394], [4, 0.893391, 0.622911, 0.213219, 0.712347], [0, 0.703023, 0.5, 0.562422, 0.973052]]}, {"filename": "000000143125.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.317582, 0.614078, 0.436056, 0.495031], [0, 0.656643, 0.602305, 0.356948, 0.524109], [6, 0.871631, 0.251766, 0.062559, 0.08]]}, {"filename": "000000143874.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.319023, 0.68723, 0.083578, 0.11611], [4, 0.37957, 0.694774, 0.076922, 0.115874], [4, 0.453406, 0.696798, 0.07875, 0.117014], [1, 0.486023, 0.55556, 0.763891, 0.365422]]}, {"filename": "000000144725.jpg", "coco_split": "train2017", "split": "test", "boxes": [[3, 0.500602, 0.78717, 0.998234, 0.400896]]}, {"filename": "000000148437.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.386289, 0.642703, 0.245641, 0.188784], [0, 0.380922, 0.567759, 0.121375, 0.246509], [0, 0.152125, 0.448018, 0.0235, 0.053423], [0, 0.07682, 0.467748, 0.019203, 0.09518], [0, 0.10425, 0.464673, 0.026156, 0.106554], [1, 0.501148, 0.486036, 0.107172, 0.13036], [1, 0.667391, 0.486239, 0.118438, 0.115766], [1, 0.284344, 0.445698, 0.040094, 0.038333], [1, 0.599453, 0.465957, 0.090344, 0.110203], [1, 0.782422, 0.461149, 0.153906, 0.167793], [1, 0.387578, 0.461194, 0.04375, 0.061937], [1, 0.348273, 0.447005, 0.044141, 0.059414], [1, 0.437078, 0.474662, 0.049625, 0.100225]]}, {"filename": "000000151101.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.487096, 0.472711, 0.616487, 0.708484], [1, 0.370258, 0.232164, 0.034707, 0.030516]]}, {"filename": "000000152398.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.90389, 0.857147, 0.18702, 0.259733]]}, {"filename": "000000152465.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.747664, 0.45595, 0.204547, 0.793101]]}, {"filename": "000000156292.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.481963, 0.884875, 0.106542, 0.20525], [0, 0.220362, 0.727062, 0.152453, 0.271687]]}, {"filename": "000000158058.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.37735, 0.73902, 0.74582, 0.502242], [6, 0.13693, 0.351475, 0.21102, 0.258505], [4, 0.05562, 0.642697, 0.11124, 0.462929]]}, {"filename": "000000158744.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.517, 0.285165, 0.030813, 0.033962], [4, 0.82857, 0.520436, 0.139047, 0.267429], [0, 0.761305, 0.474682, 0.114422, 0.576863], [1, 0.972766, 0.255837, 0.052594, 0.070307], [1, 0.962273, 0.252441, 0.071641, 0.069741], [5, 0.841156, 0.279316, 0.009313, 0.026415], [5, 0.595258, 0.284623, 0.012141, 0.028208], [1, 0.992891, 0.269481, 0.014219, 0.059245], [4, 0.128094, 0.451097, 0.177406, 0.229741], [4, 0.344156, 0.522594, 0.234031, 0.225991]]}, {"filename": "000000162366.jpg", "coco_split": "val2017", "split": "test", "boxes": [[7, 0.700727, 0.619845, 0.283453, 0.539335]]}, {"filename": "000000162732.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.291523, 0.543361, 0.234422, 0.913279], [0, 0.5285, 0.526967, 0.349344, 0.946066], [0, 0.420195, 0.528782, 0.180734, 0.914239], [0, 0.131187, 0.513478, 0.202406, 0.950562], [0, 0.913969, 0.5, 0.172063, 0.968899]]}, {"filename": "000000165713.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.432188, 0.557172, 0.378165, 0.679375]]}, {"filename": "000000171181.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.254016, 0.401171, 0.268313, 0.154567], [4, 0.286477, 0.549309, 0.325297, 0.225457], [4, 0.732469, 0.495937, 0.285531, 0.333326], [4, 0.818625, 0.550527, 0.340031, 0.222693], [4, 0.228359, 0.901499, 0.456719, 0.173817], [4, 0.407883, 0.25726, 0.118578, 0.115644], [4, 0.842187, 0.904719, 0.315625, 0.167377], [4, 0.791992, 0.232939, 0.111953, 0.174871], [4, 0.252359, 0.160258, 0.155313, 0.302951], [4, 0.586094, 0.259262, 0.097281, 0.127799], [4, 0.705094, 0.153583, 0.125344, 0.105995]]}, {"filename": "000000171422.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.610258, 0.528823, 0.063547, 0.161646], [1, 0.335258, 0.680688, 0.236578, 0.460833], [3, 0.559125, 0.850219, 0.125, 0.270271], [1, 0.664352, 0.115781, 0.186297, 0.129729], [1, 0.794656, 0.124573, 0.159031, 0.185771], [1, 0.932648, 0.115302, 0.132953, 0.092188]]}, {"filename": "000000172595.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.199422, 0.547417, 0.333375, 0.905167], [4, 0.034078, 0.069083, 0.068156, 0.134611]]}, {"filename": "000000178307.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.279146, 0.533492, 0.413792, 0.682266]]}, {"filename": "000000179034.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.418961, 0.585374, 0.272953, 0.505595], [3, 0.447945, 0.602247, 0.637641, 0.489868]]}, {"filename": "000000184065.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.82697, 0.616853, 0.34606, 0.70784], [4, 0.96715, 0.40576, 0.0657, 0.19872], [4, 0.91869, 0.361213, 0.07714, 0.079013]]}, {"filename": "000000184324.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.788312, 0.861282, 0.137094, 0.131176], [1, 0.772148, 0.779365, 0.121891, 0.130212], [1, 0.485609, 0.597141, 0.1065, 0.078094], [1, 0.40225, 0.570224, 0.043187, 0.038], [1, 0.734141, 0.687282, 0.036281, 0.095082], [1, 0.215805, 0.5466, 0.061547, 0.122518], [0, 0.698688, 0.652235, 0.026938, 0.133271], [0, 0.666922, 0.662047, 0.042469, 0.140376], [0, 0.589813, 0.642906, 0.042063, 0.143224], [0, 0.562875, 0.613047, 0.028, 0.128259], [0, 0.536063, 0.575918, 0.021375, 0.057812], [0, 0.519789, 0.569635, 0.018422, 0.044165], [1, 0.213781, 0.550188, 0.061063, 0.119247], [0, 0.089336, 0.876235, 0.148891, 0.247529], [0, 0.904992, 0.672882, 0.042859, 0.163271], [0, 0.089836, 0.706776, 0.082266, 0.385224], [0, 0.16868, 0.739482, 0.086109, 0.345035], [0, 0.063844, 0.562388, 0.022094, 0.055553], [0, 0.027461, 0.651353, 0.048422, 0.261435], [1, 0.726367, 0.589612, 0.032172, 0.038282], [1, 0.694102, 0.596259, 0.054922, 0.049741], [1, 0.449609, 0.560765, 0.023062, 0.023412], [0, 0.376766, 0.569741, 0.019094, 0.077506], [1, 0.176102, 0.545059, 0.025078, 0.035153], [6, 0.487961, 0.526518, 0.016266, 0.026118]]}, {"filename": "000000184611.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.392773, 0.651938, 0.565734, 0.676708], [1, 0.950695, 0.227177, 0.043172, 0.042062], [0, 0.703109, 0.396042, 0.071625, 0.244208], [0, 0.428992, 0.293125, 0.059766, 0.07675], [0, 0.304211, 0.585396, 0.281453, 0.721333], [0, 0.047039, 0.478948, 0.094078, 0.417396], [0, 0.372555, 0.285552, 0.032297, 0.093604], [0, 0.406539, 0.287938, 0.035922, 0.0805], [0, 0.69182, 0.256927, 0.033078, 0.097063], [0, 0.843031, 0.237115, 0.019844, 0.072271], [0, 0.883352, 0.242469, 0.018953, 0.074854], [0, 0.856992, 0.231812, 0.014016, 0.076042], [0, 0.793703, 0.240271, 0.026219, 0.067792], [1, 0.985602, 0.224792, 0.028797, 0.032208], [0, 0.276758, 0.28299, 0.027672, 0.060021], [0, 0.302258, 0.317354, 0.060891, 0.10825]]}, {"filename": "000000187012.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.66451, 0.677296, 0.213497, 0.409199], [3, 0.431618, 0.584077, 0.073987, 0.11884], [3, 0.786446, 0.580507, 0.091716, 0.090817], [3, 0.850637, 0.595743, 0.108268, 0.109199], [3, 0.732941, 0.554175, 0.040523, 0.065082], [3, 0.491716, 0.551397, 0.073987, 0.048709], [4, 0.798211, 0.579036, 0.062304, 0.096373], [4, 0.722843, 0.554698, 0.019412, 0.059265], [4, 0.853219, 0.596471, 0.105523, 0.10415]]}, {"filename": "000000189226.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.643953, 0.677529, 0.145437, 0.483162]]}, {"filename": "000000192716.jpg", "coco_split": "val2017", "split": "test", "boxes": [[6, 0.505309, 0.516859, 0.58861, 0.498875], [1, 0.053591, 0.959422, 0.107181, 0.0775], [1, 0.358996, 0.965148, 0.185907, 0.069703], [0, 0.088755, 0.953125, 0.027355, 0.044844]]}, {"filename": "000000193884.jpg", "coco_split": "val2017", "split": "test", "boxes": [[6, 0.636609, 0.309635, 0.098, 0.125229]]}, {"filename": "000000196775.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.117771, 0.470406, 0.235542, 0.120688], [4, 0.196375, 0.551187, 0.087958, 0.038938], [4, 0.162427, 0.631313, 0.124521, 0.147094], [4, 0.053969, 0.613711, 0.096771, 0.056453], [4, 0.092208, 0.803352, 0.180583, 0.270484], [4, 0.282562, 0.732281, 0.188583, 0.222969], [7, 0.528833, 0.680211, 0.338583, 0.534828], [1, 0.186969, 0.443656, 0.093312, 0.050469], [3, 0.809281, 0.488312, 0.146479, 0.060469], [4, 0.032344, 0.548977, 0.061854, 0.067109], [4, 0.282719, 0.622719, 0.147896, 0.163969]]}, {"filename": "000000197678.jpg", "coco_split": "train2017", "split": "test", "boxes": [[3, 0.512766, 0.648314, 0.599719, 0.671897]]}, {"filename": "000000199339.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.410117, 0.680023, 0.48271, 0.521453], [1, 0.73368, 0.821625, 0.436519, 0.18125], [1, 0.140736, 0.775352, 0.148902, 0.168328], [1, 0.084509, 0.690578, 0.169019, 0.212344]]}, {"filename": "000000199555.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.388094, 0.774797, 0.183781, 0.450405], [3, 0.520453, 0.853255, 0.183781, 0.070788], [3, 0.501094, 0.850563, 0.997812, 0.077297]]}, {"filename": "000000203488.jpg", "coco_split": "val2017", "split": "test", "boxes": [[7, 0.7205, 0.603812, 0.126406, 0.3955], [3, 0.869289, 0.836344, 0.261422, 0.325896]]}, {"filename": "000000206579.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.340203, 0.495313, 0.677375, 0.984458], [0, 0.854695, 0.197896, 0.231547, 0.395792], [0, 0.218859, 0.045312, 0.121469, 0.090625], [4, 0.723273, 0.321927, 0.278609, 0.447646]]}, {"filename": "000000209222.jpg", "coco_split": "val2017", "split": "test", "boxes": [[3, 0.832016, 0.758068, 0.194594, 0.322412], [0, 0.202258, 0.351159, 0.072234, 0.203536], [1, 0.620266, 0.279883, 0.037844, 0.058689], [1, 0.499508, 0.263817, 0.085516, 0.048993], [0, 0.339969, 0.342014, 0.056156, 0.112974], [0, 0.36393, 0.33596, 0.019359, 0.105363], [0, 0.354328, 0.295492, 0.012875, 0.032061], [0, 0.759078, 0.673653, 0.235719, 0.422155], [0, 0.778586, 0.345855, 0.043547, 0.111756], [0, 0.432734, 0.28733, 0.015219, 0.044965], [0, 0.749648, 0.332412, 0.065734, 0.110211], [0, 0.687984, 0.318068, 0.026312, 0.075246], [0, 0.518602, 0.318255, 0.032234, 0.054871], [0, 0.641094, 0.291311, 0.020469, 0.089696], [0, 0.723992, 0.293712, 0.014453, 0.028876], [0, 0.71407, 0.295855, 0.012672, 0.032365]]}, {"filename": "000000214539.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.531953, 0.528162, 0.351625, 0.617119], [0, 0.943789, 0.46966, 0.112422, 0.651686], [0, 0.776938, 0.491265, 0.0465, 0.243934], [0, 0.058555, 0.505164, 0.039516, 0.120351], [0, 0.166305, 0.559075, 0.078453, 0.212014], [0, 0.226336, 0.557061, 0.070109, 0.205831], [0, 0.342516, 0.542482, 0.095094, 0.202061], [0, 0.153187, 0.558009, 0.055062, 0.224965], [0, 0.276227, 0.547377, 0.114203, 0.19466], [0, 0.188977, 0.482799, 0.019172, 0.048642], [0, 0.328766, 0.44089, 0.029406, 0.067728], [0, 0.084727, 0.470304, 0.030516, 0.075035], [0, 0.178125, 0.483267, 0.018625, 0.047283], [3, 0.617852, 0.552822, 0.144359, 0.021756], [3, 0.668586, 0.521429, 0.081266, 0.045574]]}, {"filename": "000000227134.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.474906, 0.461797, 0.626229, 0.739312]]}, {"filename": "000000227511.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.29195, 0.88252, 0.32098, 0.23496], [1, 0.49295, 0.7412, 0.08582, 0.088053], [1, 0.33208, 0.703307, 0.03712, 0.036267], [1, 0.51146, 0.724747, 0.06188, 0.07104], [5, 0.16421, 0.735893, 0.00998, 0.02112], [1, 0.37806, 0.701987, 0.02552, 0.024827], [1, 0.29152, 0.706333, 0.03048, 0.021413]]}, {"filename": "000000231879.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.853344, 0.811244, 0.288687, 0.350563], [4, 0.068742, 0.919894, 0.058453, 0.160211], [0, 0.609039, 0.502347, 0.306766, 0.995305], [0, 0.277969, 0.649542, 0.357594, 0.677207], [0, 0.326695, 0.176901, 0.272359, 0.348404], [0, 0.447008, 0.50507, 0.151766, 0.471784], [0, 0.786602, 0.494613, 0.171766, 0.989225]]}, {"filename": "000000233771.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.050273, 0.600633, 0.097203, 0.189328], [0, 0.806055, 0.788375, 0.168172, 0.302687], [0, 0.90525, 0.678523, 0.116125, 0.233453], [0, 0.681422, 0.848945, 0.171312, 0.302109], [0, 0.481406, 0.844172, 0.244531, 0.311656], [0, 0.281883, 0.796047, 0.295297, 0.407906], [1, 0.435664, 0.606578, 0.706266, 0.208188], [0, 0.851195, 0.638766, 0.037891, 0.053219], [0, 0.582008, 0.800328, 0.071328, 0.071844], [0, 0.920867, 0.818383, 0.152422, 0.363234], [0, 0.419328, 0.75282, 0.056844, 0.118734], [0, 0.780609, 0.674406, 0.048562, 0.118594]]}, {"filename": "000000236516.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.603469, 0.582014, 0.607219, 0.808993], [0, 0.262055, 0.515234, 0.524109, 0.93904], [4, 0.813953, 0.835691, 0.372094, 0.328618]]}, {"filename": "000000244019.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.478977, 0.551833, 0.128641, 0.609002]]}, {"filename": "000000248268.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.341573, 0.529961, 0.479396, 0.885391]]}, {"filename": "000000252294.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.38952, 0.50914, 0.497387, 0.82472], [0, 0.71356, 0.32654, 0.198107, 0.33364]]}, {"filename": "000000257084.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.27545, 0.455841, 0.03142, 0.106396], [0, 0.50558, 0.283243, 0.33524, 0.561802], [0, 0.69696, 0.2647, 0.14828, 0.523093], [0, 0.59556, 0.177688, 0.09584, 0.355375], [0, 0.37163, 0.453243, 0.05782, 0.23952], [0, 0.31548, 0.438108, 0.04552, 0.195435], [0, 0.82389, 0.280901, 0.19606, 0.561802], [4, 0.28093, 0.604715, 0.06998, 0.11027]]}, {"filename": "000000258119.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.574198, 0.369344, 0.366896, 0.456375]]}, {"filename": "000000259597.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.249758, 0.531282, 0.425203, 0.883397], [0, 0.601156, 0.267612, 0.117031, 0.447853], [0, 0.466953, 0.362997, 0.082188, 0.215224], [3, 0.691898, 0.726458, 0.614297, 0.511186], [0, 0.522164, 0.385128, 0.044672, 0.205641], [0, 0.039336, 0.340353, 0.047109, 0.18859], [0, 0.386508, 0.377676, 0.053891, 0.169006], [0, 0.764766, 0.407596, 0.117812, 0.658269], [0, 0.964977, 0.348478, 0.067328, 0.342724], [0, 0.919125, 0.329215, 0.081062, 0.348301], [0, 0.848469, 0.324712, 0.082156, 0.361795], [0, 0.106563, 0.492372, 0.1195, 0.383462], [0, 0.542266, 0.29258, 0.037312, 0.135801], [4, 0.639039, 0.770769, 0.115297, 0.361282], [0, 0.69575, 0.379343, 0.040719, 0.217981], [4, 0.030945, 0.704535, 0.059359, 0.259071], [4, 0.06643, 0.595256, 0.033203, 0.160513]]}, {"filename": "000000267985.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.514672, 0.456542, 0.30475, 0.702042], [0, 0.210672, 0.537083, 0.30675, 0.530333], [4, 0.771906, 0.688771, 0.229219, 0.168542], [0, 0.499672, 0.75325, 0.566875, 0.467542]]}, {"filename": "000000269022.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.101719, 0.480183, 0.203344, 0.158513], [0, 0.738781, 0.434224, 0.397375, 0.800905]]}, {"filename": "000000270066.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.522805, 0.478604, 0.755078, 0.635125], [1, 0.503937, 0.161427, 0.80225, 0.319104]]}, {"filename": "000000272049.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.446047, 0.526276, 0.730156, 0.901124]]}, {"filename": "000000272171.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.891828, 0.399854, 0.072875, 0.071708], [1, 0.774109, 0.406313, 0.033812, 0.053958], [1, 0.715445, 0.39676, 0.049359, 0.059354], [0, 0.332297, 0.340677, 0.033875, 0.097438], [0, 0.370961, 0.346417, 0.025578, 0.090042], [0, 0.262078, 0.614604, 0.133156, 0.137083], [0, 0.422992, 0.388219, 0.048547, 0.121771], [0, 0.624461, 0.359677, 0.012828, 0.066854], [0, 0.73932, 0.381417, 0.028859, 0.094625], [0, 0.809484, 0.503687, 0.054687, 0.099375], [0, 0.881367, 0.521437, 0.063609, 0.062083], [0, 0.942414, 0.513437, 0.038672, 0.093458], [0, 0.95525, 0.496635, 0.033, 0.107646], [0, 0.182727, 0.24224, 0.036047, 0.059979], [0, 0.812109, 0.424531, 0.022469, 0.083146], [0, 0.798984, 0.403562, 0.011437, 0.061375]]}, {"filename": "000000281687.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.72508, 0.678147, 0.28492, 0.643707], [0, 0.85491, 0.714707, 0.26502, 0.570587], [0, 0.15, 0.658427, 0.28652, 0.65168], [0, 0.43221, 0.7818, 0.06818, 0.434747], [0, 0.54693, 0.6498, 0.06258, 0.108667], [0, 0.28508, 0.719547, 0.0326, 0.124587], [0, 0.49783, 0.710227, 0.03686, 0.11], [0, 0.22577, 0.72244, 0.03942, 0.123227], [0, 0.19217, 0.767533, 0.0671, 0.323653], [1, 0.30966, 0.70576, 0.01652, 0.066507], [0, 0.47155, 0.7228, 0.02406, 0.13648]]}, {"filename": "000000283038.jpg", "coco_split": "val2017", "split": "test", "boxes": [[6, 0.45338, 0.477527, 0.4496, 0.869644], [1, 0.048, 0.714698, 0.096, 0.101495], [1, 0.1778, 0.653968, 0.14292, 0.142811]]}, {"filename": "000000288685.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.200102, 0.372282, 0.146703, 0.385849], [0, 0.379688, 0.338291, 0.057813, 0.148096], [0, 0.719523, 0.308991, 0.097984, 0.177523], [0, 0.83482, 0.355011, 0.074641, 0.170986], [0, 0.297922, 0.295229, 0.063844, 0.156193], [0, 0.466258, 0.306697, 0.082266, 0.142294], [0, 0.545102, 0.313658, 0.089516, 0.217821], [0, 0.634453, 0.298165, 0.077031, 0.151789], [0, 0.784883, 0.485791, 0.077547, 0.295757], [0, 0.240703, 0.266422, 0.055406, 0.120688], [0, 0.052641, 0.32328, 0.105281, 0.163486], [0, 0.043633, 0.504048, 0.081297, 0.317546], [3, 0.66975, 0.614794, 0.327344, 0.103899], [0, 0.072328, 0.273945, 0.071281, 0.141927], [3, 0.830273, 0.527053, 0.047609, 0.025849]]}, {"filename": "000000292499.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.415246, 0.504492, 0.580681, 0.914609], [6, 0.755845, 0.034391, 0.091408, 0.066875]]}, {"filename": "000000293858.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.23665, 0.204058, 0.27546, 0.395698], [4, 0.52343, 0.15663, 0.13522, 0.29122]]}, {"filename": "000000295713.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.746188, 0.552037, 0.199813, 0.630117], [0, 0.9045, 0.095539, 0.097937, 0.135808], [0, 0.846312, 0.136452, 0.044031, 0.058759], [3, 0.233891, 0.689508, 0.311844, 0.292131], [3, 0.773672, 0.725457, 0.436344, 0.272319], [3, 0.662312, 0.51952, 0.514625, 0.300164], [0, 0.88532, 0.071511, 0.075703, 0.138899], [0, 0.320227, 0.065222, 0.052953, 0.127775], [0, 0.928898, 0.018115, 0.013828, 0.03623], [0, 0.961492, 0.056991, 0.044422, 0.113981]]}, {"filename": "000000297191.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.280588, 0.123399, 0.118431, 0.085163], [5, 0.640449, 0.500752, 0.485408, 0.92134]]}, {"filename": "000000297343.jpg", "coco_split": "val2017", "split": "test", "boxes": [[6, 0.240953, 0.580231, 0.296687, 0.444167]]}, {"filename": "000000297578.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.47928, 0.505623, 0.34732, 0.957295], [0, 0.09324, 0.612313, 0.12812, 0.548541], [1, 0.89337, 0.547171, 0.11134, 0.398612], [1, 0.29753, 0.856459, 0.11106, 0.263523], [1, 0.902, 0.790036, 0.106, 0.27758], [1, 0.13099, 0.518416, 0.05882, 0.110427], [0, 0.25597, 0.527598, 0.18078, 0.944804], [1, 0.01991, 0.664413, 0.03982, 0.218577]]}, {"filename": "000000298203.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.757086, 0.500105, 0.180516, 0.255574], [1, 0.902047, 0.797951, 0.037063, 0.036932], [1, 0.856539, 0.800867, 0.031641, 0.035644], [1, 0.820922, 0.818185, 0.02425, 0.0263], [1, 0.925398, 0.772658, 0.027547, 0.020023], [1, 0.964031, 0.777459, 0.045906, 0.026956], [1, 0.878047, 0.783607, 0.026594, 0.024309], [1, 0.822789, 0.805351, 0.030828, 0.031827], [1, 0.8245, 0.791475, 0.029469, 0.018407]]}, {"filename": "000000300876.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.213961, 0.612613, 0.427922, 0.740985], [0, 0.538266, 0.34081, 0.232937, 0.286995], [4, 0.298773, 0.290943, 0.268891, 0.368564], [0, 0.728867, 0.268856, 0.283828, 0.531569]]}, {"filename": "000000310020.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.600187, 0.358173, 0.456906, 0.716347], [0, 0.845117, 0.381604, 0.063203, 0.11089], [0, 0.985773, 0.375281, 0.028453, 0.175293], [0, 0.391805, 0.407365, 0.049266, 0.089485], [0, 0.776852, 0.382658, 0.045453, 0.10644], [0, 0.434328, 0.412822, 0.046406, 0.11192], [0, 0.126063, 0.451066, 0.040281, 0.086581], [0, 0.289102, 0.391827, 0.045203, 0.096815], [0, 0.178398, 0.414063, 0.029203, 0.112061], [0, 0.477844, 0.426991, 0.023906, 0.074637], [0, 0.502563, 0.439836, 0.994875, 0.30459]]}, {"filename": "000000315232.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.726672, 0.588506, 0.125812, 0.376123]]}, {"filename": "000000319299.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.160953, 0.544284, 0.023906, 0.053873], [1, 0.181531, 0.547254, 0.029469, 0.046995]]}, {"filename": "000000322829.jpg", "coco_split": "val2017", "split": "test", "boxes": [[3, 0.485406, 0.449941, 0.716812, 0.676276]]}, {"filename": "000000323751.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.876141, 0.576415, 0.025313, 0.168538], [3, 0.875156, 0.967193, 0.045562, 0.065613], [0, 0.34468, 0.608833, 0.038453, 0.047948]]}, {"filename": "000000332318.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.678195, 0.941807, 0.119203, 0.116387], [1, 0.884617, 0.956608, 0.079328, 0.075315], [1, 0.76768, 0.954429, 0.053766, 0.0538], [1, 0.834734, 0.94859, 0.074438, 0.073776]]}, {"filename": "000000333956.jpg", "coco_split": "val2017", "split": "test", "boxes": [[7, 0.275105, 0.596508, 0.435444, 0.659141], [3, 0.072512, 0.767203, 0.145023, 0.251156]]}, {"filename": "000000338191.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.503937, 0.498875, 0.418992, 0.916844], [5, 0.888732, 0.395242, 0.094189, 0.203016], [5, 0.904748, 0.623086, 0.09589, 0.180766], [5, 0.106315, 0.158562, 0.110457, 0.194375], [5, 0.900622, 0.851477, 0.10874, 0.161641], [5, 0.092512, 0.629375, 0.104488, 0.18475], [5, 0.893197, 0.171477, 0.133984, 0.216828], [5, 0.097685, 0.84875, 0.083402, 0.153688], [5, 0.088819, 0.393547, 0.139654, 0.207844]]}, {"filename": "000000338428.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.173992, 0.255246, 0.347984, 0.505574], [1, 0.509437, 0.460273, 0.84625, 0.85541], [1, 0.941258, 0.346913, 0.117484, 0.185191], [1, 0.92925, 0.274085, 0.1415, 0.097131]]}, {"filename": "000000338562.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.392255, 0.65815, 0.234729, 0.14382], [5, 0.588818, 0.62022, 0.209419, 0.44044], [0, 0.186784, 0.62786, 0.032525, 0.07176]]}, {"filename": "000000340611.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.076823, 0.210992, 0.150979, 0.419234]]}, {"filename": "000000343561.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.136703, 0.550958, 0.231844, 0.455234], [1, 0.42925, 0.767173, 0.220688, 0.396308], [1, 0.637188, 0.621355, 0.171312, 0.397757], [1, 0.957586, 0.473949, 0.084828, 0.256495], [1, 0.163055, 0.785397, 0.265984, 0.393271], [1, 0.943203, 0.442243, 0.029344, 0.153645], [1, 0.871906, 0.47611, 0.036, 0.154883], [0, 0.554898, 0.400479, 0.131453, 0.317874], [0, 0.64, 0.424416, 0.155094, 0.40528], [0, 0.80357, 0.33771, 0.109453, 0.242056], [0, 0.66643, 0.111227, 0.020953, 0.041145], [0, 0.247961, 0.428084, 0.144266, 0.447196], [0, 0.846078, 0.686519, 0.303562, 0.604486], [0, 0.200625, 0.141974, 0.090813, 0.283949], [0, 0.274406, 0.080315, 0.057656, 0.160631], [0, 0.410672, 0.142675, 0.044094, 0.141098], [1, 0.744031, 0.229159, 0.051156, 0.060935], [1, 0.910758, 0.389591, 0.049797, 0.16965], [1, 0.246664, 0.577675, 0.086141, 0.18736], [1, 0.555148, 0.607196, 0.105078, 0.307664], [1, 0.721766, 0.524299, 0.09075, 0.23285], [1, 0.271438, 0.615666, 0.151, 0.258341], [0, 0.059211, 0.19271, 0.066172, 0.263037], [0, 0.305859, 0.211682, 0.0645, 0.297243], [0, 0.415875, 0.586647, 0.209813, 0.556519]]}, {"filename": "000000345882.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.461352, 0.830141, 0.213734, 0.339719], [0, 0.616961, 0.623033, 0.166422, 0.674169], [0, 0.242891, 0.502248, 0.392812, 0.995504]]}, {"filename": "000000354278.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.195184, 0.15468, 0.390367, 0.130328], [1, 0.607112, 0.134281, 0.247857, 0.081187], [5, 0.510714, 0.55618, 0.387429, 0.820234], [1, 0.964408, 0.138516, 0.071184, 0.050812], [1, 0.833469, 0.139453, 0.209143, 0.063906]]}, {"filename": "000000357816.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.90099, 0.324582, 0.19422, 0.443143], [0, 0.31973, 0.563055, 0.58618, 0.840088], [0, 0.09802, 0.405725, 0.1912, 0.458396], [1, 0.30851, 0.100308, 0.6035, 0.176967], [3, 0.64343, 0.387978, 0.57034, 0.324527], [0, 0.66773, 0.288308, 0.19618, 0.571429]]}, {"filename": "000000362293.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.51607, 0.441176, 0.126297, 0.153929], [0, 0.698945, 0.288824, 0.060547, 0.161035], [0, 0.810344, 0.401341, 0.021344, 0.107106], [0, 0.373125, 0.412353, 0.032219, 0.150729], [0, 0.336523, 0.434541, 0.042859, 0.157459], [0, 0.642711, 0.292141, 0.052922, 0.119153], [0, 0.823211, 0.404588, 0.012641, 0.082965], [1, 0.544102, 0.375071, 0.066891, 0.056165], [0, 0.183445, 0.400588, 0.025547, 0.038165], [7, 0.446289, 0.369976, 0.009172, 0.036565], [7, 0.372453, 0.416224, 0.010937, 0.046235], [7, 0.45575, 0.399353, 0.003688, 0.016494], [1, 0.932883, 0.359247, 0.133891, 0.179435], [7, 0.536852, 0.310753, 0.020047, 0.066024], [1, 0.173344, 0.520788, 0.34325, 0.392212], [0, 0.860836, 0.420082, 0.016391, 0.061906]]}, {"filename": "000000362657.jpg", "coco_split": "train2017", "split": "test", "boxes": [[7, 0.685177, 0.358055, 0.251014, 0.469672], [7, 0.340955, 0.351039, 0.326769, 0.491766]]}, {"filename": "000000363673.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.057822, 0.109367, 0.073208, 0.096766], [0, 0.095035, 0.232539, 0.149742, 0.231859], [0, 0.435047, 0.110203, 0.124426, 0.180125], [0, 0.019696, 0.190258, 0.039391, 0.159734], [0, 0.207904, 0.170383, 0.120867, 0.256766], [3, 0.047108, 0.315594, 0.094215, 0.082844], [0, 0.430351, 0.471531, 0.571335, 0.628719]]}, {"filename": "000000365021.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.498594, 0.335956, 0.210875, 0.276387], [0, 0.331711, 0.378811, 0.139453, 0.211562], [0, 0.540797, 0.603625, 0.036344, 0.120023], [0, 0.655609, 0.583555, 0.036344, 0.149068], [0, 0.866672, 0.59359, 0.040437, 0.068718], [0, 0.92025, 0.547133, 0.050219, 0.151841], [0, 0.610914, 0.593438, 0.042484, 0.13303], [0, 0.683383, 0.589441, 0.028078, 0.13049], [0, 0.960422, 0.597145, 0.030406, 0.041841], [0, 0.794359, 0.629103, 0.011781, 0.011725]]}, {"filename": "000000368684.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.377227, 0.488417, 0.268453, 0.413958]]}, {"filename": "000000369217.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.559719, 0.680304, 0.322406, 0.586136], [4, 0.759188, 0.549215, 0.109219, 0.448407], [3, 0.317141, 0.520445, 0.239344, 0.470867], [3, 0.25, 0.310187, 0.210844, 0.086745]]}, {"filename": "000000372260.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.13774, 0.65793, 0.27548, 0.329453], [1, 0.310773, 0.499172, 0.067728, 0.045656], [1, 0.247752, 0.499086, 0.08815, 0.064672], [1, 0.045422, 0.477664, 0.074824, 0.024203]]}, {"filename": "000000372466.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.098133, 0.092167, 0.101234, 0.091222]]}, {"filename": "000000373705.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.837211, 0.323501, 0.136516, 0.341007], [0, 0.356758, 0.388981, 0.067859, 0.170187], [0, 0.179922, 0.335585, 0.065969, 0.27192], [0, 0.232, 0.420012, 0.054781, 0.117166], [0, 0.007969, 0.405035, 0.01575, 0.063185], [0, 0.984414, 0.403056, 0.031016, 0.17103], [5, 0.719906, 0.691358, 0.296531, 0.6074]]}, {"filename": "000000374977.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.28557, 0.548927, 0.525766, 0.300438], [7, 0.104047, 0.41749, 0.044125, 0.085479], [0, 0.763969, 0.695354, 0.150406, 0.524792], [1, 0.766727, 0.493104, 0.464266, 0.329375], [5, 0.468391, 0.446896, 0.02975, 0.02725]]}, {"filename": "000000379360.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.03108, 0.69365, 0.06216, 0.1177]]}, {"filename": "000000388846.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.866938, 0.860023, 0.266125, 0.178454], [0, 0.546125, 0.72993, 0.049844, 0.074988], [0, 0.575805, 0.720234, 0.027828, 0.070445], [0, 0.544617, 0.672728, 0.026891, 0.122646], [0, 0.486555, 0.741487, 0.091297, 0.048267], [0, 0.129852, 0.819016, 0.116422, 0.076066], [0, 0.288078, 0.754684, 0.036687, 0.067073], [0, 0.768023, 0.746674, 0.155328, 0.068384], [0, 0.662789, 0.626569, 0.017047, 0.068126], [0, 0.414633, 0.743021, 0.029266, 0.071101], [0, 0.207375, 0.806897, 0.042469, 0.043208], [4, 0.839969, 0.821874, 0.234875, 0.179578], [0, 0.132367, 0.809801, 0.038391, 0.028361], [0, 0.585828, 0.655234, 0.016344, 0.030843], [0, 0.949797, 0.53055, 0.006438, 0.029297], [4, 0.856109, 0.90952, 0.010625, 0.025738], [4, 0.776797, 0.743361, 0.075375, 0.018478], [4, 0.724047, 0.73178, 0.030094, 0.022108]]}, {"filename": "000000389197.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.391383, 0.638674, 0.253828, 0.301338], [0, 0.848109, 0.050563, 0.077781, 0.083146], [0, 0.774523, 0.041925, 0.090766, 0.083427], [0, 0.92707, 0.06615, 0.081141, 0.128404], [0, 0.424094, 0.140329, 0.134406, 0.280657], [0, 0.516891, 0.064566, 0.053688, 0.125798], [0, 0.67757, 0.048908, 0.072047, 0.097817], [0, 0.739141, 0.047465, 0.050219, 0.094507], [0, 0.571562, 0.060469, 0.066781, 0.120939], [0, 0.619773, 0.019401, 0.065766, 0.038803]]}, {"filename": "000000394713.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.833328, 0.767031, 0.159281, 0.396865], [0, 0.225117, 0.635819, 0.054828, 0.138907], [3, 0.942477, 0.764572, 0.115047, 0.183872], [3, 0.564641, 0.7112, 0.187688, 0.177126], [3, 0.361086, 0.682435, 0.123391, 0.137031], [3, 0.244977, 0.649181, 0.059234, 0.095511], [0, 0.516969, 0.601271, 0.028688, 0.069952], [0, 0.36518, 0.596746, 0.017203, 0.04209], [3, 0.545812, 0.607779, 0.016344, 0.038314], [3, 0.889875, 0.629952, 0.074094, 0.026651], [0, 0.46775, 0.595689, 0.035062, 0.032185], [4, 0.543984, 0.609679, 0.019094, 0.040736], [0, 0.306602, 0.588587, 0.024203, 0.047625]]}, {"filename": "000000395575.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.30557, 0.401785, 0.05706, 0.13885], [3, 0.49711, 0.417271, 0.12818, 0.102743]]}, {"filename": "000000395865.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.98325, 0.734573, 0.0335, 0.067104], [1, 0.897133, 0.728906, 0.041859, 0.035896], [1, 0.876195, 0.720865, 0.014984, 0.041521], [1, 0.722008, 0.741115, 0.047203, 0.046354], [1, 0.809391, 0.722708, 0.038844, 0.028667], [1, 0.74925, 0.736469, 0.025281, 0.040729], [1, 0.82382, 0.72024, 0.025984, 0.033396], [5, 0.862906, 0.836344, 0.038469, 0.100562], [0, 0.660766, 0.729552, 0.010531, 0.050063], [1, 0.698063, 0.737, 0.007344, 0.012458]]}, {"filename": "000000397354.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.895789, 0.873115, 0.204109, 0.231104], [4, 0.429266, 0.918469, 0.23975, 0.140396], [0, 0.578391, 0.50751, 0.097906, 0.374062], [0, 0.474766, 0.536427, 0.183156, 0.341063], [0, 0.869664, 0.741573, 0.256172, 0.480896], [0, 0.279414, 0.53849, 0.127016, 0.369146], [0, 0.680805, 0.719948, 0.303547, 0.553771], [0, 0.875812, 0.736344, 0.134125, 0.120104]]}, {"filename": "000000402897.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.359614, 0.247523, 0.411472, 0.288578]]}, {"filename": "000000404229.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.220719, 0.404931, 0.330437, 0.735417]]}, {"filename": "000000405093.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.534438, 0.533299, 0.549063, 0.706557]]}, {"filename": "000000406514.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.080273, 0.501908, 0.052297, 0.113655], [1, 0.695773, 0.713207, 0.272578, 0.134], [1, 0.453828, 0.684218, 0.072469, 0.058644], [1, 0.357508, 0.683195, 0.048297, 0.041287], [1, 0.28968, 0.679356, 0.049828, 0.050851], [1, 0.496367, 0.696126, 0.019234, 0.018874], [1, 0.453367, 0.684483, 0.074734, 0.059034], [1, 0.299578, 0.678805, 0.073937, 0.047126], [1, 0.549133, 0.69331, 0.043109, 0.034989]]}, {"filename": "000000412136.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.220229, 0.626969, 0.440458, 0.696625]]}, {"filename": "000000414133.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.864047, 0.558427, 0.271906, 0.289896], [1, 0.054453, 0.492583, 0.069219, 0.043875], [1, 0.089477, 0.476031, 0.032234, 0.028563], [1, 0.35668, 0.475844, 0.088734, 0.069729], [1, 0.124781, 0.465635, 0.034625, 0.011104], [5, 0.140594, 0.480802, 0.008219, 0.016063], [6, 0.523352, 0.517115, 0.126922, 0.166271]]}, {"filename": "000000414882.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.470346, 0.56989, 0.94003, 0.48778], [7, 0.651717, 0.52717, 0.486265, 0.46638], [1, 0.952636, 0.38606, 0.077078, 0.09848]]}, {"filename": "000000416639.jpg", "coco_split": "train2017", "split": "test", "boxes": [[3, 0.150719, 0.764948, 0.177063, 0.125479]]}, {"filename": "000000417043.jpg", "coco_split": "val2017", "split": "test", "boxes": [[3, 0.195508, 0.728094, 0.391016, 0.521354]]}, {"filename": "000000424975.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.470227, 0.70126, 0.044422, 0.149271], [1, 0.874508, 0.674896, 0.045734, 0.0465], [0, 0.406148, 0.72999, 0.055766, 0.192688], [6, 0.287687, 0.485479, 0.0565, 0.098708]]}, {"filename": "000000425221.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.074021, 0.704609, 0.136042, 0.165688], [0, 0.770938, 0.774891, 0.096167, 0.068844], [3, 0.761042, 0.895508, 0.437458, 0.186516], [3, 0.349833, 0.885789, 0.196917, 0.225172], [3, 0.485563, 0.839234, 0.134083, 0.116656], [3, 0.179323, 0.862672, 0.111479, 0.083062], [4, 0.855729, 0.908531, 0.201792, 0.182938], [4, 0.635719, 0.857289, 0.215646, 0.091266], [4, 0.74725, 0.865281, 0.127625, 0.111781], [4, 0.209948, 0.92925, 0.288229, 0.116781], [4, 0.363594, 0.845594, 0.175521, 0.129719], [4, 0.194385, 0.900898, 0.166062, 0.032922], [4, 0.405167, 0.788187, 0.044958, 0.019656], [4, 0.789198, 0.888117, 0.200104, 0.148703], [4, 0.656792, 0.783516, 0.043333, 0.019906], [3, 0.633625, 0.809547, 0.157583, 0.063], [4, 0.362021, 0.78607, 0.034458, 0.023922], [4, 0.335125, 0.806008, 0.13075, 0.061141], [4, 0.227979, 0.93775, 0.183458, 0.040031]]}, {"filename": "000000427160.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.743375, 0.732588, 0.275062, 0.503379], [0, 0.886203, 0.690586, 0.174562, 0.480195], [0, 0.201789, 0.692822, 0.392828, 0.582949]]}, {"filename": "000000428280.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.18404, 0.756156, 0.35376, 0.376336], [4, 0.55661, 0.509114, 0.13702, 0.103574], [4, 0.41949, 0.444459, 0.08814, 0.199429], [3, 0.18293, 0.753363, 0.36286, 0.399099], [4, 0.51243, 0.749024, 0.21054, 0.419069]]}, {"filename": "000000428562.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.132297, 0.170792, 0.176969, 0.332583], [0, 0.523047, 0.05525, 0.05575, 0.109458], [0, 0.352711, 0.06751, 0.035328, 0.131646], [0, 0.45375, 0.061302, 0.028562, 0.122604], [0, 0.487648, 0.052594, 0.024047, 0.104646], [0, 0.664117, 0.089229, 0.103578, 0.178458]]}, {"filename": "000000430048.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.51067, 0.681333, 0.16854, 0.110133]]}, {"filename": "000000432553.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.265351, 0.243562, 0.157588, 0.2425], [0, 0.69034, 0.407656, 0.428735, 0.808563], [3, 0.786405, 0.22857, 0.41267, 0.457141]]}, {"filename": "000000435745.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.53964, 0.579033, 0.01592, 0.064698], [0, 0.6185, 0.550402, 0.02568, 0.084171], [0, 0.50467, 0.552977, 0.03222, 0.073392], [1, 0.83929, 0.319397, 0.17934, 0.246834], [1, 0.71446, 0.407035, 0.07844, 0.160402], [1, 0.82861, 0.852123, 0.1891, 0.227261], [1, 0.813, 0.826897, 0.20512, 0.274548], [1, 0.67095, 0.750314, 0.0653, 0.232186], [1, 0.63005, 0.712161, 0.0461, 0.173769], [1, 0.65684, 0.693857, 0.11432, 0.164146], [1, 0.7458, 0.811947, 0.13624, 0.292789], [1, 0.41918, 0.764234, 0.19828, 0.318719], [1, 0.68619, 0.724523, 0.1399, 0.198342], [1, 0.36242, 0.83059, 0.32028, 0.258819], [1, 0.6528, 0.431156, 0.04668, 0.127085], [1, 0.25032, 0.707965, 0.06964, 0.165276]]}, {"filename": "000000439994.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.862757, 0.811063, 0.15271, 0.237594], [0, 0.143119, 0.199023, 0.08493, 0.116797], [0, 0.525561, 0.250047, 0.08743, 0.062281], [0, 0.204603, 0.514648, 0.077243, 0.074172], [0, 0.498446, 0.520281, 0.094743, 0.064281], [0, 0.421145, 0.472047, 0.034112, 0.036437], [0, 0.405023, 0.519945, 0.071075, 0.071578], [0, 0.65965, 0.244906, 0.054112, 0.042406], [0, 0.33729, 0.750563, 0.192336, 0.194531], [0, 0.319778, 0.521297, 0.062126, 0.065219], [0, 0.552967, 0.481695, 0.062196, 0.067484], [0, 0.661332, 0.458203, 0.067617, 0.040656], [3, 0.505829, 0.170125, 0.988341, 0.335062], [0, 0.752056, 0.279375, 0.066916, 0.03125], [3, 0.405339, 0.68157, 0.650771, 0.036797], [3, 0.499089, 0.424609, 0.998178, 0.311688], [4, 0.448119, 0.133781, 0.033762, 0.02025], [4, 0.567477, 0.445734, 0.03715, 0.015937], [4, 0.430245, 0.173898, 0.044182, 0.012016], [4, 0.377021, 0.472164, 0.045911, 0.018234], [4, 0.605339, 0.447164, 0.036893, 0.015703], [4, 0.867991, 0.438883, 0.032243, 0.013984]]}, {"filename": "000000442323.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.809797, 0.616437, 0.23475, 0.328417]]}, {"filename": "000000443209.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.021458, 0.274977, 0.042917, 0.090359], [1, 0.523458, 0.21693, 0.953083, 0.432547], [5, 0.300604, 0.626961, 0.589083, 0.746078], [0, 0.053375, 0.29243, 0.071208, 0.267016]]}, {"filename": "000000447849.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.50075, 0.493794, 0.9985, 0.984215], [7, 0.514258, 0.511674, 0.449797, 0.948314], [0, 0.496266, 0.770785, 0.194906, 0.458431]]}, {"filename": "000000448365.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.405984, 0.43639, 0.346094, 0.71236], [1, 0.712133, 0.568621, 0.031328, 0.068505]]}, {"filename": "000000462602.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.515187, 0.533352, 0.353888, 0.456766]]}, {"filename": "000000465353.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.9075, 0.783679, 0.1845, 0.209784], [0, 0.555883, 0.808208, 0.051734, 0.252857], [0, 0.167375, 0.725135, 0.039125, 0.120296], [0, 0.266641, 0.761213, 0.022719, 0.156712], [0, 0.234734, 0.76531, 0.028531, 0.174501], [0, 0.412383, 0.78093, 0.035641, 0.103315], [0, 0.283859, 0.776105, 0.048562, 0.183477], [0, 0.229617, 0.773275, 0.035797, 0.143046], [3, 0.473195, 0.798423, 0.077422, 0.061806], [3, 0.639773, 0.776321, 0.056109, 0.058733], [0, 0.603445, 0.694218, 0.020516, 0.094097], [1, 0.113062, 0.820795, 0.2015, 0.231725], [3, 0.685641, 0.753908, 0.013938, 0.046846], [3, 0.276625, 0.798504, 0.068188, 0.012318], [0, 0.383797, 0.777385, 0.039875, 0.114879]]}, {"filename": "000000468501.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.63337, 0.424947, 0.0421, 0.177093], [0, 0.8565, 0.529147, 0.287, 0.385653], [0, 0.54775, 0.595507, 0.22246, 0.782], [4, 0.46158, 0.499827, 0.13112, 0.332347], [0, 0.38789, 0.530707, 0.25674, 0.632613], [4, 0.53377, 0.542067, 0.04286, 0.14024]]}, {"filename": "000000469914.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.656656, 0.770604, 0.083687, 0.388417], [0, 0.515937, 0.509156, 0.357375, 0.959729], [4, 0.90118, 0.845729, 0.197641, 0.288292], [4, 0.953555, 0.516417, 0.092891, 0.126708]]}, {"filename": "000000469921.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.39182, 0.740603, 0.097953, 0.14158], [1, 0.424937, 0.575582, 0.076937, 0.063098], [1, 0.213648, 0.619096, 0.420547, 0.415738], [1, 0.448805, 0.540863, 0.079141, 0.090582], [0, 0.752969, 0.575863, 0.016812, 0.014012], [1, 0.787883, 0.6371, 0.188797, 0.184927], [1, 0.562273, 0.555249, 0.042453, 0.045676], [1, 0.586898, 0.551123, 0.020359, 0.037256], [1, 0.602758, 0.547121, 0.026016, 0.031497], [1, 0.617086, 0.545842, 0.017578, 0.025821], [1, 0.688867, 0.544802, 0.020484, 0.021455], [1, 0.378664, 0.554626, 0.017859, 0.02131], [1, 0.511148, 0.563067, 0.058266, 0.049834], [1, 0.707805, 0.543025, 0.019641, 0.013867]]}, {"filename": "000000476902.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.425469, 0.117717, 0.141781, 0.122037], [0, 0.943805, 0.362927, 0.097453, 0.501124], [0, 0.402719, 0.465937, 0.159281, 0.782084], [3, 0.827531, 0.111698, 0.344938, 0.083115], [3, 0.413727, 0.084602, 0.315016, 0.105176]]}, {"filename": "000000477149.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.2425, 0.890883, 0.086375, 0.061734], [0, 0.956656, 0.880305, 0.085187, 0.061953], [0, 0.255167, 0.881953, 0.087583, 0.068969], [0, 0.648729, 0.881461, 0.0845, 0.043453], [0, 0.132646, 0.883078, 0.048583, 0.038781], [0, 0.608083, 0.883391, 0.064125, 0.038156]]}, {"filename": "000000483108.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.612103, 0.773, 0.152617, 0.243594], [0, 0.572862, 0.700961, 0.21764, 0.327797], [6, 0.755561, 0.357734, 0.144019, 0.108031]]}, {"filename": "000000485188.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.776844, 0.890732, 0.108781, 0.214686], [4, 0.731609, 0.824498, 0.087875, 0.186276], [4, 0.837453, 0.930178, 0.108594, 0.13818], [0, 0.621289, 0.588096, 0.101859, 0.284644], [0, 0.423305, 0.591621, 0.106453, 0.379937], [0, 0.355891, 0.742071, 0.157281, 0.451004], [0, 0.104055, 0.582029, 0.140984, 0.4], [0, 0.159445, 0.500146, 0.070484, 0.231464], [0, 0.107789, 0.4134, 0.053703, 0.191025], [0, 0.870891, 0.533494, 0.095781, 0.238745], [0, 0.460992, 0.553954, 0.051984, 0.25251], [4, 0.907859, 0.977364, 0.119063, 0.045272], [4, 0.691242, 0.786998, 0.079578, 0.166381], [0, 0.050891, 0.404226, 0.055156, 0.216151], [0, 0.019609, 0.45251, 0.033625, 0.183222]]}, {"filename": "000000486123.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.233422, 0.830781, 0.176969, 0.148312], [1, 0.408484, 0.804125, 0.049937, 0.047458], [1, 0.493023, 0.798823, 0.061047, 0.073854], [0, 0.707469, 0.803135, 0.031625, 0.132187], [5, 0.603922, 0.81324, 0.012875, 0.033813], [1, 0.540773, 0.782865, 0.011203, 0.013562], [1, 0.81657, 0.768854, 0.046516, 0.018417], [1, 0.729477, 0.775635, 0.012234, 0.008521]]}, {"filename": "000000487698.jpg", "coco_split": "train2017", "split": "test", "boxes": [[5, 0.655281, 0.6, 0.20675, 0.741563], [0, 0.387773, 0.402148, 0.273984, 0.62418], [3, 0.021352, 0.519121, 0.042703, 0.207891]]}, {"filename": "000000490171.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.272594, 0.653466, 0.370938, 0.146792]]}, {"filename": "000000493329.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.503365, 0.498875, 0.587271, 0.417969]]}, {"filename": "000000494188.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.349823, 0.415055, 0.309688, 0.455922], [4, 0.18351, 0.467742, 0.367021, 0.935484], [0, 0.622854, 0.324797, 0.187792, 0.40475], [0, 0.460635, 0.371883, 0.130104, 0.326672], [0, 0.7235, 0.413109, 0.031375, 0.098094], [0, 0.296792, 0.219984, 0.378833, 0.439969], [0, 0.819073, 0.408531, 0.033771, 0.035031], [4, 0.847729, 0.441695, 0.019042, 0.025016], [0, 0.075885, 0.275672, 0.151771, 0.132594], [0, 0.520094, 0.425844, 0.028396, 0.065562], [4, 0.792229, 0.441859, 0.021167, 0.026625]]}, {"filename": "000000494913.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.779578, 0.505537, 0.058469, 0.14514], [0, 0.897437, 0.664451, 0.164719, 0.251005]]}, {"filename": "000000497599.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.810234, 0.707383, 0.177799, 0.245078], [0, 0.390703, 0.495508, 0.781405, 0.986516], [4, 0.958829, 0.702172, 0.082342, 0.169344], [0, 0.890691, 0.39707, 0.218618, 0.239609], [0, 0.921932, 0.570445, 0.156136, 0.226047], [4, 0.759297, 0.556523, 0.106557, 0.075453], [0, 0.499602, 0.752602, 0.30445, 0.450516]]}, {"filename": "000000498286.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.818695, 0.787719, 0.295797, 0.174563], [1, 0.818562, 0.788635, 0.286125, 0.172896]]}, {"filename": "000000498533.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.51404, 0.49892, 0.55264, 0.379547], [7, 0.38596, 0.43752, 0.06404, 0.211253]]}, {"filename": "000000498638.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.5941, 0.56644, 0.38485, 0.74156], [3, 0.850287, 0.55846, 0.297325, 0.2484], [0, 0.161375, 0.35901, 0.07365, 0.18966]]}, {"filename": "000000498730.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.482023, 0.543817, 0.205391, 0.215738], [1, 0.368664, 0.51144, 0.075766, 0.213232], [0, 0.65375, 0.499098, 0.158219, 0.689016], [0, 0.260719, 0.555176, 0.172812, 0.734239], [0, 0.034102, 0.527272, 0.049078, 0.201335], [0, 0.192477, 0.504356, 0.062734, 0.404684], [1, 0.112078, 0.532951, 0.078, 0.085152], [7, 0.56957, 0.414028, 0.023359, 0.159766]]}, {"filename": "000000511076.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.313531, 0.313361, 0.064625, 0.08274], [0, 0.699281, 0.525937, 0.297031, 0.711405], [3, 0.850102, 0.782178, 0.287859, 0.382014], [0, 0.226523, 0.324496, 0.016797, 0.076159]]}, {"filename": "000000512564.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.415852, 0.622795, 0.035922, 0.040542], [1, 0.322203, 0.521144, 0.026406, 0.01908], [1, 0.650305, 0.735802, 0.110984, 0.165189], [1, 0.377492, 0.514057, 0.013391, 0.015849], [1, 0.426281, 0.526392, 0.018906, 0.014906], [1, 0.372156, 0.506297, 0.010156, 0.013302], [0, 0.197492, 0.59658, 0.009234, 0.041604], [0, 0.155891, 0.647476, 0.009969, 0.052123], [0, 0.650664, 0.730802, 0.012422, 0.028302], [1, 0.221617, 0.714151, 0.058453, 0.079198], [1, 0.231539, 0.871014, 0.088953, 0.140991]]}, {"filename": "000000521059.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.362805, 0.69625, 0.037266, 0.041583], [1, 0.196578, 0.688573, 0.037906, 0.041396], [1, 0.637922, 0.683771, 0.135406, 0.066708], [1, 0.499383, 0.688448, 0.138047, 0.076938], [1, 0.974969, 0.669042, 0.049969, 0.07575], [1, 0.854906, 0.674302, 0.191125, 0.114479], [5, 0.711227, 0.743333, 0.033391, 0.122042], [6, 0.190398, 0.608437, 0.015141, 0.041542], [1, 0.178398, 0.696156, 0.012172, 0.018688]]}, {"filename": "000000521819.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.794664, 0.776406, 0.058984, 0.056187], [1, 0.906953, 0.780313, 0.065844, 0.0555], [1, 0.175844, 0.78924, 0.024906, 0.030229], [1, 0.130625, 0.79326, 0.025281, 0.026979], [0, 0.209703, 0.734479, 0.088563, 0.286333], [0, 0.385797, 0.718417, 0.0895, 0.320833], [0, 0.449367, 0.718167, 0.090109, 0.328833], [0, 0.856414, 0.741635, 0.053422, 0.271063], [0, 0.547766, 0.759354, 0.055687, 0.220333], [0, 0.609609, 0.755604, 0.090094, 0.246625], [0, 0.342516, 0.711146, 0.0775, 0.295583], [0, 0.304937, 0.764656, 0.043656, 0.134646], [0, 0.500281, 0.725698, 0.091656, 0.300646]]}, {"filename": "000000521956.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.34706, 0.496258, 0.69412, 0.984273]]}, {"filename": "000000522273.jpg", "coco_split": "train2017", "split": "test", "boxes": [[0, 0.850023, 0.392135, 0.093266, 0.438219], [0, 0.142383, 0.582886, 0.284766, 0.585773], [0, 0.652234, 0.442704, 0.07075, 0.224721], [3, 0.674414, 0.433455, 0.298734, 0.084421], [0, 0.740062, 0.407897, 0.127, 0.416223], [0, 0.368039, 0.457371, 0.418516, 0.756888]]}, {"filename": "000000523194.jpg", "coco_split": "val2017", "split": "test", "boxes": [[5, 0.298961, 0.649412, 0.049859, 0.160565], [6, 0.868555, 0.268871, 0.092672, 0.1208]]}, {"filename": "000000527501.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.315711, 0.42393, 0.218818, 0.161859]]}, {"filename": "000000531134.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.220789, 0.743823, 0.064047, 0.224729], [0, 0.083719, 0.942177, 0.106188, 0.115646], [0, 0.589187, 0.805667, 0.061531, 0.254333], [0, 0.643672, 0.775604, 0.062375, 0.22525], [0, 0.685922, 0.767542, 0.034969, 0.195542], [0, 0.769359, 0.534708, 0.021188, 0.075458], [0, 0.195805, 0.965792, 0.043078, 0.068417], [1, 0.853617, 0.969781, 0.091328, 0.060437]]}, {"filename": "000000534669.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.78757, 0.756896, 0.126234, 0.0805], [1, 0.787805, 0.75849, 0.128516, 0.079979], [1, 0.024133, 0.78201, 0.048266, 0.071438]]}, {"filename": "000000534827.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.700417, 0.606479, 0.418186, 0.241258], [1, 0.937892, 0.562614, 0.124216, 0.154771], [1, 0.550172, 0.541757, 0.130866, 0.140801], [0, 0.735106, 0.523962, 0.151422, 0.30969], [0, 0.246144, 0.496111, 0.138105, 0.254869], [0, 0.632672, 0.472173, 0.083644, 0.130033], [0, 0.979534, 0.499665, 0.040931, 0.09116], [1, 0.231667, 0.564436, 0.402778, 0.213807], [1, 0.955531, 0.600449, 0.088938, 0.155833]]}, {"filename": "000000538458.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.804578, 0.417342, 0.02725, 0.03281], [0, 0.88082, 0.491007, 0.238359, 0.483138], [0, 0.542742, 0.255187, 0.159016, 0.344567]]}, {"filename": "000000542499.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.731883, 0.263361, 0.217422, 0.379778], [0, 0.894312, 0.314597, 0.094812, 0.373028], [1, 0.995469, 0.176028, 0.009062, 0.033389], [1, 0.99182, 0.163111, 0.016359, 0.017278], [1, 0.968063, 0.161083, 0.032, 0.0285]]}, {"filename": "000000542625.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.677531, 0.372667, 0.182031, 0.130333], [1, 0.169266, 0.419729, 0.077312, 0.080417], [1, 0.115453, 0.371917, 0.224156, 0.739333], [7, 0.428937, 0.772656, 0.096062, 0.359562], [1, 0.484461, 0.190542, 0.019172, 0.018542], [5, 0.803133, 0.289062, 0.007672, 0.0245], [5, 0.802828, 0.302458, 0.025563, 0.056083], [1, 0.906508, 0.393917, 0.186984, 0.162917], [1, 0.724023, 0.729865, 0.551953, 0.535063], [1, 0.987539, 0.423427, 0.024922, 0.085521]]}, {"filename": "000000547097.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.287219, 0.58733, 0.27525, 0.493255]]}, {"filename": "000000551822.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.122625, 0.159205, 0.127438, 0.239426]]}, {"filename": "000000553221.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.618219, 0.372963, 0.171125, 0.261944], [0, 0.634703, 0.704204, 0.007625, 0.048735], [0, 0.339695, 0.744555, 0.023078, 0.058618], [0, 0.372961, 0.682869, 0.010453, 0.022693], [0, 0.389617, 0.682775, 0.010078, 0.019274], [0, 0.42075, 0.685609, 0.008594, 0.022646], [0, 0.670477, 0.738934, 0.028109, 0.05281], [0, 0.717945, 0.733747, 0.016828, 0.06459], [0, 0.754461, 0.685457, 0.009891, 0.029649], [0, 0.347039, 0.786429, 0.040391, 0.063302], [0, 0.491992, 0.697155, 0.010547, 0.03829], [0, 0.291922, 0.713407, 0.008406, 0.041897], [0, 0.403969, 0.688407, 0.008094, 0.016815]]}, {"filename": "000000553669.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.671563, 0.235656, 0.218406, 0.083729], [1, 0.864031, 0.256792, 0.194094, 0.069042], [0, 0.24468, 0.695073, 0.344734, 0.587438], [0, 0.169484, 0.253167, 0.065625, 0.073917], [0, 0.122555, 0.248562, 0.041422, 0.091], [3, 0.276406, 0.785396, 0.542687, 0.240458], [1, 0.378977, 0.185042, 0.170109, 0.052208], [1, 0.504492, 0.256729, 0.256547, 0.083667], [1, 0.243406, 0.214479, 0.2285, 0.087542], [0, 0.807563, 0.190344, 0.015688, 0.038937], [0, 0.023422, 0.245948, 0.046844, 0.098604], [0, 0.742523, 0.272708, 0.048234, 0.092542], [0, 0.765344, 0.176448, 0.017531, 0.022813], [3, 0.068094, 0.271031, 0.083156, 0.045854], [3, 0.704992, 0.295229, 0.157797, 0.046458], [3, 0.87482, 0.296604, 0.164078, 0.046458], [0, 0.036484, 0.581719, 0.072969, 0.244896], [0, 0.964086, 0.28376, 0.071828, 0.107188], [1, 0.947836, 0.234573, 0.104234, 0.081604], [1, 0.693344, 0.189333, 0.204594, 0.04625], [3, 0.224375, 0.262667, 0.0675, 0.030625], [3, 0.331867, 0.272625, 0.140984, 0.038042]]}, {"filename": "000000561885.jpg", "coco_split": "train2017", "split": "test", "boxes": [[1, 0.494367, 0.484555, 0.939203, 0.923933], [0, 0.975992, 0.645642, 0.025672, 0.086581], [0, 0.597312, 0.621146, 0.064187, 0.127154], [0, 0.348461, 0.45252, 0.083766, 0.212115], [1, 0.013859, 0.610435, 0.027719, 0.178419]]}, {"filename": "000000565563.jpg", "coco_split": "val2017", "split": "test", "boxes": [[1, 0.226688, 0.585385, 0.193813, 0.213479], [1, 0.067633, 0.529677, 0.128422, 0.127854], [1, 0.427156, 0.463135, 0.043, 0.047521], [1, 0.457641, 0.442531, 0.034781, 0.034021], [1, 0.192562, 0.486479, 0.075344, 0.075792], [1, 0.25357, 0.455573, 0.060797, 0.030063], [0, 0.487711, 0.434573, 0.012547, 0.033979], [1, 0.337914, 0.469573, 0.118016, 0.076771], [1, 0.438813, 0.458229, 0.034063, 0.04875], [1, 0.413492, 0.435927, 0.019359, 0.025604], [1, 0.382008, 0.435698, 0.029953, 0.017979]]}, {"filename": "000000566436.jpg", "coco_split": "val2017", "split": "test", "boxes": [[0, 0.371711, 0.334016, 0.512766, 0.668033], [3, 0.410812, 0.606358, 0.788625, 0.759555]]}, {"filename": "000000575199.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.29507, 0.523501, 0.022422, 0.030749], [7, 0.466281, 0.70493, 0.182906, 0.525831], [7, 0.145781, 0.606639, 0.023063, 0.114215]]}, {"filename": "000000575867.jpg", "coco_split": "train2017", "split": "test", "boxes": [[4, 0.1925, 0.5, 0.385, 1.0]]}, {"filename": "000000575970.jpg", "coco_split": "val2017", "split": "test", "boxes": [[4, 0.517437, 0.822104, 0.165781, 0.332625], [4, 0.762906, 0.815052, 0.187094, 0.311812], [4, 0.925758, 0.810427, 0.148484, 0.330438]]}, {"filename": "000000577310.jpg", "coco_split": "train2017", "split": "test", "boxes": [[3, 0.075984, 0.941393, 0.062594, 0.087235], [3, 0.253492, 0.911393, 0.044297, 0.06162], [0, 0.668117, 0.89433, 0.072203, 0.192592], [0, 0.470797, 0.874957, 0.033469, 0.102268], [0, 0.760953, 0.673132, 0.0105, 0.040346], [0, 0.780055, 0.670734, 0.009109, 0.029676], [0, 0.813164, 0.672538, 0.008391, 0.032333], [0, 0.610266, 0.746566, 0.010906, 0.041901], [0, 0.276195, 0.651857, 0.010547, 0.02635], [0, 0.21175, 0.654082, 0.01175, 0.034687], [0, 0.737203, 0.894158, 0.071063, 0.207365], [0, 0.558656, 0.663952, 0.020937, 0.045572], [0, 0.548312, 0.679114, 0.00975, 0.04324], [3, 0.414914, 0.891566, 0.024234, 0.041145], [3, 0.520602, 0.886004, 0.018516, 0.016803], [0, 0.532086, 0.875086, 0.010391, 0.045853]]}, {"filename": "000000578292.jpg", "coco_split": "train2017", "split": "test", "boxes": [[6, 0.576781, 0.210109, 0.380521, 0.375281], [0, 0.473406, 0.732586, 0.305604, 0.440453], [1, 0.979896, 0.410109, 0.038417, 0.045313]]}], "visaitech_video_split": {"V001": "train", "V002": "train", "V004": "val", "V005": "val", "V003": "test", "VEHICLES": "test"}, "pothole_hash_split": {"5824ccea6a7e": "train", "83c68383a235": "train", "912764692abf": "train", "7d1960f130aa": "train", "622834f6ae5e": "train", "875fe019d68c": "train", "867408c6fde8": "train", "13783435f15f": "train", "2d61df973a70": "train", "18d7116aaf44": "train", "5c2a0b5926dc": "train", "ea96a52e1d9e": "train", "306fde7e0eef": "train", "d9b7421cfa9f": "train", "c7657fd9382d": "train", "02d99280d41d": "train", "f7c1bee81462": "train", "35392ada0cd2": "train", "96b2da7e1fc7": "train", "d1b87f84bf03": "train", "9c4216fc7e5e": "train", "c262475c799f": "train", "57a36dadf648": "train", "2638aa5c765a": "train", "64bf42f62bd4": "train", "005be57178f7": "train", "39931b3d67a8": "train", "3af7ffc89ae5": "train", "a40a065ceed6": "train", "188eadfb8472": "train", "4a5f8613040d": "train", "eb8354dc9242": "train", "205436224c52": "train", "f0e6f79116fd": "train", "cb4eb00fd062": "train", "fee95a47b026": "train", "e268c84cbf05": "train", "91e04cfa8b6c": "train", "24074dc04417": "train", "09a2276df429": "train", "b54df4b2a214": "train", "8e364731c0ec": "train", "21ecfa948361": "train", "110eb2e9f1d6": "train", "f93cf1e5e533": "train", "94e46a618f02": "train", "b6d110aa3c31": "train", "8bb37bc6566d": "train", "720950ee1483": "train", "85cda723ebab": "train", "ec109ec34b97": "train", "95caf7cdb775": "train", "d5169026a644": "train", "b81c66a1d2ae": "train", "606a3c558805": "train", "2624fc3f9a4e": "train", "71e92d5d12ed": "train", "1f0c2b7b72f6": "train", "cfd4f0d2b0ff": "train", "43c5df20279e": "train", "2be8d5e7d0df": "train", "77b0d907d10d": "val", "70b88e2386bc": "val", "8ae09f652602": "val", "ef69ece9bf93": "val", "5ac424858c97": "val", "cf58d00e8dee": "val", "6ee61522680d": "val", "7042d5487254": "val", "17e3ff7509bd": "val", "5d858683c387": "val", "c123758c8307": "val", "896419e36621": "val", "eb80a8b9caf3": "val", "fa4396ccbbee": "test", "932202568a3a": "test", "69ef91d139d9": "test", "6d2659e48cd6": "test", "7efc64cac6ea": "test", "9a7be5655d81": "test", "95d869e14a39": "test", "f901e43b715b": "test", "25e0b01a21ff": "test", "cca532c77506": "test", "debda1bbba65": "test", "7482b4b0672d": "test", "dd1526d2c946": "test", "2eb738d57725": "test", "28891a2b9616": "test", "bcf2a55b4801": "test"}}'''


In [ ]:
import concurrent.futures
import hashlib
import json as _json
import pathlib
import shutil
from collections import Counter
from urllib.request import Request, urlopen

manifest = _json.loads(MANIFEST_JSON)
assert manifest["general_classes"] == GENERAL_CLASSES, (
    f"manifest class order {manifest['general_classes']} != GENERAL_CLASSES {GENERAL_CLASSES}"
)

DATA_ROOT_GENERAL = pathlib.Path("/content/yolo_dataset_coco")
if DATA_ROOT_GENERAL.exists():
    shutil.rmtree(DATA_ROOT_GENERAL)
for split in ("train", "val", "test"):
    (DATA_ROOT_GENERAL / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_ROOT_GENERAL / "labels" / split).mkdir(parents=True, exist_ok=True)

USER_AGENT = "EgyptianAssistiveVisionDataset/1.0 (research project)"

def write_item(name: str, split: str, boxes: list, image_bytes: bytes) -> None:
    (DATA_ROOT_GENERAL / "images" / split / f"{name}.jpg").write_bytes(image_bytes)
    lines = [f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in boxes]
    (DATA_ROOT_GENERAL / "labels" / split / f"{name}.txt").write_text(
        "\n".join(lines) + ("\n" if lines else "")
    )

# -- avn_coco: re-download each sampled image from the official public COCO 2017 mirror --
COCO_BASE = "http://images.cocodataset.org"

def fetch_coco(item):
    url = f"{COCO_BASE}/{item['coco_split']}/{item['filename']}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=30) as response:
        return item, response.read()

coco_ok = coco_failed = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as pool:
    for item, image_bytes in pool.map(fetch_coco, manifest["avn_coco_items"]):
        try:
            name = f"avn_coco_{pathlib.Path(item['filename']).stem}"
            write_item(name, item["split"], item["boxes"], image_bytes)
            coco_ok += 1
        except Exception as error:  # noqa: BLE001
            print(f"FAILED {item['filename']}: {type(error).__name__}: {error}")
            coco_failed += 1
print(f"avn_coco: {coco_ok} downloaded, {coco_failed} failed (expected {len(manifest['avn_coco_items'])})")
assert coco_failed == 0, "some COCO images failed to download -- do not proceed with a partial set"


In [ ]:
# -- visaitech: same re-fetch as Option D, split assigned by this session's video_id lookup --
from huggingface_hub import snapshot_download

VISAITECH_CLASS_MAP = {
    "pedestrian": "person", "2-wheeler": "transportation",
    "3-wheeler": "transportation", "4-wheeler": "transportation",
}
snapshot_dir = pathlib.Path(snapshot_download(
    repo_id="visaitech/vehicle-mixed-traffic-detection", repo_type="dataset",
    allow_patterns=["images/**", "labels/**", "classes.txt"],
))
visaitech_classes = (snapshot_dir / "classes.txt").read_text().strip().splitlines()
video_split = manifest["visaitech_video_split"]

visaitech_written = visaitech_dropped_empty = visaitech_unknown_video = 0
for split_dir in sorted((snapshot_dir / "images").iterdir()):
    for image_path in sorted(split_dir.glob("*.jpg")):
        video_id = image_path.stem.split("__")[0]
        if video_id not in video_split:
            visaitech_unknown_video += 1
            continue
        label_path = snapshot_dir / "labels" / split_dir.name / f"{image_path.stem}.txt"
        kept_lines = []
        if label_path.is_file():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                target_name = VISAITECH_CLASS_MAP[visaitech_classes[int(parts[0])]]
                kept_lines.append((GENERAL_CLASSES.index(target_name), *(float(v) for v in parts[1:])))
        if not kept_lines:
            visaitech_dropped_empty += 1
            continue
        name = f"visaitech_{image_path.stem}"
        write_item(name, video_split[video_id], kept_lines, image_path.read_bytes())
        visaitech_written += 1
print(f"visaitech: {visaitech_written} written, {visaitech_dropped_empty} empty-label dropped, "
      f"{visaitech_unknown_video} from an unrecognized video_id (should be 0)")
assert visaitech_unknown_video == 0


In [ ]:
# -- pothole: same re-fetch as Option D, split assigned by this session's content-hash lookup --
import io
import pandas as pd

POTHOLE_URL = (
    "https://huggingface.co/api/datasets/keremberke/pothole-segmentation/"
    "parquet/full/{split}/0.parquet"
)
pothole_target_id = GENERAL_CLASSES.index("pothole")
hash_split = manifest["pothole_hash_split"]

pothole_written = pothole_unknown_hash = 0
for split in ("train", "validation", "test"):
    request = Request(POTHOLE_URL.format(split=split), headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=60) as response:
        df = pd.read_parquet(io.BytesIO(response.read()))
    for _, row in df.iterrows():
        image_bytes = row["image"]["bytes"]
        digest = hashlib.sha256(image_bytes).hexdigest()[:12]
        if digest not in hash_split:
            pothole_unknown_hash += 1
            continue
        width, height = int(row["width"]), int(row["height"])
        boxes = []
        for x, y, box_w, box_h in row["objects"]["bbox"]:
            cx, cy = (x + box_w / 2) / width, (y + box_h / 2) / height
            boxes.append((pothole_target_id, cx, cy, box_w / width, box_h / height))
        write_item(f"pothole_{digest}", hash_split[digest], boxes, image_bytes)
        pothole_written += 1
print(f"pothole: {pothole_written} written, {pothole_unknown_hash} unrecognized-hash "
      f"(should be 0; a nonzero count means the upstream dataset content changed since this "
      f"session's local copy was fetched)")
assert pothole_unknown_hash == 0


In [ ]:
# -- write dataset.yaml, set the variables Step 7A needs, print honest counts as a sanity check --
yaml_lines = [f'path: "{DATA_ROOT_GENERAL}"', "train: images/train", "val: images/val",
              "test: images/test", "names:"]
for i, name in enumerate(GENERAL_CLASSES):
    yaml_lines.append(f'  {i}: "{name}"')
coco_dataset_yaml = DATA_ROOT_GENERAL / "dataset.yaml"
coco_dataset_yaml.write_text("\n".join(yaml_lines) + "\n")
COCO_GATE_PASSED = True  # already validated locally by build_general_dataset.py's printed report

print("Per-split image/box counts (should match this session's local report):")
for split in ("train", "val", "test"):
    label_files = sorted((DATA_ROOT_GENERAL / "labels" / split).glob("*.txt"))
    box_counts = Counter()
    for label_file in label_files:
        text = label_file.read_text().strip()
        if text:
            for line in text.splitlines():
                box_counts[GENERAL_CLASSES[int(line.split()[0])]] += 1
    print(f"  [{split}] {len(label_files)} images -- " +
          ", ".join(f"{name}={box_counts[name]}" for name in GENERAL_CLASSES))

print(f"\ncoco_dataset_yaml = {coco_dataset_yaml}")
print("COCO_GATE_PASSED = True")
print("\nRun Option F next for accuracy expansion, then Step 7A. Do not run Steps 2(A-D)/3/4A/5/6.")


In [ ]:
# Option E only builds Part A's (general model) data from public sources. Part B (Egyptian
# hazard model) still has zero real Egyptian-labeled data -- explicitly mark its gate as not
# passed (Step 4B's own logic never ran) so cells 7B-10B skip cleanly instead of NameError'ing.
EGYPT_GATE_PASSED = False
print("EGYPT_GATE_PASSED = False (Option E path: no Part B data; this is honest, not a bug).")

### Option F -- Accuracy expansion (run after Option E; recommended for the next model)

The shipped model was trained on only about 1.5k training images. The local class counts also show
just 490 bench boxes and 936 chair boxes in training, while `person` and `transportation` recall
were only about 0.52 and 0.57. This cell expands **training only** with a class-stratified sample
from the official COCO 2017 training set. It leaves Option E's validation and held-out test files
unchanged, so the next model can be compared on the same split. It also adds a small set of genuine
COCO background images with empty labels to reduce false positives.

COCO officially provides 118k train and 5k validation images. Rather than downloading the full
18GB image archive into Colab, this downloads the 241MB annotation archive and fetches selected
image files directly from the official COCO host. Run this after Option E and before Step 7A. Do
not run Steps 2-6 afterward. The 90-image pothole source remains small; this expansion primarily
targets person, transportation, bench, chair, signs and hydrants.


In [ ]:
# Expand only the training split; Option E's val/test remain frozen for fair comparison.
import concurrent.futures, json, pathlib, random, time, urllib.request, zipfile
from collections import defaultdict, Counter

assert 'DATA_ROOT_GENERAL' in globals() and COCO_GATE_PASSED, 'Run Option E first.'
DATA_ROOT_GENERAL = pathlib.Path(DATA_ROOT_GENERAL)
COCO_ROOT = pathlib.Path('/content/coco2017')
ANNOTATIONS_DIR = COCO_ROOT / 'annotations'
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_JSON = ANNOTATIONS_DIR / 'instances_train2017.json'
ANNOTATION_ZIP = COCO_ROOT / 'annotations_trainval2017.zip'
if not ANNOTATION_JSON.exists():
    urllib.request.urlretrieve(
        'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
        ANNOTATION_ZIP,
    )
    with zipfile.ZipFile(ANNOTATION_ZIP) as archive:
        member = next(n for n in archive.namelist() if n.endswith('instances_train2017.json'))
        ANNOTATION_JSON.write_bytes(archive.read(member))

COCO_CLASS_TO_TARGET = {
    'person': 'person', 'bicycle': 'transportation', 'car': 'transportation',
    'motorcycle': 'transportation', 'bus': 'transportation', 'truck': 'transportation',
    'bench': 'bench', 'chair': 'chair', 'fire hydrant': 'fire hydrant',
    'stop sign': 'stop sign', 'parking meter': 'parking meter',
}
MAX_EXTRA_IMAGES_PER_COCO_CLASS = 2500
BACKGROUND_IMAGES = 500
rng = random.Random(SPLIT_SEED + 1)
train_json = json.loads(ANNOTATION_JSON.read_text())
images = {item['id']: item for item in train_json['images']}
category_names = {item['id']: item['name'] for item in train_json['categories']}
category_to_target = {
    cid: GENERAL_CLASSES.index(COCO_CLASS_TO_TARGET[name])
    for cid, name in category_names.items() if name in COCO_CLASS_TO_TARGET
}
annotations_by_image = defaultdict(list)
image_ids_by_category = defaultdict(set)
for ann in train_json['annotations']:
    cid = ann['category_id']
    if cid not in category_to_target or ann.get('iscrowd', 0):
        continue
    annotations_by_image[ann['image_id']].append(ann)
    image_ids_by_category[cid].add(ann['image_id'])

# Never include an image already in Option E's train, val, or test split.
reserved_stems = set()
for split in ('train', 'val', 'test'):
    for image_path in (DATA_ROOT_GENERAL / 'images' / split).glob('*'):
        stem = image_path.stem
        if stem.startswith('avn_coco_'):
            reserved_stems.add(stem.removeprefix('avn_coco_') + '.jpg')

selected_ids = set()
selected_by_name = {}
for cid, target_id in category_to_target.items():
    raw_name = category_names[cid]
    candidates = sorted(iid for iid in image_ids_by_category[cid]
                  if images[iid]['file_name'] not in reserved_stems)
    rng.shuffle(candidates)
    chosen = candidates[:MAX_EXTRA_IMAGES_PER_COCO_CLASS]
    selected_ids.update(chosen)
    selected_by_name[raw_name] = len(chosen)

# Background-only negatives: no mapped object category is present in COCO ground truth.
negative_candidates = [iid for iid, item in images.items()
                       if iid not in selected_ids
                       and item['file_name'] not in reserved_stems
                       and not annotations_by_image.get(iid)]
rng.shuffle(negative_candidates)
negative_ids = set(negative_candidates[:BACKGROUND_IMAGES])
selected_ids.update(negative_ids)

# Keep new COCO examples training-only. The original val and test are unchanged.
train_images_dir = DATA_ROOT_GENERAL / 'images' / 'train'
train_labels_dir = DATA_ROOT_GENERAL / 'labels' / 'train'
train_images_dir.mkdir(parents=True, exist_ok=True)
train_labels_dir.mkdir(parents=True, exist_ok=True)
user_agent = 'AVN-accuracy-training/2.0 (research project)'
def fetch_and_write(iid):
    meta = images[iid]
    dest_name = f"coco_extra_{iid:012d}"
    dest_image = train_images_dir / f'{dest_name}.jpg'
    dest_label = train_labels_dir / f'{dest_name}.txt'
    if dest_image.exists() and dest_label.exists():
        return iid, 'cached'
    url = f"https://images.cocodataset.org/train2017/{meta['file_name']}"
    last_error = None
    for attempt in range(4):
        try:
            request = urllib.request.Request(url, headers={'User-Agent': user_agent})
            with urllib.request.urlopen(request, timeout=45) as response:
                image_bytes = response.read()
            if len(image_bytes) < 1024:
                raise ValueError('downloaded image is unexpectedly small')
            dest_image.write_bytes(image_bytes)
            lines = []
            for ann in annotations_by_image.get(iid, []):
                target_id = category_to_target[ann['category_id']]
                x, y, w, h = map(float, ann['bbox'])
                iw, ih = float(meta['width']), float(meta['height'])
                left, top = max(0.0, x), max(0.0, y)
                right, bottom = min(iw, x + w), min(ih, y + h)
                bw, bh = max(0.0, right-left), max(0.0, bottom-top)
                if bw < 1 or bh < 1:
                    continue
                cx, cy = (left + right) / (2*iw), (top + bottom) / (2*ih)
                lines.append(f'{target_id} {cx:.6f} {cy:.6f} {bw/iw:.6f} {bh/ih:.6f}')
            dest_label.write_text('\n'.join(lines) + ('\n' if lines else ''))
            return iid, 'downloaded'
        except Exception as exc:
            last_error = exc
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"Failed COCO image {meta['file_name']}: {last_error}")

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as pool:
    statuses = list(pool.map(fetch_and_write, sorted(selected_ids)))
print(f"COCO expansion: {len(selected_ids)} new train images; "
      f"{sum(s == 'downloaded' for _, s in statuses)} downloaded, "
      f"{sum(s == 'cached' for _, s in statuses)} cached; {len(negative_ids)} are background negatives.")
print('Per raw COCO category image cap:', selected_by_name)

# Hard gate: verify labels/classes and leave validation/test untouched.
train_counts = Counter()
for label_file in train_labels_dir.glob('*.txt'):
    for line in label_file.read_text().splitlines():
        if line.strip():
            train_counts[GENERAL_CLASSES[int(line.split()[0])]] += 1
print('Training boxes after expansion:', dict(train_counts))
for name in GENERAL_CLASSES:
    if name == 'pothole':  # this class is from the separate public pothole source
        continue
    assert train_counts[name] >= MIN_BOXES_PER_CLASS, f'Insufficient training boxes for {name}'

expansion_manifest = {
    'source': 'official COCO 2017 train images/annotations',
    'cap_per_raw_category': MAX_EXTRA_IMAGES_PER_COCO_CLASS,
    'background_images': len(negative_ids),
    'added_train_images': len(selected_ids),
    'raw_category_image_counts': selected_by_name,
    'reserved_option_e_images_excluded': len(reserved_stems),
}
(DATA_ROOT_GENERAL / 'COCO_EXPANSION_MANIFEST.json').write_text(
    json.dumps(expansion_manifest, indent=2) + '\n'
)
print('Frozen Option E validation/test splits preserved. Continue to Step 7A.')

## Step 3 -- Discover images/labels, resolve class ids to names, assign capture sessions

Resolves every box to its real class **name** immediately, using the `SOURCE_CLASSES` the Step 2
cell you ran just declared -- never against a different target list's id positions. A box whose
source class maps to `None` (explicitly discarded in Step 2C's review) is dropped here, not
miscounted as some other class. Candidate-only pools (no reviewed bounding-box labels -- e.g.
this repo's `datasets/data/detection/egyptian/images/commons/`) are excluded by path, before any
counting happens, regardless of any `candidate_class` guess in their metadata.


In [ ]:
import pathlib

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
# Known candidate-only (unlabeled) paths that must never silently enter training even if someone
# points DATA_ROOT at a parent directory that happens to contain them.
EXCLUDED_CANDIDATE_ONLY_SUBSTRINGS = ["datasets/data/detection/egyptian/images/commons"]

def is_excluded_candidate_only(image_path: pathlib.Path) -> bool:
    as_str = str(image_path.resolve())
    return any(marker in as_str for marker in EXCLUDED_CANDIDATE_ONLY_SUBSTRINGS)

def resolve_boxes(label_path: pathlib.Path, source_classes):
    """Returns (resolved_boxes, malformed_count, discarded_by_review_count) for one label file.
    resolved_boxes is a list of (class_name, cx, cy, w, h) -- never a raw integer id."""
    resolved, malformed, discarded = [], 0, 0
    for line in label_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            malformed += 1; continue
        try:
            class_id = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:])
        except ValueError:
            malformed += 1; continue
        if not (0 <= class_id < len(source_classes)):
            malformed += 1; continue
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            malformed += 1; continue
        name = source_classes[class_id]
        if name is None:
            discarded += 1; continue  # explicitly reviewed-and-discarded (Step 2C), not an error
        resolved.append((name, cx, cy, w, h))
    return resolved, malformed, discarded

def discover(data_root: pathlib.Path, source_classes):
    images_dir = data_root / "images"
    labels_dir = data_root / "labels"
    if not images_dir.is_dir():
        # Roboflow's YOLOv8 export nests train/valid/test; flatten them into one pool here --
        # Step 5 performs the one split this project controls (by session), not Roboflow's own.
        images_dir, labels_dir = data_root, data_root
    records = []
    excluded_candidate_only = 0
    total_malformed = 0
    total_discarded_by_review = 0
    for image_path in sorted(p for p in images_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS):
        if is_excluded_candidate_only(image_path):
            excluded_candidate_only += 1
            continue
        label_path = (labels_dir / image_path.relative_to(images_dir)).with_suffix(".txt")
        if not label_path.is_file():
            flat = labels_dir / "labels" / f"{image_path.stem}.txt"
            label_path = flat if flat.is_file() else label_path
        if label_path.is_file() and label_path.stat().st_size > 0:
            boxes, malformed, discarded = resolve_boxes(label_path, source_classes)
            total_malformed += malformed
            total_discarded_by_review += discarded
            has_label = True
        else:
            boxes, has_label = [], False
        match = SESSION_ID_PATTERN.match(image_path.stem)
        session_id = match.group("session") if match else image_path.stem
        records.append({
            "image": image_path, "session": session_id, "has_label": has_label, "boxes": boxes,
        })
    return records, excluded_candidate_only, total_malformed, total_discarded_by_review

records, excluded_candidate_only, total_malformed, total_discarded_by_review = discover(
    DATA_ROOT, SOURCE_CLASSES
)
print(f"Found {len(records)} candidate images under {DATA_ROOT}")
print(f"Excluded as known candidate-only/unlabeled pool (never enters training): "
      f"{excluded_candidate_only}")
print(f"Malformed label lines (dropped): {total_malformed}")
print(f"Boxes explicitly discarded by reviewed class mapping (Step 2C 'None'): "
      f"{total_discarded_by_review}")
print(f"Found {len({r['session'] for r in records})} distinct capture sessions")


## Step 4A -- Part A / general model: validate labels and gate on data sufficiency

Counts real, name-resolved bounding-box annotations per `GENERAL_CLASSES` class. This is
expected to pass today using `datasets/prepared/avn_coco/` (real COCO ground truth, see
`class_counts.md`, remapped per Option A's note) plus Option D's two public HF sets together.
**Passing this gate only means Part A (the general model) may proceed -- it says nothing about
the Egyptian model; see Step 4B.**

In [ ]:
from collections import Counter

def validate_and_count(records, target_classes, label_prefix):
    box_counts = Counter()
    image_counts_with_box = Counter()
    labeled_image_count = sum(1 for r in records if r["has_label"])
    unlabeled_image_count = len(records) - labeled_image_count
    for record in records:
        seen_classes_this_image = set()
        for name, cx, cy, w, h in record["boxes"]:
            if name not in target_classes:
                continue  # a class outside this target set -- not an error, just not counted here
            box_counts[name] += 1
            seen_classes_this_image.add(name)
        for name in seen_classes_this_image:
            image_counts_with_box[name] += 1
    print(f"[{label_prefix}] Total candidate images  : {len(records)}")
    print(f"[{label_prefix}] Unlabeled (no label file): {unlabeled_image_count} (excluded)")
    print(f"[{label_prefix}] Labeled images usable    : {labeled_image_count}")
    print(f"{'class':20s} {'boxes':>8s} {'images':>8s}")
    for name in target_classes:
        print(f"{name:20s} {box_counts[name]:>8d} {image_counts_with_box[name]:>8d}")
    return box_counts, image_counts_with_box, labeled_image_count

def sufficiency_report(target_classes, box_counts, image_counts_with_box, labeled_image_count, only_classes=None):
    missing = []
    if labeled_image_count < MIN_TOTAL_LABELED_IMAGES:
        missing.append(
            f"- Only {labeled_image_count} labeled images total; need at least {MIN_TOTAL_LABELED_IMAGES}."
        )
    for name in (only_classes or target_classes):
        if image_counts_with_box[name] < MIN_IMAGES_PER_CLASS:
            missing.append(
                f"- '{name}': only {image_counts_with_box[name]} labeled images "
                f"(need >= {MIN_IMAGES_PER_CLASS}); add more real bounding-box annotations, "
                f"not more unlabeled photos."
            )
        if box_counts[name] < MIN_BOXES_PER_CLASS:
            missing.append(f"- '{name}': only {box_counts[name]} boxes (need >= {MIN_BOXES_PER_CLASS}).")
    return missing

coco_box_counts, coco_image_counts, coco_labeled_total = validate_and_count(
    records, GENERAL_CLASSES, "Part A / general model"
)
coco_missing = sufficiency_report(GENERAL_CLASSES, coco_box_counts, coco_image_counts, coco_labeled_total)
COCO_GATE_PASSED = not coco_missing
if coco_missing:
    print("\nINSUFFICIENT DATA for Part A (general model) -- Part A will be skipped:")
    print("\n".join(coco_missing))
else:
    print("\nPart A (general model) data sufficiency gate PASSED (floors only, not an accuracy guarantee).")


## Step 4B -- Egyptian final model: validate labels and gate on data sufficiency

This is the gate that actually matters for the project deliverable. It counts real, name-resolved
boxes for `EGYPT_TARGET_CLASSES`, and separately calls out `EGYPT_HAZARD_ONLY_CLASSES` --
`microbus`, `tuk_tuk`, `pothole`, `debris`, `open_drain`, `curb_or_step`, `low_hanging_obstacle`
-- which have zero COCO equivalent, so a run that only ever used COCO/avn_coco data will show
exactly 0 here (by class *name*, not by reusing a COCO label id against this list's positions)
and correctly fail. **If this cell reports missing data, the Egyptian model is blocked. Do not
skip ahead to Steps 7B-10B, and do not report the Egyptian model as trained or complete.**


In [ ]:
egypt_box_counts, egypt_image_counts, egypt_labeled_total = validate_and_count(
    records, EGYPT_TARGET_CLASSES, "Part B / Egyptian"
)
egypt_missing = sufficiency_report(
    EGYPT_TARGET_CLASSES, egypt_box_counts, egypt_image_counts, egypt_labeled_total
)
hazard_missing = sufficiency_report(
    EGYPT_TARGET_CLASSES, egypt_box_counts, egypt_image_counts, egypt_labeled_total,
    only_classes=EGYPT_HAZARD_ONLY_CLASSES,
)
EGYPT_GATE_PASSED = not egypt_missing
if egypt_missing:
    report = "\n".join([
        "EGYPTIAN MODEL BLOCKED -- training NOT started, and Part B below will not run.",
        "This is the real, current state of the dataset, not a bug to silence:",
        "",
        *egypt_missing,
        "",
        "Hazard-only classes specifically (zero COCO equivalent -- these can only improve with",
        "genuine Egyptian-labeled data, never with more COCO/avn_coco images):",
        *(hazard_missing or ["  (hazard-only floors happen to be met, but the combined gate above is not)"]),
        "",
        "Also noted: images with no label file in this run were excluded entirely (see Step 3's",
        "counts), consistent with datasets/data/detection/egyptian/metadata/commons.jsonl's own",
        "\"annotation_status\": \"unlabeled\" tagging -- a candidate_class guess from a source",
        "category is never treated as ground truth here.",
        "",
        "What to do next: collect and label real Egyptian sessions per",
        "datasets/collection-protocol.md and datasets/labeling-guidelines.md, and/or add a",
        "Roboflow source whose classes you have reviewed and mapped in Step 2C. Then re-run from",
        "Step 2 with that data (concatenating with existing `records` if combining sources). Do",
        "not lower the floors above to force this to pass.",
    ])
    print(report)
else:
    print("\nPart B (Egyptian) data sufficiency gate PASSED (floors only, not an accuracy guarantee).")


## Step 5 -- Train/val/test split by capture session

Whole sessions move together, so no two splits can share near-duplicate frames from the same
burst/video. This is a greedy bin-packing by target ratio, not a random per-image split. Run
once; both Part A and Part B read from the same `split_records`, each filtering to its own
labeled subset (by resolved class name) when building its `dataset.yaml` in Step 6.


In [ ]:
import random

labeled_records = [r for r in records if r["has_label"]]
sessions = sorted({r["session"] for r in labeled_records})
rng = random.Random(SPLIT_SEED)
rng.shuffle(sessions)

session_sizes = Counter(r["session"] for r in labeled_records)
total = sum(session_sizes.values())
targets = {name: ratio * total for name, ratio in SPLIT_RATIOS.items()}
assigned = {name: 0 for name in SPLIT_RATIOS}
session_to_split = {}
for session in sessions:
    deficits = {name: targets[name] - assigned[name] for name in SPLIT_RATIOS}
    split = max(deficits, key=deficits.get)
    session_to_split[session] = split
    assigned[split] += session_sizes[session]

split_records = {name: [] for name in SPLIT_RATIOS}
for record in labeled_records:
    split_records[session_to_split[record["session"]]].append(record)

for name, items in split_records.items():
    print(f"{name}: {len(items)} images, {len({r['session'] for r in items})} sessions")

if EGYPT_GATE_PASSED:
    print("\nEgyptian hazard-only class presence per split (val/test must each have real examples --")
    print("if a class is missing from val or test, collect another labeled session for it; per")
    print("datasets/split-policy.md, report the real counts rather than forcing coverage):")
    for split_name, items in split_records.items():
        present = Counter()
        for record in items:
            for name, *_ in record["boxes"]:
                if name in EGYPT_HAZARD_ONLY_CLASSES:
                    present[name] += 1
        for name in EGYPT_HAZARD_ONLY_CLASSES:
            flag = "" if present[name] > 0 else "  <-- MISSING from this split"
            print(f"  [{split_name}] {name:20s} {present[name]:>5d} boxes{flag}")


## Step 6 -- Materialize the YOLO dataset layout + `dataset.yaml` (per part)

Writes two independent dataset trees -- `/content/yolo_dataset_coco` for Part A and
`/content/yolo_dataset_egypt` for Part B -- each re-encoding boxes to **its own** class order by
name (`target_classes.index(name)`), never by reusing another list's raw id. A box whose
resolved name isn't in a given part's target list is simply not written to that part's dataset --
e.g. a `transportation` box never ends up mislabeled as `pothole` in Part B's files, and Part B's
`microbus`/`tuk_tuk` boxes (if any existed) would never leak into Part A's single
`transportation` class id either, since name-based lookup only ever matches an exact string.

In [ ]:
import shutil, pathlib

def materialize(split_records, target_classes, root: pathlib.Path):
    if root.exists():
        shutil.rmtree(root)
    for split_name, items in split_records.items():
        (root / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split_name).mkdir(parents=True, exist_ok=True)
        for record in items:
            relevant = [b for b in record["boxes"] if b[0] in target_classes]
            if not relevant:
                continue  # nothing in this part's class set for this image -- skip it entirely
            image_dst = root / "images" / split_name / record["image"].name
            shutil.copy2(record["image"], image_dst)
            label_dst = (root / "labels" / split_name / record["image"].name).with_suffix(".txt")
            lines_out = [
                f"{target_classes.index(name)} {cx} {cy} {w} {h}" for name, cx, cy, w, h in relevant
            ]
            label_dst.write_text("\n".join(lines_out) + "\n")
    yaml_path = root / "dataset.yaml"
    yaml_path.write_text(
        "path: " + str(root) + "\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        "names:\n" + "\n".join(f"  {i}: {name}" for i, name in enumerate(target_classes)) + "\n"
    )
    return yaml_path

coco_dataset_yaml = materialize(split_records, GENERAL_CLASSES, pathlib.Path("/content/yolo_dataset_coco"))
print("Part A dataset.yaml:", coco_dataset_yaml)

if EGYPT_GATE_PASSED:
    egypt_dataset_yaml = materialize(split_records, EGYPT_TARGET_CLASSES, pathlib.Path("/content/yolo_dataset_egypt"))
    print("Part B dataset.yaml:", egypt_dataset_yaml)
else:
    egypt_dataset_yaml = None
    print("Part B skipped: Step 4B gate did not pass.")


## Step 7A -- Part A: fine-tune each candidate pretrained model (general classes)

Every run starts from published pretrained weights (`pretrained=True` is the `ultralytics`
default when a `.pt` checkpoint is passed) -- none of this trains a detector from scratch.
Ultralytics itself tracks the best epoch by its own fitness metric and writes `best.pt`
separately from `last.pt`; Step 8A uses `best.pt` from each run, never `last.pt`.

In [ ]:
from ultralytics import YOLO
import pathlib

coco_runs = {}
if not COCO_GATE_PASSED:
    print("SKIPPED: Part A (general model) gate did not pass. Part B can still run if its own gate passes.")
else:
    for weights in CANDIDATE_WEIGHTS:
        model = YOLO(weights)
        result = model.train(
            data=str(coco_dataset_yaml), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
            seed=SPLIT_SEED, patience=PATIENCE, close_mosaic=10, amp=True,
            project="/content/runs_coco", name=weights.replace(".pt", ""),
        )
        coco_runs[weights] = pathlib.Path(result.save_dir)
        print(weights, "->", coco_runs[weights])


## Step 8A -- Part A: compare candidates on the *same* validation split

Selection is by `mAP50-95` first, with per-class recall, model size and GPU inference time
(Colab T4 only -- a relative comparison between candidates, never a phone latency number).
**This produces Part A / the general model's numbers only -- these are not the Egyptian model's
metrics, and its `person`/`transportation` numbers mix generic COCO with South Asian dashcam
data (Option D) -- see Step 1/2D's caveats. Do not report these as Egyptian-accuracy numbers.**

In [ ]:
import time, pandas as pd

def compare_candidates(runs, dataset_yaml, target_classes, sample_split="val"):
    comparison_rows = []
    for weights, run_dir in runs.items():
        best_path = run_dir / "weights" / "best.pt"
        model = YOLO(str(best_path))
        metrics = model.val(data=str(dataset_yaml), split="val")
        per_class_recall = dict(zip(target_classes, metrics.box.r.tolist()))
        sample_dir = pathlib.Path(dataset_yaml).parent / "images" / sample_split
        sample = next(sample_dir.glob("*"))
        timings = []
        for _ in range(20):
            start = time.perf_counter()
            model.predict(str(sample), imgsz=IMG_SIZE, verbose=False)
            timings.append((time.perf_counter() - start) * 1000)
        avg_ms = sum(timings[5:]) / len(timings[5:])
        comparison_rows.append({
            "weights": weights, "map50_95": float(metrics.box.map), "map50": float(metrics.box.map50),
            **{f"recall[{name}]": per_class_recall.get(name) for name in target_classes},
            "size_mb": best_path.stat().st_size / (1024 * 1024),
            "colab_gpu_ms_per_image": avg_ms, "best_pt": str(best_path),
        })
    return pd.DataFrame(comparison_rows).sort_values("map50_95", ascending=False)

if not COCO_GATE_PASSED:
    print("SKIPPED: no Part A (general model) runs to compare (see Step 4A).")
else:
    coco_comparison = compare_candidates(coco_runs, coco_dataset_yaml, GENERAL_CLASSES)
    display(coco_comparison)


In [ ]:
if COCO_GATE_PASSED:
    COCO_BEST_WEIGHTS = coco_comparison.iloc[0]["weights"]
    COCO_BEST_CHECKPOINT = pathlib.Path(coco_comparison.iloc[0]["best_pt"])
    print(f"Part A selected (highest mAP50-95 on val): {COCO_BEST_WEIGHTS} -> {COCO_BEST_CHECKPOINT}")
    print("This is best.pt from that run, not last.pt. Part A / general model only -- not the Egyptian model.")
else:
    COCO_BEST_WEIGHTS = None
    COCO_BEST_CHECKPOINT = None


## Step 9A -- Part A: final test-set evaluation of the selected general model

The test split was never used for checkpoint selection or candidate comparison above.


In [ ]:
if COCO_GATE_PASSED:
    coco_final_model = YOLO(str(COCO_BEST_CHECKPOINT))
    coco_test_metrics = coco_final_model.val(data=str(coco_dataset_yaml), split="test")
    print("[Part A / general model] Held-out test mAP50-95:", float(coco_test_metrics.box.map))
    print("[Part A / general model] Held-out test per-class recall:")
    test_recall = dict(zip(GENERAL_CLASSES, coco_test_metrics.box.r.tolist()))
    for name, recall in test_recall.items():
        print(f"  {name:16s} {recall:.3f}")
    # Promotion targets are deliberately checked on the frozen held-out split. Do not tune on
    # this test set after seeing the result; a failed gate means keep the current Android model.
    promotion_checks = {
        "mAP50-95 >= 0.45 (baseline 0.387)": float(coco_test_metrics.box.map) >= 0.45,
        "person recall >= 0.57 (baseline 0.515)": test_recall.get("person", 0.0) >= 0.57,
        "transportation recall >= 0.60 (baseline 0.565)": test_recall.get("transportation", 0.0) >= 0.60,
        "bench recall >= 0.20 (baseline 0.05)": test_recall.get("bench", 0.0) >= 0.20,
        "chair recall >= 0.35 (baseline 0.18)": test_recall.get("chair", 0.0) >= 0.35,
        "pothole recall >= 0.65": test_recall.get("pothole", 0.0) >= 0.65,
    }
    PROMOTION_GATE_PASSED = all(promotion_checks.values())
    print("\nPromotion checks:")
    for check, passed in promotion_checks.items():
        print(("PASS" if passed else "FAIL") + "  " + check)
    print("\nPROMOTION_GATE_PASSED =", PROMOTION_GATE_PASSED)
else:
    PROMOTION_GATE_PASSED = False
    print("SKIPPED: no Part A (general model) checkpoint to test (see Step 4A).")


## Step 7B -- Part B: fine-tune the Egyptian model (only if Step 4B passed)

This cell is a no-op with a clear message if the Egyptian sufficiency gate did not pass --
it never trains on an insufficient/COCO-only dataset and calls the result the Egyptian model.


In [ ]:
egypt_runs = {}
if not EGYPT_GATE_PASSED:
    print("SKIPPED: Step 4B's Egyptian data-sufficiency gate did not pass. No Egyptian model is")
    print("trained here. Re-run from Step 2 once you have real labeled Egyptian data, per the")
    print("'What to do next' note Step 4B printed.")
else:
    for weights in CANDIDATE_WEIGHTS:
        model = YOLO(weights)
        result = model.train(
            data=str(egypt_dataset_yaml), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
            seed=SPLIT_SEED, patience=PATIENCE, close_mosaic=10, amp=True,
            project="/content/runs_egypt", name=weights.replace(".pt", ""),
        )
        egypt_runs[weights] = pathlib.Path(result.save_dir)
        print(weights, "->", egypt_runs[weights])


## Step 8B -- Part B: compare Egyptian-model candidates on the *same* validation split


In [ ]:
if not EGYPT_GATE_PASSED:
    print("SKIPPED: no Egyptian runs to compare (see Step 7B).")
else:
    egypt_comparison = compare_candidates(egypt_runs, egypt_dataset_yaml, EGYPT_TARGET_CLASSES)
    display(egypt_comparison)
    EGYPT_BEST_WEIGHTS = egypt_comparison.iloc[0]["weights"]
    EGYPT_BEST_CHECKPOINT = pathlib.Path(egypt_comparison.iloc[0]["best_pt"])
    print(f"Part B selected (highest mAP50-95 on val): {EGYPT_BEST_WEIGHTS} -> {EGYPT_BEST_CHECKPOINT}")


## Step 9B -- Part B: final test-set evaluation of the selected Egyptian model

Evaluated once, on the single model Step 8B picked. Per-class recall for
`EGYPT_HAZARD_ONLY_CLASSES` here is the real measure of whether the Egyptian-specific hazards
are actually being detected -- not just the generic classes they share with COCO.


In [ ]:
if not EGYPT_GATE_PASSED:
    print("SKIPPED: no Egyptian model was trained (see Step 7B/8B).")
else:
    egypt_final_model = YOLO(str(EGYPT_BEST_CHECKPOINT))
    egypt_test_metrics = egypt_final_model.val(data=str(egypt_dataset_yaml), split="test")
    print("[Part B / Egyptian] Held-out test mAP50-95:", float(egypt_test_metrics.box.map))
    print("[Part B / Egyptian] Held-out test per-class recall:")
    for name, recall in zip(EGYPT_TARGET_CLASSES, egypt_test_metrics.box.r.tolist()):
        flag = " (hazard-only, no COCO equivalent)" if name in EGYPT_HAZARD_ONLY_CLASSES else ""
        print(f"  {name:20s} {recall:.3f}{flag}")


## Step 10 -- Export to ONNX and reconcile with the Android app

Exports whichever of Part A / Part B actually produced a selected checkpoint. **Do not** just
drop the new `.onnx` into `android/app/src/main/assets/` without doing this -- the Kotlin
decoder (`YoloDecoder.decode`) reads class order from `DetectorConfig.labels`, and the two
parts here use two different class orders. Part A writes a versioned `yolov8s_general_v2.onnx` file only after its frozen-test promotion gate passes; it does not overwrite the current app asset. Keep class order, Kotlin fallback config, and manifest synchronized when promoting it.

In [ ]:
import hashlib, json as json_module, onnxruntime

def export_and_reconcile(model, target_classes, model_asset_name, version_tag):
    model.export(format="onnx", imgsz=IMG_SIZE)
    onnx_path = pathlib.Path(str(model.ckpt_path)).with_suffix(".onnx")
    assert onnx_path.is_file(), f"expected export at {onnx_path}"

    session = onnxruntime.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    input_meta = session.get_inputs()[0]
    output_meta = session.get_outputs()[0]
    expected_rows = 4 + len(target_classes)
    actual_rows = output_meta.shape[1]
    print("Input :", input_meta.name, input_meta.shape)
    print("Output:", output_meta.name, output_meta.shape)
    if actual_rows != expected_rows:
        raise SystemExit(
            f"Exported output has {actual_rows} rows but target_classes implies {expected_rows} "
            f"(4 + {len(target_classes)} classes). The decoder WILL mis-decode this -- do not ship it."
        )

    sha256 = hashlib.sha256(onnx_path.read_bytes()).hexdigest()
    print("\nSHA-256:", sha256)

    detector_json_snippet = {
        "modelAsset": model_asset_name, "inputSize": IMG_SIZE,
        "confidenceThreshold": 0.35, "nmsIouThreshold": 0.50,
        "classes": target_classes, "labels": target_classes,
    }
    print("\nPaste into android/app/src/main/resources/config/detector.json:")
    print(json_module.dumps(detector_json_snippet, indent=2))

    manifest_entry = {
        "version": version_tag, "file": model_asset_name, "sha256": sha256,
        "license": "<fill in: inherits the base checkpoint's license, e.g. AGPL-3.0 for YOLOv8 -- "
                   "confirm against https://github.com/ultralytics/ultralytics>",
        "input": f"[1,3,{IMG_SIZE},{IMG_SIZE}] float32 RGB, letterboxed and normalized to [0,1]",
        "output": f"[1,{actual_rows},N] float32: xywh plus {len(target_classes)} class scores "
                  f"in this order: {target_classes}",
    }
    print("\nPaste into models/manifest.json's \"models\" entry:")
    print(json_module.dumps(manifest_entry, indent=2))

    from google.colab import files
    files.download(str(onnx_path))
    return onnx_path, sha256

if COCO_GATE_PASSED and PROMOTION_GATE_PASSED:
    print("=== Part A / improved general model export ===")
    export_and_reconcile(
        coco_final_model, GENERAL_CLASSES, "yolov8s_general_v2.onnx",
        f"{COCO_BEST_WEIGHTS.replace('.pt', '')}-avn-general-v2",
    )
else:
    print("Part A export SKIPPED: the new model did not pass the frozen-test promotion gate. Keep the current Android model.")

if EGYPT_GATE_PASSED:
    print("\n=== Part B / Egyptian model export ===")
    export_and_reconcile(
        egypt_final_model, EGYPT_TARGET_CLASSES, "yolov8n_egypt_finetune.onnx",
        f"{EGYPT_BEST_WEIGHTS.replace('.pt', '')}-avn-egypt-finetune-1",
    )
else:
    print("\nPart B export SKIPPED: no Egyptian model exists yet (Step 4B did not pass).")


## Step 11 -- Run the accuracy update and promote safely

1. In Colab, select a GPU runtime (T4 or better). Run the install/config cells.
2. Run **Option E** to reconstruct the fixed baseline dataset from the sources. No dataset ZIP upload is needed.
3. Run **Option F**. It downloads COCO annotations and adds up to 2,500 extra train images per COCO source class plus 500 background-only images. It preserves Option E's validation and held-out test splits.
4. Run **Step 7A**, **Step 8A**, **Step 9A**, then **Step 10**. Training fine-tunes YOLOv8s for up to 80 epochs with early stopping. The final test is evaluated once; the notebook exports a versioned model only if mAP and weak-class recall pass the promotion gate.
5. If the gate fails, keep the current Android asset and do not tune against this test set. The next iteration needs a new held-out evaluation split.
6. If it passes, download `yolov8s_general_v2.onnx`, copy it to Android assets, update `detector.json` and `models/manifest.json` using the printed snippets, then run the ONNX verification and Android build workflow. Keep the current asset as rollback until the app runs on an Android device.

Do not run the old Steps 2-6 or use **Run all** after Option E/F; those cells would replace the fixed dataset and invalidate the frozen comparison. Part B remains blocked until there are real Egyptian bounding-box labels.
